# BioCLIP 2 — DIMER E2E species-classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bioclip2-biodiversity-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bioclip2-biodiversity-pipeline/blob/main/tutorials/bioclip2_biodiversity_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-imageomics%2Fbioclip--2-ffcc4d?style=flat)](https://huggingface.co/imageomics/bioclip-2) [![Upstream](https://img.shields.io/badge/Upstream-Imageomics%2Fbioclip--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/Imageomics/bioclip-2) [![arXiv](https://img.shields.io/badge/arXiv-2505.23883-b31b1b.svg)](https://arxiv.org/abs/2505.23883)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** zero-shot species classification, organism image embeddings and bounded species-classification fine-tuning

**This notebook is standalone.** It carries the repository's package (4 modules under `src/bioclip2_biodiversity_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `2957b322090f9cb17ae72c71981c7218a28d81e0` (~1711 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned BioCLIP 2 snapshot (3 files, 1.71 GB), builds the CLIP model from the pinned config and loads the weights strictly, decodes the 48 CC0 iNaturalist sparrow photographs carried inside this notebook (no dataset download), validates the image and dataset contracts, splits them into stratified train/validation/test sets, shows what the input validation rejects, computes image embeddings and checks they are reproducible, classifies the test split zero-shot from scientific names (the model's own prior), measures a majority-class and a colour baseline, runs a bounded AdamW fine-tuning of a linear head initialised from the zero-shot classifier, evaluates accuracy, macro-F1 and AUROC on the held-out test split, classifies six held-out images as new inputs, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about two minutes of model time after the download.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled organism photographs as a zip of `<label>/<image>` folders, or a zip holding a CSV (`id,image,label`), a JSON array or a JSONL file with image paths relative to it. They pass through the same image validation, stratified split, zero-shot and trivial baselines, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the sample. Supply the scientific name of each class in `BYOD_CLASS_PROMPTS` so zero-shot and the head initialisation can use it. The expected layout and the image ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

BioCLIP 2 is a vision-language foundation model for organismal biology: a CLIP model (ViT-L/14 image tower, 12-layer text tower, 768-dimensional joint space) trained by the Imageomics Institute on TreeOfLife-200M — about 214 million images covering 952 thousand taxa — with hierarchical contrastive learning over taxonomic names. Because images and taxon names share one space, it can classify a photograph against **any** list of species names with no training at all, and the upstream paper reports that this scaled training also produces representations that separate traits and ecological attributes it was never told about.

This tutorial exercises three things on real field data. The dataset is 48 photographs — 12 each of four North American sparrow species — from research-grade iNaturalist observations whose photos carry the CC0 licence, carried inside this notebook. The four species were chosen **before any result was seen** to make colour a weak cue: all four are streaked grey-brown birds, so a colour baseline has little to use and any separation comes from finer structure. First the model classifies them zero-shot from scientific names; then a linear head is fine-tuned starting *from* that zero-shot classifier, so every number after training is directly comparable to the number before it.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, sample, dataset and metrics modules; stage and digest-verify an immutable 1.71 GB snapshot and build the model from its pinned config rather than a library registry; validate images and split a labelled dataset without leakage; extract L2-normalised image embeddings and confirm they are reproducible; classify zero-shot from taxonomic names and read what the scores do and do not mean; measure majority-class and colour baselines; run a bounded fine-tuning whose starting point *is* the zero-shot classifier; evaluate accuracy, macro-F1 and AUROC on an independent test split; classify held-out images; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** the published BioCLIP 2 benchmarks (NABirds, Rare Species, Meta-Album, NeWT, FishNet and the rest), open-set or hierarchical (genus/family) prediction, image-to-image retrieval at scale, the newer `imageomics/bioclip-2.5-vith14` checkpoint, and any claim that a photograph shows what its label says. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — image embedding is about 0.3 s per image and the default head-only fine-tuning runs on features computed once — and CUDA is used automatically when present. The snapshot download is 1.71 GB.
- **Knowledge:** what a scientific (binomial) name is, what zero-shot classification with a vision-language model means, and how accuracy, macro-F1 and AUROC differ.
- **No remote code:** the checkpoint is a plain safetensors state dict loaded strictly into `open_clip`'s own CLIP module, built from the pinned `open_clip_config.json`. Nothing from the Hub is executed.
- **Data contract:** records are `{{id, image_bytes, label}}` in memory; images are JPEG, PNG or WEBP with a shorter side of at least 32 px and a longer side of at most 8192 px, decoded to RGB and centre-cropped to 224x224 by the CLIP transform; unique ids and unique image bytes; at least 8 records and 3 per class, 2..50 classes. BYOD accepts a zip of `<label>/<image>` folders or a CSV/JSON/JSONL of image paths.
- **Validation is decoding, not biology:** nothing checks that a photograph shows an organism, that it fills the frame, or that a label names a real taxon. A photo of a rock is embedded and classified without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. Location metadata is the sensitive part of a biodiversity record — the sample carries none, and a BYOD set should not either. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `imageomics/bioclip-2` snapshot (~1711 MB in total) at revision `2957b322090f…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `open_clip`, `safetensors` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'open_clip_torch==3.3.0',
    'timm==1.0.29',
    'ftfy==6.3.1',
    'regex==2026.9.10',
    'huggingface-hub==1.32.0',
    'safetensors==0.8.0',
    'pillow==12.3.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bioclip2-biodiversity-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/bioclip2_biodiversity_pipeline/pipeline.py',
    'embedded_modules': ['src/bioclip2_biodiversity_pipeline/pipeline.py', 'src/bioclip2_biodiversity_pipeline/sample_data.py', 'src/bioclip2_biodiversity_pipeline/metrics.py', 'src/bioclip2_biodiversity_pipeline/samples.py'],
    'module_sha256': '34f3a8cc12e7bf05189813919339f05438d6a3a2fd45722e4ab3bfe339466979',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, open_clip, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'open_clip': open_clip.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bioclip2_biodiversity_pipeline/` @ `uncommitted`)

The next 4 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/4:** `src/bioclip2_biodiversity_pipeline/pipeline.py`

In [ ]:
"""BioCLIP 2 (`imageomics/bioclip-2`) DIMER pipeline: verified snapshot, organism image embeddings,
zero-shot species classification from taxonomic names, and bounded species-classification fine-tuning
with a portable adapter.

BioCLIP 2 is a CLIP model (ViT-L/14 image tower, 12-layer masked-attention text tower, 768-d joint
space) trained on TreeOfLife-200M with hierarchical contrastive learning. It is published in the
`open_clip` checkpoint format, so this package builds the `open_clip.model.CLIP` module from the
**pinned** `open_clip_config.json` (not from open_clip's built-in model registry) and loads the
pinned `open_clip_model.safetensors` with `strict=True`. No remote code is executed: the checkpoint
is a plain state dict and the architecture is open_clip's own.

Two design choices are stated rather than hidden:

* **The text tokenizer is open_clip's bundled CLIP BPE**, not the `tokenizer.json` the upstream
  repository also ships. Both encode the same 49,408-token CLIP vocabulary; identical token ids
  were verified for the tutorial prompts (see docs/WEIGHTS.md). The HF tokenizer files are
  therefore not staged.
* **Adaptation initialises the classification head from the zero-shot text classifier** when class
  prompts are supplied, so fine-tuning starts exactly where zero-shot classification is and every
  epoch's validation number is comparable to the zero-shot number.

Everything model-related is imported lazily so that snapshot verification and input validation run
(and can refuse) before `torch` or `open_clip` are imported (fleet RTM-001).
"""

from __future__ import annotations

import hashlib
import io
import json
import warnings
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "imageomics/bioclip-2"
MODEL_REVISION = "2957b322090f9cb17ae72c71981c7218a28d81e0"
MODEL_LICENSE = "mit"
MODEL_KEY = "bioclip-2"
ARTIFACT_FORMAT = "org.valcorza.bioclip2-biodiversity.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
CONFIG_NAME = "open_clip_config.json"
WEIGHTS_NAME = "open_clip_model.safetensors"

# Architecture facts from the pinned open_clip_config.json; asserted against the file at load time.
EMBED_DIM = 768
IMAGE_SIZE = 224
VISION_LAYERS = 24
CONTEXT_LENGTH = 77
VOCAB_SIZE = 49408
# Ceilings. Images are resized to IMAGE_SIZE on the short side and centre-cropped, so anything from
# MIN_IMAGE_SIDE up is accepted; MAX_IMAGE_SIDE bounds decode memory. Batches are bounded so that a
# 1.7 GB tower on a CPU runtime finishes a call in seconds, not minutes.
MIN_IMAGE_SIDE = 32
MAX_IMAGE_SIDE = 8192
MAX_IMAGES_PER_CALL = 32
MAX_LABELS = 64
MAX_PROMPT_CHARS = 200
ACCEPTED_FORMATS = ("JPEG", "PNG", "WEBP")
# BioCLIP's prompt convention: "a photo of <name>." where the name is a taxonomic string. The
# upstream authors trained with full 7-rank taxonomic strings and scientific names; a scientific
# binomial is the recommended minimum.
DEFAULT_PROMPT_TEMPLATE = "a photo of {}."


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    for required in (CONFIG_NAME, WEIGHTS_NAME):
        if required not in listed:
            raise ValueError(f"manifest does not list {required}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = hashlib.sha256()
        with open(file_path, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                digest.update(chunk)
        if digest.hexdigest() != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest.hexdigest()} != manifest {entry['sha256']}")
    return manifest


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed file)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    return _verify_manifest(root, MODEL_ID, MODEL_REVISION)


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest but
    git-ignores the 1.7 GB safetensors file). `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "1..MAX_IMAGES_PER_CALL images, each as encoded bytes (JPEG, PNG or WEBP) or a PIL image",
    "images": [1, MAX_IMAGES_PER_CALL],
    "image_side_pixels": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "labels_per_zero_shot_call": [2, MAX_LABELS],
    "validation": (
        "decodability, format, mode convertible to RGB and pixel-size ceilings only. Nothing checks "
        "that an image shows an organism, that the organism fills the frame, or that a label names a "
        "real taxon -- a photo of a rock is embedded and classified without complaint"
    ),
    "preprocessing": (
        "resize so the shorter side is 224 px (bicubic), centre-crop 224x224, convert to RGB, "
        "normalise with the CLIP mean/std from the pinned open_clip_config.json; embeddings are "
        "L2-normalised 768-d vectors from the image tower's projection (CLIP convention)"
    ),
}


def decode_image(image: Any) -> Any:
    """Return a PIL RGB image from bytes, a file path, or a PIL image; raise on anything else.

    Pillow is imported here (not a model library). Decoding is the only way to know an image is
    valid, so validation and execution share this function and cannot diverge.
    """
    from PIL import Image, UnidentifiedImageError

    if isinstance(image, Image.Image):
        im = image
    elif isinstance(image, bytes | bytearray):
        try:
            im = Image.open(io.BytesIO(bytes(image)))
        except UnidentifiedImageError as exc:
            raise ValueError("image bytes could not be decoded as JPEG, PNG or WEBP") from exc
    elif isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise FileNotFoundError(f"image file not found: {path}")
        try:
            im = Image.open(path)
        except UnidentifiedImageError as exc:
            raise ValueError(f"{path.name} could not be decoded as JPEG, PNG or WEBP") from exc
    else:
        raise TypeError(f"image must be bytes, a path or a PIL image, got {type(image).__name__}")
    fmt = getattr(im, "format", None)
    if fmt is not None and fmt not in ACCEPTED_FORMATS:
        raise ValueError(f"image format {fmt} is not accepted; use one of {list(ACCEPTED_FORMATS)}")
    width, height = im.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image is {width}x{height}; the shorter side must be at least {MIN_IMAGE_SIDE} px")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image is {width}x{height}; the longer side must be at most {MAX_IMAGE_SIDE} px")
    im.load()
    return im.convert("RGB") if im.mode != "RGB" else im


def image_digest(image: Any) -> str:
    """SHA-256 of the encoded bytes (or, for a PIL image, of its raw RGB pixels and size)."""
    if isinstance(image, bytes | bytearray):
        return hashlib.sha256(bytes(image)).hexdigest()
    if isinstance(image, str | Path):
        return hashlib.sha256(Path(image).read_bytes()).hexdigest()
    im = decode_image(image)
    h = hashlib.sha256(f"{im.size[0]}x{im.size[1]}".encode())
    h.update(im.tobytes())
    return h.hexdigest()


def _check_images(images: Any, names: Any = None) -> tuple[list[Any], list[str], list[dict[str, Any]]]:
    """Raise TypeError/ValueError naming the first violated rule; return (pil_images, ids, facts).

    ``embed_images``, ``zero_shot``, ``classify`` and ``validate_inputs`` all route through this
    function so their acceptance criteria cannot diverge.
    """
    if isinstance(images, str | bytes | bytearray | Path) or not isinstance(images, Sequence):
        raise TypeError("images must be a list of image bytes, paths or PIL images")
    if not 1 <= len(images) <= MAX_IMAGES_PER_CALL:
        raise ValueError(f"images must hold 1..{MAX_IMAGES_PER_CALL} items, got {len(images)}")
    decoded: list[Any] = []
    facts: list[dict[str, Any]] = []
    for i, image in enumerate(images):
        try:
            im = decode_image(image)
        except (TypeError, ValueError, FileNotFoundError) as exc:
            raise type(exc)(f"images[{i}]: {exc}") from exc
        decoded.append(im)
        facts.append({"width": im.size[0], "height": im.size[1], "sha256": image_digest(image)})
    if names is None:
        ids = [f"img-{i}" for i in range(len(decoded))]
    else:
        if isinstance(names, str | bytes) or not isinstance(names, Sequence) or len(names) != len(decoded):
            raise ValueError("names must be a list with exactly one id per image")
        ids = [str(n) for n in names]
        if len(set(ids)) != len(ids):
            raise ValueError("names must be unique")
    return decoded, ids, facts


def _check_labels(labels: Any) -> list[tuple[str, str]]:
    """Return (label, prompt_text) pairs from a list of names or a {label: name} mapping."""
    if isinstance(labels, Mapping):
        pairs = [(str(k), str(v)) for k, v in labels.items()]
    elif isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of taxon names or a {label: taxon name} mapping")
    else:
        pairs = [(str(x), str(x)) for x in labels]
    if not 2 <= len(pairs) <= MAX_LABELS:
        raise ValueError(f"zero-shot classification needs 2..{MAX_LABELS} labels, got {len(pairs)}")
    if len({k for k, _ in pairs}) != len(pairs):
        raise ValueError("labels must be unique")
    for label, text in pairs:
        if not label.strip() or not text.strip():
            raise ValueError("labels and their prompt names must be non-empty")
        if len(text) > MAX_PROMPT_CHARS:
            raise ValueError(f"prompt name for {label!r} exceeds {MAX_PROMPT_CHARS} characters")
    return pairs


def _softmax(logits: Sequence[float]) -> list[float]:
    import math

    top = max(logits)
    exps = [math.exp(v - top) for v in logits]
    total = sum(exps)
    return [v / total for v in exps]


@dataclass
class BioClip2Pipeline:
    """BioCLIP 2 pipeline: `embed_images`, `embed_texts` and `zero_shot` always; `classify` after
    `adapt` or `from_artifact`."""

    _image_embedder: Callable[[list[Any]], list[list[float]]]
    _text_embedder: Callable[[list[str]], list[list[float]]]
    device: str
    logit_scale: float = 100.0
    load_warnings: list[str] = field(default_factory=list)
    classes: list[str] = field(default_factory=list)
    _classifier: Callable[[list[Any]], list[list[float]]] | None = None
    model: Any = None
    preprocess: Any = None
    tokenizer: Any = None
    classifier_model: Any = None
    weights_dir: Path | None = None
    adaptation: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BioClip2Pipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(f"no snapshot manifest at {root} and allow_download={allow_download}")
        # Stage and verify before importing model libraries (RTM-001).
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        cfg = json.loads((root / CONFIG_NAME).read_text(encoding="utf-8"))
        model_cfg, pre_cfg = cfg["model_cfg"], cfg["preprocess_cfg"]
        expected = {
            "embed_dim": (model_cfg["embed_dim"], EMBED_DIM),
            "image_size": (model_cfg["vision_cfg"]["image_size"], IMAGE_SIZE),
            "vision_layers": (model_cfg["vision_cfg"]["layers"], VISION_LAYERS),
            "context_length": (model_cfg["text_cfg"]["context_length"], CONTEXT_LENGTH),
            "vocab_size": (model_cfg["text_cfg"]["vocab_size"], VOCAB_SIZE),
        }
        drift = {k: v for k, v in expected.items() if v[0] != v[1]}
        if drift:
            raise ValueError(f"pinned open_clip_config.json disagrees with the package constants: {drift}")

        import torch
        from open_clip import image_transform
        from open_clip.model import CLIP
        from open_clip.tokenizer import SimpleTokenizer
        from safetensors.torch import load_file

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            # Built from the pinned config, not from open_clip's registry, so the architecture the
            # weights load into is the one the snapshot describes. `quick_gelu` is absent from the
            # pinned config and defaults to False (LAION-2B lineage, not OpenAI's).
            model = CLIP(
                embed_dim=model_cfg["embed_dim"],
                vision_cfg=model_cfg["vision_cfg"],
                text_cfg=model_cfg["text_cfg"],
                quick_gelu=bool(model_cfg.get("quick_gelu", False)),
            )
            state = load_file(str(root / WEIGHTS_NAME))
            model.load_state_dict(state, strict=True)
            del state
            preprocess = image_transform(
                model_cfg["vision_cfg"]["image_size"],
                is_train=False,
                mean=tuple(pre_cfg["mean"]),
                std=tuple(pre_cfg["std"]),
                resize_mode=pre_cfg.get("resize_mode", "shortest"),
                interpolation=pre_cfg.get("interpolation", "bicubic"),
            )
            tokenizer = SimpleTokenizer(context_length=model_cfg["text_cfg"]["context_length"])
        model = model.to(resolved_device).eval()
        messages = [f"{w.category.__name__}: {w.message}" for w in caught]
        pipe = cls(
            cls._make_image_embedder(model, preprocess, resolved_device),
            cls._make_text_embedder(model, tokenizer, resolved_device),
            resolved_device,
            float(model.logit_scale.exp().item()),
            messages,
        )
        pipe.model, pipe.preprocess, pipe.tokenizer, pipe.weights_dir = model, preprocess, tokenizer, root
        return pipe

    # -- backends ---------------------------------------------------------------------------------

    @classmethod
    def _make_image_embedder(
        cls, model: Any, preprocess: Any, device: str
    ) -> Callable[[list[Any]], list[list[float]]]:
        import torch

        def embedder(images: list[Any]) -> list[list[float]]:
            batch = torch.stack([preprocess(im) for im in images]).to(device)
            # no_grad, not inference_mode: tensors produced here must stay usable by a later
            # training epoch that shares this module.
            with torch.no_grad():
                feats = model.encode_image(batch, normalize=True)
            return feats.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_text_embedder(
        cls, model: Any, tokenizer: Any, device: str
    ) -> Callable[[list[str]], list[list[float]]]:
        import torch

        def embedder(texts: list[str]) -> list[list[float]]:
            tokens = tokenizer(texts).to(device)
            with torch.no_grad():
                feats = model.encode_text(tokens, normalize=True)
            return feats.float().cpu().tolist()

        return embedder

    @classmethod
    def _make_classifier(
        cls, clf: Any, preprocess: Any, device: str
    ) -> Callable[[list[Any]], list[list[float]]]:
        import torch

        def classifier(images: list[Any]) -> list[list[float]]:
            batch = torch.stack([preprocess(im) for im in images]).to(device)
            with torch.no_grad():
                logits = clf(batch)
            return logits.float().cpu().tolist()

        return classifier

    # -- public stages ----------------------------------------------------------------------------

    def embed_images(self, images: Sequence[Any], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """L2-normalised image-tower embedding per image (EMBED_DIM floats each)."""
        decoded, ids, facts = _check_images(images, names)
        vectors = self._image_embedder(decoded)
        if len(vectors) != len(decoded) or any(len(v) != EMBED_DIM for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "ids": ids,
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": EMBED_DIM,
            "pooling": "image-tower projection, L2-normalised (CLIP joint space)",
            "unit": "one unit vector per image; representations, not species predictions",
            "images": [{"id": i, **f} for i, f in zip(ids, facts, strict=True)],
            "n_images": len(decoded),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def embed_texts(self, texts: Sequence[str]) -> dict[str, Any]:
        """L2-normalised text-tower embedding per string (same joint space as the images)."""
        if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
            raise TypeError("texts must be a list of strings")
        if not 1 <= len(texts) <= MAX_LABELS:
            raise ValueError(f"texts must hold 1..{MAX_LABELS} items, got {len(texts)}")
        for i, t in enumerate(texts):
            if not isinstance(t, str) or not t.strip():
                raise ValueError(f"texts[{i}] must be a non-empty string")
            if len(t) > MAX_PROMPT_CHARS:
                raise ValueError(f"texts[{i}] exceeds {MAX_PROMPT_CHARS} characters")
        vectors = self._text_embedder([str(t) for t in texts])
        if len(vectors) != len(texts) or any(len(v) != EMBED_DIM for v in vectors):
            raise RuntimeError("backend returned embeddings of the wrong shape")
        return {
            "texts": list(texts),
            "embeddings": [[float(x) for x in v] for v in vectors],
            "dimension": EMBED_DIM,
            "n_texts": len(texts),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def zero_shot(
        self,
        images: Sequence[Any],
        labels: Sequence[str] | Mapping[str, str],
        *,
        names: Sequence[str] | None = None,
        template: str = DEFAULT_PROMPT_TEMPLATE,
    ) -> dict[str, Any]:
        """Classify images against a label set with no training: softmax over scaled cosine
        similarities between each image embedding and the text embedding of each label's prompt.

        `labels` is a list of taxon names, or a `{label: taxon name}` mapping when the reported
        label should differ from the text put into the prompt (e.g. a dataset key -> a scientific
        name). `template` must contain one `{}` placeholder.
        """
        if "{}" not in template:
            raise ValueError("template must contain a '{}' placeholder for the taxon name")
        pairs = _check_labels(labels)
        decoded, ids, facts = _check_images(images, names)
        prompts = [template.format(text) for _, text in pairs]
        text_vecs = self._text_embedder(prompts)
        image_vecs = self._image_embedder(decoded)
        classes = [label for label, _ in pairs]
        predictions = []
        for iid, fact, iv in zip(ids, facts, image_vecs, strict=True):
            logits = [self.logit_scale * sum(a * b for a, b in zip(iv, tv, strict=True)) for tv in text_vecs]
            scores = _softmax(logits)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": iid,
                    "sha256": fact["sha256"],
                    "label": classes[best],
                    "score": scores[best],
                    "scores": dict(zip(classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": classes,
            "prompts": dict(zip(classes, prompts, strict=True)),
            "template": template,
            "logit_scale": self.logit_scale,
            "decision_rule": (
                "argmax over softmax(logit_scale * cosine(image, text)); scores are relative to the "
                "supplied label set only and are not calibrated probabilities"
            ),
            "n_images": len(decoded),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def zero_shot_evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        class_prompts: Mapping[str, str],
        *,
        template: str = DEFAULT_PROMPT_TEMPLATE,
    ) -> dict[str, Any]:
        """Zero-shot metrics on a labelled dataset: the model's own prior, before any training."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        # An evaluation split may legitimately be small; the coverage rule applies to training data.
        manifest = validate_dataset(records, classes=list(class_prompts), min_records=2, min_per_class=1)
        classes = list(manifest["classes"])
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_IMAGES_PER_CALL):
            chunk = records[start : start + MAX_IMAGES_PER_CALL]
            result = self.zero_shot(
                [r["image_bytes"] for r in chunk],
                {c: class_prompts[c] for c in classes},
                names=[r["id"] for r in chunk],
                template=template,
            )
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in classes])
        metrics = classification_metrics([r["label"] for r in records], predicted, scores, classes)
        return {"baseline": "zero-shot (text classifier, no training)", "template": template, **metrics}

    def classify(self, images: Sequence[Any], *, names: Sequence[str] | None = None) -> dict[str, Any]:
        """Class scores and argmax label per image from the adapted head; requires a prior `adapt`
        or `from_artifact`."""
        if self._classifier is None or not self.classes:
            raise RuntimeError(
                "classify requires an adapted head: call adapt(...) or load from_artifact(...) first"
            )
        decoded, ids, facts = _check_images(images, names)
        logits = self._classifier(decoded)
        predictions = []
        for iid, fact, row in zip(ids, facts, logits, strict=True):
            if len(row) != len(self.classes):
                raise RuntimeError("backend returned a logits row that does not match the class list")
            scores = _softmax(row)
            best = max(range(len(scores)), key=scores.__getitem__)
            predictions.append(
                {
                    "id": iid,
                    "sha256": fact["sha256"],
                    "label": self.classes[best],
                    "score": scores[best],
                    "scores": dict(zip(self.classes, scores, strict=True)),
                }
            )
        return {
            "predictions": predictions,
            "classes": list(self.classes),
            "decision_rule": (
                "argmax over softmax(logits); scores are softmax outputs, not calibrated probabilities"
            ),
            "n_images": len(decoded),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "adaptation": dict(self.adaptation),
        }

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Held-out species-classification metrics (see metrics.classification_metrics)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import classification_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        validate_dataset(records, classes=self.classes, min_records=2, min_per_class=1)
        predicted: list[str] = []
        scores: list[list[float]] = []
        for start in range(0, len(records), MAX_IMAGES_PER_CALL):
            chunk = records[start : start + MAX_IMAGES_PER_CALL]
            result = self.classify([r["image_bytes"] for r in chunk], names=[r["id"] for r in chunk])
            for p in result["predictions"]:
                predicted.append(p["label"])
                scores.append([p["scores"][c] for c in self.classes])
        return classification_metrics([r["label"] for r in records], predicted, scores, self.classes)

    def _build_classifier(self, n_classes: int) -> Any:
        """A fresh copy of the image tower with a linear head on its (unnormalised) projection."""
        import copy

        import torch

        class ImageClassifier(torch.nn.Module):
            def __init__(self, visual: Any, embed_dim: int, n: int) -> None:
                super().__init__()
                self.visual = visual
                self.head = torch.nn.Linear(embed_dim, n, bias=True)

            def forward(self, pixels: Any) -> Any:
                feats = self.visual(pixels)
                feats = feats / feats.norm(dim=-1, keepdim=True)
                return self.head(feats)

        return ImageClassifier(copy.deepcopy(self.model.visual), EMBED_DIM, n_classes)

    def adapt(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        classes: Sequence[str] | None = None,
        class_prompts: Mapping[str, str] | None = None,
        template: str = DEFAULT_PROMPT_TEMPLATE,
        epochs: int = 4,
        learning_rate: float = 1e-4,
        batch_size: int = 8,
        trainable_blocks: int = 0,
        weight_decay: float = 0.01,
        seed: int = 42,
    ) -> dict[str, Any]:
        """Bounded gradient fine-tuning of a species-classification head on the verified base.

        Copies the image tower, adds a linear head over its L2-normalised projection, freezes
        everything except the head and the last `trainable_blocks` transformer blocks of the tower,
        and runs AdamW for `epochs` passes. When `class_prompts` maps every class to a taxon name,
        the head's weights are initialised from the zero-shot text classifier (scaled by the
        checkpoint's logit scale) so epoch 0 *is* zero-shot classification; otherwise the head is
        randomly initialised. Validation records are monitored per epoch only; the final epoch's
        weights are kept (no selection).

        `trainable_blocks=0` (the default) trains the head alone on image features computed once by
        the frozen tower, which is what a few-dozen-image dataset supports: on the tutorial sample,
        unfreezing even one block of the 24 degraded a saturated zero-shot classifier (recorded in
        MODEL_CARD.md). With `trainable_blocks>0` the tower runs every epoch and the unfrozen blocks
        receive gradients.
        """
        if self.model is None or self.preprocess is None or self.weights_dir is None:
            raise RuntimeError("adapt requires a pipeline built by from_pretrained (no loaded base model)")
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not 1 <= int(epochs) <= 50:
            raise ValueError("epochs must be in 1..50 (tutorial-scale adaptation)")
        if not 1 <= int(batch_size) <= MAX_IMAGES_PER_CALL:
            raise ValueError(f"batch_size must be in 1..{MAX_IMAGES_PER_CALL}")
        if not 0 <= int(trainable_blocks) <= VISION_LAYERS:
            raise ValueError(
                f"trainable_blocks must be in 0..{VISION_LAYERS} (the image tower has that many)"
            )
        train_manifest = validate_dataset(train_records, classes=classes)
        class_list = list(train_manifest["classes"])
        if val_records is not None:
            validate_dataset(val_records, classes=class_list, min_records=2, min_per_class=1)
        head_init = "random"
        if class_prompts is not None:
            missing = [c for c in class_list if c not in class_prompts]
            if missing:
                raise ValueError(f"class_prompts lacks an entry for classes {missing}")
            _check_labels({c: class_prompts[c] for c in class_list})
            head_init = "zero-shot text classifier"

        import random

        import torch

        random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        clf = self._build_classifier(len(class_list))
        if head_init != "random":
            prompts = [template.format(class_prompts[c]) for c in class_list]  # type: ignore[index]
            text_vecs = torch.tensor(self._text_embedder(prompts), dtype=torch.float32)
            with torch.no_grad():
                clf.head.weight.copy_(text_vecs * self.logit_scale)
                clf.head.bias.zero_()
        clf = clf.to(self.device)
        for p in clf.parameters():
            p.requires_grad = False
        blocks = clf.visual.transformer.resblocks
        for block in blocks[len(blocks) - int(trainable_blocks) :] if trainable_blocks else []:
            for p in block.parameters():
                p.requires_grad = True
        for p in clf.head.parameters():
            p.requires_grad = True
        trainable = [n for n, p in clf.named_parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in clf.parameters() if p.requires_grad)
        n_total = sum(p.numel() for p in clf.parameters())
        optimizer = torch.optim.AdamW(
            [p for p in clf.parameters() if p.requires_grad], lr=learning_rate, weight_decay=weight_decay
        )
        label_index = {c: i for i, c in enumerate(class_list)}
        examples = [(decode_image(r["image_bytes"]), label_index[r["label"]]) for r in train_records]
        self.classes = class_list
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.preprocess, self.device)
        loss_fn = torch.nn.CrossEntropyLoss()
        cached: list[Any] | None = None
        if not trainable_blocks:
            # Frozen tower: its output for a fixed image never changes, so compute it once.
            clf.eval()
            cached = []
            with torch.no_grad():
                for start in range(0, len(examples), MAX_IMAGES_PER_CALL):
                    pixels = torch.stack(
                        [self.preprocess(im) for im, _ in examples[start : start + MAX_IMAGES_PER_CALL]]
                    ).to(self.device)
                    feats = clf.visual(pixels)
                    cached.extend(feats / feats.norm(dim=-1, keepdim=True))

        history: list[dict[str, Any]] = []
        if val_records:
            clf.eval()
            val0 = self.evaluate(val_records)
            history.append(
                {
                    "epoch": 0,
                    "train_loss": None,
                    "n_batches": 0,
                    "val_accuracy": val0["accuracy"],
                    "val_macro_f1": val0["macro_f1"],
                    "note": f"before training; head = {head_init}",
                }
            )
        for epoch in range(1, int(epochs) + 1):
            clf.train()
            order = list(range(len(examples)))
            random.shuffle(order)
            total_loss, n_batches = 0.0, 0
            for start in range(0, len(order), int(batch_size)):
                idx = order[start : start + int(batch_size)]
                labels = torch.tensor([examples[i][1] for i in idx], device=self.device)
                optimizer.zero_grad()
                if cached is not None:
                    logits = clf.head(torch.stack([cached[i] for i in idx]))
                else:
                    pixels = torch.stack([self.preprocess(examples[i][0]) for i in idx]).to(self.device)
                    logits = clf(pixels)
                loss = loss_fn(logits, labels)
                loss.backward()
                optimizer.step()
                total_loss += float(loss.item())
                n_batches += 1
            clf.eval()
            entry: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / max(1, n_batches), 6),
                "n_batches": n_batches,
            }
            if val_records:
                val = self.evaluate(val_records)
                entry["val_accuracy"] = val["accuracy"]
                entry["val_macro_f1"] = val["macro_f1"]
            history.append(entry)
        clf.eval()
        self.adaptation = {
            "method": "gradient fine-tuning (AdamW) of a linear head over the L2-normalised image projection"
            + (
                f" and the last {int(trainable_blocks)} image-tower block(s)"
                if trainable_blocks
                else " (frozen tower; features computed once)"
            ),
            "head_initialisation": head_init,
            "template": template if head_init != "random" else None,
            "classes": class_list,
            "epochs": int(epochs),
            "learning_rate": float(learning_rate),
            "batch_size": int(batch_size),
            "weight_decay": float(weight_decay),
            "trainable_blocks": int(trainable_blocks),
            "seed": int(seed),
            "precision": "float32",
            "trainable_parameters": int(n_trainable),
            "total_parameters": int(n_total),
            "trainable_parameter_names": trainable,
            "train_records": len(train_records),
            "val_records": len(val_records) if val_records else 0,
            "selection": "final epoch kept; validation metrics are monitoring only",
            "history": history,
        }
        return dict(self.adaptation)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Export the trainable tensors as safetensors plus a JSON manifest binding them to the base."""
        if self.classifier_model is None or not self.classes:
            raise RuntimeError("save_artifact requires an adapted head (call adapt first)")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adaptation.get("trainable_parameter_names", []))
        state = self.classifier_model.state_dict()
        tensors = {k: v.detach().cpu().contiguous() for k, v in state.items() if k in names}
        if not tensors:
            raise RuntimeError("no trainable tensors recorded; nothing to export")
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path))
        digest = hashlib.sha256(weights_path.read_bytes()).hexdigest()
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {"model_id": MODEL_ID, "model_revision": MODEL_REVISION, "license": MODEL_LICENSE},
            "requires_remote_code": False,
            "classes": list(self.classes),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": digest}
            ],
            "tensors": sorted(tensors),
            "serving_state_tensors": [],
            "adaptation": {k: v for k, v in self.adaptation.items() if k != "trainable_parameter_names"},
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Rebuild the classification head from an exported artifact (manifest verified before loading)."""
        if self.model is None or self.weights_dir is None:
            raise RuntimeError("load_artifact requires a pipeline built by from_pretrained")
        art = Path(artifact_dir)
        manifest_path = art / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("model_id"), base.get("model_revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError(f"artifact was trained on {base}, this package pins {MODEL_ID}@{MODEL_REVISION}")
        classes = [str(c) for c in manifest.get("classes", [])]
        if len(classes) < 2 or len(set(classes)) != len(classes):
            raise ValueError("artifact manifest must list at least two unique classes")
        for entry in manifest["files"]:
            fp = art / entry["path"]
            if not fp.is_file():
                raise FileNotFoundError(f"artifact file missing: {fp}")
            if fp.stat().st_size != entry["bytes"]:
                raise ValueError(f"{entry['path']}: size {fp.stat().st_size} != manifest {entry['bytes']}")
            if hashlib.sha256(fp.read_bytes()).hexdigest() != entry["sha256"]:
                raise ValueError(f"{entry['path']}: sha256 mismatch against the artifact manifest")
        from safetensors.torch import load_file

        clf = self._build_classifier(len(classes))
        tensors = load_file(str(art / ARTIFACT_WEIGHTS_NAME))
        if set(tensors) != set(manifest.get("tensors", [])):
            raise ValueError("artifact tensors do not match the names listed in its manifest")
        _missing, unexpected = clf.load_state_dict(tensors, strict=False)
        if unexpected:
            raise ValueError(
                f"artifact carries tensors the base architecture does not have: {sorted(unexpected)[:5]}"
            )
        clf = clf.to(self.device).eval()
        self.classes = classes
        self.classifier_model = clf
        self._classifier = self._make_classifier(clf, self.preprocess, self.device)
        self.adaptation = {**manifest.get("adaptation", {}), "loaded_from_artifact": str(art)}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BioClip2Pipeline:
        """Verified base snapshot + exported adapter, ready for `classify`."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe


def validate_inputs(images: Sequence[Any], *, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, verdict).

    Decodes every image (that is the only real check), records its size and digest, and raises
    exactly what `embed_images` / `classify` would raise. Nothing here knows what the image shows.
    """
    _decoded, ids, facts = _check_images(images, names)
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": i, **f} for i, f in zip(ids, facts, strict=True)],
        "n_images": len(ids),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "requires_remote_code": False,
    }

**Module 2/4:** `src/bioclip2_biodiversity_pipeline/sample_data.py` (carried verbatim; see the note above)

In [ ]:
"""Embedded sample images for the BioCLIP 2 tutorial: 48 CC0 iNaturalist photographs of four sparrow species.

Generated once by the build script recorded in docs/WEIGHTS.md from the iNaturalist API
(https://api.inaturalist.org/v1/observations, research-grade observations whose *photo* carries the
CC0 1.0 licence code). Each image was centre-cropped to a square and resized to 224x224 JPEG (quality
85). Observation and photo ids are recorded so every image is traceable to its source; observer logins
are recorded as courtesy attribution (CC0 requires none); observation locations were deliberately NOT
collected. Do not edit by hand: the dataset digest in the tutorial depends on these bytes.
"""

from __future__ import annotations

SAMPLE_SOURCE = "iNaturalist research-grade observations, photo licence CC0 1.0, fetched 2026-09-18"
SAMPLE_IMAGE_LICENSE = "CC0-1.0"
SAMPLE_IMAGE_SIDE = 224

# One entry per image: file, label, scientific and common name, and the iNaturalist provenance.
SAMPLE_RECORDS: tuple[dict[str, str | int], ...] = (
    {'file': 'song_sparrow_00.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 79016324, 'inat_observation_url': 'https://www.inaturalist.org/observations/79016324', 'inat_photo_id': 129376982, 'license_code': 'cc0', 'observer': 'andywilson', 'observed_on': '2021-05-14'},
    {'file': 'song_sparrow_01.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 267636534, 'inat_observation_url': 'https://www.inaturalist.org/observations/267636534', 'inat_photo_id': 480991086, 'license_code': 'cc0', 'observer': 'lyneisfilm', 'observed_on': '2025-03-29'},
    {'file': 'song_sparrow_02.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 302980489, 'inat_observation_url': 'https://www.inaturalist.org/observations/302980489', 'inat_photo_id': 546060381, 'license_code': 'cc0', 'observer': 'swpollinators', 'observed_on': '2025-08-01'},
    {'file': 'song_sparrow_03.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 177450028, 'inat_observation_url': 'https://www.inaturalist.org/observations/177450028', 'inat_photo_id': 308625896, 'license_code': 'cc0', 'observer': 'radrat', 'observed_on': '2023-08-08'},
    {'file': 'song_sparrow_04.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 275349085, 'inat_observation_url': 'https://www.inaturalist.org/observations/275349085', 'inat_photo_id': 494793016, 'license_code': 'cc0', 'observer': 'k-simpkins', 'observed_on': '2025-04-27'},
    {'file': 'song_sparrow_05.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 193339933, 'inat_observation_url': 'https://www.inaturalist.org/observations/193339933', 'inat_photo_id': 339623726, 'license_code': 'cc0', 'observer': 'rawcomposition', 'observed_on': '2023-04-05'},
    {'file': 'song_sparrow_06.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 369029444, 'inat_observation_url': 'https://www.inaturalist.org/observations/369029444', 'inat_photo_id': 674054489, 'license_code': 'cc0', 'observer': 'ben142', 'observed_on': '2026-06-06'},
    {'file': 'song_sparrow_07.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 90171417, 'inat_observation_url': 'https://www.inaturalist.org/observations/90171417', 'inat_photo_id': 148994242, 'license_code': 'cc0', 'observer': 'glennberry', 'observed_on': '2021-08-06'},
    {'file': 'song_sparrow_08.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 130949329, 'inat_observation_url': 'https://www.inaturalist.org/observations/130949329', 'inat_photo_id': 222768957, 'license_code': 'cc0', 'observer': 'davidfbird', 'observed_on': '2022-08-14'},
    {'file': 'song_sparrow_09.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 107953669, 'inat_observation_url': 'https://www.inaturalist.org/observations/107953669', 'inat_photo_id': 181658744, 'license_code': 'cc0', 'observer': 'gcart043', 'observed_on': '2022-02-06'},
    {'file': 'song_sparrow_10.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 349374463, 'inat_observation_url': 'https://www.inaturalist.org/observations/349374463', 'inat_photo_id': 637315932, 'license_code': 'cc0', 'observer': 'sooji', 'observed_on': '2026-04-12'},
    {'file': 'song_sparrow_11.jpg', 'label': 'song_sparrow', 'scientific_name': 'Melospiza melodia', 'common_name': 'Song Sparrow', 'inat_observation_id': 262252507, 'inat_observation_url': 'https://www.inaturalist.org/observations/262252507', 'inat_photo_id': 471146686, 'license_code': 'cc0', 'observer': 'jeanpaulboerekamps', 'observed_on': '2025-02-17'},
    {'file': 'chipping_sparrow_00.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 117809422, 'inat_observation_url': 'https://www.inaturalist.org/observations/117809422', 'inat_photo_id': 198992636, 'license_code': 'cc0', 'observer': 'k-simpkins', 'observed_on': '2022-05-18'},
    {'file': 'chipping_sparrow_01.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 144599194, 'inat_observation_url': 'https://www.inaturalist.org/observations/144599194', 'inat_photo_id': 248210057, 'license_code': 'cc0', 'observer': 'w_mark_c', 'observed_on': '2022-12-17'},
    {'file': 'chipping_sparrow_02.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 94266719, 'inat_observation_url': 'https://www.inaturalist.org/observations/94266719', 'inat_photo_id': 156350853, 'license_code': 'cc0', 'observer': 'ellyne', 'observed_on': '2021-05-24'},
    {'file': 'chipping_sparrow_03.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 11327134, 'inat_observation_url': 'https://www.inaturalist.org/observations/11327134', 'inat_photo_id': 16128796, 'license_code': 'cc0', 'observer': 'reuvenm', 'observed_on': '2018-04-22'},
    {'file': 'chipping_sparrow_04.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 220982684, 'inat_observation_url': 'https://www.inaturalist.org/observations/220982684', 'inat_photo_id': 391300648, 'license_code': 'cc0', 'observer': 'carterdorscht', 'observed_on': '2024-06-06'},
    {'file': 'chipping_sparrow_05.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 193252042, 'inat_observation_url': 'https://www.inaturalist.org/observations/193252042', 'inat_photo_id': 339456083, 'license_code': 'cc0', 'observer': 'rawcomposition', 'observed_on': '2016-04-14'},
    {'file': 'chipping_sparrow_06.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 26076708, 'inat_observation_url': 'https://www.inaturalist.org/observations/26076708', 'inat_photo_id': 40457652, 'license_code': 'cc0', 'observer': 'andywilson', 'observed_on': '2019-05-27'},
    {'file': 'chipping_sparrow_07.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 52921135, 'inat_observation_url': 'https://www.inaturalist.org/observations/52921135', 'inat_photo_id': 84151914, 'license_code': 'cc0', 'observer': 'davidfbird', 'observed_on': '2020-07-11'},
    {'file': 'chipping_sparrow_08.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 168861389, 'inat_observation_url': 'https://www.inaturalist.org/observations/168861389', 'inat_photo_id': 292695018, 'license_code': 'cc0', 'observer': 'tim_kirsten', 'observed_on': '2023-06-21'},
    {'file': 'chipping_sparrow_09.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 291074747, 'inat_observation_url': 'https://www.inaturalist.org/observations/291074747', 'inat_photo_id': 523674610, 'license_code': 'cc0', 'observer': 'rwp84', 'observed_on': '2025-06-19'},
    {'file': 'chipping_sparrow_10.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 266314208, 'inat_observation_url': 'https://www.inaturalist.org/observations/266314208', 'inat_photo_id': 478480946, 'license_code': 'cc0', 'observer': 'russnamitz', 'observed_on': '2025-03-17'},
    {'file': 'chipping_sparrow_11.jpg', 'label': 'chipping_sparrow', 'scientific_name': 'Spizella passerina', 'common_name': 'Chipping Sparrow', 'inat_observation_id': 129611350, 'inat_observation_url': 'https://www.inaturalist.org/observations/129611350', 'inat_photo_id': 220257388, 'license_code': 'cc0', 'observer': 'gcart043', 'observed_on': '2022-07-07'},
    {'file': 'white_throated_sparrow_00.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 193338380, 'inat_observation_url': 'https://www.inaturalist.org/observations/193338380', 'inat_photo_id': 339621218, 'license_code': 'cc0', 'observer': 'rawcomposition', 'observed_on': '2023-01-24'},
    {'file': 'white_throated_sparrow_01.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 99992799, 'inat_observation_url': 'https://www.inaturalist.org/observations/99992799', 'inat_photo_id': 166821399, 'license_code': 'cc0', 'observer': 'dziakj1', 'observed_on': '2021-10-29'},
    {'file': 'white_throated_sparrow_02.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 261505977, 'inat_observation_url': 'https://www.inaturalist.org/observations/261505977', 'inat_photo_id': 469820434, 'license_code': 'cc0', 'observer': 'joy4birds', 'observed_on': '2025-02-12'},
    {'file': 'white_throated_sparrow_03.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 150969515, 'inat_observation_url': 'https://www.inaturalist.org/observations/150969515', 'inat_photo_id': 260413022, 'license_code': 'cc0', 'observer': 'andywilson', 'observed_on': '2023-03-12'},
    {'file': 'white_throated_sparrow_04.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 62040646, 'inat_observation_url': 'https://www.inaturalist.org/observations/62040646', 'inat_photo_id': 99351488, 'license_code': 'cc0', 'observer': 'bradenjudson', 'observed_on': '2020-10-08'},
    {'file': 'white_throated_sparrow_05.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 344731686, 'inat_observation_url': 'https://www.inaturalist.org/observations/344731686', 'inat_photo_id': 628148203, 'license_code': 'cc0', 'observer': 'lavenderdame', 'observed_on': '2026-03-19'},
    {'file': 'white_throated_sparrow_06.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 145903421, 'inat_observation_url': 'https://www.inaturalist.org/observations/145903421', 'inat_photo_id': 250718938, 'license_code': 'cc0', 'observer': 'stevestevens', 'observed_on': '2023-01-05'},
    {'file': 'white_throated_sparrow_07.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 65043951, 'inat_observation_url': 'https://www.inaturalist.org/observations/65043951', 'inat_photo_id': 104660609, 'license_code': 'cc0', 'observer': 'allan7', 'observed_on': '2020-06-07'},
    {'file': 'white_throated_sparrow_08.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 10793852, 'inat_observation_url': 'https://www.inaturalist.org/observations/10793852', 'inat_photo_id': 15105971, 'license_code': 'cc0', 'observer': 'schylerbrown', 'observed_on': '2018-04-11'},
    {'file': 'white_throated_sparrow_09.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 142471526, 'inat_observation_url': 'https://www.inaturalist.org/observations/142471526', 'inat_photo_id': 244260317, 'license_code': 'cc0', 'observer': 'deejay', 'observed_on': '2022-11-17'},
    {'file': 'white_throated_sparrow_10.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 102554447, 'inat_observation_url': 'https://www.inaturalist.org/observations/102554447', 'inat_photo_id': 171460784, 'license_code': 'cc0', 'observer': 'w_mark_c', 'observed_on': '2021-12-04'},
    {'file': 'white_throated_sparrow_11.jpg', 'label': 'white_throated_sparrow', 'scientific_name': 'Zonotrichia albicollis', 'common_name': 'White-throated Sparrow', 'inat_observation_id': 115373159, 'inat_observation_url': 'https://www.inaturalist.org/observations/115373159', 'inat_photo_id': 194732675, 'license_code': 'cc0', 'observer': 'wildreturn', 'observed_on': '2022-05-02'},
    {'file': 'dark_eyed_junco_00.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 102901486, 'inat_observation_url': 'https://www.inaturalist.org/observations/102901486', 'inat_photo_id': 172110799, 'license_code': 'cc0', 'observer': 'schylerbrown', 'observed_on': '2021-12-11'},
    {'file': 'dark_eyed_junco_01.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 29901256, 'inat_observation_url': 'https://www.inaturalist.org/observations/29901256', 'inat_photo_id': 46691943, 'license_code': 'cc0', 'observer': 'haida_gwaii', 'observed_on': '2019-06-09'},
    {'file': 'dark_eyed_junco_02.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 386266764, 'inat_observation_url': 'https://www.inaturalist.org/observations/386266764', 'inat_photo_id': 707222551, 'license_code': 'cc0', 'observer': 'ben142', 'observed_on': '2026-07-29'},
    {'file': 'dark_eyed_junco_03.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 196961623, 'inat_observation_url': 'https://www.inaturalist.org/observations/196961623', 'inat_photo_id': 346777340, 'license_code': 'cc0', 'observer': 'zacharyfoster', 'observed_on': '2024-01-14'},
    {'file': 'dark_eyed_junco_04.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 6892999, 'inat_observation_url': 'https://www.inaturalist.org/observations/6892999', 'inat_photo_id': 8793471, 'license_code': 'cc0', 'observer': 'truthseqr', 'observed_on': '2017-07-02'},
    {'file': 'dark_eyed_junco_05.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 114006980, 'inat_observation_url': 'https://www.inaturalist.org/observations/114006980', 'inat_photo_id': 192557376, 'license_code': 'cc0', 'observer': 'k-simpkins', 'observed_on': '2022-04-30'},
    {'file': 'dark_eyed_junco_06.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 141959574, 'inat_observation_url': 'https://www.inaturalist.org/observations/141959574', 'inat_photo_id': 243303909, 'license_code': 'cc0', 'observer': 'andywilson', 'observed_on': '2022-11-13'},
    {'file': 'dark_eyed_junco_07.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 159160633, 'inat_observation_url': 'https://www.inaturalist.org/observations/159160633', 'inat_photo_id': 274980085, 'license_code': 'cc0', 'observer': 'andy71', 'observed_on': '2023-04-30'},
    {'file': 'dark_eyed_junco_08.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 126031618, 'inat_observation_url': 'https://www.inaturalist.org/observations/126031618', 'inat_photo_id': 213798376, 'license_code': 'cc0', 'observer': 'nathanael15', 'observed_on': '2022-07-09'},
    {'file': 'dark_eyed_junco_09.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 39977347, 'inat_observation_url': 'https://www.inaturalist.org/observations/39977347', 'inat_photo_id': 63482066, 'license_code': 'cc0', 'observer': 'chrisleearm', 'observed_on': '2020-02-23'},
    {'file': 'dark_eyed_junco_10.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 255914048, 'inat_observation_url': 'https://www.inaturalist.org/observations/255914048', 'inat_photo_id': 458710472, 'license_code': 'cc0', 'observer': 'joy4birds', 'observed_on': '2024-12-22'},
    {'file': 'dark_eyed_junco_11.jpg', 'label': 'dark_eyed_junco', 'scientific_name': 'Junco hyemalis', 'common_name': 'Dark-eyed Junco', 'inat_observation_id': 14046286, 'inat_observation_url': 'https://www.inaturalist.org/observations/14046286', 'inat_photo_id': 20784016, 'license_code': 'cc0', 'observer': 'gambolingquail', 'observed_on': '2018-05-11'},
)

# Base64-encoded JPEG bytes keyed by file name.
SAMPLE_IMAGES_B64: dict[str, str] = {
    'song_sparrow_00.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAGwAAAgMBAQEAAAAAAAAAAAAAAQIAAwQFBgf/xAA/EAACAQMDAgUBBAgEBAcA'
        'AAABAgMABBEFEiExQQYTUWFxIhQygZEHFUJSobHB0SMkM5NUYpLwFiU0Q3KCg//EABgBAQEBAQEAAAAAAAAAAAAAAAEAAgME/8QA'
        'JREAAgICAgICAgMBAAAAAAAAAAECEQMSITFBUQQTImEUMkKB/9oADAMBAAIRAxEAPwCvNAmhmgTxWiATQJoZoE1EODT1UpqxTmgh'
        'h1o0tHNQjCiKUUwqEYU3alFGgBhRFKKYVCEU9LTCoCUDRxQNDIWpRIoGsgShUqVEQ0hpzSEVEDvQqGgKiDQNGgaCKCaRjUJqtjXY'
        'gk0etIO9EUEWLVi1WtWJQQ1SpRFQkFOtKKYVCEdKI5NDvUFBDiiKUU1JDKaYUgo5qIepilU0woAh6UjdaspG61kBanapUqIhpTTU'
        'pqIU0KJqUEQ0DRoGojExqsmmJ4pK6ixgaYUgNMDUA4NWJVQqxaBLB6URQBphUJBTDpQHWjQRDRWp2qCohgMUc0KgqIcUDQBqE5qI'
        'KmnB4qkE5p1NAFopWqA1DyKGAlGh3qVETtQNGhnmogGhRNA0EShUqUkc0mhQJqdq1YsaiKUGmBqsh1NWoapFOhqIuFEsqqWZgoHU'
        'k4ApVz0rRJp19JbhrdijngL5e4sPx4H86G6Nxjszk3GsLEw+z2v2pSCDnK4PqPUe4zWK9126jgWRbWQH1RlcHHtgH5r22leFbe1m'
        'glu4YpLh+S3lsdoPbk4H40+uaH4VsLaSa+tbeGGTs0ewSc8YyQQR7ZNZ/I6rVeDw9r4ljkjkeYRROqF1UgjeMcY989RXVsdQjn2p'
        'IPKkYZUZyr9OVPfqPes7eDfCGtQy6fpWoXVhqQJlCyqWRhx9JBJz/OuTd6Tq3hqIWl/Na3tpKQYijYIycE9iMYH5UqXsHGL6PVHN'
        'EGuJYa/abEhvXa3mx96QfQ/ww4rsKysoZWDKRkEHINbcWuzjYxNDNAmgayQwNEHFIDTZqIsDUSaqzTA1AMagNKTQzQA2aGaUmoDU'
        'Q+aU1CeKXNBBzUpc1AaiObUoZqZrQhphSUwNQjinWqwatiUvIqDqxxUR1NHW2if7ffyiG1hOST1Y9gB1NUa5+lSx0uVrXSbEOQcE'
        'mMhj+J6V6HTLW1hgbzY4pHT6kV+R2wcdeK5XiGy8NiRtYutHacRAC4ESE59DtPHb471znJnpxxVHHX9IXi3UAjW+n7kP1Au27cPT'
        'givI+NNM1nxMi31peW9xehPKuLW7UBoQD95ATwPX09+te70fxDYXoItLchAfphLCNc+gAHOPWuL4p8QS3WsQaRpnh9L66ncbmYAl'
        'yvRQwHQHrzQrXNi6fFHK0PTHisLSO4c3UtsymVkcgHPQep/j0719M8CeHRcRQ3d3Hp2yNQGuL4b8ADoFPcdz0ryt3p2o2tkk+tfq'
        '7S7VMPLZ275aU8ffLdQD7j0rH4J1K010NGtvItt5vliFTxnuxPXHTAz365pV2ZbVH07xBq36P9LBWaK1vpscrBZqFJ/HOa8BdSQX'
        'mofaNK0aPR7XOSMkNJ/+Y+kfJ5q3U7SLT7lXEUS732qEXlOOnrRzXV7I4NrwAnFDNQmlJwaAGzRBqsGpu5qAtzRzxVQNEmgizdQz'
        'VZbigGqKizOamaQGgTQBZuoZpCaAaohyaIPFV5og0Ec+pQFHrWjQRTLS1BUBYKezlK30bSRS+UpyNoyX+AOcfz+KwapfQ2MI80v5'
        'rkCKNRy5/oB1J7Csttr2ras1paaahtoZHaQzMu3dGvGRn97H4Csykoqztix7Pk+k295bCzVpJ3SV3EapkIB6Lknn5rkatr9loVo8'
        'eoXUbh1+m3lyDnvwTkj3OM+9L4c8L3LaHc6k2pSTaiTugZ3IjjJPGAPatvhXwjocGr3V14mvbXXtYnkDhNpdYz1GQTjisJOR3k4x'
        'PEJ4i0s20cl94ZcwvIGWaMlTtPU8fgBVT+ItAnvrXXNMuZ9NvbcOsMK/XhM8gg9OOfxr0X6VFS71uGwsoLme/wBh2QxDCoDwGIHX'
        'HbtXFh8Ltpms6jdzadBLHa2BxC/1NcPtxkd8ZJ5HpWVF2O0a5PDePvGOs+IL+QfapZLNHwkCrgAAck+vqSa9B+iaK+sLI620by2U'
        'h2XEeCAoJxv+Oa7OjeAYb3UV8QwhI7B7fckSHIJZMOp+DuH5V6TS76RPAsOrW/ljT2m8i4jZMNCgO1mzjHAx1yPzrvD9nnnzwjg+'
        'ItVU+J7O0jjkeGBj5rBuUcggZHcc10wwPIrh6rpl7e/+cWVnI0blYoCVADD90noV4yp6YIHYVq0t5Inlsp3dpYTuIZNpUNyAfXjv'
        'XWa2Wxx4XB0iarY1CaVjXMQhqJNVZo54qAfd3o7qqLcUGagh2agHqotSbqhNIfjrR3571mD0Q9AUaC9BWqnf71A1VkaN1EGqQ1EN'
        'URlphV9pY3VySIoicde1WSadcxvskCK2OhcUpMTJWfVb0adYSXZTfsIAHuTjNdcaVdbggEZY443isGuPaQRNYTvHllKMCpbzH67Q'
        'Byce3PX5olwjUI2+TzsN19rbzQhkeYFCVXOQeqj27cenzXr/AAxpA1O+R5YUt4FUgNjc+B+yCeB8VxfCWkpqN/Gi6fJP9nUeZsl8'
        'u3t89ASAd746gHA6V7/WvEGleE9CaS7KLIfpt7dPvSntgdx74riotu2ep5ElSOhrF5M9umkadLHp0M8exbiQAHP/ACjjJxXNspLX'
        'wxqq+H/C+m3Wo6pcMH1LVLnkRLjJ9sAV5ybXNVFxZPqWi2UviK9IFjBIvmLYxH9tlHfv/PHSvqXgrRr61tpZNW1Ka/kl+uSSWNY1'
        '9OFHIHyfyrtFWzhKVI8Xa3eqC01XXrWwknvLq4+zW0jrgBAQDI3sOMCrdY0220zxNBqj3L3k95EtjtRgF3EjO324OaTXPEF74mkk'
        '8O+FUmWx+0fZrq9CYVfUKR0GM/8AZr02oafDZWlgLSxE91AVWAM+FTHG5j6ADNKSfQN0eMvtP1nQtO0yHTWKW1vqDNdxg4LI3UE+'
        'n9QK4vhPWLUePLzSLeAjTnbzfJfoW4y2M4r3GuLd3FrcRWrLcXMN0JroE4DqV+6PTg8fAr4ZrNxd6P8ApTIsZnkcSbI2bkkHqPzJ'
        'q8l4Polm62N94i8NtcCSyDO9vCmd8GfqwAeoBP8AGuNo16LqUF5RMxh3QygYLRbjwfcHirNU1CG9urnVrJBHc26SRq7f+62MZPr0'
        'Fcnw19rjvvKltY4IPJZlx1DMwZh8ZJx8V35UDi3cj0pPFJu4qE1WxriaGLVA3FVbqhaoCwtSlxikZqrZuOKiHZ6Qvmq2bmkLYoEv'
        'DUQ+az7+KitmojTupg1Zt/NOG6UEalbJps8VnVqfdxQQ5ubjB2Ssh7Fe1ea1HTtflnaWPV5nJPGSBivQA0RUSdHm9P07XYJVlfUb'
        'oupBz5xo6SLnXf0t2mjajG4sXJeYMv8A6gBS2Gb93P7I4OBnNelopbrcuICSvmEKWDbSAT6jpSlyh2Z7/wAR6k+i6alva2KKrHai'
        'blGF6AIg5Yk8AVxfDXhFp9RXXddVLzV5DuVJgGS0XsqjuwGOenpVWi2Ur3MnijWUE99KdtnC4yLWEcL1/bPU1ttn1u/udltepDC8'
        'm5yi/UqAcjJ7k9fQdPWt1fI3XCMiXl7qH6SJpLW3htNPtF8ma4dPruXB6KfTjt2zXf8A0g+J/smgx6Rpe+fUtTzBbLGcEA/fc+gA'
        'NeWtY7u18YXeq6gZbkJ/l7C2gBK5PDSN2Fd1LSzXVBqEdmn6w2bRKTkRp35z3Pp1NZi20alw0dvwRpVv4P8ACA0yOUSyljI8g5Z2'
        'bv7DpR8I3Es/hCE6jciaR2lV5QMggMwGPWqLm+QxFA+yIDDlsdfWq4DDaoYmAhtxkgKcBQeeB2710VHOzVFHZ20b2znfceSu5wCp'
        'lAOAf5/n718H1Dw9eT+MbjUdRC+Wbt3gEZ5wW4zjoMV9n1jUY1HmK6Mip9UmemegHvXzjWr4K880Wzl/8NQP4k/0qhFNlJujkmZd'
        'Oins5JUFw7BkjI+6rNgZ+SDXZsY9ke9jkt0+K8rJG+rXkQY5mM6M745ZVOfyr2OAqgKMADit5JcUYS5sjHrVbHmix4qtjzXE0TPN'
        'DdS5oE1EFjVZaox4qtjwaiITS7qBNKTQI5NFTVO7mnQmgiwdabPNIvWrMZNAjK1WBjSBcUTURYKYUophUZGplyDmkFMKiPRaLf3F'
        'zKkc+ySJFPmM/G1QOAPUk/GK3Jqsb2840xUwUJicdHY9Me3cmvMWMgScbl3I42Ov7wPGK2y3Edlfy2top/y5C7SuAvAwPfpS3bOk'
        'Fw2dyykSztrewhAmljA3SyclieWY+uav86FJgDICjZILc7iTnn4rzkeotLfLdSMI0C4x6+5rLYXMl49w4k8xImYR4OMHv/elOifP'
        'LPRTXEBu/Khj37gH/wCXIPX5H9aw3Gpzbz5p8zc4widhgcc1m0q4H2hxLwUU8deeSawS6hHzcooEMTfUSeuD/fimwaDr92yRTR3B'
        'QTjDLGhzt6YB+B/WvISh3kAWQkFd8hJ6fHvXQ1i6LytgFmdyXOPvH0FLpunSGXzrn6R+56/NdItRMSL9CsRApuGA3MMIPRa6bHih'
        'wBxSsa5t27AVjSMfanNVOcdKCFPWgTikZqRmqELmq2aoTSYzUKRCahNQKSQBkn2q0W07EKEOTjAos2oN9IoPWmQ8mrZooLeRBNIe'
        'QCwHUU63NnE+xIDN9P3weh9azsb+p+QQo8jhEVmY9AB1rfb2FxI5QqIyAc7zt/DmqINVuopRLbRxxMFwGb6ivxWe6kubxi1zcu27'
        'qFwoNVsdYLtnXS2sotz3N6ipGMyAfeHHGB3rLcanYJcL+rbRnAXBkkBIOe+PUZ9e1YljjHOwE9cnk0SaGmO8V0jBDrLEkFUIHIJP'
        'UVoGsRAr5kLhTxvQhgP61kbTdMO3z7UW5JwHhlIXPrx0/KqpvDTozPZajKjH9mVQw+OMEVlORqaxdNUd+3ubefHlSqx9M8/lV4rx'
        'E8Wr2M++6s/NQcLLCu7H9R+IrVZ67cp5aKRyxBWZ87fbPtmtKfs5vBxcXZ7GIFpFVckkgDFatWdTrN24yMykkZ5rz+k66rXiBo1L'
        'I4DbH6c+h713PE8sS6gt5C4mja184gY3uoxkD1I64+fStbI0sMtXZdPeRLbKVtVnKRkLHnGD/WuTYXRtI97wMCwyYo153Y6/lWaD'
        'XdPlQOryAEZwYzVq61ppwPtQGf3lIq2Ry1mvBVcXcs19JbW0bwIUxno3ufk/wqlY5GsRZqWbaegPFdEXlm7BhNCxHcnkVEkgAxG0'
        'ePYim0Dcl2jz15+sNIvI7swpdW2MPj7yf2ru2F5Be24mgbIPUEcj5FXoombyyV2nqT0x71XDZWMd6trawFXxuLo3B9iar9mow3Q5'
        'bApSaoE4MXmMCik8Fuh/GokqSEBWBz0qs56u6LWaqnNao7SRiS7xRxqMszP29hXM16903T4xi9E0h5KrxWd0dV8efb4GZh3NUStL'
        'vXyym0fe3d65UfiSFHzEIhkclgWzXOudQNzcCVpJ5MsC2BgflVcr6NrHBLl2z1AeJWHmyBVJxkc0ZpYY5d0EckwB4B4Brm2t0WX/'
        'AA7N1XGckirXuZEGTAf+qr/o0/8AMTpNqF46bAsMK542DnHoTWeV5ZJWleVy7dcHGfyqiCdpTjYFPf6qcSDO3bk+xqpA/sYdi5zt'
        'BPrVgHNOtvMQpETEEkD8KcwSiMSGJ9hON23jNVoy8cvQqcU4PFRIZGwQjH8KhjcEKVOcdKbDVhzkUAeadYZShcRuV9QOKYWtwXVf'
        'JkDN90FcE1ktWcTSNPvLQ3unanNbfSA0BjnEgPqAQfj86w2HitU82Cd48wfQDjrj3zzWvwXrP2srYTwTT2/l4MoT7oA6E++K9AbB'
        'bTEMkVu0Of8ABKqMbewOa4Y3To92epQtK6M8WtaM9nFJ+sojOy5eNRlV/wDtVNzL4YulW4uGgadSCuY/q/6h/KtpsrJvvWdsfmJf'
        '7Uo0zTT10+0/2V/tXTWXs8ay41/k4kFhoA1uG4s7mUZlXh5OSSRlcHNdXVrsQ2MOolkP6j1p4ZIj+3Ew5DY6c5q8abpy4KWFqpBy'
        'pEQ4PY1QyiRYlGx7e6m82dWUYc84J9euOfarWjrHOnwkcm/0fSU1BdUt9TFrpl9l4yz/AExsDynf8K1y6H4Z+yKz6ncOZWyBlA2P'
        'UAn8cV02tIxJEEihEC53RlBgnsQOgI5q3y4lH0xoPhRU4fsP5SXg809lpNo0f2WWQMVYsN4O0DOO/PFIba2mTYZHCvhjn09OK9OQ'
        'B0AFVs1Ki15Mv5N+Dz8UdvEsaStK2/hU2t0z8fjVp1Cb/wAZQG3hmW3VlhVtjBMAY3H2rp3cXnKvOGU5B/gR+VXKoaKVicLHGzH2'
        'AFapmVmbaVHEgu3mbUdFMbiPzGkgmZT5e4NkAH5NcyPXvEWHEGmsjMerRlgPgHpXpAubcRnIBTb/AAqxBhQOuBjNKigeZ3weNnTx'
        'TfH/ADHngHsWCjHpili8O6g7DzTGo75fNezfFJ3rSSRh5GzzsHhzaBmRAfXGath0y6iuFR5IgjEj6V5+a9GzRlFCRlWA+o7s5rM5'
        'zcRZx1P8qzJ8o6Y22pMoismjXAnfB64AFBrBXzumlOfQ1vPSoBTSOf2SfkwrYqDxLKMe4ox2rQvvjncNW0ioRRSFTl7BbXWoQMSt'
        '1u3cHK1si1S4ULvhhdwcluRk+vzWPbRAFGqOiyzOhHr99CzGC2gUOPq55z61JfEOozWbQSQxCTs6AA9a5xBPQVdBazSNhVHHXJAx'
        'WaR0U8no222u3McMcRg2qFO4KR1PfPrTR63NFczXEVs5kkGAzSfdz1I549azLYzlHcjCJ1btWaJoy7b5Y0jXqc5P5Vml7N7ZfRpA'
        'AHAx8U0pM1lLamQozDMTgZ2sOn4V5SY65BMSlhcMgY42Tg5B/GrP1lqkZiLWl7g8Mpiz+fFUmcsUNHakj0ds0wjVLlQswH1AdD7i'
        'rgfxrzUN75119bSR7RuAdGVt3fk9q22urE747lRFIncjIxjrxVHKn2OT4vP4Pg7ROaxalFv+zsm/6JkyE7jPf2715698WG1lQFbZ'
        '1bvkg1fF4qjkg3hYd2eRvOPzrdqrOH1TTo9MxzVbHmuNa6407f6cKrg/VvOOKvXUkfGTEgP7TPxQ8kV5NL42R+DezetVsayw3tvJ'
        'O0RniDAdj/3x/SpqUi2jp/nYJQQN3k/UVz+HNCypj/Fmu6LyaokZ3m8nYTFjLsRwfQCsy6ipmi+iR4s4lIjIbHtxXShv9Iddr2up'
        'AqQ25U+9/wApyOPnmndPwP0NeUV470cYq261OxSNFs9FnlZwdxlZvo/ufeuabicf6dhL+LcfxNO/6BYV5kjU5pcjBrMJbjduNiGB'
        '6h3GKW1FzDIzRWlsm7s53AVbv0P1QT5kMbkPKUG4nvgcD8axaXO8+rtCuWWNSzYGQM0NYvWt4drACU/TiMcH3ry+h3OqJqkUlmQs'
        'rMcvk4AHUn2oi3TbPRNRpRXR9LFrOYnlEbFU+8cdKpHHDbVOM4ZhXKsZ7u/nnS8uTIEKnMf0qTXQjsrZeRHk+rEk0/m0ed/VF1Q5'
        'lgRSXnjyBkc0j3cC4AWSViOdik4qxY41P0oo+BTijV+WX3RXUTO074JS1YntvOKUPeM4IjiQdwTnP8K0mlHtTqi++RU5vHU5nRfd'
        'VoQRTImDdSnr0wKtFHoKtUDzTfkUITGY2lmZCMYMhwRQW2gUgiJfxGaYGjnJppGHN+y0moDSnrQzzQcSwnI55FYp7VV3vEoJYYIP'
        'OB7VpJqZrMoqXZ0x5ZY3aMNvHbblW4s4JEb7kpiU5Oeh962HTtNPXT7X/ZX+1LLCGIdMBgc4I4z6/NG3mQSTCWbZgltrD7o9B61l'
        'S14keiUFl/LHw/QWsLD/AIK2/wBof2ql9P0//grf/bFbUBkgEyfUuMnHUfNIyt7Vq4nHTL6ZRHDDH/pwxp/8VAqzj2qEYDHK4X73'
        'PT5pVZSu5WBHtSpIHiyemEVKUOpOAy5+aYHrWk0zDjJdogFQ0aDHitAJij2qE0ufSojgeItJvb6dfszKVY8ktjb/AHrTJp8Wn+XB'
        'bIZZNmAQOSOM5rsKc9eKpldW1R4xy0aYPHxXKS5SR7cWR/W5PtFelWv2S32k5djuc+57fhW3djqaqcskZcLkDqewHrXFM19qE7my'
        'hH2YEqJJTjPxW7RwWNtbS4O8DnvTA1zIHnhdYpYSi7eZVbIyPat6v0DYyRkUbK6GWFpbLlFnY0MYqZqE0nIUURyKTPNMDxSJKgoZ'
        'oZoKyzNKTz1pS1TNZOY5PFKDQJ4oA8VAWg0s0UU6bJUDD+VLup0OaiTplDW0qnMN9cxcYwCCD27isk2nXDxvGNUuUDHOVC5rpMaU'
        'ms6o6rNP2c2TTriRWU6rcgsoViFXOBVcekSxRLHDqUy7c4OwE81081M1rVF9077OU2l3gO5NRBIP7cX9jT/522DM7Rlv2dp4auiW'
        'pJRvjZAQCQQDjOPesuC7R1h8iV1N2iy2mE0IkAIJ6r3U9xTsayabam0iZDO8xZtxLAfwxWquiOEqvgVs5oCmqcZpAaMZYCkuAF1Z'
        'x2EIOR81ZH99eM8+lYdblisnnlAwxwigf2rEv7I9mBXikjO876lePZRki2jIMzA9fRRXXRFRAiKFVRgADgVl0m0FpZrHj62+qQ+r'
        'HrW3HFKSR58mRzYhXIwRkVlmDRjAyR1Q56H0rbikZBIQp6Zol7NYJNSr2LbuJYlcDqKLnms+mjYssQJISQgE1fJWk7MTjrJorJ5x'
        'RGcVOp5o44pMg5pqXvRJ5oE//9k='
    ),
    'song_sparrow_01.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAQUBAQEAAAAAAAAAAAAABgIDBAUHAQAI/8QAPxAAAQMDAgQEBAMGBQQC'
        'AwAAAQIDBAAFERIhBjFBURMUImEHcYGRIzKhFUJSscHRM2Lh8PEIJHKiFkMlkrL/xAAbAQACAwEBAQAAAAAAAAAAAAAABAECAwUG'
        'B//EAC4RAAICAQQABAUEAgMAAAAAAAABAhEDBBIhMQUTIkEyUWFxgQahsfAVwSOR4f/aAAwDAQACEQMRAD8APga7mmhmlg1FlTyj'
        'tSDSjSTU2FHq9Xq8Kgg7iugV4UoCpJPAV3FertAUcrtczXM0EHTSTXiaSTQB4iu4pOa6KCTuK7prwpQoIPJTTqUVxNOpNWQHg3Xt'
        'ApYIr2akgQU0kppyvVJA2EUtKKUKWnnRQHkopYR7V5JFLChUgU4FdAqR4Jrhbx0rLaXsjmkkU+Wz2rnhHtRQWMYNKSDTwaPalhna'
        'igsYSKWBTwb9q74ftU0RYxivaa9Nfjwo6pEpxLTSeajVG7xLCXJbEZ9PgYyta2icDuMGqSnGPZpDHOfSLvSa4U1QzOLrdHYUpbgC'
        '9ijCDpWnuCop+1QXOPLchxKQ1JUc7pU0lII9jr51V5YfMutNkfsFSgaQQaEF8fQUysOfgoKwAlxISSn29WD9/pVrB4ljPWw3FXhv'
        'Rkr0qcjqyUf+aTun9RQskWD080XQBpaRUa1XK33NBXAlNvaQCpIO6fmOlTQmtDF8PkSBShXcV6pIPA4pWukVzNSA6F13XTOa9miw'
        'HtdeC6Z1VzUaLAkhddDgFRdZr2o1NgSw7XfFqKFGu5NFkUXSovtTZje1Xy2U0yWB2q+0gpfLe1dEX2q5DA7UoMDtRtJspRFPau+W'
        '9quSwO1cLI7UbSCoEX2rximrfwfaveD7UbQMa+LcebAjpeROL8ZSjqjnn/rWaWx11D7aowWFuqw0pZPpHXOOXSt54+sTGlcwFkpG'
        '5ZUrTqPTes0YhLjl16cFNpUc6EOYT9uWeVIZYNzZ0sM0oIqk2uXMedjtvpdkLIUrKPEbA5H1ch8udNXTgiUI6pUdpLzm/wCGtIQM'
        '53325ff2oii3K0RGUpiIYbWPUAoEgbb57mq2+3/Qt9128tMjQAfLrOQScAHOBn9MVVQSLPI2Bkqw3+KB4dtZVnUElp8KI752yOVQ'
        'FvcTQG/NGHIaSNJU8gED29SdiDirpzj6bElFsuCe2FKJLqRqA5nCkkg7Z2qyicerehurYUXQXCkIICk6D+Unr7H6bb1fy66M/Nfu'
        'R+GeJg/NYlxJHlbkjPiMobx4x2GcDbffNbfYrgm6W9qUGy2paApSD0zXz1cLjCnTI8yLHjR5cYaysYSnWTsSP3iO3yo2sHGsiArK'
        'ZaXmdO6XR6c+3UVrhjXbMsz3dI10g1zBqj4X4ysl6WlgvIjSVbBClZSo+x/vRcIZ7VtQs+Ct0GvaDVoIh7V4xD2ooiyrKDXCg1aG'
        'L7VzyvtRtCyr8M17wzVoIvtShF9qNoFSGj2pQaParXyvtShF9qKJKtLPtSvCParURfau+V9qmiC4U6O9ILgqv8wTXPFJ61ayCyS4'
        'KWFiq5Dhp5ClGpsCZqFdGKYRmnkA1ICwBSgmvJFLGBQAPcVQ1GI7JU2l5KR+XTg4+ea+bPiZc5LN0EJplMLJKQMkY75r6S4+uzNs'
        'tKS68GtStRUc7BO+cDc74r5Z43kCdxE5dFuJeYUpPhNr5pBUQAR3wCce+9IZ5qM6Q/p4OULZXGDcZTaQlbjoP5woHT8xU228Ossx'
        '3he48tUF1GC9GOS1nkSOvy/lRjBlwHnTEguo8VIAcydJAxkgDkcDbeo/Ej8SGwpYdejgDoMgK6nHI9ayjybSMtVwlDt88vxrmzPS'
        'FBTKWWl6lHoFAjCflVpGtj1rjKD+jxX/AEIaSsZJKs8uwOP/ANTU6bxVHDq0skPO6VJBTHSHPzfxDlkZOedWBabiQTd78pCZSUZZ'
        'jkYCNttyClWeeOdbi7VFExGkNQjDl2+QXXCXMoQCQeWcdqJODOCb/f3EsQI7cdokeI9KVgAdfSMn6VM4IuJktvy5RbSpxz0HGpQH'
        'IDPblW3fCtgJalPqRgKdASTyIA3I9s5rSONSfJlLI0uAe4d+CVpjKD95uUia7nJbjDy7fy29RH1FahBtrEOI1FjN+Gy0gIQnJOAO'
        'mTvU9JTSgRW8YKPRg5N9kfyw7VwxxjlUvavYFWIICo/tSPL+1WJSK5pFAEAR/alCOO1TwgUoIHaooCv8Adq6GPap5QO1JwKKJshh'
        'j2rvgjtUo4FIUoCpoCgQ33NPIQKQCBXQ4BVAH0JFPoAFQw6B1pXjjvU2QTwoClBwCqwycdaQZfvRZJb+MB1pKnwOtU6pnvUaXcQy'
        'ypwnly9z0qHNJWyYxbdIEPis9Jush63w33kYZDSg2RhW4Uc9e/2oEXBtaYbpMZS5LBSlrWPQltAGxP8AFuTR9MisCe7LdIUpI1u6'
        'Tvvvz6b9OtBV8gSZsxfkSMLRlsNnKNPbOeZ61y5tznuOtBKMNoKXGcl++tXCOwiF4YLbjYWEqV3wO5waHOOLvMnJ8sw2lqOlQAGD'
        'q1EcqN5jCg889MhKJZwkOtueGokfu567Z255plEvhV2ZFemgsuNbBLqSAnt7Z2qyKMZ+G3DEOHa27hNjJU+gEua+mDkYzy6UFcd3'
        'OLduK1KckuvsY8NtJ9I1Hbt+UHvWq3ydbnLC8mHcI7YdaUEnUMDVn1fzrBpKXo3ELD6mRIQ2oLCTyUlJzW8JC84ujfOFbFFhwo0J'
        'tolyI14klakakn05wk+5Gfata4VKIFhgxk4ylhOT3JGT+tZT8I7u5Ntcp+5oeKVSG1Ic1aQdagFJz2xjblWjlwxiGQcoSAEkHmBt'
        '/SmFKuReaChEz3p5Msd6FUTcdadTO96lTM6ClMsd6WmVnrQwib708ib71beFBKJANLS8O9DqZ3vTiJ3+ap3BQRpdB60rxU45ih9M'
        '/wDzV39oe9TuILxTqe9NLfHeqVU/3ppc0nrRuJouVyAOtR3JQHWqdyYe9RnJRPWquQUSVP4602p896ilRNcyapZJJMg0kyFd6Yrw'
        'osBwvLPU0guq7mvYrmmoYCS4ruaoOLLo7HREajqStTstLKmgfUr06j+hFXM9wstICCnxXXEtNBXIrUcD+/0rPeNT+yL5BBeLghMu'
        'S1KVjLri1EHGOW+NqXzy42jemhb3EbiK7TEInW1KPVqLjy0ZOke59qhI4h1W5tqAsMFQ0N6QV6cDc6u+2KGbrOnT7g/BjlLki4Oh'
        'TxbJVpzySPlvUjjiHIslmiQGZMWMlKCTof1LeO2fSOW/8qWhF2OTkifBuEi43F2C8HcpPrAGyz8idumah3fyDtzXFvTTDcYJBbUh'
        'WlKyD07/APNP/D5qM7C9MZS3f33n1YGojAx17ihrjW1PweJEOSVxYbe62mwouYA5KOOprSPdGUurJ1w4ftbsnykVT7bbnqRpThBw'
        'Oe+/M0KSLWxGcXGdkLXoJxtg4H96IrJc/OPeIpuSFq9CXHXM6/8AxT0/4pUixNPP+YUpSno+QUKOdYznerUyvBCiXRyHBj/s2Y48'
        '4QpLkZ0fhlIO2QNtx+u9a78Org285OtaX3n/AAgiSlTmSpOvGtCieZCiCD1BrJuE37bHuExm5MNsRnmltJ9WQleNyMjb/iin4Z8T'
        'tQuIYDTnjyI63129UjVlsML/AMME43KVYUD2raPPuYzXFUbAEmlAGpnhdCN+tJLVX2ixGBUK7rUKeLdNqRUUAnxlDqa6JKh1NNqT'
        'TSs1FsCX5tXc17ziu9QsGvYo3MCd5s9695knrUIClVO5gSi8T1rniE0wDXc1NgWVcrxr1XIPV0Vyu1AHa7kUivZoAqbnJZF9YU84'
        'lLNsZVOc9z+VI/Umsj4lkS7o3MvchRDK3y20RzxqJx9P6ijq7huTwrfLopWH5hIQSeTaFYwPoCaC3pMU/DhSH2kpcRJPgDkVbbk/'
        '76Uhklcr/B1MUNsEhizt2u2qTOdfS0AjW6CrKgcHAHuakswrXxI49ck28MR2kaC444dwP3j2NVDViXcbYyUrS3pwpZVyJIyfqBtX'
        'OJ74hixR+HrWhXhI3fWNi6rtt0qq5fDLy4XKCTiCZbbJwy9JiKSFPq8NjBHqJ6n2x/OsqlTFzvERIcSfTs6tR2A/1og4ctP7QQ63'
        'cXUoS34i0oWrHqxg4+36UJ3Bv/8AJqZSFhpDmFDsM9a2ikjCTbRd8OplMIj3JxSEnT4EYEZPMkqq8ubrkWUhxClDKQV9SVAbn51Q'
        'XW4ogXO3RU4UxFT4ih3JHL/ferSStU0ISs4LqCoqHMdc/Sod9kquiC/Bjzoz0lK0oQ44HSFH0pPXB7Gih5US38IMssvlCUTESdB2'
        'OCNKh9ht8qH7VZ5CWltS3W20a/WQdzn94DvnpUriK3FFhgIZeU8A6W3XMd8eo9u1Xhw6KS6s+ki425lxCgUnkc8/f68/rTLshhuS'
        '3HWsBx0EoHfHOo8HwlRDHGHG20oa1fxgNpGaqL1bLghCXbc94waOtDLivUk/5FdPkdqaEn2Eakg00pNByOMVy4ku3JSIN5baJbTI'
        'GlJIqx4DnPyuHo6589MqWvJUrGM78h3oCmXa0U0pupRGaTiq0RZF8P2rwbpyU/HipSp9xLYWoISVHmTTjK2nka2lpWk9UnNRSuia'
        'dWMeH7V3w/apOn2rxTRtCyKUUkpNSikUgpooCUM10ClaaalSI8VGuQ6ltJOBnrROcccXKTpIIxcnSVjmK7im4khiUyHo7gW3uM8u'
        'VOZBGQQQe1EMsJpOLTvn8EyhKPaOEVDu8hMO1S5SsfhMLWAepA2H3xUwmqbiBozYFzj5T4aYwQonoVEE/oB96rmnsjZrp8e+YCca'
        'voZ4JsECKtYDmoOqx+YndXz3J+1C/HUZcIw4jatUdLecjkVE0YcYwA9Ym5DakKjNZVHQFbhAQkH/ANgaFr4WpYtKZDxBUkrKccu3'
        '2pFO+TpNUqPcSXB2Jwqy0w34bjxIGn2xk/yqi4MYTokzpQHpADOv+Mnn9ADRJNSh0ojPqACm8AKVjSrOCfb94fI0O8VykQ7rFZja'
        'Uxw1jQnkVEEb/eoiq4XuVbT5fsQ7VMQ7xA7oSNOSQD8+dQUw2pF1lLQ5hOvOOprtoiOMuqkvNKwQRgHel2iO+5eHykHSSc46Ctmq'
        'tmae6kyLfGl3W8AtoCinCVaRy/4okiseCDqwFJaAz22pu0Ro6HpDjSslLnrJ713zbbjYc1D8Z4NpHcj/AIqspN8IvCKXLH1kpCs4'
        '2G/0H/FSJmXeFXYSHcPPPNtpGOeV4x+oNVylKkzENN7jIUrsE89/rgVJtzni8QxCEhKGUqeQkDPiuDZv7q07dhV8apmeWVo3Dhta'
        'l2lpwgAKKggD+AHSn/1SKss1Ht8cRIMeKDkMtJRnvgYp+mznNgd8VLBFunD0iaEFEyMgrbcRsojqk9war1X6zr4MtxU8YT6EoCEg'
        'YUkp5/Q0RcfW5y48NyUsynYzjSC4lSDscDkR1FfP3FcK+WnwV3E+L4qA41pOQUnqKyyZHB9DODEprv8ABovEnxNk+C5GghDK9tK+'
        'aj/QVF4E4/vkriSNBnuhxl9zQUqG4+tZQt6a4Gnw0BkE4J/pVpwrcVRr1BlSTgNvpUpXYZrGWdXwxmOndO17B/8AGe+TJtyctMV/'
        'SzEUFek4JVjff2q3+ENzfhcJpXIkNaHZHhtoJy4VkgEn2rMuK5QunGFxWwVNxTIUokc8UW/DhmZcuKILMaMgQoavEUFD0p9z3NXj'
        'JOW4xnFqG03akmlVw01QkJNJIpRpJoAmHAoN4zeV+2WWSrCfBykZ9zmrniDiG32dKfNLUt1W6WWhqWR3x0HuazniK+Iu3FDT0RSv'
        'CSkAA88DuOnM15z9R6iEtJPDGXq44Ol4biksqm1wGtrJZhpb1k6/3M7E1dQkpaZOAAVHJ+dC/D3mJEnOMDGx7e9FaQEpCRsBXlv0'
        'l4bmnq3q5v0xTS+rft9l/J0PFc8Y4/KXbFLcCEqWo4SkEk9hQpc5nmeHIsNpZTKus1wuDPqGnH2A1D7GidwIUAlwAoUQlQPUEgY/'
        'Wsw+IFzk2+92oQ0ZLcVZb0jBJU85k/PYfavf53ukonN0q2wcvwW85UV26O2xYW3FQwGgOyB+UAfP1E9aBrwlMy9Nvxzpjx06QcbY'
        'TnP+/ejabH8eS7dH9SVKbbGM4BJSCR+tAVwlpU+9EjJKNKlE7YyB/c4PyA7UvCI1JjKJMmZIeKlJOTkkjIB+X2pdut3/AMgv0SOv'
        'TqW4DgDkMn/Wq6EXIljfd2Wtxwaz/CDnH3P8qsvh8/IYlTLqgYc8Py8PPVatiof+Kcn5kVs1x9jCPxfcl/EdaLcyIkQp0JXoXpHb'
        'v71BZKGuETNbb9Sv8ZwjcE8vkMVZ31uBGj+Rmp8V9WHFBSt+uCffIquipUqK+6VluOoYUjOyxjGMVWL9NGkk1KwZtcx9VuuzjIUv'
        'TpWQOeM7/pSrGlRjW7xyVhC1vqSOeo5CQe3PNSXWI8YqbjoCEL3VTkFlTrngtIUSElTmkZwBzrbswfHBKcVLWtpEZBBkrDSAkYCR'
        '+8rtyO3+lFvA0Ay+I2brcm0QYsVI8NsjGdJ0ozn3BP0FWzEFMWyQVsw2vOOrDaFK2KATzyeuM9OtC1/h8S31yLEgMPBTzi31kbIS'
        'CcJGeuE/zNWi0nZRpyVG3svsPDLLqHB/lVmnBWOwuC+O7LHEqHcWX3eamdZB+h5Vf8H8Z3ORJch3mJ5Ty4/GW/6NI7561vuQrsvo'
        'MeKXPC4fmqGkEsqA1ctxWOcdxuKF8IQXrnb4jkJgJU1IZJ8RpJGAFDtyrTmOJLJxA+qFFWiUhB9frAB+XeoXFGbxbZ9kjvpZYEcq'
        'WrqEjoBVG7ZrGO1GKu2gHhFm/sOq8RMox3U9MYyk/wA6Hbg94QABGrnttvR3w+ytfwgvGv1eHNSU79RQHMfgMJS8+krd6IPKlMmK'
        'PmWkN4ssvLpsm8POEpcU4SQ6oZUfatQ4Hut4gWpUTh6yOS33VkuSVJ9OOgrNOEvDuV0jLeSEMKdSnT0519TRUtMMIaZQlDaUgJCR'
        'gYq+CFycr6M9Rk2xSrsGLFbuNHZSZd3vTcdvUCY7SArI7E9KMCqm9Vc1U4lQlKTkOE1ykZroNSVMQcmLckOFxTj77pytR9SlGrvh'
        '+2oay662lvVvpHP60RzLlw74i/2TaIpWtXrcEcISB9Rk/TFedu8ONH8xLdjtNpPItpSCe2AK+U+KRhhl5ccm5vuuT1uCTmtzjX3F'
        '2l+5trcRGLSW+YAayQPnV/GemaEqdWk55ZSBn6UHJ4lYmgojBK2VHdDekfLIyCaIrGuPMR5tCfGKFFBUoYUlQ5jB5UlqNX4hosSU'
        'ZTjD29kSsWDLK2k2WU2W2ExwRjEyMFnoAXRWdcVPKVfLlFcOpUaY601nfQM7ge2cnHej3i1wJ4RlSUJDS2Sl5Ccfm0KBGf0rOHES'
        'Lv8AEt1plshq4yhKQs8vDV6iftz969/4S88tNCWok3Krdqnzz+xzsyhF1jVK/wDwLuMRpt7LXqbYbUlSsDdQPID6UFX+GUzHnAwN'
        'DyNIwd9WkDn960a4NRHH3pNxa8OMV+HGGd1b7EdyTyFD17ttxkWUPoiCK4CSQsjUdtj8v7CnYS5K5E6M+W0wwgwitKw4Brzyzz/S'
        'rjgER3EXK7yBhiI2otpTslCUjbA/3vQfdnHYkh/zXpeTkaeoyNqsGJrkTghm2Jyl6e/4rgHMNjOkH5k5+gpmUbVC8ZbXZ0CRfLu7'
        'LU2QHFZ9kpA2H2qVcHENtFhpIKEEJI7qq1YhmBaW2Egh9xOpSuwI/wBaG7o9lWlvJJO2OZJO6sfoKonvdLpGjWyNvtlepS1OEDKl'
        'E+nrk9/p/OjfhS2/s1CfGQFSJASjHPmQT9MA/wA6qrVbYkCKZ1yUEqTjY8k9dI7nH86jx+JZcuWpUNoLkPOKbjjngbZUewHL6Gt/'
        'oLfUKb3dpF24ij2Cz6gGyDJfT/8AVkEfcAn61pENiVBjtxozUcMtjShOSCBQz8PLC1bmS7kuOfmddPNbh5/b+1GJVWsOBfJK3RXX'
        'K9i1tpeuTBaYKtJdQrUlJPegHh6bD4649nuvhD1sgtlDLR/K4TzUR1qh/wComdcY02MwxKdbjPs4cbCtlYNRfhLIXYZ1sE1SIzcl'
        'tSiVHGpKuR+4ocmTGHptB7c/hlY1r8zaFv2yUk6kLZWcA/LtQdxhcL5wu74ssoXIWytpSkn0upIxn+ta4LpAJA8236uW/Os3+L8J'
        'XEL7LEF9gBlBC1qVt74+VUlKEGrdWTFTkurAb9qKY+F0W1sEF+4SVurSOekHA/Wgm6wH28oW0rX+8VbEUfWWzwbZFeRJmsuScJRH'
        'I5Np5qO/U1TfEOfNmzWJUmOynS2G/FaGy8cifelss5qfQ7hxx8t8lJww44ghCCUqQrPyNfTfw9dmvcKxHZ61LdUCQVc9Odq+cOE5'
        'lsFyj/tZp5lLZHilvfxE/wB6+g7RxrY3eHzcWkuMxGSG8achPROe2e1a4eJO2L6hboqkFmaRJeQxGceWoJShJJJOBWWXjjyfImKc'
        't8tDEcfkQE5J+ZqhunEF+vURyI5McKc5KRsDTDlSF442w8tvHMpWpcuGw6gpGnwjoIPXnnNG0CUxMgNTWVZZcRrBPQe9YoxCl2+2'
        'wfNJKVSY/jD3ypQz9gKu7LfJUfh+dakqyh8YbOd28n1Y+YzXl9P4pk0eaeLVStLp/wB+Z0sumjmipY1RHtPD1zk29yZcLw3FBc0J'
        'aKsnnzynb9aq75b41qStyTdI7qAM6VOEkntg5qrVd7w8wY1uab9Q3ced0p+QA3P1FU8rhTim4finy0kJOfDae3/9gM1zIQxtpzah'
        '+V/HsNObqkmwp4Tg2KTMiyHGA1IDgcR4a8oVvnSUnZX0xWv8JPpHmoWgJDatbeBkBCumo7nBBG+45Vgtht70cLhOomftAHPlkp3S'
        'PdJG9ad8PeI3GD5O4OFWSEKUtOHEnkM9/rXaxa3TxyQu0rrrh8fP/Ynkw5JQfuFHxJkJZ4UkjGSttafp6f6igjgie3DmImcg0xkq'
        'Cc/u8vqcUa/EGC/OtCG2zlGpSVo76kkbe/X6Vn9yiJsMGPbmyqQ+4A46TtlAyEj2zg11ci9UkRha2RDazKlXGN5q4LQ5hRWku74P'
        'Qpz+XHSh/wCIU9DsGOYj6vILOlxxCttIHbnsds9aq1SLjPkJgvTBHjR0DzbgwlKSSPTn9PvU9yZEYg+YQ2nyrP4ccYz4yiPzn27U'
        's1KD5GrjJcAVxciOq3R7nqaedThKnNGxH7pIPWoNpZS9e7e0pZe1rSVEn8xJ3P2/lTHHrL0aDGj+LvJWXPDHQDmfucU1weVty2lO'
        'El0fhtAckk7aj8gfvT0U9liLa30H3E0hCY4x6VSlFeB0b5JA+mPvVBaGmlXbQrQXQNRydm+2ex/p86h8b3tqJIkyGwHHQA1EZ6BK'
        'RjUfbOTVFw0qbfG0xbctLciSsJccWrCiSdzVccdqJyS3SDCU8m9Oy2YrjaYsZYa8dSsDWdlaR1OM4+dW3CVgYfnhi0NqKMJbXIUn'
        '0NtA7kHrk5A7nJ5Cp9s+H0S1WrwpssqcByGANj/EVHueW3IbcyatrVcrqwuQ26hhmPr/AAy0oZwBgD8vIAYAq7texj30wzjMNRY6'
        'I7CNLbYwkf7617UkrUgKBUnBI7ZoaevdyiW3zCW2n3XFhDDLjmp11R6JASBy3ydhTlokQ4z8i43Zflri9gOBbwKQByCUjsPapWW3'
        'VGbxNK7Av/qHgKessOYlOfCcKVH2I/0oStFme4g4YYu/ioY8BbcUtpQeXIEe/etR43XF4q4ekWu0EzJeQpCEpONjzzypvgTgGbbe'
        'FXbfLlaXnXQ4QgAhG3LNWtXwy0U6pg/abe9b3VMCe3LioQrQoakqz3Htz60w8pC8xlBO26VZxipUpl+3PvQ3AkuR1FvUnmrufrmq'
        'K5TEqSSU6VDrXgdfqp6jWuVfC6X4O5gxrHir5lHcYCo81b7DrKWyrKy7uB8v7U7wtaL1xEgnyCPBUpSSPDITgdST6f1qxtsplS2l'
        'uMIe0OhRSoZCh1BFH12fs/k2X2TcZRWhK0IakBISk9AMEZHbFej0uqzZcFwpzXzdf6YnkjGM+egIifDiGw9419kx2EFQAjxV+I44'
        'c/lyPSP1rVR8MJfFUKFw9E8vZLBHdC1pbIW6vHy21HuTQxBbtMshwMSo60LwnzDqlKPvsnAq4f48nxpC7bGc8tEjehpCNjt1J7mk'
        'tVq9VpFv1KTvpR+n3/v0NMcMeXiD67sur3wzwPwzKREjw4gYaRpUqQ0HFqIG5JPPegO9McNS2XvK25ppDnLwipGR32O1S+KruxLj'
        'tyn/AMVSxz2J+tANxvCvF/BwkJGAM9K5uPU6iab3tW7pG7hjTSo0Cw8MvXXhEtty3Hno0gohtrAy2kgZSpR5g9McsU5B+G/FPiF0'
        'R46G+pW8BVXwp8SJjcqLHaDaYzTKWlBLY1KUOas9zWnXu5ruNphJhy15xqdCjgEnlXT1WPS5canP1TquOL/kWg8ilS4R82WuQUEY'
        'QpX8qI4l6TbmPMSHUtNZxyzk9qooiADkJq3jKRoLbiQptWy0KAKSPcVxdQoTdSXBvFuPRRXG9IunELctiMtxKRpSlQwV8+3IUa8D'
        'cPcRzboxNkMluOXUurccOxAPTvyxSuCrbaWr6mTbWo4lNpV+C8vCN0nffOB796P0y7pcrxETCeiQIzKgpaVvoUt7bdOlJIxz+1dr'
        'QYtPlik3Si+F7v8APyF8zyLpdhDcpLceGouDVr9IRn83tWb8Tys3HSy2lcoqABA3U6fyj5J5/aie+3Hx3H5y9IbaTpZR0+f1oSgS'
        'UW6JN4onALTHJaiIV/8Aa+eo+R/QV3Xk3zcl7dfczhj8vGosiSYrUiWzwhGBUUkP3N449iEZ9/5VAvdzZYDkxbSBhSWkJTslxaR0'
        '9hVhw8hyJw3JvExZ89d31fidQnqfYAbUM3eA3c2lym5GyUFLKdtgDyPYnn9aiMbbfsiZPil2wF4imvyL64+t0KXsCQcgbZx+vTb+'
        'dElqU9Ct/mACtZTqShCcFR/mBSbBwxGl3tllcksOH8yHU5JPXflmtXuFmtFs4WlNszI5lOODSVDKykc05pjenFyMFiluSaMltFtc'
        'nzUv3DSlCzqccWcJH1Pb+lT7W3FsF+t93tEZciDEdQp9xYyhatXP27b/ADpi8oChoK9TaTtntVlZbi5GtTsNNlVJiyW1JdPieog8'
        'tPp7gH6UlHWRnLbdUOPRPHG3y2aa9eoMp5yU4ytyO4g6G9ZSQT7jt3oZSw+u/MshxcqN4oytP8Odwexobsd2cFscju6vGY9Pq2On'
        'POjrgWRESFeZZQ4gkKWlXMjrg9KVxazL/kPJl8LXH8mOXDFYdy7QbyOGbFfFq/7Rl1DSR4S1n1JB6ahjf5UKXL4a2mO846bstpCj'
        '6W8BRT7ZPOtdstmiSbamXbmW0wm0KK1Kd0hGBkjPes24lXLfvKvKxHikEaCuurn0+HLxONi2Oc4fCystvwvm+Ybm2TiB6I8ndKww'
        'P6EUV2bgTiloOP8AEnHL7MdZJQlhCQpQ/wDJQ9PyAPzq84YiyLJw54lwfDbrvqSMflz13oB4/e4qRqmPiW9EAyh1GSkJ6bdK5ery'
        '49FCsMW/omxzCp5nc3+wr4icKWu2WsXmz3oyltq/7puVKCnF9iDtv7day6e+1Jd8JaAlRSFdM4PI0OcXcUr1Kj+Mpa+eAc6ffHU0'
        'L8OG6vXR12EhSypJKws/u/3rm4/DJatSz7dsvb6ms9VHE1C7QSzZarfckNNu58X0pOeWaNPhleLfMgLgylanYilFBJ3CO/8AWged'
        'ZJlwjKUUlJbQVtpwQpRHbGd6orBfnOHL4xPSk+Mwr1tqGNQ6g0/o8EsbXzXZjmmpL6M+kn7zbIDcNpyP5hC8630gJWkg7H3+vaqd'
        '5i2Xe4uS5vmNSjkJj6U57BRPKo9uv1m4js/no0Zfl3iAUggKSvYEDO2Rmq2K4Y0+XHZd1padKUqznUByNI/qXzKhOL4V/uaeHqNN'
        'NBBcuHbVNaCEochgDYtuFRPz1c6yfj3hy8WkqlxXFS4X7xaHrT8x2+VaI9dltRxqJz1FUrt7WH1pOFNnpXmfDs+qw5N79S+T/vA9'
        'nhCUa6ALg2a6/dI0OOkl11QSnWcBJ7k1t6rHIXawxI4ndiqIyUss6k5+ec1lE2JFiTBcoWlkpyrSnOST0qzd4nmyGk6VFJxjnzrt'
        '6zJPJtlg9xTClHiYw3oB/DO42UD0qwjvsllTUltLkdYKXE4GfoaDPiZMeTxy/JgyHGm3kIUlSDsQAB6u/apXESL3BsUUpKg/ISA+'
        'SnZvVnAB5Z2/Wnp+D5IzUsc7RgtVFr1Ij2mdHTxE41ElDyiHNPiadRUjG4raOAYjTcR68M6yJLWmKlQ3S2dir6kED2Se9Yz8L+EX'
        'rreVGYgNQI34kl0KwS2OYA7k4H1rfuF0Kkw5cwsoYaLyUoQNkttJThKQOgG/612MmOCSVJv50U0+5+p9EDiBTrjTUBpKi44sBIB5'
        'np+u/wBKDeLZH7Sv7Nghk+RtA8BIHJx87LWR88j6Ud8VzYdiC7q6FeZZy0wzjdTpHpO/QZyaCOBLbpkmXOJCU5kPLV2G+fqTV8Pp'
        'jZfM9zovuN4uiLFssR5tqQzCQ0yDyK1+tefbGlOfc1irVwudtuqojwWVpWEqQU5IA7Uc3ifJvHEkq4y3nFtO5S22yspUlA2Gk4PX'
        'GBud80K8YWu5zGG5UNDkoRlkOK0gvIPZRTzGevQ4ojGmmvszPzO0+i3elrdLctsFD7OCTjc45GrCdcVuxm3icIXnYdFDmP1+xoR4'
        'Pj8QXOa3HkvpYSslCDLJCsAZJ5ZIA6nbtWg26xxYNkfjy3I856Q4glSVgIYATkDJO5JPTOQOtK6jG6cbHIaiKpgXMlJU7lxY0nkn'
        'POtitfE9otFsgWy0uOIjttjxnCd3HD+Y/fOBWQXPhlhy5vOw7i9DRGbU6tuWMKJG4Kc49J996pF3O7x3g34SJYWQElBycnlkDesF'
        'pM+JLy65JyamGbv2Nt+K6bLbZlq4w4auanmpqTCubZTpUFaQcnoQR/8AzVBBvFvtzzgLpkocOkpaUG1tEZwdJ5g7e2KFeNLM/aoE'
        'G4OXR2aqWk+A0pQw0EkZ2GwJPTpuN6pmlTZrkYRIjz0kENrSlOCAfy5yeXTPyp3JgnxkXxIWhLG3tl0zZfhtxDcoV7aaiTXXEnU6'
        'tpaPESMD82k7c8b0YXfjCwwnn5Uy4HzxGpthSCVAn9OdYlGY4nsc1mQ1G8N5zUhBbfRqUAd8Z6ZFEDlyXdRGZm2mdcbqV+GWHEgI'
        'SCDgggA9N8npVcGTNjwuL+L5s11GLFly7oVXyRqXGfE8W5cCRriqY0hSNQWgKGVnG2BWHcX/ABTv0GzuWSHc3D5kYLWoqLaf6UcL'
        '+DqF26K/xHxN5RRbJMWESpe++55A9KYifD34axNJkW2W66Ff4ypi8n3IBpTFpIZMqy5pfhf7M55ZRjtgjI/h7arRdGS5cbs2w426'
        'S6yY4U4Rtg61HbPsDWjR5VjtrKo9sjAhatS1rXkqPcn+2KvU8AfDdpTbkaMhohwpKhJWlQB/zat/qKl8McLcIcP3hcy4zHX8bMMS'
        'EghCue5Gx6YPKmNb5yTlDJUTHCotpOPIi0x5pjuMNMMxxIToI04Ksj70HXTgGI5N8KStEh5KvUtacgfKtXvXxBgW1gpiW5rxgTha'
        'wMj6UP234iWyS8yu4Rw46F+tKWwRjPtS+DxfSOkraXv7Gk9LlXPBmt7scOFIbt0Zxb61KCUNNo9JUewFEt04dk8KSoDL7qXBJjBS'
        '0g5LTg5pP6Uc8Q8XW9UpEi1NBpKEkhSmkp+3asz4o4ikTytLTo8yMlBUM4Pc1zPEvEoav/gxRtfP2GNPp5Y/XJkLiG+woKyh5wFZ'
        '5pHMCge4cRgyFrYJCTyz0qulW29zLoqP5d6XKcJV+H6s++apLk3KhSnIktlTDzZwtK+Yro6HwjFCCd3/AALZ9VK66LpXE8kEpc0r'
        'TU+Feoz7eASlXUZqgtNsYmoUouKUoHGOQo34Z+HplArkSUtAAnSCP5mt9Ti0mKPq4MsUssnxyXkhLzUpiLcoa47MlS2HFqQDlHUb'
        'Zx/pV9NMxmGmE1Ejux07K0nUlaCB/FyIwM/Snb5a1MXNqE0GVrcX+O+7nSgLJ5qO/I5I29qMhwdZv2C8l6Y5JjFPiqlqwglIBKUo'
        'HMqOD9845U3ur6kJWVVrjWmz2mc6suR2nVodlIwFeG2nISlJHQnJ+tDvFvGAmWd1mLbpkS3Z0oUpQQHwAcHvpzvgDJ9qubWl5LMu'
        '3BptsOt6EthatbeMErWFDkQeefpvUK+wuHIlxZYfW/PWh8x1jVt4mM6e5AGOXU1VVu5N03s4M9s92lrewq8tSC0ArVMaKkDJ5IHz'
        'NX7HEdyvS3mZ5aaisoOG2WtBeIGAcDfoTjoTRlcbK9Dt7TU6PZoDqWEPoaeaytpsqzjYeo7AEbn9aGLU1xM2ZbQafQw8S9qKU+lv'
        'PqXk4SCRvgewrVcsxcnQJ2Ti9yPZHmrTbiuU7qJK16iwobgoHPPv7dKi2hyVJtrJlPSoSC+UrSYpUHCeQ5jJ2NGdutioYk3C3xUv'
        'OvqLslQaGSc4AAPLbJPvRLaE2axsu3Wewww6hxOglAWlKic/lxucZ+RqspxTdIiMW1ych8P2i12GJd3mpMuTIhr1sEaEIZCttKQM'
        '6jtk9ciqpV/jzLY0HIiQ1JdQ08hxSkEJQD0JwOe+3QUSz4TM2AWl3F55QWt1KwrQCl1eAgkg5OQD9KhscLWeM47ZHW3PAhvLD77z'
        'ZSVEnGpOTjfI33wByG9ZJqXqNGmuEBLHFqoE10FhpbzigkANgKUMaE6VHoOwHvVs/DMO6obtVsw4+GwX2opK0DPqzjbkB9/ep3FF'
        'tt0eQloWhiGpCNLb7KUqdzyxq6kpwfr051dwb6izWuFbI0UuNqZX4RWrKs8vz7A4JO/TG1aPNSTSM443bRTWvhmdxQ/+NHKYERSl'
        'yXpSfCCtSs47525DYD51fRPh3GQZFwg+DBcUdDT0OQoJHoUS4EnOQMYwcZq54lcEnhBhpuXHdmKALfhyQWwoJA5/vb7VTPyJUGNF'
        'uUbxMKgIclwsEKJQFJIA6AlI+9ZZM2XduNYwhtBy58IzGWYxuHEpfnyXUOtr8PUk6hslWTkDnn60S8IXS2cLxG58qGL/ADEuKSy8'
        'yvU00DgAhB6nue1QGbhLkSXo64wjtywjUp1zUtKSrkAfy74AxUPiNbamTGgyXYKmRpb8EaNweau4NTPflW1uisWo8ovOJeL5z/iP'
        'y7XLYbSrSpakHSDjONtuQNCv7bk3BXhxIzykqGUqLRCVYxsCeZ3HLvRpwHcEz1t2+c+2855dTyUlJDaXcbuJ98Z3P0rnFbzbTbK2'
        'xJmq1tlwFKUhgq/IEJwSVAjrjv1pCPh6UuZP+/g2eoddIDL3a+KYlpbuTsHUxp1LbT/iNerSAU98/bNI4QuU69MSGbn+Ha2QApS2'
        'xlzqNJ5gDH6ijW9uN3C22+O5cJ0Ntt3UEZTrLiU+lOnnjOSd+Zrszhmci0Ieded1ySXD5dkZdVz0lBGyjjGegA96dxOOPG4y9/uY'
        'TUpStAdMcVKUtgW1zSAPDK0qc1JyRknmOQ50MsXhq3pLbsdqGpS1BRSRlKQOZHPJNaUzw/c7janX4zaYUfxSyVOk+Is9TjmcVlPE'
        'vDr6Q9JE1CktqIU2UEObdcdetcjHjw5+sNR/6/ZDT8zGuZck9jje1OM+FLJRpGN07VQ3O92xctLzPhPNpVqAyRn2NUQtSn/xFlxM'
        'Y5/FCc9e1XC+AraLc3JF7eWtwEoCI2QoasDG/bffvTmDwnBglvTaMJ6uc1tGr5xlmN4FoSi3NFOFhkfiLPXK+3yoVmQn5DAkBvOo'
        'aitSiVKo04M4JeTxM0ua2H4bA1uJdbxrzskY355FEd9tnDrLj9rgocabZQoqcZ3T4unOMK/droz1Cg0oi6g5K2Y/AefhrS62BpB3'
        'HetDs3EXi20pBLbmn8p6+4qJa+DnbowluG0tS9nFLJ2GTjH1p7ibhCZYoTriHCthpeklexSojO39RWWrx4tStkuycUp4+V0f/9k='
    ),
    'song_sparrow_02.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAAAgMBBAUGAAcI/8QAPBAAAgEDAwIEBQEFBQkB'
        'AAAAAQIDAAQRBRIhBjETQVFhFCIycYGRFSNCobEHJDNS0RZDYnKCweHw8ZL/xAAbAQACAwEBAQAAAAAAAAAAAAABAgADBAUGB//E'
        'ACwRAAICAQQBAgUFAAMAAAAAAAABAhEDBBIhMUEiYQUTMlGRFIGxwfAj0eH/2gAMAwEAAhEDEQA/AMgCmL5VAFEBXiTMkEKIcVAH'
        'FT50GGiT2oakV6q2GiMZqVWpAJIpipxVMpBA280Jp+w0JjNKpAK75xS2qw6Uplo7hSu1LNWGWkstWxYKFmvCiIrwFWuiUEtGKAd6'
        'YtJQVEkVOO1SBRgUaLFEHFGO9e28USrzTJBUSV8qalAo7UYo0NQ1aNaWKNDS7Q0NFFmhFeobApGXto1Wj2VOMU1lLQAHFTjmiwKI'
        'KKrlIgCrmjCUYXimKtVuQAFjFMVOKYq80wLVEhWxWzjtUMnFWAtQVyKSyWUZE9qSyVfdODSXTmipBKTJS2SrjpxSmSrozQSmyUG2'
        'rbJSilXKQyQtRRKOaILRKtNYyiSgpiioVaYo7UyY6R7FeAo8VO2nGoECpAOKILUhahKPLRioAosVYkGgganNDip5p9pKEbagimle'
        '9RtrC5FDFhaMLzRBaJV5qmUhLPKKYFqVWjC1XvFshaYBUKtGBSN2A8BRba8BXLdTdZQadqUejaVAdT1VzhoY+REP+LHn7U+DT5NR'
        'PZjVsaMXJ0jpWXnFKZOee/pXzS51bqee7ubU6vCJQd0sUUqrsGfpHHP2Gfse9dPoXUllDYww38ngqqhTJ4DbQ3nufJz9zgV1M3wf'
        'Jjhuu2aVp/Tw7Z0DpxSWSrY2SRLIjB0YZVlOQQfMUJSuVtcXTKEimyUto6vGOlslXxHSKeyiEdPKc0Sp7VYWJCVSiVabsqdvaipD'
        'IXipAo9teAq6LGIC1O2iAxRUyAABRBaICiAqyyAbajFO217bR3AbFFe9QE5q1s714R1ynIzNlbZRBKfsqQlUuZW2LQUW2j21KrVV'
        'gIAqQvFHtqJELxOgdoywI3IcFfce9WceSWVtWurexsJi11Gl0VKxpydhx3bAOPt3PtXyTqS/1vTNO+E6fmMBjTw5pQQJJyW3MWcj'
        'IyxJ/T2r6SelwchNa1WJTniN0UjPuFBzXLdR9ODpRWvdaurzWNMmZFF9HNia3JPmCCDjI8sjFeq+GZNM1twd+b8mrFKKVGR0rP1h'
        'd2EUtx8FleTcTh2YZPYEMN36Y/StC41zrPQ5m1K80rT54A+yO4tYxHhj/CwDZb3BrW0vp64tims9J9Xm+0y7jKz+PKJWjYcgqMd8'
        'cY4IrKFnp/TF/c30jyXl0XIjmmjMwtywBY7cgZJJ5wceVddzd0aI7TrtD1n46wF3qFg2m3BIWRAf3ZPltXnb/LNa5TPI5z2r502p'
        'XF28bG7MTKQqvboETuTll5xn27elfQtHlefTYPFcySquGY4Bbnvxx+lcH4zpoqKzLvyVZkmtyCKdqUyVeKUtkrgLIUKRU2VISrG3'
        '2rxT2p99jqRX2V4pVjbXtlHeMmVtte2VZ8OoKU6yjpiAtTspwSi2cU6yksQF5piLxTAlGqU3zRWxeyp8PtT1T2otntR+YK5CtteK'
        'DinBeTU7K5UpmWxBXio28U8p7UO01U5i2L21IWmbcVIWpuBYAFeIFMI9qEimcw2DmhmghvbV7G7XfbT/ACyL7ev3FGVrwFHDnliy'
        'KcXyMnXJ8cuf7O+seh9dFppl293bT3IC/DygIJgCwSQd87c8edEvTmr6h01P1dcavbRhZjsjl8T5X3AbQRz3yBX2PU2S3vJdcsGa'
        '6gsbyCXUbaIb58RiVd7KeCAkiEEcnZ61yep6Joup9R6xD0veXKRX9kl/E9tOGgJZ842eXzDOPcivpChuSkb45OKOEiD6JqebppZ5'
        'CNolMPhbHxyrNn5sfgmuq6L6lSSKBDAYEcnxERMg+QYZAb+tUHkGrSy6F1BprWV+VLRyBS0ZbsWAzyD79q517HUrPfJaTwyy24eN'
        'YVfllzu5/TvWbNjUk4yVpmqEYTjTPt+0EKwIIIyCOcigZBXzrozqu4jt5DNFI8CTbXA52g88+hAOfcD1HP0pCrpuUgivIa/Qy0sr'
        'XMX5OdmxPE/YrlKjb7VYZaHZ7Vz1kKlIVtqdnbimYqQKnzB1IVtr2z2poWjC1YpjplfZXhH51aCcdqkR5orIHcVgntRqlP8ADx5U'
        'QSm+YJJiVWjC0wJzR7KKyiWVwtTtogKLFY3IpbFFajbTsCvEVW2KJ2+1QV5p3FAe9RMgGKggYo/OoIo7gi2HtUYopSVRmC7iBnHr'
        'UQuksSSxtuRxlTS2AoaFNaQXmoW2pRXMMrag5t7yA/OYnWFjx5qr4B47GsW76G0FLL9qaBfz3Gr2UjRSzW7AbwJCzROvbdhsc98C'
        'utXTbSdxeOHS4t33hkbBIKFMH1ByM/8AT6Vk/FWNnqE0kKmOWTAnKoB4pIxvbHmMY+1fSPhedZ9LCXtX4NCnxwcT1nf3K3kV7c2T'
        '6fGs3hGQyYjkU/S4YDdG47Y5Ung+RODqXxciyPDYz3F8oJk+HGWmizw+BwSPUD71rdTKsslxatI3wt2A0crjdHBMCAMg9kY4z6H7'
        '1btxpmr6RY3E94mhalZTGHxYDwkqjBRlPkcZ79jxWqXqNOOdHDWfUEdpPM5iEkdzhWZcq24AjBHkcH9c+WK6/wDs86/tLdE07UZt'
        '8cQOGkBDPHk+f+Zf9farWvdJF7NtUuE0+WSMBpLm1AKTp/nKD6TnuRkd+1ctHplsl3DdtEqTRjxI5osMNpGMnyZT2/71lzYU4uM4'
        '3ZpU1NbX0fcLeWG5t47i3kWWGRdyOpyGB86kjmuP6N1e3sLePTWjSOFm3Lg8IT3I9V9u4967QoQSD5V4b4hpJaWfs+jnZsTxy9hZ'
        'HtXgtNVKIJyK5ykVpigntTFTmmKlGEqxTLEwAgowlMCUaqKm8lidntU7Pan7anbTKZBGyvBKftqNtHcIzPBqc0gPRB81WyobmvE0'
        'ovxQmSkYBhPFLLUJbIoGNEgwNRgjFV1bmmA5oMIzHOaxbe4/ZutPYzkLbXLb4GPZWPcfbNbQrP6h04X9hlVzLF8y47+4oRdPkWV0'
        'aUrIkTCRtqOCrHHAB9a4vXLO8tLqe5aSQkAbBnIYEgfnv/WtPprXVZk07U5VVidscz9m9m961OodOSTTI2xM8UcpikiRfnBHOznt'
        '3HJ8jXV0eo1OCO3H9LZITvhHDabqS3UMsdzCJjb5BUD6h7evFBLZaNd3N46hmE6KJ48DbkcqxA59s+Vc/qc1rZasyabfSQXYmEc9'
        'hOh3I/Y7WGQwoNS1GeHVp47clJoGZkOMh+M7SPcf0Fevwa1qoz7NEG0dfZ3jdLXMFxYaTJc6Nc83EEA3SwydiQnmpPcD71znX2na'
        'FcaT/tB0heTWyIxmliiyrQ5PJVD/AA57jy966TpAX9/qaRTKsdrcWiXKnPML+IFK59PlY/auvv8ATdA0O01TUbqxS6ncSJAscAle'
        'ONyxIUHhSd3JPACjt559T8b0+GSi+b6LoZD88ad1hJH/AHfV0SIOw23CrtjfHYkj6D5hhkewr7N/Zz1KuoRppd3JumC7raQkfvFx'
        'naccZHcEcEcjtXKWPTnSuq6Ze3t5B4UbgE2gyGiPkyEZDcd9vf2odT/s31no+00/qDpK6llt4lE91YynxGiH1CWLH1DHdO/Bxnyv'
        '12HDnwOORpX0/fwPkaapn2QJ3ogtUOk9U/bmg22pNB4LyKC6g5XPqp81PcGtYLXz3Jjlim4S7RkqhapTFSiC+VGBSWOgQtEFogOK'
        'ICoQELXsUeK9inTAxZFRim4qCKNgOXD0Qkqn4vvRCT3rQ8TEaLXicGh380jfxXt2TSuAKLIcY5oGak7+K8X5oKAaHK3NNRqqK2TT'
        'kNBxJRaSiuGlS0kaF9rhcj3xzilxninRjd8pGc8Y9aWD2zTFa4OR1nTor6J7yyCse88A+pD6gd9tbXQHVu2+tdM6ijhuLUMkaXUh'
        'wygfSJD/ABKD5nkfaqN9bz6bqYZo5YmQjcQCGx5MPf8A+Vk64IfiWvGdHiGWkaCIDcPP5ff0rfLfpp3D6XRmTqV+TH620aK766vr'
        'iWPbcR3olcA90JBVgPQ8D8e9VnsFnvfj727W3s7X97LuPEi7DiP3w4BHmNzY71315b2Ov9O2+t2DNHc2amzni8RcscAANwcntxkc'
        '49KoaX0u8/Tv+0Mxt544S0qWkgwGKAqHJPGMk8Y8qslrZQnGuHVfn/ssqSyenyDJq1n0vodpJNeQHVJIFWNZpRGqjH1Hd58njy86'
        'wepLnqO40L9tafqdncBkHjxkpODk87jyCPete26Y0TXLNNU6gt5ZLyct4YVWzMQey7u/ucACqWs6Hbjp+S10kW2jHTLjYm4ljJkZ'
        'K58ye32Jq/T4dNg9ef6vf+l/dGrGlD1z6LHQWg3M0shJt0doVk2qoRee/A4yO3H6V9IsbK6tIVtZtpMS/LtJOFB96+X9EX95p93Y'
        'aLb3LTXlqjl5Y04UyZYEgjsMDk/6V9Rh1WW33kzNLqEu0k7dxC+pA7Hz/NJ8b1UNsIR6q/8Af0PJ3T8MJLZLUGJIBAMklAm0Ankn'
        'FH50cs8tyTLcF2c+bHJ/8UArz6lu5FJAohUUQ5pkyE4ohUCio2Q951IFeFTmjZCKB6JjSnNLYGcAJeaITc96oGTBqRJyOa9FLGhq'
        'NNJMijV+aoRycd6ej5NZZY6YtFkvUFs0rdmvZyar2kHo3IqzGaXpcUdxfQwysVR2w2O5HoPeup13StNin3WCPHHAvzoz7t/BOc/f'
        'is2acYVuffRNtozdKsri/uFgtkDORnkgAD1JPlTo/irC6JUvDNGxXcD2I780/RJ/DspFhkKTFT9XOPcex4yKLULzda2U0qkyYEZC'
        'rkiRudx9gB+uKyyzxWXYu1yBpbbMzqaSfUFF3d3imWNNoL4Xcvpn1rlrSSweQzBVljbicbMMAM8Y8u9dtNBE+YrkRuh+sMNw+3ua'
        'w9S6X0l1ae3P7Pmz8sivxj0IPcV1I6qc1Wbm/v2Z5Y+bRiWk8PT+qx6Uka3Frqdz45OOGGBsx6MD39MV1d1HbtZXums8yabLHI77'
        'G2GMNltoPlzk/Y4rh7yybQtSD31xhFdQCRuBOchiPb19627nVvHtJ7VYpJXz4Lk7QArcZ574BP8AKsGtxSlODh2n/vwSE9r+xb6X'
        '2RWN3dGMl0cxxc7v3QA2KP1/JJNZLaPeapqc949rJJCJSVhVwgkftncew9x9qt2cEOjpMNNuDPCTl7e7n7Ad9qqOD7k1tWWvwwyT'
        'NEUEcYzuXlRnHy/1q/VScfXd/wAjXv8ArZxWsNrGg3Zn+AtdMhvWHiSWb7nfGBhmb0zwO3tWzFqc66hBpGkXV1cXrf4xeRXjt0xk'
        'k4HJ9qzNXvdd6qmms9Os2isI28Rp5k2FiBwVz+cAV0fQWm2mn6KJIoyLiUkTu31Eg9vYe3vV8q/TKco+pf5WRLfzH8m1YWhti7yz'
        'NcTOctK5OfsBnAHsKuCl7qndXLbvsvVIb5cVINKDcV7dQIOBogc0gNTFaiAaDXiaDNQTUISTxQN2rxNCTxQYD5YzHJokbtQMOalO'
        '9es4LbLCMcd6sRvVMNimo3PesuRCsvK3FGpyKrRHNWYxmscuBbNjRvFRkkSO2lMbAqPECSA+x8/sc1t3N+puiZopI2k+Vo3QjluA'
        'MVyUZKMGU4NNd5WIkgnaGVfpwSB/4rnaqDnXHTsVyaXBreOILoO0e0xY3KuRj1yKRHfDDnxP4i0ce7OB5ce5xzSrC5a7um+JmtpJ'
        'QFB8OTl8d8g4Oeaw9Svk0fXo99sTaO+ydj/ugcEMPbNLLBKf1R7ffkpc6Rpx3t9pF68F4Tc2Tkss+7mPuTn1Ht3/ABWlbvo+srFc'
        'D4e5K4ZG7lcVVcftGeWObw5F8Fkg8RdyqMgb+CCTnsc9hXCTaV1Do9+0klm8iA5+JtCSrD1Ze4P3FdLZim7i3x19xN0o+LR3XWel'
        'Leafa3dqzkLE1u8WO5UkbvuMD7iuYv8A4mK6kntR4fiALzyUxxgf8RIxWnomra2UZJmWWGPDus6hQOR/FgYNVbzTRq9iSLqNZNmx'
        'QJAFZ8/Rn3HH60mWaclL3DJLIjD1K2e3u/CnuY5mlKBEt5SuxyM5JH1Ht7Vfi0y/a5Fta2eqNp6uZGIQReK5OSxLeXpx2pVqwste'
        'tbS207CQzAyQxDewOwZQE+9dbL1AtpG8upeBDtGTFG291/5m4UU3rb2pWLGMXdl+yWx0vT2nlt0s440zIzsGOPc+dYPSF3c3jzWt'
        'jJ4NjDIWaVl/eMD2VQeBwKzm/avWt3G+2Sx0SNtwYjBmPqAe/se1dBp8dvp2vfs+2jWKKW3Uxr7g4PP4oqFRnjXdXS9i+DcuFwv5'
        'OgBJ7c8VAevQoxufBwd/l5c/+5p0llttBciUNnOUA5XnAzXKbS4fZZ30KD17f70pWC5DcjHAHmf9K8GzVcMimrRB4amq1VVamK1W'
        'WQsg14mlhq9miEImhLULNSWekbFs+cuMUJODViZcZqs4r0cco1kb6ZE3IqvT4R81LPJYLL8HYVajNVofpFWY6xzkCx6YDAlQ2PI1'
        'ZhuJlwFIA9lAqsveneGsi7SXA89rEZ/I5rK27Axk9mlzKJpFIk9cd6R1ZY2qXNu1tlra5iKTozfKkgPJXjIHYgV79kabJktbDJHJ'
        '3tk/zoJLCW2hKQRtcRHnb4rbl9wGJB/BFNFx2uKfLKpK+0U+mLq6tLZ/i42eS3224UD5nBLEY/l+lOv7q7uI3up40WCF/CZIm5kk'
        'JxgeuO3fk59KuwpA2nxzrMkciyAbJDtdZF5CkHnz/nVWVUhvII3P920xFZsjO+ZvP375/Bq+EpTmopciVS7KfUN7Pc6He6Jex3Ec'
        'xjDQPKMGQqQcZHGcedFb20lpoTxLbpLNIqtMZR8i4C5ZvzjGOc10q6lDJBcpcafM8IXGGKEnsQQuc9qy9Nkh1C8tnW4LQNcs06n5'
        'SdpyAQe3YUM8c2KUXKNWy6ME/Jk3FvqFpaA/ByQSupMsltbtJLKDgZK5wmQB3OT6VV0X9meIJNR0bWLt0O5Wmtw0SY89g4/XNd3q'
        'Go2lupnmmA3cqqjlv+wHvWdFLPqzLO6mOwjO+OPP+Mw7E+oH863qGWGJvL6Y/hsrcUnwPstQjupZolyDFIE5UqcEZXg9uM/pWP1i'
        '/wCz7zR9XYkKt2tu+P8AK5xzTbRFm1vUR4iobuJHhG7ncvn+KzerLqPVOmoIZfER5WSQBFyQ6SDIH3wR+ay4lHDm9fFcP90PDJXL'
        'Oo1C/wDhXttQicEQyhZQx9eOf5Gtp9VsXsZiLiOFZwCG3Bl3A4Y5HYfeuE0aO6vv7xeLiGYgG2jOXchvTuBgcmtea3ha/jlljlKw'
        'khYkTdAvsRzuPuaw5cLz3J9f+AxzpF6C8sJGJ8Z5iDgLEpwfyasu5LArtUEcBT2+9L2xuoMMFvEP8sPH8s8VKqQcEVncHi9PgsVk'
        'rxTENBiiQetJYyHKa8TxUDioY8UUyASNzSGajlNVyealiM5GZe9VXWtGde/FU5BzXUhIayoV5p8A5qNlPhTnGKscuCIfCDgVaQUE'
        'MfAqyiVnkyMKNTmrEa+VRFHW7pmjLdWkkiXETvwEUZBB9wftj81RlnHHHdIiTfRmRrXmubaLBeeMAehz/SrDQqAEcDd3Kn747fij'
        'Vdq8Ac9s5qQUWxOTi+pfhoreO3jlWdHUkybsur57nzwQe32otCu5jpLtcgyyA3DHe2C22HCnPnjdn8Vp9TwXE9nKJLBJgB8ksPLI'
        'fcd8frXI2GotAnwM/wBG/MbE8rkYI+xB/lW3BN4Z7ouzJNbZ8m9Zao0vTsMTy702/vPEOG44+VhyO3nxWRousLD1XawRbmhmLKx8'
        'yAM8+9YXUMWp6JJpEsdvcXtrexuktqu7LMjsCVI5DYwcj+ddh0/pmmQanpmo6XZzz2E0ZM5lB8SNjkMDxww7V0NTqJSxKXaZZGLZ'
        '2Euj6dctBPPbrKdiugZyykEZBIzgn71z02v3snxOk28SC/jmMQc4ChM43Y9fKuvaG1t9Djt7eVWJdY1cEl3GfIeWeB9ga5K502ax'
        '/tDXUJpIY7cqZZDvA2DZg5XOe+K4eLLLM7k7LsiaqiempdTPxYsdOtLm9hIiRrlyqpyc8AZPYccduaGxtdct9NvAYYLmRwwiSABd'
        'pY5JLk88k4VRj3qvddZaXbtdQaUkskbs0kpX65fU8Zwvt/Osq3vup+oHB0i3mtYM/wCPLK23H5P9M10pY8mT15Woprz2VXGKpclr'
        'pzQ5VuFu9T068TJ3CSKUMv8A1AfMPxXafCQiQSxxBHx9SDGfv6/mh6e0OK4sBba7d/HyhGDS4KDBII4B5wQOe9adx8HOpit7aGKK'
        '32oh24ITHAB71y8+XG57VItx49qKqpzTcHGKgYHAzjyzTVFZW2i5IDZUhaZj2r2KWw0ARQPwKcVpcgwKlgoqy9zSCDmrMi8mlstV'
        'yyUTZZzdwneqTpzWtcLnPFVHj5rrxkIUhHViFOanw6dChzRciDoE7VdiiyKVAmMVciWqwMKKPawOAfvV5JlSL91aIZRyrK7Kc/ri'
        'kxgcZpk8tvHCfFt18I/U7EnH3wRQcN6qyLgpW9w1zqWJFdJfDw4kGDkef2q5rBe00xr1SrKIlZcHsW4XP5IrCu9PxIl1p2qSwn6o'
        '3dHeMfY4IxWNddQSLcSabrckDECMR3Ft/hkBgQSO68ZHt6U+m06xwmpLnwVSnt7O5toWihjjIIZVA981836301bTVJCFKpMd8TD6'
        'QD9Q98V2mq3tzZ2cYjv/AB7e6ZY4XaQcA9zk8ds85FJ6lgN/aJa3hs7NhzE81wikH7DyxWz9JijHfHIr+z7FyeuNUcY1jcdQ6AdG'
        'luU8aF/FtLiOQMNwGNrY5XPrj0zz32OgdQ1VNAcam8s9xZXTQEyf4iDaO58+575rA1Dp3UYLxPDQh2b91PE3yN9mHFX9C1rUxM2m'
        '6lMkseTllcFiB55X6uKMsm7A1FlcJu0pI6C/vbwvELTULKxDHG5kLsBt7qoHofXimvoNpf2kcN5PLLuXAeWU7pvMtgY798emKwbY'
        'fGGOUxXKrI2ItjqCyjz5/hGPYcenfRa5ttLuXuLS9S8uRGzTLsDNJjvjzBx6cGq8GCbx7YVZY5c2+jT0fpXRNKcSW9r4kwORJMdx'
        'H28hSepeqbPQrGNzC91K7mKKODsWHlny+1cfc671P1aTa6RaTW1mTiScfINvn83+mTXQ9GafDaytBa+ItnE7M7StvKknIwcdz+tG'
        'UIY3tzStvwHfbSgqNu3vLyDT/jJoxEZVCpCfqVmAJB+1XrEqIgHnVnPLbQW5/FZVzDqd5etPsighUkRiXDkD/lHGfvW3ZWc6rFG0'
        'jTvJwhICg8Z4x9j+lcmOOON75LktimNAX+Fi33GKagpartYqSMg4NOXvVU5I0KLCAr22jGMVOM1neRIdRYGOKXIvFWMUtxVcsqGU'
        'Cm64pTCrUi0kqazynYyiYEiYzkUh1q3L51Vk7ivSRVGFC9tMjHNCKNODUYS1F5VbiGapReVXYhx2pbAWkHanIOKSnlT0qEMi8SbR'
        'JGv7CNpLMndcW6/wHzZPT3FVOoDpXVNzbjLvFOrASLgPGw5x/wDa2L86skTNZPatwcq8RJx+tcxpsCrqIuooxazgnxYoz8jj1x5f'
        '+8VavltKU+KfZVNv6UZsej6hYI+iT3C3Fm2Xt34+VipwMHtnPNb3RtzJe6a2l6rbLIFykZmAJYAfMMHvtrxgtr291Jr62eZvFj8B'
        'txUk7QcLjy5Ao4rS5TUINUvN8UysSqudnyc/IA2Ocf15q2cXKHzUrQkVUuDL1rpdbG6xY6hNp0FyfDDqT4WT/A6+WfI9vt5p1XRU'
        '074a7vlht1gXEjREkMexIJ5GRnI9a7dv73pcyX1miiWM5hY7sD0Pv9q5TrN7eKxstPa8+JRV8RWbhmyBtRv+Icgn2zVEJvLJY75L'
        'HCMU3RgWfi60lvbQzXEYPAghAU+GuMGRzn/8gY9eTTuirHSL2O8MdlHHqkYJWYEggnzAzgc/1rsOmtMs9L0mbZci41C8RfGkTGYi'
        'CThD5DnHPfv51yMNu9h/aVJ8N+7ikYmVeyqrLuJ9hnmurqsb03/Ffiypp2pM6jTr1bjp0TtIluFiKufJD27eufKqWkvbSxGGOa8R'
        'Sxb5InRcnz3Y5NKudQWDabO0lmhV2ILDZGNxJDFj55PlnFN0HVJdRvJIJLq2jVELM6xM6g+Q3Ejv9qwww75brSv/AHgttKkx0FvB'
        'a6ysZW5aJkGJGncgvntnPpWqt88GrW9vEZJI4gXxu3Ek9sk9vvVPR7QGS41W9iEcJcBACwM2M9hk479+1THqQF1I50uZA7cYUMCP'
        'LsazZnOl6/5/gaKSX7m2k2VI2hcnnHP4zTkNZ8V3DOfkiMJ8wQw/katxNXGzvY6uzdBJ9FxPWjB4qur8UW/3rA8jZoUBzGlswpbS'
        'e9Ld/ehbYyiG5FKY+1CXoS4opMlH/9k='
    ),
    'song_sparrow_03.jpg': (
        '...'
    ),
    'song_sparrow_04.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABAUDBgcCAQgA/8QAOxAAAgEDAgQEBAUCBgEF'
        'AQAAAQIDAAQRBSEGEjFBEyJRYQcUcYEyQpGhsSPRFVJiweHwNAgzQ3KCNf/EABkBAAMBAQEAAAAAAAAAAAAAAAIDBAEABf/EACYR'
        'AAICAwACAgMAAgMAAAAAAAABAhEDEiExQQQiE1FhMnEjgfD/2gAMAwEAAhEDEQA/ANBt883WjVGR1zQETYPWjoz5a89HoMguwAAD'
        'UNseWXqaJu8Gg8EPnFMAG6PlM0DdDz5PrRVs2YwK5Nu9zciKMddyewHqaVdBh2mKxtWlwfDUeZsbCqnr0zcQ6gmn2Lt4CbySDog/'
        'uaI4q1V47iHS9ELTyA8rD8rZ65FN+HNKSw09YgirIx5nxvv6Zrsi2SOjzoNb2MNjbRwQIFjTp/eiYcIrSt0UUdeQYSkfENx8nph3'
        'wW3NE3rGzox2lRTNdkN9r6xA5CmrUsPgWkceMELVU4Uia81Zrh9/NmrjdnJxS/jRqFv2N+RK5UCoN81MuOWo12IqXHeqkyc/cv2r'
        'howx32qUfrXQGRg11nUCNERXBTbJFHmPbptUTxYorBF8qbEY2NUri3T9mkCj1FX5496V63ZC4tmHLkjtQyVqjU6MfjkaGbBG6mtD'
        '4Q1ATKm++KpPE+nPZ3JkGcd9qI4U1AxTIpbG9TL6Soo/yRr6KGwwHWuwuaG0ecXECkHORTBVqhqxV0RLHXMyYoxU8tRXC4HpXUbs'
        'LZdhmhgxZ8UTP5jgdK/Qw+1Z4OsYoPNRsJIAqO8h8C+dAPLnK/Q71Mg8ucUnw6N8ohnbL4qOQYTI610wLS7UbNawWtv8zqk3y8OM'
        'hPzt9u1bKajG2wKd8I9JjedyoICqMsxOABQOuaySG0vRQXkY4llHf7+lL77V59dkXT9GVbW0TZpB1+57mm9hpsFhCIoVyfzOerGl'
        'r799BeDrhvR4dOh8V/6lzJu8h/gVZbOIHBxS9BhVFOrJf6WTR0DfAa8iDHAFZ38RbjlR4lPXyitLufKkknTlBrGOLZ5LnWvBByOb'
        'FLz24qI/46puQ64LtPB08zkbvsKYXLec5qayj+WsYoh+VRmgLqXMhHSqIqlQlvaTZKjb1Mp2xQcTBsb0XGNhRWdRKBmu0FeINqmU'
        'DpWHM/ADFesoxXQG1fjRoBgskdeJBzdRmi0jDbnpUoixuOlGCZ5xzo4eNyE7Z6VlyO1ldNHnDK1fQmt2QubVsLkgfqKxH4g6Q1nc'
        'GZFIGetT5I+x+KXovXA+qCeFFLb9xmr0oBAI71hfw+vnS9SPm3Jrc9L/AKtqpJyQKZB2jMipkwGBQt15tqOfYfWhjGTTBVi/wjnp'
        'U4iwo2ou3gyckV28IxtQyQUR3qOnveWEV1aJ4hgzG6qPNj1+xoS1spXg8SQeFEOruMD7etE6RxBD4WVBSV0VnAHboSBSfiia4ubu'
        '2gtZ2aHkbGH2wa6eOP8AkBCbvU51bU7PTVDabILiQ+VmZPOG9FHQfWkB0u+1WQ3OqyOkRPMIQ3mP19Kb2VnHbHJPO/ckVO79RUss'
        'cZytlCk0qBdKt47eYRRII0XoAMCnMhAYHNL7Yefn70WTkim1Qug+zXncU/gTlix7Un0sAlaeKNgPWuBYo4kn+U0qRz3BNZHo0R1D'
        'iIysMqhLGr98TtQ8Oza3U7naq5wDYkWM9267ucKaQ1tlX8KY/XF/sZ3D4U0jupfOfrTy9U8jb0ikjHiZO/tVJOgmxORnuaYIaCs7'
        'eWSOSdV5YYt5JTsiD3NL9a4p0fS1eFLy1lnQBpJHlKxL/pBAPM3sDj3oWMXSyR9B0qZQaynUePWuYgLCS4nQrhnW1BVW9QOcb0ts'
        '/iK9l5rmCeYq3LmW2ZAD2yA233rFJBaM2wV6iljt0rJB8SUuFDGR7IsDzyJ/UVMH/KwyevqPvV54d4v066TE15bSpz+GJosjDAbh'
        '1/L9elHulxgfilVlnCY6Cpo1yMV+Kdzt9aljAFNEs58EEHNUH4haD8zbuqrnbatD9NqB1eBZbckjpXNWZs0zBOFdFuLHVy0wACNl'
        'a2fRbyFbUAYziqFxVcR6fIzAfelGicTySXyxBzyn3oVUODaeQ2IMJPMNwakWP2pfw/OtxbLk7kU6jjHXFGLOEj5RtXLpkelGFRjc'
        'VzDbyTzpBEvM7nAFDI1HFnBBHHcRXC4QSHwmC5wp3x9Mk0skj8K5kcMGVgOT/SPSgZtR1eSUISUgQk+G2Mfr1NHuGkiaTAGcN9ul'
        'DklsmbGOrRGzZJ+lQM3mJNTPsv2obv1qdPo8KhY4xRULA7UCjEbgZoi3YlsUwWx5pWecU9EgRCx7DNJNMIGDRWrXIh0936ZFc+Ap'
        'W6M44+ujc6iYgc74q1aRZCz0KGEDB5ctVKtI21PilF/EofmP2rRrkgQ8gHQVPg7cv2PzukolZ1E45h3pdBZyTh5iGEKAl25Sen0p'
        'pfR5Zmb8CnzEdapvE+tXct+9haQutoVwkMwUK+B+YbkgE9emd6fKVC4RsTcdcVTxSJYwZjh32mTyovfAIKZ75ODmqC8+rawCsV3O'
        'sXWOae8YAn0IBZQfrimOptc2bPc3l1FYvNmJI4QZ3bfcgYIXf2ovTYIZ4EjnmllI6g2687fUgDNZjx7dkFkya8iLrfRZZIyJVkku'
        'WGGaMc8RB/1LyN9xmp4eA7fUpmVl5E5eV2S4lAxjcgt2+2CexqyNYLGisLK4t0xzR24IBlGOpUbgfeoOKeIouHNB+Zurt2e6YwRW'
        'UJCICAMlz1Ip9RjxInuUutnsnAGmaLpbahq5mure3C+fnVZQvQ5VcBh223ONwaL1fg7R9Pu4JtIurqyuQSee3kPIygDAPXY/rWRn'
        'jL/FNQjsHSEeLIRG0EjsVO+AwPUHJH3q7GW5HDyul5IHbyjlcqAxGMe/Wlzh6YcJPymPR8RuIoH+WXWJZ2ACpyJ5mI6sfY1Hf8e8'
        'YQMnj6uLcN3l5dvtiq/b6Q3lsdGee51GZOsW/Jtudx+5rSPhl8MLLSUTVtdnbVdQYh1WU80cZ7dfxH9h+9PeWdaoWscV1jr4Z3vF'
        '2pob3WZIzpxX+i0kASSY/wCZQNwvuevb1q7SIrKQehr1f2r9IwAwK5f0yRl/xE0Pn8Q74IOKyO35rK/YYIZGr6V4kshc6e3lBYCv'
        'n/jKyez1cyYwpO+1Izrlofgl6NS4D1AS28Tg5BrRIcFAw71hvw/1EQyiAtsdxWzaNOJbQDriixT2iZkVMPY5rnW9Ui4Y4ek1Gb/y'
        'p0IiGd0X1+povToYiZLq7IW0txzOT+Y9lrO9Uu5eNuLHlf8A/mWjZYdmPZR/3pU3yZucvwx9+f4huCC/zl4Q+IacmaPlCDKvkbg/'
        'T/vWpIw3JEEmIBGNh5T/AMUOZmjuPEiGQ6lpP4zj7URGyrAOTlYA8wz9aqbE0c3I5QF6HvQDE89HagwDjcbjNLJJAJDU9Uxy6gnx'
        'MEAUVbk81KfEPiDG5pnZKSQWNGmCyw6eDtmlnG174Nn4atjamljkqABn2qocdGV5vCIILHApXyJaY2zcKTl0j+HVqXmnvnHXZTVs'
        'uWG4YhR3JOwFC8LWQs9HRcYJGTQnEd4ILd18ORh0yADk+mM5NdCOsEjpPadlf4i16PleyQHnyfDPl5VB25mJ6deu9VlILSCMW0ct'
        'ld6h4ZEksaFoo1G/mIIGe2B6VFqF1Gs7Sc9nOUUlQGIHMBj8Pfc1VNX0rU7mNILe4uYUKmWYq5bm7k9PL6daZFOPX051Li4N7jS9'
        'MvLyKW91u1EcGD4cVq0Zc+hI7e1Sza/p+kAupggiG/PIdz9AR/zWWSWus3N34Qg1C4JyQDJyBQPQ7ZOKCfQ9a1aSW6n0++kYEpDF'
        'IWYgDHmP3NEslgvFTLRxl8SUujJFYQC4YqUEhUhR9Kq1is2szEcRWt/d2sq8qPbneLpgjY/TGPvV24c4Jj0qCD/EYYEvXXMqzN0P'
        'XGD0o4WUKYFtzROQWDQtyqvLnB9h3yaLZMFx14UvQeFNKs9SlOltqF9fHyQma38NYSerNgnJHStA0qyF1dabpNtEbiEu0kmWBidV'
        'wCTg9c9Mgde9UjUtf1PUb2PQeHLmV3EnhRqi+YrvzOxx7nr671pKaFF8NdBuryC+Das8ADpdYIzkFlRT1U5HQ+vtXS6YuFx+F0+h'
        '2vFWo6FZwQRlomYOcB84wR7gZ+303q52GRbhTjbavmL4a61PP8UrWS5ljh8VnkYqeVQdzyls7YOcN2zvtkV9IaVqMFxb8qyZdQOY'
        'EYJ96YkopIDyNHk5RuajVyxyaEaXnbeu0eiOC2w6Mp6Gsm+J+jFkkdV3ByDWqI+aS8V2S3dmxC5ON6yStGw4zAtFvHtrxSMqYzX0'
        'L8Nr1NQtY4xgs1YHxHZHT9VbykIxq38EcSNpFhceGT4jryoc9B3qTf8AGnRS4bo0n4na68kkHCeg+eSVuVmXuT1Y/wDelS6VpkGj'
        'aXHZQ78oy792buaA+H+llIJNevjz3l3+AN1jT+5/tT2+KsDyn7VmHC4R2l5Z0pp/WPhCiVmjQeE7F4xlScHPqP2r2UoyhyfJ3CnH'
        '1riUc1wMHHcYqaNl52JTlDHlIPr61uxlEd6ojA5WLLy5Bz1pHcXOZNqfsjtAYMghNx7iqvqEZivSig4JyuR2rGEhrp58Rwxqw2se'
        'AKr+m4CqOlWixAMQ+lMghUxlp1wlsRJIPKOtIeItT0291iGKPlLk01mYLZyE9MYqoaLZJNxJJPy55NhS82fWSx15Cx47TkX6CKQ2'
        'OYIGl5VzyLsap/EPE2jraCDVNIe2lJxGk6EszdD065q4WGvx6ck8klpP8vAuWlK4H275rM/iF8VeEbiVbk6Kt3eRHEbumeUD1p1c'
        'Frz0p/E2rK14LJIVglcgrJNHgKpOwB67egqB4ua3dfFF60gzyKp8F2z3JPLjal3FHGema7Z26XWizTyTSKrpbx8p5SccoJ9uwofi'
        'u50vTLQ6XouitBdEiNW53BDHGC2/X2rTeex4Pk3uILi4j5be2QmWWMACRjnYHO/phR3604kj0i9hI0u4W0uuQGNo8tKD1UMcfzS7'
        'h34ccXPbLfaprNmsckalIWh84AGwDbEZ7+tEXqX+iaOl69np8D27NI3NJzeM3TmZRgDp61koteUZGSfsT6vHxlqvh2D29o3izKqX'
        'n/tuD9D26mmA4W1KfTI411TT7UylYkDHPiHG4yNgDsT/ADSnQ+P+J+IruZTpGmAIoCNliqAE+blz19ycDHSqtepqUr315fXsTLau'
        '7Qoh3DE5ON9gKW1Khsdb6afwLwVwxwI91r2oXi6hqKsULQLkQvkZXbpuRk+4rMfixxjc8S6pLMMyQW0jxFWjP9EnAJIx3xnNLNFu'
        'tcu7qGyknma3PNcAtKW5yPUHr1rzS9Nvo59UcsvzNlILho5B5Z4znmA9RuR+lHBS9gycPRxwfplw3EVpeQqs628iTupOA8eQG3Hb'
        'B39BX0JbXumQX9vqdhqq3miXDiF43jHPZszcnMpXY8pAyD1BzvtWb6XfWvBGp6TxJo1ql1oWqKDdQSDJtpO+PTY/sfarXr3AnDY1'
        '+WSz1YWp1PmubG2yDGWZckAZyBnBBHQ0yQqLpmk6rbR2cyeBcpcwSLlJFP6g+9Do+9UT4b8HatoKXGqaprouIZl5IIuYkMDuvXoR'
        '0q6RSZrU3XTmkvAcrV3yiWNlPQihkYZp5w3YreXBefmFrAOeZgO3YfetMfDIPiloi22n/PS4VnP9FO7Du30qh6NKFlXm3UEZHrWr'
        '/FIz8T62Y4Y+S1iXJIGAiDoB7msnmhNhqDwkEAHbNRW5LZopXEbZwlqPjWarzbEbCmd7IQMg71nfBGoYYRFulX4f1IwScmqcct0A'
        '1Ts4UKAQSSVO2BuK8AOV83U53HQ11NyeITv1wcVxMrIRk59celTexpNbyBWZWQcytj9elDa/ZpPEtyp5XiwAMdV/vvUiENz5yRjG'
        '3cdP1BpjauJozBKiMc4OehYb7fUZqiEbQiTpiWyi6Yp/ZZEeKXG3+Wm5Mkr1UkdqYW7gAViVHN2jvUm5bA+9C8F2bylpuTPPISGF'
        'c8QT8lqVHZaY6HheEHt7cPFcSLy+Jnv3P/c1Hyfye+kPdxxf7AuMdZ4lfVBpem6UlxaAASSCTmxnrntmo9N4c0WGdrtdFtBcE+Y+'
        'GGbm+u/7Udw3piaHZTtcX8s+cszTtsv61TPiDx/qejXKW2iaesuWDNIYycj27fevUxJR7IgyScvrHwP9U4Q0lJZ9XksRLdIOdY8n'
        'lDDocDHSsuk1bVbG4utRfQIp7mSfMch3AA7KMbfWlWpca/EHjbUf8G06P5a2aUC4MS4Cp3DN6mrrxDxBovBGkWdpeWxv74nyIgyw'
        'P03wKY6fQY7LhXLOf4gX6XOu38LmKY8ttbgHP2HYfWuLLgHW9Xvo9Q4q1CWOFRzm3BGQfQ9hinNp8Qda1BJpINAmggjG3N1J9APW'
        'lOoaXxpxaIpNUujpljMfNBCeUqnv6k0ltehqT9nj61wdo0V5oWiwrJMqFpJIhlVHqzdz7VQf8E1DWNLmk0yRza3U5EkjDzcuela1'
        'o/Bmg6HpVxbLBtMP6shOWYdhnrVM4o4v+Q1CPh/Q7SOOGGVEeXryL1P3xQPnkNd8DNeDrfT9Q0a5Ut8vaw+Ey82ebbGf5pdaSaDr'
        '8utWDq1vfjxbeLBxkZ2x+lLfiLxu9hbWaWjhVmZVGT3A3P7gUXpPC1xfcPQ6tpis2pTtzEbg+YjBP0wayb5wKCV9IuF9KMXD88eq'
        'rcGztLhbeUoMghjgE9tjgfejPiJ4umcEae3gzJq3DOoKIpm6tExyuGHYgfqtWvTtXvtMN5oWt6UhtJADcO2wkDbZJ9Qf9qWrxTpu'
        'ia9dcP8AEqmfSr+45oZnJcRqxyoJO+N8+2DWwVgzdEeuaRxJaanwxrIluH4e1C4iFxCjktbBnDp/+Qen6d605injOYnDRljykdCK'
        'zS+4x1rQeKbjg3V1a80a7kSOwvEUB4I5chO2G5W2+1XjhyxutL0O106+uWubi3TkeU9XOTv+lF4dIxO1bH1hE9xcxwp+J2Cio+O+'
        'J9S0UR29jqQhigx/SjjAV8f5u5JphoDx2kc+oXDBfCQrHk9XP/GaoYhbibjB45cmytT4k/ozZ8qf3+lQ5c03njih/wBlGOC0c2W6'
        'MxXmmieNAPmFEp265GayL4g6aYLj5hRgA4NbWYwqhUAAGwAqlce6cJYHIXOQe1VTXBcX0zbhvUfl7lctjOK1nRtQSa2TzZyKwuVX'
        'trp4ycFTV+4N1QusaFum1Lg9RjVo0l0UjmYkg9cVzFyM3Kcgnp7n0p5eaDIkTvb3EVzGu55djt7Gq6/iGRQDjAJyRsKCKM24dgsy'
        'MqRqzZzGN/xd/wCKMsJfFjlkKlHhkUN/qwdiPt+4oOFXjnQlDiRsEemx/tio7vV49M8UiDx/GYksTyqm4HUd9vTtVeNV5ETd+Cz3'
        'UUF1pjP4g8SPzxZ2JXuPfqKVwfiAqJbmfV440TnitXRuTy8pyGBA9BkE4oxBF4wMeSqjBz1yBXZKrYzG2nqKuIW/Cp7sBRs19fWu'
        'iW1tptuZ7mRsBegAJ3JNKtdl8TUIogem5o7iXjey4b0mz0qVQLm4UBCM58xI69sda8z4v2cpv2y3LykUH4jXWs3/ABJBpH+MKILM'
        'iWZIhzc7jfkC9z9elJ7B+MuJeIVW55rbSItnV1wxA9AOp9zTnge20q71u9urOOa4uucma5lySCT0HpWi2em26xyOxLsRvvsTXr4V'
        'as83M6dFZ0SyXT5ylnapFAuXblO7k9frVZisYuIOPZ9Sn02SC2gwYjMQXlkHfA6AfyRV21O4tLCwub+4uooYItmYYwPb3PtVb4O0'
        '26i1+/1a/u5JDOiciHYL1OAO3b96olToSrRbLu4s9Ps05YVLybIip1Y+uOlVzSpNZ1rU5bqfkg023kKW8SggykHdmPp6Cm3EU0tz'
        'pZNi6IyIzLI/rggAH3NG6cLfT9Fja6mSOK3iHMznAGOpJpM4tsbGSSM0+JWp6pY67axGUWmnkEk5yZW7A+1ZfpN1Z6fr1zbTTJdz'
        'XZcyMTnwwB1P3xUH/qK4ln1TjSOC1d47e1UGLB6k/mpZwtwdqd5oM2stFIRz55juT1J/ilLHa6MeRp0iqcS6jc3WtuJ5DIsMpWP0'
        'AzX1f8BdVttR4MikJKXVoTFMh6sOx/76V8/Q8EXN/FzzxtCCjMrBdy4HQ/cfvX0J8MbOKz4Vs2ltVjvkhWG5ZfzEbgn1yCD96JJN'
        'r+A20n/QwG14x0x4763EEjD8LfiaLn2cex5CKh464M03WOHzZWcCpcoYjCx7BG8oz6YJH3phJZwHVJNTjzGwgEKYOPKCTgj6mqdY'
        '8V6lpWh6HNqyyMY7meC4dlwWjAyp39DgfatdIxWw74iRR6Tw9809p49zZRpynuVV1OR9CoP2NXqMmVvE/wA4DfqM1Tvi0l3qHCF/'
        '8m+JPDyhAzncZH3FW7htZjpdiL3Am8BPGz6hRmlSklbGwi6JeMrmHT9CgiIAkVTMx7jI/tioeBdPay0GOWUYuLsmeXPXzdB9hikX'
        'EVwdd4gtrFTlLmcBsdo13P7Cr6oCrgDAHQelef8ACvI5Zn7K830ioHh9KVa/bePZuMZIFMycUPOQysCOtXvpMYDxnZ/L33ihdid6'
        'j4TvfDuuXON9t6unHuleIJQo36is60yGSLU40bOC+KU10fF8PoOHUdRtG5YroyD0apllWSFZjHyu/MJAoyBSwPmTrUkcjISo2Dfz'
        'QYkoPgMlaOWvVgzbrI8QmVisjPuD3wDQM1nqBZxaGUxyjdicrnqGI7e9TXcBupw9wDN4bHwwduX22oq3TxVCBg8a9Q24wM5A37U5'
        'TvgOtdOLWw1EzJNczqsSBcBc7tkb+o79fWn1qUMzHJDMGU9gSD39Dj9RQscjyKjrC0ihfMMdR6H3xX64jVpPmIWLNyqGTm69s49c'
        'd/btvTYxTTQqcndiNWa44iZSCORuX6UXq+l27a0ms6i8bx4W2tozjyse496NW3he/wDnYwqsygHHRjjqP+9qX8W6X8/caNdOw8Gw'
        'vRPIpOA22Bn74qbHg/EtR0su/UKeKde0bgeL5C2iBuZP6ghQbkt0z96j404h1iw4YsbHTeVtX1NxCGH/AMWR5m+wqs8RGDVvjbfp'
        'NCJEtIUBY9AR0H13zVm1KO1XUbC+uHzIp8OFD6nv+lURk+iJRXBFq3DV/dz6JZNcSzadZFZJQejMoyS3qSf0q329nNcOpZnWJRkq'
        'pxzZ7VFbamraS8yLlnkKJ328Tl/imbXfLzBSM47U6MkhMk2QXj+Hi3IAjA/THYCqzxlFJq2nx6XLK0Nmd5pC+5PZcfvTS+uC0fju'
        'w8ucfSq/fS3F1ahJMIo8ox1z3OfXGa7azdaK5pnw/wBC1vic3Nzz3Xhp5ubYYAwBWoWVna6fp6adaW0awRDZPWk3B1utkviqSRIM'
        'En09f2o4Xh+flBPlxsKG6Cqw9bKxWEL8rEnL5gAo2NetHbxu3yqhXmIZ/TIHp9MClU+ock+WOVK11YXEk8ryZwoOAfWu2O1GlxGp'
        's3iY8pIIB79OtVT4iI54bEUcayvIyRhiAcKWHMfvTq9vgkbyybiNCcDqaQm/a9tFUqfKQmD+UgDP6GgnIOMQHgHiM65oxtSB81HM'
        '0QVhnKpIFzj6Gr/rt8llZ+Ep/qTZVR/NUn4V6NarrOo6hAmEhJgXbALluZz+wFGcUaik2tXDB18OzTlAz0PevP8AmTaxNL3wrwxu'
        'aGvANv8ANcQXuokZS2QQRn/Ud2/YD9avTbCkPAVibHhm2Mi4luMzyeuW3H7Yp4Tsao+PH8eNRF5pbTbI3IoeY9cVK/WoJjsacxYp'
        '1KziuPPIuaqV7o1pb3omCLgHNXiYExMFPWs+4vnuoslQQVNFFpeTOvwXNAAQTUoGcVAXw1TxHIFRJlFEpAUhiDjGCR1FQQxOJQIX'
        'C5JbJGRnvn60bGNgaFvI2ihZ4OUMzhuY9Bj8uPQ0RyonWYsqSIxjlyQy9Mr6Y9v9qYxuqLHI4C8gHLIN8dsN7ds0ihnWRnXlLON0'
        '3wef0B/739aNjvFaARFeUOxVt8EMAAc9gcZ+4qnFKxGSJNeM0E5lXlEZHnHoc9f4P3auNRt11TS5bPxSjMQMg9GVgf8AaiVMQ5Xu'
        'eQMeaCVhuDnHKfof2pRYyT2botyGVHzgtvnBx/GMeopuSOyFQepSdWiaP4vaiqYEc1qkjD/X3NT3FxbXHEVnC5y9sjsN9uYjbb6Z'
        'q4azocNzeHXbaEzXqQeFyL1dM5OPf+azTVVhtNTvNSlWYNnkjjC7kAEhh9ebGDUzkl5KFBy8F0s40hQBSPD/ACg9jnNevOJJ/DbK'
        'ltsUktL+4ewhuWiljQgHzKRsRXKXzSSJPg88Zz9RmmqSFONDOVA4MHUIMKDSPUkeOcRxZOWDEHse37Z/WmK3QEzTyN/TYbHPShHZ'
        'p75nx5c7fStsyhxorstsYicbbio7qDEyyeIAScE52ArmCdEblUgMdqFuneRkVeinI96HboVEt4Y2l5F3A6VLHP4FtjOAooaF0bck'
        'bHJNSuUeNcqMN29q5yOoDuL2VIpWERkPISq/5m7CksOoyXOpwadYRl5XlMWVGxbHNK30U7Z9aP1GO7vpP8N0rIkdSrSj/wCPOxbP'
        'qBmrTwpwtY6EfEhXnnMYiDnflQbkD6ncnuaU/sxkVQdaxWvDfDcroAqQRs5P+Zz/AHNUPTLdtQns7Rt5dQuAZWxvyA5b9s1Yvifd'
        'sLWw0mE+e5m8R/8A6L/yR+lfvhzZifXLm/5f6VnEIIj/AK23Y/oP3qPJ/wAmdR9Ipx/SDkaIuFQIowoGABXMhrwbY9K5YjrmriRo'
        '4c7dagk3OOtTNgA1D1JokYRso6Ug4p05J7Z2C5yN6shXbFRXEIljZCOoomrRl0JyPPR1rHkAmg8eY0xsyPDziokilvh3JhQdqg5g'
        'xKcueaupnHSpNLhEt0GxkJuaJtJWzErF0paC6imjzEYSVJTqfqD36fpXdrdIYXMr4KkAv3DHfJ/nJ67g1xreoR3Gq+AoHkXPKo3f'
        'fff1pVMrxSNcQEiOUFQxPQjfp6jr+tFgyqcdkDkjXGNdT1C9iHy0VtCzeFluYkKVI6DA3U746YyOlVTXNR4rnZ5rb5SW0Qc0LRq3'
        'M2MgkgjY+oq1W0kxe3SXlE/L06jfGSp7j1XtRBgtmD+HyolwM82cYbOCPbPX2INXVuvJMno/Bn/DXG/EMmsrpGoWYW1SFpZ25N+V'
        'VJIBHTJwPauPhQkXFXHt1Pc2JfT445JBzElc5AXfv3qPieV0nubSCHwby7TlJzuq7DP/AOiOvSjfhBcS6NqNvpkEKia+kXxWK42U'
        'nmB7dCT7YqOWPtln5ajXhs2LXtHt9R0v5XkQMAAjY3AHas41vh+SwkYhdux9a1dGJ3NZ38ReNrXRtTa0urFbqAeUFdmVu596PaMe'
        'MRUpdRTFtrhHljkbKucgHpUjN4SjBOV2z61HqnFPC6lZItURlJOFWNsjbO4xSWz4p0+91COKFJDbk8rOy4z7gVzkckP7FSJGkkbL'
        'E13cTBpfLjcdu1M30aQIksHNJG68wwKkstAu5386+CgO7N1+wrLb8G0hBdXC2kIkkZEjTdmboKeabo0+o2sdzJdAQTKGDRMCWU+h'
        '6CrB/genHTJbCWBZYphiTm6tVf4atrjhq7fR55S9g7c1s5/Lnt/ehl9X9g4rZc8lgsdPtNPgEVpCsa9z3P1NFw/izXQGcilvFV22'
        'mcO3dwjcszL4cJxvztsP06/atnJRTb8IxK3RQuIdSW/4jv8AUiwa3tB4EO+xx1P65rReAtPNhwzbLKMTTDx5f/s2+PsMCs14d0db'
        'm/0vQlBKc3jXJz+Rdzn69PvW0oAFxjp2qP4kbvI/Y/M6SijwjNQy7UQBv9qHnHrVpODSORsD1rqM7b71G+71MgwKNIBs6A5jUyIM'
        'V4oAHSulO/WmIArR/FR0J5YfSg0GT0op9ovapaKLInbLbUXNerpumNkYkkGd6i0zwDdhp1LIm5HrUHE11aay4t7fyMDjHTFJzQU4'
        'OKfQ4OnbXBVw/Cbi4lvpfznC/SiNUWUMAzExqSUXGRkjH69vvR1nCttCsSjYDFdSqH2IB9M03Fj0ikDOWzsSwy8wgl8Pn5ACwz0X'
        'BBx9j0ppdXFpa2Uk10GWFsK3k8sg68wP0x75pfNGLd5IwW6cwI6/X3NIuJjdajEuny3TrHImMFjyqu2So6ebGPsaoxyrgqUbdh0N'
        'outXlxrBjDSsDyxEYDQjpg9tunoevWvbW0uNN4ks9TUIsVupK9Dzs45cn0GNj7nPSmfCixC0it5ZUhu4MhT2z0zj/K238dQKdXFj'
        'DcQxuixBnLZifGx/MmehUgkj/nZyitRMpNyssun3cNxardIT4eMnPUY6g+9fNPxW1hNQ4lumjbMaucVtAnu9P0+7htVZzJbuUR9j'
        'tt19QdvoR9a+ZNeuZGupjICH5iGVuoPpUuTG1NfofjmtX+xfcSeJIApJJ2q1cHwc17bRDcmQVU4wUAb8zfh9qvXw5g+Z1y2QDIDA'
        'mgyuosPGrkfQFsnh20aAYAUDFSjpvXjgISoOQNq/CqI8ihUvLPcUHqllHeWxQgc43RvQ0WK/MO1dJJqmdFtO0K7DU7YyLaXUqxXI'
        '8oVj1x79zVe49uhdavBYKw8G0Txpd/zHoP0/mlXENxGeOHEiKzQSJzKRnKHG+Om2cnptQ/EcfiST/wCErmS/ljRU5wShfYDHYV53'
        'yozcNI/+RbFRbU0Wb4U2JmW+16Vd7l/Bg26Rr1P3b+KvwXbehNCsItL0q10+Ef07eJYwfXA3P3O9HMMCqoQ1ikTTltJshJxQ84qd'
        '1JOTUMx3xTEAwdE81SqPauRXoO9GgWdk14G3rxiK4JxRIEWKnLLipp1Iiom8tmiuemBmu1hD7t+FdzS5rUZB2KZ5hY2bOw87DJpZ'
        'w5A9xdSX8oI5iQg9qk11mutRS1ibPMcHHYU/s7ZYIFRVwAMCocCWTI5opyy1ioojdK5CcpyaMZQB0oabarGiZAt9CJYy6LmVAWUj'
        'qNqrGo2srzgxyFhMOVsgKykdAM9gO3r9atDPgk5OaV6pbI/9YKCARzj07A0PsMF0gutusV3A8vhhhyKBlsHGx6qfUewqwWEqOgLz'
        'h4c8zKAQVTP4t/T/AC+xxjpVZupJ7a8eQcnYc5Gd8dcfbFM7OaZZYpWKeICcZbZy35NvsR96dCYucP0WS5tgsngmVcRuOR+pTIyD'
        '9MjB7EGsf+MPA01x4nEGmWpM0f8A5cCD8a9BKvr03+la3ps8VxBHJHzM8D5Ab8QGcFP5H1AosxiRnJZSkeAFOAUz7d1bY/r6YpzS'
        'mqE24Oz5Ceyu4JMXCMpYZGe1at8F9AnfOqOh5FcBc+3etSk4P0C8NwLqwjMjtueXBXHp6UytNPtdMsVtLSMRxINgKjeKT+sipZYr'
        '7RB85Y17neo3blcivQSTTxJKK8frXo6DpXuMsM1xxn3xHtjbXk2oR23iNLb+GzA4KjG5/gYpbYX88NlwzxDA4lyBDNbooy2D5duu'
        '2+9S/FHUbiNb0TPyw8hES53GBSP4KC7sNc0WKe7SW2uFkdOXfDHmGDsO/SpFPaT/AIVxTUe/o35BntXpG9dLsK5JxTyYjcACg5Nz'
        'mi5jlaFK771qOIm2r8DXTjyk+lQM2KIxkjNtXDNtULy+9cGXIrbMNG1XQ7aSEMxAfGM1UeMYIrDSz8vIA2N96d6jqE0i8sUmfTeq'
        'ZriXeq3cdrISiA5bHoKbljGSaaF4m4uxbwlYTPI99deZ3/D7LVo5QPSu7a2EFuqAdBXDntU2PFHFHWI1ycnbIZhig7nYGjJelL7t'
        '8kj2opGoDkYDNCzMDkHcEbg96ln+tBu2Gx0pTQw9uoYJMOyZyMbdfb96WLOola0BBMYxynOeQ9j9MbEf7U1HKUw4GP4+lC30VidT'
        'UIkqKFVVdiMMxxk59Cd8djXeOnVfBhp11LCk7SReKCpOcHmGO4/c49QfWn2l3SXIjdSvjY5V5k2cHfGff/oqt6ObqYSwwPk2+TKr'
        'Z3UYH69P+4omwmdnCtFJbvHJzRADOO/L9R+4p0ZP0Lkl7LWJo5HGzLMqqFLDAkGw399wPY4r2QM6MdtuozuO24qITRywxpzq0jTY'
        '5lAPK+x6Dbm3GfUZouW3jk1Zb0GSK4jXllEe3iR9iQeoG49f0pzWwhfURXAxJ1r8me1H63amGcsAPDY5Uig16UDVBrp2ldufCgkn'
        'bYIuRn17V4pBoDiy5FvpqQg7yAu30Gw/3qf5OT8eNyGQjtJIo/yh4h4xtLBjmCJvmbggflU5A+5wKv0nDmmTcRWmuPGwubWMxxIu'
        'yD0JHqMnFIfhlZf0L3V5F/qXc3Ih/wBCbfuSf0q7r60r40NcSv2NyyuXCQttXBO+K5Y4NcMaeKOnOTtXjKMV5Ga6zRIxnVtaGcn0'
        'rzUNKaOLnQ5HUg0Rb3It0yQMV3fatbyx8ikBvStM6yqzZB9CKhdjijLxeZmdSDQLbDeuCP/Z'
    ),
    'song_sparrow_05.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgIDAQEAAAAAAAAAAAAAAgMEBQABBgcI/8QAPBAAAgEDAwIFAQYDBgYD'
        'AAAAAQIDAAQRBRIhMUEGE1FhcSIHFCMygZEVQqEkM1JiscEIQ4LR4fAWU3L/xAAZAQADAQEBAAAAAAAAAAAAAAAAAQIDBAX/xAAh'
        'EQEBAQACAwEAAwEBAAAAAAAAARECIQMSMUEEIjJRYf/aAAwDAQACEQMRAD8A42VeaSwqZKlIK1txrm5QkDnpWxTNtaArXWeCQ801'
        'OlJHUU1OlTTkNU4NGrUrvWA1nWmHbuaxjmgHI4rYOeKmqjRzWd6L5rKWjCyKBlpxFaIo0sJ2ZoGTrTwOcURXiq0sRdtEq807bWAU'
        'xG4xjFSF6UlB0p8fSotVxnbeDihI5p3bigYc1na0wll4pUq1JPSlsuaNPETGK0VBpzJQ7cGl7F6k7awrTClGEo9qMSHXNIaPPSph'
        'AoNvrWvHl2m8UQxmgZKm7fahKA9q0nJneKHjBohT2iFAYyKV5aMDnmt960Rg81rNTaqQwdq3ihU0WanTwQ5rYFbReMmmBaVqgBeK'
        '0VpwU1vZS9hiMF5owvFG0eOQKEdaqUsCyUBAp4oHFGlhY44psZ5peKOMc1Fq5Dw3atMMmsVeKPFZ6vCiKHHJpu2sIo05CWXillea'
        'kEUBFTp+pG00W2mheK2UparBZrWOKDdWbq1lYj71lBuFb3VpKmweB0oGXrRqQcVjUryE4klQRzQNGKeQK1xU+x+pGyjRCTzTkXnm'
        'nIme1L2GAjTpR7afHHmi8nFMSEpHkCj8vj3qQicVtk4paaC64pDLg5qZKKQw60SjCO1Ya2RitUXkcCRiij60Jo1IFTachy9KLvSg'
        '3FEDWdrTB9q0RW16VsjmjTwphxQ45ppFaxmlp4BRit1utgc0rVSIZNBuPrROKW1beznsbMmKxZKU4oDxVSknK9MD5HNQYmPFSo2B'
        'FFBvetquTWIuSMU1VqbTxtE6U9FrUSjvUhVFLRjI06U9EycAUMa81KthiQHGaVok0posYyMUp0xUy4bL9MUgjrR7L9UGVKjMnJqz'
        'Zfao0kfXijaPVAZeRS3GKlSR+gpLoR2pbS9UYnmiwf3rbL7UE0WoxxG700JdxlWE8MbeYRj+baCSpHTp80b3i5x3qGLk05RxUDWf'
        'GrxXkdjL4UhmiWQwyz2isjRtgZY4yOnPpz2q3KRGCOeGQPHIgeNhyGU++eoPBp8uFzRL3lLUYrZGKLArG5rJcmAbpQmiPFC3WgAP'
        'WjHShNGopU4ibc9aWygdBUgjgUDAVprPEdlpTLjtUvApbLV8aiwqJTnkVKjQYoEj6cVIiXB6UWiQca04LxmsjXpTVGKnV+ooAWIA'
        'GalwwluNyg+5qErEKVBxmi8tSAd77vUGp0YsfKSNwkjjcfSjh4cgGoURCsJCN7DpmpOnXTPe+W0Aye5pWnOM0c3Lmhx60+84mOVA'
        'PtSc05ehQMOKS6nmnGhwKNGIjpzS5E61LkXFJYUtLEJo+a9APhW1Gl6XfW9hFJ5tmkt1I67nO7IyM/8AjGOtUPh7RF1KQNM0qxeY'
        'sYEUZZmZjgDjofTPz0r1a48S6dp/3fSY7eOGeAolsZyCsqJwxHOWwAcAcn98dX8aSXeTHy+3zi+dD4jbUNV1LQUMQ8Q2LvPZS20X'
        '4WooBkpIgHD7RtPQ5HqKt9I1DRtQ0q5NhHcQyCTMtvJ/yJcZOAcHaeR06j5rqNe07QdZv59b0yH+C6tLdyDTrmGTa0zcApIyArk7'
        'WAB54HOciqi6traPVzaR366rJv2mX6VcP1ZJAQMOOSB3AyM1XPhZv/D42XO+1b/LW15OKOSJ4pGikUqynBBHINYBg5ridIXXilOO'
        'akMOKTIOaWgsUxetCBRdDTNGJoSKKtEUJrQWtFPQUaimRgEU5UWNRpwKcq1uAYcbgCPepdykZAeMYzTtOQlBzRHGKxeFGaJRk1Nq'
        'oVjkUY4Ga03BxWuaCsSLYeY4UkCnwKRLuB5XvUaKJ3UsnbrT4JvKQnHU0WjjBzXIecCTKn371jZQ4JBHah8pJly5HPIqPFN5chjn'
        'zx0o06kFq0G5oZGBwV6UGTnNAOfBFKC5cDOOetb3/FaLDrQeOu0xIrLQ7vVJ5uHIhSOG4HlhNmAGGQGdiVz8KOgrx37UfEeqan49'
        'tF0S6kmutLiEktsiALAoAXAJHJPPPavTbTULKK00yGRC0MR8hUVeBKUB3n9Mj5NVU+j+HB4hfXY7ZLe9ZjG5Qn8Ud8/0/avQ4+Oc'
        'pLHFz8nrah6VeWxglj5itrqMySxSqTiVifqyO+ev+1Vet36tHMPEOiPdXVpAgza3LKbuAEHzYx1MiZOVJPBNXN7cWUqrEVW2ZUYA'
        'x8YPQGqOBtZguo5I5rS6jjctbmc/UyiP6k3DuSCQfQ10+snFyznbeltDNaXen2t3ZXzX1vJGNkzAgsvYN/mA4PuK3SdJZP4dcIyr'
        'GpuWlt1QDGxwDgkcE5PWpC15Hk65V6vC7GiOKBlpprNue1ZtIjFazHNOK9a1t9qasQeBWGsNYATUyoxsLk06OPAOK1GMkU9Vqonl'
        'GkQ8ZpjkmML0xRRglgAMmjKlThhimULfotajbkGmzgBcKOnNRgfTmjkqUyYASGtAcUUoyFPtWlqTMjZgpAJANPhQPCR3BpNuVWQF'
        'hkU/zPxCQMKe1OiQK8AA9qB40dxu9etboi4C4280BlzLb4EcfDAUgNQXSZKOg+o9TR7HC7iOD3ogxmaEmsJ6Vo0BM01ly0T42Fgw'
        '47j/ANFVt+ZbGdZAm4KWZfRsnOaaxfGUODVbq+qedZSW5jdZIW3gEYIHcf8AvtXX4fLkyuTzePbqu1h57icuwCITkbepHc0KTRS2'
        '6IpfHnKY8H8pCn/bP71XrqLJMQBvYpgZ5xVTqWqxabZvIzEBcu2fQdv1OKu+aYxnh7dZ4WzEbq284SLEY9uDnaCuVX5AwK6FORXC'
        'fY88tz4WuNQm3l7y/kkUscnYAqqP0wa7iM8muPn/AKeh45/WG4ohigzxW81C4zHNYV5ohzR4zS1SpxkVirxTNvPSt9KSG04IqZAo'
        'cEk4xSba3edsLgAckk06GeKNJEVdzjvVamxouwOAeexolcso3tkist5bcSb50IPbFIL75n2DaM5AqsI52G0j1FRLfb9Sg5ANPkDx'
        'gEjg1Xw3UaXbR9GPanO4PlWQPAX0rWMVgI4P70RwalYlAwKduMUYkKbgDS4lLEKoJJOABUjVnt9NsUS/kdJnmMS2+36nIx+VunfB'
        'zjGO9I4S53HcBgGsjXe4BrSlWUFG3AVvleaNLGahH9OxDxioVtcNhrd85Xpmpbknk1Du4yrideo6+9OCzEmGCWfd5KFyg3EA8geu'
        'KyeB4ZZELK+z8xQ5HpUS4T7xaSIpP4kbKCD6g1zWjeKVXR9LimffNcXXkTnq25Mgj5P01HLlhya60jA5HXpSLiCKbBkTJXoe/wAV'
        'Z6y9yblILiA2whXakJGCinnn3PU1GtBEbqPz3CQ7wZGJwAo5P9KqUrPxx/iO2tNJvboTfebZoIxKhWISAKcY3cjGSQBVZqGh6H4v'
        'ggsIp/IupoInlkgUSNGwJ3Mylhx3IGTjp0q6+0/VmtLefWFhWaGWRVALEeX+ICG9xjIweOhryLTtbEWtq9rHI/8AbA7AHAdVP0rk'
        'fJzVTvtHOSdPcdI0VfD+k2ulR5aOBNqybcCQ5yWHyTU2M0rzp5Ui+8bhIsagqX3BT3C+2aJTzU/+tJ8PBrA3NADxWUqqHIacDUdD'
        'Tg1I0FxWuMUcvC0sdKlI1Z1/KxAPpQjCDAHzRL6UMnenoxtzuX3HShaQAqSMZ4oNxBFalJkTG0EjkVrxrO9JLPhRklh6Gq25WFby'
        'K4wQxODU+3WWaAyBMhevtUPVCFtHkwCVGRVcfqb8ToJVkU7TRxyAkoeoqv0Xe1gk7A7X71MCu0wZFJ9cVF6q+PcXGm2U97bTrb29'
        '2XUAi4gUt5B7EgA8f9q4PwjFq+sazbX+qbblLCW8illdyxL+Z9AwecAMcf8A5+K7fTtSvbFXjtbhoQ/5ioGemOvUcVyng3UWTVdY'
        'so5oY5Y5cyRhCQpzgtz3JNT7ZVZs10dv5cU0saA7QcgVO+8WJs8S2TRyx8l1kOHGecjsQMkEenNJW1nh1iaxkeGaRVDGRGGwggEH'
        'PAAwR6VVeKfMXQdTWJ/MkFu5XYcnIGePXpTu/hTKvtVTTrZzb2q3MrgD8SUhQPhQOf3qrmUupHY1XeAryC88BWVzduguDaqYnDnJ'
        'KggowxjPHzXQ3FxYtodtbwbTcoQ0+I8lmIGQWPQDOMCluXsRSWaT7pVEUjJHyzKpIX5PauH8cae+i+JtPu7SF7X75uvNykjLk7N4'
        '9CTg/IzXoVzeR2luZliaGRUYNJE5+tcHIYHI/auR1K5sfFN5Zx294rJD+DbXE34X92qMd+44CnDAd8jPfFTeX4eda6qNVRAqgge/'
        'U/PvSNUi8/TbmDzFjEsZQsV3YB9PerOEWlzYQXXmLFBswrom5pmJJyORwBjmoscck7FIYpJWwSFRMsQOelVKHB6hcPqnhDWtGkuI'
        'dg2hZOuzZhipPbB4HrkV5Pb3c1u0IQeW8JJjYDkt/i/7V6umlmx0fVpLZzOdULRpkFdp5MoIPIKgHNeZ6zaslzvKkKzcEjg1p451'
        'WXk5bZj3PRL1NQ0WxvUP99Apb2YDDD9wamg81w/2SXzTaLc6c55tpPNUf5W4P9cfvXa5xU1rxuw8GjApCtxTlNQqGLx1pimlCjWk'
        'ZT7ZVynGe1L8th2ptvFtZ4j2OR8U0xkcUko2MGgcVKePIyKTtzxQCCKAkoc96k7SWAAyaXKnOMYrTjUcgwXMsKOikhH60EsavDtk'
        '5VqaoZbVm2Ej1oJShVSh5xyK0+9on3FZay3tmxs0Ia3DZCntV5YsztlEdxjkKM8VR37mG6glXjJwa6C78VWMEyoQbO+1BShSGJVg'
        'YoN27r9J4zgDHBqOd62q47vRcJaWUiNc88DPWuZNjfaZ9o1xeJCIrTU4vKMxTcBLtBwfRvpbGavo9XuNQur5oiBayXIDFQpZmABO'
        'WHOM9ulc34vvbux8XSwXVy7R30MMtq8j5CSxZAHsPX5NY3l1sa5v1P0TxLLd+Kb7Q3tlX7uwM1wSd0jKEXaB029D65NdPcFBOH4Z'
        'SMY9a4nwBZR639o2rzw3CQo0JmctyV4Q4x+hHyK6+7KLI8SnLwkFyOgz0/1FaT/iJHN60n8N8L3tnAvlm1l3Q4GPpJ3qf9R+lWXg'
        '4yN4bs3lO6R1Lu3qSxJNRfHbXL6cslqiTLOgtpOcFSxARvgEn96udKtBY6db2YYt5MYTd6kdTUZ2qCvIVmtnjcZVlwa8X066/hcO'
        'tWMijzoPOVAeuSNo/wBa9uPIxXnes6RN/wDN0vLCCH7xcbVVpUDICoyWZTwcbQcHOarr9K7+O18DCO/8MWxu7horhbJWt14IcqOV'
        '9uMmpv3Atp5v5JUjVWIiUsd8jY52gemep4qF4l1cWOkG/v1aVrMCRZIlVHbb1GBgcjcKK61fTLu0bU9PneTTVj3wlx9SpgHDD/Fk'
        '8+9TL+Q8/a5zTfENrcaxNaT2UkmoWlnPc3Jmb8LzMDI24/wqCT3JqkvNce80z71feXc3iTO2GcNHLCUYspQcJtCnGMcHvgVP+z7w'
        '5d3MuueKNSAj+9YhjhLAs0Uh2s2e2PpGD61TeM/D7+GNWtls72KWxedzNOY+FxGd25SMYIJx/wCa1kxFk+rXwfc2V14sspoY3t3v'
        'bMQvAyHbkp9BU8cY2tk9zXWnPIPUcVC8DTWsOhWl82mRpcEiW33gNtTACA5GRhQOB171N7mpt1Ugl6U1WpI6UwVKoeppiHmkA8Ua'
        'n3oNIuV2Ms/YcH4qV5YaMMCDmgeMOpQ9DQ2jIhNsScp0z6U5E1toz2pDIO4wamqys5VSCR1pV1G0Z56e1GFqMYtqiQON1Q2JZmY1'
        'OdQEznioxQfVtNOFSoZ2iBBGVPUVGkG5yyg805xg80AJVuO/rVSpsQdUVfugd/5Tn4qj8WXMEum2l1bt/abSXc+DnKNgcehHf1z2'
        'xz1b20cjJFKwKynGPeqbVNGjSxu7IR5YjHuQe1VePtxKcsqT4L0nVv4REXtJRd3QN0I1XloxhdwA6jgdPWm/aH4Yv9f8IJqltFtl'
        'spFZGbjeTwVUd+n9Md6geGvGk2k6Np1nJtxYXKRebKx3wxnrjnoSMHNXlp42vtW1AeXdL+BO7GLy1aMAKFVQDwByTxisJ5OPtjT1'
        '5Zri/DR1G6u7Sx0xEtNYOpQTAS/QcmN0cE4yVOOfmvQZJY7gSXcFrmNpHYnGQuT0z+g/aqfxZqllabPEc0Rt7+wcNCbaIBXVztZW'
        'HYDO4fHvVlod0p8NWtpHPLJEqh8k/nZuSx9eT/SnMnIfYC6hS80+WFV2h0IHsex/fFOsZhcWyyg5JH1ezdx+9bt7gxT+VsGDyKG2'
        'VY7u5jXgM/mgfI5/qKdmCU8jpVQYseIImP8AKJCP1Aq6UdKg6gI4ryKUq28ghSOnUZzSUgeM7RrvwpqcKJuc27Mo9wM/7V534S1d'
        'n8CWGkw2815Pc3skfkRLlmiUhj8AkqufevZbESNceXEygyIyNuYBdpBzkn964j7PdGsUn1TXIrJGS4mJSNThVizwRjkZP1Y47VUz'
        '9TyrevzalpPg64nuXK3+xjNgcKzHoPZTtx8VXGKTxXf6Gb+5kubWy02Ka5SU7hIzDCKT36EnPYCrbx3fR6vp9/p9wyJdNCIA5ztA'
        'GGEh9cIuD71B+z6+stHjstMaGW6uI4I725MoGGd1wkQX0GEwD19qUsm0r+OlwAoCgADgAcACsxRSuZG80lN0mWYIANpzyMDgfFAT'
        'Rihr0oxS0NMPWkcEP6UVLFMoNYJOrPs5DYzgigu0IKzp1Xr7itzzRyXTvGAF6D4og+Rg0SpR4iBM00RyrD+tNaZzHyc/NR2U27vj'
        '+7fn4NNI+gYPOKfIoW7goRQwgmMsTj0oZt0Yyynb2ND5hKAdKNwWGmOFo98swSo8se08EMvqKGXa67WAIrIiFTgZx2p6MRruBmiW'
        'RHKsh4P+lV8OoNJrDLOcbwB+tXTguhA6EVz91ZH+IRyMcLu5PvVcOWck2bxQvHuhQ3Og313ZwlLrCvJ5efxVU85HqBk/pVT9mlzH'
        'Dr90txZyOrosnMhXG5QTj1+a9EKGFkwcjAINRE0iztLq4uILcI1wq4YdiDnb7Z7ft6VP6qfDNf06HVtGfQ0mt4ru/ZjamUgFjGjO'
        'Fz2yQOPaqr7L3uE0X+E6nBLa3lmSGSUYZkJO1vcdsiq2a8Y/aFpcLt+GY2CqecsQR+n5a7G9tpJNlzaqDeQ/3Y/+xT1jPsf6HBpc'
        'bc7OxLdIVlDv24BqPcHyriCX+Ut5bfB6f1A/epc0azaXbXAfyvOlKPHMhR4/pyCR+hB+KiXUYNrcJHvdlB2ZHLEcjjtyKqzUSpo5'
        '6VXa4+FRAPqALZ/XFTLOdZ7aOZekihh+tR9YjzAXK87QAf1NTVyofinV3i8Ox2c1wIklUpvWMFoo8ZkcYweFz+pFB4f1fR9O1+Tw'
        '/cTRW9i0v3e2lhjLLcKqo27jnnPU9M4qILd9Siu5WRHR0NhBu6BD/eP+p4/6RXKfZ/Zz22tWa6jbtL9yE0UMcj4JYkFnbvjgAevH'
        'pSt0sF9oc0UnjzUIop4fuzBEBgUomGwpwD/l3VM8E2zXeoXetzJt80/QvoemP+kACm61o1tqvjeeeWICxigSSWFWO0yEEKuevqx/'
        'Sr+yijhiEUUaxxoNqqowAKr8E+nHjpQGmdqWfzUtUOL3pp4NLSnYzSMINGDQYo160Fobe4jM6wcliuc44qemcEjtUSzVVUy7BukP'
        '9KmxgsMetTx/9FwN4oeEIGyT+1KhZxL5cnDD8p9addeWlt5TOSxOeOooJbcSQKFY7hyD71p+I/RajG5t8l8r6VW8hBUh5nizFfA8'
        '9Dng0FqYniZWBBB4+Kfr0W9lJyaai80DI0bcqQfSijfPWpxWnLGdwIOKi6pZGaJhCCzj6sDrVhbnJFUOualLYaiHjOAQQafGzZ7F'
        'd/Hpvg/wVB4jjjit707o4xvIxwap/FXh690HUJbK8XdE6fS44zj/AENb/wCHnxIllq81s74+8Pu5PQ16r9q1va6noxGAbkjMeBzm'
        'u2ePx8+O8XJPJz4c85fHy7cQTz/aBpMvlYEcchkO3g7ejD0zzxXcWrTi4U25kEq8qYz9Q+Kq5bOeHVFjKf2mE5UHgEnqD7HpUlHS'
        'ZVkVR1OAw5Q9CPkciuT5XZuxR+N9X1HSb7TbmGaQx3N55cu4lsblKOOemVP7810umeIdPv7K7a0ZprqGRbZ2nUKfXcBnJOMLk46f'
        'rXmnjLXZW1abwzqdnGIWm862kc5AcqTG4PXG7gjP8zVH+zXX43uoJLxWe6vNQbzFRVXftU7SRwABkZ+Kn/PabNdf4p1m48N2n3aC'
        '2Mt5NcItrGwO0iQ55Pp+b+lWXiO+kjsjHCrefMywQj0bGWb4Xk/pVN9pMUkSaB4juzmD+JiJ5i2emDz3GBVpaONUup9RXm2iRoLU'
        'g8Pnl5B8nCj2Bo5TD41L0VEj06KGPO2MbR8VzGgLjUNS1y5Qx20SttZjxgZLEfuavrV2XTZwoJYAhQPU8f71Fv4/vd7b6Iig2kKL'
        'Ldn1AP0R/wDURk+w96lSPYpNHpS3FyhW5vZDPKP8Ofyr+igCrCEHyhjvWasyqiMxwAawENApjboKqfCtwbcCl7vqrNxxz1pbdaVM'
        '9DxTQ1Rlb6aNGyaRpGc0cfWlIMinJ7UBH0eRnsYi2cjgg9qs4WwQc1CRkIDR42tyMU6N8UvqZMiRfJ+FuUduayEt5S5HGOtNBDqA'
        'SQMc0iKVfNaFWyPSr+wr9BfQieMBznFVVxOYisaDleQR1xV0WViVGeKgywKt2jBDuY49qrjc6Tymttci8ZV3AbRznrStpHNMtYra'
        'TxDIl9GVTYNrDvSromG7aNAZISfpPpRS4n20mGAb965LxhITfKtdL5irLsJAf0Ncv4n+q9U1z+af1b+H/Sd4MVtL8RW12HO2TBFf'
        'ROm6ol89vPIoaFF27j6185vNJBpkdzGMvGeBXsHg3UTP4HIZdsyjfXX/ABOdnFzfyuP9tc/9p1m1l4mjuYgQkpyGHzXMmQQXTSE/'
        'hTsN3+WToD+vQ++K7rx9cpqHhe0uxGXdHClh25rh5USWNo5FBVhhhU+Wf26X4e+LmPtM0CXV7CO/sI99/afyY5kTOcD3B6fJrlfC'
        'lvqTayutWkPlnT0DFGjwzOR9SbT3xn9cV6ZaySBTBKS0sXBJ/nU/lb/Y+4qLqAjtdVgvETal6FhkPpKuSh/UZX9BUTvpVis8Ualo'
        '+r2enWGrXd1JDe3jXKSxyL9KDk7l29MfTkHIyOvSutsJbRtNtYNO0u306xhi2QxRs7NjJ/MWJz8+9eIeJ75bzxgRGQLdLpIIgBgB'
        'Q43Y+T/qK9xRlXhQAo4AFG2TspJvSqnuo9Ot5pXDNhwFQdXPZR7klR+tO0yE2dti5Ie7nbzbhx3c9h7AYA+KgvOt3rsMgQG0tHYK'
        'x/5kx4Yj2UcfOfSrmba3PUjvU/q1brSyPGg2/Tv60cCbLdV9qVruoOrRQbOCaev92vwKus597A1AaY9KY1LRm7gUcTYNLPSsU80G'
        'nI300xG5qLGfppqGkT//2Q=='
    ),
    'song_sparrow_06.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAGwAAAgMBAQEAAAAAAAAAAAAAAwQBAgUGAAf/xAA7EAACAQMDAgQDBQYFBQEA'
        'AAABAgMABBEFEiExQRNRYXEGIjIUgZGhsRUjQsHR4VJicoLwByQzQ/FE/8QAGgEAAwEBAQEAAAAAAAAAAAAAAQIDAAQFBv/EACQR'
        'AAICAgICAgMBAQAAAAAAAAABAhEDIRIxBEETUSIyYRSR/9oADAMBAAIRAxEAPwD5RmqtU0N65mRIY0JjVmNCY0Egns1BNVJqM09m'
        'JNRXiaig2FE1IqtXUUrCWA4qcVeNGdgFUsT0A60dbYkuGkiQp9YZuV9xWTMot9C6irBajx7FWIF0TjqRGcU4bVzHvQ71xnoQfwNO'
        'nY0scl2hUCrqKkqRwRVgKIhIFXxXlHFXApWEqBVwK8oq1AxZaIBQ1ooFFAIrxqTUHimSFsqRVWFWr3FFmAkYIPrWfKDBf7hxhtwx'
        'WowpHVE+aNs9V/SuDzYXGwtWgpFDcUciqMtdTYEKSCgOackWlpUpVMNACa9moYEGvAUxiwqagCrqOawTyjNFjQkgAcmvItEQYINB'
        'mHobnRIdJuE/aarqRKqpVGIQc7hkDr0rmnu43cWb3Jji3Z8XZjOSeuT2H60rrlkbLUVvYgpiuMmVHAxu8x71ZnivBHHDOIxnDRyt'
        'uTPbBPOPSuhQi0XjNxWjsoJ/h3R4FhtoReTON0kjNnntgHtWbcfE9mZA7vDZo52hpM5Dd8BfLzriLxpYm2yZboE+Ynbg8e/es6UX'
        'knhvb24mCZQp4e/H4edN1oFt7Z9TW7W6kEdz4W4gGOWEFvEU/wARNQ0ZRyrdRXOQ3WoxSW7zuUndcyx5yRnGcnsc849a6a2lW4gz'
        'kb07eY/t/OpM0o6sgCvV41GaRskWHSrChg1YGsYKvWiZoKHmi5pkKyag17NTToBWoPSrYqCKxglpB9omEZYIoBZ2P8KgZJpDV/mU'
        'yAAIrAIO4HrTWaq6eKrRZ+sY+/t+dcvkYnkj2UhOk19kFKo64pxk4oMiUGxRNhQXTNNstUK1HnQTPki54oe00/IlAZKvCdgoCFq6'
        'ipIxXqpYC6ijQrlwKCtaulJFBBLqNyR4UP0rnBducD8qFWxox5Ohqz0SK9gmiuVEsiANsBBxnnGCRg1l6z8Nfs+2lmtYEDY3AFQT'
        '9w71t/CNzFOrqrOWnkOGc5DN9TZxzwPU1panc2zwMiuZmnm8JVb/AC8s3mRXaoqijtOj5Fqd0siGz1GPYVPyukfO7yPPUfdRbfSZ'
        '/CF3D4d1ERgmMFWT/UOPxGaB8Tqf2lOkyEOZCVJXBI7Ghafqd1bv/wBvIQwKtggYJHf0qEpU9F4RtbOg05pVkWyt4reNmHOAQ+fX'
        'dXQ2dmIVQOix7e+eTntXLSa3ZXtyk1x4yTY2tsIZffGOf1og1sKQHK7Q2EOSQaTlfY3A6GVCjEH7vWh4p60n/aMASWNVdh8rL2P9'
        '6SkVo3aNxhlOD71OVo5pwp6IqM1BJqAeaWxKCqaKp4oCmiKapFihatVAeKkH1qqYpaqmvE1BNGzEE0Mnmpc5oZNTbGWjUZaE6086'
        'UCRa4PkGoQkUUNhg01IvNAcUlhYu9LuKYkoD1WAoJqrVjVQCzADkmuhAGLKFp51jBCjqzHsB1NM65dy3drbWNpaLFFHnaehbjqc9'
        'eBTuhWl05mS0gj8WMbnmc5wM9FHc/wBDWva6O/iiWdV1DUXzlGk/dwDzPrVYxZ14o8Fs47QYljuY1uh4VvGXd51f6eDhV7ZNak2s'
        'aS9/FD4MstoHCxPIxMg9c0lq9lcLvt7m7hjVmLiFD0GcE4+6kLSJWie3lSR3Unwiy457ZNOptaGcL2G+Lbm1kuEa7shJKVCgrwyr'
        '5nturntU0aztrMXUU+A5wsbNyPf8a0Eu1bXVNy5ulGVcdVHXHvzXQappNprOhi8tY9kyA7UI8hg1nsH6nHWeg3V7YCWF2Vz9ABwG'
        '9M1aDQdQtgJ2t5NmcP8ANnFdH8Gzqu7S7oDLDIz247VW50DU/BuGtJCNpJKBjhh5e9KlrQXd7Jt7TwLUXFrcSRTBdxV5zg+orW0T'
        'VbO/LRajaoZsgGQtgt99cjY3c/gyW03HB5Zfpx3o1pdx+EjiVGZDtbng/wBK1oDj9nZTaVDNKxsrpdnXbICGX0pC6t1t32t4pb1X'
        'Aq+nO4bMUmJOuAcbx5+/61oamftECsGZGA2svbI8x2rTxxq4kHExQeauDVCu08kZqCa506JUF3V7fQSxr2abmYOG9a9mhA1cc4A5'
        'NFSMeY16CGW4lEcKFm6+w8zUNnkfjUieVIHhRyqOQXA/ix5+npQk36Mv6dCw4oEi0cmhSGvNsoKyLS0gpuSlZeKdAYpKKVk601NS'
        'slXgKCNOaNEJLsZj8TpgZxjnk/hmkyKL9pS0twW+uTdtwcY7V0w7DBWzTuPiR5L99N0i1K+IRCsicsqjgAVsW0R0XSrmKKceNuO5'
        't2Sn9TWBo95pvw9bxXao1xfzDCooyVJHH3/1pr4Y0/UrjV57nUod0ZO8qT0PlXRFNnVJpGNo2l/a7xr26c7/ABtzAnnaOefenvi2'
        '4gh0aR0Aid2GGHBHGT9/GK0dQVNK8W8kCtNcnYq8AAduOw618++JdTuLgtasQwVyVPatXFUDlyZofBFvHNeZyMKDgHzxWnbre2N/'
        '4sjssLyYZVHy/wDDXPfCcsvji3QHxGO4MD0Oa6rU7822q2cMuGtpJfmOOg/+/pWijSeyLywS4Mt/Cgjli+ZRjBbHPFe0rW7+DTJb'
        'ieHxlYFlKjkf1ra+NJIYNIYQyJGSgZD3JHOB+FYvwxE02lGOTbtuAcA9z3FNxadIVNNWxDTr22v5JfEtgwclsMORnr07VH7Ot13/'
        'AGdSN2cK4zSUVrcafeyW88RaNWwSOCB2YEd6WuNSureSRGZmKuSjEDkeuKVtew0/Rp6ZM0k5tJWNrNH/AOMnsfKu50O9h1C0+zXi'
        'COUDYXA6Hsw9K+bzajHeRpOQFlQDDjz8j6V0+gzC8sWuIW+dB86d+O486k/4N12H1S2a1umicYZTg8cH1pSmNQvmvEjMnMi8EnvS'
        'y1A5p1ejxqKtivYrCkLV1JByOtVAqwFFGNGG9aciO9VbhcYBcDcP93WljHbyTEQs+zPfGR6UfS4l8X7RKP3URzj/ABN2FKyRx/an'
        'mIAZj18qRri7RaMoKNzNotQySxwASfSn20ueJDJet9nQdQRlvwFY4u2YFoZ3MW4hdpxxnuPOuOcXjjykiLmkNG2bdtllihPcSNgj'
        '7utCNtbybv8Av4VC92Vufbilmd356d8nqajA781D/TXSEcw0Wk/a38O21CyL5xtkcx/qMUBvh/U2OVjhdM8Os6sD+BrxwPSql+mD'
        '7VWHmJdxBzZU6JL43grd2jS8/JvK8jtlgBn76wbu2mk1UpKrILUorKwxg8k5rdLHOSCc+lY/xDciCCaOM/vrg+IzHqAAABXX43kL'
        'LJqqLYZNyND4LBv/AIjubmRN8cA3RE/SrHgflXVPeukz28PO0E/L1b3rj/hm6bSPhpbuQ48bLNxk98flU/Atzc3uo3d9O52P8qqe'
        'mM8/yr1McktFJq9gHivtbuZbW7cR+EAy46nOcfpXL6vapDeJaIcyjAx3JJwK7nTLV7nV5buEMxnY8Z/hzx+VY+qaPMvxjBdsmIyQ'
        'Sw7YGM0ZQvYFP0aXwVoP7NlN5doC5QqvryMY++ntP04axrNxHexmOOEl4iO4zitXULn95brGAIt4VsdAoB/tTqSQhRKuDtyCR5d6'
        'dRXQrm+zmfisLPqVjpczqI2HOPTj8+KnX9PfQtA8e2mb9ywcZHGciqhI9Vu0vpco5BCDuCDmmfim5Z/h+e2nXpEQx6+1DW2a3pCd'
        'hcLrFn9sKIGA2OvXpSGnJp+oi6t3hVZd3I/wkeVJfAN2Fs5InbjftI70rqSXGla/LqNuGNvuBbHk396k5Jq2UqnSG77SbayvVPKx'
        'Ovzgcg+f5Zomisml6z9naQ+G5/dPmj3s0Wq6TvQ7ZAMg9CDWTZ2kt3aHxGJkjbCt5+VSnS6KRbfZ02qQLFMskYxHIMjHTPfHl7Us'
        'tWSadrJI5/8AyR/K/kfI/rV7SB7iYRR7d5+kE4yfIZ71zy0znmqdECrbasY3jcpIrIynBVhgirKpJwBkntRQhQCmbe3QqZJiVX+E'
        'Dq39B61aNFhbdIodh/Ceg96HNM0j+ZNK5ekBug11eM0KRYVY487VA/5mub1zW4rIMNheRTjbmtTUbuPT7aSRn/7llxGB/AfOvnl9'
        'NumZ3Yu7EnJOTmktX3syruR9Zvb+a6fa0rHttHH59aXjRYxx1qyKFXJ60J5MnCDJ9q8bJnnP9mSou7gDnFQu9vpGB5mqBWClzgEe'
        'fNeG/G5lLZ6jNR5/w1Fyg6u34VP7tBkRt94qyyKeQACPTmi20scdwrzQrcIMgozEA8eYp4yDoXEmfpA/GsTW1jl1aO2Ay5ift1Yg'
        'Vu3HhtKzxR+Gp6LnOPvrEngYfE8Fy5/dxoDk+fP9K7vBmvl4lcLqQGCV0+G5bG5AEsSlVUjnGOK98DrcXOhygfumVyA3Q8gc0vre'
        'W1OCUMQoZjNz1BPGfxFdLpUC2umtHCQYpXDRnzBFe9DbLTdIJ8Lyrpl/DYOxI2OqMfTPemfiVGPgywgErMu5f8QzzWegNzJDdRAB'
        'reQxyr5jp/etaSdZolKgNx27EVdPVEX3Z64liRIwBksaXW7K7otuFL7eRUShHnXbzxzSWqu0d4gUEKjA/wB6Vs3YtqyzJq+nrbFg'
        'iMzvj14/TNa92i3NmVwNsnn5YocrqAWJGdvf27UVDH9jXOchCB6cdaxrPnUEbWepyxoxC+LtP48V0mtxtHozSoisGwDnrjuKJfaV'
        'BNG9xF8rOQ7fcR/SjaoBc6ZFHFjAI3DuBUeNWW5J0YN47RaVbvCcYXa/HWlbfUHhuBFyATn8K1NUYw28McsY2sQGx5edUutPhdo3'
        '/wDYZBznHaptMomma9s/i6aGALK3c9R/zmvKuKBoasiXVsx4Vty+1Pyz3ENrwyFeirMu5T6D+1SyE8iuVAL3UbmS4jtTtlkMe4M4'
        'yQAcdfKmLcypH+8ZNxHJVcCvWNjPe3by29qSzAZCg4UDzPYc9/OuoFlpml6YWvrcXc8jEKCwGDjjGOo65oxx8o60hcudUor/AKcq'
        '0jSSbIxkjuegpa/v4LCIgOGl8/Km9UWb7HJJZxxo4JJjjB59R/SuBlee/nIBbr3rmk30jms9quoTXkxC5JJocVgsaiW4OSf4e5p6'
        'C2S3wka+LOTgADOK3dN0ZQRcXuJJOoj7L7+dNji7pGSs6F1I5QBj/m6ULbL12r7A0wYmGP3hwfSpKEdyB7V4KjFBbFv3h5MZqOep'
        'Qge1NiPn6xiqvlWxjjtTUKxcAGvbcdM0YhGPzDFQUI+lsjyNI4sAMkAcisb4pYx2HiqcPwAfX/hraPkVOay/iaHxNHmcf+vEmPaq'
        '+LLhmiymN1NGc1u0t8rT/LDcoIj/AKsZ4/A1uSW8sGlRWcbkGJlRJO3Byp/IVjtcrdia2OMxbJY+2Pat3RrwajpG1sbldlOfMV9R'
        'iaOmdjFwgjhaeAhZG5cdj2/tVrZ1+zFlA+Yb8eR7ik9OuAk72lx8wwdpPUjyqYibe5eAkspGVPmD3+6rpkQ0su24UDnA4/Cs3W1e'
        'S/R0YlNmSM98f2o0sgiv15yv0+mD0pe9OLhFGVBB2nypGwpFoXdtPIYHejbVJ8u35U/9pVLESt0KjI9MZrKsZm2sjDDKQMeZxV7g'
        'u8YiBG0DFBMzDxyyiN4pFwp5B9M9KXulKRCVW2sJOfbpRryZVbceBuxjzFYt1qBLiAYIP1c9DWk6NENe3kV2hhkTDJkdfTg0lPfx'
        '7rdt5yjYPrxSqSoHkct8zDg0jbrLdXSLCu7aePSpOfssqR2guIbWzWQ4M8n0xjqR5nyFL3t3cQWv2qdUlYMMR4xtzxx+VU0yxKyD'
        'c+6ViAXbotaKaQst0rXE7yeE25FVcIT2bnk1zz3tkZZVZ3Xw/e6Vo/wrbajD4p1SZSXhZxjJHcDsPLvXL35nvbnxppSHAwMdAPID'
        'sKLDZqo4D8cdKJ9jbJYAD/UaE8kppJaRyt2xSNfDZC4Z36Ar059KW1DQmuHaUKIJGGWZMNn3Getawt5lA2pkHnOa8sUy9QRnoKT0'
        'blRlWWl2lgeIZmkI+aRuf/lNiGM89P8AdTqlwMPuA6elewGHKjPtRUq0heTMx7+dkCqEQAk8Lz+de+2TNIGfDADG0jjFLgVdVrla'
        'X0dVBpZVdSV3IeMc5oxuVDKhAZNqgsBg5xzSyrVwtJHFBN6NxQz4lu+drFB5MM1YwgqpjdSx/hBpVUogXFU/zY5CuKJYHGGFBlhD'
        'IyMNyMCGHoaaSVgMEK477h1qNsZORux3Fc+Twprcdi8T59qqS2dyqhir7fCLeeDwf0rY025Nta+IVaNpHG9fJvP76Y+NdMFxAtxB'
        'nK9Tjkc8E+1ZElyJI4o5BhiNrH/nlxXq4ZNJcuzrT5RNm+d0mjul+bDYbFEa63yKqklwPkfP1DypC4lYRoM5wAGHmKEl1FHcKjfK'
        'Cec9j511cqJ0acrxyMGwMkkEffmq6mrFI3B5H5+v/PSlnkYu3fuCKam3XFquDyBj7+1C7F6Et3h3Xig98H2piKcGRgPpHOfWlo42'
        'kOG47YqkwaK38ROhyD7+VZB7E9b1EwuilgBkZJ/T9K56W+lErGKJpGJPStDVlaZGYjIOOO5PYUxoumyJGFYkH+NgeD6ULXseKSVs'
        'QtrW+vJsSL4StztBzgeprq9J0kRQhYxsyPqIrQ03To0VTjqM81qpFgVzznb0c2TNekKWljHbgN88j/4mPH4Vpw+LsIjRvuXFDQhe'
        'nerLI3YsPvqTuyFlykx52t95qwhm7ADv1qImOcHv60dmKEAsC3+WsYrFb3AP0jk5o4hfA8RlH+6lmnbnBP41Hit1zgfhWs1j0aqA'
        'dzKR5k1EiwA/UrZ8qTJ3DcxJHvUK46BaF2azGUURVrsfiPTvhXV5/tvwhqtoA/Mlk7ldh81J7HyPSuWkgkhkMcqFWBxg1PLilB7O'
        'xA1WiqmalEo6pUbGoCFqdtH2VBWqRmBoBtrwFEK17FXjMWgbqCCCAQe1ch8QaS1rMLiAEwM3zAdUPY12DUKRQylWGQeoNU5WGMqZ'
        'xU0jeIiSja+PlbzFI3UjCXDjgdK7iXT7WSMJJCrADAPcUodEsypDqW7Z6Gntsfkjm9OuXR8HJTJOOp6f2rdt5FbIQ8EfjWfo2kkf'
        'EF9IHPhIuxQTweR/StkacInO18A+XUU6bRpIHJDtIk/H+tL3bxLGwkGQ3Ix1JrYmKW0OJfmlI+RV/i9fakVtRuM1wBu7DstGUq6F'
        '0uzKgsGuJRPONqD6VFbmjWwlfMcY8NeBx1NJuslzkITHDnBbu/oK3tKtDBaokahABk85qVonkk6DNFs+U4yOwqdhYYziiGOXZuCE'
        'DzNC5zgtk+lDiczPFAo6VXkg7Rjzogy3HavNydqjAHn3pWAGikMMH3PnV2BUcVJ4XjrXlVicmkbMDG48Hg/jV8BV9qswAPSoOHHA'
        'pezAy3mcVYAefNRt6DvVlU5yaZGOQtLe1uX+zJF4LAfI8Zww++m4LXU7SNVTVbrKE7C2HUjyKnIpLTGmTUEVF+rhl7Ef1rqLuIkb'
        'c7QeuO9cORzxq4NnRZl3GuzWI3X1gzqT/wCW3Hy/gen41v2E8F5F4tu+4YBIIwQCMjjy9ay2XaMLnFVECM6urFJEGAUJBAPtSLOp'
        'fstjKTN7ZVSlKwalKse25iEjBh+8T5crjnI6Z6eXetBQskKzRncjdGx+XvT9bTsonYqy0NhTbJQnX0p4zC0LMKGRTDLQ2WuqErEY'
        'IihzMI4nkPRVLGjMtVZA4KsAQwwQa6Yoxh28klndNLJCpjmbBKdvWr3Fveza7DcWdyBahD4i5yrE+nn60reXE10zWFrL4F7at+7f'
        'I+Ydhjv5YFK6XrE9peeBqOIjI31oPkb1x2q0lrR0SS7R0YijgUuxye7Glbnay+NdHZCDlUB5f1PpUT6haeEbjxFmwflUH5fv/pXL'
        'a1q0tzK3zHFceWfH8Uc8td9jGq6w0sqpGxjjUgfL29q7zSHb7PGfCZRtG2PGNg7A+uK+b/Dthc32oRmEfMjBtxXIXHc5r6ZG+1Fi'
        'EjOVGGY9T6nFbCu2yMrYd5JJBsyFz1NB8PnAPyjqfOrBuOWAA64qdwcZzhf1qhJlSo6JXgqqvTmvEhe3PeoGW+Y9OwqcgE7CcnPt'
        'XmIHArxcAY70s5LHApaMXcswODg16JZIlO6QsT51VWHJ6c16NxIc8nHpWaMEycZOakNgEEH3rzHjihgs/fK/zopGOV0yQR3yF0yn'
        'Q5GQP6c966u4OcDzpe10uC3ihuo3kUsvKsOcEcqfMZ/Srs/BUge4rz/ImkqLlCgPTkVUIqk8cnvVwcLVWPHNcKYQBJjfB5Bpuxun'
        't5dwG5D9SE8H28jSz8jkcVVchiO/608Mji9BR1JhEsH2i3YSxEZyOq9uR1H6UqyVgSw3KSQXMU80IUnlGIBB6jI/50rcsb2CeIR3'
        'T+HKFO2XHytgdGx0PrXdFwk1x0yin9kMlCZaakUqxB616G3ed9qD3J6CumAOxIRM7BUUsT0AoGvGGxs/Dky08wwFU9B3++teeWKz'
        'QpBteU8F8dK5qa3MmqSaheh2ki+WBCfl2kDLepzn2rsjrbGjSZzNjp11NqMk3gkTEgrJnAUDvnsKYj1u3k1dreVYTPygm2YEnmQf'
        'OmvijVFi017eFsSSfUR5eVcRquqT3dvbWzpGBbklSgwST5+1H5IrXZS4pW2a8mmqbqQws0RZuR059qUt7G4lvfs7REylsBPP19u+'
        'aHHq91MiLcku6EESDlvv866XTbuxv7Qx3K7tw2kqcEjyzXHlaUv4SaUmNtHqFnoxt9CtGuXJxNNGMsxx/COu0ef/ANrc+HtN1ZtF'
        'tfHtJkmwysJPlb6iRkH3pG00rTnULDdzwgDgB61IIJrSF47K6fL9ZDJl/wA+lFZq6Q/GlVD0ujlYgb6/gtBnLKzfr/arCTR44oxa'
        '6nHdMvBZxtTNczcaPLcsTdahKST1YZP58UzY6bFaR4cm5OOsigAfcOKaM29sRw90PzuUumEjBl65A4PGa9JcoNrAEjuKDMxkYEjk'
        'AD8OKHso8hHjjZM90dyqo5cnAz0FGjdRF1GScUAxjOcDPnUEYj2DgZzRWxHi2Enmi2Bdwwc5NR9oijjxHkEcYIpV19KERzVFBGWN'
        'DclypkUBiMjLGmVlVUYFwCOgrMBPA8ulTW4Wb4kakkrkBGfIxwBQSAT7VMaq23LYA60QxHBZeleFmxzt60FxBH2ofLH0orZ7DioV'
        'eeDXLYDwjIAPnQ5QM7u47Ubkd8UGb5uMjmmWwl4ZGVso23PUEZBHqKWur6CObw2jKSddoPB9qNbxybwiqWLHGPWgalo0tzqCShPB'
        'RRhpG6nHYD+deh4WJZX+XR04OLvn0bGjXDXKiKXIVBnLH6R5cfpWXr3xmuj3o0+4iJjxlZYwA231Hf361p2NokNuIYl8OIck/wCI'
        '+Zr598VWcuq68854iXCLnoAK9OShCqGjwbfpHUWXxNZ3Um6CMsv+bqaQ+INdG1mlcKQMADt7Vz013b6VbiCDDS9OOxrICz30xklY'
        '7RycnpU5TciE5J/r0RfXk17KdoIUn76m006SR9qISe9aFjYvPMILSJpZD5Cur074bvIYhmVQT1RTyfSpp/SEiuTORlitbCPddMVO'
        'MgAZJ9qJplxaXHitEzJ4Y3OXG3AzjOaX/wCoMT2/xJcRMMARxED/AGLXPJK/hNGHYIxBK54JHQ/maovGjOFvtlvjSZ3NrqkKj91c'
        'wsT/AJxT9rqM0hxHh/VTn9K+cxtzhuK7H/pmLf8AbgM0yAsjIsOCTISM+wAx1P3UsvFUFaYyjfs6AapNDw7up9TRU1rAGHGfbH51'
        'urp9mOWt43bOcsM/hmhyaVpz/VZQ/cuP0qCcvsVxa6ZmJrSFRv2n3GauusWx/hiP3kf0pg6DpjE5hkX/AEyHj8aWf4YsWUhZ7lD7'
        'g/yplOYv5FjrFlk5iI/0yf2qp1fTD1Nwv+0N/MUrJ8KKT8moMB/miz+hpW/0GS1syVzPL1GzOMeZ8qpGcl2jRTk6NJtS0k//AKZg'
        'O+Yf71aK+0FmbxNRkQY4Bixz+JrnrPQNRuIBOzJEp6bicn7qK/w3dHAa6g9Tyf5VRZMjWogf4umNft3RVyJJL0OMjCwKw/HcP0qD'
        'r+jqOBevx/gVf5mkH+GZCebuL32mhTfD0qJlXEozjK8fkaDyZYraNuT0jsYzzTcL8EZ61mo9MRSUjjemYaeILznIPelwOoFHPjPA'
        'zDBjU4Jzzk+lCZXQgum0nrn+deT5OJwlpaEcSsgOyotbeW4lCRqWY07a2j3YCouwL9UhPGK0Q0dpGbezHP8AE56mqeN4ssm30FR+'
        'ylrBDpgLbhJcN1PZfaqSKX3T3D7I16k1E8kFogmuTknovcmua17WwQXmdUjXhY/KvWXDFGojt0hvWtaQxNFbkRRdCT9TVwWs6wXc'
        'w2vTkEil9U1G41GXYhwmfbNet7WOHlvmfyqTle5E5SbAW1mT++uSfMA9TW7o2jXOpsCB4NsOrkcH286c0rRGl23N8CEPSPuffyFd'
        'qJIJ41MEKQGNVV404HT6gO2fL+tLjlHJKrAkY+n6THpmpRPbs5hdCjAno2O/vitwcDiojADKxAOCDg0VEzzV1Hg3XRVHz3/qVod/'
        'f66L+0gknjmijQhFyVZVAPHlxmk9C+AbmYrLqbeBGRnYpyx/pX1JVIPBxXvCo/I0qRXkfMdW+ALqJvE0+UTx/wCFjhh/I1o/APw1'
        'eWWpyXl5H4Rhyqhh9WQQSK79YsjkVcR4FJLLKqMpULbKqUxTRTFQUzXKYUIqtMulCdaZMDBGq7lCvvUsSu1eeBVz1obVWO9CvQs4'
        '7dqC4plxQmFdkOiYq61EzjwVjRAuPqPcmjOKXkHWnq+w2f/Z'
    ),
    'song_sparrow_07.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgIDAQEAAAAAAAAAAAAABQYEBwECAwgA/8QAQhAAAQMDAwIFAwIEBAMF'
        'CQAAAQIDBAAFEQYSITFBBxMiUWEUcYEykRUjQqFSscHRFjNicoKS4fAIJCU0Q0RTsvH/xAAbAQACAwEBAQAAAAAAAAAAAAADBAEC'
        'BQAGB//EACkRAAICAgICAgICAgMBAAAAAAECAAMRIQQSMUEFEyJRMkIUYSNxgTP/2gAMAwEAAhEDEQA/ABO+s7/euJVXNS8VgYis'
        '7qczmtS5Ucr5rVS67rOkreKw9HD8RZweBUQLJUAOpozciiDZFeYcKKeKNQhLSMGV3YbSJWqFOuHCUHAHvVrqtjaYiEtp5Ce1VXFk'
        'SIkxUtCSW+vHarH0XqSJJZCX3ATwME06akvGPcswzIz6FNLKVAitQfanK4WpueyXWAk8ZGKUZ0V2G8UOJIANZnI4r0nfiDmuaxmt'
        'Arits0tidMnntXCVHafQUrSCPmu3FYNSDOidetPltSnovHfbS8txxtXlrSRjsas5xIUCCKXrjY2pDql7SVHpT1HKK6aXDRQSyX+A'
        'OTUN+C809uTx9qNyWHYL2woJx3AqfZkNzFKCwMitIYsEL3xIGm5qYtwa/iDX1Ec8KQo01X+x298xp1oVhCVhSm0ngH7ULlWjdJTt'
        'AAHNM1sTHhxQhxYA7jNcvHGMSTaeuBGTT90aUwGpCsLQOQTyftTno9D8l9TobPlE8E+1VaidASou8FST1zTdafFWy2yCmOSPOA2l'
        'IGcmvL/J/EdEZ6Vyxlahk7llyoYdwnbio021vx2A41wociomjrpN1ApMtTC2mOqQe9NVxfbZaw5tAHvXi34NvH/Ow7/UcrdQ2RAt'
        'qmy3Gw28kg9KYIzLa2wHMKzVe6j1nAt6yhsjfnomtdOasnTZSAptSWyeprjweden2AdR/uEa2vtG7U+l7fcYLgW2nGDxivOmo9OR'
        'bbe3Wm+U5zg9q9RsqEmFgnqOwpQmaFts2eqTIRvUpXJNeh+GvsoQ/c+RA3VZ/iJUMq0yEZ2DcPtQ1+NIaVhTSh+K6xtWOtkB9JP3'
        'FSoepYMl/a6ACT3r1grUxTAPiB1bs4wc1o5uT1BGasuz2y1XFvzgG1YSVHPYDqaFXnTrE1SDBdDhcVhtDQ3KV9hVnp6jOZeuiy04'
        'QRUscdUqcn2Tyai68fe3oioUeevNW7YvDC7wtKuXR36aDLWSUtzn9mED+o4Bx9jS5N8MZEiOi53XUdrbCmvN/lFZShJ6ZJTj80RV'
        'Kp4hBxXHsStoMdP0flrwcily8R5dpkfUxVrCCc4FWW5pK7bSbc1/EkIUEFUYbsH7dcfNALpDUFuRJbe1xB2qSecGlgXrOYuyNWdx'
        'j8NNbNOMNxpXpVjBJpwv0aNNjF9GMdc1S9ts70e6MrjKykq5FWhcn1xrEdxOQnArUrccishpU4bcAvBDayEnIzgc1pu7Z/vSNPvz'
        'rMxQWF9eMVo3qdW7CmnQD/VWRZx/yOJJSPoNfE/NK8e6vONggLVnpgVuq4voT6kOI+TQxQTK9YyK4HJAHya5B1sZJUTj2oFHnl3+'
        'pSjUxMphlJLjjaePfJoycY+5YLibymUy1EJaye3FD4trehSCsgAKrSVqaHFyUbnFD2oa1qOVPdJS2QntxTtYWs+ZLAw7cprcYZUe'
        'UppEvuqJO8eXnGe1GbxHuMu3KkNoV6ThWR1pElnDhbeBQpPUKoz2Z8SQI0WW6/WcuvFIPanvw2sNquWo2lvq3qQQQCepqm2nxHRl'
        'B565pl8ONRSrbqBiWVFTSVjd9qQ5QdkPWExPb8JiPabYhtlKR6e1IXiNfXYsJflqO4jAHzXa1atYusFoodChwDzXZi0xLpLD0tSV'
        'AK9KT0ryPGoL8jNo0JGdYEqK3Wi8XGWmWuO6tG7JUqrKsQbjpQlxvYpJ5PanMItkVBYT5Z4xx0oTMt8FbxdUsJHwriic/wCQpvQ1'
        'HX/ULXUDsmF7fdm3EeW2Tgcfepb1wYaO1a/Xj3pMm3S3WdpSm3UhWOuarq+aulyphUwshAPv1rN4XxL3qSpIHrMPZyFAwIqrCTwQ'
        'CK6xbQxJSp9+VFgsISVF190IyB12g/qPwK4LPNA9W2+43JbMqLKQp5hAQlh8bmlpHQfBr3SEA7mdWFLfkcRib15As9tfhWWNJnJW'
        'SFPvN5yBjGewTkZxz96O+DGvo8O5POzHJjl0W0pRUEgJGMnCUgDaMY5JqlrhOS2PpbpYX4i0j9Ud1QT9wM4orpe46TZDhhz5kCa4'
        'goUtwlacHrlPf96eaz7AAABiaVOEGAdGXFq3xDuF3lsLkR1uuJV6Qp0hoOAelK8jBGeSM1S/iX4o6mcli2JuVyhSor6FHyHAlkjy'
        'xn04yo7s4PQAAYqwJttn3C1R7m/EmsxHUhta4DeTIAOQ44k529juJHGaMXDw5tb0ply7oTOfBUWt7SnEhKcE7duVKAHuRnd14qFc'
        'r53CsnYa1FnRviXqhvTcGQ9HdceeCgt5hIS4tA/qHYH345BHFNC9RWnVQYD7p+rDSkAuJAXu7BSgACOetEk2JmbbGkx4MsQ0xiz5'
        'jbAS46ConCRuKUJOQO6vc1VepbE/bpxVZoktpxojP8zcpBz/AFAZwPvQi2dGXNSsMGWFarTJiX5qPJQMkbgQcpP2NM2qoYNvSk8V'
        'Wvh1qK4x74iyXtbqVKO6P5oVgE/4c9AaYfE/UL8OMlLWQoDtTnHVUqOJk209HxAk3TkNxwuLcAJ5JqAuDAgElSkOp9lGl1m83mck'
        '5ykGujVrnylZdcWR7E0gXrQwWl8xjRqG1xWtjLSd3+EChdw1AqWChlgYPuK3h6bbBBcI/wA6LxbPEa/pzQX5Y/qJBceosMtXB3pl'
        'IPtxUpuzSXDlxauabG2Wmx6UAVnG5zgcCgG92le5i+jTbRR6zzUy3WtuIonak+1FiK1UPmg9yfcr2MJWtD5tEhqLEEkrVyjj2+ar'
        'fxE0zIiIbmLZKA4M4I5HxVw+GbqE3NTa0hQPv2qf4u6fizYX8tYCkeobhjI9q1qN1Ax1Cpr3PJy2nCvyzkc4pkszbESLuXgKx3oz'
        'KsTZUpzaAEnj5oJeoxDKkJ4HxRCmswYeMGk9YKtt2Qwl0llSxwD0q0rlqiZHLT8B70rTkj5rzfCbUw8Fgng5zT3Z7+pxtEd5WcDA'
        '5pR+MrN3AksPYlhq1jdlKJLnNaO6rurqSnzikUCYbU82FJ710Ed7OAg5pX/CqBz0EH2M7SJsmUrLzqlH5NaD962+jkpGSyr9qx5b'
        'gOCk5ovXroCVzOKjzXNRNNLwtYtwWlLZO3O7vmhkREKQrBCPsDROkkVmGNJWS033T14hXJtl1QZStoKxuCsnoftSJZ9L2O1anZZl'
        'tMvF1zaGik+oA8jI7cfHWrKtrUq3WZ9dqs/1jq3G3X3AnJQwjJUAe2Tj74pl0lom0y7uLlLWn6lYDzZV2JxwK0+PQHrB9iGDFMQl'
        'o69QbNBWzcWo0W0qd9DqnMIaSsAJSpOcgbhjJ46feoWuNL3VbbV00hdpbLjYUpqMqSlxjHXLZIz/AHpH/wDasfZiMW+HHkNSZWFo'
        'cLQ9BazlKTjr1J/egngnrK+NItFou9yiKsj8vyEl18pejkJyBu6Ac8Z6++KixB4mhUxwDBupb54k6ZuCJV9lPsIeOwJU9sZWcZBA'
        'HFTLNqaFeVJU5bkNzMhLpbljeQThRKVfqH25q69baKtmvIf8GektOSms+Wp04I9RAVhJwTwelVdK8FZOl722xO8yWytzDbkVAK8j'
        'njkYIpNz08xpPzG4UvWh0XHTMedBuIW/CbLzCWllSVJ/7JOUqzxxweOKRLm65clriXDcJsfCXQpOAeMgj8US1GvVBvKEwFuGPFTh'
        'CmkKSp5vukp5569Ce9D50pyZe1uLkvPOPJC1tqIIB6ehXf2I65Fd2JU4i3KQFCBOEOM2yMBAzU9BA7VwwUqxmuiaz2mFJKDXZKqj'
        'oNdUmhGdN3CRivmehNc3QSnNbsK7e9R6nToea0UK6fmtVAHiq5nQ7oFKl3xDaVhOcZ+easXXUKaizPeQyXHSnIzjGPvVSW+5MWV9'
        'M998s7clJFTrbr+4akTKSiM+GVApClc8e9a/EcCvoY7ShZNRD1Q99AlbZPqTnOPekyJKM1S93Jz0NP8AcrWqU2tEkkubjyRzSvcL'
        'KYCstpznngVYXjPWB94gqRbChHmnGSKxZIanZQwOQa2lS5S0FtuO4ojjPapdiEiOkKcQUk8nNGyCRiFXPuPFrfTFZSl3CsdqIMXy'
        'GzISVoSB9qQ7lefKSdpwB3NBk3vzF8rKqP8AYq+pxUGXyNQWhUQDc3uoLMuluL5KQnHXNVI5dUcgOYz81yN2UohAeOB7Gqfcp8iQ'
        'UEsVuGp4FKQQkdTniuQjBt3a2VZPTaa1uc5+NHQGuDkipVjnOQ50W4gb3GXEugEDqDkVlYGMwOsCXHobw3vrLNqkzH1KakFSn2Un'
        'aW8AbEL7k8qUR2wKsWPbY0B+LJYjuhKVBtICN207SeR7108IrmmZa40qUoBTkdTqQCSEpz/MdJ/6lgpBPZHyaiOeIMFmAZLDapDj'
        'kZdwDLYyQxv2oWcdMj1fat7j4SvAjTJkgDYE84r0vcZmqb/DvZehSFurdgoW0QP1HyyM+/tQHVPhXqSNYf4xNjvxFLeCCFJ2ggDq'
        'Md+cfarj8fIN7du865x32lf+5tLZAXlaORxj9+a7SPEZOpNSWvQt2iFtC47LpcUchS9vGPg5OT9qUYYY5mim0GP/AGVL4Yalmacu'
        'kaNJkzpitvl7fO5JzlI57A80/r1brW4ySl2zull9HnxD5iVueWMggEjnjHUe9NPj3oC0p0hI1HarcpifGOVuRWuSgY646ge9RfAD'
        'W8C+2ZCby03Idijy0ANjdj3yccUF0bt1PuGW1WTsB4lcbZc+Q9bbldF2mYrBS2+0sJKunoWM8H4+ftU9rQLluh5SpEhx1wFSgkHK'
        '+MqSex747/mnDx3tiNROtXHTqraFNN5lxHJOHVhOPUEZwMfBGf8ANYixZ38OWblKddkKUs7GjtBHB43dSPtVLM1+ZesLbsRV1CI8'
        'SclBaXE3HYUvHkrzjP8A3uorggYODWLgzPdiycxkvR05y1IdO99HcE9MjtQ23TYzjbbkKQst7dpjSFZUkjqEqPJI9lfg0m4D7WIc'
        'v44/yrhhArsgVq0AUhQ6GuyE0mxmIcjRnwTxg9KhSipo4z9jRJKfatJTKHUbcZV2qUJzJBkeE+XRtPWpJFcI8fyiPepJHeqt5kmR'
        'n7ZHusiK1LSpxttwHb7jPSrjt1htVutqHmITaGSnOAnp96q6I66yyv6aN5r61BKVHogdc01wbi4i3lmdPVIdUkbW0nISfn4rX4Xb'
        'p4zHOO2F3FzVMZx69LVHy4h31IwnAAqMrTRfaCn084plhiJHWt91W9xfx0rlOvkVpJSSE7R3p08Edu5MVY/lAcHSkAkJUEgjrxS7'
        'rKHDgIWG9vHAqdcNVpYdV5a0gkngGkHV17VLezuO0cnmjsyquJesEmBb00pxJPQe1LLroYJxU+6XlJSUgknpQN5anxwCc+1KNuHm'
        '7k0KPB4rCJSk9+aipgTFr9DSjmikawTXQCUEfioYovudnEue5hH0ayrtXG2ueZHSR2OK6X4bbS4vn8UO0q750EAD1bqzv6Rb+kv6'
        'zajd0rpGbeJ8RxNoZ0wxDbA5W884VrJHsB5nX3/NGPCXTxYkRtUPyo4ZlWdmK5HCs7CnPpx7ABI/eqp15rpqy6I0/bHFOPmQCy+3'
        '5W4OoSSNv4GPxQrwz187pdVs0rcLg3KbudxeW++HSoMZTlKAfb1JPHQ5rXrfOM+JpqP+P/caNealuUvxSvNwDaHUWmIpLbKT6X2w'
        'CSPbpmgmo7tF1Hp23eIdojJjz7QkNyIzPpPlJPUH4B4+1FPECdEYS/cYS3ExY5H1SgkEKQSAoj5AJrhp7TzGn7FJbgXGLIt92Rtb'
        'XypOwgn/ACqxTJki7Alr6J1Ha9VaTEmDdX5keQ3l1C1gFCsYKSB0qgLpaJEHxCm6aWyYca6qzDkx14QhRPCj8e4pZauU/wALNXMs'
        'wri8bPcSlTiEnA3A8jnp/tViTrJI1ZEeXKvb8dY/mRnoxCFt8dM9xUMv2DB9Sq2dDlfBibFiat0jqRtF6ecft7D3okMHeglOCQrv'
        '05/epUjWUl+7KQ4+pTDqippxatuftn5FJE7UV5s8qTpy/S3ZRLwDU9albztPHOeewxnoaH6hW7bIjLTy3Po3HN6kBW/Ye7jRPI6c'
        'ppO+ruMDUeo5Br2dywmdSpkx5DjiXMN+pbSOVYzyQaHz7cwmK4zEeD8aefOjvJ/5iVAdR7H/AGqtLdc5w819l1xxA9IUk4IHvXdu'
        '63FVu+hFwcbT5gcScdD+Ohpf6cHBhzfnYjlp3UdxgIU3cWlvNMK2uqCcEJ7LHx7jt1qxIy0PMtvtkKbcSFJI5BFVJHXPfJmzrk/I'
        'KkBKglOMp9jnuP7imfT93nNOIjQ2VKhJ6haefv8A+u1Vuo77HmZnJ462jsNGPYFbDA9KU5PvRGz2tVwShxOdiuc01RNPxY+1ToGB'
        'QaeFc/rEyehGoktwpC/UUECpLcBKU73MkfNNklUJtKgVJHsKD3x+M3BUlBTkjitGv45Acscwi1/udLO5bEx1tOqQF9hS3d3w3c/5'
        'DnozzVZ3jUFyh3hZaX/LzgVq7q5X63leqnFsrVeohOmJaTsxCWfU8Bx70l6snoQg+W6VHHY0m3DWLzmQg4T96Ch++X18tQWHHT32'
        'jgfmrNeCMCUFZzMy7s6mSta1kqzwKhSp7kg7Qck0aRoe7PBO6VFceP6mULysfHzRmw6IcZkAyGlFYPIUOlIXP9fmXyFihA09MnrS'
        'opISfinWx6LQgBTiM/ena32uPFQBsBVU0JAGB0rOs5TNqBawmL7WmoLYB8tO4fFTWbZFbGEtiiRTnNY29qB9hPuUyTBk1oPR1sqy'
        'UqGMUmwJ/wDCLguIs4KlenNPS00ja8iNolMS8cg80zUd4MJUcHBhfUjj07T6ZJB8uK2ogjqPUc/ghX9q7af0na37DpaS8E75MlT7'
        '7q1YKAUlXHbokCielo7N3s0myuOhtu4RVNhXsvGUmomu7XNt2iI/lOjy4jYQgDqFbQkff3/FanHIKZjddvYYhtrUduvMe76VYLSV'
        'JWpkFWAFJUkEEf3/AGoJ4eomW+FNtTspc+LBdISsKw3nafQn/EPkcUr3S0RrchnUMZiUtLqEJKQSVY3BJWT9if3p1ukZyWxBahFc'
        'YNJUWto/xDBB/wA6aXZzIY+pCujNt8QbbHhstBpuQSW3EgbkLB789R7Uwx9BXvTejWYSroxKKF+Y27kp298Ag/GCKg6M02mw/wDy'
        'ZWVvPKWSeqM9hTeuLdbo4iPcleZHZHo2J2nJ9yOv5oqj2ZUuBqV3L0hHuqXrqLv5dwK96oFyby0tYOfQ6OMHpggdRQLxItyJtuet'
        '4huMzYqwqPhQKSkpGQCMgpzmrQuVtNvZdDDq0hXXnH/9pVdYhR3FPqCEqUcqx3pdsZxCCwysdE6MuzjwkPbmmieUjtVhwtGRI5C1'
        'toV+On2qQdSQoKFBvChQ/wD4wEhWCQlP3oVvUb9yVdv3DxscNDISdoHTFFbFboDLwDiRtFK8e9tOKKSoHv1rs9dkBG9L2D0Aqa2x'
        'swbOZcFtmQI0U7ClCRQm+aiUElDagE46mqlXqaS24E+aSM9Cah3jVyUtEuLIIHHNEe/OhKBfZh676oeTMOXON2BzXN6+eayQ6+VE'
        '9s1Vku4yri+p1jOAepPArlMnTm2QkO4PuKXDkSSYy6ieYIKwRx/elGYsrBUnOO9aQv4jcpAaYQ8+4ewGafLX4fS/4YmROfSXlDlr'
        'bwB/vQrLUQ5JkFsDcQ9Pwf4lc0R1E7N3qwa9CWGz2yFpRuGyhthbj/lqKB61D2P3qurLoWfFuSFxpCENn9RKckU5IcvlpcQSj6lh'
        'BHqR3+cVerl05xmcrrLn014fRDYvrmQ0HEoCm0gZUR8ntQHWdhQ1FUfqzGdKcBTff70Q0LqGZfrclmClxrAPmDdwMHA+w+KlqisE'
        'yH706ogHGFIKsgnHT2rQ/ArCgEmVVGOW9pWFqT6Sc9fmuoFGtQ6ci2qY6/bUp8l5W5SEHO35x2oRjHevMcmv67CPUSsXq2Jpisba'
        '6hOaztpfMpI7TSHcNrSULHUmhOqLG3LilHmBRHIIpqkXe3LQQQk8UvXS6RkoIZxzXozw6wM+40y53BOn2lQWU5XjaePjFNMqYxeL'
        'WqDJSlSc7h9//RpEn3PkgEDJrW23dTSiFOf3qtX4OR6lgoGxHVqPHKWorwT5SRgAjjFMK24Ai4KmxhOOO2Kqq46gzyHP71Bc1RIQ'
        '3jzjinBao1I6knMtNi6x2pA6HaepNMTOpY4aIGwcV55c1Y4FEFfFcBrR1Ksbj196oLcSCpMuDU10+pztVjmkK8PeskrJx80uydaK'
        'Un1L3YHvQK4akdkJO3jNLNknIlxqFLhLaSSCRQSTNOTtVihb8t15eSSc18E7sBSgCfc1bH7ls5jBarmQjlw5+TUuTd3QnhSsJ70s'
        'sRZCXDtST9lCpkhUqM0l3yQ4g+lSM5496gjPiU67mZtymb9ySSD0IrjJclLTl9sjbgnPtRW3GK+hhLb7aVdVIIzg/wCtGtTQkvQ4'
        'f8LQl1/OyVlJGR7+wAHf9q4LCKkAuSG2m9kdsISoA8D4pj0hoqTfW0TZqy3FUcjHU05aMsViVpxiLcWmlOpUeSncsjtwKeotvabj'
        'JYgsoCUgYysA/gVnci6xchRK2V2f1UxQ0toyDYpr0hlanPM4SFdhTRsGMYGKluQZTXK46wPcDIrgUlPCkkfes12YnLRNg4P5TmEB'
        'PQYrBFbq4rQ0OVhbRExFtvCUjCW3nAVDOEg55NObVrl6jZm3BN6VFhLcUmPjhPBwCe+KrJYz7irP0Yp2RZGGnVNeS00GsJVkKHXH'
        '+9eg+Nu+xPrb1GqXOMStLjYbwxKe23jJQrCnUL/5gH/T7VDZ3FsbsZHBxViapiwGFnz4xLfOSFc1Vl1ulsh3Jtu1ylTY7qvLU0jl'
        'TCvn4+aJ8hxu6dl9SbV7CFB7VsBWE9uK3SK8/FZU8u8qbz/MNCHtQKJI3ml2bcFuOYBJJ7UQtump1xaDwkRmkq6b3Oa9GWPuO+Z9'
        'MvZVyVZqEu8udQSDTXA8MLhKIBuEJAPdTv8A5VPX4PX5gecy9DlNgdldaobF9yMGV45dXldz+a0E150lKcn7U2yrXb7e/wDR3iIG'
        '1/4mlUYs1l0rtBallDh//IP9anuP1J3K6MSYtCHSk7Vng1xkRXkDkGrff0xGZVwfQeQO3Pt8Ggt2ssQv7UIUvB5UegH296gOcyes'
        'rBtl1S9uDn2orboRUsNqSCVg4z8UbutnEZYfQPTvwrHtWfJMaUhQSFLxuAA+P9ScVdmyJwWCmrSQQ60QptRwQTyg12kQ4b730L2W'
        '3k8pUOMg+1SIKHFvupbVsDnqAPZQ64/cGot2UuUlhtbYEls5Q4OMD5qoznzLYAnwdFscEeS0Vt9EuHkH4NFdNXG3mYtCYqN+fW2p'
        'GePfBODXyWfNajiQeFgJXlIUDUC82eWw4iRBCQpvjb3I+Kr3HgywAhnVEAOtJlwWYzLoc9XlsBBI+eD/AJ13vdydj2hppLiypSAH'
        'AkgE/wBsmglvvU2WBE8oJPRZWOh+1ELNabndbj/NS4UI5ACMZrjZ18y40I3aZv5j2hCmXkNOlOSVIyB+4rE+/KAStt4oUDlXkZx9'
        '+D/pURq13JghqJGU4CdpQpORXaLoy4PzG3nlCGHOQGgfT9x/pSNj9j5jlfLVBCMPVEnalP8AEM99ricE/wD6mmCBe7u+0EI+kkpC'
        'SQFvqSR9shQqciIs25uHKhQJhQAkLdZHb8Zod/whAK1KwY+7nEZakgfbmhM4T+2ZVvkKWGHWd3L1NaAVLsE0IA5XH2vJP/hIP9q0'
        'a1Tp9xYQ9P8Aol90S21NKH/iGP71yGmpUcn6G+ym0k52upCxn+1RptvvwAElUCc2jgBXoyPbCuP71QNU52ILpw7vGozRExZyA5Bn'
        'RpST3aWFf5VYfhwqJChOMznkJUVqUEleMfNedZdjbecU7/w/9MoHl2KVJI/7yPT/AHqXb3L5ATvtWqJ7QR0akpTKQPz1FO8Vq6X7'
        'id/goNoZ6HvsGPe/MQJLTXljKVKRu3Z7A1591UwxpzU7DWHGZqnTtQlPoWkq4we+aYLZrzXkZtRetkG7tA8rhulpf32nipada6Lu'
        'twYc1PZrjb5iOhkRiQg9iCOtP23rasqeKwkuO2t5SUITlau1TpNskRUhToSM9hRW1uaVmIckWW+MPrA4Rkbs/wDZ6/2qLdZa4y0K'
        'kAOIV+k4IJx8GvGfIX8jiWD/AI8p7Mz3oZB+QlFN+HM62usOXBnIdTuSc5yOx+KbrXp2Iy0NyEg+wFWhpTTzz7E66SIsXy1x2ysy'
        'eSpJB4A3EhQx16e1ayNGfUsIk2ec24lRx5bmU85PQ+33/evQMLXHYbhLqXXaxEVb45SE7TgdK6uuXBiJ5MSQUp6c84om9ap7Dq2n'
        'Y6myj9RUQE/ueK1TD3S2oiJcRcp0kIYS6CtRHUD37dPehqtudAxcLYfAMSX9LJuD/nzleYrPepLWlIDYwEAD4pudixGbq3a37vCa'
        'nObcR1FW859gBzUO8SrNa3lsyb1HC0/pHlr9f2OMcfNEKXHcKaL/AD1MiRoDbcQRSslKeEKPJT/5UE1DZXFkK81aQMElJ4VjsaKu'
        '3+xoOBcmln/pSr/aoMrV+n0hTa31ODHACOv2rksZTuXrrvHlTFq9xJa4m2OwgqT/AE5/VQq3MOh8GQ0rIGCVdse35re9apUm4oRa'
        'W1OMkEkPAZ+3FGdNuP3t5mMiEsSXFYwk5T0znPtTH2kYzDsjL5EAKZ2p85aUc4J46ZzUWOmOXG1utq4T6ge3/rmrIuejLvGYC37T'
        'KS2vOFlo4OOpzilWXpmexK37nGwT+laeo/NQbB7lMwPKlwFelEhKUKPQnGOOv2rZEtM1e1GVHHJSc5+RWuq7BHbibtqvOWcIAJ/J'
        '+1T9CaVLHlTnVLAA9CAf1fJ+KgqpXtmEzgZMGvS49iCzcFJfdyPLCQN+Ovqq6PDm+We7WGMUx4+NoBISNyVfI7Gqh1PoO53e7vSo'
        'LzJUeiHPST+a56SY1TpS7JD9tktoBAWdm5BT9xwauoQLo7gyPs1mX06zDQ6VIdBGP6eSo1yyM8Zx81HjLS/FZlNhXlvJCkhQwRXY'
        'VlXNlsYxEGyDgzas1gVsKXMrPsVoRmup6fPatSKrmSJBkQIryt6mtq/8aCUKH5HNA9ahFitsdTU1anljzCzKbS8k56Y3DP7Gmk0i'
        'ash3G5X14KWuS4kbW0kZUcjgAft+1a3xYV7CG/UPS7A4BiPI1+zEleXLtWxZ58yI+pBH4VuFMNp8Rn5bKfppbrjSDgonRA6kfG9P'
        'I/at5fgjdJMQTZz7cZ5Y3BvBJoX4fabvmm9dm1up3wnmlKeBTlCgBwee4PFPciutFJA3HvusQZjArUFhlfzLhp+PvGCX7a/tWPnY'
        'cGj1pv1pkJQ1b9VO+n/7O6jcPt6un4NEH9M2OQoqdtrG49SBg0Fufh9apCcsuSGyP+oKx+9Y1lyFd+JUcxW0wl46ZnsuxRbXrkp1'
        'LIDS9ihjfydpJ/pA9+lT763bAllMR1ZdSgYCEJCVAqyTkcjqeRVNWzxF0r5nnuM3CFcFkNfSgjyQMFRcK8nJxxyM09f8T2UxIqFS'
        'jHeWgqU26DjYOdxIxySTx24rbV1AxCg53EDxBseodRz0CzFwykNOiS28FhplpXDeMf8A1CAVc+/4qPqy2SfDHw1lxRdPqrumT5b8'
        'rCUvBJSDsxz6SVEAZ5CTTfaL8xHUHYLqkS1yVSXFqcKkr3Ac8jpxwD/rVZeMl/l3PXlktHlhJTLZeWopwFqVtVyMncBnvzV1YHMk'
        'nE28M7FMizhrfU6i0uPBdVGjbfWkEfrUcgk+vPwD8cG9PeDGvtcPqvUy2Lbs8hKpMYLmIDqwf08ZJ5Gaa9UWw3GxF+3yiJceE95i'
        'XE4Lza0EEY+MZHtg1aGgX4K/DWwImMomToMFDfmsrU2sI2Z3FwHgH5I9u9cFWwYM57GAwJSdx8GXrZcf4e7YnpLgQFqU2srQBnHK'
        'uAOeOcUGufh1aLZJDM6zqaWsBacuEhQPcEHB/FXbd7jfrzpyK8FNx7KHsl5Tq3XCEn0kE4IRn2zkp+aGPrN3nORLzABgtIUlpSX1'
        'KcYASAkoVjGDjlKh1NJXcdV/iTuAKWMCQ5zKaXorTynAtMZxtQ6bHCKcNBxY8C8xGYzSENJOCCnJUPk96JNaVlSHX0syojXlk4Q8'
        '4QSOcc4xnH2rlGtsm0XZky34qEBKXCtL6VJAPuR0PxQUrsRgSIuBcXAbJnoSJdkxrQ21GjoCNv6TyKrPxEW5dHEx24DTi1E4CUDj'
        '/apETxD08XzaWnnZU1CT/LbRkEDrz2A7mq+8TtUTojRXbJrbkqQrYWmSSkJPbPTPT5rWsdSmI9Xxi5/LQlW6/W+1cpc9cJlSIzgZ'
        'KE8NgJ/oyOpPc/NcYXiZZPpy27YHWXEj0Fp0FOewwR0qP4pX8ixs2VlKk+QMuKKeVrPKlH7nNVPEdddkbQMk+1A+sERd3DHXiek9'
        'M3SJd4AlNthpav6M80XAxkDoetV/4SxHXEKecdwGxgIqwwKx+UMPErRhpkZPBJrdIrCUiugFKmDnwrYCsgdq2AqkkTXFYIrcCvsf'
        'FRJE54o14Z2NlN2n3KWkOlTn8vcM4SQOP3zQnFO+nLaGmoc8SXWmkJz9OklIWtXVSiOo+MVqfEgm0n/UPQPyzJ2rXm4zRb81C/5Y'
        'CG2uVYP34FU25NdOrokWPHcLTqHA6tZOQByP2PH5q1tUvJKVyVoAwCN3Ofbqe1VFGuZRrZDABeRIbUAs9UqHJP29OP2rS5v/AMm/'
        '6jL/AMCI1N/TJClPvhsJ6jvXaHLjSUrbhx9xSOVqpfnblSFlYGc1rEefYJDJwFcE14G35F+xHqIZ3iVNqfWN6Q9Hiy4kFmLwpP0r'
        'DYWO4UpAyAroce/WpjfipOt9pVGaU5KeKuS802pDieOFZG9JGOoV1Jpmn6OsUlxxwRVR3F9SyspGffHSluZ4bO5zCuTTpP8AQ+2U'
        'n9xkV7NLkPqNjkAydbNZxnY7Cp1vYbD4AW4QtwqKu5G4f29vxQfV1xDniJGclPMoeQ+225IcRhCAMAke456dqSbo1HhsFhMgGQFL'
        '8xCM4CkqxzmoBky3G2curKGR6CB+jnP+dNomAZf7Mz0zb9ZwXGXmGwy8tSynz3lEoDQ9inpkcY44VXJ++RIVmbhW65IQWsJGckKB'
        'A2pORt6Y5/eqbg+Id8YtbkFySpDEoIbkJYCUrcQgggKVgnGcZ96KxL1Bkxkr+uQ2hxWNjroG0443dBj55/FUKlYTvmXppe/vz7VI'
        'iPXBvy9iQWUKAUUoTytOeMY647j5oi35TMhxsPOSEbwVlxYHBGQolX6h8D5qlo19kGI6yvKojrYQ2po/yx/4eUnjOf8AenKw3K4o'
        'sT6Wn2G0KSf5aShbeOCcZGSrt15qC4zuXUx31jcpccR7dY5sRp+Yyt1Trw3JiIQkZUoe/IwPnPNefdNWu7651H9I87OaDZUuRMQ+'
        'SlaQf1ergEnGKsafZ9M3qAbhOuEpFxigFLaXQjzc7d+4YxtSSAOcZGBROdc3GLebdaJ0ARlICHnmEtuqHT04TgZ56fBowcMeskHM'
        'B36zx/D+3qaTNm3DUV4UiLGYkY/kNKP6kbQMk/4h2HzRPTFolXPxKtujbG35aISy5dX5aMuKUCCo5OCpRPGPgVWn/E8aZ4pxpbkh'
        'DrduW7KISjaje02doGPkdsYpq8H73e7jqC/3yVMcMmcUAssupKsDOSSk5Tj54PfNWKhRuXNnYYEt/wAWvBHT17sYk2Rx1u4eVuPm'
        'r2jr12bSSPseK8sTdDX/AE1NWi42x0Z5QpsbwR+M4+xr1HG1tcnb+ZSn4Fu8gDcmS6txSgBjalad3H6j0545p6ntsXOBHnzoCBIe'
        'IDSHwhe7jk5Sr0j3znHxihshOl8QPVCMHU8qeGLU9mY4t1p1ltWBhaSk/sasdJB+atNFpTLTLZkFtDACcMIWV+T68fqzjkfb81Hu'
        '9gDsxNsYS3FaUtCVh0JWsoI3DGMbTwR0/sKRv4lrnIi1lAYk5lcJromm6/6MZtzLi2Lkla2+Njif1H2Sof7UrKjvJU4jblTXDgBz'
        's5xz7dDSNnHsTyIsa2HqaCsivuc4PWsgAmljKT7vXwFZxWcVUyRN47K33kNNpKlLVgACrasibMhAYdnPIbDICEJaKyOP1HsPtzVc'
        '6UhLm3VLKFbPScqx+kdM0+vRWbeU+VHBUjKkL9yRz+TW58VUQhf9xugYXMBa9u7MdtLLzLkiC0FKxgeYE9ByPnmqe0nbWJOoZ+oW'
        'lb4q1KREC/1pH9RP+VMHiZeQI8prftTsUQrZlSV/f27VX/hRKvrlouSzFAti3lGM6c7lK74+P9aLzyfpODJuOo43QI+tGx0K3e3a'
        'uzSUIR70vtqUmSQSd/U5oqHUbQpa8HHSvnHLQ/YYspBmhFfYrNfV7bEHATmk9PuTPrHbWy6+V+YVryolXucnmt7lpix3DP1NuZ3H'
        '+pA2H9xRoda2xVizfuT2MRJ3hhY3ypcZ+VGcwAk7goA/mle8eGd8YQXIklqclOcJSShWPsetXIOtbAZ5oi8ixfcsLWnndDt4sryf'
        'q4Plkt4w80pGR+MZzjH5ro1qCW9hOWozbriS55bfqJHTBPIx8V6AfZaeQW3W0OIPBSsAg/g0Bn6L07MSf/h6GF53BbJ2kH3x0/tR'
        'xylP8lhVv/cUoeoYFvtkt999m5rcIQksTFIWhXPrShwHeQSOSMVreNePwtGtvR0hqXMbMZpSVeooTjctXTKs8bsHODUK++H1xt8p'
        '6dHLcyGyN6EpB81fP6doHXnr8UKlMxLgxKtq/NizW9qWWXm/0oCckA9vf80xUyZ7CHR8+DAmiI5euDjq2FvhLZKkhezcCQCCrtkE'
        '809+H7j1gelxWWleYuSEutg42p5BwrIBOM/54NLukLZdIzEuZb3YKH2mFFbUspGQD0AVzu4BqFbLyv6sOF9mM04cKDgU6lKsckj5'
        '6Z560RyWziEVuuJfDWqrTAlR0yYjzqMtp3tygPcYBJBJyc9e1GIerLnscah3UKgJcQG0SGCQhrIJAUnuSEpJJ6A4qnrdeET7pBgJ'
        'g2yS2F5KogJJx3KScnsOSKbrffrfZ5jLrrL6HG5BDqpGA0doyEhJORnjhXHfJqFYiczgncs6Hri6wZmZLYnTZu5alsueVHaSEjOO'
        '+SdxzgkYAqTp28KfcZ+skzUPOrLzzy1pcCwCMoCQnjgYBACunPNJDuo7DdYJVCcw6napCXG0IDyir9ISknBVzjPvziprWorfHQtS'
        'lP29xxX8x2QAMEDolZyDylIAHxVnsxITB3LLti4ybo8zJgmO0tRUoncraFH0qOf0k45xnBNF4tp02hmZ9H5TTj6vOkqX6VFOzJwT'
        '3xwMUh6Vu0O4OYuTUtgFKVOPofThRC87ijI+eAO3fmmlzU1ruYcnsJfcj2971MuM7S4QsBoq/wAI35URjoke9UV87MISPAmsnTWx'
        'tCp7YaErCmglPKE45yrsEgjrx370KumknmFqVb30SmwM8HqPg9DRlF0vcuA27OWwhaniVbEKLjySMYyrPHKRycnB4AGK3enybJLe'
        '+kg+U6IiC0Hm/NQr1478gkYHTIqltFVgwR/7BtUH8xef0fqRllLrlml7FAFKko3Ag/ausrR16hSWGZzCY/nJCk5UFHB6ekck54xT'
        '9Zb1fZaVSrM0qH9WkKUwphKUpXnCvSrgZwefzg1Nho1DPvaH7uXExg24HA20lYyR0Srg+1AHx1P+zBf4wHkxXsVpuNigofagsv8A'
        'muBbhcUoFxoHHpGOvU/t71FdtYhypM2FPnkSEBamXiPLSo9NgyVJB9XHPSnjVdt1JNRGREkqdajPjzmZLaGw411CkOAggpwFEHGc'
        'Y4pNvFpvBgMiShtceUUxXJpdC1sqSnk4IACCeDnOBndmngoqQKusQq4EpPUkhV91fHtCo6W1oVulpIwlTaev5J7U3R2m47CWGW0N'
        'spGEtpSAkD7UAg2S8RvEG4XKbETGhqjBuOoEEPgqyFpwBhPHt3x2pkxWB8lb3tx6EVtO4JmWRmRJL6F+Wo9QBXF+wObcNPJV96O1'
        'kCsd+NW5yRBERQBratayK38ShmRWaxWU1EibCtxWorNVxOmxxisYrTzBu2j963SrOcVxBE6ZrRMdgPl8Mteaeq9o3H81uKz8VWdB'
        't+sdsvUZTFwhtOkpISvaN6MjGQfeq8i+ExFwUmRckmDjjy0YcUe2c8CrVPzWQKutzoMKZYOR4lMaj8PbjY0/X2+5x/p0AlS3leUp'
        'PcD2JPTjvStDkSXkbowkOS9u5BS3v2HPXIPxjkGvRk6DDuDHkTYzMlrIVscSFDPviukGDDhjEOIxHGMfymwn/KirzmVfyGTCC39z'
        'zWidcCsR3pL6EpVnyQooST8gY5pih6pcjx0tgIU0rO1jZlR5HIUQfbHNXHqjTVvvcB9K4UVUtSD5TykYKVdskc4qu4nhFdJDSHJt'
        '1YiLQ1htDSCvBzn1HjP4oqcypx+eoVbRiF7Vr2bdWoMRaIKXIQLiVFhps+X6QkdBlWSokjPwBTfqbWchm3MWzTzDMtDpSJyW45YZ'
        'JznbjdvB4JzyDk/ipLro3UtgcQ45C+rZSpKvOjesI2nOcdR780Jt1wef1bFuskMONxXGyrzXCNxAA5VyQM8/Bo6FX2p1CrYJ6hia'
        'rk+ayZ77JP04ypQ2Ddt9gSDyccjp3phiXZa5FmkojPndvU6su7m1qCTnCuxxg47YqiWtRtxLnItyZLSG0L2rcb/m7SMZSFAerBye'
        'AM/anfT2rJDsqFCLUG4QVLO8Rn9ydiRkqSggbcdwcc8VYEiGDZlkPOONFL7UmQna6WXELICWkkbspSOpPT8561Pl6wtllZYWxZbp'
        'cHorS2mVrQ20lwE4KkBascHnIGexxVbSL6qc0lSAH4e/McPJJWGyoAE4wU5A6cgdK6w58RxTYeIjYWFJDr42bd2Vdf0nFRWSGJPm'
        'WJDDBl86XnXC4wkLMsynFMlC4bq21tMf1J3qHJcx+o8j1cYqNOn3iPammp0hM6Q4FLSuK2Q2DlRDeMHAA4xzwKUozVjiy48y32uP'
        '9Mtf89xDoWXnVIB5IP8AhVnqPtR6LFV5ccNzLj9RJQWfqJTRUWA6QE5GeF4KhjIxg5I7ntDsMJBKQDuLb0KHcW3FOR0tLVlJKQN7'
        'bgznPvkg/tSzPt8mE5h1Cik8he0gEVZ1xctzdmMZpcFi2uvLSqe69lopB/UF4PqOVHb2wfeu0Epu02Pa4V1iyQ3GK1+UjeUgY/5h'
        'Bxk5GB759sUpd8cLhn3B2BXORKhxWQKtS4eHkWZEmyILL8WSHP5SHXRtOcE5BHpHJx3+KTNT6UuNjklBBlRxgfUNtqCM4BxyPkc9'
        'DWTb8ffVsjUXKkT/2Q=='
    ),
    'song_sparrow_08.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABAUDBgcCAQAI/8QAPhAAAQMDAwIFAgMGBQIH'
        'AQAAAQIDBAAFEQYhMRJRBxMiQWEUcTKBkRUjQlJioTNDscHwFjQXU2OCstHx4f/EABoBAAIDAQEAAAAAAAAAAAAAAAIDAQQFAAb/'
        'xAAuEQADAAICAgAFAgUFAQAAAAAAAQIDEQQhEjEFEyJBUXGBIzJhkaEUM7HR8OH/2gAMAwEAAhEDEQA/AHayeKGfV0/iNNJDGASK'
        'UTEKGa8I50ac9gUg9qCdcweaKXnJzQclG+ajRDgHce+agU96sZr1/ABwaBC/3mKKcaYi3oZR1KU4ADVmtDQ9NVqCPUCOKtNqXsn2'
        'puPGtiaotNtbAA2p9FbTgbUgt7mwFO4znG9aESLDugAe1RLONq7S4CPmoH1Y3FG0QQSQFDelEkHqx7UwdJzzQcrilXISBNgdxXXn'
        'JAxmhXXCDzUJUSeaUq0NQRJcUeKhBJTUjCFr9qmDHTyR9qXeN12Nhix1I5oR9WEkU0lsgAkUlm5GRjas7Nic+y3jexLc3dyM0kcc'
        'JXjemFwKi4QM0EGDyazMlpM0sUnxKQmpGQlfqPtQ0hKgDXcQEjGa7FS3tjLQxbVgYFTsJKlCoW0YTR8BrqUK1IyS/RVroa21nHSa'
        'fRj0pG1AwWcIBPtRycjatTCloy872zl38O9LZraSMmmLyxjelUx0DIzSGhkoUSE4WSOKBmrATud6JnPgA4pFNk5JArlG2MrpEchz'
        'J7CoAoZrhfUs716hGDvxTVBSscwFYSKsFvcCcZqt29YG3tTxolDHnEFLf852T+vFEp0V/ZaoL2wpzFfPTvVNg3BlrHmupQknAJ2B'
        'P3p5HmtKwA6kk8DPNWJb0d4V+CwIkb7GvnHc+9KBIINdmTtUujtBTru9CPu5GKhcfUtW1eZAGTvSqew1JA62pW/AqSNHT710twEA'
        'VIznihmUmMSJdkABIqFxR5NTFNQPp2NFdaQaWgd9wAd6UXHBScDFHyVpFAycKFZXKy9FnG9FdlNAqJqDy6aSkpFAOK5Ary+an5sv'
        'RlF8hrJruO0BxUqk5r5v0qz7V3k9DPPZO2DjBp3amgQNqStHqUKsVq6egd6scO270xOWuh1GQEoqTO+BUKFHAGambTnevU4r1Jm0'
        'tsVSn+aST3zuc1PIdUeDS+SCoHNSOhoWTXskgGlxbUteT70wfb9W1cNowdxRp6Jvs4bjDpzRVrsk27z0xILYKiMrWo9KG0jcqUeA'
        'AAT+VTxmVPdLTSFLWshKUpGSSeAB3q5Xm2u6dgsaXkSYsORPS2u4rS4PMjtlJUQv5PTsMH8A9iascfG8jdP0hHh5VoC0vZrQ7ckW'
        '+1PftWUo9XnrbKW0gAbjIIAyfcE5KRjnF1neG1yVCROkXR+I+2cqU0DkjGyRkKI5xkDOO5O2VXbxIagKRY9PtNRWY6A0wGx6jud1'
        'K5J3Jz3z3qp+LvjHqW4MJtdpmzbfbY6UsHylFta8D8a1clR77cHFaGNKn4zIdN41tdGlaptkKwuxoN51GlEh/HSJDshsEbZ6lKCg'
        'BknJIoa4RtKqaV0akbuCGxgFDuUAnbCco/CPdWB9xX50ssm8SoE6U/OWYwI3kKLnmuE7BOTnOMkkfnVw0ppvUl9IS1NeaYCC4tSm'
        'R0DAOBnOMntRZcKhPb9EY8129I1OHFvKYfXZX33VhQDbTnlvNu/zdJa3I/8AaPvTmJOdUfInx3IctJAW04kgE90kjcbVldysdxsF'
        'v/aKTIdLeRIMfrZcSnPvg9JOfnep4WvXVtsiXeJktpAw2HZAQ81vjIByFfZW1UnHlO47H5MKp6paZr7SkYrtQByQaoWndWJekCJK'
        'cQQf8N5JwFfce3t8A7Z4q4NyMnBpTTXso1Dh6YalOSBRTIwcYoSMsEimDY9O4rtolHpHeg5J3NGLBxtQMgHNIz3pBpCyVuaAfPtT'
        'Z5gr3ApfKa6M52rFzthpiWavfFL1K3oycMqIG5qBuKtZ34rBul5NsbNHAwRXKk9qOEYJG4qNxsAbUtWtjPmaB0HpUKaQZwbwBSd0'
        'H2rqMolYp2NuaVIXWTfRbY8wrxRzcgkbqqtx3Cng0UmSR71uYs7fQMycOFvGRvS6XkjtU4cHvQ8hQNbGhSYvXkGuoEaVOmtxIjCn'
        'XnDhKR/cknYADck7AV84eQK+Q/JRb5kJh5TKJjYbeW3s4UZyUhXsCcZ74FTPj5fV6C8mywRtWWvREkpsjUe+31CfVP8AxxYijylo'
        'f5ih7q/TvVVvep7rexJburocbmOByQ4ttIWojgjbI+/NAw7BFjq/7qY52C3BsP0p5bNNW26vCG5KfYfdOGVqIKM42SRj3PvWlHNh'
        'r5ULS/8AfcsQsMLf3KA/YlMzDLtsn6glSsoAKijGDz+f9qrypDTchTzr3kyC4UKbcT1ZT3ORg+9a1M05P00w/IhJbW+yOpxognzE'
        'jc/Oa40jL0fqScxHuYYhIcJSFygA2hwZJQVH8KjuQTsr77VZxy5nbRXyUrrSMzgSUPz2C+tZczhKQ0AkoVtlCdgD+ma3rTej7M3p'
        '+PcYjJmBSTmSmWlQB7K6QMYx+VXef4OaVvmkFwoTMZDqwlbFwZPWUbbFJGMpx7ZxVUsfhBqLSHkuPaicEZxRS40h0NjpzyPXkn/m'
        '1VuS3mnUvWixxlON99ktubtN2D8C7OSW48bBcaQsK6wo4Gdsgb88Yql+I/h9bmZ0h6JEehW59QHlPwQA2ocqQtO/SftzmtviWPRl'
        'ptRustRlocb6VMSG8h3pGB1J3USN8ZrPdU6/0YltNtg6auCXkO9SHHFBBRv/AA5wenbYH9KRi41Y39D7H3mm/a6MetWnpkKc5EYW'
        '70BXWlspUhShj8QCsf8AO9aNYJC2Y7UWU6HFj0pWVAk49jj+xOKZWe9WHVNwEdmyi23JST5LqlhxtWBsMcjPwcVzdLOy/Gf8hpxF'
        'wjtlIYbOFHf8SAeftjP+tW7w1a+pFOphrobRV7g0xafTjeqNpvUcZwiEuS5LkNdXnkMKQtkDGCtJ9t/xDjG4HJtbKwsBSVApIyCD'
        'kEd6zM8XiemIlDRLoUdq5W0lQJ2zQK5KW9gcmvhIWsc7VTyZkvYzR0+pCdgaT3JSnAUpFHLJ6t6icSDwKy81O+jtCJEQ9WV1OEIQ'
        'MACiH09OTQLjhBql8hL0jmzx5QTQby0mupK9ic0vcdPUd6qviU6OTJFdJzUaSEqrhS6hWok0xcekRsax3wTiifUr8O9KYaVqUO1P'
        'ojeEitTi4W/Z3zNCxS+1ROKOK6UMVC6oYrb2LYOs+qiG0+jehVZB3qdLqQjFdS2TLPsZVRUZamlJcScKQQR9xQYdSDzmu0O9WRQy'
        'mnsZs0PxFAExqMVhpM5tag704LbmUOBJ+6Qr/SsUFvmt3566wnYrZTDcW8dlMOK83p6VpOxyFDY/lW53CwXe8abbvbyEuBEcTFpV'
        'uHAoqIwB+HCSf0ql+FmiXJWo7xY25kZ2FPt/qQ4n1JWSlfHOR0j9TXoZtuX0cplNGk+EzFt0Z4dxrveFJZu6mXHkpwpIbbKtkBOd'
        'x1HgVdrc5cr63Gnz22XLfKiFLzefwkn2+MZ2JrnTybFH0rGiX1bCyhIiLU+B6FcBJ7Z2qq6tZ1Lp/VUIWWQ4uzyGyhSCepKCNxt9'
        's0pRX8zGOpbaXv8AJS/GUzLLdFo09dXZ4zluCsgqYP8ASR7diftVL0DeZk67S3tRQGnHiCl1L7KeeANxUHjdI1FA1YiexbfpSW8L'
        'lN7of+4/4fvVElXm/oxJX1rS6jC1gg5I43pqjxraOeXylSzQp15slsu+G4kxiPJc26d22HO42GM/H6Uuumv27rJctd2BjT2TiLNa'
        'SOkjIwpaeCcbHuD3AqlsTxNjKbccWlw5zlRH/PvVevBfCVPlZU8ztvzj2+9WY97Kt31o0LVcHVtjTC1LcLemRFBSBPtslST0D+ob'
        'pOOCrBztuMijrVrdNtlLlC5PXyyPJSVLDIbkxFn3Wgek/OMZ555i8Gtcz47DcGS6042G19CZA6m8Y3SpJ2KTjGCDsTUd+sWmbhPc'
        'nWa4DTt4ejNyTDCeqOtLiQspRv1dIzwc4we1dlwxmTloFbXaNJs9yg3eEidbpTcqOvhaDwexHIPwd6ZtkYrBdM3OXpm/Fiej6R9z'
        'hxlWG30+xI/Cof37EVr+nNQRrp+4UgxpYGfLUQUuD+ZtQ2UP7j3FeV53w++P9S7kdNqhy73qHNSrVQzygk7VjNdhbI5IBSaUvpOS'
        'TTJ1z0EmlshY3Nd0A2ASlYBFCBAJzUsk9SqjQMV3j2C2cqR6ahCcHfiilLSkb4oZaute3FdUgOg6GsDFNWHNxvSSPtTKMo7U+Mmk'
        'A2AhRWdq5WEpG53rwOpSnAqJwqX8VeTLLkikOoSDihTI6jgVK7GUvvXLcUpVgirM6aFtaPNzvmpm9veuvIUkd6+QhQVg1OiNm96U'
        'nu/9A2mYndEOMwl1vq/xitxxpKT8AAHHzVQt0Nq1eK8h63yTDUi7IKkr5LLiVHj26jhND6dvXk+H0+M5IS2YSUuoTjdSkPIW3+W7'
        'ld+INl/b9svdxYkC3aphyDEdQlX4g0eptX2UMEH5rZw0rlDl1O0WnxdDVmdelPqdVDvclpt3oI/cK6dlY7bY/Os/1N4oT9Jajbsd'
        '1Dj0FxgORpHT1ZHtv7j+4qyXy9ztZ6CtFwTGQ3NQofWMZynbIJx7b70o1Ja7Xerc1br0hpsq2jKUQFJXj+En3+PerE4u20LrJ0kx'
        'U/4haa1SgQJTiElxIBCxtk+2ayzxBtN20zJVKtpD9rWc7DJb+/xXetvDqbZ4D9yhrWVx19R6OFI70y0RqCPqGwKt8wp85tPScnn8'
        'jUb0D+hSIslu4D6qKpLUlI3RwFUqu7zzj3QkFtZGFoO+3cf8NG3u3iyXpcZYLaFqyg42I7iu1w1T3UK9PmMnJOdyKlVpkNbAIs82'
        'ydH6SMFW4NOdRrdl2+JcmXEokwUBGBsVJBOPvVP1A0+3NPWlWM4FWO2Siq2I89tKutPQrq5B701taTIW+0OWbu1d9PLgzWEyAgFx'
        's5wtlWOUH/UVZNLaKut90u3c9OXZM19pfqjNOmPIbcG4KQfSs4xwc+1Zba3lw7otnrPQrPR8VZdO6hf0ve49yZyYDriS631HCSDu'
        'NuOOaNKa+mgdv2atonVLtwcNlvHUxeGVFBDrfll4D36fZXcCrFIdAJBPBpzrnRMDV1uauCZH092ZQFsTx6C5tlClEc7fxfnVNtbt'
        'xkspt1zabYujLJWsJdSoOoG5eJBwEgbn7jvXmPi3wh4P4uJdBrJtaCZkpKEkdQpY5ICzzQch4uKO+ajaQ4o+nNeZVOnpInZO68hI'
        'JJoCTcUtnCamkRXSndYFK34O59Rq5ix17aIbZMiYXlc0axkgGljEYtqpgw50kCmVHQOmMWcUewQAKBYUCOKJScVUaaZ2gdLGSCRR'
        'DETJyriiIzQPNHNt/FaWi3QudYQhOyRQimiVcU3ktnJ7UIUjPFXcS6FNgqm/Tih0t9Lm4pgoAA8UKtQ6uKboBhtvUhBWh7dh5Bad'
        'H9JGM/lz+VMdRX1+D4hX0X+GXIt7tTBbU0SeooSMOJ+5KgfsKR+ZgDANW/T7TGoI1tbX5ZuNlKuhCx/3EZR9ac905zj4HzVvi00/'
        'FEzaXVeirIlXuz3FX7KeDyQ3l2OTuoqGQQPv/c/NTSbrafEXSTlqfUYFxVu11bFt5PBSfff88Gm14s0eT4gJAUpjzIhSEpOA6ge6'
        'T7LQTx7pINVN7TU26N3SzCchq9QZPUzJbAQpY/EOsD3Iwcj3rVm6npi6SrtEugbteHosrT2p0l+TGJaUtXqJHG59wfY1lmrLS/o3'
        'WPmxeoQ3lZRg4wD7Zq26+b1JZ5duvyytEtJDUhbY9LhHuR89qjv8+Hq+0/SSUIbuLSepscdR+KRddhyiv6qmMyILX1QOD6m1FPB9'
        '8UoRlDDcqO4UuIHTtx8URb5TD8BdnuZCS2cNrVyk0GmG/EW6x+JkpION8diPil+WumMU79AU+Y9MSS6lBcbO+3PzUsS4hwpacbSl'
        'C09JPZVCTOpk+cU+pB6XU9x3oN4hiWEqP7p3g+3waauwWvFnc1l9i4BQV1JCsg0dBmtLdchSQFsuJyRng/7VAHFhwFZKinZQPbvU'
        'LTTS7iSyodXbvTFX5F1On0b5oPxZuLcBq23J1lKEtqiMDqIC04bS2lON8ISlWd+AN8mh5aYch6PGdmKYUHD9HIWn1tAnPSo+6Twp'
        'B254rIdM3Vy135BkICgyoqb6wDg4I9/bBq1aik/VWxCyFtLU6VIUTupS1rUf0KsU+cicuKW0wHO+y8uMFBQpSVpWpIK+ojBPuU45'
        'T2NdyH4sSN50mQhpHG9UXT+q5HlttzllaG8IeBOcDP4x2PGe/PJJq23a3xLtBDTvqQoZQtJ/uK8pk4OLi5/qX0Mal5LoIS80+2HG'
        'XEuIPCknOa5KQc7CqGo3XSczB6n4Kj+X/wDDVutlzj3GKHoyuoHke4o8vF+WvKHuX9wpf2YWtoDevEoGc100FOEJCSSdgKbrsd0a'
        'jB9yE6lsj8XTVd4apfStkMgtTjTUptTwykHeudV32AxIQGE4IO5A2q66S0db7jb/ADJT5S8obDOMUPc9AR47i1Eod/lKvatHjfDs'
        'kx/ElOX/AHImHT0Ko2CKOaTSuK6E43o5Dw5zWbMj6OpiQRttQBaIox1xOMqNCPu+nCdqsz9KFMEfV0+nO9DJSSc1OtBUcmuQCDXe'
        'QDO2mc80RCceiy25EZxTbraupKh7Go2zhODzXTIJXtmgdtPZKLq+2zqK0fVRgWbhGHUgJVgoXj2+Dx/+VWL/AKdnXWU1qa1vKg3x'
        'uOGn0dPSHVJOUkg8HkfY4pjaFriOh5skK4I9iOxqzNSG5I81oeoJwtHuD3rZ4vLnKtN6f/IFS5M8tN9a1M3Ksd+iJYnoHmFo/wAS'
        'fcj5ByKpfihp9FskW+829KvQry3S37jGUq/1rSNV6ebmXONeYX7mfGX1BQGy08KSfuKRSi5FlrtN1aKoMhRLDpGQgncJP967Lk77'
        'Q2J36MxuNjZvcRV0tq0eec9aP6wOD9/96V6dkLlNqTktzImxSdyU8Y+RVqctMmxaykQmgr6GejraB4Cqrl7ZctWpIt2Qjy1eZ0Pj'
        'HPcKHyKCHvoZXXYRcbaxKb8/ywyVJIcQBsce4qp3C0P/AEZaTh0I9TS0nIUn4Na7IjRpkJxsJKUrwUlJwU+4II9xVAnvyoE1UGYp'
        'qJJCv3bykhLEgf1ezaj/ADDY++OabG13It3vqiqsreWx19B89gYWD/Emopo6W0ToyiPUNx/CafzXul9ZVDDD6dlpVyM+xHz7Hg0N'
        '9NCcQvpWR5o9bOPfuPtTVentoHSpaAZD7k2D9WwB5rWzqccDuPimluuSpdojtyHDlCg0lR4SoHKf1oyLpW5WS2t3acyY8G4IUIbr'
        '3pD3TzjuN+aUuttvxH4kKM4HFkKdTjIGPcUz+mgWmMdxIccT0hR2Ug/zdv8AUVo3hWh67Nqhtlf0zYK+tW5Z7oNZhFYmTI4SWXBc'
        'GBwQR9Q2O39Y/uMdq0/w0vke16bucd1CmnX/AElzpwM4x+VDeCcq8bXQWN9jee7GnTP2VbbW/cUFfluvJbKkpPftQ9j0HIs3iPGh'
        'JceTbpCepZKNh8dqdaM8SbboyxohqjtPZUSVpxkknk05e8b7FKYUVwx142OKsY8WHHOp9B1tv6ix3vQCWJLMq1rbCUkE5PNHu63t'
        'NraFsvCGm1pGAfY1iF78Xbr9QtuG+UsE+kdqpF8vlwvs5Lv1Hmu89J5rv4ePblaB2kbPr3WkJIDthcUlwHboquw9a6nmtEOrUOxp'
        'Xotpmbbz9VFUh9tWFdXvT92K2E4QgJA7CsvPzLTcz0MdtrZ0l7GN6JblDIyd6Sl7auEvqJ2NYqeiHRYVSArbNfAg8mkzLys80Ul1'
        'ZpiAbGORjFRKR1K2IFQBZJxnapkKxwM0WhewlltHKzR0VtsEYQPzpa0pRXgnamTKvTQ0tBSw9GNqnZfLKgpKilQ4xS9DmK9U5lXN'
        'V6vxfQ6VscmUh5CiohtWMkn8JHf4oKfGaeZ6XUBaDg7/ANiDQa3AppbXWU9aSnqHtkYzWa+D17mQ7ledNTX3Hm2D5kdDishHSopW'
        'E9huDgbVp8bkeeKqt/yg1j01ovV9tSJ0RDTa0tvskKYcIz0kex+DwRWf+JdolvW5Mwx1JcR/ihA6gMe57j3BrRXJsU/5haPZfH61'
        '4tYcb6ekOJIxnkEVZxVGT6oZzbnplO0SBLszTbyfWlICV+x7Ubf9Owr3bjGmsAOp/Cscim1vtsaEs/Sp8pCv8v2prGhvzX240Zku'
        'vOKwhKSNzTlv0Kp7e0JNC+GWmYdgdVqRa5ymI6pbrzqsIt0YEgAAHKlK9kk9I7VlD2p9DRNYplRdKqlW6O5lDTklQDuNklQ/vgfb'
        'itY1CLmiz64sJZWmU79M2oKOfSlPH/5SDxZ8HbLpHw3ttxhoku3F1CFSJKnfSVKG4CeAK1VK8FpHNMzbVmqLjqa5mdMmF5SPTGaU'
        'MNtI9m0jhO3HtQ1unKQlTsY9DoBStBG4pqnw8ukYNNSJMfrlICmFJSvy1nGcdZSBx2P60uTpq8t3liC7GXHmKcCELUPSsdiePz96'
        'pZLfzPfYUbXtB1jv9zempiMpbddB2SpO/wB6dXTWN3TbJFnegR+hZPWoI3z96u0Ww+HmiFom3GYu4XdCMlKT6UKxuMf/AHVA8RdY'
        'sXh0Li29qMyg4SUJwT96fTqemEtKd7KkiPLfQVk+gH3VxVnsWlS8y2+8+SlW/SkY/vVXtspUh91PWPLWghTavc1cNCXRKJf0Dq1e'
        'S9s0Fndt0DdBPyBkVn8q8kw3H2JhzT0yxx9PW5tCUlhJPcD/AHNdN6YtjU1EptCkrSc4HFPYzZcP2olDGTxWXOe37ZYeJfgiYKUk'
        'kJAJ5wOaIThQrxTBG2K+CVJ2FTvYLgr/AEEniukt4O+9MG2E9O/NS/TIPFZ3zUKcgDXOMUWkHpr1TAQc10FDGMUc5U/QDk4azmi2'
        'cmh205VxRzScDcCrE1sXSPmxhXFGsKwN6FKq9DwHHFFS2iJYWp4J96hXI2yDQzzgI2NTWSA7dboxAbcQ0HCStxZwltAGVKPwACaQ'
        '8LqtIfNnC5RI/FVNt2lL7cPGVEmxx0LaXHMqa446lttpnZta1EngKI2GTUt81nqG+SblbvCvS7r9vtxw9cRH8+Q4nOyiTsjqwcJS'
        'M4rYPDCwSIul7R+0lPybreJrMeTIez5pZaw++kD2T1JCcf09ya1MXAfHVOnva1r9R+GPnUl/78v/AAVXWGmzYrUqfcLzb+jbpQ0H'
        'FKXnjpJSkH9aokfU0VpKVWqPPfcIJcccIQ22ckbpBP6k/lV41La5Op4V41SqA7NQzMdceekLV5KEJOQnbtkJwPf86WeFt/0xGYuL'
        '+otOB6MlSER1Kt5DDQIOfbAJOMZyce9HxeBjlp+n+5fzzgxTpJ017/UbeHF/sMtclrVs6M0sdIjlleBk89SuO3FXh+DZowM60ajQ'
        'wpI/dqCUqCVHg9yO9ZVrLUnhs5CmQrFptlcuYAEFprAQvO2OxydiP96oMC0z2XT9e4423yGuog/n2rQvJGGfqMq3Pl5JGla+nWwv'
        'ibd72mBIffPnPNsl5KultO6CDsSQcZ23rTtGao074haQTbZLanEtHy0CW30+cE8KA96/NmobfLnWj6aMS4216ggqyR9s880gRe9X'
        'WZiM45JksJinpjrIx0/Ao8XLi10/2Bdpn7JuVzi2e2Fi9oact6VBCVdPpbHAHx96Wav0NarlFjTLSgIaUjrBbVvxsRWVWvUGpbvp'
        'ZtE+RHkfVMjzW3kYz23FTWC664gQ/ok3OMmOhHS2FZWQOwqrXxTiNObr1/QPculSMh19Zbxa7++zcQ4lSlkoyclQzsaSvxnEQ1qK'
        'OtCgUOoOxHz8Gtnjabcn3n9o6juBlrCupCEp9INF6l0jbbjl/oCFY6S42Pb57ikLm483+1WyOt9+j87qntJUyttktyGmyl4nhas8'
        'j74pqHFqWiZG/GQFKQDjPYjsQRtTDXeh7haXvq47XmsHhxByk/8A1+dRR4DztoiyUNBpYBSnt1clCu3cferapWuhWvFmt6LubN5s'
        '7M5vIUcodSRgpWOR/v8AnVhaQM8Vkfhrflw72be/6I0pQQUkYLTw4J+Dx+la/GWg15vlT/p83j9vsaeKvmRs+ca9OaFdQANqbqSF'
        'N5xS59HSTRY8iZFrQsbR8VL0YGcUQ1HVjip1RiEbjmsxtlZil7fioRgGmL0cj2ocRXFuBKElSjwAMmmY2xdHjWAc0Wg5SCRjtXH0'
        'bzZ6VNkHtUNk0zrHUeq1P2plP7HhI8l1broSkrxkpSPc7jNa/G4t5NpL0iu0whYGDvXEWM5MmtRWltoW6oJSXF9CQTxknYVoOktE'
        'vRby29qRhn6MdQSnr6gpftnHtRU7Q1yteskXiBHhz7cgBxqIMBZc4xg7Ee/3FXsfAyePlXXfo5S2zPXtP3hp9Da2GyFL6AtD6Fo+'
        '+QTt81bbzA0vaPDe53e0SFPXJlhSVrecIW4j8LvloBHsTjG+1Zdr5fheL9KUu1X+2yvMUmTHgyvKQlYO4CTkJ3HA2+1UC5WhmY4k'
        '2QXdEdXCpUkHb7gCrk4MGBt+/wBfsMS17Ld4aa8heHNinrsVxEoPTEuriOt9ClZT0hSV43wANiO+D30zS3idL15dol8cWLdDZDlr'
        'kLCv+2U8nqDo37ApBPuawhnQ0JSU+dNeC/foAq9eHLds0rHu0BbLkmJdWA2/5hyUqSFdChjGME0jJyppeOy1xcszkT/PRp/iNf4v'
        '/htDOn5LUaI1JcW1DXjDrbKkpSD/ADEnKjznJNZ5dde6t1JbXIUViFHhyupDiXVAg52JO3Pz37UE5Cn/AEcW23dDyQpRciuO+jrS'
        'dioD+LO2R71DLjW+LKEVia+w+Mh2OVgoz3Qrse3I+arxmyY9r8mjysU3j3j71v8Az/yII1oFsnBuTID7i8lJSMIKvjvTB3Kj6iTR'
        'GqYj5Zt7EXPmpPmuFXKCTx+gqNxsk5xv8UrLTrtsys8KVLX3Ry2pKMYOai8RITP/AEzZmHGlfWzJRWkEYPljAH5HevQkg8VItv6m'
        'e3NlLW8802Gmis7NpHAA9qjDc4m6fv7FYsduUGY7TQ2ShISB9qZB1SvV0gA8Y4qux3jsM03jynfp/J6vRnqxise8KbGyw7rx716m'
        'Ups+kkUJ5oxuaiU6CrY1XeNx2vY+NP2MFPMqSoLbSUKHqTjIP5f7UoXpW3/sua3b8JRKX5wB3Sk4xgfH+lduPADGa5iLeYkmRFe6'
        'Cr/EZUf3bnz/AEq/qH5g+2twvimn45v7/wDZGTj7W5MqucJ+JcluBCm5rCsKR/OB/uORWo6Ruou1raldQLg9DuP5h7/nzQ+t7Abr'
        'GROhJCZzQyn/ANVP8p+ex/Kqvoe5N266LD58pqQeh9J2DaxwrHtvkHtz3q58S4y5WDyn+ae0Dxsny70/TNaQv90ATzXDrfUKgbUe'
        'oDnFGtgkV5zHkZfsJjx0qdSkj08n7VWLRdHbrq6c03n6Vj0DHBNaZp2z/tIuNIcCSsFJV7ge+Kg0voKBAl3SM04pC0KyknlajuTv'
        'XpMHw6647Wu2ZrfZWXo4GSRxXF2ltWKyxFl1DC58pDciQBlbLGfV09ie9aLpfTTLkWSm6RT15KQsnb8qznWWl7svUTEBTIk2p5YQ'
        'p3q2aSNyo/YU7g/DbwUslL/4dOnWmPLx4faWv0ZqfpPUciBJP8f1BcQrvkE1UtaI1HoTT6f2ZqaG8PNWpxpKfUoqO6uayDWt8Rbt'
        'ZS2dLzHoVsYV5KPLdOF45V+ZpPcL3NuDKn3nn30tjKlqJIzWveRfjTI3r+prOj/EPWb8eVGkj6xlxBIcVsUHsDQbHinr+zyyJa3U'
        'tJBCVeUV47HPxSTwfuT8yDMivkrQwsKQT/V7f2q5y1JxgtJI/qGaz75VxevaEVlfoorb8bUdyduM6auXKz+BSQkAD4HNGqdwrpGw'
        'HAo2bEhofMlqM2h4jBUkYpUonzT3qrlfzK2LdtsZxVKyFZ3BpxCmSGcLaVHBHuqK0v8A+STSCOTtk0zYUAKiY0Mm9BtwcfuLJZmv'
        'uSW1LK1IUrBBPJQR+D22G23FCSrFCcabDCz1oTlC1DcHOcK7/qaIaWjPNFt4I5FG9v2NnLU9pi5ptaGg0uH5ah+JaHApB+d8Ef3r'
        'lbQBzimqkhQwOKgWwDzSrnYFVsWKaHOKjWgJFMHGiPahHwT7UioB8gZt3pWN6YMTEpHNKHyQqo0OknmlPFsJUPFzR32qB2ac+nml'
        'aln2zvXTYUo0t4ENmhgy8pSsk0Yy7jG9L2UEDJO9EJ296r3x0yxFscxZKkDGykHlJ4NIdX6c+tbcu1rbP1KB1PNDlwdx3UP7j5o9'
        'teE4zTuzdSgClRBBo+Hyb41pN/SMyY1kW17Kz4eahQ6tu1TF4JGIy1H3/wDLP+36dq0JpNZ3r7TCmVqvdrQQM9UlpvbpPPWn89z2'
        'O9WnQV7F6tY85YMxjCXgP4uyx9/9Qatc/hSq+fj9P3/3+4rFlbXiyzP3e26SmuXBF9605KgyV5ByaW3rxcXfYak6dtjpmpP+IOK/'
        'NjE24T5KI5cdfdcUEDJJ5NaXCsd20dDbnMqW8XiOpA7V6LLyHK+kr1ZdJF61/c4XW7OVBVjAbQrA+9eSV3nTvh1cLjfbi7Nul8WI'
        'NsY6yekH8TmP+cU7tD8VyKiVckveUlsrU20MrWQPwj70iW5d9V6qbvl5hpgW+Cny7ZAH+WOOpXzSMPJtp1b/AGF+bM61L4SXBu3C'
        'RAkpkuhIU4hW2T71VI9h1hIjizpt76GFKwoFISnbua/S4WenFBy14TsAKD5lfcFWUbSenGtNWkRgrrfcPU8vue1GSTzvTGYrY0of'
        'c6c5pNS29iqFs7GDSVasOmms9wEGkzy8K3NQp0L2HMrGPtRQeA2pUy7lORwOd6JQoqotBJjJh3JppFPUBk4FJowxinEHJxUhpjFp'
        'tPTXziABnNSNpATua4cUMUDRzYG7jJoVxAVnIoxwA1AtOxwKW0QJprO5xS8elVPZTeU/NKJDZDnFL0g0zlGCc0Q38UMn0mpUqOKh'
        'oNMNSr018HMHeoW1HFenJofFDVQYh2nunn/Vj5qsN5zir74cWRctf1biT0A4Rng/NJvjq/Q1ZvFbY2YgvuoJ8sAEbZ96z65253R+'
        'p2r1FbULa8vokNJ/y+ojKftnBHyB3ralNobV0JxgcnvSO+xIEtLrD4S626nodSoZBFXONUzHybfTKjz+V7KtpXw7stkCH1MpckAA'
        'lR33qyTmmXWvKWgKQPYii3XQE5zSuU+BnFTTb7ZzBnSlHpQAAOBUSHCFVA6/uTmoFSB1c1EsAcKcAR1ZpVPkgZArlczAxnal8twL'
        'OQatJ7J0RSHc5pTMXziin1EZ3pZIXUsFi6V1EnJ2pTLVvsabylAINI5Bysmu2Ko6YXg70ziOpVgZpQnap2FEHY0LOTLGwR32ppDd'
        'SjGRVWadcGNzR8d9WxJJpbsYWtt8KRtXwBJpTDeJ2JpghZPvQOiSfoBG9QupANFMNPup/dtLWfgVw/AuCz6Y68UDeyBNMJwRSl5X'
        'rp1OgTkJ9Udf5b0lkIUFkLSQR3oCUegA18EnivmvaiunKc1KJREhNTBJwNq8Qk+1ENNk42qWhiZzGYU66htI3UoAVsFkcFus7TEd'
        'GXCkJSPjvWb2RCE3BpbmwB7VpzE+1x46VrUkYG2Tuahtr0Rl7Rw+3KcbK1Ege5BxVUv9xaiJUjrUpfZJ/wB6bXfU0Z0FCCnGMDB2'
        'FVG4PNSVkpT1E+9VrevQnwp/Y//Z'
    ),
    'song_sparrow_09.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAQUBAQEAAAAAAAAAAAAABgIDBAUHAQAI/8QAPhAAAgEDAwIEBQIEBAUD'
        'BQAAAQIDAAQRBRIhBjETQVFhBxQicYEjMhVCkaEWUrHBJDOC0fBDYuEIRFNUc//EABoBAAMBAQEBAAAAAAAAAAAAAAECAwAEBQb/'
        'xAA5EQACAgEDAQQHBQYHAAAAAAAAAQIRAwQSITEFE0FRFCIycYGRoRVDUlPRBkJUYeHwFiM0RJKi8f/aAAwDAQACEQMRAD8A+hhE'
        'B5UoRj0p3FKC16244tqGRGKUIxTwUUoLQ3G2jIUelLA9qcC+1dC+1bcHaNjHpXcD0pwIK7sFazUN7RXtvtToSu7a241DO32ru0el'
        'PbfavbfatuNtGNntXCvtUnbSW2qMsQPuaG420YK+1c28dqRc6lp1vnxryFMerVHk1/Rk73sR4zwc0dzBSJew17aRVR/i3RmbbFK8'
        'hzgBEJzUu/13T7GAS3LPGCM/UhFZtmVPxJu32r2Pahi462stubaGSUeoXNRT1lFcRnayoM1vW8jJx8wvO3zxSHeNf3Mo/NBk2uKE'
        'Lm48siibo7S7nWFS6u43iszyM/ub/wCKSU3FW0PGKk6TJuFPYg0hniBwXUH0zRLqNhZafpE0tvEI/DXcWIycCsztNRsJtaW+nuVN'
        'k5YnchCsfQGkjmcuiC8aj1YT7VPYg14x/Tu96GNVuo9ORL6G5YLOd0UJbgL7VXv1XeQqN2x0ckrjuPvVIycugklGPUM2SkFDQxZ9'
        'WvszPBuHqO9WUfU+lOF/UKlh2IprYvqhGBSgBSwtdCVOygkAUoAV3ZSglazHAK6BXm2qMsQAKhS6tZJJ4Syb37YWtYSeBXsVT6lr'
        'sViqM8X0t255qIerLYBcKGLDyNZW+QNpcMJAKlxWoI3M6sB3C96gRTlLQ3VxtjVkBQN51W6trV1ZwOYWgTjdyai5tukVUUlbFX2q'
        '+BfOgi/SU4571Ki1Gze3M/iqqgZOTQbrPV+ktpaLcD/iWJYlOT9qzfVtevbtyEkaKLPCg11YcE59eDmy54Q6cmtav1Ey28ksFzDb'
        'Iv7S5yzfisp1fqXWL2djLfylckYU4GPxVPNLLNzLK7fc02SMbVruxadY+XyceXO59OB1rqeSUKZCSxxlmoz6B0KfWep4YISJ7ODD'
        'XDEfTn/LQIwzzjNWej9Sa1o0Jh028a2QnJ2qMmq5YSlGodSeOajK5dD6fs9E0qwjPgWVuh7nCiss+It5J1XfXWm6bd2sNvp3Misc'
        'GR/QVn6dc9ViXxP4xOzEYIOMEfaqGe5mlnkndyZJGLOfUmuLDoZwluk+Tqy62Mo7UuC203WNW6cv2ZYwrshUpKuQR61WT3s888kz'
        'sFZ23MF4GaTd3l1dbPmZmlMa7ULdwKj4r0YwS5a5OFzfRPgnQXsyuP1iB7mjDRPiZ1DplukEc0M0ajCh17CgAr78UnZg8GlnhhP2'
        'kNDNKHRm46P8ZLSSF4tasGAIxmPkH14oW+I/WGl6wtra6LtjtIwWICbSG96zchvIk0nGO9QjosUJbkWerySjtYVS6+bm3tobmQmG'
        'EEAdyKY/iiE4iO5QPPv9qHY5WhYEEEZzg1x5N0hkUhCfIUzxwTFU5MLdOuRqN1DZLcJC7nCs5wPsajXuqixu5IpipkibYQDxkeYo'
        'UlL7s5Yn/SmXY85JJPcml7tX/IO90fWYroof6g6jttGiBuGDSH9sa9zQjd9dX93J4VuqWyH+Y8muGOGclaR3SzQi6ZpzyRxrl3VR'
        '7mqbUOo7WFmitQbiUd9vYfc1nJ1a7vnaKW7lkcnCjOBU+zvIFhWzYhAvLsDyx9K0sTiCOVSCC+1lpUyQJsr2V8AVHTVbGzMcs0Aj'
        'xhtzHkmhTX722kvla3YwHGHXPGKrp9RjJWQN4hhP0qxyP6U8dPuSFlqNrZfdZdUabqdyDao644J9aHDqrxshgGCDkE1A1K5jubgy'
        'xQiLPdR61FHq39K7seCMI0cGTNKc7CK96p1q98NJr9tqcAAYAqFqOqO7Za7luXK4YuePxVTI7EjjAr0JjV90q5HlW7qK6IPeSfVi'
        'JpSW+onJrmCeTS7iTx5tyoFHlSMHjJzVV0JvqJY988/ak4PFLwopBIJwOaIDwz60vuCTyBScE/tHFOALtwT3oMZMQNpbhaSE3HsM'
        'CnQgBGO1cJC9hmhfkNS8TnhKRmu+HHxtXk+tc3kHOMCkM5520LYfV8j0kaKwGSfUCm9ij1r3HdmxSHlXsKzbFpHdpDZJyPQUhlBY'
        '/wCldBY+wrzFSeAfxSOx0MMOeRkUgqwOe3NSGUnzC/emmPHccVrBVDT+IefQ1x0UjJ7mlknn6jScnPPIoMKCe/vbm+umubqZpZGP'
        'JNLt0LoziRQVHAJ71CClfuakIdqcjB8jXQ1SpEU7dsUszxnKkhqW95I0Sq7HjtUYnJODz614he5OTQaQVYos0rl5CTSAoBOKTknz'
        'wK83PANbobqJche3c0kbmGBSxGT+7gClEgDA4FDcZRG2Kj3IpGC/JP00o8/aktyK1ho9yBgUkjAzk0oc5GK7t47ZrWahosD2BNdA'
        'b/LxT0aheTya6c8ngA1txto3jjniugKvbmvFc5rwwFOP60rYaOEk8eVJ2n1pbcAc1wkYbduBxxQsNDbcdzxSG88nC15mbnAz71yK'
        'J52wDwO9a6MlY0WA47murznKYNPeCoUkjGO3vTRPoOaG6+gdtHv5MN2FIzg8Z/FeLhuO9J3KKAUdbLnJ70jCg966Tn9uaSygr70o'
        'aEuwBxxTbMQeOfSlhMdxzSSQPc1rNRf4+rnmlNzyeBSec5J/FcZiSM10N2RSo6WGMKKQSM4AyaWCq4LkD2pLFSMggD2pd66IbY+r'
        'E/UTjuaWhCcnvSQeMj8iuuh2l2O0eWaVyGUTxk3mknceSOK6u4YKjIrjuyr+oQOeKFpBqzwx3ry++AKRK/Gc4ApiCZJF3MSOcYrb'
        'ldG2urJYZRzSWYMeTimp51jjJ25AFNRyh4w5wu4Urml1MotkreoH0mkM49CTTEMqOGwexxXZZUTgkgeRpe+gvEZYpPwFmQltpr2S'
        'TtC9qiX1ysVsWXG719Ke0+5MsSuFHI5NI9RHwHWGQ+hRT+4lv9KQ+7nLZGe1KuGQRtJgDjypiGaN1Byctxj3pfSIX1D3MvIfZmVQ'
        'CoApsyOo44HtRRpHTcd1aCW9naDjJXGTj1qk6gtI4bt004vNCmF37eCfWhHPFujSwSSsrvEdlHc16Rh2Ixn0qNcXEkClQcPTluvj'
        'De/AxwPWmWeNtGeGSSZ0lTxSvBY87SF9aixSI100IHIqWrSSp4bNsUHNZ5VVpgjjd0NzIEI28k9zUd3IGBnJp1wY8sTuH+1MG5Vm'
        'KgDArLLHzC8TsWWIA8zik8d+a54sedgPJpJfJwo5A5FF5EZQY/JqoaeGZSwDD6h6VItdUjku2jB3Ljgiky6RdRRSXbRILaJAhBPO'
        'fUU30dpZvbqZ7b6vD5xXD6VN82dnoqtIb1K6llnKKpG3sPOlaRczM6wsCuQSdw5oi66jS3ksLh4lW4K7HVMY2ikfMRXPUQe+SOIQ'
        'wBYVXzyOM1Jaid34lHp49DtnY3FyniW6PIAOyimbxp1XEyEEcBcc0R9G+NFFb2xklimyzPtTIwTxVZ1tbXFpqImWR3PdOP60z1eT'
        'xYfRYJcIh3RWGAISY325wRXv8NdQXcEMcVmWaYbozvAzVTp95c6ncz2by5aQZyw5yKnW+oaml5BaNfSxtbLtjIc8VJ55tjLDHyE6'
        'z0/rmmWctxeQ5jhA8Ta2dppm40u4g6T/AIx4qESMCgQ5I+9XlzNBcJNbXN3NIJsGTdIcMat+lzpNlP4F0kc1sy7lQjIB9KHftc2H'
        '0e+nQGdEtIr7S7R7mVonlP1Fl4x61Cu/lYjLbq5xFLsBHmPWtAtNW0Z7uXTLfRDKrZKFSMKP9qr49Mt47ySQxZhfJVEXcRQ7yU+r'
        'D3aiuEVOo9PWFo1omnXEs0kwzKTyFFVfUmizwQwPbb543bgAc5o11fWdM0/wLWe0uLV1AIzFjcDSU6h0xJljKux25GRxRbaoVRTs'
        'C9Ksomsb2PUNOna4IDQMVOBUP5HUFidLeymUfatDutctTaO0do0ieeMZqs/xHaxxyeJp8kZHcOe9BS82HYU3TWnCbxI9ZtLox7f0'
        '/D458s1YaR0xi4Ms6soRyUjbz9KuINWjdYltrEMHxyOcVNuNSna4QpGFk7MWXgUlqxlB0OzTy2dk21Y1iwEkJBJ/FBbdOaz/ABF5'
        'NG1CT9Y+J4DjcOauL/UuppXZbSSxeKNs7GiOWqbp99rcVmHWFTeH0TFVUtiJ7dzBibQLq5m26lbpFMDt8SHOM+4pNv0pcNFvjuSy'
        'AE42kGr+8k6yZWmgWJWPIRkHP5rmmzdUyzr/ABCIWqDuVI5pt6mhXBxKKw6Nf53c8sg57hO9XCdFytbvK87ZDEbcc4pu+/xXZh7i'
        'G5+ZhZs7DgECuaDql9cSOs0k6OOWSVuPwazTatM0eHTQxL0dIwEe2QfRnJYYz6U9ddB2dpbR3JuhK7AbkDftqZr1reXd4l0moLBE'
        'ibWCycVXwCG3iWPxIrnLEmRpWLN7AUkWxpRRWa/0w1lEs2nr4xeTsn1YGKqNX0670nV4obna7zRiT6fQ1q3RK2z6kI5YnmBIAAGN'
        'v3FQ/iD0+lzrg1YSm1WAeCFK5LZ4GB+alLK1Omxu6TjaM61S9vr9RtQl5Gx4aDv9hTPSt1daRfXBZWVJ12lMYwa0j4eWEVvbXD3l'
        'sEu43JXxV5C4qHr9zerrVtFp/wAk3zSlZF8MHzrLIlwW7vdyuQTvFiuXHiSkkg8jnFdjhgtY4i2HlzhpCc0RNoiR3bWotg0pOMAd'
        '65DpdtG0kLwBWTup8jQc0x3HxKq06im095vlZsuRg5GeKia7qjX0IkSdiw5IznNEcOk2G5pBbJuYcnHenodIsTKEjtELscAAd6Da'
        '8CbbZm9k8xn8dP0znAYCpeoI8cRnSUGQ+fnR/NpdtbO0D2aIQeRjzqXoWk6Ze6rDbXFqjI5PGPPFFzS5Al4ATodpBcCKW/llkXGS'
        'E4OaJYk0hQMQXP08DL1aHSIheXUNjbLtgJyMdgKhjwSf2jNHemugKaCTRekjeaaNTsoEVXXCkyHJB8qc1bo3WdMsDqEgt1jiGQiS'
        'EYqog1i/t7UWsF3JFCOyqcCpry6hrOmyC5vbq4hQZdfFIAFCMnZmDuqavBJKiz2ou7gAA7uQvoMmqbX4kmt3kBgim4IWPy9qtbS2'
        '8EyxpEGjDZQucsRVjcW9uLeA/LRjI74ppyoEVaZmVrLdu4gEkocNkAZ8qsJJrmS4aOcM7MM4YUbRw26PvSCMN6haUY4S+8xIW9cc'
        '1NysZOkVmi3s0OlI0Emx04cMnAqY9xd3san5oQuOdyYO6rCNYjYyoVQDcDtx3pWnSLHMqJDFzwTt5po5DSRSakbgCN1vGRSeGU0a'
        '9Nz9Kt06g1bVE+cCnJaXDE0P6qsTv4KxLlshcr+2oqaIdQkhBkEYBwx2imnyuAR68kC7vPFml8C/mZATtw57Zq40yycQpK/iuWAO'
        'WYmmdY6aOh2pulkNwznABHYVa6XOrWUSFW37QDlTxQi2wypES71TRljuIZYpmmUcMpxihzQdUm1HUJbQ2sbxvkJIeCAKIuptHENu'
        'LlIyCf8Ame4NUejvBZ3qSMpEYBB20VCW1tM556qEJqEl8S+6N6XfXIr5rJ9ny8mx45G4c49PSjTpH4ez2U7zXtvHgf8ALUHIHrQB'
        '0/a3l7JqD6de3FrIiGRPCYgsfL71u3w/S7/w1aTX1zNNcPGC/idwahllkvbZbDOE1uSImkdMWlpfzXEdqUZ8bnPn9qpetumfntQs'
        'rsKQbeZXGTxjIzWi8etV+pwm6/SKnw8fUfaueScVd8nRGSk6aM7+Ic2lab1LaWTwi0acbvFZgBKB5Vlerzyw/ELTCq7oDdtHE6jg'
        'jBNX/wAUr/R+uNU0+4k1KezFmhX9MD6848/xUS0h6XhSy8S+nleykMkTE8lsY5r01oNQ2/8ALbfuZxvtDTRin3qXxRa3UlxB1vp7'
        'NARbHu/vUnVLG2n1u8kjniQMDwe+aTJr+jyFWed32sCAV7U9Hbx3jSaksdw0UvIITuKE9LnxRvJBr3mxavBldY8ik/5Oyui0tB/9'
        '/DirDRLa2tdUhme5ikCt2AyTUE6n0+jFMXBZTgjaeDXIta0WCaOWOO43I2f2mrQ7O1UlaxshPtPRwdSypfEF+tetbPTer7mK5iBi'
        '3jaWOAx9Kd0brTR7TVlvJHiZYxu2K4yCfKmfiJZ6N1JpV7Gto63En1xuyYwfWshi6CvUwwlU/djXLqtC8bXfS2PyPa7N1UdVjfo+'
        'LvV5o3HRuvdPtpLu+nhiAmLDAkBJyeBVHB1LpVzdt486QIWJJBBwKzyTpWYaVDBFGq3SSEvKWOGXyGKgno3Ut2TNEPya54xwx+++'
        'h6UtJqJv/R/U12XXelIRk6wze2Kn6f8AEHpyx0q5tUuDKZl2g8CsSPRuok/86Ku/4M1H/wDPF/eqbsH5v0I/Zuq/hX/yNVXq7Q1n'
        'DrcgIBgAsKZl6z0ovt+YUrnj6hxWX/4Mv/8A9iL+9Wl/0sJ9EsLS3t4IbqDd484JJlz2pZSwP736Drs7VL/af9jWNL6n6buNKuIz'
        'd2sUpYBHlkApH8W6fx9Wv6fnz/VFYv8A4KvyMfMRnHtTkHQ10+d93Gp8sL3oqWmSp5SEuy9e3a09fH+preo9Qafax/MRhprQttW5'
        'X/lsfZu1V/8AjPTBKjwSx8eRbzoq6K1iz0roK0t7/SzcRWQW3kwqlSR5gH14p3VusemZ7CSC16fEbycE+Eox9veob8S/fLw0Wo/h'
        'ra9/6g9q/U0VnaeLeWUtuTgeJIpUZPbk1Wr8QdLREQSxADuQ9S/ijq0XVPTg0WxSSNFkjdXlAz9I5FZX/gi7xk3cf2xVI5MDXrZf'
        '7+QZdnax+zpV83+prVp8UtNit2hmnglwcxMWyVobv/iIj3pkfViwJyFjAAFA/wDgi687qP8ApVtb9NND01d6W0FvJPPMsiXZX6kA'
        'x9I/p/c0Vk0y6ZX/AH8A/Z+u8dKvn/UMx8TNOEXy804mTbglnzk1UnrHQ9xJnTHkAaDh0Tc7ObmMEH/LTb9FXGSfm0H/AE0yzaZf'
        'ev8Av4E5dk6yfXSJ/H+pvXw01ZJ5zcaBsmjaEeLLIv07s/tFaBFrXUbSwLJd21vEWwwjTJxWI/Bm8h6WiuLTU7otCx3RkDgHzFGe'
        'pfEHTooJDaQSPKn/ACwwwGPrUJ5MDbudifZmsT2xwV7uhoGt/EXpLQ7w2OrdS/L3IAOxzg4qi1346dFW00MVprIuI24cxqW218y9'
        'R6ZqnUOuTanqd/EZpmycDhR5Afaqp+m5IT9V3EWVu/tWi9J+Mz7L7RX3X1X6m6HRNb8VS93p5IPO2IgYpH8D1cyNvvrYBv24j7UY'
        'NgtwAcUhwM9lFfRpteL+bPkWk/BfJAkmh63FMjx6pBlTlgYsgiraAdUQF0j15FgI+lBCPpq528ZAFc4YY2ge1Jkxwy+2r99lIZJ4'
        '/Y4BCXQdTkLP/Gl8dm3M4jHNSRouog8auduOf0xRNhRwFXd6YqPc3dtbsguZ4YTISEDuF3Ec4Ge9GoRjXRe8CeScr6v3A5NoupTc'
        'XGrvMoGANgHHpxSoOnjgKZRj7Vfx3FvcKDFNE4Pba3evGZYm2sPtUJ6fT5uWr+J36btLW6JOON18Afl6b/yyn+lJPTm7vIcgUTGZ'
        'AQWOM+1KIDDjvUvs/S/h+p2/4k7T/M+iBhOm4yM+Kw/FdfpxQvDkmiMMuCMcikM/BzWXZ2l/CK/2k7T/ADPogaTp/afqAYfelroU'
        'WCGcL5gVemTIPB4piadUUFsc9uaouzdL+Em/2l7T/M+iKyDSYI8gAEn/ADeVek0eJsHG0j/LU26vYLaBp5iI4kGWduMUGax8QZYL'
        'rwdI0k3iqMlpWKb/AGUf7mky6bQ4fbSQcXbXa+dtwm38gui0OM2rQGWXYzbiueCaR/AbaMfQuT71Y2N149tHK8bRl1DFCwJXIzjN'
        'LZwclQQD70/oOlf7pP7d7SX3j+hR3GiDvGEB96TFoK7f1NufariWUjICk496QJSVyM/mj6BpfwoH2/2n+a/oVJ0JAeNtSE0pjB4I'
        'ZNv25qVJcqhyc04l0uAQMg0/oWlS4giT7c7Sk+crK1tAjHAIINNN0/AeWFX3ihl/YwPlxUV5mMmwDgd80Fo9N02L5BfbXaPXvn8y'
        'ttunrBTiVN4PlmrGLpvQ5ADLbBv+ukq538jiptqVZeQKnl0mmS9lBx9ra+b5yv5kRukelpZDus1A/wD6Hiouo9D9JOwK2S49Q55q'
        '9jRAclRSLvwwMhRiox0+C+IlJa7VtetN/Me8WTGAFPvmm5p34yF47ZNSPBA5UKFFDur6YbwXZu13qP1YpA5BQLzgYxjtyPOvRlkr'
        'ojy4Ytz9Z0I6hv8AU7ZDPb3MalBu8Nn2qfYnBP2xzn71CbW9VWCKP5i2aVYvFkCNkhMgc98+mfWvdWW38O6Xil00kbJVmkDANu88'
        'ZPYeX2qx1KO0l0YTIEeGSBgAWGGZsEAgcH6gPOuScHOTfT3NndjnGEUqv3pEiyu9XdlMsVu3GS0ZJHPp/wCfmnb2Jr6Dw7q0ikAO'
        '5C6BtrDseaoYIrXp2yW/FnfSNcQeGWglBWJVGS+3uD6kZ8h5VO6anuYNHnF3JqNxJEQM3EQVSxJ+lD3YAYye3p510RmlCmjmnibn'
        'ujLx8qBq10Ui6uWt71LRZWZo0aTG5gcl1GeBlsY9z+DGylu/l1RlmJXjdIAWP586r9FtE+c3SW0RFszMm9Q7KWxja3ln6uKjT69q'
        'c2rSWVsBCLdsXTtB+mEIP1B9x5yD9OMjzxUsKhjjvjH4FtRvyy2OXx/8L0m9Y8oSPLinFe8XnJ48jQDa611BpGp2x1hrnwL6P63U'
        'bmRidoOf2oMANyAO/py9b6nrWn6vHeQahDqOlXN0bZm8Ihy5yQeO/I2jHGWBzjs8da5OnjJS7PUU2slhkZ7te6ZJPpSpJbjbkAn2'
        'C1Iim8SINnGe2R2rzNODhdrGutZYvwOF4pLqyFvv88xfSabmWWVQksCso5+pQaltLICCyf0qdpVjJqEpTftjUbnbHb0A9SaMssUr'
        'a4BHFJukwH6o0L5u3kksrY/N4+iMSYRj9jwKqen+l7tdrXMDRPuJlVmBKD0HPJI/FbhHpNoItsVurkHG915/vUnT9FhtV8S3tY0B'
        'AUkQgk5Pb3FeLmjp55Vk2/oz2sE9RjxPHu/UAYI2CLth2qAABnypc7uox4JOOxzRkml6LczPGWMLKfqaNsbTntgn7nt6V2bpFP5J'
        'jPGBkbW7/wBvtXoLW4/I896LJ5gM0kxUuIASPfvV109pzyE3d/AI4V5EbnG7vyfbiiK00y3tcRQwJ4w77wGY48/t9qltEoIldWQ7'
        'goCoBznmpZdZuVQVFcOj2u5uxj5PTihiFjbAAcboB9X3JqO2g2twGjS2jXYf5RswMetWo8dwyyBl+kcoMccgZzyO4pUcZOGjiEQ5'
        '2gkszHAzjnniuJZJLxOxwi+qBxNAi+Z8Izyhc8HjkffH96Z1DpZEYlbsxdv3gNjPvxROYSsfieA7McAgjkgcY45FcfMU2Cm1c4Iy'
        'SfIVRanIubJvT43xQK2/TUMYL3V48p3gDYMKf685/tUyTp/TTGAizIxwd3iHK+3bH9quvFijJeT9SR1B4Pb1x6VEFxEsWUUEEYQZ'
        'xgUJajJJ3YY6fHFVRSzdOK4f5bUZEHZSy7wT9+MeVDl7pOvxPIAi3G3Jyjjy78Gjn9RrdjJGqqBnaSAMVTanfeEmYRvKnH0vxnGQ'
        'Pv288EVSGsyQ5dP4E56XHP8AkCt31XbyWi3mn2N9co8ixbNnhuCTgHD4yM4GR6+1W0h8a3IJZWZc4x2JHY/6VRa7rsem2zT21mt0'
        'tr/ziX2JCAQCztg4ABz70qy6u0O9ju2t73d8mM3O1GwnOO5AyCex8670oV1OSW/wRB6quZIuhrxmTdJBCwbeN2Ap8x5Ej+5qysEt'
        '5NONvbWskNvHGska5ztZhv8Axjj+9CXVV3Bc6JNBdtPZQalclpjuw3hlWKN9Q4yVHHkV96NunDbLo9obdvFWSJW8QDmTgAEn1wBQ'
        'jGL4oecpRV34gXpWjrHa28mTbsHk8VDCCWA2xJgjkZJ3HHqaINUF9YaUqtfl5CQEiCgE4ByQc9/bkCoK3xTrKGyLM364VY2O3+QH'
        'P2yrGneuIGv7x7eAfq2liZUwf/UkkVEx+Fb+oqPdRaddeh0d7JTjfTqXWg3FvcWTXKKyiRyZAQQd/Yj+1MdQsJLVoYJiGnKRL9O7'
        'JJJKgfYVHE1nBM7fVFcG3+ajdSV8RR9J3J2yOM8f6VF1Pxp5v4Tpk3g3ERiRSVB2n9zsfxk+9PNJQ21ySxpyyuV8E9tStv4bdTGF'
        '2t0tgH3gsS5zuU+uOAal2gjVIYzBGJJR4wwv0jt29PKqbXIjp9jLa205kSONZpw5z9IfO32LZPH/ALTV3o8LR2iyJdvPG6jwgwAC'
        'p5DihD2+UbJXd8MmDeVI2gD2pLu4XaS34pwsMcd6bkJxhs+/rXVxfQ4efMaabBC5f7Yot6SjQ6WQjEszsWwPqzgYH9KEy+ThTz71'
        'b9PanHZJLHcsQjMpJDYA5xk+3bPsKjq43i4RbSSrJyFEePFRGkGB3UnHb/z70i41M+K8UQYDcqfSAN3uT9qlXCRKyCO6iY7BuCk8'
        'ffOKG+stVvNO0Zbuw01NVkaUIlvHKodmx5Dz9xx+a8aTVHrot7FITC7vcxhAh3PjG3nHJ7Y71W3XXEGm3ngRI98kY/dvXb5EhWBP'
        'PPbAr5z6x+Kmp6sz2USNp0SyEssc7q4btztIGe47c8VRaFrqLctINXms3PJYS43HnkkjHnSORj690vrLQtRIikcwSFhj5heWOcbd'
        '3bg+lWt84nkEmIpU4JZV8sc9ueMDz86+Vh1BrF14Sr1DHeMpwjQOhc+f8vJ/OaNOlfiPrumLHpl80C2aKVWNwRjywCeAfPnA4NZT'
        'DRtqMzX4bBh+jAcR5yewPP3zT8CrHIQ8agklEYL3H/bjyof0nXul7+OOWXUYElChv1SBtx22lT29PXFXkF5Z39nIba9imjK/vEg3'
        'YHbI++e4p7sAqMySFvEuCilirQ92OAPMdhnn3psx77sK05SNcFiDyCe2P7c+1O2zGR2ihkt2duAFkG5iO2QPKkXR/WRXeMPxkg8n'
        'HcY/FZsNCRaQrM7LceIY1KtxzkHgc/1pdlBEsBaeEElsIzMORux5+3/nnTVyDNbvBFIjyRgrnfjz7Y8yTUe9Z52lkiQbUCofqwys'
        'MYCjn0JoOQKE39/JEjGazj+gtsYZOQMY57Z9azu78e+uJUigkKgCd2dskue7k+eBgADgf6FHUN9HPPsUhYmZc4YAgkAFcd9xoVlk'
        'eO0+ciSOJnjcMmd2IxnaoPbOB+M4pepilg/h2t2kmjap4twu4OfpZd2w4DbgPPaOe/JHlVRcX+jdM6zcRXSw2kbSJLJcFWZ50KqA'
        'AFBLHKruzxwfWi630m+g1rSYpsCGaFojcONiqWYFTu8yWBH/AGq6uOmrOx6kiv8AUFGopcW7xOSMwwFMMuFH1EnDnPkRXqelQgqb'
        '5OT0eUnx0ozDqS60vqvTZJ7W9R72CXMG5SCVxu5U84GOeOMGrHozUpbSyXTlWTwwvixbVZtuTyvb/Nu7USdRavo2haFaTuLDTpRZ'
        'hpYY4BHcSAgbQpUZTk/3bv2oU6SuYHuLK3g+as7i5aaVVkjyqyLjKtntuVgPLlKXFn35bSHyYFHDtsr+p/E0vqabqCWaTKTQjw1c'
        'DOUO4A+WcDP4rsd1qt31XdJBI8sEpVnEjAOUj5VQB+0/USOOTjnmpvVlr/E9StLa5s54rmW8Y2nip4SljGEByfPcoOPbzBpvSNNv'
        'Y9f1K/ktoo7d0EENxI6/8O8ZkKhyf5DsZfxTd+o5HHpybuXLEpeNBBqvyKaV874wDt4eyT+d8MGVB5nPbHvVNLY39l42v2d5JLdi'
        'OSbwSF2ySLkyDJ5244X025HfFVdrPb9Q6xHLYyvbvbyyyrEsniRArg4I8v52474GMc0U9T6jp9haXWjlZo57izd4mVQy/WdpBP8A'
        'LhQSOMHbirTywnFzfQ54Yp42oLqxfwdhtutZta1oSTK0k0O6PA+hVXaCB7qCfPvj1rS9O6MsrTQI7WOWSKUMwWQtvzliRkZGOCPt'
        'VD8HNNtumNQ1nRNPlY2JhtrqCdmDySbw43ZHG3CjjjkVo7Xkqy/qF5eMECPgf9q8p6rK3aO54MaVUZprFjdabevayOjuhwHT9rD1'
        'FV5aYt9TKVPf1rQ+q7e0v7MuH8K5Q5y45Pt96obPQLcssl9NK2f/AE4FH92P+1eli10dlz4Z52XRS31DoC0CXEs3hxR7m5wFBYn8'
        'VySKR43jm8MqwIYYrTLA6dpluy2VjDAcfVIz5b8mqXqF9JvJj4luXnjGXFupBI8sk8fnvSvtOF048G+zZJcS5M+m63t9M18aLrqv'
        'GkoEkN7u+mRcAbW44weNw/NEty0dxHAukz7C8gZ8uSxXvj0x/tQX8T+j5ur7G2Tp42kd9aykxo058R88Hk9gMAnFGHSPRSaTpEVv'
        'rd2dZukIPzEpKKOOVAByygg43eVePkcZybx9D1calGKU+p8wdeaf/C+qtVtGJCx3T7CR3BOR/Y1oH/0o9D6H1J1Bqur6zbx338KM'
        'JtoJG/TLsWO5l7PgLwDxz2rXdR6J6a1S/e51PpnTrwnhyQUbtgcgj+tOdC9LaV0NcXt309aXNp81tE0UtwZYiFJx3GQRuI71k2ly'
        'jbbfBoU2g6JqFs+nXmg6ZLBygxbpyDnzABH3GDQ/rHwy6Zu4VSKK+sFRAkfy9wcLjtw+7PNT7DqKW3H1acxDttyuc/c8VNt+roFu'
        'VSTT74RfzEx5I/GckfapucWUSZjPVXSXXHTNrcXYMeu6aiMZZIUImVMd2jPOB5lCfXFVvSNx0/qtpbeHqmn/AD8gyqLcBQV54K9s'
        'qO+e/JzWz9T9e6XaQlbO3uJHH1l3UxhVHcnPYeWTXxX8Z0ttP+IF5LbNGiXoF1LFGAFhdyTtx2Bxgn3JqkBJKj6wSDStGtFaOaxt'
        '3ZcmQThcZIzj/X3xVTedd6TaXbLEbvUZAOcOQiduecD74r5U0Xqueyt5Wty4unXatyr7iB6EMCPyMGrG1621Wa0GnW0TyTSDBkiL'
        'lzznKopAB7+R9aZgTPp1fibYIYri+aWKGIgBpFwpHooBJ/6jn2q0sOttH1m7aLR9Q8WRmzPDGGGwnPG4jvgffz4r566YfUUu4oP4'
        'fr4Q53TXAdRvOOdoKnHuSfetO0Gx1azaKQfrfTtxChjcITyN2CT/ANJ8s0tmYW3r3M2quJFSMQgbN2XbODjHueBnknmk31tBcxQy'
        'BVZQoB3xqWdQeM89s4/v2qbMph0OOV7GRZ4nCRRbsnBPfd5jHnVJdaxdafqnCwnx95m8MY7dhjnzzzn/ALU65aFZcdeKmrtFok/j'
        '29pNNEZJQAq53ZXjOcg7Sfaqjq7Vllsba/S9a2u9PYC5WNmwcAhmXHOARkEA5Bqd8Vt0axxLFcXl8ACllbbdyqWw0mP3YHPbtny5'
        'oL606ySLTY3sNMgtQuIjEWCz7cAk+COylcjOSG3Y58kk6dnVjjaQxZ/Dt+o9a0/UhfsLE2sTqJX8RJJYmHGM/Wv7vTsPLud6XYI/'
        'WWuwXLQpcTSM9o1uwAjH0fS2AAHJ8NvuM1W/CW/j1jpSx035V42hWeSHapCopYBGH3JI49DV31/Lb6LoVp1Jb2jE2lzHLdEMQTGo'
        'KkcDJJ3bR5ce1UT4Qj9poHuptfhsOp9GuNYMcdxpOpePdR7DmQ+CUXac+uTjnjae1EXTeoaTd6NHNp4j3zqnzKNyQwVlZCDwGyzD'
        'PbJJrLvi7qaavcnVbHHh3UPhs6yHh9v6bYPPY/u9qv8Ap7qK/wBL6Ct9Rnt28Z0Cs55Z5UAX93bcdpwf60id2mPJeqqK7pnS9OsP'
        'ijePeRzxR3sciLPEAUklTPOcYHcru88EUfW/S9vJ1Bc9QQq88cVutlEisBGoUEv9PG4bivfPKZGKzDU9Tn6fbRJrkgR2tzJKZd+D'
        'JLlS6emeWxwRge1ar0hrF/JoFhIA8RKPMQwG5t7Fs8eX1f6Zoxm2trYk4KPropejLi7tdVGgX2n3vzunh/l7xtpRbZjlBuByeSQA'
        'QSMfejhoJ2kPjGRWf+Yg8/8AxVhHqNm8XFkBeBR9aABckd+exqx0x2lmaWRcKvBLHIOPKg0JdsHxYvbvmRFLnscftp1PEkBzgMv7'
        'cLw35FWgv1a6dJLVJQnA4HHHtXIbxBJ8tIscCgYQgcD/AM96RuQyUQcvk1Fp+GZYc84HK8eXkfWmm6bGoWpnt5xJMrZ3yNhRx+7a'
        'O7Z/HbjyojNxHu2RNvbftPGeB3IPbj2p8iWMRG3W3+s4ba21iPOk5YeAX0HRHsZjK6lZt23cH3BFHmCfMk89uPzVlcviZYHCvvGc'
        '7wMf35pi/v5YtWOnSlI3MYnRlO7cMhWGfUH881DuJ2c5KDenYgYOD/39K6ccajwSk7ZaC13uVfard1+of3FeuLVkiUyYUoeGBGGz'
        '6HzpiyLbsiUNKy5XjIAqVeXGWVTEdmzaUOO32pgDKQLLIsjqFwNoywyc+RHmKVdxJJDDNHIqAqd+TyMZ7ffmnbhbclVhjB3DcuGJ'
        'CnOe3rTECiKw+URQuxt2AuRk/wCmTz+aSSVhRC8WeXbErlQuBuLf2NMT28EpaO4igkiI3HcNwbHbuPSnp5XV5HwCSvI2nOft6Vy4'
        'G5QQjbQoRsDzI/8ADT7UCyl1Dobou/dZbzpbRZ5d36g+VRM/dlHnmnv4LpMdpDbafbtpPgIEiksGETIMdgcf386nu25CMFVIBySP'
        '6ZpZtQ1vEElVH7IXzjHB796Hdpm3AdqXTuv7fCg651dVXJKXVlb3HJOeTtUjnHY1KtrTqe3tCLnUNPvtiZE3yslue3mu5w/2470S'
        'SYMDDLZ3bSwK8j0x5/evGNpH4zGH7MGLA8Zx7Y8s+VDukHcZPq8fxZg1WbUNr31nDxFFYyxIp9AYpBk+mAfcVGk660eTCa/dy6Rq'
        'oBjNjcW0ifuwOMjLZ9ffjFa5Nvjt4yrZZT4j54XA9CR60D/Ecr/AZ/Gt45TKVAWZQQoDclfQjy88n81trQOAs1q2v5/DW1it7fUp'
        'lXx9RMx3KVVR9JI5OB24z50Ea50jozXmuXd7dssdiyPEHnba1xK3735wF3YG0AetXUmstYR3qSlpltZikyeJukij7Ag+annz49aD'
        'Ooep4rqTNrqdoyXhjUuX2hijhlD54HII5oZI7eWWxybddC4+C11bWE7Pf6mwv7maW3S1Cu4jVTv5J4xvJbAwPqP3GodZnxemrnTJ'
        'PDiV0zJJtBGNwbt6+/l+aw7p6GW0v5tSRLqzhgCXMhjZcsoY8oeQGCkHHmBWidP3Npf6XFJqE890yzbG8SXiTdl0Lgd8fV7HA4pc'
        'bbj7h80UpWmAWt6Or9L3iWFoZYzEPlGI5SJAxdQ3AB3buO54z5U503erffDSx0+K/gX5y7iieKQkbCpLuQT/ADYUg8Ef7y+pdcsL'
        'KwvbGz1WO58CcXMj2+GMbSBtyE4/m+o498Gsq6e1xre/S4s3uJILSKSZ7cqSoc8KAccZyuSRwM9qV8PgZcq2F/WWr2um2Wp6Vc3N'
        'ndzXcKPbwxnLmbIAKFTgYzgnAJBINXnw0v8ArS0t4oba5tLu0kmZYbW9Dx7sE7/DIBX14J9OMVC1BOnNHstPlSx0fXY7SzKKlvGr'
        'TxzJHtUnsdjMwcEc5Ue5qf0bNHYaTbXWq3+qWmjzIfDsrqRB4bM+c8YIJxnJw1PjW52TyOlRrHT2om6kjg1CwudNlb6wZGR4Tg+T'
        'qeCB6gURvqMNsiWiyLMJCCkiMDgHtgjg1h/W2t+CyLpeqXdvdLECls5DJKCMEZP34BzWWa11Pd2F0j6Zr+q6aTJtuH06Mbc9ifD3'
        'bHYnzG0+9XeF7bs51LmqPq9dTgW7kibxMqcOCoIHsfzUW6lM8gkBKqAV2kcL6/k9hXzvbdXdT6UkeqS/EWw1G0vIjIIIdMRbhyDg'
        '+KrD9M9yeTn3zWhfCvqjVNX6bj1zVBM8SGRGnLR8sFywCqoIHIIzn/eubet21lu7lt3+AfQ3jI5W2H0ZwAOCR2J57d8cVKXWrqKV'
        'WXLxAeGqswGMZ4OP7A0OdPa9p+t+HLCzJ4jgJK0TBAe37gMVL1rVuntIYWzanDLfhhk/yhu+GAHbkc8Zqs4wXDJxtlrdald3Gy4s'
        '9KgupUm3uWkEalDxgNjGR6Hv6igVvijBaapPY3ehX8NzasVniufpbI7bSuVOR25x/rRJF1pptuxbU1mgZ5GEhKhVhUDiQjglDyAQ'
        'Cc+VUMvTmgdcS/xfS1SCBrgi6vzOzz3D+giyVjByMbuf/aKgk5P1SrVLktI/if0a1q1xPdXcMpT6YTaOHJ/6cg45zzj3qJZfEDRr'
        'rX7K1nhu7O3uFMcEswTBYkYB5OAe3Ptmsq1+3QdaX2g6BYOPBmNqiKC8j4YKSx7kE5J8sY7Vr/S3TWj6fevcCENBLJJElq/6kQ5G'
        'SNwJ8gRz51vX3ULGuQsiuGa4SSDdMqkA44yeOAe1SYd3iyo1sRuJQq3bcSMYzQN1Rb9LaKjMJbvSLyWQ7GsWZHJGMsFGBtwAfvQC'
        '/Vl1cdaWsU9/Jqsdk6HT0IIkuJd2VDAKE3b8Y3FSwwOCarPh0ZRTXqs2m5ltXvWszewR3KLuNv4yhlHqRnI/3rs8phje3VvDXsxx'
        'yx4znPcdqAum+idNeyj1q+a7ub+7keaeSV18SIYLYATsVIwcnyPFR9Qj6qh6fEeka3HHeC6ke3adBOLuBhlfF3LlTnsV+3Plt7Xg'
        'Bwo0MxHeHAZMKDgEYHb148/Omyy7ok25RQcFfP3/ACKzKy6668sL+Gy17oeS7QyBXmt4RtPvlH244zk9qtLP4r9ENfXFpdS3VjcF'
        'tq+NGQGA77crj1GabvI3QNjqw4t3zsbaAVBxuA4HmD/55U8wUyon0+Gz7fZQeB27CqvTNa0nVW8HTdRtLiTIdbcsFlYcHdtI7fbP'
        'rVrIk8uMRs24fVuOMccYJ96e0LQ3ftbMvh2yuTH9DMeCT3yFHbt25rMfi5NM88NkpO2GN5NrZJ7jYTnv2J/NazZW0UUT3E3hkW0e'
        'VQD97Y4J57Z8vasu63lvbwTvJbyfMNglxJwwP8pxzjJP9TQs1H//2Q=='
    ),
    'song_sparrow_10.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAwEBAQEBAQAAAAAAAAAABAUGBwMCAQAI/8QARBAAAgEDAgQDBQUGAwYG'
        'AwAAAQIDAAQRBSEGEjFBE1FhBxQicYEjMkKRoRVSYrHB0RYzcggkY4KS8CVDU6Lh8TRzsv/EABgBAAMBAQAAAAAAAAAAAAAAAAEC'
        'AwAE/8QAIxEAAgMBAAMBAQEAAwEAAAAAAAECESExAxJBIjJRBGFxof/aAAwDAQACEQMRAD8Az2TUVuuIJY/DHiEYaQ9cV8tjMhkh'
        'VgI1Y/WlcCvBr0zH72cE0+0238XnUkZLZoNjDLh2wM9wJT8bE9TWlRWpSMRYwFUD9KnOE7Lw5Iwo71WTzBNgPikflFBcA3YruLTD'
        'HlGa+2sDqTkEU6S1EamSchVAySa7wGw5BKo8XPTyoxTYlidmkUbKcedA307K/wAR7dKZa5fSjKxIqKPSpi7lZ252Yk/OmaCdvF5n'
        '6GiOYDB2IHalDSlATQ/7QYScgauPyNpjwjZRvqUUKCS55Uto/icqpJx8hTCaPw7kudwTgHGNqmpFluNNuFj/AMxomCg9yRTfhrUf'
        '2zwkmoMI0uLchLpC/wASOMAjHX1+tdMJOUWmK1WjhZgEwTnap2IJaa3O0Ma5lGSfWvc9/gcqk14tl8d+cH4sUIGYFrMsnic3MTv2'
        'oa0lYg96ZanbkrgrvQllb8rKD3NTk22EJtLXxZRkU7905VUAbfKuum26LjI7UyMaUyjhrIvidOS2YHyqY4PXm1Ik9Q1WXGKoIWGR'
        '0qW4IQftRs/vUtaP2JYcIwLpXG+o6URyw6pAL6AducbOB/Onl/aETEgd6B4wX3G20fiOJcyabdqJCP8A0ZCFf+lOeIpFEI5GxzsA'
        'CO4plHTS5Z1sYsxg0DxPZC40u4i65Qn8t6Mt7qGC2LM2FUZJofVtSiSzDQgTPLsgHfNO9JtA/D9zBdaXBHE6u4TDAdsVE+0XliU4'
        '65qi4ZsZNBFwxxMs7F2UbMpPYUq4oh0bUnEU09zFK52XlzU5+NzWDRdDP2XqV0k3jfdJ5R61Z3PhtFmQ4HYdzUPpt7ZaVYW2lweI'
        '8qDOVGxqu0tpLiISSwtnGzNV2lHGI2fzZwnYXeowyXw+0AlJc96p9AgZdReN1IBGRn51y4XsrvR+HVnTlCXIz16Hzo7hy4eOcz3a'
        'iSQbKO1ZxsLZofCluEYyyDEar1PnTLxoJLmGG2hLyZLczdBSSyvPB0iS6nbAc5Cjyp1ws63lsL/BAfZAR0FJHAMNltVkObglz5Z2'
        'rzcrGkQ5QAB5UZdtGqZ5hmlF3LnIztQcq6ZIWaviTpSCcKqtk0/u8FdutTOrFwxCih7jpHJ2VozuKR3MvLc4BokSS4Ix0pVOzNcE'
        'dDmpS0pFFVo05YBM5JG1OLrh+9s9Kl4tsG51fMOo2MUZLTRr1lXH40GD6jNIdETlRWPUVqPAd9HYWFxe3RlaDmaFlQ/EGdfvY77V'
        'fxq8JSdMgba2E6rLHIJI3AZGU5DA9CKN0uF/GZGGOXpSyKM8PcSWtoLp/wBkXszR2yeFlY3OTyc2fhGdx16kVSW8UkOpy5T4PM0F'
        '9BJf4EXENtLHy3PwtjZxSq5sXt3STGYydmFMtQ5nBUDJHUZoKC8ntvs7hM27HGG7UEvZ6BYEQT8mDXi91IRkDnALdBnrXHWRb29v'
        '49tcxunKW5VOSPSpq3nW5YXM9whdh8K91FOk0E9cS3ZeNizUt4KJ9/Lds0Tqjaf4f20kjeYArnw1cR+8uLO2+EH4eY4JpZLR1w0q'
        '+gW/0G5sJcGO4hZPlkbfrUfp+vNe2kTXUrqtq5gIZQGLIoB+e+d6d3ep31jpxuJbRWhUZJQ5IqU4KisL/UtQvHfEYl8WIN/xN/5g'
        '0t1IZbFjz3y61CNoET3eFgQS27EUz0m2j0rTcSSPKIlLBmGSBXMlIpT4VuZD15jsK8+86hNdIs4jWEdo+tFJ/WTenjVNWsIZ3b9o'
        'QzLyJlY2DFGO+NvMb1G8S8RMspFmnKTtzsN/pQPCOmLJZ628iHxo9WlRmxv8IUdtq5ajaLNcIImLNzYK96Dk1wdRSelBwGZp7lZZ'
        'CXc75brWu6Yn2I5j2qI4M0jwI425cbVcwgxKPKlUmuitWYvxLauvDVi8YZIzj7vQUt4cgmmk5AxYs2BWh2cFlqHDraNMBzrFzoTU'
        '9wtaW+mXknvMys6EkKK6pL8WJd4NtYgIW206PYEAuewFUOmynw0tLJR4aqBzCkdpa3mqXbXc58OEnCKOpFVGmxpbxqqADFczdYgt'
        'HZrZYIGd2Lse57VLX1+q3Xhg5OardSbmtSB5VmGoTvHqrZB61D/kSaot4oplSiiWMZ70Bf2YZs4orTJhJB64rjdSsGIzRTyw+og1'
        'C0ESllWpW8yt19avL5ea2ZsVA6k4/aIXyNM3YIui34UtxMilhVlDEYrSZY8BMAsPM9BipPhKT7BewxVZosiXtzPDuAsTYbzPlXTD'
        'GiUib1iyj1XTbqwAKyr8cUmMiOUbqfzpbZatJqdjbXMgdJOUpcpn7kqnDD89/rVTqHJbeHFHgc0asSO5IqNuY/duJpII3VItVw/L'
        '/wAdfL/Uv6gUG/00aOoMMgaYyrJIreYbrS++eW5fw2nmkwdl5qOnAROVRjG1IdX1aHRIpLqdebmQogwTzMcADbzz/OltpBStjjTJ'
        'Etrv3iG3RR/6bZYfLej7v9mXzq8CC2uGOCmMA0Np/hPHz8pTIzyk7j0rstqsj8wG+af4CxHxJp1zACXjbHYgbGhOC2JvChGCDVLq'
        'd/d2lsVyJVA2EgzigtDS3vJf2jaqFlU4njHb1pZR/wACp4U3FNwLThOfK8zyLyIPMmkfAmivperRWl5h3lsFlGR3DEf1o+7c6zxL'
        'Y6SuTFbjxpvLbpTbWysHFmlyJsohkhfbbcZXf6GptbYVLqGl5aR+6mUBgQcHyI9KTqo8XmzinenzC6kmsy64nTC5IGHHQ7/lSCVJ'
        'EuGVwVZWwVPY0zeWKuImuD35DxKQM51qfA/5UpJIjnXudfhy3amvs3sZbo65qxnlNpc6hN7vHzfB97DPjzJAH0rvcWmNTzy96TWk'
        'VlSbL7hV3FjH4pDMo6jyprcTikWhsY7YDPaipncjNZ22BERxBePY3zrGeQRphmHrQns+so7+6ub6RzLzyHlJ8hReuospu+ZeYuG+'
        'tNPZ9bx2WmxRhQrNuR+tXc/y6ES0p/AWGNUUDAFeI5VV+tdrlsrkUkupGE3wkioJBaHk8wlh5QfnUTr9kBdCQCqK2lIXc0s1aQvk'
        'IpY57Cl8kfZFfE6OOlKVjxiud0czFep9K7WjLFASzAE9BQd3evbjKRDnP4jRUUloG74Ez2E508u5CKR3NZzrcVvFqS8snMS258qt'
        'p7i4urYmSVm26Z2FQ2qwkXZznrTylFLCcU7K3h9wIuRXPKardE1RNNvEcxh1ZTGAem461HcORkQr8qpIYwzovQk4FGMrpgYZxUQs'
        '1o6YK+7quR3x5+u9S+p24vVMTHlfZo3HVGG4Iqg1kuba3Dg4VmA+LOOm1JUIOoqKeWSYqZ4iuDe2XvEiCOdWMVzH+7IO/wAmG4+d'
        'SvEii613S9NO4V2vHGOybL+pqs1WKSyuG1C3jMkcqCG5iUZJH4XHqpP5VFcLSXGp6vqOqXpiM0EaWmIwQFIyTse52NTTsoljZT6e'
        'X5sZ2p/pyMTjfFIdJdfHwTneq60jBiBQdRTOyM2xTxFbgwk9yKl+HJ30zVGkyQjbOD3FWWuRuLcnHbaonVFkSMCMfbSsFT5mhxph'
        'isK72fjN5faqwOZ5CqZ/dFM+LFUy2eo4PNbzAMAeqnY124e0h7TT40aU8wQZ27111K194tZrdj95CB8+1JJ6NHotnilLEqzLg5BB'
        'wQRR/GOoW44LvuIfgiu7S3ZZoyceI3L8LL6k4zXzS3S506CYkMWQZPr0NSntWgWTTtJtMBhdapBAy92Vj8Q9Nhj60/EaEf1RS8D6'
        'WNL4C0uxx9slsrS/62HM36k0FexYuwcd6rdSiSCMNBvbyjmiPp5fMdKmL3mWUN13rPcCujnSIw3KtNb20bwAUFI9GuVEwBOKrGlR'
        'rTsdqFXoTM9QizdJ3Vm3x3r3p1ysevLbo3wqp28q/CG4W6HIQ8IBLA9qV6QY24llnklEahsdaeMcwDZfc/MtLLmJvHycKD3NEi9V'
        'IWeGFmx3NBqstw3iXLbZ2UUqjXQ3fAyL3dBgkyN5DpSrWr6XkaOCFEX97vTSNAqERgDagby3X3c4HxdTTfMAu6INLfErc5LNnqaI'
        'v0MpxjNKrq492uTynBoi1vjIRzEVyt3jOj/waWduBCQR2qV1+1UXJIXG9XGnBXiz5ikWv2TGfKjO9GSwRdPPD0WUUUdrGoppDW8l'
        '1F/ukj8skwBzEfw/Q9K78P2hSNcjeiOJLe3lsHt7q3E8Mg5XUiq+NOiclp71h4pNBSW2AaLnEiyKQQwO21TNvIx1dF7GgrVJODYw'
        'WllutBvjyFWGTbnP9+lN2tgt1bXsTBoJN1cdx/Q1eSt2TSoayxRzW7RSKGVhysPMGs64FSA2GuCFgJE1Fy6E78n3QflkY+orSPgM'
        'RZSdt6yPhGWaGCfUgqmFbyYSADd42Pxg/LYj1FRWMsv5ZaaHAGn3zkmtA0235LcHGTWdaHerbag1u+7KfhbsyncH6jetP0qVJLQE'
        'b5qsSUkLNXtTKmO1SdjaJqHHNvaKMxWo538s9qvtWCQ2bzvsFUsfpUh7OB41xeaow+O4lPKf4R0oS6ZcNB8FETIx0pDqD8lxnt3p'
        'yztjrSHWEbxVHNyhmAZsdAT1qTQE9BOH2xHNARgRzMFHoTmknHLw/wCKeErSeRET39pyWYAZRDjr6mnrwPpnF82kB/GTw0mEoGAy'
        'kHBx67UnvLKPX/azDbSRrLa6DZidwVB+3kPwfkBn6U0+JFY9subJEu4ZdOkJHNmSEj8L9x8jUrciQTtGwxykgiqIiazvEuIsgoc0'
        'HxTbKmpi7QYiulEwx2J6j861WrBeiy0jKyhsnNO1u5BCVz2pdbrzLkdq6KWZggySTWYUC2/grbTzSHlUAjNZi0/h8R+OspaLnyd9'
        'qvuJGlg017ZRnxs4x5DrUBHYpPa3GHCyKdh3q8OCM1Sxvor+2Uw/5QxkgbGjLKNbidlxsATUvwc5h0SKAndRin+m3QgkmY/uGoP+'
        'kgrgXaIrQzOx2U4FCz4KEd6+3U4tNHjVtnlPN+dAJdK4wDk+lH2SqIVFvST4lQrKWApTpbymcKWJBO1UnEVu8kbEKelS2nu6XXIw'
        'wQa5vKv1ZaLw0XRyRGCfKus8YkuMkAg0r0m4ZYgCe1FCd2nHlmqfCaelBp9sFXYdq631tlObGfMUToqeJGKYTWw5CDVE6QrJS4hh'
        'uLD3K5ihkgBb4H6YPX+lS2k6bq2m6q+lw2c99o1wxKSYz4Pl8XQEflV9c6dDJGHe5hh52ITx42A5h5NjGenQ1G8Q8UcM6VoUEfF+'
        'nahLC00mJEd8oc4GVzgVW7WASXXwE4puptD4e1W5aWKaOGJkSSNwRzH4RnHQ7/pS/RtDFjwbp8DqRK1uJJR/E/xH+eKn+NHsbu8s'
        'NH0aybwNRkgkRhk80e5Kk9Cdq03UxywchUqAowCO2Nv0qaj8Gk0lhndqjIiwZPj2QwpY7yw57+qn9PlWqcITq9ogyScVl2ruYLxb'
        'mHl8SNuYA9G8wfQ1b8I3IiVQjAoVBGDnAIzijEWSHfH0rfsC4t4jh5V5c56A0FwnbLp9lb24G4UZ+dAe+jWdXvUEmYbdlXHmac2X'
        'xSKPKi6sBRLCZUBSuVxpxuBysKYaaMRj5UwhVMknFBgoz+9N3a8Scl26NyWUjRSHpyqGK5+WP0rj7FIHvdBv+JroZudbvXn5sY+y'
        'T4EHy2J+tDe328Ok6TJfQsVke0eBCB++eU/o1U3CYg0jhjSrCJWSKCziVQ6lW+6Dkg7jOc0aQ7bqxlqcSLHnbNLLkLfaMUYfaWr/'
        'APtP/Yrpq16ssZCtQNpdSWAW4kgM1vKfBm5RuobYNny/+K1rgEgW2Ro3ZcbUt1wzXE6aJaOUmukLXEineCDoT6M33R/zHtT/AF+W'
        '20fTp7+6DGOLACr96VicKi+rHAFBcP6dNDDJcXZV7+7fxbhl6A4wEX+FR8I+We9D1+BTrRLxZJ4bc5+4vwipfSYI7gzOv3gSfnVJ'
        'xn/+HsM/FUrpFwbHxZxCZWYBI48455D91fl3J8gadYgPWUmgS27WLGCZWKPysudwflT3RbdrhpZH/wApfvE1I6JaNZ6oBJIGeYZk'
        'YDALdz8qq5rlorcWlrllfeQii427FvKFvEd093c5jJES7LXzQoi8mW7V6ngMoEaDfqTRelqsLAHr0qEor2TKReUe9UiXwW23xUHd'
        'xol9zjbervWJOWFm9Kza+nkbUCpz97NJ5tHgsK/TR9kCT8qYWqsbgAjApZpA5oUBO9Ni4SRaVrBOMstDZVC48qfW8S3MqRF+TmOM'
        '8uevpUfosx5lAqt0h41mEk0ayIg5/iOy43z9MVWDvBWF+1ezGkeznUJI3R7eC1JBC5ZWG/MMd81i9hxRccWcG6bxCNKW75ZHtddS'
        'CE/7uQCVlAA6HAzWkjiprYW2nXt1FNbyTzwagVy3gPI3JF16oud/WpzQOHr32cafq13BdzR6dZSEyxsF5bjKgkg/uk+fQ5rocrWI'
        'EYpdZkg9y0n2icMyW6vNafFNHEZC3JzkqAM9ACc4rQpNe07ULCW2hEdrf2E7RLDI+DMnUYz1znbFQOvammve1qK6SJI4hp4KopBw'
        'uObIxsTvT7inRGutKW+togb6FQ4ON3A3wf6UiwaQr4tWG6ZbmwUggZlTuDXDh/UruKW5QRoGezPu/KgDMy7lSe564+de5fFurUav'
        'AG7JdJ5E9yP+96Cs4brUNYlj00kG2j8RMjYv1AoNbZlLKL7gbTBZ6Es9wT73cuZJM+tPZLmXTmS5nhSWyLqHkXZoFO3MR+IZ3z2z'
        '5Uv0fVYdT0iO6iCh0HJIg/CwG4ozTtSjllMDqGBHKytuCD1FTcknTFin0sIm8KPIOQehFc/fmLH0pLYyNoVgIbp5JtNjz4U+CzQr'
        '2Vx1IHQN+dHT8kkazQkMjgMrDoR501hoz72tahHqnGXC3DzmOSEXIurpf4VIwp+ZHStC4hlWSwt7wDfeGX/UOn6VjmpZl9q+tags'
        'Pi+5C1iA6YLHJ/lWsxmW5WXTCqmO5BKkndZF3H57it8QzXwmLnUSLhYt+Umq3RjHPYtDIAUkXBqPntGW6w6kEGvUepT3l1LoEXJF'
        'bsivc3QfBSA9V9C5DDOdhvQjJNm9WNYCuv3Ed9JIJrDTX8O1AG00uN5yPLGy/U9xRk8rRMLZWMckq5LnpGvnnzPaulqyJOPdYvDt'
        'njEKsFAUMo+DA8tsV6s7aeMme+czXLHLM34fQeX0qyzSb0juMHkgg8UrzRMPyNKuG7IXN9HdyKRGikQxkdz95z6nGB5D5131bW4t'
        'a0a1MVs0HiBfEDHOXPXHkKP0IeHOikbYoPiG+nPiG18OSNkRuoIYfqKPi1bSE92t4L1GnnyvhYw6kDO4O4+fSi9YVJLcqBkjBFLN'
        'Pi0u4uxpt6FYyHm8To0XyPUVSOxJvGNbKEkPOBzK230r77viTxFrtPJLoBH7Qk980xBn32NRzRL/AMVR1H8Q+oo+QQr4VzCyS282'
        'CHQ5Ug9walOPxjp/RHqMZlhIOagdRtOW/JIxg7Vr95ZxGLmUDBqD1y2jW76DINR8iKRYJp8rxgY7UxhmMtwoJoOGPJCp3php9kyX'
        'is2alrY+FboFvjDE9qqNPAFwmxwdsZ6kjFJ9GRBAPPFMBKYnBB3BzXXGNI52zNvZfMdW07W7N3kn92mmtbxn+7cp4jMskZ68yf8A'
        'fai/aBqOr2Wp3OpTwR3Gm2kcSXWUb/ebc4+IgHBZd87dqu57LT7a0na0tkjiuSZZFj+Ec5I5jt0OQM1A8XXV5a2WqTC3e9hEThYZ'
        'J/gdNiebPQjLY9AK6Ekok/ZuRmGjHTk474hvoXVLeFfdbVVI51VtgybdsD86v9H1m2vdJaPmQ3EDEM6bKwwO34T6flWa8L8PjU+G'
        '7nXI1M8yXkrzW8T4lMW3xJ5lSCcd6ZwTTWMSXGnxWVxa3eWNxErKZsdVJ/8ALYHtjr6VzNtadFJ4GSXP7L10SYD2VyxLx7dcYI+R'
        'ql4W0hNOnLxS+NBOTJFL3Zc9D6joahbyA6tqq3HOy2/LvH0aN+4P86qvZ/fXVvdTaPqc3LB4mbfmGAp88/xbUFJTVIEouPRtyQ6H'
        'xFLEEYWuqnIYfdSQZOD+f6+lEWcfJfltxvRvEmnC904wZ5XyGjf91h0NJeGNSe/MvvHL7xbsEmAO4bfqMDB23pXFSMmaLp8ytbhG'
        'AIxgg9KnbQHQOI4tLNwW0rUFY2ofrDMu5jz+6V3A7YNNLaTljXHehuJLD9o6dyBhHPGwlgkYf5ci7q3y8/Qms0zRM10Q211xpxDq'
        'LMrGfWYbONgM/CMsT5dVXf8AvWrshEuVJB6gjsayD2czmK/1bTEdWuW1ZZTGoBXkycuPQY/I1rz3JEgggQS3LAFUPQA929PTqaLV'
        'pUF42BcRzWlvD+0rjGAnM8SH42cHHKB/ESMfOp3gi3N899Pcsstw14/ixr91GH3Vx6DAFHTQ29/eXl5Jck2mnwyyiZulxOq7sP4E'
        'GVUeeTSrS7q90vheK3hDR6neqJZXQfFC0xyB/rOdvIDPlQxStIZK40W8Qmvbg2djJGsVnkzzFefmmxtGo6EDbJ89vOu0DSXGntLK'
        'cyBiDXXQIINLsYbOLoi4J8z3P515UCN7mPOxfmHyNWRJsynV5YU4mgtFCoskh5QOnMe1UlmgSQNUDxvFO2nza1CsgltLpXidRlSA'
        'dwfI9xVlpupR6hpttfwn7O4jDgeR7j6HNCT/AFQUsscy8rqcdxik2j2UceoTSnJfPU0bFP8AlX6ykHvrryEZOcnoaaH1CSHNnDJO'
        '78xHgMpRo2GVYHqCKHt9Dk0eNl0K493hYkmxuMvbH/T3j+n5U2sxy4HY0Yygj0pW76FYIItXYye4XMTWd0R8MErbP/8Arfo49Ovp'
        'UrrtwskviJhge4ORVZxZZW9xpjQzIGRmGFPY+Y8qh2Z9LuDbajHLNpZ+7Kq5e1+g+8n61DyRbLQaO+jzB5FB65qutIkJVjvSjTNM'
        'gto5JBNHMsoDwOp2dT3B706s0kIAZSu29P44Uic3bGVpOyz8qfcHWu8l3hiTS+B+VjynavrsCN96qxRtDeRtbPA5+Ft1PkagPaPe'
        'SWVnctJkRzQyKvrlTVAxcPsTis89qmqt/h2a1eHxp4pCUfukeMfzIFZSdUzKO2TfBMsj8JabHZTiC7t3mkVyPM5/t+ZozS9UtY5Z'
        'b3wPDlkiE09vFtHLhuVmC9nGc7etfPZDwlJq/Dr3IdopkbllDEqMEbY9cURrVvpMN+9lai4vbu3Ul/CxhfNcnGfOpyY66FWGiXmq'
        'LNqmiqJBZyCO8hxh+Q9Gx3Hr27001LRnv9FaSIEXduvMo7sOpHz2yPWhuArxLaWWe1uzi7TwnYEqwYdj5MPI9adPdXP7RMC6iLC8'
        'Zsr4sYeGfz2O6n5Gl/imN/WBPCmsDUNFSOV2eVF+83f0pBrUi6JxCmsJ8NvcfZ3YzgZ7Mf7+lB3st/w9rhnuraGG3u5D/lSkxnbc'
        'jy33/vTq9sjq2nPDMrCC7iIV8ZGD0NM3xoVL4y30lkmgV1YMhGVYdCKLug3KBjas89kes3NrPc8Ka3JjULJ8Rlj/AJi4ByPMHOR9'
        'at7++NyRb2UnJGGxLcL/APynr/F2+fRqoUx3R86R7SNUlijjjhjdovE5Phh525Vc+gJx9a024NzD7vpFnO3izqZL65feRYztnPZ3'
        '6DyAOO1RnCS2n+K+KffYDOjoII4G3M5Z9l3337nyyasuE5mM+o2esxpBqNvIZ5Sp+GeM7IyE9R0QDscVGN0i86uz1xHJFpfD1w8F'
        'qkrLbOILbHwFVG5b+EdMdzt50PboLm80+RVYQJbC7Z2OTJNIOp9RvRGqWsraXd+9yBpntpDIxPQBThR6DOP/ALoThB1PD1pBMPDn'
        'NuJIt9mHUr8wDmnhraFl/I7iuyJQuTRsc6tdiJiOdkz64pTbRhpgSfWg74z6tcpeWQddOhQxm4WQoZyevLj8Ix97uennVLoklZzs'
        '9Itp+EDYXqGRLhcyAHB8/wCdQ/B4n0TWbnhC9clebxrCQ7B1Pb64/MVob296sAje5A5RjAWpXj3hu6u9JXVLCWQ6ppx8aEjqwG5X'
        '8v1oS12NH/BzBGqnDEAn1okBUKyhlwNjvSXhp7bifRYdXiZ1kJ5Z0Vscrjrt5HrRFyIOaSG0yYYW5JJCc88g6qvovc+Zx500H+hJ'
        'LCkj1TTlnS2a+QS8vMVUFio9cbD5HeiZtRs4X5DdwyDsyNkH+1LdEsbC4twvuwE4Xmwy/eHmD/SmEGnW3iDFvH/00JdCuAXEOqWT'
        'aeGWTmIcbAUk1C4s7hOQyAFh3FV11YxY5Rbx4/00A9raxgmS3Vj2+GkGRBPdjRgojc3FkXzJaqcFM9Xj8j6dDVnHrcFvpiT2c4vr'
        'V9i5XDof3WHY0rvdPt5LzxDCg8hiu4sgD7zYSC2ugMdMxyD9117j9a0X6PDPejLS9d0WY8k0ht2/iG1HFrJ3b3fUIHGMj4hSG3Nh'
        'qEzwS20VpfopaS1kxhwOrRt+JfTqKV6ho9mx5hEY++UJFOpWhXGitTDZIdWHoc1mHtpuI7Pg+7SVgJr66VYV78ifeb5ZIFMl0+8t'
        '5Oex1GZV/dY5qI9oTyT20cWtxc8xuRFbTr2B3II7g7VloVVmj2msw8N+zKyS2+0eW1jhgC9TJy759c5/Kkui6PJp+kvcTkvd3J8S'
        'RiN96ul0HTEs7e18LaOTxsMcnmxuT+deNVsk5CMDBoSirMnZnC6Re+NLdaRMYbxh8UecLOB+Fs7A+TdqeC3g4m0SK4t5JVKMVltm'
        'PxQSr1XzBB6eYptptsiXYUjoa58QaZf6ZrQ4l0CFZz4YW/sBt7yo6Mv8YoJYFuxBI7y2j6NrhLW5P2czbFG7HPY/oe+KY8KWNzaX'
        'vujXcqKBiOVG288Mp2/MEeVPxBpPF+kC80uVSzDG43Vu6uOxqCa81jhu7RZ7aSfTAcJIo5mhGex/Enp/Kg168Ctejn2h6brJhTiX'
        'TrSNnssCaaEYM8AOSWTr8JyQRsRn0ql0rXrLVuGIdVtSkcUcf2iLsEIGSPljeqjhq7g1TSIryKSKaKReU8o2PmCD0+VZRd2MfA+t'
        '6vpE0kraPqMBW0XJCoW8z2I3A7YrJpqjU7BvZDC+o69qet3LyrGk2E+LaR8nr8h/MVpfFEGozQxxaaUgu4ohOkjqPjbOUjOR93qT'
        '6kVJ+zqOy03hUSxwuisTOskgHhSszYVebOx2Gc9smriRp2bxJ3Lytuzefy9PKkq2PdEVxnq1tc8LnXIXkiu2ja1urQ9VK7sp7jDY'
        '+YNUFpaD9j2J5QJIoY3UKdshRtUVxqJBresyJBG8It/DuAi7nK/f9WHf0pxZavdzaTYW9lNEl1eIkVvLIwCR/DlpCfJVBb6DzrRm'
        'kzSi2kN1jj128ks4GI0tG5Ll0OPHbvEp/dH4iOvTzqj1p9P0jQ5G1W5jsofDxEvLljjphRuANt+lIdPluIJorPRNXcxwO/JNLbRl'
        'HXqSowM7k77VO8UajdQw6ro+rJC97d+7+BcKWLXMbyBXyWzgg7co2Gaqn/8ASTWl7q9sLWC0k5i3ixBiSc0HayAyKD0r1rmqWlxa'
        '2FtFI800EXLIIoyy837vN0yO4ztQJW6ZQrj3UY88yf2H60ZUuG1me8Tm54D4l1AabJGthrCZjXm3gkY4JA6AjcjPnVzHYwWeh8ke'
        'CFjChh+71AH8/Ukk0t9ouhwazwVc29tEGurf/eIT1ZmA337kjNc/ZzrCa9wRCHbNxbr4EwPXIGx/L+VKnTGkriUPDrpPYeG+RynK'
        'uDgqfMGnNnM0jvHIB48Y5iVGzr+8P6ipvh1jC0sbgrg7ZpkzGQhkkMcitzI46qR/30pvJ/QsOUx1M6mPJpHf3KiQKSNqNkmae0eR'
        'VCSpgSxjoP4h/Cf0qT1r3ie3uI4JmhmZCI5FO6t2P51Jy0NHa7vIZI1nhkWSNxlWU5BFfNPuwzjepxLqSbR7OcZGYVDDyIGD+oNF'
        'ae7BgxNH6H4U89ha6nH4dwmeXdHU4ZD5qexoG2W5tXTRtYm8V2yLO9fpOOyOezjt50Zo1yDJg+VMtTtYNQsXtriISRsNwf5+ho+v'
        '1dF9vjJh0a2uCrAqVPQ9qy72xSyvq+kW6HJkuOZV9MqP71qc3Os66bqU+bj7tndvsJh2jc9nHY96yf2irJN7SdPsipV7W2LEN1B+'
        'I/2ploKo/oSfw1MTooCyRIRk5JOB3oLVyDBnNctG1H3rQdMDgoxhXfmBB2A38q4X8xMTLTTRok9cXzQ3IIOMUy07WvGu0jQlmJwA'
        'OpNJru2kuJgkaFnZsKoG5PlXhxHYxvawOGnJ5bi4B2GTjw0+uxPfp0pE8C0deK7X3S+/xDoPiNDE2dTtonKpcruGKAdSBnJ6H6ZN'
        'O3ufFPDjpZXQWG6h+wlTHw+Xy32IpVpyPHCqg7fpikVpK3BGrmWIzS6FezZeNVybSViPiH8JPateh6h97Kb+4stSk0i+J5nJT4j0'
        'dc7fPY/lS7/aPiiGjWF63345mjHqGXy+lF6ysMPE1pqsBJR3S4zGBvjqR9P50B/tHTQzcOaeinnR7gspXqRjYj8601dM0HtBvs1u'
        'o9Q4X0u2aNfCtoOVo2Q4LknJOevw4H5+dP5YbrRopJdMxPaorObOZzhcb/Zt+H/Scj5Up9mtjdS6BpyxKXYwh2c7ADzJ6Dam/EXE'
        'g03S7h9ChivriIcr3kgzCjeUY/EfXoKV9GFWiR6e0d5f6kbmaaWTxHsBEVuDzDIDA/dXH4unlU9wTYG91/UbK4iY2ulNyW0LNzBP'
        'EPMR6/dA+lVVtoi3EZu7m4uG1GUc7Xhf7UMR0z+7/D0pH7P7i5tuOuJbTUeRbl/DlPKMK2Nsj0IINIl+tHteror9TSSwtoNQhGPB'
        'cI4A6o2388Un9qcumwrw1qswBMWoI/zjI+L8jyn6VSy3VtNbyW0pyjqVYDyrPOPtS0+81HTtIe0nlMdu/JuCOdxj7uOucYNVlJJJ'
        'CeOPszTr+eKw4PsDMkcYsyVIUABc9NvOos662oXIKoxXOAoIH5mqi4FrqvCesW2n3HvTnDrE64mUgdOXv8xUXwTpzKczAhwdww3F'
        'JKTbqPDevqrfSntBIY+b3ZgO+JQf5gVm9i8XCPtNuLKOT/w/VxlBgqI3PQY7b7fWtgWNRakAYwKyv2r6JLf2yXloB71aP4oGN3A6'
        'jP60zQFJDi1u7j9qyKzYXsKcQ3XJuxqP0bURe21regjnkQc/qaYT3TR55jtR8kqpk1hTG/5WWSMjnU7eR8wfQ0vmnhvYLjUrG2db'
        'BZBHHK0gOWxk4XH3OwY9cVMave80bWJZ0LgeP2IU7hB6nv5DbvsZwxxFHYX/AIN3GJtPuFEVxEenL2I8iOoqSno6qgWwdOe9scjM'
        'NyzqPJH+IfqT+VdVblcKK98S2sWmcZ3sNuQYJrSGWJgchlJfBoNGZnp2wtaUWkufeUzkDNV8WPD23zUZp7CMK0v0A60ya8u5iE8T'
        'wIvIdTTpYI+nTiWytriJorl15XGwzuPUetY5FYe9+2R9PudQea490dVmnb77cgKjPbyFbTBBDJssRkb95tzWLcYW3J7cZYR8PPb9'
        'AN1+yz/SmzqMuUa7w/aXKcOQs0cgks5DFIp6DuNvzom4SPJ5j97cA9d6A9mWp3us6bc6feRCUpyBZ9ssAcYPmQM0RdX0K6jJBZqT'
        'cICCzj/JTOAw7FiOnl9KLpoAv1ELbSSW1vIPecYncf8Algj7gP7x7nt086S6jb40O/PhOyrA5Kx9enb5dfpRupGNH5VGMetMNDsl'
        'aItMMpKCu567biptoOhywxx2SFcY5ARjyxSm4SK6MtrMiyQyAq6HoQaN0eFhwrpzSSsZPdU5s/KlUDul4S+2TsaHWNwmrh7vhi9i'
        '4ev5ZJNNnJfS71zkxk9YXJ8tsUN7Wr+X/D2lQSn7SFnyvNnB+HGa1DiTRtKveD5Y+I1xDcri1hT/ADnk/CyDqMHfNY1xK13Np1rp'
        'F54bNp7GFGkkHPKrNkE+Z/tRclXqGKbdmmaB7/daZa290TbWC20ca28ZIMgAGS5zvk74pjrUMbjT7GNQqyXAJVRsEXfpXiHKRRom'
        'wRQo+QFfExPxMhDBjbW3xAHOGY9/pQVI0m30oBjl5RgYqB4vkfSPaHpOqRoGXUYjZSZOBnIwc/UflVo5CJ400ixR5wGY9T5AdSfQ'
        'ZNTXtDV4tHTWY9LjnNhIssbXLNzDcDmEYONv4iflQ+BXRjL4+eeJCVQ/ExOFHzJ2FJNLtref2i/tIzxXRtkLiCE8x5sYDFjhQAfU'
        'n0pvGkGqWdtfGZ7uKRRIjSHIAPkOg+gpLwpI8mt6xej4gZFjU48sn+1FtOaDG1GTLKwFhFDqcljN8F3aeLaEtliV68p8x3HUVw4e'
        '1h5yE1SJLl8bTZ5Jf+rofr+dINCMmnM0csbSaZKctgb2znYOp7V3jnj988G6li8QsViu49opyDj4v3W/TPlSRio6wyl7LDQbmGN7'
        'J7iyZpIcfEPxxejD+vSovVQ6SmKfBVvuP5+hp5pE89vIG5njkTb1+XyrprNrYarblDLFa3DnlTJxHI3p+6f0qvwiZpdxW+luGAEc'
        'BfmIH4c9cUbJNb2NzFcajbPPGTmGBX5fGxvzZx9wbZPfIFK+ILK+habTb9GSRSVBb9PpQPBy3fElxPptuWkvbSMRhZH6qudl+p6U'
        'nn/lGirs/a5f/tDUTOsCW64wEQkgbk9T86+RISFI38qDuopobjlkidWyRgjFMbCCVR4jfAnXc1z+BNy0SLsbRiPWOIPd1OboaVHy'
        '/FsHWR8rj1BB/wDuuSAQ7Lu4O5Pavt4I47/RNYsVMDTQTWsxB6yRsrqfqGP5Uy1pFlhj1OBFCXBxIo6JIOo+vUV0J0XaumF6SOaA'
        'SE8znrmu12x2BOCPKhdEZmTlA+lMb23ZouYbEU61CPB9wtJb8oL4HzrFONCG/wBoK+kBUJ7sQp9PCH960SC4miHMj/d6isa4onnt'
        'vaXdT+LJNcywOqliM4dcAD5A/pUo+0ZaUyUTWvZHHFMLiZZFeMc8UbjcHO5b13GB6CmesWfgRXT2o5ZZIy0ZPV8ZOP8A4pX7GFjt'
        '7OOKQlIlmUMfJc71RccQiKeUW6sjRyGSPDZClTt+Y7etWSqJK02RZaWW2Rph9oQM1x1m51Kx1Cw1m3zNbW0D280CJlwGyQw/5sU+'
        'WIapYLqEMYQq3h3EY/A/Y/I0RZcPXGpxSWrRFo5UKMMdjtU+jLDppiSrw3p1uVcyC2jUg7nPKK6xpBpvjtJaLe6lCFMdrIwWNSeh'
        'c98eQr3wTqJ07h99O90lueIrB3suWRcRw8mwlY+qkEDv2oNbVLWR5JGMtzM3NNMw+J2/oPIUHlUMkts5S2l418+raldG9vpB8UhG'
        'yD91B+EVmHtKjK8S21xHErGQICxPTB32+VbJy80W/TFY/wC2K4zqtjamSOJUjPJhPiOWJOSN8bD5UFrGfDU9NV5QqIGck4AAya88'
        'PQmLVNWuo5La6muJxko3NHFgY+Ij7zfwjYdz2oO0u2ksuSNjFZheaTDfFcEDOWI6J5L+flR/AUJXRRKVC+NIz4A8ztRigSY3gtQs'
        'njTSvPcEY8WTGQPJQNlHoK6arax3ulXFjKPhniaPf1GK7uVjwCQGPQHvXOduUDm2OMinW4TZE+zq9A4QutPuTi50l3t5lPUAZwf+'
        '/KuvA0SLoV5eynlDXMjsfJVApDxlLBw7xzHfC58K01mEx3caYJDD8RXrvtv86q9CtZIuCpUgQyKUc8yjI+NsAn8xSJNO/wDCj2H/'
        'AKNOGriK21S2t5wGgnfwpkYfC6kHY/XFDrZWVvq2ocMXpRFedms5HGwJ35CfIigrsFQJE+/EQ5P161ScU6ZbapfW13yFfHgQlge+'
        'Nj86diK6AtONxYyfsu9LBVblgkc7of3GPkexrxd3DwOysuSpwymjo5BeSLouqKTfIn2FwBtOnkfI/wBaD1l44itregxXGfDR2X/M'
        'x0B8mH60s/yrXAbJ/wDZNaldJNcxS3MXjRRsQ0bscY8s9RUddTxaZ7UZm0Ey28V/GrW45ssrkDbI6/ECPrVVfQsJJLfHxEkVDcX6'
        'feaZHZ6siOvu1wOVwOnf+YreXYpG8bftRpOuR3uu+z6DVbqJY7yxvDDO/LgsjDOfzqMMzuBHzHkXpWq8Na5pOvaJcWerxrayarAA'
        'LpTiKSTGVYj8LeZ71BTaBd2WpPaXkLxuhxg9D6jzFLxYB504NMycPtOI1kNheRXIRjgENmJh/wC8flVHw8YppJdKnYCO6+EMfwuP'
        'usKGGkXE2kavZWkQknnsJBGmerLhxj1+HagbRzJFBeQk4cLIp8sjNMMng5sYpbGdo3TDoxV19RThJPeRyqMHuDQeuXMFzbwawj8p'
        'kxHcjGySDofqP5V0glkMSlVY5GxA6/Lzpogkfp7fw0YoDWMcWLEntds2aNeWZFUjPkuP51tskLxxOlzdw2zH8EjfafRBlv0rFeN/'
        'Atva3YSwO8ixRq5MkWPiAO2G9fOnYEax7OrYxaI0zoyB2JQkYyPMVS8a28k8kF2kKJDcQIcnY5Kjt570Dwbc2V7w5a6jbxyu4IE6'
        'ynmOM7g+m56Va6nHb6r7Pprl18GbT32wM7AfCflij61oE0Z9oOnXdhP79bQmaA8sd1APvSqe4HmNqc8f3N9aXFhofDuoe5XUn2l2'
        '6ICYo/4j2z5Dc0Xo2pLZwyzWyRy6kwMduh6Rg9XPoNvyqbm5bGe6trmZ55rmXxVn2DSM3UN6D9BioPMKx3gmvZX4f1oap75dT6fK'
        'FhvJZG5mJI/zmHbDYHyOKcznx2zjA2II6EEZBHoc0KwDrJFIoeNwVZWGQR3BpXw4bixvpdDYF7UL41lKx3UDrET6Z29KyleBrLKW'
        'd1jt8Ag9tjnesj9swtY7rTi2GuZFYLjqu5ABHQg5rQ727XxFMbFQZF8QA7MOm/1IrO/aUo1LjvQtPj5CzmNX5Mk45s7+RG/50Yx1'
        'CtllKw07hUrIEQiAIGBwCcb/AF61a8NRLFo1pGBjEK7fSoLjWJ3s009Tz5vYkIPUI2f6itB034YFUHoMUzpMXp11BVkTlK8wpZNb'
        'oiQiAyoIkCMviEj0O5phPJhuXOSa6RxIXXxB8J2b5GpxesdrCB9o9nBa6fHrK21rPeCQLzTwhsLg7Dof1qn4DupV4fs7Z1gjS8sH'
        'kIEZOCrKcYznvmlHtXjjteHmSY8xEnwY7nHX+tH8ITmKG3kLQqLCwVU8RSwLcgdtv+kUVN+zD6/hH//Z'
    ),
    'song_sparrow_11.jpg': (
        '...'
    ),
    'chipping_sparrow_00.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHQAAAgMBAQEBAQAAAAAAAAAABQYDBAcCAQgACf/EAEQQAAIBBAAEBAQEAwUF'
        'BwUBAAECAwAEBREGEiExE0FRYQcicYEUI5GhFTKxM0JSYsEkgqLR4QgWU3KS8PFDY4OTstL/xAAZAQADAQEBAAAAAAAAAAAAAAAC'
        'AwQBAAX/xAAsEQACAgICAQIGAQQDAAAAAAAAAQIRAyESMUEEEwUiMlFhgUJxobHw0eHx/9oADAMBAAIRAxEAPwDElYHoe9VMlDzR'
        'E6r8ZCGHXqKtMBJD9RXlwPRnsVWR1lPSi+KnYMBUF5EEftX600JBqjsn6HGA+JAD36VFPDsdq9xDbj5SauSR7BoHIamCEUKdHdWY'
        'T0rm5XlbdcxuN1sWaX45CBrfSplIIqjGS3ftVqI66A0Zh5MuxVOYaBoiwBFUrqNiu9UqUfJwOkGzuvAu6kZST2rqJDvdJySoxnAR'
        'lbmU6I7EVpHw/wAzBkrRsNk2Dqy6XmpAZQF61+xzXEd8ktuxUqd7FL+tUMw5XjlYxcRcMy4bKOVQ/hnO4yO2qquwjj9OlaZhL214'
        'lwIx94FF2F0CazvOYyS0vbi2nJURb+4psJuSp9jMkFF3Hpi7eXHPIevSqkvl3qOaRfGYL23UiaOqpSpEzZcx97PEphMrKjDWwNkU'
        'SxcKzATSxxxRw75GL6kk9xo7oVClWXMhhWPnKqrcw0BsGhcfsamOVhdLpjCUUtvUPKdFAO3Sv14ZZ0S3lZIYmbeuXYk3/QjW6W7D'
        'MPCGEvKOUAAg9h5H6ffyooZ4bkLBcSMzRgMgLEcp9vShTdh2eX8cRsxHEhmhBJIHXmIGuxA1SdxRFfMpmnRQquCQi6HUd6cbq8eS'
        '+isY+ZIlcO7cu+Y+33/pUmTgiZmsyV8Rh/ZgbPp50aBaszOzv5bYjl0QD2NMWP4oZFALlWHluguXxE9oTIq7QsdKFJI/6UIWaS3m'
        'jnibkljYOja7EHYP61ksUZ9oBtrRedzz7onjU8ROp6eQoRok9ATuiWLk5ejkge1ctDSLM2oUF03Qu0f59Uw5ECSI8hoBJCyPvWq2'
        '10Kkt2M+HlUkDdMHJzJvdJWInKyqCabIJ9xcvtSctpnJlPIKASfOhXiFXIo3coHG9UIuoTzEgdazHI2yxDJsDrVqNxrqaFxMyGrA'
        'dj3PSqEEFoHDaFTTxgg0Mt5SCNGiUTM6dfT0rJGoGyw/mdB3ruKA81Xlg5mII61MkJXpqosp1A1oGZxEB1J6UxYXBEQB2A2e9eYu'
        'y/NEjjrTKk8cNtoEDQrI2jYx2Abm5fDTLcxNpk9KKSz2/FmLdrYD8Vy/MfOkvjPKJJzorb6+VQ/DPiH+E5xFmP5Ep02/KmtSS5Ls'
        'bCUW+EumBL21ltb6WCVSHRtGpIV7VqHxX4aiuLJM7jkBUjb68xWYW50adhyrLG0KzYpYpcWXYV0BU7AAVBG61+ll6aposp35HfZU'
        'jsVJBruxyjOY440AmiXl79GHr186qX0nQ9aCzswlDIxDA7BB6isqzLo0qxnjaLnS4Xxtfysfm79den3qPEvNNkhLdheQbHOQe32+'
        '1JWNyHdLpecnXKxOh96bcbcclpIdMBI5+YkMGceuvTuPtQ9BJ2FL64DOskdqiPPthHzb5F9f/fvWfZrFNJLPLaq/Mu3aPW/Pro+f'
        '0pvFxbi5WVy87aIMfN1bfTde3gVr205/DDEBZH1oL07Eg+Q8/P8AqSdHPYf4U4S4ewdo13xSjT3LxsIolbSg66Gs0kaNLqURghOc'
        '8oJ8t03fFzMyPmhbxy9Yfl2tIYYu29mpsClK5t9lXqFGDUF4DCOrIPPYqlfx9N6qayIC9T28qkvFDIfenNUyetAi2fkmFN+J/NRS'
        'aTSpWT702cPTAoEJ60OTcRaQWaEqda2Kq3VpvZAoysXMgPSvHiQqd1BycWc9CpcwsnlUS8xFGr6AbOqH+GFPSrcU7QSOIDo9aOY3'
        '59DdBHBBB/pRXCSbcA02XQSDL2utFRUlvBzOSfKr0ERkTeq4dlgY0ikw6o6YrDF70sZzLOqsiuaIZS8ZgQp0KWMhEXBauikjpfgB'
        '3bvNLtjvZqSCEjTDpquo4iX61fig+WtnOuhDNU+F/EUWWwz4DJEFlXSk+YpB40w0mFzUsXKfBdiyHy1Q3HXM+Ov0uYGKsjbOvOtU'
        'ycVvxpwos0QUXMa/cGkQftz5LplsZL1GLi/qRkok6d6/F+bpuobuKS2uZIJRp0OjXEcoDgMelX3rRGu9lm9xd1+A/GrGzW/ygyDs'
        'Cd6B9+h/b1pflQiQhgditZw1lc5XgK+lieMxQKTMhbTbXlOx67AT9DSPe4wygso0R2NJx5rk0/A3JipJoBRxk0RsLr8MpV4hL/4Z'
        'La5D61XMbRsUYaIrx+1PaTE9DDZhNlI0a3lZgTttFft3170cigjuo1F1PFJOF+WTm5QAB2159P3rP7PISWNyHVtoSA4I8vY+VOkE'
        'qnka5dDHyjwnjYMeU9epHl5+tA7QSaEvL3bXuSmuHYku5PU1Ap11FQht96v4iwu8pex2VjbvcXEh0iKOp/6e9aqjH8I63OX5Z7bS'
        '9hTJYcO5zJWwltMRfTIR0ZYTy/r2rWOBfh/hOGVhvcmgvMgF2XcbjiP+UftvuaP5nMQxnme6kRSflBOt+uhXjeo+KLlWNX+T3fTf'
        'CJSjeR0fPuX4Tz2NhW5u8PeQxMejtEdb+vlVHEzmKZfSt8fijc6Jbs0SqNbK9x9Ks3+DxfG7QWcVpZT3raea5S35HTXcFl0Bv6E0'
        'z0/r3P5ZxE+r+Gez80ZaMvsLgMgLGuppgSdHpTXlfhZnILlosWnirzEKsjBWOvMex8qWMpgM3iJRHk8dcW5JIBZdqdehHQ06cDyp'
        'IGXLb7VRm6CrlwOXYPeqE7eRo8XQKODo96s4tuW4Ue9Ug3lUsDBZAx77qpdBI0TFHxY1jHc9qmyWNVFLHqdUJ4au9PGzdgaYb6Tm'
        'Q76g9jSktjHtCPlIGRuYfymg9116Uw8QSqiKuxstQKZeYhgKKUa2dB2UBb6cMB3q8kY5Klij2nWvx6DlqbI9Cpxoozxdd6otwXn5'
        'MNlVjdz+HlOmB7CqhXY7ULvY9P31o96Xhl81MXGTi7Q9fFThy3NpHn8aeeN+snLWYPonvWxcJZjHZLDDh135zKmuvcGs6424cvOH'
        'sk0M0TCFjuN9dCK9CE1fEdkjrmume8H5mezlubbxmWGaB1dfIj/Wr6EH5GBGvWl7hfJx4fP21/LaR3cStqSGQkKynoe3/wAUa4ty'
        'NvHxXdGzhaG1l5WjRu46Df77pMo1l0u0MUrwq/D/AMlPKWauSyjrQOcFSVboRTPC6yp1IO6E5mzLAvGOop0JeGIkvIuXjaB1VOzy'
        'U9jOGQ/JzbZfP9fKprkNsgjRHehs4+aqEk+xTYfdAra1W5/BHh0Y/h987JDu6uVJZmXpFD5fQtrf6VkPD2O/i/E+OxhBK3NykbAd'
        '9E9dfbdfR/EuRscP8NbGCGMyZDLt8kUMgcqicpAOv5delef6uEsuPhEv9Dkhiyc5AbN8QRjnkckwxeSdS7eXSl21nuMlkDdSI3PI'
        'OaGHe2I9/wDlRk/Dniy5u7KO4svAiuuVi07dIuYb6geY9K2TgDg/hLBII0y1rdZLWp5A6lge3KBvp1rzMHoW5U9Ht+o+J44RuO2K'
        'HAnwzGVgiyeVm5Vm0/gxL0A9CdVodvbYPgPDyXBQpbvIEQomyCfLf2otkszw3g7GSW8yUYjiU7XnAAA8vrWLfEr4mw8SZLE4yxR4'
        'sTHdxvIrJytJpumtg7XfX3r14+mxQSrs8LJ6vNmb5PRqeMvbjJkyy40WMZPNGS6l2XyJA7d+26jzlul9bTQv1ZNBjrbAdx7Ci0EI'
        'EYMRGuXoAneqlwhWQqqkAjbEt3NOgvuSyf2Mk4pw+LjyypeWT3du46q8Zcr6nakEUuSfDrA5WaSLHZmzhnJPhxrdhv8AdIPUH/3u'
        'tM4qzH8Jfx8hE89uW14VuvO8nsFB7VSYcMcWY/ma2hsrhFIXxI1WaLfblJAANeZkyOEnXRfHGpRVmLZz4b5/GTLEqw3LyDmSKNx4'
        'jD2HY/Ymla9sbyxmMd5bTW7g61KhU7+9bxf8S3fBZNjnpbrPYaUciiRXLwr/AJmUBSO/v1onZ4bhzinEtJgL2SK3nG3tGmFzA5Hq'
        'sh2h91Ipi9S0rfQLwfYxnh4cypy6O6P5BZkszy0Ry/CkXD1yJb23lwsZblEkjGa1Y+zjZTfo361amtd2o5ijqw2rowZWHqCOhpqm'
        'ntC+D6MuyDO9wfEPWuIxsD2olxDbCK6ZtdjVCMjoR51T9SFrTJivKvTzqJoySTqrMa70KkEeu4qDPpBT2iiE13oZkU7nfaikzBd0'
        'Jv5Bymk4U+VkwOxeQnxeVjvbdtPG29VtomsPiLwoIrkol6ifIB33WC3B/MJFEuHM7fYW+jubaVgFPVd9DXo5Mbkrj2Pw5VH5ZdMg'
        'y2EyFhl2xctvIbjn5UVV2W9NUycXYW+vMHjswsQM8cLR3kY1uMp5n19fvThkYbXjPhmbKW7OmViHy+GdNrzHSl/hvw5uF72zkmFt'
        'JDEYriF97f0ZR5HW/rS3kk6b7XY32kriun0J2JvNERuenkaNlVmjpScGKQrv5lJHSjmIvgyhHPUdBT5R8oni/DBefxpXcka9fMet'
        'LEsfzndaVdKk0RBpQzGOMcpkQfKe9HjmDOPk1n4OcEXZy1pxPkwbWKFxJZo/ys7eT9ewHl6/StmTIWtpkxfyY6DL3NrITZyWsXW2'
        'jI0R16evUHZ3WP8ACfGeS4u40ssfPHZW73cqwo7w8yISND5d9u3/AFr6O4d+HL20Cw5jOXV0Y2BRI0CroeQ86W2npIJUttiBdfEb'
        'i7L3kMWIxVuULrqKSTkcLvqCB1PTXX3NZwvwX+IMvGMl9ns5ZW+Ae7/GSzC5Dsx5uYKEHzb307AetfXcvC2HlxDW8ePihcHmV4xp'
        'wd9w3cd6S8zwplYJw0U11dRqfywUDtr0Ou+/pRxTiC2pGaJ8PcJ4y34aQpGNo08nv6dtn10KbsbhMLNLYyGyJurdgIRGAevmdeYo'
        'mMVNaSia+SCzB0dSyhS31B7fSidtksZj+aRDHLJrlWKIb+5PTX2oJV5GpvwEl6MYzz9T/Nsj6jr2qrl5RDaNyjmP907/AK0Av+K7'
        '4SN+FtIuQ9nHzHX+lUZbi+v4DdyTaU6PzMdg/fpXPLHjoxYZXbJoo5xK0koCJITzMkgVtfUj+lA8z8O8Lkne9x0yWWRdTq4MzsVP'
        '+I7PU1e8eSMgvavM53oo/cfTdBMnxNf4aR/xeNlNm5/tk3tB7ivMeOnalsvUm1VF6BuM+H8c6cSYmPN2SdDdWt4viFPUxsOv60qZ'
        'bhpsE78U8F5z+EQzKXmtruItHsnZGgPlFNEWavJ7L8bYN/ErTl6iFisqj7HvXVll3eya4scsBbDZntMin8o8+p6r9e1BF09f7+v+'
        'jWtbAvDHHKX9q1txC9pCT8v4iMiS3lPp7H2NeWmBaXLOMbl7abFMPEMAjEfzf5SBr+n1oZlcXxFY3ZzvDthFfWdzvraorqOvZlXo'
        'f0NeJbXmetnjyGAyXD+QVCyZCztnWHf/ANyMADXuuiPeq8cIJ3YmbkypxxwbeyY9sli45bmOPfjQlfzI/fp0ZfcVmEEoVyjdNGjt'
        'pxnxphr3wIZ5LrwHK65edXAPkR1IqPiPiDh3OXAusvh7rh/IsPzLi3TcMp9WjOtH3B+1WpqKJmm2VrRw+jVuXQj2O9QY/FXlxZyX'
        'mLZMpaRjbyWp5mjHq6fzL9SNe9RSTHk6+lJyQ59G34Bt9LyyN1oNez731q3l5TskUCklLN3rceJIS1R+ZtnZr8DURb5q7WqDBn4D'
        '4ln4eyyyj5oXOnU9qeOOeG4762TirDn5HHPMIxs+u9VkhO6evhtxQbZhhb+dhZynXU9BvyqbNja+eP7K/T5bqEv0JWWUC8dwrASa'
        'f5u/Xv8AvuqkcpjfYNOHHWJNs0sULmUWsraYkElG6r761qktxqm4ZqcExXqMft5Gg/YX3iqBzfMKtywiZO1KkUzROGU9qZsRdJOg'
        'O+vmK6UeO0DF3phn4MXtpivirgLu+QtCLtV1vszfKp/Uiv6ERQroEjZHnX86uDscL/jzBWTOEWfIwIWPYbkFf0Tkk0hVP/it9PTV'
        'iJMr393DaczBWdwOyf60qcQ57LCLw4XS1LEfNrZ79RvtV7IzNHPIoGyut/X/AE86FLai5uxLMXaP/BokfX96LK21SGY0rtme3GVu'
        'L2SS3uIJp5wdO7dfmBOtmhiXN1aXCpcqqh20mydDr5+laVmcFj3BeKC3jkJ2WJ0T9l70uZ7Dtk7EizkSJ9cpk5GBB+46V5+duGpd'
        'F2JqX0le78eOwM8SqVZfmA6qOn7UEwd8uVguseJnivLfo0Y6EjyOvMVc4YvL2xt5MZk2E5j2qty6bXbr6/WlTiuF8LkrbLxie3uI'
        'H5TOkZKsh7BgPLyqVZH9F78FCxr6gpZ5swX4w2Y5I5ebVvMenN7elGrbJxzTtjzHFNOB0gmJUuP8p/6Gg3ENlbcT4aDKW9skh5eb'
        'nDBfDYf5j2oviPhpmeKbTHX9zetYG36pJz9V/wA1Z7fuPrZrmoK2xZzmUxXDOfjEGLvbAXP9rIq/Ireuh3Hf3HvU9xlpFzcTZHFt'
        'e4y4TrcWkR0m/wC9778xv/lWlcQ2aWGGWyijtc1fjYllcDlUD269SKSLzj+wxkIxvEGBvLCEdntACv8Aoaascrqlf9RTnFq90SX3'
        'DV1w5ibrJcG3uWlurtSUtrcARdfNuYdCPbrWYzcffEXDZGNM3e5V1Q/NDdFkDD9ga0NeJeEsnHy4r4i5XByN1USuyoD78w1+9Vsr'
        'H8V47TxLO+w/HGKP938PFccw917/AKGnQTX1f3Ftq9HOP4sw82NXM4jC/iVcgTwRRjmif0I9Car5T4jcYSp4Nl8P7CSHsq3Vu0u/'
        'qNaqLDRKmFvbi84Gl4dyYICmzEsSSDf91WJGx6V5LljGqePxLxZYOvYmyJH/AA0lY4xk2kNc3KKTYsy3PGV1e/iI/hPg1nbfJJZ2'
        'kts4+6MAfvSVe8I/EW1eWaXhTLFCxblWEyADfbY61qcmds3Dc3xfy9qR001nIG//AJoLlr5hIZrP41ZZnUAgmyn5Drt/J59fSnwn'
        'NeP7MTKEX/6jH7ud/Ha2vLaa0nB0UmjKHfp1ofcQPHI21I0eoI7VrE3FPGZZre148wnEQc6/D3fIHby6Jcou/oDSLxM2SF+W4gsH'
        'sLiXQHNa+AjaGgRocv3FVwyX2TTh9hZ683WpY96qW7tJIJSjrog1yo0tGpISfga6TasGB0RXCnrquqI0c8Vl4cgtq16T4lvpJix+'
        'V4x2B9/IUP8AibgosTlxLZqfwdwoeNtdCCN0HxkxhvEYMVU9G6nRHvWy28Vpxn8PjiZAsl7ZdIpF7ldbHX9vtUc5PDkTXTL4JZ8L'
        'v6kYA511qzjrl4JAynpU2Qxklndy206FXjYqQRVf8OUO16+1WWmiHi0NWMvHx2XsclG7I9pcRzKygEjlYHpvz6V9+8O5OPLYO3y0'
        'Lh4rmESqQd9CN/61/P2RPl9q+kP+y1xtDNgZeD7qXlvLUO9rzN/aRk7Kj/y9enpUXpclaF9o2WeJ7meR98vMdD6VM7RAgsOSRRrm'
        'AqQqAi6UA8vRvWqL75jpVZD/ADLvdWmpnF74bRM8g6cp3tQd0nXQuI5omsnh8E7HjRnRT6gAUfkt54PFVHVoDsqrfOAD9dH7daD5'
        'PJXWGnt2isY75LhCVI2QNeSt2P0JFIzK4j8TpilleI+Hcpex2F1DJkruGYoJLR2iaEjpzfXfpXttj+NM1YTxYaFbiGOYpAbvl6qC'
        'QdsR1o7jeKsPZXRuY+FZRdAHUgRV5ie4J1UeV4zvb+G3iw8AxSxOXccxVmY+XYdO/rUSjCO6/wB/wWXOWi7jOA8BYYKy/wC+kvJd'
        'c7SaViFDdNr0/mFUczfz5AT2nC2dktrVAFS3D8p0BrsetB5b7M3cqy3kt7PrsQRKgHsB/pUZhsckwjuI1/Er/I9u/JKp9gdNv9a5'
        '5l/E5Y3/ACFnJ8W8YcLT7yFjDc24Oi3IVP12P9aL4v4l8JcQWptc7YMisNMXjDgfcdf2r29tuKsdA8VqbfiawAPNZ3actwq+x7n7'
        'g0uWXB3DXEtybixXK8LXanUsNzbM8W/Zh2++qJe3JbX7X/BjUl0Ecz8J8Vn4XyHA2ZtX31Nu8oK/Tfl96k4O+H3E/BpfLZaNba2R'
        'CSIZwW2PPp613lsPY8L2bQ4bM/xC4nXl8WKP+z9SSN0uO9xIpS6uoX8/znXr9QXWiblVXaMVXYw3vEmUv0dLm4upLYsWSOaXajr0'
        '/m1X7HZ2+tU5EWbw/QOWH7Fx+1LCMqN+U9n9Y5Qp/a4q5DPdAD5JpR6BpWH7SMKmlik/A9TiMMvEc8+45rZCD5yWcDj7+Iif1oLk'
        'Rj5tmbD4G5JHUviJYm/9dtIQP0qAX1zFI3KlxGfaKc/r8tenKyoPzZF+kkBA/wCJayMckXo6ThLsUcnjMVeStAcWq7/u2OQE2v8A'
        '8M4Dn/1Uu3MV7hplgxOWEUZfbY/II0Ecns0Um4jv1BrRrvIW88fJcwY2dfMSy6H2Hal7JQwySBbSzVYD3igv1df/ANbqU/arIZJ1'
        'TRNKEfDGWTg3FcUYi1vzhJ8fcPEocWk6sF6f3VO1K+miKU8z8IswGb+D30F702IplMEv067Un6GmbhHjPP4IxxrjYJrJpCq2sa8r'
        'QIOx5lHL+uq0Hh/4o8N5W7aylyFpjrlW5dXiFUc/5XA5T+tNT8oRKC8o+Ycrw3m8Q7i/xs8XIdMdcwX667UPXTCvtK8yuGvLkRvb'
        '4jKOBy+JEyM2vT1FL2b4A4BzDs95w/c2crkkywDWt/TvTFl+4HH7HyhrVP3w84g/hl5FcyyDw9BGUbOl9T+9PuW+BmCn5jg+JXR/'
        'KO5UdP6GliT4bZXhy6a1y2RsUs7hWAmDEKSFJHQjv0oMvDIqH+nnLHIs/FXhyLJ2EXFGKiHI4/NA8/espaEE+lbD8ObzLQ2P8Iy1'
        'lcSY+df9nkeM+GwPb5u1JnxG4buMDmHcQkWcp3GwHyg+lKwtwfty/Q3MlJe4v2Cym11U3DmavOF+JrDO2R1NZzCQL/jXsyn6jY+9'
        'dIBydaHZBSQRS/TtEGKmj7w4fytjm8BZZGwmD215Es0Lb8mHY+hB2D9K/SFVfkkQLo6JFfM3/Zs+Ij4u6HBmVlAs7iQtYzM2vClP'
        '9w/5WI6e/wBa+j2uo5FWRiDv5WB8jVvOtMxxokZ2ibQbz0p/9+VeLZxqkkXgxtb3AIlgYAI2+49j7+dfkZFfl1zL0INSxFCSVb/r'
        'W8kajN+J/hykRa74Zz+TxTDosXisyJ7a7gfrSNfTfFHDId52SeDfLzkqdfcg/wBa+gJ41lGiOWTyJHc0PntE7mIE/qCPQg96nmne'
        'iiM9bPnOa+4svZhNeZS1aRf7819HGR/xDVH7Di7JWcKxXmTsr9kI5FAF4v0JCkj9a2SLF41jzS4+BhvZBUHX0/5V3PhrJY9QWkKo'
        'R11GCGHlv3rOEZr5gvdcXozc/EieJYRctYW/mvPZTBWHsxUgfY1DxD8RL7I2DpYZbFpzLyMIZIyxH++wI/StAfEYqVBDPaRr16FN'
        'ro/Y0Ju+ErS4ZjbrC6jyZd/1pHGMX0Hy5Iy+KVowfDxczN3+WKBw3qdq6mp2vfkOsflIj6pbTBf+G5NPz8F44gh7a3jPobaM7/Va'
        'o3PBdknywrbhj6Wsf/8Amiv8GOQl/kyjckGZ2O/+xykf8RapVwtrcKGWxvZeY+WPBP7w0XvOFpoGIjuQgB7KqL/QUMu7S7gXl/Hz'
        'A+niEVv6FPLReteBrcjxZozap6ywLH+xQGhuax+FsNRrcZGblPzPDb+Go/3nIB+1QoJmjdHnlZx6uaSOMLHZMnLze5oopN0FyklY'
        '3nN8L2/SWblIPX8Re83/AAxk/wBa4bjfgaM8rwTTb/8AAgb+rmsdc8rFCO1cgndPWCIPuyNfl4u4Dm6/wfJy6OxzCNdH7VVyPFvC'
        '8qER4a/1rp+YoP6jRrM7aQqavowZe1F7MTOci5e5ThKOZ5ouGrhXJJ5jckkk+Z2TQZOKsvY3JbDZXKWEI/ljF2za/oP2rq8twynp'
        'QW5gKN26USgkBJtjzj/irxzAFU5gXSjyuYEk39yN/vVniD4hZDiXFGwy2Nx5BIYTQRsjKQd71vVINoN96I26brnCL8HRtOzYOE+P'
        'YOG8VbWGSs3lTl5keDl0qnryka3sb9aZb3irgbi7FNa5TGZA2n96RIW/LPrzJzfvWP3kNzNjUnaNWgjRAWVdaJAC/X0qzwPxBccP'
        'ZqORJStvIwWVT2160iUPluO2h3LlPbqwEd+VRTxBh17mp0IBOzXjabRFR4LTJ8UQVNalAWB6jzrY/g/8UxDbDCcTXYUwKPw1zKx/'
        'MUd0c+o8jWWzRl16DvQPKxlB1HnV3FTVMbONH29Bl7WeBZoX+RgNdd9+xBH9avWlyyqNnYPr518TYLjfiPD2CWdnfuIozuLmO+T2'
        'Ht7Vsnw7+MlvkIRa5wJaXSADnB/Ll/X+U+1LlCcN9i0b+JkkblJ5WHvXgnYOQwJHny/11Szh87ZZC2E8FxG6n/C2xo9qIfjEK8ok'
        'AZR0J9PrQrJZtBhijjf7jpXP8rb5tj0J6GgoyfhKRIRo+e9g1w2Zg+Vo5gyn9q1zNQelZNaZVXf0NVXjj2CjsB5ne6GPlIGbZePt'
        'oE+tQvkRGnPKxRAf5io0PuOlC5J9hBchCvKHD7PY1WuiV+VrhIVPYAj+gqmcvjltzNcXSCMDqV0dUtXnHfCTSchvweTyaYRsPtzV'
        'yaAlIL5GyW4gaWCd7jfUcqb6/esQ+J15xJhJydzLZF9CTk0N+h8qO8efFzhm2s5bfB/inv1JUq6jX15tEEfesIzvEGYzUjPf380y'
        'sd8rN09qfCFgb7NU4Yy5ukDO22Per+ftxPblgN9N1nPBuRKFVLdR0rTLWVbm15e+xSJ/KyqD5RoyzMW3gzsR23VRAKcOJ8YQzMFp'
        'NlBikKHpqqYTtCXpkwGu1XLNtfzdqHRzA1bikUgHdNs1BF4gy9PShV7bEnoKIQzA9CelSCNXOzXJmNAGG3dJOqnVFLZANDVWWgBP'
        'Qda6aGSElZUZSDrqPOsbNSJJ5rr+FmBLfxI+dfmHVho70KqXA6miVnJ/ssyt8yKyuV9+o2PfrQuVwwJ1r29KXB/M0OnFcIyR4z9D'
        'XqFthQe5onwdw5e8S5Nra2+SGFfEuJiNiNN+nmT5D/TrTrffD2wsMcmRjzC3gDFXgLxwvG3kGJY/fW6jgtgQQjQw7iJaguXjHzA9'
        'VrdbbgThLKCGzwuauTdyKn9tIojkJHUoSo2AfXvVofCKPCx3Et1bS3d34iJE08KyRxDfzMF3pifLfaqoOtsKVPo+YioGx6e9WrEB'
        'W619S5PAY2bIOuTxuHu7Nv7MyY9eeHQHXYI2PbXSqNvh+HcZfSyY3BY2DnQFZBFr5h6c29A1vvL7GLB+THeF7XiqYpHhBexq3UMC'
        'VT67PTVaBbDjDEWwuM9xNFYxKvUFQTr/AMx11+m6YjlchPdbu4Bbx7+RrS4jboPVWCgf0pd4ux/w8yYe5vcjxHNkF6mOQMdLrqR0'
        '5ensfWkSbn+BvCEfyUrr4v2FmfwrpNfqOhnChQT66/5CkniX4lZSZ+XG3CxRc21MQCnXvr5gfvV3j34eW2IihvsVdNkrO5XSqJAG'
        'jYjYB0Oux/SkOPhzJvP4LQPGwHzF1PKOvqKdjxQqyWfK6GzF8f8AGOXuYrJLsMR3k5OoX3I6EexBq5xJxlnkePF427k8dVDTSQMe'
        'hPUAenuO3lVDh2w/C46SGBLgTwnmufDhJd+vTWhsj2r1rtZrO6ssfjntJ9qGedCrtvfza7+XnQuKcuhiilHfYCy19xJLGZZrqcKx'
        '5j4TcvX3C6pVvZZppjJPI8jnuXOz+9M91HnBFFEZF8OJiAysOp9z50P4gx8kkkN3BAdTxgyKo6LIOjD76396oiqESS8AaMsddd1d'
        'jQlR51+tMZkGYBbG5brrpETTDg8Bc3V0YrqKW3jVeZ2ZTsD2HmfPVdJ0joxbBWLleC5Vt9Ca1ThK+EsaqxpHm4bvEHjQFJYD1V2Y'
        'IdepUmjXDizQFGiljuFJ0DHvqfQbA39qlyJsOFxY+5SyW5tyQB1FZlxNjmhmYqutHdatZtILJnuY2iRV2zSDQUe+6XOKcfHcx88b'
        'IeYbBDDRB8/pQY2+hmWPJWjKlPzd9VZSQ66VeuMTGn5hvoSztpI41aQ7358o0BUz4aRrlP4d4ksI5VaaVQql/PQ9PrVMZAKLS2VZ'
        'Y7m3EbToUWVA6H/EtWbW6BXR70wwx397epZ5m0s7iyVt+IpKMNHWlK9gfMVzksBiLfMTK10Le2RdqbeYShj5Ab0d1qyeGhjx+Uyh'
        'YzJ+Li3GZAGBKjua+gDhsLjuG7tcslpHHdW3h8rr+a50dcuz0Oz3HbQrEMTYJYW6ZWOdJbiJw0cUhC6YMCAy9yDrrUV5msrfZe6n'
        'nI5S7SeDJcr8i9+QE9SPT2rzviGLJmcVB0l2V+jlCCfPthKwwz2yXts9kuRmUpJzRSFHjQE9x11vp+lJ2VLQ300T28luQxBjfuKZ'
        'bPJOuXN3cXCvcTJyFYXB5Bo7B6emh5028Npw/kJ4MTmJY4LOYc+7lPEMe/Q9+49atwqXcuxOeUa4x6BXwZ4sx3Dl9kLPK/l216g1'
        'MF3yMoOgQOujv9ab7rIWcqHxMel3i5AUQsQJG3rRUkHX1JFYog3Psdt07cGzTtcRWIyMFlDMdeLcJzJEe+9aOt/p61JB7Ewk2qNJ'
        'gsXtobOaK3hhJT8p45udiNdN+ZblHWnCe8ee3tprvIXYQ6UCIHm7dup8teXrS9w7hxi0F5NkLrLQ229+GySKhO/nRUGwfb0NDctf'
        'Yr8Yk91l8n4ZcO8MVu7+EN7IAK6BP1Pemcg/Gx1x82CvQkF9ehC56mc6JHbfzdB11VlMXirfITrA65GFoUZVDqyhuZhragfp7Upn'
        'jLCyiXmw7XlqrHwpDIOeRfIaGyOvkRuucPLGuQm/ALFj4pJGncnTrzb0IzrQYjXfpRKSozd9jTZcJ28lwxe0Cwn+9y7Zf18qSMtB'
        '/FuHbaW3sgl0JGaBRCAZCsjd/bWuopxxN5eKrl47K+ikU+IgeSN/qFG6z0yx4WyuntY/w9xBIyJHcvzFl5iQdaGubuN1zdBJ2LML'
        'RjLG3yePlx0V5KrMl3IFhdwW1ykHS+Q8hvVNGMwdn+KvZpLdYomj0FmdnR1TqTsHWveqn4ls3ci3v0smE8YjCLCrkD/dG68PCfB/'
        'BV/ZWuQ4mWzu5/nNi966RqpOtld9Nkga11rHKw9RP2D4WsrjMwzYLMxLcqoLyJIxAAJPKrDr22DV34nfDzOwZZc7iLD+JwvaFrkG'
        'bW+XRJ2fP08z1pgiewxl8Zbe4jKErylZtltAdD06dSe460YTirI5CX8NiLyKJrUlbqOQE8+/5SvoO/WkzzuL2xkcdrRicEeFv0ZT'
        'ctYyFiGS8geNQfduXXX13QuXBQsVhS9txDJIV8RHDR9D5Hz1W85HiO1x1p42WsbSJGcrK8JDbJHRtHlHfvSpxHlPhvKwu1hW+ncl'
        'xGRzRBv8QXt19qZj9RN9LQOTFCt6M+nZrS/eyivMfcR7MfioAN9P3PQfpVR8dK91HLaTus4bmilWRVBYHyH2p2uONbFrX8NcYbEr'
        'Gy6jjEI5lGu+iOvl5Ur523h/GgtZ2drE6q66hTY36GqYScu0IlFLpg15b2RSkNokf5g8VgnMOYbPbyBPXX+lc3VvLHeyw3sovkTl'
        'cr0VQeUb0O3Tr+lHsRwVj79ZZ4rq68R/nYmZgSfsetfrzhPJxorWUUE6TgRgSgnWv72t9enrW+5FOjuEmrKd1kLSeLwEkmUOqL4E'
        'g0SCR132I6b/AErvKizkkht7aOzhR4wzzOWBB0SR1121v0qtxNYZnCWSSZnI21pbI3IiR2aFt7/uggt990Jw81lLPM0F+90WQlS1'
        'm2z23s60Br1rou0C9FlsvjrWQw28lmQp0yxIzKT6+ZNXY7+0nhEMdvduxQkBIuUFvoRrVPzYmzl4atLtpib5QXYySDoO/n7ACgt5'
        'b46CT+I3CvKkUfKrAfLptgaZiAevpWRyJ+ApY2l2KL215HDHLOkscd0dq76BOt9OnarUslvCgtSkhdl54wo0Wb/MPMfpViXKSpdx'
        'HmuJX/lVyBpF7Fuvc6NRZDKRxxm3hM1rcLpY+UiSZh582/5R9TRtsFVQPyEcguY5piBsEtb75uYADrodjrfWgmSmu7RriO1htzby'
        'EaYx7caHTf8A19atXX4p2nnjK2iyKYlPMSTvQ2T59vpS5jzJZY6dpZCytcMumO9qNa71ktoVJ7DNs9zfOl09tFGNhJCsYjVuwAAF'
        'HboQzMzRyPG8AVVDdeqjoPf3pXiyTwSQ+FFzQn5XXxR29z3359qJy39u8iKJAwYb5uXu3t+1dFm2DW2k2hTNiR+QGYgdKW7ohZj0'
        '1qrUF+RGAGNSQxtaNwOnsvZy4kiifw5WA8wp1SjLl8or8keRvFQHYUTtofvRLMX3MrfSl/mU9T3qzHClsLLK2EbfiTO2XiC3yM35'
        'vVy+mJ+56j7UQxfFubhIZLnlYeakr/rSrK+21ur1p/JRSxxl2habNTwfxhy+LsJImsIbqbXyPK50D6kAbP61nuf4izObyc2QyV/N'
        'NPK3MxB5QPLQA6AaA6VTf+U9aqsevehWKC8HWw9w9xZlcJJLNZ+C07pypLKnM0Z9R70PGZyD5mTMZK4kyl62j4ly+9EHp+nkOwqg'
        'tesNiu4RXSObY5T8R5NbC3yDpDdxyMVVmkIIfzDDR613a8YXN47m8muLB/DIR7ZgebXYMdb0aVMfcT26vGhDROQXjccysR2OqkUb'
        'YvygbPYDpSXghd0MjmnHphzJ5zIynw5b25u7du0jTjr7cpB/eqsnEM0SLDA040dljo79j5Ee1D5f5KpXGiOtHHEjHlky9d8WXEfi'
        'xQxLCraCtvZT1I89/wBKbuCOIcPdYd7TJMss6Bl1IP8A6Z1rR9e/T96y28Qda4tWKSAgkfQ02WO+gFkaeze73O4nG21ukbrCkgAH'
        'j73y9CSDvrv+lWP+9uAa4Wc5SYyNr+yfQDAdwPKsdS8mu3DzylyAAAfIAaq2ZAEGu1KeC/IbyvwaBnOKvxl/G+Rx1tcvGNwfin8S'
        'NV+x7kdfWh19kUGQmvIcpiLbx4hG1qijk0p6AEnmHU770iXV+IFK7FeYfLRvP4csSTL/AIX7V3tOK0zvcT7GuXPcQTSCztLpXYqd'
        'sYVCfcknYq7isjnZgx4gzsskPRVkKKsEXqF1oFuw86m4WxtnMfHtlx0Ddwslszn7eQorkLKKOSa9uUa7YDcMTjlhT/dHXX6da5OK'
        'Nab2UYktxBeTi5na1hTaM7b8QnoAvTzNKttmVxylGtzCXbbuVDkKfTzoRxHfXyySWzTOkDSGQRqdKp0B0H0Apeadubq5P1NMUG9i'
        'pT+w6XfEEDssFpHKyswA8QDqd9qG5e7nZWY2Z5Um6p0Kjp26UGt5z061fjmYrrmPX965wOuzmETtJzhVXbcwHTp+1ELVLhWXnDuv'
        'fqxKg+fbtVaIjzA+1X7WXl6AkVvBGJF7KRnZoHLcSRtpTrVN2bt+VSdapMu11MwPaujFGvRG8jykl2JqKUqF7V2T08tVWnbZ1sUy'
        'gSHnJbZFErNtDrVGJeY7q0h5FrGaixcPodKps49ajnmJNV1clqw6whE2+1TKu+lVrbrV2FdsKXJhLZJDGR5VcSLY7V5Cm9Crscfy'
        '0N2FQPmi6a86pzQswOqNTQgaLVVuVUg6AFMQNC3ex8tDy/K/ftRnIL0NALkkP0o0KkFbK40epq294VTqelL8EhBGqsSz/J3ra2am'
        'c5G6aR9A17j5GSRXB6iqRPO/WrUB0RRtaMXZpXBmV5SoLdqc8hdrNbd9gisgwU7w3CnfSniK8ZrcAtsaqKcKkUxloUuMQBM316Up'
        'SPptU2cVfP8AMftSfN0k1VeLaJ59lqCU770Tt5S3SgsXeiFu+tV0lR0WFo2OqtQMdjyodG++1WoZKEM//9k='
    ),
    'chipping_sparrow_01.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAGwAAAgMBAQEAAAAAAAAAAAAAAwQCBQYBAAf/xABJEAACAQMCAwMJBAUJBgcA'
        'AAABAgMABBEFIRIxQRNRYQYUInGBkaGxwRVSYtEHMnLC0iMkQlSEkpOU4zRTg6Ky0zNDRYKj4fD/xAAaAQACAwEBAAAAAAAAAAAA'
        'AAABAgADBAUG/8QAKxEAAgIBAwQBAgYDAAAAAAAAAAECEQMEEiETMUFR4SKRBTJhcaHwgcHR/9oADAMBAAIRAxEAPwCr88sv6wP8'
        'N/4akdQ0+KMyNM0hBAWNYnyxPdtk464ztTCnO2aFqC2/mcr3QjMSKWJkOAuOuenrrz+OWPcvpv8Az8Gtp13Knyd1bTIdPSC6vk7e'
        'SeVo+GNuFhxHYYG5B92auV1GwA/2kf4b/wANZryGjjvrXz6driUQXDG1EuQqAjPEv3v1j6RzWsEh6GrdS8fUfHP9/QGO9vcH9pWB'
        '2Fx/8T/w1BtRss7SyH1W8n8NMcZ76i7eNUbsfr+fgst+xfz+yIy07IO94ZFHvK4osLRzLxwSRzL3xuGHwrhcg86VntbKduKe0gkb'
        'veME1N2N+BbY1NPbQEdvc28R7nlVT7iait7YE/7fa+2UD50O3itoBiC3hi/YQD5U0r5GCaKeP0/v8B5PRy20gyl1bOPwzKfrRhEz'
        'fqjiHhvQuygf/wASGJ/2kBrn2dp53Njbf4Qo/Q/YyUgxgkx+ow9lQaE9VPtFcGm2I3W3Vf2SV+Rrh0+1G6m4U/huZB+9TVD2/t8j'
        'VI8It8Vwx4NRNnGp2uL0f2uT+KvG1GMC9vx/aXPzNSoe39vklP0S7IGhtAKkts2Nr+/H/Gz8xXmtZD/6je/3k/hobYe/4+RGn6Be'
        'bZ/VGT4VGS3dN2Rh6xiiPp0LjE013P4SXD49wIFc+zraP0oVkgbvilZD8DUUYe39vkG1glTeiIu9Ra3uAfQv5x+0kbfErmhiG8UY'
        'Oouf+BF/DT7F7ByvAzgV0d9LCK766hN7IYR+5UxBO2zajd/+3gX5LTKC9kUxtYzLFIMHhCHiPcMGlvJTjl8mbKXhb0I+zYdV4dsn'
        'wodzYzSWrIt/dMvEHKyzNglc43XBX2e41nPJa2vNWmhnFzeQRWk7rOxu3JuFOSqgDGFGRvsfCtcMMHAV5JKRuGDEbAn2UJopD/Qc'
        '+yofZds27NcsfxXUp/eqJ0iy6xOfXK5+tNDTIaWdiyviuSMkiGORVdGGCrDINBJrhJrjoVshpjhLUooChZHAA2x6RppZMdarrAns'
        '5B3TP86aFPm/OwR7DKyZ61wuaEu1TAqpjHeKompEgChM29LYGTU0VDS/FUlY0UyJjaNvR1akkairJtVsWWxY3x0N5QDS7y4FLyS9'
        '5prGcxxpfGoiUHrVe0/jUVnOedKyt5C0DjNS481XpISaYRs00YtidRDikHnRMAil4uVMIM1uw4Gw9VIi0WaE0Rzyp5EzUzFmt8dH'
        'ZRPKVvZnurwXFPmGoGHwq2OiRS8wpJG7wukchjZlIVwMlT3791VnkXpVtpVjcwQcTP5y4lkY+lIRtk9M47qvhGR0pfTERZLxV/rD'
        'FvWQDV8dKlwJ1bG1zXiKmBXcb1pjp0hHlMyOWa41dO1ROSa8Y4nUaFrDZrhe6dvkDTa5zStkP5zdr3Sg+9RTdHL+b7CRJgip5oPW'
        'phTWdj2eYk1zhzRODNSVKCQAJWuqKKUrojphSA5Vwk86JwYrhSjZLAyk4pOWQinpU9Gk5I96sjyRsVLmpxAk70QRb8qYhg35Vtw6'
        'dyM83QW3Qkcqbjj35VK1gOOVPRw7Dauni0XszPICiQYpqJBU44PCmEiwK6ePTqIjyMiiADlRAgqapRFStCikByYHss1wxAdKZxUW'
        'xUpAFmQVVadtqeqR90yN741/Krp8VTW2F8odSX70cL/Bh9Kj7oiHDXVwTXDjNdUDNOKZoivcG+aY7EiurH314dnoWittxjUbpe9Y'
        '2+BH0psJmhrHw65IvR7ZT7mP5072ZB5UMq5X7IpSALHg0ZEPdRAnhRo0x0rPQVEGse3KpiLpR1SiKmalDbRUQ17swKaK+FDYUVBs'
        'DSQo43oZFMOMHlQmxVsNO2UtgHGaGY/CmuHPSuqma3YdI7K3NIWjjHdTUEQ7qmsQo8UYFdrT4KMmSdjFvEuKciiFKxkDlTKPXRUU'
        'jONKigV3gFCVzRA5ohO4xXcgVwtQ2egQIWFQcihs+KBJPioQlI2KpuIr5STfjtEPudvzp2W4qonmxr8DfftnX3MpqPwRFuG8amr4'
        'pIS1LtD3UwCBjrgSuNMKH2u/OvBKVnpJyQCVQuvWp+/byL7ipqw4R3VV3sgXU9Nk/HInvQn6VapIDV0sbkl/fJmUlbPdnUlAFd4h'
        'UWbFFaZjb0TzipZxSrTYqJuPGr4aUV5Uh3jHWosRSfnA7695wK3Y9IUSzBpAKEQDXO2BrwYGuhi0iRknlJKtTVfCuxqD1opCqhZi'
        'AAMkk4ArVHDFFTm2DzipLvQPOLZpxCtxE0hXiCBwSR34onFw08HFq4iyUk6khlBTCUgs1FWemsBYKwqYakFnron8ahB5n7qgWFK9'
        'sO+hyT4o0AYlcUnM2etDe4oDzULCdfc1XXgxqlg3eZE965+lOGQUlqTgT2L91xj3qwpZPgK7lkgqZG1CWSu9rTAEDNnkaj2hqHDX'
        'uE15jDomzo5M4DU5Dmzk+5dJ8cj61aRuetU+tArpxk/3ckb+5xVogYGunj0a7Gd5/I0H2rjSHFQG/SuGtC0iQOuQkkpdnPfR3TNC'
        'aKnWnSEeRsgHNd4z317szXRE1WqCQrlZ1XNFSQ5qIhYCpLG3dRsFDMUh2qs8spXm0OS0t5olZmj7did40LhV2HefgDV3pVtCxnuL'
        'xitvbxGR8c27h7TWKm027PnV9MZTDIG4lmHAeHPEpxzIyRvyBHOsGv1G2PTXdm3RYN0t78Cuh2EdnqsuozRxyzrOoMrFgUySMAE8'
        'wFyR0BGa3TyDPSsVptxJPJPFIYWninVyAxbJ4ccRDAHBT8q1Xk6638VxaSRG1vIuEwxEECROHJA9hHxqjQ5uk9sn3LdZj6i3Jdhj'
        'jyamG2oeOHbGKg5J6V2DmBWkA5Go9qaBhs0RIzzoWEKJTjnUXdj1r3Ca4UapYKIMxoLMaOY2qDRt1FK7YUL8RpTVWIghf7txGf8A'
        'mx9afaMjpVfrakaZM/3OFvcwNBp0wp8j6salxGuKhNFSInpVlC2DSMGiiNar1v5RzsovZdH+CiLqD/1FP83/AKdVxUY+Au2e16Dj'
        '0W9AGT2LEewZ+lOxLmKNzyZAw8QRVZquqBdLlgksURZlZJXM5I4COXFw+icjqMYPOg6dr8x8w0xdLjZTbB0cXW8a7HD4XY77bY35'
        '0eolImx0XgUCuhARS/nE7H0bKL23Z/7dTE8/9RT/ADf+nT70LtCmIGurAKELi5/qCf5v/TokdxcZ9LTzj8FyCfcVHzqbkSmFS1zR'
        'VtR3VFb63jXMyXUB7pIGPxTiFdGpQsP5C3vLg/hgKD3vwio5RCkwhgUDehvGANhXjeTkelpk/smj/OoNeHfi0+9HqMR/fpdyDTLC'
        'MwReTdy0qniaUemrYZMDYg+uvmeqxm31sTnVbw3Dxdq0pJcEZx6R5g9ORGK276qqW80EukanLHKpVgsaZHiPSrGeUMZlvIbazeeO'
        '4VS4BUrIoI8PlXL1WJud1wb9PkqNJ8ithFLJfLxosVqDwo4cyFiDnEewEeQSMjlk4xzrZ+Qeoyz6yk093FJC8vAkUaApEVGyoT0x'
        'zI2O1YyAvbT3MpVYTGsLSKyFFdgQWJ6ZKZ5881ovIu50vT5hdahemCKIkRRmJhw9Og/DWSGNOVruap5Go14NZqdsEv7hByEjY99K'
        'm38KkdZ0q4czfalllznedR8zRI7ywfdb+yYeFyn516BSjXc4rTvsCS235UVbamo3gYZFxAR4SqfrXnvtNgbhlv7VW+6JQzf3Rk/C'
        'o2vZFYEWvhXfNsdKYXUdMP612sWeRmR4gfa4ApuJFmTjgIlXnxRniHvFRNMjsqmt/CoNbjuq5Nu/3G/u1A2rn+gfdTpIUpWts9KQ'
        '12yLaJfADfzdyPYCa1PmxXmuPXS2qxQpplwryRhpUaKNWcDicrsP/wB40JVtZFdlXY2va2kMoGzxq3vFOxWeOld8l5IvsGwhuZ4U'
        'u0t0EkbOFYbDGx35fI1Zl7VdjcwD1yL+dBSTQWuT5mzGo9oa7LnuoWd6xvPFFqR67uZYrSZ4ld3VCVVOZPcKD5PpPZ2ASaCCB3bj'
        'xDyIO49R8KYG/KpopYAfdyv1+tNGSk0w+KG4rl+hpqOdqVhgOM0dIzVqkLSHIpz1FMxzriq8BgKkpPdTxYrRarMuOeKl2gI51Wqz'
        'dM0QFqawUNtIO+hmTuNLszcq4M550CUM9oO+orBbteredkvbqhjD434SQSPgKXyQdzUu34ageRy6t7W7t3guoY5Y5McasAQ+DkA9'
        '9MpKFUKmwAwANgKqTdb0RLimVCuy0Lhh6QB9deEcDfrQxH1oDSKXA76Mk476ahRjzLT2PpWFo3rhU/SnLdIYF4YIo4R3IoUfCkUn'
        'XvogmHQ0vCCWHaZGDuO6ln03TpnLPYWxY8z2YBProazDvoqXCDrUdMKIjQ9H66bbH1pmu/YujDlpdn/hCjC5THOudvnlQ2r0S2C+'
        'xtH66VZH1wL+VKavo+mLp8skNrHbmFTLiGFMMQOqEcLe3l0Iqw7U0pqiQX1jNZXSl4J0KSKGIyp57jcVJRTXYibsyH6O9KsdYtLH'
        'VbmOwK6aZLeKO3jBEuw9KRiTxHfYdCOdbdrHTFGRp1kP7On5VSeS4t7DR1traGOFFkkBCjGSGIye84A38Kfnuxw7GsryLHDksfLM'
        'fIobYUpIhLcI5000igZoCuDJnNeeeV3QtjNrbHAJqfZ8Fy68PNAw+X5Uezblmp6gypcW8g/pKyH3A/SuthmlAPNi/a8LUaO4Qiqq'
        '9uQuTypFdRUNgmsmTVSTJaNUsiscZo8cYYbVndPvlkkGGrTWOHUEVv0mp38MjXk6kZ5YqRTwpsRYqLR+Fb2BCbIKG4xTbxml3Q91'
        'FAAEVEjIo3ZN3GuGFu6jRLFiK5xEUyYT3UCWJhSye1WTudSTHM0ZZgORqulJXrUUnAYAmsEtek6DtLhJetca54TzpATjHOhTzAjn'
        'WbUfiFLgm2i1889HOa553n+lVEbjHWiRTZwc1lw/iTb7kVF/bz560/HIAOdZ2CcAjBqxtpi5xXYwapZCNFhJLttS8khO9EC8QoNz'
        'hFIra5cC0VdlMUW4QH9W4f4nP1r0l1sd6RDlbi8A/wB7n3qKTupyud687rsrXBd2FZrgGgrcYcEVXR3SsN2o8OJCAu9cmTblYjVF'
        '1a3gGN6NdzlxE2eUg+IIquigIIINMTKRblzn0SD8a1Ysr3JEi+RbUVLITVBchlJ3rWXNqSu3Kqi6sueVqOVgaK/SrkxzYzW40W94'
        'uEZrENacEmRkVcaTM0eN9hVuDL05BXo+hxSq6CusQeRqksb0FACasIblT1Fd3DqYyXIGqDlTXBEc8qLFKjd1NRKp7q0xyRfYQVSD'
        'PMUZbVT0p1IloyRAVZZCuNkMUneWqqhq/IUKaq9SKcJqnPJbSGN1RgjEVTSXLdpjNXOtYyaongZ2yoNeS1Mvq4HY3FcNjPFUZbhj'
        'nnU7SxlIGxpp9PYputZcm6SohUm5wcUeG5GOddubFk34arJ2eAnIOKXHjaAXsFyMgZq706VNt81gRfgEb4q70q/O3pZrpaTI4yCu'
        'eDfRSoEzSN7KGztSNteEpuajNPkHeu7k1CUBox5K2Vwt7dDvCH4EfSq25LFyB7acuX/nzn70Q+BNDWLIya83rcu5l20wSSyRzNC+'
        'VZGKt6wcGtNoeHAJqn8rLZrfyx1GMLhXnMigdz+l9astIkESjJqSVCZo1KjRBlXAqN1IptpU6lDj3VW3N6qpnNVj6nhyC2x2oQXN'
        'la4NmsqPAhPVQar70rg75qoGq8CCMnJUD5UpdaoGXHEatcKbGbHmHG2M0zbKqLWfgvSXBJ2q1in4o8g1NrFLTzoRAYNTh1PB/WrO'
        '3lwy9c0qbokZBxQWSSGfY3Vtq34qtbPVwSBxV8yS7kHJqcsNTkR8sTT4tTOMivufXLO/RxzpqS8RVyDXzrTtZIxvT1xqxMezV1F+'
        'IfSTaam51NQD6VVF5qPGSOKsreas+cBt6Fb3bysCzVgz69zdE7FveETPvRrS2jXmBVabjGDmpfaATrXMll5CaWFIguwFFSNG54rM'
        'DWVAxxU3aapxkZNK232JRb3VnGYycA1kNetHHEVFak3oMe52qk1S4Q5yc1djddyUYOa2uO2PhVlpxmjxzp+OESyFsbUz5oqjYYqz'
        'HxKwrgNa3jYwx5U6lzx7A1SkEOVXarLT48AE71oyZbVFikenGLxG+9Gw9xFS7UYwaJdAec2/rYfD/wCqXuVOcKKy5IqSX7f7ZbGQ'
        'l+lOBLbygiuB/wCZCNz3qSPlisr9oiMbHFan9Ls0Z0PR9URiROoJz+NAf3TXym6vyRsavjHcg6uFZX+vJo7zVgy4DUOxmjnjuZp5'
        'GWKCLiODuznZFHrPwBrJmZ2bOaur0ta6Za6aCRI+Lq5BXBDMPQXPgm/rc06hRRtLC/vHS7dQ3LHyFAN27Hc5FLank6jKD4fIUu2V'
        '65qyTW5krkura9xselWkN9lMg4rHGYggg4NHivnXYtkUoHE1DXPGdzXVXi686ooL3iIq4tZhwhqolAFj0cLAcqmVwp2xQ0u1C7kC'
        'hzXa4OCKGwUYhmMe4anIrl3G5NUUcvG5361ZW8gUCklwgDJBdiTTltGQBikklBOKsbaUcIBrOo2yBHRuDequ67TiO5q1lmQ7GkLp'
        '1INN0bdjoru2aNvSyafs70jkaQmiZ9wK5EOz2zVyjSAXp1JsYJpK5v8A0t96rprjh6Uq83Ec1NhDR2EzSH0V2q5ihJiLMKodBnRV'
        'Gav/ADtez50XaIxKSIGXIpq39EUhLdr2xANMQShlxmqrcgWFu5B2tuc8pR8QRRXAO9IXvoqjZ3Ein40eScKtaIpqKsO50U3loJNS'
        '/QZot7wg9hbxHi6+iwT9418a4996+5eTMA1X9AXmwPE8cdzEF8QWYfMV8JBzvV+HyvTOnqY2oy9pFvoMUU18jXC8VvCDNMM44kXc'
        'r6zsPbR57p7y9mupyDJNIXbHLJPL1UC1c22gTSA4a8mEWOHmiYZt/wBop7qWilJcAd9SUWY2i71Ag6hL6x8hS0rYqeohxqMwwR6X'
        '0oLgld6El9TFFpWycCojiG9NQW/EckZNHe32xw0Ug1YKwfLjvrR2SkpgVm7VTDcelyrWaU0RUHNWRSaAooWvleMZAO9VvbSce+a0'
        'OoKrjHdVY0KlsYpJJIkorwE04u3SriBDik7GMLjarNeHhyDWeULK9oCc8HLnRYJ5eHcEVICMnJ50xBEJDjnVahTCoCcl1ID1qAld'
        'zvmrk6YGXND8w7M7jNXKDYjsTB/m4kXocOO7uNV93cKpOKvYrKV1leOPjVF/lEB9Lh6kDrjnVBPYE6rFb3LERGQB2TfIPd66mxp0'
        'NtfDEpLgt1ockhxtTepad5rd4QHsZUWWI96sPocg+INIz7AjFFxoNVwMWGp9g2Casn1nMeFNZqZVFssn9MyMPYAPzoImYHFKFxNH'
        'b6iWlJJq1t9TQbFsVihcFTkV17uTGxNLwmJtNvPqKyqQG6g/Giy3ayE4basPZ3khnRWY4JAq0sr0F2UnkSKtf5EBrg+g/oejspfI'
        '28sra7S4RL123iaMLxINsEmvjdz5P6XBfXFs3lFGGilZGHmUmxBIr6z+iSXsp72IsQ8rxvIvRj6Sk+3K1k/LXQzB5Z6onAMNOZB4'
        '8W/1oYslTk/Z1szT08JfuZ3UNK0gWthAdcdVjhLbWbNxFnY5xkY2wPZT/kta+Tlok88N9c3WqrlbYPbqiqGGO0VS3pOu+BkbkEZx'
        'XdW08q8YCkYhQb/siq23tFivUmkQOkZ4mVlyGA3xjG9aI5YpmOrL7VbnRLu0jguvtBLi14kimNuoYqWDcDji34fTxuOfhVVLBoON'
        '9Tvh3/zJf+5Ur6ImRiYGhwf1C/Fj1HuqlvSy8hS5JqU+EK0XlougLgeeai58LVB85KdZtCA3XUm9kY+prJWkp48GrWM5WqpT2+Cx'
        'VQ1ImgvJkrqieOY2x7NvnTcFtYcGbXV0X8NxA6H/AJeIfGqkkZoiDNBZPaKpstu0sEXFxqpf8Ntbsx978Irsc2hnfzjU18TBGf3x'
        'VPJH4UrOzJyJxVikn4Fs1KzaTjKarInhJaEf9LGuvJZkfyeuWPqdJl/crHGV8ZBpeSaReZprXolG3VhnC6vpLf2nh/6gKudLZRu1'
        '7pzeq9i/ir5Qs57Tma0OiXJVgc4qVH0TsfU7aSEj07yx9l3H/FTBjgflc2hJ7rmP86zem3eYhnFOm6iOOJUPrAopxK+C4t4Wt50u'
        'bW6hjmjOVZZ0yD76JrOjQa0sM+kW6JfKCZrSORWLY344xnceHT1VTteWiLgxRE/sCl7TVora9SeACJlOC0YAYA88EUXKNDxa7MhH'
        'Z+d6dHb3FxaxzQcUgHah2VTu4Krk7Hfl1NUmrWNjBEZfPhLg+kiQyKxHhxLj31ornUOyaTtoczIxYSxKFLesciCPUd+tZ/UZo7qM'
        'XFsshhkYqgZcMGGMqfEZFRuLVpD7ebEb3R5LmytW0rtLtSZMgKOLIbGRvupxsaQPk5rvMaVdn1RmtXdznSobePR5hLdwJmSYxgqX'
        'YHMakjPCQRk9+KUuvKXVNa0IaTeaCbvzaTtDcTMqykb5HEq5IB5UFhi+TTLEmZ4+T+vddJux648VxtB1lV9KwZP25EX5mu6jo4GX'
        'iAZDurY5joapLqExZBQD2Vm+hv8Av/DNKFGs0XycljtpdV1GSzEUJMcUHnSEvMccAYg4VdydyM8BFN32l20yR3lpcabaXADLc24u'
        '04GYMArockAMpyQSMFT3isHo88yarDFHLLHHK4SUIM5XrlcEH2g1cTdgNQKxPEy5yWROHJ8R09XLwrY4w6fYFcH/2Q=='
    ),
    'chipping_sparrow_02.jpg': (
        '...'
    ),
    'chipping_sparrow_03.jpg': (
        '...'
    ),
    'chipping_sparrow_04.jpg': (
        '...'
    ),
    'chipping_sparrow_05.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABAUCAwYBBwAI/8QAQxAAAgEDAwIEBAQDBQYE'
        'BwAAAQIDAAQRBRIhMUEGE1FhFCJxgSMykaEHFUIkUoKxwRYzYnKi0cLh8PE0Q1Nzg5Ky/8QAGgEAAwEBAQEAAAAAAAAAAAAAAgME'
        'AQAFBv/EAC4RAAICAgEEAQIFBAMBAAAAAAECABEDIRIEIjFBE1FhBRQycZGBobHBI0LR4f/aAAwDAQACEQMRAD8A30t1tGSaAlvi'
        'x4PeqblzzzS2WTGea+eUkGTFoynvOOtBvekdDzQMs56ZoZnbORzzVS5Ism4ya7JbpXxnZhgml6y5PFXxn5eaItcHnDIvmaiUAHJw'
        'KCgfBAzRDP8ALQBAdzgbhe5dvBFcRwTyaWyTlB15qtbvJ/NWk8Yd6jskHFQkOAaHgk3Ac/vUriTC560RcVFA7lMhJf1rofih3lJO'
        'eldjfJJNJaiIw1UJLHaKqdjzipA5UZr4DtUjWIhoLPkEHvUomYciiWiBHIr5UA4pVWZqiEWsnApjCd/FJWYo2VNG210RxmqcS/WE'
        'FjuJFwOM1GQquaHiucj81UXc4JJBqqgBDoVDolU5J+1TIU8cUqjuzgbTk0bBICOTzRAiZVQxIlI5FQlgX+k4qLyuCNtdM4PXOaYG'
        'Ah3qAXYZcqVpZOWFO5MSHkA0Bcw4yQKjynkZw8xbcyjBwcmldySTjOK4s8jnvirVhLkmmnFOq4E4wQeaiXwe+KPe3bpihjbNv6EV'
        'nxtOqfR/NjHNFKrBOlTs7fbgsKLePC+1N+I1FMu4uXKvxRBlO3pmounzcVbHFkduaBSVMy6gUxZqqiUhqYPEAcVHywDwtY59zgbl'
        'luxAq9uVquJAasxjIpJa5jagsg+b2qagY4NSmHtVSnJweKnOUqYIaExgdasUc5qsKwAxzV0SMfpXF+U4yxfmFTW3LJmr7e14BFHQ'
        '2/8ASBzXY8Z8xg0Ile3IPNVMhDZHBrRyWSkDjmh5bAFsharCGFcVxhwAeaolaQt0IFOmtwFAxQ8yIoOQK4qROAuA242nuKPtXy+T'
        'zigZJFDfLjHpUbW9SOXaxpIyU24RFR4juThhn/So3C7gSCQR3qq3uFbkEYPpREjKUIU81YSCkXIQjpuNWTxKwqgkgDpXxkYD6V5p'
        'Ygwi0zEFtijIIwBzxV+wDtXyqDmvU5bms06saMM4qtrYHmiI1K9+KtK8A1Qp1FhpQsAUDAqm4Uqpo3JxwKDvASpxS8ramk3Fckqq'
        '3vXUuB07VXJD+Jk1ExDscVCHsxZuEGUHGamGBGc4oYxEYxyRVsKMQR2rcjWISal0b8iiAAaHSNgw5oyJAelJQzn3KpEyKoWM7qZG'
        'LIFQMYU80DpZgAbnbaPIAxRSQhVqMI2gVazEDFEEAhEwuz6CmUCp170otJNr9Kd2QDnPX0pyTVnWjz2qBj74o0p0qiY7fTFM+Sp0'
        'AuEAX0rP6nIybgORWjnXeeO9L7iwL5yM0QYMI7GZlFMhJ+Y1WY2LhgSK0j6ZjoMY9q7HYKQeM1Nlxbua8VWpl24yaZW5cDLE0Qlg'
        'FA4q82wAwBTEsijEM0G3Z4FVTOccA8UakHPTNRltuSanyJuZZitju4om2ttwyaBL7cEmmdpMNgx0r0MJBO4bVLGtBt460O8TDjPS'
        'jHuAozxQZuFYtyKdkdVnLPiuFoW6ZVSr5JxjApRqVwcHFRvlB1OMpllQyYyKkAG6Ck7SuZsgUwsZHfAIzUou4Lahij5ugq6OPPSo'
        'jqOOasMu3ijCkzpbHFg9KujQA80Os+etTWbisKlZtQsACoyAZqj4gYxmq2lJbrTLFTqqFLJtGAakJc8EUEsmGx1oi3JLgHOKATeE'
        'PsomkkDAU6tj5ZGRVWmwqUGBV10uzOapCdtzQKl7zrjrQrsXzil73JMm0HgUXDkrwe1RvZNQTuWoORmjYYYyvNCIBjnrXHuCg+U/'
        'at5FJqGoTLbx7eO9AyxqmSBxUZL0dC1UvPuzzxRjJy8zne5ajLjmrgFbpjmlkku0cGrba552mjXILqBUOSNd3FTkiXacjJrkDg8n'
        'rRGVxzTwgPmMCXMI6ngGu+ZJCvy5xVqqSASKseJdnrmgXe4HmAz3svQ5rlvIzCvrmLLYWrreIhMkULgncJTUrd2zjpQdwvmZo2de'
        'civooQx+tQG+UwxQLQs4ODTOwtcEZFMYbROKJghQNV+NBMg/w4xkCg7hMHpzWkS3DKDQV1bKCSRVZQARgGomjUtgAGiBAyr0o2OJ'
        'FIwKKVAw/KKnZLgkxDJG4bvioAFTyDWhe0UjpQk1ngE0p8RnCB20ayMCfWm1vbrwcYxQkEflsDxim0LpsGME0CtRoxoOoVaSCNcH'
        'tQ+pXG4YzXZD8u4cUsvJSaazkCogt6kAwL5FMLWfC4NKU4Gc4rpnx0NS7LQC8fJKHPFC3pKEkVXZSFhkVy+bCnJqgpazOVxfvZny'
        'W4qxZGxmh2Yb/YVCSYgECkDHUEkwtpcKK7azZbBPSks17sOCa7Bc5Pyt1rlWjcIEzVW12vHPWrxOWOdwxWbtZG45NNIH+X5jVA5G'
        'OTIYIYdq1WQduDVzEtgnpXCV20zIhXxBT7yuG2818dKPayVIsgZqi2kAbrTITKU2njisUGtxlCZu+XYx4qFnKNwBPNGapCXYntQN'
        'lAwY157qQ8GOImO3NcLhWzXYUIUA11kB7c08Mwh8RUPt5RsGaHvW3cL3rkKnb1qThfrVYckbgsaEGjG3GeattzJJIUiRmPoK5tZ2'
        '2Rpub0rLePbhBfXGiW3ia2gsL+KNXeBSxiVBmUF8Y3OTwOeFx3q7p+lbONeIlAC2zNfLdWlsga5vYEP91W3MP0oOfVdPdW8uaWQ9'
        'eIu3r1rJrr/8NPDk0Sme1uIEj8nyppDKZhnJLnseDijvCX8bvBl1Lfiw8IpZw2wJknt4RICoOBvVVyoPY800/hzXReeqr9MF/QT/'
        'AFjxJopE3wssgAycdvrVlpMWk2jpQ0vi3wF4me1uBePpE+SuIiqbgexBx/lR95YW1oi3+k36alpkjbBPGwbY46o2OhqLqugyYjy8'
        'iTuuM7T+DCiT5Y7UtvWQA1x9RXbtB5qtFM6nHNJ4chJmxm4HNcbV2gmvrZzJznOa5cWExkyOlE2dmYhlhSlQg7mNj+kbaegWMHFU'
        'akcg9cVOGbYMVyceaRgfrTF7tCbwFRO+5cgd+1DSMVXnk07az3jGOarbTvmAxR/lyIorUzFxE5O4ZomyTaMNWifTIvK5HIpTfReW'
        '/wAvFSZlOPZhgak7c4ORyKJW4IOPSvrCDzIwe/tRC2RY5xzTgbTUMakYgcEHpVUgAY0wRF2A0MkIeU5I9K1izUZkqiAJFEngDqat'
        'e2ESFgegobz0yQp5FGB6MHkZKc71xULOIbqqnnCqcdarsLrcx56VO7DludcbtGduRQbLIrZ5Iq4XqgY9qjJdwyJt4FYzrc1mMiJS'
        'MVZnKkjpQwUFvlOQKibkAlM1Qte5wES+O9D13WLJItJ1dLNAMvBJESkp92U5H71il/ht4ot5kttW07TwLiMPHMbx2Uqe49D9a9h0'
        '8/FeXAB8zsFH3OK3nibTIXtoY2RFEcYQegxXtdCWZeN6hqwE/Omm/wAIobODz9PS31Bhgy2s2FxIOuHHPbjNERXN3pMb2UfhWPSf'
        'iHHn7UX52xwSQOf9PvXoPiLV9N8N3C3SzI1wFC7VbJcE9Metfa7468Kz2aJcWKXlxgGKNyQR9cc164C4xuD3P4mA1eLSJ9M8vVIX'
        'YccbSoX3B9a2HhXwYbHQoZNB1eaaKVfxIJ0/DlQ4wx6YI9efbFZibXbzxFZXi6TpcYmtOfh2Clc54zzn9h965J4huo5rXRdb1i/8'
        'MaixBilUq9u/tkZHPvU2bNhb7yjHhygVGmuWF5p2qtbSODjoy9GFN9JYpDh+tZq/tfEmnTXFzqLm8tYFDxSQNnzFPUsjHjHXKnB9'
        'KbaBqttPaQy3a/DiYDy3Yjy3z0wQSBn614j4QrEp4m5MbqJoo2jbqK+uCoTAwKiFSMA5HtQl1OGbAOcftUzsWFSdb9z7I3cHir43'
        'XpmllxL5S7waC/mDF9wJxmkqOIhqZphKirmqJdQhB5IpO1yzR5BpJey3DTkKT1rfzLnVRb7mxl1GNkwGHSlNwxmclelLoElKDdnN'
        'F20wQ7G61N1LHINwFBJjCxdosAj5aOiugr9eKUifn2qy2k8yU4PFMwN4EdQuWKZfLA3miLFFJJZ8HNJT8dszjFWWYvcnJx6Zr0hi'
        'CgUIrvYamg1KVBb8NztxWUhmYXx54NEahcXCIQTmlMUoeTpg5qXOvI6nBXU7jy5iMigqc5qdlalIyaFjnaOIE84qy21iLJRiAaib'
        'KgbifMB0bzLYSTcbGNW3loDHuiY0tu7nMu+I/vV2m3ckmQ+eOlSYc15ChEMMKh2nh1XElU3AxIWU12aY5wpxxzQReUydc1Y1gVcw'
        'Az0P+GdhGol1u7VWht/liVujyY6/QCsz/EbxpbPfLbm/UPI4UQx4BJPGMmtfKZY/A2k2NuSDNA0j4Pcsa8u/iTp8Ol3CK1h59jNb'
        '7J3RAW3HuT/pxX1fQY/y/TBhsmNUB3CnxE0r3PxrJPa2MkU3yRp5ocSA55LA54PXFAWPiOLRdasYLvTFsk81oppJD5m09gGx09Dy'
        'TnrWb1VdT02w0i6s7gahDpUxIjj/AD+Ue7r1BA4+lNNU1rSLu1kt9bkNqJV8w2xTfsX+nOOhpD5nfZ3PSTEiaBh/jMN4e16PxNoO'
        'pXGnPfDyZrqICeBieMupwyHv0PStBPq9t4k0BNO1zUdP+PRMEwqjrJ6HZJgnI7D7V57Y+Hb2/tptLj/nV1otwfOWRrYqIiBu3eYx'
        'xjvyalDZWVxafAajbQ649qNkMwm2yqg6DisN14m6vzND4Iu/EugXM1vpcNt4h0+BistvHOY5Uweio3KMPTkGnk0HhzxVp93N4emT'
        'S9UhO64tZEMMi88iVMgEe/brmsXcX+lwWiW1w0+lTRn8Oe5tZXdT6iRW3V24a41eC28u70fWryEMI7uHVfIuGXHIYPgn6Zx7UIW/'
        'VTSfvNd4Y1iTT9uka2fgRG4iJnfiAn8rZPJjb17deRnGkSCU3LhnBAP9JyCPbHUVgbvRFn0S2bXLibwu8YCj4u3MtvPjoY5QcjH9'
        '0HHcUd4c+O04hNK8U+HdYh4xbi8ELn/lEoU/oTSsvTDJdeZNlxg7E3cltCYsMRn3NCSWUflfLg/SrZUkuNMgu0R7eRm2vFNwwb0H'
        'Yj0I4NdQ+VCDK+cc15aqcTEZBI3uopkSWOXaw4xxV9vaqzhmwCKKLJdSrt5PTiiF0+VTlSa1LJsQFthPnt4xGCB09KTXShZCRwfa'
        'tB5UixYI5pfJZSSzEhePWk9UrEaENRBYELKATxV6ARZZSOKKFn5agHuOtDS2czyYViFPauVWCWo3A2G3LoZWIAccD2q03kEXUc0s'
        'vbtpI/7OOQOtBWizSPiZycmvSTOzaBlQW4wuZ7eefkYFXQ2VmVyCDmld7EETKHJoCNrxTlSwGeK8nMci5d+5y1dRnc7ElKZwvvSf'
        'VLRfNDxPj71Zf/E+SXk7CsmmpXsurLb4IUtjNDmALXUSzXqai0l8slHbtXwvJI7jKHgdaXoZNxL5znFWKzqx+QtnvS8QVtzFxWNR'
        'w2qBF+buKoi1QGUZ7mgLhRJEGICnGDmgJZYoYyQ/IpuQXDTEfc9vtb1p/Cej3jkN5UboNvGFViP16VjP4jW7azoa/BSwLJDIr+bL'
        '/wDKTPzH36VX/CrXnuI7jTJXYq8ZeEMfT8wA+nP2NFa5EYbpoI4w8LowYZ4YHtntX2HRPz6ZR78QB2tc8x1HQINGdNa1O9hutNuC'
        'qxppb7JX3ZJYknqAPpzXBP8Awr0+3nksLDVdVuZApMc0hzgHPIODzx69KH8S+F5bGZpIDFd2AJOHYRzxD0I4DfVaz76ZoMrbF8Ra'
        'ejDnyLp2BX/ldQSP3FILFDREuADCwYwm/iH4mu9LOixas0liSVFmCIZVU/0AgcgdMUiXU985guJIhL/9HUI/KP2kH+tMp/DNiWjS'
        '913SLqE42lrjEyf/AJFBB/xCq7t9I0/dZafNNq8yNgLfKkkGPZsbv8qHlfmGABGGmf7QXDLb2Mmp2xbgRXlsb21ce0ig4H2oq90n'
        'R7RlfxHpek3E0i8fyW8dGJx1ZMEL+gpDNr3iZFRLO2t9LgAw0engwB/c85NL1uZIwfOsmGTnlQf3yDQs4A0JoUmbOy1GH4Uadpl3'
        'cW1irF445ZknZDjBBDLjH6fWuzQaj5ZRrDQtRiIyRPZmFmH/ADJgZ981lIrqzkUBp5oMdmJZf+sGmtnNKB/YtUt5D6Fyh/8AEv7C'
        'l/J6hcajqweCCMF/COrWy95NG1UuAP8A7ZDA/TitNp/irSJ0FrLrsiSAYEWpWZt5h/iXcjffbWHuL/U7eRFvofMQjKuB5oH3AOP/'
        'ANaIi1lrqPZ56zxjkpKBcRj/AAvkj9UrmRcophcB0DDc9V0lfOCS2siSxseGQ5B+4rRzXa2sH4ifNXkPh66urFmvNPgNkEG9mgmP'
        'w7KDyWjkyMe6vxW+vrx9QtoXgkSYSqMtCcqGwMj7GkNg+FSfUjOMYzcdQXkU7Dpg0W0WxDICAAO/elWn6PLDGGkk5A/SrdQ1FbaJ'
        'EkkyTxxWE8UthOSmOp1rgO+PlIz3owGIwZ24b/M0kihiuLmU21ys0aPjcAQR3wQe9OorZzbn5sADnPasxMFQuPETlLKYihsZxGNs'
        'QC0C5e3uNrLge1NtF1priJUaLaOmTVWtQeXKstwrKrcq2OK8tsHHGHUmMIZjQg5thMgYHA6nJpVcX0EN8LcAYHBPrT+HyYiI5WGx'
        'V3EDnPpQeqWlq1lcTrECSMqccjFLX5HPI+BHJhNeYqvZ4ruRYUBBB554IqUeg2W9ZUCtIDwc96AtrUw6nAHlDJKG2sDweM/6VpNN'
        'jWH8UPuc5MSk8D3qrACym/cQcR5VcqOiql9HFJtEhALgf0k9vrRVz4fMQ3Abh1rmm6gs1wzsMkNhmHr61rLCzTULXPxOwdy3am9G'
        'mPiy13CaqsGoTGX2g+fpbyRheOoHUVk7bwbdXE+/zcgtjBr024t3sJ3eKTfChJ2uRkr70suhcWsovrNQbST+nup96HqF4ntE45OB'
        '7oFB4dm0drW9tcCaBw498dj7HkfetVthuv7RBGTFKudueUYdRWS1TWr1mSMEYZgv61d4S1uSO4IfOHU7kPTrwaq6X8QGBwpGjBLh'
        '/Ep1vSEvWkQZUnnBGazF9/D+xnJl8oo2eVUAjPrXo90YJpWeMABumP8AvStrSUlpA4xkYySRX0CsmQX5nBiPE8zk/hvatPmJiknY'
        'xOQaX3vgO6hYvDqU2QP6xnFem3CXAlJICnnnoaXXcswiLSh9w/rV8g/Ud604sJ9QhkyA+Z5LeaLrlmWYAPjptTbn9KA8+9hbdLA4'
        '9cAn/PNem3imfDyllPT5Tx/69jSW8t4cmNWYsagy41H6ZUmQnzMnbXmmyEC6V1PqYuaNlg8NtAGFzdNIegFuD/4qIk0p1kL+SXHU'
        'Gi4ETbsaNc/0gLg1NyqMLVEtm9ojj4abUrdgeoZFH/S2f2p3Z/yyedZLy+t5ZFOQ8qbZB/iXaf1zTfRPC99qfMNnbgf3pXCj96c2'
        '38PbiaJ5pp7GIJxhULE/5VnyjHuA+VQLJi2WXQL6IRXms2Y4wpS9KY/w7gM+5BNBXZ8UWjB/D/i7TbmJTkR3bJIxHpuAz+5pnD4L'
        '0+eV4Z4zJtOMpEOTRa/wr0IxGS5kuLVmGY2BXbx2IIoj1q13eDO463L/AAr491KZXsPE9vY2coAEM9vcBxJzggqTn34orWPOvZ9q'
        'XaLg8Bsrn9aSw/w21nTZzNol9ZyODkxvEFJ/YioP/tdaPJBd+H3mEY3ObXkgeuFOMe5U0TrhyABjRi2UA6m11HS7jS9DS/imBugV'
        'W4C8h0YblI915rRC5Nj4bt7qcpJdToGjQHIJPQ+4xzXldj4htEmVZbq702Qn/d3KFB+oAH/RT20uboJB5couLdFOwEeYgUnPBTLA'
        'f4a8vq/wrM4K4G7T5H/n7+4sKR53HOhWyx2ciStmW3me3Yd9ynj9Rg/en82tQXGmi1vLDcYRhXx6VmZIdRh1a48yeCKPULlAki/M'
        'AyriRgTjoFHOOSae3NkJnSzsZ7q5kEW93WXIP34FSP1pr401WtxIDDc4lxpBYSR2V00LDEiiPJT/AI1x29R9xVF5DpEsAlsdW3SL'
        'keTIcRup9ff0PvVdi+sQvf29razgwDEUEnD+U3IbA5PoetCeGbBnl+F1CFkdnJyUI2nsPoT+9ac2ULxXwf7Qqb0ZBtKtbz4A2F5G'
        '8rysHhdsSQnYSQfb0PSueJNM1e1xCmnyhMhVYDPAppqvh743X7S1sd9tPJG7CaIA7QCMDnqmeT9Kq0nxVdQahDo88e++t7ja0okL'
        'W8q4IJViepB6dqWzFr+T+3uab9nzO6VoD21vChmjjlnwfxDgZ7daZw28kFuyR3irs5ZjkCQ+q/8ADRMtlpd5JPqc17L8Skm2OGVt'
        'oTPAC54zz1runapaWnkQz27XBtX+Fm/CJWN17k9Ccf51Z8Ltl5Ecfpu/5jcakRbHp9xcxTSzynAGSD39OKVnVbfSvL+Iills51US'
        'kj5FJ9/Wtx4t1a00aNLryLmWOQ5JEYITP970+9LNd/2a1rQYrVLe2nuZIlG13AC5HJHoc11Y1ycSe4D+YHx8yQREGqaHJHcRSWsY'
        'ltJirRODnr2pbLaT2d2Wni8tiMkEYPU1P+YanZw2emG3ubeOFyVUOOYUI7g9CcD9aZXbyXbfFT2EpiCZkIGSg9fpUhtsp5aAmHCu'
        'NTR3CdNMc1nL/Z3lLL1TqD9zigWEkIMTrJGDg5K9Kokuo4bWQ6dfizkK8tNEHjOOmV9Pcc0otNeuE2pr1rHZxkErfwK5tnIPOepX'
        '7imD8Uz9MwVF5L+/+ISqhT7wu/vdrBZXVfTPQ/Q0pmvohISxDoeoxyP+9bC20iw1qO3jjexuV5Ilhl3qwPqRjn9KynijwjbWdgby'
        'NbiADIkUHevXrx27/avWX8WRgLsXDGKINcvFtgWOFULuD/0sKz9vczahMGZTEo5UjoR61prbRbmWV9OZ2KRvkSjJCknGPof/ADpp'
        'beH7CG7aBvxnhXfM6njPTA+/X0xQv1aty34jFXiDfqJLIR7QpLSHp+WtRovhNzKt7fW7KnDIMYB+p7Zoe3GpNew6ZBoiJbSTOs8t'
        'yvyKgxhx7+mPStdFYRy+Zbx6hIluCFVYztMgA4yT0JpXz817T5g2Ce6aO08ORzadmKB1XHzGNhhftQs+h3dvZssEUsyluT5ZFLVu'
        'tattdt7Z4ZYrJo22RAj8eTPBJB6f6/WiL3XtXsbVDZ3/AJU6uUcyqUjjUH+rdxnPBHHt61Mci8e4n+v+pucY8qUNftKP5TeQItxL'
        'bYhRvmCdfuKh4mjsdS0mSK2naOSCL4iSKU4JA6KB+p+gphqvisx6TBqVj8LO6/8AxdsMblPUspPVf/KrIJNRuLL4+L+USrdo03lt'
        'bEysWX5VBPoOMD0pOIvn54mGq1ObuGmuZ6xjvZL0XOnyPuVfwkcEiZMZ4PcjvWn1DUbOw0rChfi7mMSS45MaddufU+lZzRtemtTb'
        'hLeOO4iiaJY48qYjnaqsx53ZBJ56CjzdLNp7SXccRZXJchO/UYPU9cVL1PXDoMKqLYnX1q4lkGawp8RLe3ct5aTNqNnauFG+GGRQ'
        'zMQc8j3HGPekWn6Po8mk+bawz2uqJieRbdyg8pmPHHHYj7UdHeaTJ4msYYncNC3xF0078ZB+SMcd25PstU6VfPBIjYAlmia1cryC'
        'u5mRx7DceTXl9PkyYGJGQk169GZysR3q2+58Y2kkwWG1VJIk/s7yCMYBBK49R1HrWl0ofy/SYrxJ28yeR1ZfhpMRBcdOCQDkdqw1'
        '6LzVVju4royPaymRyjnhiOc45HB6GlCX+rC/t5zrt3Zwy5jjkNlI4BJOBuBx25J5xjivZ6fJsgC7/wBxnx33EzV3erC08aC+S6ZS'
        'tm8SNJG4VTkEDJA9T6UXqvixr7U4YNPvJVKRD4iRl43Z/KOSG5AxjGayGn614qmuJrNtQu7a4t2McqByxzuHJzlTkYI9jTO4vNUm'
        '0uGSKbRpbm5ldf7RFtVwuOhQBtwzjPfvWthfJeNNTArjTHRjjwxpk2qa1MJ5Z1ESLGyx7nklUnO5mUEKMEDGR0pn460O0tYILCWy'
        'nlszNH+JECiqCcc4zhuh3VmPC2snUI00qHS7y0tYEdHvra8kiRRknd82dw5OG5HTmiPFnh9pPJudP1/VDIfLmtfNbJO7aBgbuBkg'
        '8g5IPtXo4eiTDhK/9vc10SFi6k0W5Vb1rnUoGQLDHMEVsdCr56MP3pxqN1Jp9w+qaIthiSAeZDvJ3KFPJXuQAftigI7LX4zGt/DZ'
        'yxCPE6uvmecwOGJXnaO+7PBqzV7PTJY7e5XULiC4twdkbwksvGCgHHH65zxRFDWtQwyjtJg41e5cLdzEyQFcJsUMHUjkHswHHHcU'
        'ts9ENzffFRw3NtJb4NwgQ7HU8jyj7jHB5Ge9PNHaOx0yRmtZN8ZLgo52yoRn5l7MPUY75qN5rd3qenGCyl23dxEWWSZcLCo+UnHU'
        'cgbfUk+lLKI5AfZi+1DZNmZ+1uBf6/e3Tos1wCv4BONkKn5V+p5J+ophGda1CK8iUA3aMElhzllTHykD0rSW1pYyWMaxxWkiQxhJ'
        'pLMbzIe2Tw2e596zMEV1p+r39zFDqEcUyRhZHt2wcE5GQOBikZMGTE3cbBgix+qK7kS2Nm8VxF5UoIVfNXClj0B9uD+hpTDHqNjY'
        'fE6gIZLW4JEYEhKphTyVIz0H7Ux1y/tbrXRcXwvYrBFwHUBvTJbAwOT9f1qjW7eXXEtbTT0LQSMXicPksFzwfTJHQjp60vghFV+0'
        'GhxsRdDeXdlaK+nWNvpwBWRhATGzsrAgnHTK5Bo3S/5kPGOraVqU8wspCr2u4jaS/wAyDI6nOft1q23sr5raJ9VsyitGB+JxhupJ'
        '4HrjpTa7tZLqxsrpUJaOMJGJs4G3v6/pRdKeRZW+soTKFbu9TAa14su9LM2m2iCCaSZo7i4fLKBkDA7AjB561rdIma1t3aOSydZW'
        'ESySAhWwCxxjJ5JPQVTqGlaVHqN3ItraveTRfO0u7J4wG44z+9F6JotxBod/qdsI5WVGtrdSvPmt+eQA/wB1SRx/eouow48g4kXf'
        'mtQMufmKTUX6V4h1TVIriC3SFYbQNIDKN29OAEDduSAPrjqKL0vxTJpkKpLaG6jmmMkhViGDE9s8enHHSqNQh0fRdATTdXjxczus'
        'tz+IVYsp3LGAnQDO4gY7elLbK3vLy9g+BeUTNMobcg2uh6naerD/AC96HFhOJeOPR/mTPiOQWTN0us31xcpPYvYRzWrtMkUkjK8u'
        'ByMYI+uD6cVXd3el+M7S8k1C8ayuJVZLmIlGjdlHTzE4LDAwSMjGCK8/uoTJPdxzeekZujJEY+djY/VCcgg96daWsUJn1W8ECNDC'
        'iMVbG7nJZsHkkLyTin40ORyMg1HhSMdmEWq+HbPT7LTIrQN8PIJbhp5wXdCRwSOGGP7vTB4GK0Gk+J7/AEa9imm0y2mh2sYWkBdF'
        '5O1gyg444pVONB1PSczRHzpQ0hmjj2GM5+UDpjA496AsFtobq307UbqeaNvLaJIPl+IbPG89lA645OKnfG3ygKa+hH+CICtwPmXa'
        'raXOuahdXrk2265MjuwwoVvzHjp1FOdEj8iERTmQiGPaI5Ad2Ou4Z65GR7UB4p15pL425ZoLZB5amJeUbHJP3P7VktB1K8hnnimu'
        'DLKsm1Z1JIlI5bB/9Gl9S642BC2biiaGpotLt4dVN8j6c0uoz3G+4cyjYufyqRjjCgAAH60nge4udevY5Le5gsIVRI4vJIkdgqnA'
        'x1AOfbFaKB4Le1iw5WCaTJjjU5Oc7s4BJ/8Aalc2raRpF6sFvdwSK422V4VZzGxxlWJO0emO1B03SY8n/I+rhlrW1jrRrLULB0a9'
        'is43vDslhRo1bJ6MxH+8x6ZJoObTo9P1ieJLWyjlluURp0V15ZcEKUOFPTGRjJNWBbp9RhAmmkhjkXGXOSM9PpWheKRtakjEotDI'
        'oZ5Qu5ioHoQRVKZMRbljB0a3G8tgA6i+98Pta6Jc6hfzxwkxEhr6cn4ZT9Aeev5QcD60m8LW8FqsVjpMM13jE6LligJwGJDgFQCF'
        'IPfPHNaWeVb2GTz2m3DgSSNhmA4yQBgZ61Oy0i/t4JJ9Ki+EkZgQ5m3ZTPQluSM805W+QsyDx/MF2awpjHVtMgm0J5ZUYSuPmt0Z'
        'Qq4ye+TjnikGs3Ov33h0sGPnWVxD8KzndK8fHDYHRWA6Z4OaY3unzG5uL26vt3mbBLEhBVto7enFL77S/wCZWs38ttVeaAZcmTac'
        'H0PbiiydTYIAv7e4RLY64w8avcOba41qCImW2WDz7ZwvlzEZYFSMhd3pg/rUNbXU9SC3FjHaCWHhvKlZy45IB3dCM88+ooOCG2is'
        '44b17hJtoZOCBx1GOn3ozS7q5niZ7IxKI1J/Id27uSen7VPk6zippfP8weXPTxfZaje6dpb/AMw3OPPOVdcD3/8AbpXbm6Q2tzJp'
        'SBZ54uSzeXg9MZwcEdQfWoDSrmRUlla4lm80ys6P8kZJyMjuB6VO6utPvUEljOEayj8nd/fbJJ+vJNEjPiUEmLTGP1DdQOLSFE0M'
        'kk8vnyKN0qkfMQOr88ZOcnHfNVW5vlikLvPYmM7du8p+XOcYwWHvT/QbeLUDDbuqIyrkOw4d+oBI5FR8RRRvDLbGSMKo+YKpJyO3'
        'XpRZQr4+fuNXISCTF9xZvd2SfEzvdW11FmWOT8pKnIbd1Df51T4Z09bAkW7S7SQVkbPA7DFTsk3aWtwI5JTnaIwMbeev0ppc/GfL'
        'deR5cbBQFJHYYqPqncoprQmDuHKJfEd48t2sE7MQVCx84zz39ac6fbJCmnnyWby3Pyoccd+KqlsxfpA0tukhjcFJd3KjqRUo57pL'
        'vyQMyGQ4GeFXPrR42GNhkPuJKsWqURadbfE3cwLnYQmerBM//wBYpvctb7EnilkiaJQluAOY1PXAPVm7sfXijcW1nZH4RBOzEmTc'
        'ec96M0b4cwR+ZHG6bixYnJx0249qtxoQbjVoNqYY6xYS6x8IbC1geAny5JozIZPdT1zWpNzbtLb3sZsXfKhFKbWDAcgn6ZpR4qtU'
        '06/dxbQSKXJVypyAemOcVbpkct9EWhYQqnOxUwXPXg9zS+m6gF2xlrP7TiSPUG1fR7XUTbyQyC2yRG5LFtyqThOD9B9qzut3+naZ'
        'ZS23leaI5RvRIt7Ej2PQY7VtvEVj5EaXFrYTmWTBIKbthxyevBzSb4GTV90MsKw3qoWeN1BJXseDTchYNxPuGcRZAYPpaaNrFhaT'
        'C5W1Z+VMn5XAP5WB4B/T2pjY6TZ2zvd30aF7JWIP9IyMgrx04NBaba2mmTJpc5icP+bcM81PxdeSWdmlrGktxcSSAxxxAksPt2oC'
        '6h6imthVQK10221oTtG7QAs3lvIMgtwaxV/4e1SG/Fvf6nErCVWi+ctwD2wOM1urZZ5JPiJFeKWQjKCPbg49B0phdaVaS/2trwyy'
        'j5drAACp83EIeXkTsahiZkG8q+jnszJOiRtuAQ43ceh6iq28L6dqvhuOxnjBimnzAwG3aR34HBrZIlr5NvYxafBE0HPxWMtjOdv6'
        '0wuoiLaIAC5iDbsEbArH0xUHTALoNdbholNQk7OG20y0kiljJlaPcJCOc+lIdVZWVJbdpRIV+Z9xyKe6jc/FQyMqZU9KjYaXDZaa'
        'JrkqxnPPfAFenmw86GPwICni1QeNJz/Lvh1QJGh37+d2R0xTWG6u5beVJIlTGQFHy89qXw31qLg+ZKURFJQjjP0q/TdWsbyWWKVS'
        '7N1YjHI9qb06my1//ZpexXuVS28lpYST3EazIc/1Z28UFZ2uvz28LaT5ZtXk8xsnGR0++OeK01lZb9PkhicSJKCcN2HPFZ+71DU9'
        'MkWCEmKJQeF6GhKIlEg19oaniLaWi2KSubi8beuVaMjjHt+tH2cMWk2s88aKIZgPtzWImvdYfVnubmI+Qwzx2rU6dcSahopgkVht'
        'Hyg+h6VPicM/GvHiC7AiGQ6gZNMa3RAh3EH/AIqyl1otskoVkljWQlgYxxu+1MRBfxW8jMjRYyFwK03h22kTTVe6lWXB4z1H/en/'
        'ABtmNtOXQAmXgibTNKkVpts5GI27gnvS3R5L+W0lt5P7Q2MAgcnP+tOtd0177XI5hIVj6EDoBmn1rBaada/gRjf0Jxj96UiF2PoC'
        'G6GqlFjaT22hSI0aIjAMMj5z96VvI93C0KMyxrycimmp6zNJaGFYwoC9cdBSnQFnuFc5C5bbn6UWZg5GJPpFtqgJT58scJhj+QZ+'
        'ZiOvYUw0S3t4kF1LOilvzA1OfT5IYfLC7t53ZquSHbbsSn5Oq4zmlY+n4OGfdRigizD4hbTQvKrMrJ+RAucihxBcIm4ZXjPy8YNX'
        '6K0lzkQbUIAzkdKr1G8ubaNreXaSx4xVTZA+O5hoLALm83wtFf4ki7MeoocGaBo57KYfCjsR/rTNbW2uIkMo+Ruoz3oO+s5I4vJg'
        'kHwy9ianOMEWTuYT2wptR85GjjiMhx164Hc88Ukk8qO/W9LXCzAbQ27nFH6NeAStEsamMDDHvUprKCYSOhKKD/U2aY+Tnj15ED5P'
        'oYKZbS4niMkbyyB8o57e1OJ7l41CxQAzOdgK8ED60BodsPjhCUDgc4XoMUbql0YJ8xwjcOFOc1iFuHN9Qzl5LUGFnPaXAkdvMYEM'
        'xz0JofVriyFqrlGScMCQo/N9qMbUEMPTfIxy2e1VwrayyBlVQw/MzCkZArGhAXWoPpt/bTRPaRwlsnO5uCTTF7+zeAwvEYpEwCFc'
        'HP2q2DT7NPx2RSwPUHn61Q9rYtO03lYYA8YqhERU2Nx7lQKE/9k='
    ),
    'chipping_sparrow_06.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABQYDBAcCCAEA/8QAPRAAAQMDAgUCBQIDBwQC'
        'AwAAAQIDBAAFERIhBhMxQVEiYQcUMnGBFZEjQqEkUrHB0eHwFjNikhdyJYLx/8QAGQEAAwEBAQAAAAAAAAAAAAAAAQIDBAAF/8QA'
        'JBEAAgICAgIDAQADAAAAAAAAAAECEQMhEjETQQQiUWEFIzL/2gAMAwEAAhEDEQA/ANHuEblME6MUmTXH1zilP0pNON7m6geuKU5L'
        'zXMBOBvXmWnpnoJNBi2OrWkJWk1feZC0Y0ChcCW2EAIVRBE9CXAk75qXBeillm3Q0KOkozmirlqaEcjSOlQW5YWoFAFGC4ooAOMA'
        'Ui7OZnF3sqES1LCfUelXLZZVuJCt80wzWkOyckUUtbLYSPpAFVxxcuycpV0Ky7LJ17pyPaqM62BhWdJBrR1BsJ2xQy5QWlerGaeW'
        'L8BHILlnWsoCAnBq1c1htnJPqxVhiOWHSUjahd8ktpClEjasuSMl2aINMr2psyni6tvYHYmnK0xmeTlWBSfZXnnGtk+mmBU8sx0t'
        '77+KEKQJ2y9c0NttlaBS29ceU4QUbUTfnpeZDad1ULlW9+Sv0o2pZTt0h4xpWWYc4vHptVtQASSlODVSJb3o4BUNhV4ujQEEYNOo'
        '6Fcin82tOcn8UQhFDmla1b+KpqihzKiM1ftUdsqwoEYoqmK3oKoba0ggbVTmuNZwE70XbaRycDxQG6oUnUE03Bk00wbdJbbaDpO9'
        'KciY87IIGwJ80wPWuRK+rKRUsGwR2jqc6+9dJNIaLKtjbeK0r9R+5p4tqhpGtW+KFMsMtpw0gbd8VFJlFlWArfwKWMWF0MTqmN1H'
        'egktTDj2+KHvT33E6Egk1xCjyHngF5A6mjKLsCaF5lUiQ0S4kqz2xS9d4skv6UsOBI9qfbA1rYCiAT7CmRu3MvNjmNJz9q0wgmTl'
        'JxMmtLTiE6VBQVRBlp5T25yPFaWm0Q845KB+KnTZ4XXkoz5ApZYZemd5ULthZ0oBNEJD6UHGRgUXVamNH8MEfahdxsLzuS24RSrH'
        'JHOaYIkrQ4VKBO1ArlxB8krlhfU+elHJ9qnRoysJKzis9u9qmTbglCmljJwaeXR0aH2w3hcxoEKz+aNOvakjNAuFLIuFHSkpVnHe'
        'mFVvWsd60QcaJSuwXOcVoOk9qVZ8d2Q6cAk52p6XaVK23qm5aFMLKgnP3FZ8ystjdAyxw3m46Qdsdavy4uUZx0qZpZbGCN/GKrSp'
        '4SSk+ms6ovZUYiq5mok5pitqMNALxSm/dUtubq6eKhe4taZTpKwMdqKgrBJ6HK4qbSggEA4pWlyyl4AnAzvS5L4vDzpShZP5qjMu'
        '7rrOdyTVJYmtk1JGi2panQMHINF0MLQdSRWccJ8QEuJaUcHpWlQpSJEcEHtV8cItEckpWWkuFLWCarupaOSRk1XlTmmTgqBNVfnF'
        'ObpI/euyNJAgm2Tzn0ttahhNCmluvu/wwpR842oizCXNWFOu+j+6OlGIUSPH3CRWWEuTtlpa6ALsaYEYyU1AxCUVkKJUrzTHOdbU'
        'k6MVXgMaSVrG5qnO9IWvbKSIOOgx71eYjhpBVVhzCR6U1XW6SCkZUf6UfC32LzSA/DdtcbjpyshQHeic6TMit5SkKAq9FdjttgEY'
        'NUrotK0nSsGmcUgptsXJfGTkd/lrbwferULjELOFN0scQxFOSDnTnzVS3Q3UoKsHbv5pfu+mNUfZp1u4liOkJWdJo5EuUJ5Qw4k/'
        'msKvr02K1zGgtKvahlr4uuzMoNKClHO3auWWUX9jvEpLR6UcbjPoOwxigsu2xEOFwtoz5xSvwxxPIdYAeCknHert1uUhSCpvJ+1P'
        'LPBoRYpJhVUyKyjGBtXyPd4hXpKgN+hpRPzkpBwhW/4oLcYF3aXrQ4sAHas/n2V8Rr0d2M8MpINczWWyjOxpC4clzW0BLqlEimZb'
        'stxv0dxTubrYnFFSWyC4VJA+1DrhBS6gkAJPvUz0W5l4nVhJ8CpkQ5Oj1kkCoN7KoT7pZOYk+tQPtSjeLPI0qSCpWPzWpSmdJwrI'
        'rqLamn9ygHPtTLI10FwMEbts5qVkIXgHOcU4WtoLjhLpBONwa0ydww1yiW2xk+1ZvxfbrjbVqcjIJHsKM8s2gQhFM7jpaizNaEjG'
        'd6b7dekIi4QkkgeKyi2SLq/NSl5OlOe4rV+GLap5hKSBSQnKI04xYEvN7IeBKSCTtU1kuTrkhIcUUpPmj944bbWjSlkFZ9qHxrMu'
        'GSFJ1Hzii5SmBKMQ+m7R2GsBwZx0FDnuIHZCyzHO3kGk7i+RJiJPJJBOx81LwmzLUyHE7k7gmi36QK9jrCuaU4D6sH3NGY0ttwDR'
        '6vFJTtqmKcDjyjknoKYbU0uOwCR0HerrIl2ScWwtIUtQwo4HgVCt5tICc9KXb/fBH9BWSeyU96CG7zH1BCm1NoPYUZfJS6BHC/Yz'
        'wLol4BJIV7VNPwuOpbZIOKy6Vd3bfL5jThwDuFU3cPcSxp7QQ64NRHmpT/pWKBUp6YZimn8gatj5FMVnYSppKVGrq4EaQOYkpNct'
        'xlNnCNqbG1BAknI6lWaPITpWAfvQtfC8RK+ZyU57EdqJzJqo7YCvHWglz4uj2qE/JlOJCGxsCfqJ6Afemnlv0dCD6L6zCtMZUia8'
        '2ywj6lLOP/7Shd/irAac5NoiB4dA7IyAfskbn81mnEV7vvG/EUaGysBx5zlRmtWlKc9SB2H/AJGn2N8JrBZW25HFfFKVN60oU1Fw'
        'kqJ6DUo53rPDDkyv6LRsbwYF/tdv8A034k3l0rJvK4wz9EdpKQB9+tQNccXySh39OuV4nKQPpbVr3z1OQRitIvN0+E/AvD78yLZr'
        'RJfbaKkMLdS8652H1E/n2zWOQvjfaRfC6zwlZ7b6MtSoTCQ61nqCNIBOM+1aYfBcXcmQyf5GElUIDxw98TbpapiWeK7LJjRtWnnv'
        'slperTqwFAaFEp3A9P3rceE73ZL/AAES7VOZktkbhKhqT909R+a86XT4pscT2B2DMhuSi8ktqcYY1N7pwCQSCCQSNjkZNK0Livhy'
        '0Np/TYE+csL1F4rVHU0sjdCVhWdI69a1Swxkvw89ZZL+nsWZJhsj1uJGKqtTYr5whWR7V5ftHxTvt6uP6Za+H7ncSk40tTFSFD7q'
        'UkjH3I+9bXwkLkILbtzhqhPEZLZdCyPuRtn96wZcTxum0asbU1pDRdEIScpSFZqeztn+YVTMltagCoGjFvcbSgLURUZ8V0Ui5ey6'
        'G9SdOKE3m0xnWlKWhJPjFEXbgw0CSsD80s8R8URI7ZAcBP3pHMKiK1y4eRz+Y2AnBztVy3SnIBSjmAY80MVxGHVqWtWE9h5oBert'
        'IdXqisOq+wro829Id8Utmtx7jHdYC1KBVihN4vERjOVJK1bACs6tcjiB9IShpbST3UaKMcPzpKiqS6oqPeq5HJKkJBRb2fpr8OVJ'
        'JdKTnzRuwfKMsjlqSBVaPwXrILjhPmjbPD0eK0AknYeaTHGS2Gbi+j7IukJpQBOpXgdarv3JbzZDbSgD0GKsx7YkL1JQj74ok1bE'
        'OJwpY/FNKEpewKUYiYbaqS+HXB3osxbGwnJAOKL3GAzFRrS4f3odEcC3FYVtU4xcdMaUlLoQOJrJ87rLKNz4pVi2K6WqYlfNVoUf'
        '2p3jXNKApUhYFDbpfIr2GmfWvpk1ohvsR36C1mvC2W+U44cgdzRmDekvqKetZ+WZbidbexUavRmpUYpcSooWB0PQ0ZRUegXY+3BL'
        'cmCQg+sCsqunDz/FnFX6YXXmLfBID7iAFFbpwdIGeye/bNMy+JVRWVrkoKdKSo++BWT3298SrsMuLZoy/wC3vKelykLOpa15ISgd'
        'SAnG/wBqpgh5Hvonkm8fXYc4y+LnD/B86Xb+DLJDcvAUGnbittOAAMYSBQW3cNcS8cMqu/EN3lNqfPMDYb0IB/8Aqn7UE4C4SiNT'
        'ky79CQltpzlnGVerrkk9x7ZrVl3d5NxMWJKh6EJGlKU61EYxk7Yr0V1Rhd3YpQeELJYVrXPhNTkr9KVJy4obYzpOMnc9KsW/4a8N'
        '269M/L2ydOfnq5DKUuJ5TOpOS4pBwtIAJ65Ge9M70kOam2ZCvnFjZDbYwT0OMCn3hqOzw7Hdut0Z5IdYBefAzn2z1/FcdZlt2+Fx'
        'ihrhm3uzERkgvhTwIbAUobrVnSVDfIA32/HzhT4PWNmSpXEMpdxUh0hLaVlDSgDtnG9ard0MSbL+rRFykNSnPQ06DlIJ9I36dSfz'
        'VKPZ5/PQOUQhPvWb5EmtIvhSe2MnD0C2WqAiHaoUaIwkbNsNhI/p1/NFRBekJ6YHsKs8PW9CWk81G9MqWWWm+gFYY4LdyNby6qIo'
        'MWJZf1KJow1alhHtViRPix1HJG1BbpxvbISTreQMds0/HGhLmyvxBZpDrRS2sg+1Kw4KfdJW86pVE/8A5AhzHCllYIzjNE419Q43'
        'sRvQjji3pBcmlsVU8JsRnQVZP3o1brRFSPoRgeaIKdEg6sVTnSERxq1YrXxVUR5OyxK+TitZ0oGPApfuPFEeGrUoJCRVG43Jp1ZS'
        '696PvSfxO0l+OpLSyQem9Z5Y76LxlQzyviVEYSooyojsK+2fjWXeJCW2WNIPdXak/hrgx1+OZC99uhowxDNmloJUEis7dOmP/TR0'
        'pe+X5j75xjOAcUON+aiLJSrIHXUqhrs95yBqiFTqleaTrnZb3Ml80ulAO5SnrR5X0LX6P0riFFyAbj4B+9WrbbZGOcSR3pP4LsE+'
        'PPDklC8Z2zWovXCJCihLykoOOppXNXTG4+0ZNxHaHG4pUzqJA371mcuXKh3FKVpUEk16euNqRklCAQeoxS3euArZdf4gaDbvYpFa'
        'Z4ZY3ZOHyFJUIfDMxEmMgEEEe1Ep65LaFBTYcQehA3FWneHZVhT62ua2g5yBviiNtuNslNhlwBKjsAayynJOmWSTVozrjuU0eGHI'
        'zQUZMkhlOE5KR/MT7Y/xoRBbSEtONEtqGAhJwMAdh2yep+9GeOmoj3EKmGiCGAArrgZOVHb8CrtvYjholMZSUnBaKk5yPOPNet8a'
        'FQv9POzz5TObTHYeaX84tpxtKQMrBCsnOoDbv389qabLw5wyUCU1YYykjOlDY5Zweu+2nPjtVByNhTaWm3FOHdpaEZwRvqx0GMfb'
        'arkJiSlIcLJlOOoCkhSjhP8A9iDur+laKIWPfDzXB7CGn41tgxlpA6IGvPfBNXi/brxNaacU04wwoO8oDKQsZ04HTI/0oBaOGUXJ'
        'lNwu7C0OA6WEJP0Jzkn3JNPsKzWiBBbS1FQFIVkYO+T/ALV2/QNC/wAXiLFjsz5wEaA24hS14JCUpI3IH3/pRVIhqbS62ptSFpCk'
        'qSchQPQg+K/cSpYfiuMKcC46k+poozk/5f7156tXFV24Ru8uzOJW7ZQ8oMpJ3i53AT/479P+HL8pOP2Rp+O0/qb25cm2HQkKAFXB'
        'LU81srNYW3xDcJU0KQoqZKsgg9RT3aL7IaQCoEpx+1ZVPkrNDjTGibb35CFBGRnvSJxNwK7NUSp1wE+KcIfESHAQpYBAzihV84xj'
        'xdKlKGc4xQklJBVpi/YuBkwMFSVLI7k0XlsuRvSlkoA8CiNu4ttz7QUtaRmo71fraIxUh1B2roTUVR0lZViTCAQVbiqF5ecfZUEZ'
        'OaU5PGEdVzEaOQVqOKZ4TmuPrKkkkZqnM5RM+vTMxh1Sw4rSe1DWLkoPtJcXq0q3HWrvxHvKoitKW9WTgECgfB1om3SZ8w5qAO4H'
        'ilk1WikWbRwzJb/T0qSk4KdqHT2TKuWA2rGe4qzaWjb4CG3V7gY3olFdScuaUb9CKyNW9jFKQr9PZOlvYDOAKrWO7qlSDlheT0ym'
        'iYejy5fyzikhR7Gnfh+xW9LSCEIz5xRXegPrYKiKcDA/g5UaWOK7Jdb05y2dSEZ3Na4LYyhGwGKicTFjpI2yaPit2xHkpUikrSfS'
        'R1qu4wprK0AHviuvXsc9a/F4pPq3Feu1fZ5qdCtxUZMhotoSpBPtms6n2KZbm37pJfJZaSVjbGPFbVLcYU0BpBKvas8+Mzyo3Big'
        'EaUOSG0Ej8n/ACrNkwcnZohnpGew3Gy8GW2EgLGpagRk+5JpotbkZTur+GsMjoo5BFJUIFMXXuFrG+e/+lM9qaeYgIykDUfSnHX7'
        '4rZHRmexigFyRMWlrAS6MLyP5T39hR3h+zoF4LocJYSvIb7ZoNamwhRddUSte6gO57D7Uy294JjpAWQEnfsc53P/ADxVExWMjLzJ'
        'KmDnAGEnoBU0Z5JIU4QSU4+3vQiKpxttTq/UgjORXCZnKZKioHeiAgvUtB1s8zBA2J6/isQ4mtTC+I5c1Mtxl17CCrSFtKPQEpVt'
        '7HPX7itO4gmJcfKicAnYg9P96Qr9KUyVSEMK5wVpyjAUck9QcgjG2PFZ8yUlTL4m4u0LnAPECLLem+H+KmEREPL5cWUkYZKum+fp'
        'zt+/3rbWLYhKOWobEbV5q4svkcFSZ0FtcRClh2Mt1IWhzHp0Y6A+xNa78BeNl8Q8MNRp0WU07FHLaedBIeQn/wAu5AxWBSeJO0bW'
        'lkqg1fIZikrClI09DSXcgq4P8tYU4exTWv3C2i4NFs4INQW7gWPH1PA51b00cqkrQHHjpmWx+GZ6mstOLbB96nY4PuKnsSZC1IUN'
        'hk1q7FsZjHRpyBXN0YZLZ5atCsbVKUuS6GSoxm8cKRbW0ZG6HEnOc9K54f4jSp0xW9SkjZSjR/jewcQ3FpTcQpUg0F4U4OmxVhM1'
        'IbUKm+UmUi4xGGfa7ZKhpccbQ4FjJJGai4eXbrc4WWxnB2z2olPs0uPEyjIQO1JcJSzxMht91KGtXQ1oePjHZJTuWjQrgpl6PzEp'
        'qvZXUKWphWQrsKd7Tb7a5akhek5T1oBc7Xbo8kusr0LPfNZnCnZZSvRUdt6TIDzaML8ij9gmyojuJClaB03oNHkOhehCgog1dkSH'
        'VNg8oagKSKSdnS2qHo3dC443Cc+9LV4mv5yn6fNCbXLVrBfJG+wNS8QSULYKWiQqm5TnKkKoRirY0MoWdic7V8UAj61DftQ+TceU'
        'kEr0feuGHo8pSda1HHcd69d9nmLoIsFshQKRt5pU+L0Zm58AzW2Rl5haHwB1wk7/ANCaYXlpRGUphWVdADVSS0qZBWh1jlhaSlR7'
        'EEb0UAwS1x3Hm9QClJSoDAPam2GFLbbbbyCpQxkUtOxF2S4XCC4tQ5DpAP8AeSfpP7EU1cPvALZWtOsJORk9sUyZ1BibGkMCIG/S'
        'VqyoZ/amBhB1JyrJI3APXAqjKdblAOISSAgFP4qu1JVy9epQWhRxv1pk9gGK6T0RGUsYOOmrr1/5/SgMq6JTEWVFR7AnxUU+4tzI'
        '/TGfSoHsaVbjcuWCw4pJwdyf6UXICRZkS0SEOoU5nbbUe/n/AJ7Urz3JAbKkua8gjHU4Hn+nv0qG8SyMKBSttRxhPahMuYoJ1alA'
        'Eeo56ioyZWKBUe+WnkKMjhWBPlJeClOoSkhwAnrtkK+4/FdO/E+8tojR47DENiO5rWhtvBKQcg+2B187mlHiS2YupudkuXycte6m'
        'yvQFKPg9N/B71Vsz0mTcFN35t+O8pXpkhn0k/wB1QAx5OR/WsjhGN0a02+z2LwheWr3Zot0hbpdT60A/QodRTaq4fLw9T3p22Fed'
        'vgffWuH3XLMJrcuM4gvIdbOUjsAPtjH/AK083+/3W4SW2IbKi0o7r74rLag6RVpy7G5U9U59QaJwfHSvr8eYlor5esCoOGYD7TDZ'
        'UkEn6s06R46OVunFGKk1tAbURFt89/5z5dyMU/iu74JDSQ58trSfA3pxVHitOlakIPvVG8OR5CA20BnyKqq4iXsXYQXKilJBAx0N'
        'JvEfA7kmV8zFyh0ZVkU+yn3YEFZTFKj5FXrfIS9AKnE4UU7g1Pc9Mpaj0YHcp3GNvX8uxIcKBsMCqCb7xYqSGJAcc962VMKLKnqW'
        '40ChB8dTQ662aKqTqSnAJ6Cs3C2aeWugLw1cJbLDap6UoVnuadI0yLKRqQolWO1A5fDaJGgICsdzmmaw2JEKOkKUOlMuSFaiyhJa'
        'dKdLIyc1Qa4f4gnSFl18JZPTSN8UzSpMKCvUvTR213uCWAU6OnemhJJ2SyJ1oGSGW5oHNawQdgRXaWQwlIcbShHkVSjPyFSeUHBg'
        'HfJ3FWJchD6TGDhUehr2meUiV5nQAUEKQqoX3lpShtSxpPUeKjcMhuEQTpUBhKRvmvjLgRETzUgq/mJ6g0LCIXxVsy3VN3dhpISA'
        'GpGB1H8qv8v2pRg3BUdlttwkFJAJz1weu1bPcWQ7GcbIbcaeRoUlR7VkXFlikWiYklHNjK3bWO/lJ96D0d2H2Z63SFtKCM7qTmrC'
        'pgcPLSsNvKGU6hsfaktuQthaXGlHGnrU7k110BSVkKSMpNK5ncS5PclNy3FFzQkjt/LmhMlS5Y1Pp9SOiuyvNSG4rJS6oE59Lg7V'
        'AZGGwnIOk7eaV5BlErzokhLCA2Q4F+pHn7YoQt3K8FOSg6VJKeo7ii65AOpOc75HbTXEhKXcdCrAJPn3NDnZRKhUv1iiyWlOBsOJ'
        '6be/mlUQJdpWSlRVFUeiydI9leB79q1cQIikZLqkEjYAY2oYq3Id5rDpQpISQVeaD2MnQM+Hs6GzPj85OhKHNDpxgoJ+oEee/cKG'
        '4PXHpG1txWkoSlpKlEDC/IrxsuU7YuJVRnkqVGCwltQOCpGc6D7jqk9jt0Neqvhdf48nhSG6ohZbHL1nwOn9MVnyw6aLQlemaPDU'
        'kIACcfirDklYGlJNKsi/hLgCNx7V+/W3XfoB2ruU3o7hENzFuFJKlEA1QgEoeKj6gO1CXrq88+lgklSj0pkgMJajJUR6iKSacexo'
        '16KV7uLaWShyOsjyneoILzTzGELwCMbjFF3PlSMPIG+wJoTxEGIUBT0dQ1fygClcnWgpKwZf5se1xyS4E996XY99bfc1JUQaEy7D'
        'xBxW6FKLjMcnfJ6im6zfD5qDbwkLUpzHU71manLaNcZQiqkQxr4OYlJcTn70bNwlOsgM6SSPNZxxdwtxBCnIegqDiSrdJ2xT5wjG'
        'catzYnkF3HqOelCN3Ug5HGrgRrt70klyQd/fpVm1WR+W6W0LOnpttR9yM040EtqSc9qtW5tcBGQgpB96tjxRuzNPI6oASLewZ4VG'
        'beQo7qWVYBqwFMFvDTC2wlWlTqx1PtQ8315hxDDhi6lI1ZUDlPj71fiNqmNhQmfw/q64BJr2GeWSIlx86XHQkhJIGO1V2p8SS0pt'
        'Ta0qz9RGMjzXT0Z1vmjHMUnookDUPFVI77bkvAdW28pIBbUNk/miwInQiAW1lClAA+eprp35WVEW1IYakRSNKkEZ/P3rl4AOq1IW'
        'nlp9QAzk1wh0vpDLenKfUSTggfagEzLiuwOWxfPjKLkRajpPdHsf9aX2FjUUEbePFbf8s0p5sPxm0N4+onO3+lLnEPA9uuMtblpc'
        'TAeCc6T6kLP27VKUH6GUl7MudXpeKQcAjNfEkHcb6Rk+9G7vwdxHEf5bkAvbEpdaUFJUKBz2H7OuOm6srimScMpdGC6fA89qi012'
        'WW+iumZHK/VgkeP3q0yUKxgkHtVGfCTHd1OR1suK9WlSSk4/NcMugKGSrSkbVzGoLqaU6oJACQOpxVedFDDSnkqICRnr2qCFc2lS'
        'DGQSpaRqUkdhQ/iy7MMNILz7Zbx9AVuo/bFB36CqKUnhiDfYyvmlFK1ZWVjA0iqPBt44m4X4lt7aI8p22rV8vICWytChrwFbd8Yq'
        'ncuKkOW92IwhSdadOwxkfemD4VcfmzSW4s9PNj5AyfqAoq62G9m+OMKadAQgrB3ASnejtssypkYKAUlXbtVvh2RbbvAbnQXEONLG'
        'c+PaiqHRGUVoICR1oed9UHxf0DQOGjHkc11WSk7bUYYKdWhQGE7CvtwuLSY5c5if33oVFkpeZU6ler81mdzY60j9fJTAXymxlVKl'
        '5FzVKaSGyY6TleRnI9qNxn31XYBEdSgTsrGwpjcguONlK206SO1FK0dfFg20SoqoSdC0J0jcVWuXE0eJ/CbBWr2q/buH4UR5bqiT'
        'qOSCavuWO0ykHlx0Bw/zEb0yTXYvJNmW8V8YGO+3z9OlR3BquOIIcxKVfMhv+6M0c4q+HTxkmUP4zX9zG9LVx4WQUobMYo3+rdJF'
        'Zs6blcTXhnBKmGLffn4jqHEuh9pJ3GcnFOLnGNkn2hxAfSl8I3TnBpRg8OIYioKVnONwRUirFDB564ydQHUChDJOOmjskYT2g2ba'
        'h1LUmQ6z/EUC3j6knxV9uNB5yjHWiQpKcrQlQJBHegcNt6e1zJUNlgpWdTXNCVpHY7HNWrWlEVaorTaklxQWXNRJA9/avds8aidi'
        'S5IUpGtDiiSE7Yx96uSWmYzAS00MqT6ipWP61TcbT+oDkPNyH0D1b4SN9znGPxU7kyMshKy11+hexz5APb3odHH5Ett1pLCi8jG4'
        '2JBP3q0hguNAQ4bKSnOpZOFK81SW6y0hpl95ZAXqQlw7JH3B6UNvHGNhtOWfm0vLSrPLj5OnbvihVhsYGlRHWHA7rylOgoCcjHc1'
        'WQhLGuUzGKh/LpO+w8Upq+JluXy1MRJZH/kBXM74qReV/BsyytQ+kuBOw8mmSYLGwFt+Oh2SAjO+nPQe9ZRwnFjcZ/FCbxI8VKst'
        'nVybale6Vr/vb/8At/61xxr8SnZHDsiJBt6IsiS0WlL5hKkJPXGO5BI/ND+FuL5fDnCDFpgx4idBLilrTlSlq3J/y+wpJQtloPhB'
        'v9NmuFsgXGAtlcFnU42oanEbnI2xnpWTngWeJKGksLU2V8vUOmaF3n4hcQyHUqElfKAGnTjKf2ofF+IF9adKPn5iW+utRBGeo2Nd'
        '47JqbRBfbHIsHHsphzbLaAQO4UBuPzkULvdqZMtxlai7HfPRYILTnjfsacv+o/8A8Eji+4pZnzCpDTBcSd0pWTj7dT+KK274o8L3'
        'C2SbVxLYW+RMSQ49HOpTWeihkA7bGoyTvXRplUUr7aPP92tDtullh9KuUf8Atr8UPUhxl0ajgj6VCtRlMQrs2/bVSG5KmyQy+Oji'
        'OyqQ7lb3YMlUSSDgHCFe1ddnUO3wh+I03hu4oYkrUuGs6XGydseRXpSfd483h9u52iQ1IjvJz9W4Pj714pQ0ppwIWDp7EdqeeC+K'
        'ZNlkNxJTi3IbigQNRAP+9SlFex03Wj0Hao819lTrjhUhR3SrtUgUm1KWG5SXCd9GdxXEWZaZdiZm2+4qPNThLQ3JPilqTbrs7ckv'
        'NsLUhzqemml4qC+rGT5vY1NXZ5XqCgjG+R1qwHb5LUhcS5BCT2I60Ftlvkrd+VKSVk4z2pl/Qp0OKlaHQFJ3wKilIpLiuwPdI3Gr'
        'ag41MbWB209a+wr7xdHUkOQkLwd9J60Wk31bFuJDfNeCtGkdc13CXclhDrkdWpQzjFBpt6YqpLaDtq4l+bQ2zIjrS4eoIo4YcR9G'
        'XWUkEeKRLhP+VbU8tQaWn+8MVNwxxnEUktXGa024SdJKtiPvTc3F1IV401cQxxDw0iVHP6fJVGc7YO37UkXNfFVjd5TtvTcI4GSt'
        'vAV+xrRm5cO4I/gS21HtpUK/IkBsqZkNawB9R6Gukoy6VCpyXbM4E9ydzEuR1MTFq0tpKgNCfOodPfrUcydKjynG1vJ1haUKI9aU'
        '7bg57fnvRecxemEAQGba0dkLUUkqwT3z3xUKLSpm7ply2IgjqwlakKKdSuylA7e3SvUZiOUfqyYrLkZbLPMJ/s7aMlaT0O336Ug8'
        'bcexYNwdsTVtWuZgZde2SVK6gYzjFaPOtLIeQiLzY6GlH/suaUEnvSvePh5DubrapE1CPUTzl/8AcO3btjr1pnFrvoVNejO5nEV3'
        'mPyA1Ply0EehCFBKAfb2GP60Hj3BuWteoArZBK/4oCk/bpmmu8fCWY2yHLVcFTyws4Q5kKCMbkJ6EUMmcIX22QmJzfDbyk6EapbT'
        'IKwpWwGAc03JHUxfTNW3KVynFLQrOEqOMbjO/aqka4NJkc954IZ1qy5r9YHcDPWopBkonoaatT7kwakLQG8ODBzuOmTQriZ65W1A'
        'EmKht14FbbeOYpOeuw70OSKLFJkrN3/Urq6GXVBDZKk/zHI6VzcpUcR1EJLy1ZygnBJ+9AItvluOMcoBLz6C4rsB2NEY3DVwmvpb'
        'X8zzOqUtggHHUgY3pE2VycGEbdPK08mSymK4kAYDuyvBBxXSnyNKFrHr+kefNWk8OXAusMxoU10FxKcrQo6iR9v8KNW7hK42+TGf'
        'n21akJc1rQoEEgbk7n6c4ouddkoY3N0iTjRxMaLarOgafl2gtYCttZGOn70Gs1ilXSXybewqTIXkkI/lwPfpTlIsTF2uLt2nOtxS'
        '7objokKIBWcZKgN9I32G9bDwnbG4Nq+XXbYsV8ZSFxW1cpzwvUrBwffpuO1SUlWi3yE+bbMptHwh4gXBFybksMutjJStRKvcbdaF'
        'cbcGXNuCBcowSvHpdQcgnwfBr0A7Keiy2mlxXw0EY5gA0Zz0O+TnNRsMNOPct+IhkpCkKQ6gEKSTvvmkkrYsZtHjCYlyBIESagpz'
        '9CiKsx0okx1RVK3O7av7qu1ekPjDwNbbvw02wxDaQiGC5zGx/FBPfpuKzL4bfCe7Xua45FcHyTRwXXtvwPes88ii6l2aoRclyXQv'
        'fD/i+TY5SGZutUfVpUO6PcV6Q4Zks3CMhcBTkpC0atQ7ZpV+Ivw6Q1aI7bViEhaEhOqOn+IT747UtLkK+FsO3JlXZ1Tcl5KXWUnH'
        'KHcGs7yJsqoOtGvQkRYlzQhD3qcUOYCclJpmuDaVN+haikGlqMzbH2IsmC4gok6VghWVEnemhxAZipcedShHRYPiuTbEkkgHbzCi'
        'POPLZQrC9u+9NcO5219QaK2kOYB05GaSOI7cizTmLvGUt+C4cPNg5IB7iqEi9WpqWiSxAalpONCgsBafxVMWRrTEnBS2jSZ9rtlx'
        'YLT7DbiT5FZzxx8FeHr+2p1qVKhO42Uy6Rj8UXt/G8Ay+W7ElNFXYgEfuKLf9SxnlaYykOZ/lUcEVplFdsgnL0ZdwLwFN4T4oXHd'
        'v8mXCSj0pcXlQNNPxKkXpfDrieG3h8y2Oh6mmVy5MKcK+RHKu5xvUJukNa8COkqxuNG9QbitJllye2gTDlfqqOW/z0lsalpISAPA'
        'Kh5G+xBFB5Vz/srjCGnZam3VFcZqQhzRjOyiRuB174yKKR0hyMyZMdxR5XLbZYWpLfLHTSk9M46neunIcVktMlaYaWlKDR5YwtOk'
        'EjI7/wBdq9IwdC4ri6xswlti3yo8oKLaNXrKVBO3Q4PT8UPncRpeYU65GLge0pWVuqCsDc4T4yCPuab4vD0J+MZT0UNFRywhSS2r'
        'R+2+d9/8Kh/RYP6kIUdsqbUkIS2UF7l+r+8d8H/WklFvp0FNIDWy8OPXDEy2F1lSMNSDEUAkY2CT2x5orcojM6HJix3JGpatSklp'
        'aVowBjATnTv/AI5qZlMiPem2HGpSucxoQ24OSyg5PQEdNu3mujYVxp7kpERlTgILWlZAGN1A5VgH36HpXJMOhI4k+Homy7dokOsg'
        'oL7rxSSpBIwU6lnGTuc9qYI1lslhssZq2C2B9bZWJawHFupz/MpWemaJ3q2/qML5dxxLT0lz1N8kkpwc4yPSCB7mqFgtsm2TeQiY'
        '7JQpooSHMupG+c4wdCiCev7iurdh5OqIrfarPG1ohpZekuf9/wDgJGSP5sYATn7f50tXziKI5cEso+biQgl3+3Q4qG+Y4nolLhz6'
        'RnOpI7YGN6epXDcKUy60ZDjodOlalnBaSB0yAPOMdceaCcRcF2Bm1tNpguXBqMvmBoOavWrsAohJ6dOmNtqHH8Cpb2Jdtvsltxf6'
        'ffWrg4oA8iOjUvUfGCcH3NQv3GfPuarYmPclyScFoq1dtitQ2SBk7DxTpFjxbXCSzaLUIi1qBcUhCEo3OfUEK2PXp4q7bmXA5zEM'
        'JUH3tf8ADCQl0nqrY51Ebknrg0rg/ZdfIS3FHFj4Ks8Fj9RuKE3Wa2vmmQ9hRbUd8NpP0pBPcZplhgONKW2yyXNAPILpyEj2I60Q'
        'aS2mGYyEuvhAHoSMZIG23TNBZcdRf1JYUJK1AELGgq2zpKv67VyVEpSb7DDDBcWlMlRaQCF6Uq3B8b9RXamGFNuNnUU6gUJUSQdv'
        'H7bVUiCQ5GX+puhoq2y1gqSPuc56b5G9fVSwiUhqOt+U2UcxKuSQnGR/N98++9MkiZOtEYJCHUKTgBRyo505xg57e3avltDNpU8/'
        'bEavmHCp6KQMlWfrSRsNuxx/WvqJa3GfmmrclSy4QESU6T5xjr/rUqW0hZU3HZbj4ytttODk98Dr4pJ4lNUxozcNojuvELsdYMZc'
        'b5nGEsvnHMPgHsaHRF8I8aOmTO4djPy4yuXIblx0lTSh2IP+NdvxIzsL+0tMy0lwuDWkpSgE7YJzg/Y1Wu/Bdl4jShz5h6DcMFPO'
        'Zd0Pt49x9Q++1YJYsmKVLaNinCavpjSmx2FyVHlrtEZl1gfwXAdOB4wDirVyt9tuCUtP6VJP8oeIFALJwBZREaYeut3nqY9CnHZa'
        'khxXnCdhQlj4fRo/EcuTMuco2wDUxFQ8rmKPgnrsapKM/cUKuN/9DX/0zbVKDaPmC2kYCA+VJA/NLdx4Q4JRPXLkyHIbivSULd5Y'
        'Odsj7+1RcaXm92jhhdu4Q4ZfdDqNBcU4QprP8xzvWGXRjjh5wi5W+Q4wpRQmQXVOtkj+XUM6fzilm1F0o2NCLl3KjZ53AfDD6A5Y'
        'm0SnG1kLbXKWjJA7EHrWaN8W8PR79JgR7PfIsyGpaHg1JC0gp69TvViVxPeLMlJghUKQGS64oY5aFAjCcnqFZGPv7Gj/AMN+J+F+'
        'PJq2rrY7cm8OJyt0tBPPP470JZm2kPHFSsqcLfFLhSe61CRd1GQtWkNyGClefGR1rTQoCJ8xzUtEDYj/AHqlbeD+Fbe+4/brRBYl'
        'Z2cQyAR++9XpXDdvnxeXcA4sj1atZSQfxTO5aBaR9Tc4cmD8qxHbkx2HAS3yVKJwdjlOM7+TXC2575cetyG0lK0pX8wlQQkDf/Tz'
        'X6Etu0uJiR0MqLq9QPOUcbY6AdMZ/PWuZbS4vOluvvFgZcOHhpWB0CtIGMZ2zmvTZ5qPrc+VDgtR1PQ7hLS3spHpSgn6fTvpT0G5'
        'zUcJmTMYlOOL+QU88PlghKAtSgBvud98+9Brtfn3bemRw7HdmqQtC3lkelSOugHGcYGdyB169h9jvl6VF+YejSZT7xyw0WSnlAqw'
        'TgEae++d/wA1waGtq03CQ4tS58jWlAylXKJbV5JH1HuNtq+yFvQbU8ozlOJYbOlTgTqBG6sg/UeuBj80u3Z+4SeHgiWFx3uYFOqZ'
        'UtLaB3WCclWNtugzRK0NMMyEMIVEVHfGtIeWMZSNyD5PX813ZxQt8uXcLemWw7GdkFwFa3XjpYSegPUhRGdt8eDUt5lJjNMRY1wi'
        'PvuBIUlMlSnCSOmxGdwd8djtRb5N1E12Wox21lIQgFAc9CSSNOSMYBO53xillcHh0TETYpbDhKlZYaUoknJVgoGOgIyd966mztBe'
        'bPdgtm3NqU/ISlKdQcKi1kZwTndRGcDPiqzzi7JGVJRd2rgstkguSiFZ84wRjPkk+9fJN7t8tTi2bY44rShTS0OJJc204OT6dgBj'
        'GaAz34k61MQzbWrdMeUpbjTj+VBCdyArBO48YI3otAQUsN1mzGnnorcVp1a05VHTgLwDrO/cqI3wcio7mzCky2GluT50hbxLOpsJ'
        'QkhGCndYJ2Gdx27b10m0WZVlfdhMuQloSAXWipb2nAJBXkHOexOem1X5qIsplL8gGE8MNplOHK0EbJwfV7nYDruaRp3/AAa1QBs9'
        '8vbFxWxMZ+Y5IWtC1KCCoDBwMZztj/gotBvKZMh+e3AUz8xG9T5X/DyASACTkkdeg2JqrHtsmFElxYV2UJa9b68k4kE5wkLx0G+x'
        'wN6p3BuEYHPZYkutsJ5upKVAKIIwD0BSN9uv3pkjiZziR9LiYkHnturJQ86rShSVpOSAMnAIPXG+xqNd8uxI0xUcxpwhKzJS6lex'
        'yBuMg5HbFfU3G13Ftlguh8NHJU4CFZAx6wU76RnBzjpuaYYsiE4oR408sBvAbwkHKVDIIBTvt0oNHWR2qXORbVF8Nx5To9bzjhWh'
        'CwMJOkK0p65xkZq5apstuKz87OE95841KZDS1dTkpST0wcb+9Un7amWhIuT0h10uIUFsL0hGDtt3JHUb/iiTNjhRHQltppnUjISU'
        'ZUf/ANidjuN/fauOILaW4jzshxUvnvKK8kEpwc5CQSRgD9+veiipkRLKpCUBUlvJLhIR0z19uuPualZgpWgpQE6gAVNADV53Vmuh'
        'bJL7rokoiKaLewUgKVkdOo6fmgzls7tvEnysCQ6qGHCN2G2ElKnVHscnA++ayjiv4sPWu4qTdbo7alKc1Ijx0pdcRnJ9YI6b/fwK'
        '0xERqNEQ3yWkFB21JwkdMJA6gbeaUOOfhvwZxRco9zvluL8jQlLkiI+WdskgKA+rG4zudutTyRcuimOST2JMD4txrRLNzZ4uk3Ry'
        'SAXYPJ/hp99ShkH2BovFuyPlGroy3cbXKlPa2ypJbaWFjqWyMHrjVjHTNBrvwDwZ8PLhEu8VUmel1zS0mXoWYysjGAcBR3zk+Km4'
        'vl2G7PRLnBuq5lwjlQVrmaQ6nG6cEYCh422NefJpM3RimfuKuF2OIYT/AA+Hl218OmQlwI0IWo5I074KRjcZGPAqWxfCa72aDBv3'
        'C0pSp6wW5zbqgpLDmg5W2e4CsHHii/w5iTXrHLYus+NcJcdQKWG1YcLWPSpKuhUM+47ZwcUYtl6LClx7bcAttKTlGNxtjQtPZWe/'
        'tXKmjna6Eq7SfiHabZFvt5ZU5MQEtLDDmHFqJ6aelMdh454mXMt8OVb3Cqa6ltKnEZxnrlQ2/emriee09abfcDb0yEqTlwLdCeWS'
        'MaiPZWx8ZHmqt6DFvt7amY63nXkZHKBXo27d+/apNVLTodNOO0f/2Q=='
    ),
    'chipping_sparrow_07.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABQYDBAcCAQAI/8QAPRAAAgEDAwMCAwUHAgYC'
        'AwAAAQIDAAQRBRIhBjFBE1EHImEUMnGBoRUjQlKRscFi0RYkM+Hw8QhTQ0SC/8QAGwEAAgIDAQAAAAAAAAAAAAAAAwQCBQABBgf/'
        'xAAvEQACAgEEAQMCBQQDAQAAAAABAgADEQQSITEFIkFRE2EGFHGBsTKRwfAVM0Lh/9oADAMBAAIRAxEAPwDCMdqljXkV8qZFXLC3'
        'eeYIozXJs2BKlRmRx5UgkUYsCODtzV0aQGiwRg471XjtZbckN71AcmGAk4jDTBiO9GbUKIx70GSX5hRK0kyRU9uIYKJcC+wqS2/6'
        'yD64r5WXbUbPtcH2qHvMKzSumrPKI2OKbftKWyKhPNIfSfUMMMQjmIzir9xqy3eoqQ+IxV/TagQYMMuMcR1if1mRQeWNOWg6dbW8'
        'XrzuqDuSxrKJNZkRglkC8oPHmvtT1Dqee2UXMdwIh/IpwfxxRvqqO4TM2C+6s0PThgTKxA96B6j8UNOVSI4yR4IFYlfX84JBDZ85'
        'obJcTMeSaE+rb/yIFrQJsMnxMtZTl4Mc9xRBOs9Gv7H55djHgg1hXqv5rpLgihfm3Hch9czVLnU7aSKTY6svg5pZ1KFJU9aMceaW'
        'Y7tgMbiPzq5bahIilQ/ynuDVZrXa5dpEz6gPcthcCvicVW+0bhnNRSXW3uRVbRorveR3y7moZXHihc+o4OM1JbXPqjmiXVuoxNbs'
        'yaZ+KqvIwBwakvJraK4S3klVGkXcjMflb358EfWvhbO6ZVo2B7Ycf70rXorPqZYQhpfGcRc1JXnnKjJ5q5pOn/OqBdzk8DFEE007'
        'yzGIH6yKP80Y6Zk0zTtTjudTvrOKNDnmUH9BXSVrtAEgtDk9TQ+jOj0tbSGS4XM7YZvp9K1PR9PWO3Hy+KzJPiz0HZ4UX1xduOAs'
        'EGc/1Iq8PjBPNE37C6C6hvlA4keBkT+oUj9as0sqUYBjwpYDqfkKy0l5kDbTzT50d0TPLbm6KceOO9TdDadFd3EcMowpIrbNL06G'
        'C1WONAqKOBXPaHTG8lm6iVdQ7mR3vT9xbxktC2fwpb1DS7lYnmMRCj6V+gp7OKTKtGCPwoD1Np2nQaXKZAqDB7U+fHAcgwhrE/O7'
        'nZMRmr1pKB5qn1MsUV9I0LHZu4qna3f15pFlOcQW7EZkn4712GLke1Bo7gEjmiEMw2jnNDKyBcmEYH2uCDRW1uGJGCaAwuzNgc0x'
        'dP6dPd3KRqhyx9qnWWzgSaZmn/CzT4rpjNMgf2zWxWWhQGIFRtGO2OKVfhzosem2kayD5jya0WJwFAFdHQm1ADG5nvV/w703UyZE'
        'X7PN/Mg7/lWb618N9TskeSKWKdV5wMhv6V+i5fSkXD0l9dXNvY6fM0U6rMVO0MfNY9KHmRZQe5+c7m1aGVonUq6nBB8Gqzw+wq3f'
        'PLJdSOxySxJPvUYDHxVa6jMTI54lNkI7V4rEHzV/0S3ivjaZGcUIofaawZUNyFXkmqFxebnxmiF1ZORwOaCXltKjZwaXttauRMsr'
        'H6w+tGtB02e4PcpCp+eUj5V/7/Sq/TWl31/GZIYSY1OGZjgZol1Kuo2djDaIirbqOAOA7HuT7mptci1fVccSarjkyj1LoOmai0EV'
        'xrMQEROwJJtzk85xXFr0roUTbDLAxPvO3P60vXdlcjBY7QeeBUZvtTgXEd2Rj6Ck6/J0E8iE/NWINuSI2N0vozAhEtGx5LsaMdPa'
        'NoWl3KT3lvp7x4yB6G8/0IrNJtb1wcjUpQPODXFnd9RalexWdrdXdzcTNtSNTyT/ALec9hTa62lmBCzX5lmPZM/Tendc9M6SqQ2O'
        'nh7jIVI4rUJk+MbRmh/WfxO1K9tjplssFsRzcNE5bYP5d3v74rMtGs5dHt3jF215qUif8xd7j6canukef1bufoKB9Q6rBaW7xKjM'
        'M9gR+vuTVi2pYL8SwqrZuWhP4c2zT3IZeCpz3rXoLhoolDjtWG9K6lLpk6yRH8RWk6b1VbXKBJRtbzS+hsrVcZ5i1eMRivdait4y'
        'SoJ/Cs36z6ia+ZoRhYxRnqXUN9uzRBCuO4PNZtqkrO7HPc1LVakrwJljYi71GqSNiPml4xujZGaabmHdnIzQ6a0O7tmq4XAmJFoP'
        't5JMgc0Ys2JwDVaO1x2FXbVNpxipZDdSSgGM3TkUCfvZsN7ZrU/hnBFd3huCgAXhRWLW0sicDIFa/wDCO+jWHYxwac0bDfiM1fE2'
        'K1l9IgjjFGF1SJINzHt7UmX2pLFDlWHah1pr4VyHbK/jVvuxDRm1frG1twyq+W9lGTWZ9V6nPq9wXJcJ7E06DVdLlXMkULH6qKBa'
        '3b219IGsoNreyjg1BycSJ5iIbIE8ivRZL4FND9P3pBPpkeaFXUBtR8xUn6GlmbA5EEUlG30yedwsMZY/Silt0tdsR6zRxD2JyaNd'
        'LfZ1QPM8aO3bewGf61L1Br1lbxOYHMksbFW+UgDH9/yqYVduTJbVAyYJuemLaKDdJPk+MkAUtXOiorO7QPNg4VV8/UmoV17U9T1J'
        'tmUjBxx3/rTLaybEHqNz5JNLPTXcYPKt1Klsi2FivqIY4kHEanAz+fmhmpapNeRGJooVi/hTbnH1z71Prt093crFyEQfKMYB+v1o'
        'dJCijLHP0rkfNeWcOdNUcKOD94NnIOBB2qQJ6MTqDgrg/TFLF7CCWxkCm2+mj9ARlhhc4HtmgMyG4uUgt4zJK7bUReSxPiqnRuc4'
        'AgbTuPEAW9pPfXsdlawvPcTNsjjUck/7fXxWjWGnW3S+lyWllsuNRmT/AJu6UZx5MaHH3P7+eO1rS9Ji6es5FQRnVJkIuLgYPpL5'
        'RT4HufNKnVGr+mz21uWQZwSTyx85rttJpvopvf8Aq/iWOk04qO5uTJdW6je0hBtgDM33dwGPxNIrI0zyTStuZm3NzXsnqySMSzEZ'
        'Pkn9a7gcRI7CIySD7in7ufJNDstPUzW6wE7a+ofgOFz7VZW6eNeDUUEYOBirEtuFTkUuCYsCRIH1KZwVZ22+1V3beffNTi3DN901'
        'ZgsMYOOKg7M3UizZgxoSwqvNDg8imQ2mF7cUMv7cjJAqKo0hiCRFk9qsRwbSOK7gX5uam7N2p2tCRJoJJb2wJBIpi6fvH0+UPESP'
        'cUCifBojZOGYCmFG05EJ1HaPXLiddpJINSQLcSHIDc1e6Q0AS24uZh8vfntR5YYI5PlUEDyOafVHYZYwuD7yp0/o9zdTrvBCeaZu'
        'o7i06T0Rbt4N8kjbIl/mbGeak03W7e0McNtZGadiFUE9z9AOTSv8QtauNdt0tbyJIoreTemxcHdjBHk0S1WSo/T/AKvb9ZLBxgRb'
        't77WOotVlaKRFm2FnkeYRpGg78nA/KhOoi2a4WK3upJML88kmFH1xzyM9vNAtW6t6X0q/j0rV7aaG5dwqudzJ9CT28+1WdZ6Wu4I'
        'ZL3Tr17ssd6Rsf4T2GaoX8Ze9e60lm98H+P9MPRpUc7bGxLgNv6wWL1pivZ2Hyj8PrXd4J5IVhiJAzn3JrOv+Kbqy1D0Jd8LA4Ks'
        'MYwea0v4b69p+rpLFcNEt1ENwJOA6+/5U1omTP08YjGt8Q1Cb1bIlrRNFuRjO4ZOTgYpmttNghAaZgffPNR3Ws2kPyQj1GHbwKBd'
        'R6ndPZhWJiEhwQOOPan7ra9PWXPtKnhBOOp9WsbhxFbQNuhJHq5Az+XtSpe6gypkfKv8xOBQzUdaEV29qse1kXktzQa4uWlfJJJ9'
        'ya4rUVnWWm1xyYo9m6Fnu/XfaGdmPACr3py0rT/+GLRLmaAHWLhTjPP2aMjnn+c9j7dveoOktGOh2qarfR51OVc2kDL/ANAHtIw/'
        'm9h479+0PWvVAlsY7W3cR3iDa77c7h2JHt+f1q90Hj004+o/ft/vzHNLRj1N3Jtfvvs+jSTpIr+pncC2HIHn8M1nU0z3MhaR889v'
        'I+lWr+59S3SCJi21cFixOcnJ/Wh7h0AONzdhnzR7rixxM1d+PQktC3VjjOBjLN7D/euNsaRkeT/5iubsyvaROwCKuNoXyfLGoMbh'
        'ls5NAUHOTK8mPGnWqxgPLGTntX2qFSQqrgVdhKqNp8dqr3KetJx4qTrhMCPE8SrZQFjnFGrazynavtOteBxR21txgDFTpoz3IAZg'
        'g2JIzih99puc/LTo1sABxVS5tlwcDJpv8uBJ7ZnNxp8kUhIHFVxbTM+1UZm9gM1pMGhxSSj7R3PIQd6YLHTbG0i/6aJjvgc0RdMT'
        'JBJmPT/TtzfyH1ILhU8fLtB/M0+6P0vptjCDPEksoOc0Rm1CKMmK1gLOeATgD9aAyateNeokWx5C21Ywd24nxijLSicnmS4EaHuI'
        'oI9u1ERewJ4ofc69ZqDvnXA9jS911BrKXM1zqqQWzJtVollUBTj5UCgnnGD/AH70vW8Ah9OW9kQBvmWJmwSPc1J79vAEkqu7bVEf'
        'rDXHS4+02peAhSFkGQ2Dwce2aE6tLJNmRHGB4JxQSDUluy0UDhMdjj5SahlvSAqThgmceoORjNbVywyY3s2cSjqmi6ZrUI/aNmj/'
        'AHsErzz5pc1zqvUfhtpMMSGO6spJBGnrFnMQ79x2U47c9jitBuLGWeNXtGQoR8wJPFLWuaHPeSek1jHcbk2kS7drAE7ckg+T4GaI'
        'MiRzmA51tviX0/a6pExs7uNHYDcOGPg+ccf2oL8O1uLTrGz0+5lKXDXr2bFOVIC5JzQO61216R1y5tG06ezlcZJjysTeRtyP1px+'
        'GU+laj1TZ6yrySS3DvGGlBAWV8Dv2P8AL9Mj3obUo53Q5ttqrK54M2a2tLK0haZ3CqgJkmc9hS/1J1Hpt3GtnZIXEOTvIxt92Jon'
        '1XDI/oaWrbbdR6lww7sfCj+9Z78QpI7W1t9Os1WFWPqSqoxvA7AnyPNc/wCS8gj2/k0GSe/tKaxioi7qRH2+SV5UlLHIkHYinnoL'
        'p/7JaxdSarECzjdp9s4+97SsPb+Uee/bGRXw76dgu89R64pfTon2wRt/+3KD90D+QHufPb3pm6x1/wBGN5Lh1MzHAXHA9lpzS6Va'
        'lDH9pLS6fd62lLq3XXtlO6ctczElyG5x9KQJZpLydrmcjc7cAHxXl3eS3Ny0tyRIxwAcc454/Dn9K4iWaR8qm0VltpY4ENqdRsG0'
        'S1GArc/jiphsZyCQWHDKf4fp+NVEWQ3Kqjtle5FSIAt4wHbjigAEnJlWWnepFTBjGMdqHpvchRnk0SuyHTAHHtUVnGF5+lTJ5kTz'
        'GqF5nk3En8KKWMRY5NffYzHKQR2NX7ZQgwPPepJUc+qP+8tWKYopAQMc0PhIUVMJsU8igCbEKmQFe9W9MtYnJnmGQvKj3NBYZd24'
        'nJCjJC9z9KtfbTYWhubuT0SV4jJ+6P8AejrzzCiX52igdnHDse/mguu6wlnZm4OfT3FM/wCrvilvV+pbp/UlRQiH7pbufypL1O6l'
        'upN9xI7BjnGeD+VCt1QQYEE1uOoX1TqR5yyo5Xz35qv0t1F+x9ch1aW2F61uS8ccjHYXxwW9wD4oJcbEbESgn+1F+jNJbVdTQ3kq'
        '29jCQ8zsp2n2Xj3/ALA0gLmZ+DzALuewAdxm0uR7q+l6h1n99Pdu1xwMRxZbAHtuJzwOwH1qCQxSTM0KPJ6r7pZmGWP4+wxxx2qX'
        'qZdA0ixkvdZ1D7YglDIkLbUwudsZxyBkgkDknjsOa4vheWomECWsLfMqKNgOfO3APP1p2qsjlpe+mpdqya1kgjYRRx4IOcZzRB1j'
        'YHcBtPGPBobDDHHtcHv2q2kyFAm4EgZJ9qcAgCcyW1aSKZVSQBSeA3+1e6xp73cb772eJNpyIm2k/Tio96ggqPmHapPtMjRNgqGG'
        'O/mtnEjmI2svZpK9hcytJaSRtFIDy0ef4gT/AOd60j4YaBptx8PYtNTTvXEUk0IijXLgpI2HBHZsYbP1pD6pNuxbaE9UHv2/91vX'
        'wN6b/ZnT0vUV5byW4uIMWllkjAKqHkwe28qMDwPxoNedxE3Y5YAGIly08t1JJcn94znd9D7VU1PpPS9cxdakJY4bfG+WNsMw7+mP'
        'cn9O9OOvdMXdndS3l16draPITuL7to78e9J3UmtQhFKER2sA/dRE9/8AUfdjXHaPxF51jXajIAP9/wD5ApSWPMra3qsVtEk0kUdv'
        'BBGI7S2QYWBBwOPes71KaS/uxcyMSu0bFJzj3Nda3qEmo6hu9QtEDmoA4AAGBV/deCcTepvFQ2r3LCD1I0jOBHHkqMDue5r6UiNN'
        'sfJPaq32jaPuk+57CvSzEhg2dx7dqAH+JUls8mWIIkjXcOW+teQpunZuMmvY2O3ODUliMqWwck96wnMjn4kV1GFTk4/GuYRhBnti'
        'pdQHyLn+I4r4BVTGc/4rWRmZia5r1isV25UYGaGom00y9WSJ9sdE8GluVgKtrFAaWFgGZ88mBUE1zt81HNJ9aht4Jbi4T5SUDDd+'
        'FDJzwJEcw/YMbHT2unG64mH7tWPCj3NJl9qhnkkN1cGaTccMewH0FXus9dCr9lgbLEfMR4+lK2k3EMUk73iFmeMrFxkA/WganVLW'
        'QgM07c4EpahqJuJtiElQcflUd6qySlbZ2dEXPzcEe9cm2ZmLBdoz2qxZ2DSzLGiM7OcADzVYHstMXkWj2M+o3SRQqxJPenbXGi6e'
        '6f8AQhkjYx535OFL4/iPhV8/96JaBpydO6S+oNbia5xtjj7b3JwB/UgVm3xO1e9t5jprwRzXc4VFijUlXdmJAXPOCeT+B9sVZ06f'
        'YOezLTRUhBvMrza5GtzCLu3/AGvqkgBtbbHyW6//AGFfBPcZ5p6sdPvBYCbUHU3DDJXH3c+BXHw06Cj6f0Zb3VlWTU5xvmZju2k+'
        'M0Y1WYQo0aglgMYptV2xh23HMFfOdyow9iTUBla2k3Z3ZH9KhacCMmQ7ef61VmuPUD7jxkcUXfxBGEF1SMtsJCv3APmobvUJNjen'
        't485qvMsLxqDGAwHJHeqUqLEXCOQoHGeaC1jTOJLZWD651Lpempl5Lu4SNseAW+Y/kuT+VfqXVNQTT7NpppBb20K4QAeAOAB+FYh'
        '8DotNsdYvOpdVnjhhs7dljeQ/dJ+8w/LI/8A6r7rHrSfq28LQF4NJiYhFJwXHuaLU2ELfMwcmXOs+tZNfuCkEq/ZlGAisDwPf61n'
        'N89zrWoHT7VlGPmeQj5EQdyT4/z2qteMzaksGlQSvLcsEihXnn3Ht+PtTbb6I3T3T1yGHqXhiJmYDu3sPoM1Q+U8gaBg9ngRq+xK'
        'avT3EOWOGKVobWRpU3cErgn64qyLSMPkkuPwxUe9Fyzbt57jaePpXaT7VO4heOM1Cthj5nOMxY5Mr3pDyiJBhV7/AI19j5lAz3/p'
        'XFufUkLnnPNXbG2muLjZAm4jOcVIt7mBIJM7A2wOc5496ntMLGAQc/3qO7ieMLCyFWY8g+1HdC0drxX374wqDaxHZvahWalKRuY8'
        'QqqTxAOoAmeJUVic5KjxRXSNFfVIppYn9Nkx3HDE+Ku2XT17PqBMkYjRWCl27YHfHvmnaDT4NOt1igVFRstheeTVTr/K7UP0eT8/'
        'EPXVnk9SfXifXLE80BuHIzTddac91cs5B25oRremrDETjFdtYjHmNMhMB2URurlYx2JpmuraC20541G0hD8w7g4pe0SVYL0Z96Yb'
        'p47mJ0cnY4wcHmt0AFSfebQYEyzUBG8sgIJkDYBP968WzCsmGWQsM8eKcL/RtOs9Lurpl7AhXY8lj2ApRm3RsVTGQPBqhtqNLZfs'
        'xV02nmV5mAIAAp96J0Bo4o7iaPE8y5Gf4F/3pW6Z0xL/AFu0glBIaQFh4AHJrWLme30uzmvpyFjQEgds47CrHQIXzY44EnSmTmA+'
        'uryDSNKkvZWCxWcZZQBkhsEZx5ON36Vj/wAFLabrL4g3/UN6oeDTMpGCcgTP3x77VAUfQZ81f/8AkN1Q8XRkMYbFxclZnA7DPKL+'
        'gNGf/jNpL6P8PlM8TR3F1MbiTdgEgj5f0FPgqfV8yyHCgTRdXkEULZYHA8eKz7WL4qGyRknjFM2sXDsrknzwTSFrEmJGC+/egWW8'
        '4ksSjcXZWVsnLZyBXtpdFiwYDcfHiqEqCacyLyQMDPn2ryIvGhaTK/Q96EXxI7eYWuLhYIFGdzntmqlp9pvpTFu2og3SSY4Ue5q3'
        'oXSvVPV8F/8A8Oae0y2Nu8ss8nyxqQpITPlz4X+wpQhQRRwWNrd3j3F18928w+RHYjaoAIyCvk+4Hg1IAkZPUItW7oxn+2za9Ouj'
        'aazR6XbkCaTt6uP8f+6MTtJLLb6LpVu1xdTMsMMKLnexOAP+/buaG3MttoOnDT7RVWVv+ptXBz+HcV+ifgD8Nz09pY6p6hRTrl3F'
        'mCJ+9nC3jHh2Hf2HHvRa0LttX95HCpzPOgPhjp3TWlepqlwZtXmQm7uEI2xA9448/wAI9/J59qTuoXthqs6WksksIbAZzkmtl6hj'
        'uLuwntbV1R3GC7HGBWH3kIgu5YA6v6bldwPBxVB+K/TSiheM9xW5ieZn/Vd80+rPDEq7ITtBx58n/FBFUSSF37gH+lGeoHgm1WV7'
        'OIBS+0lf4j5NS6boT30Msgb09p2ruHf3pKm5NPQpbgYlYQWbEB6dpt41t9rWF3hD7MKMnnzT503oqafb+vMmLhxnGfu/96LabYra'
        'WMcUS/u0GM+5qZwSM47VS6vyltgKKMD/ABDJSBzA99pVrdXYllUk45UHg0ZiKiNQBgAYArkx8Bsd66QY4IqsusdxhjnEMExJVJqT'
        '1m24PPiuEUnnx714ww3vSoJAxN8zRZYAg4WlDq+VY4XzitC1KD0kYkVkHxAuZPtJQZxmvbrjtQmME4EX0uNku8HsambWZEUrnvQt'
        'ckZqJ0Yk5yBVYHKiL5Msa9e3N9pUQQOyo/zBR/ShsUDRQeo4Jfvj2pi0m0K277d20oGdvHPig8sfrXiwwuZHkO0AUtaoJ3nkmBcZ'
        'OYw/De2xfzanNxFBGQCe2TVDq/XJtVuyiPstIvlRB5+pq7rV5DpWmppNrMN4UGVh5Pt+dJV3dFuI1JZu4Hii3btgqX94UekYETfi'
        'bcftH4j9J6JbRC4VpbeeSN+zFyoIP0AU/lW5Xuox2N+0kIXZ9wqOMjNZnBotvoF7d/ETV3WWY20dvpcDcEMF2sx/HBAx43H2qrcd'
        'W/bHzLOrTtzsj5waNY5VFUdiPp1maLql+l0jGBw+eMDvSFd3Qm1A2+18oOePqQR/epenm1GW6W6gSR3Y7YYwSc47k+wH+adJ/wBk'
        'wr60Vkj3ksSibkmNZMksV988d+OKSu1CIpaxsTTXonBihaaVczruij2Q+ZH4XP09/wAqaug+mtAv+pbSHqPVBFYAlp3b5E45Cg9+'
        'e3iqsheZyT972HYCvkGG2ls/6R3qm/5aw2jYuR8fMVOpZjxNu+IPVnSOkdAS9I9FzWSJcRtCWhQskKn7zg/xOfBz35zxWCW2k6ZZ'
        'aRJbvIxVkImnkXJ7e5PtV69kFtBvkUA/wrnLH2z7U2fAzoKTrfVR1Dr6t/w9ZS4ihbgXcq/w48oD3Pk/L71fV3azVuAQEH94VDZ3'
        'nAl74W/CrVNU+y9SxFLL01SS2vL+IyNJxxKkX3c+ct5wR4NOWqfC2C5uXutQ6v1S7um++5jXn+pNaT1H1Lpmk2+2aeOPAwFH+1ZH'
        '1R15u1Efsvcy/wARPY1YXJp6UxYc/v8A4E0wB5adt0ncaRFIln1bqkSN95dowf1pIntbiK7dEujMoYjcY+SPeija1qd7IxmkOG8C'
        'vGY7GJJzg9qrNd46vV1AoSAP3/mDbBibaaHI11O90NsfqZXHduc5/CmKNRgAAAV8zHPIruDJbPivO9RdZY2GgAoBwJbt5EiQrKCV'
        'bwKr3MkZYCLcF85NS3TKYVHO4VWRQeK09xCfT4P8whJ6k9ll5NpOAeOauX1vsQMGUhOG+hqtEFVeRj61DcXZjidW+6xG4+9MafVV'
        'jTtTYuc9EdxitlC4aTCT5doPy5rvBIGRwaoo/HB71NFcsPlfJHiq1k+8ASJt2uRhixI4FYv1u8Ul+0e0HnFa71nqEdpaSNuAOKwL'
        'Xr9p715Ae5r2jVWhF5hnbAnBit4FxtDHzVa6kjdcbQPwqlLK7eTUR3mqxrgRxFS/MLXdzdPohUOEDPjjjIAqLQLeWztZNQ9PdK5M'
        'UBPj3arF5AsVvbtKxEMcYwpP3j5omJopLOEBQI414/Hya0lWbMk9Cb7MWY9Kmd5prqQSSSPkYHYVd03RIJJGac+nbxLvmkPG1R9f'
        '/P0qxc3imVIIELyOwVVXuSewpL+K3VYtbP8A4X0qX1JCwN5Khz6j/wAg9wP1pkKqQlSbj9hFL4qdUN1FrIgtBtsLX93bxqMDHbP6'
        'D8gKvfD/AKRnuoVup821qT+9uXXufIUfxN9B+eKudLfD9ooo9S6ikMLn5ksQD6rA/wATn+Aew7n6U9tJJLHHEeIol2xoOFRf5R7C'
        'qrX+SSj0Dlv97k3u2nA7nzLawKtvp0DRQBQg3Nl5AP5j7ZyfbmoyFXJZgSBzg+P8VNZzwR3kcOz1JpCFRW7EnyfoO+KO6crNq+bC'
        'PZGdwD7AQOPvnPHfmqQUPqnV7j/UcQKoGILHuBrOFJ4y8sgW3UbjsOAfpnzUKiO1WW8kATJxHGf4B/vXeoWPUct9M+mq08Z5MDQb'
        'YuT/AAlsbR5/EULvNC6s1F47eexjgV2AeT1V2gfXnsK6PS6CvTLwMn5j30a6+M5l/onpLVPiB1YbXDxadA267lB+4ufuj/U2OPpz'
        '2FfqSy0z7DpcOnWSJa2kEYjihhG1UUdgKXvhonT3TnTcGl6YWdUGZZfTIaaQj5nJxz/gYFH77WR6Z9FSBjz3q7prCLyeTNMc9RE+'
        'JWkwLp0k7EiReck5JNZfAuG4p46+1C9uyUMbrEDySKRGlVXOTikNQqPbkjqK2nmFIHC8Zopp1jdXzbYUz+JpW+2xq4BanHpjW7IM'
        'sKIAxHeR8CmKmUnbMQgyjrGmvp8sck5VlY4IU5x+NUklRpGCDA8U4dQaxpi6a9vc3UM28cQwLx+ZrP5bxTLmNQFzxVF5D8P132M6'
        'nvnH3/X4mNgdQhNubGPFS2OwMxlTcSOPpVCK9VmClcA+c0Rtj6kZGMEc1yOq8Zfo8GzGT7dyI7zPjIUJKqDx5oZqQaWH0R95zRN8'
        'iMjHFDpPvKw8VVVtg8zDJLVDsAbuODU4j2nnse1Q2rn1Cp5NXUG/AbgDzWwjNZtWZtyeIZ+IWtG6mMEb8eeaz2eHJJNGZ98zl3OS'
        'aqzRgDtXoeo1DXPn2krDmBXj2ntU4028BUm3Zc8gMQD/AErjUfkQkcUOh1K7Ey4mY7T/ABHIrEOYt+saTDNcCJJ7Z/lGFBU1xrVt'
        'qcuxLLT5tgTHC4rqz6guJY0inVpAD3JOAPYUxie02RNFMTLKdqRM3DH2Hk1ZLt7EOq7hgRBudH6ottNkm03S5ZdTuAyIxdVFsnYt'
        'kn77dh7DPk8cdMfDebSPs1/NEs2st8zzSyD04CewQeSP5j57Y71oyXRaFnEkCmM4cF8Y5qW31HS44mn9QzyKDwvO3Hf8BWrVqZMM'
        '2B+sKBldogWDomeRg93qdvEO7BFZiT+eKvS9K6NYQi4vb2YxDtuIUGuL3qeOTTTNbIY5nbC7v4V9/wA6VNR1a6vSFuLiSUDsCeBQ'
        'U8fo0G5VyfvzBYVehHC0j6btojNai1G3kyZ3P+ZOTVm2ubK7iWa2lDoTwewNLYCaDp++5VP2jcocI2P+Xj9z/qP6Unap1Vc2bfs7'
        'TFM0krj04wOSfp7Zp3dsABAhlqYjImrT30Uf7tDuPbAGcVT6aez1/WpbE+syohMgXgnkDv8AnUWk6frl9o3qQWcjXIT544+R+APc'
        '1q3w36Ih06zhvplPryJkM6bZFRsHY3uQc80VQ1jDjiRCsTzD2haTBY6Zb2UEZKIuF3ncQPbJq7+x7eMmWUZNFoYkjASMdvNezbFX'
        'LcmngoELEXqWzguIJI1gRVx3YV+fPiIkWj3R2ShgT4r9DdaX2m28bC/1CC0UjgM4BP4DuawTrBNJ1bUJEQNLbKflckgsff8ACqry'
        'erp06es8wNxGOYmW8095GJIVYgnv4olaxXqsAZMCrqxQwoIoECovYCvVOOa5R/OsHwgiW/mfQJOSd+ce5rlnWM5kcAZ96g1/VotP'
        'sAFVmZu5HilB9WaUk+oxBPk1cU66x1DcTZsjs11ErB45A2OcCvtU6g9CNHtZBvPLfSk23vSDlW4PippP+aUsrYajuleqKm0ZxJBz'
        'NP066XUNPjkjYEuuTzXhSNW2yt29qz/oiTUIepba2UOyTH0sEnaM+a0jVdPnt5TFIFBHOQ2RXG+R8W+mc2KMr/H2h1ORkypsCyhw'
        'TxUtxIxiOw84r0LkAD2rwjA5qmH/AGDibnUcJcAAZJqUaXJKSDwaK6FbBm3lc47UaMcYfJTBr07TaIOu5pMLnuZn1HpNxBbu20kA'
        'UkWJluNQFrEDvPJ/0gdyfoK37ULGG6tHXAOVrHRpktnr2qFFARFVS2fBbsP6fpWayj8uhsWQsT3Eur6UCbIJy21dxLDAJx2qSyht'
        '7u8t5ZpGhubdh9lmycK5PG/Hdc1SjbJ2v908GjWl2dmpW4vrpY4Ek3ekjfvJSPuqAOw+v9K5fSu9mpDkyFbHPp4kUQlW0DXIb1o3'
        'aG6Vu6SgnOfoe4/7UUjUW2gSzSKFN2PTjGORGOWb88Ypf0i2t9Hubs2N9c2wuGLGKX9+hGezhjz/AIrx571EujfmWa0l5intsyJB'
        'zkgqfmVc+OMeM10W9LB6RzD7FJJUyK+uxI7NnAPYDsKN9OWEen2i6/qaZ82UDD75/wDsI9h4Hk80P6b0uylnlv767t7y1gG5LeB9'
        'zTn6r94KPPGfHuaodZdSyXrySM4GBjZ2247DFO1HC7m/aSo07MdxEHdadSGSaQ7y80hzk84z5po+EHRzyzR6lqkbGeT5o0bvGp8/'
        'Qn+1IvR+lPrGpx6lfEtHES0SN/8AkOeDj2Ffobol4E3sSAFOOfpWUN9W0g+0PZZj0rNH6X0u3tYF9NFRQPFMcbA/Kp4pVsL5pFAU'
        'lYh+tX59VSFAqH5qtxxBRiLxovJArO/iz1dLo2nLDp7hZ5yV9TGdg9x9aq9b9XS6ZZEhv3hHC55rHOpOo7rWCBcNkqeOap/LeTTS'
        '1lAfWRxBWWhBKOp3lxeXDT3E7yyOcs7sSTVPOV4NcE5GK858CvOrLGsYljmVzNuPM7Br4yBMk9h3rxQWPsB3pV6n6geNmtbUKqDh'
        'm96c0Pjn1J3dCRC5MtdQ6zpzWzwMQ57CklpxyVPGe1QzzLIxZjk1E7IOT2rpKtGlA2pN7ZdivNg4NWLbWnhcbjxmgjOmcLkZriTd'
        '4IphcibAImkaDr0BdXXbvH9acLfWxMo3tyfrWCR3U0DhlYjFMGldUPDhZskCoajR1apcPCB/abjpV3E0jK0iqpHmuNTuBbRs7fdP'
        'bFIGka3HOA8UvPsTTPLrFvLpkCTYEglyw78Cqdvw/g+luBDo3GJpejx+iAG9s1fcbmJFCbG63heRRmBo9nJBrt6gAoxGBIZJPSBJ'
        '7YrMOp5YvtbMq49Ry7sBk4BIH9ifzrQteuAkDBazfVIG3q54eYltp8Lng/nzVZ5xyumIExupTt4o5S7btqKf61GkxkkIibPOA+Oc'
        'fSqutzODFZQg4A3SEDvnsP8AP9Ks6TCwABBrmtNp9q7j2Yuzc8S5a2jSsEUcnye1GNCtLddRRprhBFHG7SKCRvGD8n51CwENqi9m'
        'c5P4CoHkwpHf6/4rT6sU2ADnExeDzEjrmF7aBm0zR5bYwNn11yAVzjkc/wBRQTQLe71WWSW6ZxFGAZJMk72xwoPuf7Vo11KdhGRz'
        '+tCmm9CI2wjVQSSdoxwfy96ep8j9QEFcRn82yjAl3pZCbk4BASM7QB7dhWh9NxzLEplYhc5NJOgK9i88sikKFXG4YPJ7Ee9Nq61C'
        'LVUjPzGrXQFVG5jzBJ94+6ffTXBWC2HA4zR6eOPTNIm1K7OfTX5d38TeBSN0hrMKTLFgZY9zVr4xavImiW1uk3384Ufqf8f1q1s1'
        'C11G0+0IThczNeqdWuNX1OR5JSy7uOe/1oNIAG5Fc7znPmuWYt3rzTVal77C7dmVzvk5nQAY+1d7gF7VGvbFdE4FKE5g8gyvfLNJ'
        'aPHC+x2Hes31izuLa5Kzvkk88099TX4sbAssqrKewrOL67uLqQvI5Ymuo8VXclfqPHxNgSBgqea6RRIvf9a4KMRyea4MTg8PxVuJ'
        'ITt7b2NVXjlQkjOKtL6i9zmu9+RhlqWBJACUOTw4/Su0tlkPyHBq20KNyBg1XcNG2VzUpnE7tGvbOYNHuwO+DTHpuutMBHNwfehF'
        'rclk2sor7003bl+U1sHEkOJv8WoSxchjwaPabdXE0YxnJpeggDXpjPhqb9KEMBXcVAHLE9gPentMrAcx1eJX1hPsti11eMREvceW'
        '+gpGnlluppbl0/hztHhRwF/sKKdXa7+2tTZICy2FuNqLnG73Y/j/AGFOOl9GmPpm1mljxc3QWWXPOAeVH5Aj881T6gnyWo+lWfQv'
        'Z+TInLniJWgdOTXA9edS7yHcePJo3H0yyuMR4FHNUkS0thb2+0Rrw79t5/lHkiljqXqvUoNOe1uLm2i9T7gXCPt9gMDP61bnTVKu'
        'CJNcZxK3ULWcbiK2dW9JdhYDO5s88+wpav7+CGMjcMjilu91+cTvbRleMuuPbP8A6ofE1xcJukYklskmuS1ej3Xs54HtAW8NGCbU'
        'DcFo4wuE/qRUtspP3SH87SOR/wCfSqenQNHOHC7uf60Wtrcq3Ygjscf39qDuVBgSIUnmG9A3XqNZyxSSIRuEm0nYR4Le341TIZb5'
        '4t20Kau6TcXVvdCS3Mm7s6rzvHkEdjQnqIsupzFCR8xFNjUA0hvcGEzxC6Xn2AepHKS3g1JrerPqenRC4Ys6cLSsHllZVYseaITo'
        '7MI2OwKO1Qv8gxrKjoyDscYErEnkV5nkV2wVfrXgbJAArn2xEiBOx3zX10fs9q9w44UZAqaAA/Mw4rmcib5W5T296hUcWA4zCKgx'
        'mZJrd7Ne6g7yFuDwD4FDWdwc03dbw6fFcb4iBL2IFJsz8121NhdA2MTXE5eR8ec1z68gPmvO5rkq3tRtxmZkguGDc1LHcc8mqmxy'
        'Oxr7ZIB2qQM3zCcN9GOGAqctbzISMZoEVfOcGpoHbI7ipZmA4l5TtlJWrMUmT8wqGOMMoPk1JHARIACcmoEyWZ//2Q=='
    ),
    'chipping_sparrow_08.jpg': (
        '...'
    ),
    'chipping_sparrow_09.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAwEBAQEBAQAAAAAAAAAABAUGAwcCAQAI/8QAPRAAAgEDAwIFAgQFAQcE'
        'AwAAAQIDAAQRBRIhBjETIkFRYTKBBxRxkRUjQqGx8AgkM1LB0eEWYnKCF7Lx/8QAGwEAAgMBAQEAAAAAAAAAAAAAAgQBAwUABgf/'
        'xAAwEQACAgEEAQMCBAYDAQAAAAABAgARAwQSITEFEyJBBlEUMnGxI2GBkdHhFTNCwf/aAAwDAQACEQMRAD8ArGTcAcDAr5b22+Te'
        'EOK92TeMyoD3p7bwrHH2FeRQBlnnw3EDj8i7faiLe4JO0gVhdsqNx3NYwkh6BnCGpAbmF3RXGOOTWWnnFwQe1Cz3BEgBORmtrZx4'
        'oIPeqvU3mCWsymhhQgHg0LqsaqCRxxW9vL/JFK9buSEPpV7EbZJ6ii6kOcA45oWWbykbqzlmyx9s0JcTeY1ndmVVMr2Q4+r0pclw'
        'Q9erq4ySPSgVbe9ExgmUWjzM8oBPrVR4mIvtUz0/DxuJ7U11C6EULHPpSjsbhrYizV7ndNtBoQSFSOaWXV6JJyc+teoLpHwCabva'
        'IZBjSdd65x3rKGMqcV+huFK7Tg0TtDjIwKuwEM04QW6Ule9D2ww2fai5jxtAzX4QYjLYptlk1M5CznC5rZVMiYB7V8iI7YohNqL2'
        '9KmqhLFoUxz5+acQSgRFvig3jUgsQPmsbuUxw4WiBqSRM9Rudu41OvMXd885prK3i8HmhLazAnIbnNcV3QauCZ2I2KylnOwEelMd'
        'SsZI4yyD7UthiLKQ2QfkUrlBBqDyDU6Ro0ZAEvPNNZJwiYzXyyiUWygAdqB1J9iEevpV6LsFQOpjLdb7gnOQKI8VQoPGTSKaTY1a'
        'w3Jdl9qz9WpJBnGoVeNlhg1razMuCaDuJRn5rP8ANojAZqNOh+ZwFywsrolOT6UDq0gdT8UBaXgK4BzX2ebOc1dlJHEkniorlJVj'
        'S+9lHm5o69JI4pVdjcp+arVLgRVdXGCcnij+mrd766CAZXNJ7yN9+MV0f8LtILQeO69zmjGO+YeJNzVGttpRhtchcYFTXUjlUZVJ'
        'JroPUssVjp7Z7kVzDV7oSsSTnNUMigyzIAjVJxo5CeM5JoqK0lSPfzW1rgzjcOM06kVDGMAcVxppKgMJOrcNFLtY1SaRIssQNSus'
        'kLdqFx8030O68IDGcVdgGw3Kh3KKOzXJcjg16ngHhnAr9DfIU2E1+muAYyAc08W3Q90RXJMdwQM1uGYoOeMVjd58XJHrWgkXw8UJ'
        'MkTGS4IJGaweXxGIJ4FaSxhuAeaClR13e4qt3qTc97AvOe1e4nwQwHalUl0+7aPemmnOGXzcGrVbmpBjASxSAK/Ga9S6VDNHuQjc'
        'fal19E23xEOCKEh1mW2kCseBV25emkbqnSLJ/wCQPelOqSb5jzwDRcbGOPA9aUXsuC2cZz61QW5ld3AbyQbiPWv1u22griXMpNFW'
        'iMy7scVVkXdI7hnLjPYfNKr0us/B7dqcbSkOcfpSa6mVpzx2qsDbJIoRrp0jYGTW8s3YE0ts51BHNe7iXPmBqXG7qcZrJMN2Pmh5'
        'kBY+xoCW7xJzX6a8BQ8gUCCjBmc8Qe7jjX+o12roa1S20qPA9K4npj+NqsW49jXbtClEWlpzghaZ28cRvTkAkyW/FC/xIsKMcZ5r'
        'nTyszndyKq+uJfH1E85xUfcAoTk9qUdRcoytuYmFQDkYpjucx4X2pdp4aUgDt71S2VspiHl5qtdtyFklqNpKZA+Cc0Xp9u64BHpV'
        'adNjaPzKM0NJZiE8AU6EBHEnaImmMsZGM8URaSl++aIkhMhxtr7BYMr+XtXYwQZFUZ5ljDj0zQs0RCELxims1sQnPB96XXL+H5TV'
        '70BC3TC38zDPOKz1ABQa9W8gSQ5xXjUGBpVuRJviKWiVnzjnOa0SQo4Va8SEq2RWkK5HiEUYUnmCbjJCZI9vpSXUrNmuMAcE02gk'
        'C45HNeigmkzimCARJlbOuICB7VK6tcbGIPB9KrUPiW/ziprVLJ5ZySPLml7JgVFlhG07l3Bx808gCInJxivum2u0YKihtXLpkR9/'
        'cURIUSfy8zS7mUjaretIL5SshOSc0bZqxzuPNC6oNmapJ3Tj7hMLac7yDRkrlU79xS60XdJyMU0kj3R4xmoUH5kqOIiuXkefCBqa'
        'WGj3lxEZXDJEASSRz+1Nen9Ni3/mZkBROTn+o+1MjqDTXDgFY40Hp7e1NemB3PSeG8B+MX1chpf3gdhoMVncpIzPuzhSf+bvjFXF'
        'pfR22nJbySSNIfKGI5Y5/wAVBS64YL9Zchj/AE59DQ97r7y3AkkuWJ+DxyTUkCp6Z/pzSUAoqOOooJBcyS91BwT7VONbPdTYA4zz'
        'RU+tRvshds7vqIPp8096Usvz11tVEA2huDk96qOnL9Ty3l/Cfhffj5X9php2ktGqtg4x2p/Y2RRN5/arC10G3jtgWwWxQOqWyQDy'
        '4+1V/wDHuh3GYG2ogmYqTwKGuY5JzhU9O9fL+YLKUz60w0poigHfI701iFcGSBZi+2tQp8wOQeaKMCL296KvFVWJHGe1A+OChGec'
        'UTUshhU+X6o0RB744NRmq+IkpU/Y1R30smw+nFT12rMSGpbJksSsmK7advzGxz2PvRt3uZM0vKEXA45zTV0Pg8jk1XVixOHUBgAk'
        'cow70a1t4cOCOKAZhFcBs01M4ltQe/FSuahRhrREWHJJbtimWkYcYzQqQHY3zWui5SRlJ7Gr1U9yAKMoLS+BUYPFbOFmXcBSSM7c'
        'IppzZJIIvMDioCWJwEDnm8BiM4ryFWdSTgms9bUgnFL7KWZZNuTiq3BBnGMEt1UHOKV6hAHY+oHamUvilc+9BAPvKsO/vUipMAso'
        'HEhyM1SabYmZRlO1edMtBJID6euKs+ntL36jaW7AbZ5VU/A9f7ZprDi9QioSruIEiNYvIoYDbIQiqxOR+n+aTXF0wDKhXxFwSB37'
        'dzSP8SdabQevtQ0q/i2flrjYIVOTgnKtn2wR+9MfxR1nQdB0nT7SwR5Lq6to7ls5LEuuQCe/AP8AenToXomfScXkMOFFxp1FN5qc'
        'fgkuOQSMH2qevdUlnulEAKqPX0FKZNYW7tZFjLAKyh93oTkgZ+xpp0zouu61fQx6fbO6Sd5WXCKKBdK11XMbbWoV3XxGdnJK8avh'
        'mfsOK6l+GF1LaXET3BcRy5XnsTVf+E/4ZWcGkiTVYhPOfqJ+kfoKourOinmhWexZI/BULHEI/KoHGeMU6PH7V3Xz9ph6vyK51bFX'
        'B+ZpLexrH39PekuoXImBwcUJPNII1QnDqNrD2I4oB7ht+GIxSefMF4M8S3dRTq0bGYkH1ovTX2KOe1fpzHK59eaMtrdfCO0c4pMC'
        'zxAvmYXt2DxnnFAgOw3Dtmh9XWWO4GD6020ZRPb7WAz7VBG7gwSCZh4XiQncKndTXZKUx27VbT6e6x7h9PwO1SOvxhXOTzS+bEaq'
        'CQYqgtxLKDj1xTq6sQtnnHOKV6VIRIAy+veqW7YPace1diQheZKjiQV+NsjA9xX20uCF254o3U7bKszfUaU22BLtzVGTHRg9GP7M'
        'q0ZBOa+2W2O6YDv3oG3lKHBreORWm3DOSKZw5fbRlo5hcR23iDBIzXQLG3j/AIeWYf00kvrOOOHxPDRcc5rWx1Im1aMyDGOKuRgO'
        'ISnb3EmrYkmIUcZonT7CMxh2XBoGVmadjj1og6iYodhOKg1ABhtzboU8pGe3FK7iAKcnFAPqbifiQ4zW35xpFPGaqYgSLjzp2MNM'
        'MGm/UmsS6IlrdwRmVoXDOq99vYn+9TPTl0VuOSeT2p5qOuQaS15czgZmsnhDu3lQMRk49eM4HvimvFkvlCxnSkFwDEH4gdBjqz8W'
        'puqk1iC0iQRtKkMZlcuq42jPl54yfT9a5b0JpEPUf4i69d9YXMMtjA0kG0vjdJuIXwz6AYPI7dvWnV9+JV1ZQLpnTluYLNDtMrDL'
        'SfJqN6r1TVb2db1IUtrjk+NFwZPfcOP3xXqcmwD29zcUs3YnUuprTo630Z9FsoYLOyKByqHaTKoJznuTgc+9IL/8WrHpfVbXR7Wy'
        'SYptVtriOOMcdzz8E1yiy0jXNb1B0S+luJlied9zcnjnHuT/ANac6p+CnWd08V7arHeCRwJ3En/DzypJPcEHPFVLbG5JscT+grH/'
        'AGgdJTR4pYNKuRNGdtxEj+JGvJCMHGOCQe9YaZ/tIQw3KDVtIvLaFs5ZXEqY9+2RSvpXo7RdI6Ntun5raC9mjbfdz4I8aX+le2dq'
        '+noa31bpTS20OS0stkUwVvMVB79xj2q8XUCU2tdTaRr6Qa7oFyktpcApIUbO2Re4P2IpRLdhhknHzXDumodX6L6pez1CaVLOYjBj'
        'JMbZPlbHbH9xmus2spnKqBn715ryWPZkv7zJ1mPY9/eOILkIck5z705gvIhECzgce9TlyhRckYxSy41DaGG/gcVn+p6fcTvmM9bu'
        'kMpdJcjPvXrQ9Rkik3b/AC5qPkvTJdBASQT2qh063bwd5q4HfzOv5E6jpE8F3bcsDkVJdcae8GZY1yhPtQGjarLZXaoz4T1zV9aQ'
        'W+uW4UsrIRRgjINvzCA38Cc30xMxA7fnNGvcq0TDODjFV150xHZAiPsaTX+ho0TbQQ/ftRtibbDOMiRmozBo2BPOKlzKY7nGDkmr'
        'bUNElUHbyKn7jS5Y2LstZ7I4b3DiVBCWqbWy+LCCByO9fo5JEl2Dkg8VlY3ISbwW9R2ppFZq8ok7VyYizcSR7TKnXGc2u1STx2oH'
        'RbSV1BdGwapYLNJl83f3NHJDDBCAdoxTSY6G4wALNySvbAxs74IHepm/mzLsfIqw6gv4kJRZACagtTuiZyC4JNU+qGehJodieLpQ'
        'uGQ0y0/c8IyPSg7exmlTxASR7Vr1Ki2fTZ/M3UlmszCJfCG6aUn+lFHJNEcJYgCNaTSvqH2rGumpZwhru71uwsYVwXLyDeFPqFql'
        'h/Dvp7r+wuIH1XVby2tTuNysnh5YjIUADB9/jNcl0jpDoy+WdG1PUpJ4yC+UJKf+0Htn0+1f0r+B+lWGj9JPY6e0rxM+8tIckkjF'
        'bOhwY8be2ejbRppsftUX9/mc51j/AGcNH0m0a/0HWtQdk8xiuVV8j4IA/wAVz/U+h5ReETzuWT0de5r+wyCbbaw524P7Vyjry2gL'
        'llt0EmMbvj/vWiyi7lIyECpw+HQoLe6TUtNs1tb63O8My5Ct65HqD7U/i69vQq2t9o7pIkTIZbeQqM/05A424xii5c20rkxruIKA'
        'E4yPY1nbyWc86QTRrA5Pl3/1fH6Vynb1BLXAVvtUmmF0QIo2XIjRvp/T/NOLe9xZPbOI0SXPiMz44/WrDp/TbN7hEngtXVscFQf8'
        '09vuitHW6/iEFtEIyuGRAR+p9qvU8WIPEhuo9HspNHtLq1Fs0UJRGKjeAM8jPcZ4pZa2sNrdLvIXe2EHbPxVqklk2nXGnCxFupyB'
        '5id7dqVWmli8ia2khMjLFlSByvsc/BFL6jTrnWj3K8+Bcq1E+sIRbsUx24qIlWZn2bTkmuk610/qkOjy6g3hm2iUFm3jke+KntHg'
        'hmlDSAH5rzeo0rbwG4mFkwspoz70p0pNdYlKKM+pqsn0X8sgjdBx7U/0I2UFqBvAIHas9QuYpZgkbBhnFOnEq4wBJIAWSN/owIMi'
        'x4qh/D6VbSXw5QSBRV1bF4cFgKE0+Mxz8NhfWiw6cJzOQbTcrdbuI5h/KUCp6+uAkZBAzTSKGOWPcpYmk2tW0ihuBxVhNEw3ZhzB'
        'HSGeEkAdqSSacspePAIou2kdHMbHGe1eJ5ZLdmkKkr6mlX5NVJxsD3JafRPy18ZCh57cUVPIIo+QBgYp417FcQZODU5eZe42hWK5'
        'qnKQg9sEjmV9/qAsScnHpU5Jrs1zOyqTtzRXUDrczMqEkGgtG00mZiBnPNKHIzHaIGwxfrMrMVZvb1qZmc/mt3fmqfrJDbJntile'
        'i2Vumgy9S6jqEdpBDOEjDNs3HvnP68YUEn096HFhL5dohYcJy5Nomuu6zc6ZpcVpaKGvrg7Vj25YD1J9gP3o/pbRAltF1Bq14dVv'
        'Jo/92Bf6F9QueFHue9U9n4vU34I69Z6ZZsurx5eGR4BHNNCSCdqjzBe4A7nIz3Nc10zVP4L0vpuk6jHNp+qwCVna5j27UJJUH1xn'
        '0rd/DqqAT1ujUacUg5nQLWe2jjlt4re2SOTicQrt2j2z7/PzXTvw5vHsNJSKa2aKJ3Pggclhxz/r2rlH4bQfxq+tt0ksylTi4aA+'
        'FuB5XsRk8ctXcp7a2jtIrcAQhANu04wftT2nxBBwJ2fIX7NyiVlkTcpBU+tRHVVnFLJhiuGfkH0AraDW57ANbOjSrACcsfngf3z9'
        'qV6pex+JAbo7Wk34UHPmB4B+5/tV7CJESH1DS1lu5Y5AOCStB2GlWbqFmHmXmOTuV/8AFP8AU40XdOWY/wAvuPU96Vx71kjWMcuM'
        '4HqKrEGedQnbTrVWCrIy8owP1V5sepTeWvF7JFMVOF3EK36HsP0Nfrm0F1LJZ72IXkqw+hs+lBHQwNRQupiL8SHGQT2BI9sVconQ'
        '3T7m4vxFcl5FdSQ5xnj5/wC9Xugw+JFhlDEeXxB5WYH/ADj3qX0Xp59KuI5NxkEi7hGWO0e+Piuh6JPaNaRxtAuM+pwVPx71cEoT'
        'i4gmt6BHPC9pch5rVlIwHxw3pz6g8j9BXF9a03UemtYkgeB4oWJaAM4YlM4BJHrXd9Q1GC2PgTufMhKHt/49KhetNOh1TSxqIJe5'
        'i4YjuVzj7+lI67BvSx2IrqsXqLY7kHHrlxgKQaoNALzbZZCe9LW0vw4RIQO3Oad9PSwi2AJXgVjYTbcmZJQiOp540jxnJxWVsfEP'
        'HH2oW4YOCRgfevWj3Uf5lYyQxp31BOBs8yj0i5KyCNwNvbtRWqrbSRNnaDisbhVW33Lwe9T93JeSOVX6TQkxsFQKMX30cCXJw9Mp'
        'NNS40YuBklakeoJ7q1mG48eppx07r4uLQwY5xgjNJvnAaqlAq6ks9vJBeNBGW2g+laNG8Uqhj9XNVVhbRi7kkdFZe/NT3UrpHfrs'
        'bAzVLLY3QStCfN2123DmmWiug5NDX0AW4cEVlo5ImaMn14rPW0yUZIyQXr9bMRLLezeFESQoH1OfYf8Af/NT6z3EmjNc6T05p0se'
        'mu0kl7cDfJDuwBgvhUAwfNjj2JNVvUemxzayGkdJ5RGoiRFaRkXGeFQMwzz6D9aTQX40G/JuLWe4tpmCXVm9uGWSP2KlmYemCcVv'
        '4dP6Q3Hsz0GjwJjXjsxL0T1ZfdM6hNq2k3DTT34CS3VwHdrgDnEMZ52Aj6zgfpVk/wCF/wD+RjL1brus/k/4isZVQQXABO7ABwMg'
        'Ljn3qE6/vOnLO6F34t/N+YwX06FDG4A/peY/0eyoMUz/AAy1/Ttf1eOzt9Jv9PjhKJbr4Quoo1J74JUL+vmprEC5qPMQosTtGhaT'
        'o/Rmiw22lp4k/hqjSjJU49SfmtGvbi+2TqpYgEsceUGiL5dO0y3EUsv5yUDDSsw3HOfYAAfFeHltBpzw27M7p9RxgHjOB8frTi+3'
        'iLM27mKLq4Lxkxs/isxOQMKo9yfU55pFqk0s8UM0p2rbo3hBv6mBJ3H7tmjbq/hh5MKbg3PYAD9qntQ1tEd43QMMEqe+Pj/XvRXK'
        'jHNxcrJpdpbqxaSaMSscdvKeP8msLAb4Ymccbgqtn2bBNT1jqr+N5/pXgZPYe1GW2qFm2cYG4HnjP+hQwY5u2ZL7xF2Ayod3Hv8A'
        '6/xQ11dMZSZG8oGD6fIoBr0sI5DzInH7f6FEpcfmbYxyoDKAdpA7juM1aGEgyh0G+WSFGd2RV+gZxin1xfxr4TIYynrgf3rn0dz4'
        'GzDFCzf8P0B9cfBozVpZ7eyglQ7WY5UKcA/qP9etHv4gVLfVJrKfT0QhjINxXPODileqaZ/DdCur2BpD4kGSufMnbOPipy01sPb+'
        'O52tHyo9aO6s1mJuiJZzM6TSv4USY/4nIz/al9U49Jj/ACg5W24yYonuVl07A5YrQWnRzJCWXsKXW00htVDAjimNpeolsVxzXj8O'
        'oH/qYha5qLyVgQT34o7p3dJf7valkE8SbnLY9hTDQbuM3I5APxTCZg3zJErbm9PiRwn9K+XDLBGZCM8UBOSJ1m9M8Vnq96JLYKvJ'
        'NNY8vdy2xEWsyR3zPhQMDsan9IlFnqeCQFJ5FO7izmYZQ43HOKHXQsHxmOWznk0pmO48dysmzcK1LVfAwYnxkc1P6rM93IH5z3rX'
        'V7SUhSCSM4rSGydIhIQcY9avVGOOjIJuUWoxM9wSqk5oRj+S81vAkl1JwrSjKRfO3+o070p4roq2ee2KY6lpFlbmO81GTwocjagP'
        'nf8ASq8WFsj7lh4FZm9ouRHURmntI7a7e+1C4fc0sUMrwQBfmOEDj9TSXT9HuIbdbiw07S7SOXhBbWMru/PqZH5p/wBR9bNo18q6'
        'ZbQQyQMCyEbuD/zn14/p+farXpHWrDqXRRPZWcUUiSbpY8AeGzen6ccVuYx0rGzN7ExTgnmc5foTqS4mjjjvHhUtucXEaOgB/pC4'
        'IX7GrbobpfWum7a88W7iYOVIWKMIFx3PH+KuNJhWOA29zksBlWJ7Y96AvOpYrGULcrHJn6Vxy3oKbGML1DLk9yPv3ubzWSkzsIXb'
        'LnAJx788Dmtr/ULfTrg2omKSN5VQHLE/b1r5f38dw9zcRQhZNuUHYc8/6x7VJxTLe38hkjlNwh8jqdoyfegMndP2q38TSyHATHZR'
        '3J96mL6bDyDxQEAOcnsB/wCaayTJFfzQyyRnC7Qeecep+Knr0RzTYYjwYzkgD6z6D9PiouAwmYvLgRSlexUEH35Ix/iitOupFfxX'
        'c+bk59z/APyhbl1lmJUgY7/rik0urmG4MZwvBOPtxQlpAE6BBeRPEq5GSe+exppprhDh2wx7Emue6RqguEZQB8c96eXOsoIxGzef'
        'AAx6miD/ADBIjy+kJu2ib6lfBA7AVpPezPEbeSQmIDKBjnbjvj4qcivpW/nO5eQiiLa6fYZHPPseQfii3zqjWxDT+LJuCDYQpPAP'
        'tVHbWsuq6LawyMPChBbt3PbNJdGhe5hYysFTILjsRVJDdJb6eUTAHYUrqsgC7Yjq8nG2Ibm2jgk8POcUlv51jbjjniitWvtszH1N'
        'JJXM8mTzzXlGx+o5AmcB94S12zx7RmitGNwtwpTdyfU1ja2quvAI96o+lbVHnIk7L60Z0pSiIVVHwn8S0WN1AalesX0FnGoZwGzR'
        'Gpj8sWcdlqLvGa/meRiTtPHxVgYuxUSCZWR6ij2/iFgABWC6gZCeTipC5v5IojEucjitbHUQIhlvNV3qKCAIMpLueOVAOBg0XK8Q'
        'sQCR9NSE9+yMCTwT6151vV2jsAQT2p5NQAnMHkSw0zU7HQoZSZo7i7XvIz/yYB7k/wBRqW6i6xup3Zop2RP6Z3H81/8A4KfoH/uP'
        'PsKjNQ1tZXCQ+cJ9JK4Vf/ivv8nJpaJWlcs77iTyTTSlca7cYm2oXGu3GIwmuJLmTj6ckgZz37nPqfk810L8Dru60zqKR3RnsbhN'
        'lyxOFX/lPPqP+tTvQPTb69eDcxhsY2/mSgcsfYV0XXobbS7aK1sYxFFHwAPX5PzVL5/R93ZlbZBj/WdojtbORJPy8cTZTIcnOc1J'
        'nS7BL6S4awJvnGPGfzKAM8qD2PJ71IdP9bS6cFjnd2iZCm5fqX2/WqG01eR4zK6iSKcfy2XnJrS02rTMOO4zjyDIOIuuLLTreS7m'
        'uod146lYVEhkO39AMCoC9uk0ycyrKYi5bxPICV9AOfX19qI17U7q21WZXVtjSYbBIAHxUr1PcSzXyzmMP4fKqV4NWObllwLVbxBc'
        'l4ndlGfMe7E0rluSyjY/Hz6V7un8Rj2JbDHHYH2oFV5cMMDNLtdybhccrOjBWBIGBj1JzmlZsTJdozg+Y8n4plC4WMYQA9jXtMtF'
        'zyQ3ehnAz6kK26KFTHPcV5lV3YAehBJNbh96Ann4zWkMZI2sM5OakCQTCreTyqB2Pf4phYJLPtVVYnPArTROnr+9ZSkTKg7uwwAP'
        '+tX+h2GnaXAxVBJMv9bDn7VVm1C4eD3FsmoVeBFdov5KzS3k4ZvM2e9HhFkgwCO1Ieo7sG5LRk8+1HaWtyNPE7ZwR61lvkd2sxUg'
        'M3Mn+oFMdxtYcd6Gtgrp5e9ftfv0llYHvnFAabdqspXOaXw1vizCmlhotsHj83Gar9K04Q24lGPmo3Q7vIIHFWNnf/7oIyfSnc7K'
        'FNwG7iXre6WDT2A+pqhNLum2uTnFUPWckl0/gL3pBJbGwtDu9RWDiOTllgck3B7grO7c4NebK1d5Rydg5JoK1aSS44yRmqNQlvpz'
        'E8Pjj9adwLuNtLMQ3RVqVxDPcrbREF071jq1pcT2WxBzigLG2MOptK53M7ZbJ7VXxzQvbEoobaOauLi6kutmStx0jCbVJLbU40BB'
        'JEoI7HHJGa+6b0jKJY/zE6zJnzCA5x8GlEOuyFGM8olZIgNq8ZY8ZwfTFaR68WkmVZPDbKpnPYr2/wAkV9Zz/T/j8w9tqf5f7mZj'
        '8rqsY5o/0nZ+nmjskjtki8GKMYQAcUd1DBHeW+5fqI5ri8HUl3DG3g38qhXDKUkzwf6fsae6V11qiwxpeyRzEo28GMDzj6Rn0yP7'
        '1g6n6NYknDlB/UV/mSPIhjbgw+8ilhk2MTjNV2h3Mx6XW5i3NNYyAbQPqQ881z286vs7iYiW2dUXbmRT6tj0PsTg1X/hxrmmajY6'
        'pp1pcu13sMkcZ8pO3gnPsDWG/wBN63RZN7D21yQZv+L1+HJnUA98RVrGpQam7SQthi+DuH0/GKR6jBfTbo44mmQjCn5rXXtVuLvQ'
        '316KCJ5LK4NreRAYOQSN3H6d6O0u4ddOj1BcGFyACD2PtSmTLnxGqsRzWpn07e0bl+8jrpZIJ1EsTKfUEEYoaQkOeK6jZz2+o4hk'
        'ijkVu+5Qacp0pozwtI8MKEDsFAzQL5JC22otj1Rf4nGLfc77SDz7UTLBc+VY43Hly3FdDv8ARNJ02VXghDkjO002ntYdZ0srHEo1'
        'GOPEeBjxlHO39fb9qsGtRm2qOY3h/joxQ8r8fynK7W2kIBJCZ9T610ronprT/BW8vHWeXuqH6R/3rltzeySXbIBtKsRjGMYqv0PV'
        'riCBMOcYpHP5Fg+2qEy82odvaJ0zUW8OIbdqrjgDipO91Jolkw1OLfxdS0YujHcFyTntUO0jTGaLdkqTS+oYuVIixUgifY913OoA'
        'zg81aalILPpzgDITiprpiFY3YsAefWqbWkS40hy2MY7UwFKrLLI5nH9SuHnvVjQfU2MinVlY263cKFsAkZpMvhjXxGo4U0y1ef8A'
        'J3MUijJ7feq8YCC/mSlUSZa28Nvbn+WRnGe9EWd5ulcA5Aqf06eWTTTI5O5hk5rTQroeJIz8YFVags4uA9E8QfXNREWqhDyc1jrc'
        'hltA54yMCp7XL+ObqMhWztOSBRniS3hJRSUUYFBhG1CDBFhSIZ0zEocvIo2171yRppMRHCL3+axsZGijYMcV9MokiIA5NLrkIJkY'
        '3qTySy/mppZGwqjiqDpG7E1jMjAFnzg5rzF05c38DGOMhT3IFedKsm0y/Nuwwabw4zW8/MIMRzOFRanLI5kMzAspaTJ9AeB96aab'
        'rTpIZpwCqsWkye4YYA/WiNR/C7rSx2H+DT3EbIZC9v8AzBgehI9fipjULK+sJDa3ttNFKTuZHQqfgYNfR8Pkb/K1yx9KjfEo01SJ'
        'CrAeVEY7sep7feiobwS7TFeSRK7RYBOdp9SfmoyKaRAqy5VQ288fsK9pcNGo8NtxXLE/J7ftTia9vmUNoF+JZXOozGSO2JV4d+1c'
        '/wD7H9qeaB1wOmLC6ubJFN7cwmJQOyAnDEn/AKVzpdQYT7UYN5gAT/zEcmvL38LJKkm7b4e1doHmbPJPsKnUalcuPaTwZOn0zYXD'
        'KOROpfh3eX150prUt26zJcTv4m3IYHAYkjt68VK9N9Saha6TdaY1zI9sQ8sblyWQleMfcU//AAjfwei9auJjLHHO5Cso9k7keuM1'
        'BW0sJtkXIV9jevck8ZrHwafHld1I44m3qtQ+PFjP3BuVWifiH1RpwQrJDN5RxLGOefcVX2v4vXMiD85pxXb9Zikxx74NcviRXcFQ'
        'GBbYDn2XzH969KQypLtxnYG/RuP/ADTzeH0WTlsYv+37TBORhwJ2WL8ROnroK035yGTgMHjzjPrxTLRutLX+MQi1uY3iPZwcGM57'
        'kHnFcRSRCu/bnw+OT6qcH+1MYigAfIJUEEeuFI3H7gilD9L6MnchIP6ycGty6bIMi/E7/wBXaLperv8A+oLW2VLhHT89CPKswJx4'
        'ikevuPvUTdskOq3FrapL4aSEKCDkD2NINOv7u3iYLeXCDz7VEhw2MHGO3Yj7inOsX9/JG88l7HIx3bnCgFsLksdvrjn5waV1n0k+'
        'VbRx+xjeXyWk1DgshSzzXP8AidU6DOdDnJmjbEZz5hxx6+1Rv5OeG4kYgEOTgqwYfpxUX0J1oLTVJdH1CJGiuUILRSeSRDwcH0I7'
        '1Q3sdpp11dRWl7dR3sSpLatKpKMp5AbHcEcH2NYZ8RlZvTVSSPtH9ZpdLjxDKrmj1/uPreQ2yc8H1pslybjTnjJzxS2yij1BY2ku'
        '7d5fpl8NgEZvdfj+9OptPNnBlRlSO9L59NlxD3qRMckE8SB07T4zr7mRec021Hp6S+ul2DCqeK0aFU1E3AGBTmy1IIhYnGeBmsxy'
        '12JYGBPMRzQvYYtZMGsrlBbwyOnBdfSmGsFfFWeXsx8ppfOobknKYyKZUF8fME/mkJp0Kya7MJgcsdwq40mKGC1kxg1NRLHNrcsq'
        'ABYxjtTHSLws8viEbc+9LHhTODVxMdXuRDE75xW3Tji5VWznmsOp7aGTTXmSTBHIFa9CQSCy/MFTsHvS4wtQMq2kNOpW9zbWWiAl'
        'FHHJqQu5bWe78dBlmbii7rVLSSKGC6ZjFnzAHBNAtrWlXN/DaW2nRWsUZOHzliSfU+1aXroAFJjFjbZMrNJi1W0s4g8EuXXco2kn'
        'H6elLOovyN4DFqWmW0zdj40IYj9xkVTX3Vmo2Vguy6C7SGA2jK4Ax9uBxUjFONWk82MseWJzUhQvKmWlWPCyW1H8O+k9Wjkmisha'
        'SMckxHA/Y9ql9U/B/T5Y8aRqkySgcrcAEH3ORXWtS0q6t7f+V9OMbh2qeZrixnUyuGGc5BpldfmStrywpkUWROOax+EvVFiN9tDH'
        'eIqkgxNgk/oalb3pXXrIk3OlXkQOQCYieB37V/Wi3X5vTwke0OOVNfW6enurbxi+R37etNp5fLXIBgPmK9T+Y+nNdvtO6c1OwbxI'
        'kWFhGRwdzHGD7+tSnikBAoAK+p98/wDSv6mt9A0Y6rLLq2nW94dmxY5YwVXnJOPc+9RPVXQnS1zq7JFYz2ak/TDJgc/BzTGDzGJb'
        'JBFwzqFyAbvicUS4IYKkhVTkfbHJouPU3jdW47o2PgDAFdb0n8Bf4tKwttcSGNgNjSQ5IPscHt2pJqX4FdcW08kcdvbzKpba4nUb'
        'sHAPPbI5FamDzGFhYf8A+QGxo3MhBqCLEIwM5Uqf/t3o+2v4nQybth88m085XAXFCar0f1LpcpjvdJuo9mcnYSMDv2pJItxHJyjD'
        'GRz2/StHH5AHkG5WdKjDgy0tNUWaOBWlCnxFXJ78IQf34pjBrHh20DNMd+2PepbvyV4/+p5rnTTPuzngdsf3rT89KcJv7cj9qZTX'
        'iuZS2hs8SlluAdVtLmBVQRIFbgDyngnj7Gr9uo7c3KKLnxEEUIRicdjsZcfIxXIYNQySHHHlyfXaPaj7e/gd1iXzblVFA992TUYX'
        'xrmOUdmFmTIcIwnoGdmi1G0dCVdQiBwTnzAq+Cf2x+1H2OpXcaP4Nw6HkhfF4B3YZcH7fY1xu11SSZpGSfmTdhWPqxFNbbqK6FvJ'
        'IjB2YPkE5xkr/fIrbGqR1oi5itpHUzrWn3xubT8wZfEhbfhWXzDBxjPwaKhmheE7Vk8uAcEHBxz9s8feuUWPV81tfmKRVijkllXg'
        '8DcO/wDc09teqbZUYeKV3Lt8pwGzF3/cUk+i8fqv+zGP7V+0kjUY+iZ0NrmC+sNjyAOMY3Lg5xxQXhSNbFEG8bfIQfqqXh1u3nfw'
        'IbhVkkYbXY/QzJnn9CM/emVjqUNxBE6yLuLIxCnhc8cfcZ+5pHL9K6Bx/CJX+twhrcw/MLgccLWNpc3FwjIWJ70t066XcfNxVMLt'
        'L/TzHIysGAOJOQCX24z8H+xoT8lpExG6IQg7vNG+NoBwTg+zcfcVhaj6KzAH0sgP68Swa9DwwIgke68ieIsQhOBVzothDZdNxQZw'
        'zVOWGnwmOMwXGAoy4kXBwDTjVZpvDiEQZ0TAynNef1nhfIaJKfEf1HP7RvFnxZB7WElevZ/yVxGqsQDzkVhYzGaNLhVO5h5v1r91'
        'tbPdC3bDF847U3tNKkh0BJApWWPBGPUetYjaZitnuWlODc//2Q=='
    ),
    'chipping_sparrow_10.jpg': (
        '...'
        'AAodBnNJmco/gNRsOMOpD9z/2Q=='
    ),
    'chipping_sparrow_11.jpg': (
        '...'
    ),
    'white_throated_sparrow_00.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABAUCAwYHAQAI/8QAOxAAAgEDAwIEBQIFAwME'
        'AwAAAQIDAAQRBRIhBjETIkFRBxRhcYEykRUjQlKxM6HBJGJyCIKS8GPR4f/EABkBAAMBAQEAAAAAAAAAAAAAAAIDBAEABf/EACYR'
        'AAICAgMAAwEAAQUAAAAAAAABAhEDIQQSMRNBUSJhFCQyccH/2gAMAwEAAhEDEQA/AN7gVTKozVofC5quQ5Ga9NkJRIntSy+Q8+1N'
        'xQV2m6kTjYcWZu9iJ470sliIJIBrRzwgtjFBS2/fildGM7iy2iLOM0+020PFDWtsSwOK0OnQ4UDFd1GxYVaQFFFXZxmiIVwoBFVT'
        'JjJFJm6KIAc3LVKFearLfzMGiIR7VPGex0oUi1EJxkUQExzXkY7VaeKtgyTIj1OwomNRtzQ6c1cGIXFVRI5rZEx7jmrY48VKLnFE'
        'KBijQtlG0ZxVyJHGpeUhVHcmoOVQl3IVQMkn0rI9adS2lxDDpmnTeNNPJs8gPLDsv5oMkuqbNhG5UF9ORXFis9zoAWTx77e8xwc5'
        'bkVueto31Ho+ayEnhXdyOVkA4Yf5rE/Dq3k0jTDJeyf9UZctCV8oBOCPuK6P08bfWVNzbLElvathom5IOP1D2zXjcibk7R6/Hikt'
        'nOPhJqmoWPw61DSdcZGaGa4jhij4cBTnbj/cUr6e1q21/S/nIEaN0cxyxNwyMD2Nfdb3UFp1Ddpo9rLd33j5mCAhd7ehPYcAUXZX'
        '8NzpnGgyaZfrLtlDD/V9zkd/vTuNklB7WmK5OKMvHsou1G2l3hZlzinN5ECKGEQCgnvV01bII+H0AAAHtRKIpYcVTGMNRUY5FcnR'
        'rRIMcCpjBWhzIM4FSEnHentnUWuoAHvQc2MmiXOR+oChJMc+tZQDBJeXqswhvSrynnoq3hzWqNgtlNrajI4praQbfSp28AGOKKVM'
        'UEojccySKNlC3PlzijFB7VTcR7s8VLlg2i3HNCZhmTNGW/tXjwndxXqKUrz1BxkXuSaDIwBipOcDNVx5OKm44qvG3RJkSPI2w1XK'
        '2TQjZFTikweTVuNkGVbGUQ7UQcYoOGQcc0QXG2m2IK7+3S8sprWRiFlQoSO4yK4h0fpd/afES70y6uHmOnvujkIIDk8r+cGu2vJz'
        'XOb+cw9b395bQzXF5kIixrlu3oP+TScqv0bB14dFeZYNMkinFu8x5ba3dj6gH0oPSuodY0y4eG0W3aykJZgBksCOSPrn0pb090Te'
        'dRXVxqZvZ7K5tyF8J8OXY+vf0psul3mjazc6PeKrgFGimX+tO5GPTmvJnNTkoxR6mOHxxcpMst7SNDLcFf51y3iSkjnPp+1Qmizk'
        '0wkznFVSgbTXsxioqkeRKTk7YgukOM0JyRTO9U5IoTZgdqBmx0DKMYomNuaqfGePerCh2gg0IdgLS5Ir1ZM0EsnNWq59DTOwfUNV'
        '8j618TkUNGSO9ERsuKOMrFyjRADzUwsl4pfuG+mFkcimxEyGcSDirkj5quAjjNGRgEUTVg3RV4YFVSrRTjFDy+tLlCxuPJQJIgqI'
        'UEdqskqndgmpZ4lZdHLouRcYxU3GQarjcHFWseKxYzJZAaQYqonBohu9UvjmmKNE03ZbFJjFX+OKWu+0cGq1n82M0YsZSzKFLE4A'
        'Gayvwq01NQ1u41GchzcTySTYJzEd3k5+3pTS9uvDtZXLhcKeT2B9Kv6DtxoGk2lrFJDG82DcSSOMszDO4E/ekchqtjsHp2bSLK3t'
        'LFJIVjjQ4HoCcnk5rPdVSWN5qimGWN51QhnBztGeBxSbV+sYtOjS3s1bUhtZIwsn9ePX3pB0jG0JmmvZGTxm8QDHYk/p+grzck/j'
        'a6bZfBd9S8HF1G8Rw649iOxoaRwAa1KW1rfWzrG8UoKnDjvurEXMrI7ISOCRxV/G5PzLapkfI4/xPTtFV0wLYoa4ZQu0VKWT1Jpb'
        'dzkOcGnNk6RIt5qJt3B4JpfFJuouBwCK2Ks6TEqLsILGrw+R5QTQpJJzmpGbauAeaj+Sy3rQQz45b/apibA4PFL5Jtw5NRFwNvem'
        'wk7AlHQ0STLd6Os5dtIoZvMOaZWsmcc1XB2S5EaO2lDYyaPhbjGaQ28uMc0fBc7R3qmJLIZSEYoOZwM1410pHBoK5uM55raOTolL'
        'NzVErg8g0NJLk96qknVFLvIqKBksxwBS5RHwyMNjm2tyaLWZSO9ZbVdf0rTbX5i8vYoUxldxO5h7hQMn3zjFI1+JfTZlWKKW7ldv'
        'VYRtP25zU0pxXrKFGT8OhNKPWqJZAASKz+n9RWGpZFrcBnAyY2G1h+KJlvByM0SaatCZJ3TCpJvrVHieYnNCi4DHvUZJdqM+CcDO'
        'B3NDJhRBerrspoklvEhkurgiK3QD9Tk8Z+g7mtnqHT1n1DcQPcXMqiBY2CwEKCygADjtg81z43U2o3MLxWcimE7kWXgufoPSul3X'
        '8Q0/pZIbWDfqFyNqmIHA9cfjn9qg5St2VYJfSLrLoePTdCmuI9VkuriGRjA8gVgVJ4BI9ueaGuJFjcoS2/ud3/FQ1BW0bp2aPTZb'
        'mC4kaGeCZ5N0cuWCvj0xn+n60qa8uJpZWvIliuVkaOVVGF3KSDj/AD+aRxleSx/ItQpjBrplXCyMv2OKCkn82AaDnuDnvVBkJOc8'
        '16K9IXsMlmBQjNAyck5q2LL+leyQvntRUzAaNtrY+tEoxDg5oOeORDuxUoZSRz6UUXsySsX+IO2aCnuxkgH1oWa5JGBQxY+tRxx1'
        'tljlbD/GZh3NQEpHrQyv25r0nIyKO0jKGEE/IyaZ2tz6CkEKsW5OBTSzBA7Y+9HDNQuULHsNweMmrfmsetKvE2jg1W0596fHPZPL'
        'CORdntmq5rrIOTSc3JHrVU10SO9O+YX8Iya554Nc/wDiB13BptlIYFS7keZrWygUZLzrjfIw9VQkAD1b7VqI5ZpS3gRSSsgyQi5x'
        '96VaF0joukameoNXSO6jjcvaLv8AF+XJk3O/B9WbHbjAqHnc2WHE5Y49pfS/Snj8ZTlt0gL4UfBfXevNvUHW2pz2lo5AjhV908vP'
        'O4n9I4+p5r9IdPfDDpjRtOgs9P0u0SK2UCN3jDyE8eYse7cA5rjfxS681zR9Ot4ei7K7RxIoeQRgkrjnapB9fcU1teu/ig9np4t9'
        'WsLeTAaaOa2SRwOwVyMDdjvjGK8nh5uZmh/uYKLvxO9f5LcmPHD/AISs1OufBCxn1BdR0+ea3nLu8sqyeeRmOcn07+ntWT1jpnX+'
        'n126xHFIucJcRNlXH1HcGq9f1T4u6xrMWlx9QZ0W5l/mXFtbJHLDge49M4obo221TStCudQ6iknn1e5nZZDeXG7CA4Ug4OARg5+1'
        'URyTwy/8OeOOXRWjgCvTKM96+1y70xDHJFMkZkQEDDcn19MHn1FASM6ElgRxnkVdDPHJG0RzwSxOmN+n4Dd9QIUjV2WLsfuMV2Kx'
        'igkVFmYJLC6mNOcEemffNcZ6Cvr1L26uLOG3u8OkaMZCp3EFgO3Hpk/UV1TSNetb1Zhe2N1Aofw8kZVjtH6SOalyzUmNxQcdjzr6'
        'ztb/AKIvNPt9P8V0hYpDCmSxxnjHbn/euU9N6SsOmxxo9zJklmNx/qAk5w3sRwMV1fSFkgsXVZS7NK3y+D/Sf/uPxXLfhbFrF7q/'
        'UD6orotveyxBGH9ZYkkf+3H71vH1kWvQszbx++B50feP01U2iFTnaTW6itFA7VI2Ssf016VI89tmIt9MKnlavksf+2tc1koH6aCu'
        'bYDPFazkYy8s/KRilEkBRjitneW454pJfwYJ4pdbGHNFUjk1BzTCS0dm4GBVselsV7fmkzyRSKYx2KolLMAATTi0sWZBniibGxRH'
        'GV7U6ggUAYAqDJkf0UxihQunlRkYzUWikj9OK0ohXbyKX30YGcAUmOSSYUooUM59areTA5NTucbtoU596HMEncnirMU7J5qiuSRi'
        'ahu96udY1Xk9qEebnCAk/QVV2FUHafe3+ns72F9Nal++CGX74YEV0XQdL1HWNDEGo3VtcCSPdvMIDOGIOM+hNcsSG4nYHGB9a1ui'
        'dep05BDY6la3LoqECeOPcMZ4B9QaH44ZHs1zlBaLOpvhxr9msqdNvBd2UUbOYLuU+IuOcIcHI+hrM6LfSLDEsqgSrGA42k7W9hWo'
        'n+Lz3U0h0iCeIMAyvcJhR/dwe3p/vWVl1yN7qbxJI0uZH3nOFznPOe1dJLGrbMjeQ12l6tJawu0moSpAWxsfCxljx3rG9Y9Xpb9R'
        'Q2vzF7Z+ESkckUCFZs8HBIyfxmsr1hq2pXfUEGkSFJbeNFmlPihAxPYJyM9uxPtQq65r6Wl1814Gqx2km23juJRbyKAuNylMgsD7'
        'NzXnT/uVl8I9UOLrSodVmRNH6ot5VMjZttRLRmFu4Ifbwfp9e1D6kNa6YMdtq5nsI5WZIrpNtxay9jn0x37gVPS3hu76HWLdkhvb'
        'iFI54rsGSPcvfz48v7Yo/XdTh0y1/hXVPT15HoNw29buwkW5iDejgf04zyCP3rIpLwY5Sa/rw1Pw71vT9IMkN7YvDNL/ADBJA29L'
        'jPYj14HAGPTvWh6s1d4tHsbvRrw6faS6nFG99PGDEOcHJ5wBnGfQ1xrSOnb+aMWukahDrOmiMyobdWjePJ7FWGVYd+9R1bW9f0PT'
        'r7p/W4JNZ0q5KmaKV2WQsuCsgJHPsaOLk9Cpxiv6R+nIotZtOq4GDxy6PcQGUkd1lQgAfkEnP0pxY2kFsJfCXHiyNK5JySzHJ5rk'
        'fwj+JNjdaEmmLaX/AMvZBmUSDfNEnfHp4irz2y2PTjNdX0HVtM1i1+Y0u+gu4vUxtkr9CO4P3r0ONFJf5POzuTdfQaFFWqoxX2BU'
        'SwBqonRGUDFLrsDmj5m8tLrpuDXGii7HJpPqKgAk03vGAyc1n9WuAFIJrGw0hQlnEoyRz9ahPH5cDHFFJuY8mvJSgH1rxrs9FJIX'
        'xptbPajIpkUckUNdzRqpxjdSO6uJix2bq6kzm2jVG5THDD96BunU5Ytn6Cs5HNd7h5zimEMrFMNzWrEwXkPpmZ3xGtexwyt+oUZB'
        'GGAOAKPhEar2Bo0uoF2LTpiSLllOaq+QjgbJxTpiWGA1CzRdz3psGCwEFF4VP3oPVrQXMQcqW8NgSAO4poFTODivJSsUEzhQ3kIA'
        'PvjircNWTZG6MTe264l+TlMcyupJkz5sjJUD0HYfvWc+JWm64bWG+gkiljBCpFaggRvjIH144p9AdIttQOquDJFO3mOCTG4zwR6d'
        '81pEmsZdLVru2lWBU3bI5ACyj1z3HfPP71Hydz2V4JdYaOIz6lewS2s1zbjw5VWN4yclW7fg1tOmOh7uJJ7nqi+istEuIy1orPvn'
        'lxz5VXJH3pudW03pa7uIOntO0yKzu4/5BuV8aRD/AHZY5Bz6ZxSMadqVwyNHavdiRjs24kyfXHIYfjNSy6x0URk3ssSCy0+zkh0u'
        '2u7iFJRJFcvcMzpj+5FOAfT7ULP1Neut0ttpem3MNwpjnFoXil+5Xdgn71oYOnGsrpT1F1Fa6UIgryWjt8xdBD2ZNu1x+c4oLUNY'
        '6PgMsK9MW+q3EcpWK/vXkXxE9yqhSPpknFL61tjFNNUgv4W6tcRTOsPUmpafYWkZlezvrMyRSgfqj/UuTz271XrGvW93qd09xokU'
        '2mmVjBHPaMWiX2EsTbh+RSodS3kls1kb4taO275fx/IMduPpU7fUkQktbSfdO1BOTbGwhFes9Mul20U0/S6alDdvjNpFfpNby4PK'
        'vE4SQfQjJH1oC71zXdMdNa0ZrqxuImw+xirxH+1xwSPqRg07kvdOu4dtzbq2Rx4sO4j880DJc3lnbuumX06xMMNGj+LHj2KtnA/+'
        'I+tNxciSexWXjxatHUPhV8d4dYjTTep4fBvl4WeJeJPuvv8Ab9q63Dq9tcwrcW86SxOMq6tkGvzT09Y6LqelyX2p6JCtxHIEFxak'
        'wyDIyGC+uMd+R9acwa9rcGtM2maxp3hIAHtniZNwBJYn03HI82fTjvV+Pk3Kn4Q5OMkr+zvkuoKV4NK77UducGub6P8AEfT5yINV'
        'MdjNjkiYPGfz3H7UbedXaExKjVIM/Ukf8VWpxfjJujQ+vtSJB81Ib64aXIzSxte0yZsJqNqc/wD5BV0c9vN/p3ET/wDi4NdVmhk2'
        'owIcA7m9hQMl1PO52+RaWW5UnJG5qOjDbcscfSvPjgb2VPJQSkUY5dsmq5UBzsTipRAe/NXLjaadHCkLc2wEwgDmvYsDjFXSj0FQ'
        'jXL5oqoyy+EOuCpIoiOXH6sg+9fRL5RVhjz2ovjUgexfFtfBGKmy8dqojjKEEE1cznbzXLBRzyIDnjGcjvQUzHYyd8jHPY0wnbPp'
        'zQUyhs0yMWgG7MhqLwW19iSzPg3LEllGdkmMMjD34r6zGUSKK4kE6N4kEgOQE/t5444p3qdmlyhJBY8b1zzweHH1Hasvc2txZFp7'
        'KZnkU7ShbuTSs8OytB4p9fSGudILq0q3MsVtbTAEEw7lR/8Au258v1xx9qGtejbxUMJktxGMZOGzkfUk8U+tNRutirEjOjDad/p7'
        '8fSnmnwysC07ExMMKpbnP3rzcnZF0Ov0Y0dHalBPHdRLbmReC5kzxjGDwOKCvNDvlmbfa5OfMBCCOPzmugTWs8KtLDe/ywexOfwR'
        '3zUN11PcqslskibQFb1GfUetLrVsPt9IxljodhJblr6G9jJ7fLwPvz/8h/iq5bLp+1TdJ0/1BIewbxVQn64wcVvJdNuYtpI35GfK'
        'Mc/f1NeWSTyTmFoLdCWyu7lifrn1oOzGf9nPDLYo6pBonUjc8j5iLj88Uwt9SWMcaF1PG4H6lmiJA/JrqNtpFjEgbUyqKMbjgcH8'
        'U8m0myhtDPFbeMu3KEjAx/8AqmKNinko4742p39myW9v1PDASCd5h2nHbsfSrX0rUL6NV8Z42C4Z5EVXI+u3ORW+e/vy3hwpaxqD'
        'x5RgfgUPDZwjxDclZjKSWXGF59MVVDE2IllpmBuemb9gIR4N3luNqg/vntUpOjL5yDNFbnAwMtyPpxXRVMMMIjgjSNB2VAAKod8k'
        '1RHjw/RXzyOfjoqVMHZE5/8APGKhN0fdom6OMcf2bj/g1vwR2q5CoH1o1x1+sF5n+CG0jVSCKKmbC8GhbNvJzVjNuOKb9Absuikw'
        'MHvRUbgpQA70TEcLWGsm/JqUC1Wxq+15NYlsxsMRfKKtUZ4qEZ4xV0YpyQts+CGvSvFW4ryjSAsEmUH0pfNlWNM5+KXXAzmsaNTF'
        'tzIVO4EgjmhpIrW/bE/kdl2sy8bvb8j0oi8HFKySGI/aky0MSsmulMJFS4lO8DAZDtEq/b0ahY9P1mxuQtjcPd227OyQ4dR7fWlH'
        'VHWa9PTW1pf2E93ZTg7pI/1RN6YPYmtV0nrWkaxFGYr6OUFQyhz4cqZ/uBqeahNUMg5wej02uocXCDeSfNG5wRn0FM7S3vnKxGOR'
        'JDyJVBUj3psLZYmxM+5JODv5B+xpgs0NtHEVM2Af1L5uPuM5qaXGHrOJjo93EGkUZyeQ0h/x2qkWj30yw2cUD3KEbsh0Kj3yBTiL'
        'WtN1K9e3jO/C5YbSOD6nimmmfwnRbYtEQgJyTvyc12Pjxls6eeS0UaZokukIZZLk3Ux8weVVyn2wKQ9Q63Jcn5fwymP1+c9/+fzR'
        '2v8AUj3iNBb5VO27d3H7VkrpyXJJJJ9TVKwxXgju36GQ3GB35qbXR96ViQgV40vPNddGjZbnI96+E3PeliT8VYsozRqWjKGPi85z'
        'ViTc0AJBipJJzRdweorgmwBiiopffvS2JhgVcjHNApDZRGasKtSSgFk4AqSueeaLsDQc0gq62k5pW0wHrV1tP9aKL2C0PY37GiIW'
        'yaWwzAqOaJhkGe9OTFNDMHyiomqVl7c18z0xMBo8mGaCmTvRTv70LO4wayRsULLtaVTKASaa3kigUnnkyTU82h8UDyokmA6qwz2I'
        'zQPUSRxaPNdmPLQLu8o5wD2z7Z5pig3EUu651CHSel7y4liaQMhQKPc+/wBKmklQ+LaejFdP6p1PdTW8qazPDbKWHywckNzxuycE'
        '45x6cV1jpU6reRRlwmWORIoKDg8ZXPc4NYDoGyhvJNK0iwDS3TRMZ1C5EbsSzlj6d8fiuyTalo/R9lFb37kiBE8TaOe4A/ckUiba'
        'dJjccU12kReG1skuNSiwb24ZYiFzhVyT/nNLby4muOZWDfgVOTWtL1eORdOeVjDP5/EjKH9PHfv370PKRg1VhS66JstuWyCnjAoa'
        '6HmzUzJg1TcSZBptiSh2xQ80uK8mkI4oOV855pM0MTCUnwO9WfM+bvSzdipByD3pNtB1Y7jnyo5q9ZKVWz8CjomolM5oXI2CKIRq'
        'BR+RzVyuPeuUg2g8uAAc1AS+xoN5T71UZsA1zmYkFyyn3qdtPhu9K2mJbirbcuzYUZrFkO6migue3NH28rMcKCftSywtGO0yH8Vo'
        'bQJGgAAFNXIoH47JQrK2BjFFG2mC7sVOwkiM3nYU3+agKbRtoXy+p3wWZe6Eyd1496WXNztzninep3MSylQaV3CxzKdwBrv9VYSw'
        'iS6uNw70ukmGTzTu4sYD6YpfLpkZY4ZgKB5kw/jaKIZSIyygFgCQD60j1q5v76yayv8ATJDa3Ch0ZCMSgEeQnsCT9RWkj07b2kNG'
        'aHZyJfTuVW3tVA+YuWBICj+kDtk5798fcUDyr0ZjxdnRo+hbGHp/pSG7vIYjqdzGoKQJ5Yyf6AT3A5JPr2pc9lH1Ta3zeKrpHLLb'
        'tcumRuAXO5SBlOTn69vSnGgSaodReKJrf+ERs6QgwDKj+jnOSOP968i1Oa2069/jmkpb2sVwWWaMbN7gjD/uFxnv71M7m7kUzcKS'
        'g/BFZWV7aWtvpy2kL3AlO5rbc+9due55yOB2xiqbiYAlTwRwQfSmN/rd7Z6189bQpFt4im4ZZV4xweRxj0rN317JPcSzyHLyOWY/'
        'UnNWY5OMaPPyJN2Tln5oaS65IJ4oWafAoCaYljzRfIL62MJZgR3oZ5RmgjM3vRGkCC51W2gu5GjgeQLIy9wPpWPJZ3UsByamoyaY'
        'dX6Xb6RrjQWMsktlKokt3k/VtPBB+oIIoO3ANKbDSCIeAKKWQgV9Y2s945jtojIwxn0Az2ya0dp0yURWu5YwxIDDfgIMZJ/7sVlN'
        'm2jJvahO+aj4SY7mnMsAYYFK54mRiCD3pamPcUUOi+9SitQ6Z96sjSMYJ70ZGU2Y7UMshyihf/D97AhsU4sNOVADkGqVCHADYJok'
        'PJDH5Tn7Up5H4Eohjbo18ozigrjUZVyoBJFffNSScSDH3quSLcu5Rk0cX+mNFS6nOkockjHpRMnUTBCACD96AlUDhhQNyF54pv8A'
        'LMplk+rXLzb95I9jVkesvjDr+1KJODVTNx3ouqN2aA6whHORXqarA39dZwnNeAc1nRGdma2G/hIyHU8Vn5+sdQu9Qmi0TRo54LGP'
        '+ap3bUkK7i5A5Y8AD/FUwg5GK1PRPS/8OnluVuS3j4BBI8i9+T7n3POOK3Fj9BnkaVHSLGLU4bUSSxQxW8zJsZGLOuUXh1IwTn2N'
        'LOu9NlfoOaG6giuwyNIXcbWJXJQBcYzyv3pqbU3Jkt/ElHdX8N8MpwDwfTjH4NXanexz6BfQ7fLaw7Zkl5ZfLwSfXuOQTRsWv04l'
        'odxqNxo1vNqpia5dc7kXBK/07h7gYGa9uG5JzXglO0Adqolcknmg7BddkJfN61Q0YPrVjMaiTQNm0VFAK8/02WUJvKsCFxkk1Jjh'
        'SxBIUZOATgfYVuOmuko/FtJtTULcuolSMuMxZ7Z9jg9ue9EpfoNfh5/DZupNCsCs5hubUESfMRlMq5JH3/S3b2NMbPpCytspdTNd'
        'OVBGDsQjH0Of96lfNY6JpW+G5eaW0/6gZYjxBkkBeMEnJAHHes/qHV8jSSizDokiBkZiDw3OMfmuU/xBPH+mlmv7bRY4YrcWvghS'
        'pbOHjIHYtnBB/wDv1z191ZeaheqsERWCA7km7ebg+X9zzWdaVrkhriVpQDlVY+VfbAohHyaLb9OtLw0cc8QXIYZpdqEyNk5FI11E'
        'L/UTVct20uR2FIUaGPZfJdDeQGq6O5fHFAIBnNExqCKxo1BcdwSwyaY29yMDzZpI8RPIOD96p3zRH9RxWdbOujTtOmRkDNfCdOcV'
        'nkvJBjJzUmuWI4OK3qwew6uZEYZpTdMMnmqfmWUYJzVMk4Y96OKo27Iyd+9VH1r1mB7Go01HNn1eioHivQa0AtaeG2jE08hRFZew'
        '5JzwPzXQunbHqTVYFNtcx6fNcbQwkAZcDOfLt5GAByfWuc21p/EOotGtykrLDObptnYBEbGfyRiu+dJP4MaSlVjaSMICx5UZH/AF'
        'b26xsBq5GD65m13pjTo7/U7u5tp4IFeK6Eb7GnCMduFyqgthO2SCPSvNa66vNSn1nRHsDYFtPt7mOIoVk2uf5gPoQpwMj3PtT3/1'
        'Mi5T4b3Go21wymGVRtEnHPBJ9zWT17VV1HRdOuZ7dWv5otk820BkC4JU47DcDj/+0tS7INqjOH0qL5NWEcV9HG0j7UVmY9gBkmlp'
        'nArCmWm6Bqd9E00ds6xBC4duNwHtmtFpvS1laQRajrl3sVSW+WjYAtjkqX7A4BOKbN1JoUMPzukq17BFcEMrtklQMAKWOWIwTkgd'
        '+MUxx0Z6JJOnIY4bC3F/tMxzMyRrtUYyVJPmBPPcgVRb6redNaxqfzM/zBi2xWUbHIxjliPXgAZPrmr9b6y0TUoLgXOoDTFaUFMe'
        'cggZHH3/AMilujWOhan411PqdzckEh2XAO7BPPfPY9qxr7YSpOkK9f1m71q9ea42qhIKRKPKuM4/zS4qc0V4IX0NfeFRxWgJOyhc'
        'riioCSRzXwi47VbEmGFECJD96gsjKeanJwaokIrHEbYdDOOMmj7eTI71ng5Vhg0ZaXW3hqVKBqY/Qg4r0oCOcUNBOrAc0SrZFJdo'
        'L0peAHkcVSyMvej8A1Ex/SijJmULXzVDZ5ps1uremKHltiM06OwHoAVj2NTXnNWvCaisZBPFMozsjwio49qvWImvvDx6V1MzuM+j'
        'bQS6ubknYRH4QbdgYLA/8V0fTdQtJb9JElBjQEKDwWT+4Cub6PbzXd1BClzBbQI4aXIJd17Yz2A5z/tVGo2l9pPUkuvXF0t5ZwuU'
        'to/FUlUJ/tOMEd+M1jarwxLdm4+Lk0urfDrUtPgkzcSIzoWOAuOVJ+uVNcl6E1K81Oyu7i5zsllSeMM2Su+JCQPpkHFay81e4vtD'
        'uNTt4XgWRJdqz8O6qrYH3GPXkg0h6C0uHT+lRbwzSXF2saARtFx6lmHPPJI+ldlioJV9nRm5vY9srC6vpRHbQtI307D810LpPpsp'
        'oJupIPlGfhrxyCyHcAMKcbRz7/XtX3TOnS21t/EbtLeKCMFVgIx4n/vHmH7VkOqer7+/u73pvRFLabE5jaaQbETa2duzHJP74xnB'
        'JpDaXg6Ebf8AQ2+LOsaTdaBLotmwjZvKsULk+YjIATgBiDnJye/Fcd6P6e1y0iV9Qv57TbwIEfczYzy55H4FbeHT7SyWOeOX5i9f'
        'JklYcrn2Poe/avJBgV2NtrZuVK1QqttIsYIyGhSZ2YuzyKCxJ/x9qNWKMMGCKCO2B2r0eY4q3ZgGmVbFPRXtHtXoXmvcYqxAO9Ni'
        'rFkljBXkV5swasyAvFecGmdTGf/Z'
    ),
    'white_throated_sparrow_01.jpg': (
        '...'
    ),
    'white_throated_sparrow_02.jpg': (
        '...'
    ),
    'white_throated_sparrow_03.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAAAwQCBQYBBwAI/8QAOhAAAgEDAgQEAwYGAQQD'
        'AAAAAQIDAAQRBSEGEjFBEyJRYRQycUJSgZGhsQcVI8HR4XIWJDPxQ5Lw/8QAGAEAAwEBAAAAAAAAAAAAAAAAAAECAwT/xAAlEQAC'
        'AgICAQQCAwAAAAAAAAAAAQIREiEDMVEiMkFhE3EEUuH/2gAMAwEAAhEDEQA/AAxsOTAqDcudzUljjBx4wqXgiTISRCfesjqTo+Qr'
        '0rkyczBOlTe1lhj5nK+2GqNpzMn9TdqEhy+gEluV3BzUAmOop8xZPyn8q4IHY4Ck/hTogSYkdK7uy+amzZON2FBuMRoQAaKDIBpu'
        'oS6RqCXVvIVIO4B2Irf36WnFmlpd2LKLtF847mvJr9nZzyZ/CrfgjVbzSdSSbzmMnDr6inDlUXi+hyhas5eh4p3hZSHU4Oe1ACd2'
        'rccZabHewjWLFBhh5wBWIuG5AcqRUTTixraByMFBqvnkLE4osrM+5ziglahuykhZxQ2FMstDZaQEbeLnavriLlFNWagLRfhmnilk'
        'yFVBuTTomylPU1zFEYYJFQxSKPhRFNDFTWgkZjNMxGlIzTMVADsL42pyNs4pGOm7dXc8qIzHsAM00NljbN0waejbIGaSt7O8DDmg'
        'dds7inpIDCiMZFbm6gHcVrGyTMmND1dvzpWaKaMF4LqQH0zXSxNTTLIQazbGuyobVdXjuORpC69smrC01nVlcFVR/bFAkiAu0JGR'
        'mrK0slSXmJ8vYVClJnRjGrG4eINWwObT1b6VaWnEF8q5Niin0JxSTSqi4FKTXRJwK1SrbZg3fwXM3EsqtzSWQPsGoDcVB9l0SZ/c'
        'Jmq2FHeQMd6tYJo1XDDBoWxPRCLimIEg6DKD7pUm4tjUeTRJM/8AGjpySyKBjc1cjT0MI8g6elNJvoVpdooG/iLPbwNF/KpVi7jF'
        'Zm94s1C/LC30lQnN1bbatBxBaCB/l2NUhUDYDFRKcumWkuxtOIQIkVtLUnG+ai+u2oXzaXnPXApM1A0s2Kix/nWhvDiXTZEbuQKV'
        'n1fhouqpHIp+1kGlWA9BQvAiJJManPtTyYqLRdY4fjgLRW0sjY22pi14p01tGa2/lnIzHfI3NUEoRRyqoA9qAapcjXQsUWVrrOnx'
        'yf8AcaUXUnqKdk1nhiVcNpUiN7Cs253qJIoXIwwRfC84YL/1Yp4wemM18j8LSuwS7mjA+8Kz+Qe1dUL1Kj8qM/oWJpDHw0FyNY5T'
        '6NU4peGQ4j/mZZ+2O9ZowQSbPGrfUUNdGs/FEkYaNwcgqehpZLwOvs2y3Ok2rBRayzN1BbYGi/zu4UqLSGK3APZcmqSC+vByeP4d'
        'wqLygFcHFPG5sZVGIJYXI7bjNNy8AkP/AM2vnZjJOXLDBzXYJi3ViT71XJucDen7SIjc1NtsZRKKPApY4ocKFzgCnoVWP60JWKzn'
        'wqKQ7DJqDNyHrtUrqcBSc7VXPK8xwuwok1E0gnIZkmLnlTeiW8OdzuahaRhR71aWFuS/Ow27VKbkOccQsMIWMEjeg3PTarCRcr6C'
        'q69YItaNUZLZHT5WS7Tfbmr0KDla3U47V5pbOfGU+9ei2L5tU+lXwPbJ5EZvjJQOU4rKuN613GAzGp96yb9ay5fczSHtAkVBhRsV'
        'BhUFASO1cfYbUUrgUCQ0yReQZNCf0o5FBkFACz0JjRZKAaBHcmuqah3rq0wGYzTcJpGM707B2oENopFHjO1RgWjNFtkUAM2uNjVl'
        'C+wqnhfBwabjmAppjF15YlwOtBecAnJ3oc8vZdzQYreWR+Yjam34FQYqZRk9K+jjw3Ko3puCBivKdqsbKzRG5yMn3qZRbNITpgtM'
        'sHchnGB6VdeCsagYqcIVVGKjeTKsZORVwiooORuQndShAd8VSXMpkf2qd9dGRioO1K5qZSsmKoNCcOPrW/0t82Ue/avP4vmFbrSi'
        'fg0HtVcLpk8nRW8VHMI+tZR+tajic/0se9ZhutTye4qHtB13Heuhe9dqUMC4J6UFomJpsjNd5aaViYiYWoUkDGrFhUCNqdCsqpLV'
        'j0NBa0f1q4Zagyj0p0hWVAs3z1qa2MlWigZ7URQKeKJyKpbKUelMQQSKRkVYADFfALmnggUiVtG33abVMbEUGGVV2phZATSoYOW2'
        'JPMtQ8GQD1pwMMVIEUqKKO0idjzOKtIxgdKVyR0NESRh9umlRLdj0Gx3FNrIE71VJKc/MaLKzNHscUWND82oRxp1yfQVXzXbz9dh'
        '6VX5YsQetGiVs9DXO5ts61BYgZc85r5aPNbuSCFNdjtZD2q0YPR22XMij1Nb3T4itqox2rJaXYyPdRrtuwrdX1hqw06GLS4c3M8g'
        'SN3TKKB8xPr/ALrXjajbZEk5UkZjidDygVWWWharfAtbWM0qjqQP816PrH8OtNu9Kkhuru6+MlQclw783Idshl6evTFeP8RcM8X8'
        'LNPc2Nw9wI4/sOWdVIIbA+gG4Od6zbcpaRolFLbG7yFrWdre4RoZUblZHUqQfTelztUOCf4i6drNuujcZS20kKQmO1uJMqyuDjLM'
        'BscE4b2GasuJdMi0G7tHjv0vtJvVBtbnmXmJOcZwcEbdRVYsltIRBrvMPWmzZA96g9l70kmhNoTeRfWgNMB3pySybsaWksX7EU9i'
        '0Ba4TuaC90n3qnLZSj0pSW1mH2f1p7J0GF1GD8wqa3sf3hVa1vMOqGomKQfZNFsKRbi9j6cwr4Xa9mFUzKw6g1EEg0ZMMS8+IHUG'
        'pLdle9U8bH1o6E0sh0XMV6D3xTaXCldiKokO1HjcgYosB9pFqKuCdqDHD05iTR1iUdBVCCxgFhvTsSArjc0kh5e9NwSqAN6aA+WB'
        'Fkzy0wkQHagTTxrvmu/FjA5QTWbxTNoNtDTRjFfIq5pX4lyPlxUtNeaW8VTGZFU5ZR3AozVClF2ajR4otOVLy9idGZlMLMV5ce6n'
        'c/oPeu6Z/FePS9dubXXYvC05XYWk7kL02br2yDTkayyt4kUQWGMjliWNyF2+8eu/qMVkuNOEG12O2tI4FjS1nee5mZyOZWAwcdO2'
        'Me1Yyt+pmkUujcap/Evhq9jcQa9p/wAniKTNjb9yfavNOKv4mjneG0ms70bHntOfmi33OSMNg46Z6mq48FWQuQ0sASPOWl5SVB6A'
        'AdCD3rMcU8C2a658fEXt1wMiOcPFzdMj7Q/44P1rXDKNNtGL09ArfSdL1jUAHSG0i8UyhiOUDO5Ge65PTfFbrgPhXT9RsZOFNV4g'
        'tJPgLpLrT7rdUKl/6kROTgkdunvtWLsOGNSucNaQXps40U+NyHBO2SvtsfrgVqyiWj29mLPUJJzhRFJEEDE9NuX++ay4OKUORuc7'
        'j4/0rklcdR2bXiizSx1i4SOBordnJhyQQy9iCNiKp2ZaHNpHEej2gi1RJYLWUkxRN51BHXDeo9BVbIzg7MRW9qHpXx5M8W9lkxWh'
        'MF9arGlkH2zQmuJR0c01MHEspFWl3jU9Kr5LyYfaoLX8q+lPNCxLPwgT0qQhTHyiqkaowO60VNXXupqlJCxY3cW6H7IpN7aPPyCv'
        'pNWRttxQ/j4idzRaFTCraR/dFFSzj9KAl9ET8wphLyL7wopBsmtpHipraJ711LmI78woq3Ef3hSpDti3xKDuKi13noKryvK1FSs8'
        'mVQz4rseuKPbscbk0qlHhOGpJjG5MMlfQnK4r5N0rtuPMRU8nk24n8B48AEsQB3Jouk63ZrI1tp2lXer3cw5eWBSeUfsPqTUf5XN'
        'rBj0mF2ja7kWJnXqqk+Yj8M17Tw3o+ncL2MeiadYsLmSMs7R4Cr0Hmfr7UuODm9Fck4w/ZmtA0i6ug4urPlkUAPHzl2U4OcnsR6A'
        '4FO65wZFq9p8NdC8iUyDzRTBOm4zj5unfNM/EJHdujajJFBCf60sfkUv15Bj7IHWrKw1KCQpcJ4pI+UtHkkkda6uPgTXqOafM07R'
        'g5OEbnTYFgS6uLuMycqusY5gWG2Ax3wcelM8LcGaNaXJm4jhudQkVwIlKpGikeYsRklgP89a9C0yVMLd3EkwClmVNip+ue3oPWnk'
        'kM6tOFCsCHUqcFl71uuCKMXyyfZ5R/EXiSC3tFTQdHvlDkgMInHIBsG9Oo222HttV3o+vaFNpenvcq5ujCpdPhnYo+N16bHr/atg'
        '7xTeJD4ZcvkMoPT07de9V2p2iSvHEtuvxCITjOCy7Htv67e1P8UbJydGan4j0/VuIn0NLSa8tniDTrLEyrBMDswz6qcHGegrzzia'
        'Ozh127tbNkKRtlVWQNhT03Feu3VlaRyTTgrG5QM/LEPPgbjOMg7fsa8u4x4beLWItYtFQARlJghBEsfZhjqVOAfrmuf+Qqqjfhad'
        'pmakHWlpetOzLjNKSiuc0YrLS0nSmpBS0gpkiz9ahRJBvUMUwOGuV2uUgOA70aM70DvRU67UwGozV3wxpUmsagIfEWCBAXnmf5Y0'
        'HU/X0Hc1UWFvNd3Mdtbo0ksjBUUdzT/8QtTttF0j/pbTLgNNkG9mjP8A5JcfLkfZXtQtbYJN6QJ4iVyKjGD6U1bkSRBhXSuG2pNC'
        'BoDRU2INTAr7G9FDDwnciixDEoNAhzzCnbeJpZkjQZZmAA9zSl0XDTN/wFo5itjrEriJiCIXPVR3I9z0B7b1pJLqS6kks4pDEFA5'
        'puY8zem9KLLDZaXbW4PiRQwqzMBs2PT26/Wqaa/mvo5pYGZUYeXBxv8AWujhikjHmk5M7eRz2GnNbxXRkDyZuGk6kE5OB261Ow1a'
        'HnjSPDoqjBAPKvYZHrVXcSxrc815IyMgwoJwGFSjv4jG09oGCBuYcgA6dDv/AKrdKjFuzSm9aaYwiaQxk45H2x7+1M3GqItmcyHl'
        'i2Zz5eUbdPvegrP3KR2+mPNe3kqyFgwYn5Tnt+dZ2x1mJ7ma2vJuUyOxjd335fLjPvTyoKPTJNTtIrmLEqL4nlKucDGepP5U/Pc8'
        '/LMAgkUhVKnoPUVhbUDUUWJWaXwmBMhHUA9P3rR2CSpFGOfEcfy5HQ9cU1IR9c6vLJJc3VtCHEPkuIOj5A3P+PWstJpfxkaGwupL'
        'FAzxyRSKCEYqSDjsD0J7gjPSrlIJo76/SQFYZtwAcZHTf6UHPganNMkTzSTLGjsdxgbAfWonDLsqMsejzDVraS1vJreVCjxsVKmq'
        'uYVsON42k1E3IhKK0aZPqdx+XlrJzDrXFJU6OlO1Yi4oDrTbrQHFACUgoZFMyLQitAgRFRIo/LXOT0pgB5aJEDmiRwsxwBV9w/pV'
        'u4mvdQ5hZ2qhpAo3ck4Cj9c+wppWILY3MfDPDz6uxK390jLb+XPhx9C3sT0z/msPpNmb++bUrxSx5sx8x6n1+lWXFV5LxPxNHBEf'
        'BQgBYo/lWNR29sVcWWnsjpGqgKuwFL3fov2oJaoY4qjLIefA2okrBExtSnNzNk0mSMxu3rRAc9aAhoqmgBqLBAq74dslvtWtrdmK'
        'ozZYg9huaorY74rRcMI7XXOjcpA5SR6Hr+maY+jXanNHcK9tayco5R8vy8nQL+lBtRCGMSEeFHgqB9qkAwkvVQAhHBAUddh+1G01'
        'Vkd5AcYLIyk9K342YzVDjNbz3TQIMnlyzEbCm10uyktY4bkh4yd1zjYH0pRPDhjPOMx43bHpXLmSVf6wQny4PL19q6E6MmN6pott'
        'dqV5fG5tuZz2A6VR8P8AANpFqklxJECGjHIpk5uVu43+pq3spS7L40zlShYrnGx6/wCKvbOWI8hQKAh6gdalpMFaFtIsoNOje05Q'
        'QMZYdz1pm2WaV5UlCCIfKBnIGaBLcxvcPsoGxzX0d0I3ZgS2RvjsatNE7GwIl2ZuZk3c/wC6rIJYEv8ABDBmHPj37fjXIZzJO7yg'
        'hd8fvS7vHJeeJli8o3z6AdRUyeikUeoSLcaXdQyj/wAXiKCR1OxGPxrBXArd34V9OumUqokVnBHQFSowPw2/OsLP1NcM3bOqPQnI'
        'KC4phxQnFJAKutQEee1MFa6qe1MASwA9aIsKjtR0TaiBN8d6okhaQq0yqTgE7nHSt9xPxBoXCXBJtOH7VNQmvogZLyfYM2OgHYA+'
        'n51nrK3tNJ03+easV8Ib28B6zkHr/wAR+vSqrQdJ1fj+6le0iDQRyHPRUi5iT+HU7UOUl6Y9scYp7ZS8C2bXJm1WYFSWKoP3371v'
        'ND0u4vrgJbQmRu/oB7ntWr0HgfTdHgtoNRvEumjGBDHF4cZ+pByfrTWv69ptlZJBBFBbhC39CLCnlH07/XfvWkePFephuT0eLXVx'
        'zvgdKijUoDk5o0bVzmg2jb0whpJGpiI0xUPW5wwrS6DhGUnIORuKzunxmSUDGa0tkhTl9qpKwuhvT4xPql3cHJFu7RKAe3/un43S'
        '1LEnJc5xjvSuhR3CxXbSJyPJOz7+nY/lRL4tzqV5SWIwTV8T0ZcvYw1x4iOhA5scw26fWow3yNdeExUs6bAHp0oEo5YPMchsg47V'
        'X26xcxZjsDt+wFb5GNF/CvNIMMFdTg7dfb6VYx3bxo6p26Yqkhv42nRVHfc//vwqwmlSMIFIORvSbGkSMvjgE5UA+bHXavrMhpGc'
        'MQWOSR+1V8E6fEsnOBvUpLrwo/DPQkgnpmhMTQ/FzeG6rJ8zErk7jNfSRvE7SgknlKgE9e9JWcvI3McEn5frVu7xNbhnAyoLb+wp'
        'uSoDHcUv8KILRZf6iQYlX/kc9PwrKTVZa45k1WedjzMwUZzvgCq5xk1xXezqSpC7CoMtMclfGOgKEylfDbtWp0fg3W9TVZRbC1t2'
        'GfHuT4a4HUjO5/AVrNM4T4W0dfG1O4XVJ0ySCeSBT226n8fyrSPHJkOSR53o+lalqsojsLR5cnHP0UfUnatSeG9G0PSbjU+IdRaV'
        '4l8sFt8pb0LdSPpjvVrxJxfL/LGt9JhEdsFIVY4wqIfboDt29q8M441jUvBFoblpEdc4UdGyc5HrgmqlUOtjisnsdurzUOPeKEg8'
        'eOC3LYUkgJDGOn+vrXtHCt/p2k6Quh6bbiGGEcwYDzOe7se5Nfn3hC5ktbSRfkeRs8wbzDHb861vD95fyawsrMyIcg839qz43i/s'
        'ufg2/HvEzWgXl6kbD3/vWNtIdb1iXmgVooWOfEk2H4ev5Vqb6yjuPDklQSMpyCwzVnYQ8qjatJXJkpnlKGjIaVVqNG1YljSHemou'
        'lKR07bozEAd6BF9oEQY8xq9nIihLelUGmCSBcNt6UTUr5mTwwa1TqJL7NLouoi+i+DPkkVD5gfn3GPxxmiyqZIUboBtjHSsPDOyO'
        'GViD2INafS9W8dBFcYyF2Ydz71ClQONocnZoYeYjK4Ib/NUwk5WcdebtVpeSxiB2eTkjI5iSaz63EM1stxayrKjZwwPcHBqnMlRH'
        '7dzHP4mwJG9PyagCVZ+mMYrONdcgwOmKUur+QsCOhFLMpRNJb3SNdK0fmySRnp75qeo3aziKJXJKsCxH7V55ecVQ6frNrpsscgEs'
        'iieVf/iRj1/vVzwZrNprYvZYjypb3BSHOcyL97Pc9/ypqeiXA3cMmRGn2iM49KX4g1PGmvBBIFkYhSR93uPy/ekL7UktiyW5VpSM'
        'M+dl9hSVhYyXyvKJAQu5ycZ/Ok3KWkOMUuxFyWJJOTTGnaXf6jKI7K0mnYnHkXP69BV9BaaVZtCGVZ5Gxz84JA+mP8VaXvFENhAL'
        'e2ZYY1+Qc3KM9dv/AFVLi/sxuXgX0vgXk5Zddv47KPHMY4/O5Hff5Qfzq8t5uGuHpUFnpaGUoT487iWQe47D8MVjNQ4n8IkSTHmk'
        'AK9wN/zz+lZLWOKvE2iPN5s4wM/jmryhDomm+zecRcXXEjhZpRyklQCQfLjrt0O+Kx+scVrbgBC0wYgcvP8AKO+M1m5ptUv+YAMs'
        'LdC5xgVxNLiG9wxlbrjoKzlyt9FKKXZzUOIZry7JhgJITAA82T6j0qtutPv9SlR7lxEoHrlqu1RI15EQKvoBXRUbfYX4M/ecPScs'
        'UtjeSLPE4fztgPj6dK1unFnvI3IwxbLY/WlVFWmiQF5wcdDTQrZv4rMSW0Zx9miwwcoxintMQNZRewxXZkCZrocdCizwaOMt0piO'
        'EjqcUBJwh7Yo63KnoM1z0jVpjMQC9BVzoMBlm5iNhVHC7Odhitnw7beHbKxG53pwVshj0kMSxZI6Cs1fSBpjy7Cr3WpxFAQDuazD'
        'HJyarke6EkTDb07ZylWBB6VXimLYO8gSNGZj0AGTWUjSGmA/iLqzwcNBVZkkMmVKnuB/usHpmo32h22n3ds4IkjJeJt1cF36ittx'
        'not497bSX9sY9Mt4+eRpGALuQSAq5yd+XNZyGG5vtKurKWGMG2AmtXi8pjUHDD3G4NCi62aKm2bTh7VNK1uMCSGWyn5SwjbcNj7p'
        '7inb7U9F0xctFE0g6I45yx7YArz19Ju4oPjoLmYyKATEGPmGNx7fSiatNbw2cUqY8csCcnvjYA1WHkz1ZV35vNSk1/U5vDmumniU'
        'BBjp8wA9ABitXwxac+mpPbytBaybmNRgZ6N032xWb0tns1nuZ5JPh43jMoQeXmIPXO350Cz1WSzu5HMqiCVjJCq58vqP/VVGlsqb'
        'TWj0uMWcEDMuJycFCh6n69fTambjV7VIlVRyYiAC5ywP9/2rCWWoanfr/wBtZyFegYkqv596eg0W9mUfGXKoPux7/qaf5F8GWLHr'
        '3iERzu1vPJEWzz+bIIIxgfXHWqw317enEUMrrvgk9M9dzVtaaLY25LCHnc/akPMadMYAwBgVm5Nl0jOw6VcSEG5n5FAwFQ5OPrTs'
        'FjbW4/pxDPqdzViy0J1pUDANQXphhQZBQQLt1r5a69cWmAaJckCtVw5abKSOtZ/TITLMoxnet7odryqu1XBWyWaLS0xbqvpXNUIQ'
        '/hR7QcgApPX3wgaulr0krs8EazuYtpIXH4VK3trh2PgxPIR1CjJq6/ncVnAbvVmC23blXLOfYenvVzpk761ZFbO2l0+GRd2aPlLr'
        '6564rmUU+joTfyU9jp1xFcwrcJy8/vnHqDjv7VurSIRwDHpSXD3Bun2d0rSagiK7Asqtlt+/mNekWsegaXBEyacbsn5nlPOfy6Vt'
        'x8dLZlPvR5fd2Wo6re/D6fY3N0w7RRlv2qytP4e6uEEuqXNnpkfdZZQ8n/0XO/1xWu17i28ghMVvD4dsDlQi8mPTpmsJrXEbSrgT'
        'eYkkgliOnQE7VLhFPY7L2DQOFdMRZLx7jUWcZj5nESH12G/60k/EdvbzfD6faW1pESF/poBze+ep/OsHqOtjmZvHL4XATJA+p+tU'
        '82ukcxiiLMN8A82PypOaXWho0HG9+90/KIudlJZD/r8cVlE1aW2ee6tkCgqVbsSv2v2qpvuJDcysCzFt+Uj9e1AhlnuLiK3jkVjM'
        '4Hl3rJzZSNFPqqqI8M/iYGwbf8KSubbUNTHLFbSYBBy55VH51qdN0u3toFSOJPEQlXfG5IPXPvVnHb+1TbYGHseF7yXx7G7vkjWS'
        'NZW8JeY55sAZP0rRaZp0Gl6ctpqMktzY24za/wBJWKyk7ggdcg9far2K0US+JyDn5eXPtTkcRAwBsadiqxOOHYYGBRVipxYsdqn4'
        'VIqhIx1Fkp8xUN46AK90oEiYqxeI+lAliNFiK6RaWlGKsJIzvtSk0ZoJYi9ThQswArroc0/pFsZJBkd6BF3w5Y7qxG9bnT4AiDaq'
        'rQ7QKinFaGMBFrr440jNnJGCECqziKT/ALIt6Ue7l89I61l9PkA+7VvoXR4ZxFrM/EH8RxfTWkb2MLqUh5AIyo2A5RgYz+Fejajx'
        'KqhmMMY54x/UYFSNt/b8RXmGoa1HPdCUEsoHKq4xgenriu/E6xqTJyKFQjlV36Y9q5VOjdWa1taVnWV35s52Of719/1Pcw5+Gup0'
        '9gxwKRsuFFSNZry/klPUpGvKPzO9Tmis7eTw4bdVx3Y8x/WhyaRpafZZW/G+tMnhNCt0p7ld/wA6SmuZri58ae2RUO5RXzQGJIyD'
        'XY37ZpZN9kOK+BmG30a6LPc2+D6Oxx+ldVLaEFbYIqEYIUYBFDGGGCK+VAvQYoZIK20vSIgSml2wY53IJ603p+m6TbkPb6bFHOM4'
        'lDH9qimxp60GWxWLR0Q2hyKLK5xvTcEO3Su20flpyCPfFWZyWyCQe1GWHHamY480URe1IKExF7VIRe1OLD7URYfaih0IiD2rjW/t'
        'VmIPaumD2pgUz2/tS8tvjtV81v7UCSD2oEZ2a39qTng9q0c1uPSkZ7b2oJoz7W+WG1aDQrLHKcUO2sy8o2rSaba8ijargrZLLHT4'
        '+RBTE8vKOtRXyJSNzKSTvXUnSMwdzNl/xot0vPZsPVarZZRzZJq1Rg9su2QRRF2TJH//2Q=='
    ),
    'white_throated_sparrow_04.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgIDAQEAAAAAAAAAAAAAAwQCBQEGBwAI/8QAOhAAAQQBBAEDAgUCAwYH'
        'AAAAAQACAxEEBRIhMQYTQVEiYQcUMnGBQpEVobEIFiNi0eEkJTNSgpTB/8QAGQEAAwEBAQAAAAAAAAAAAAAAAQIDAAQF/8QAIhEA'
        'AwEAAwEAAQUBAAAAAAAAAAECEQMSITFBBBMyUWEi/9oADAMBAAIRAxEAPwC3lmH2QfzHwUpK4ni0IEgrx3+iQ7ZZRzWeUbeD7qsZ'
        'IUxG8/KaP0qTJNMLM0OCTYS2Qm063lBlj7oLsiFICbJAaNpuOahSp3PLHgcpmOawEd9MEyHb38pacBotFe60KT6mpGtDP0We9R3K'
        'b4yolhARUlTO76VBslOWD0l5DRtXmTNlrA8gWjWXDtIY0oc0N90cPI66VsEA5wIshJtyXH6SU9MS5pCRlh2uBA7SUFMyXkjhAp7j'
        '9SbhhJFoj4AW8dqNm0UYPZOwM3UlhC5ruSnsUEVa56ZjzoSfYrDsbhPsaCOViQCqSdxNKuSAUkp4eOlcyMv2SssY+E+6LpQZMAol'
        'V0sdOWwZTAAeOFUZTadY6Q64Y25wNrBbYRH8FQLgAukthBnBRmmilZHEO3IkT7WwA/GeF53SHEeEU89JKeCYJ5LA6+KKWhkIdtVp'
        '+XdJ7ILsPa7cAprWwpGYG337qb4qUKcwUstk3cE0rzPgcMemKQZmUE1YpLZDhaqkETkaRylpek5IeKSktG0yRtF4piH8Kzg3yCx0'
        'kI8e+QrDFuOgm0XCex18hYcz6gPZMgFwWNh91JmIRtARPTtYb+sNrtPRY5q+1Nps2ldNA691KUbRSefG4djhLvi5scLl5U0HSTCa'
        'RWttCi7oppgAC50ybF5GAA8JLIYBaspapV+Ue1aQFZlCwqfKabKupz2q3KaCCV0yvBjZ3NJb0lZfpK2OXGjINhVmVjAEghW6ldKi'
        'QrMBN0jvhaDSnHE0LYYNC0lOwxg0O0qw0K9keGQBBxoGWAaA3gIEhaOKU2yjak8yVrbcEFGAIzke6SkIBsIU+WL7Q97pGnYC490B'
        'aopChozjb2l5JgTyljNbab2lzId1Wi3hhh8luQnOs9oHq/UeVlpspewB/GIIpHPHKRjftHaxJkO6BR7GLaCS+Cm2NBaOFT4Ehc8W'
        'bV1HJTfst9AwuNjBztxCdjZtNIOJM3olN7oyLvlbAGZIWOj5+FUywuDzXSbyMksBaCgskDuSQpXKYoEQO7rleBc3hwVlA5m3kWvT'
        'YYmFt4Kh+zoCpkf8pXIIcFZZenyNZd2qjJDoyWuCPXDFblW1x+ElK6wrLIaHgqtmjLSnSYx0p5AaSqrKcC4hNudKYrD45B8McCVV'
        'ZU9E8FdSelqioeUsA5NByE2SvdAlnJPKA6UrYIPOnoosUtntVJl+6Zx5Ppu02GLR8pDOCq3MyjVFekyHDlVedkFy3hjz5RutzuF0'
        'f/Z8ysKXyrIxMuONzZITtLxa5LI57nWSSFt34U5r9O8ywshkgY4u2gnrlBV6Yx53iw6Z5lqeJj0IWzkx10Gnn/8AVQ5MrWjjsroX'
        '44aY2PzA5kcscoyYmveWdB3wVzuSIuNdUufl2Wxl8IQknkplpCA1pj4IXnSc0FBW2bA7n0EPklRaC5GiYb6TugPwLjOcw8KxgyiK'
        'BSkcX2RDGRzSWefPGTbLSOYEdosU7ge7Cp45C0gVwrTGAc1dE2qXgU9CyyBw55Som2PrdwmZIuOFWZzHMFpL89FaLvCyG1RcrjGe'
        '0tu1oMOY+KQWeFdYerCtrnUtFpgNmyC1zKCps+GN7SK5tSj1GOv/AFAb+6XmyGvksHhM1phGbB4sJCfEcQaC2Cw9vaUnBB4bYTKT'
        'EJYnY7vzGLTHtB3D2c33CLjZOgahi7v8YbDle8bxwP5RZgKIIsHhc8z/AATFfr35+POmZj7t7oAez+/wujj6z9Re7q/ybLKQ15bY'
        'NGrHulpnkA0E2Az46XjG1/sgK2VTnG7TeFMT9JRnYjT2OFnHxNjqHSSqwRsMY9wVflQfWRSvYMOR4pqI/R5XGz3+yg6Yio1YwAdB'
        'N6aHw5Ec8d7o3Bw/hXM2kDHa2bIlgx4nmmyTytjaT9i4i/4tWGLobDG3IjmbkY5APq4zTK2vcggUaPBq/wCVlaOiIqviOpeW4kPm'
        'P4Swati44bl4bQ921tWB+offjn+FxNmDPM8Nx8WeVxIA2RkgWLFnofyujf7wa74x4jl4njpZqD2FhdFNjksc1xpwHR6Ko4vxJ8gx'
        'M7Ix/IPHX+pDIBK7CaBG1u0Frau/7dWjyV3/AIlo4Wv5Gmanhyae8f49h6lpULnBrciXDLoz/wDIGvuPlTyfHMsYP+I6fkYuq4Aa'
        'HPnw3F3pg/8AvaQHN/kV91s/kv4yaPPojocvE/JOLnN/LZcZcw1/zVV1R91zLRtZ1+PU2674Pit/MsH/AIrCLmiKWHkltdURXSly'
        'Vx8ed/P9B07fxLqCJ1CwU7BFZHCZgmZqeTk5MGnS4mM8smx9w+l0UrdzS0+9cg/smYscB3KW5whQOOA0puiodKwihAaFh8bT8Lkp'
        'Nsiyo/LkusBP4cZDRYU/TaCmcVrKors4V4MiJbwkM9hc0ilZzADpLHa59FXc6Ma7LiPJ4CiYJQ0208fZbXDiRnmhSY/KRVWwILiQ'
        'Opo1yxnshMwZrgKcbWxZmlxSWA2lWP0jYeLTdMNgXHyhI2rpO45aXDd9Sr2QOiPuixyOY7tMjYOZBoqvyog4E3SNNNZJSWVkU0/K'
        'qOJytew2CSE1jGwLS0E5cS1w4TMF7qaDSWmBjvob2+yLjYg4JU8bltHtSyMrH0/GOVmTMhhb057g0E/AvtRppfRerp4i6wYMeCD1'
        'siRkTAQNzyAL9hyl8nWIpm1ihsMDjTMhobI6Wjz6QJDaHu91tB+VqGq6nFnTx5GQ/H9IxXBFLPG0Mae3U/kFw4Ba0muitG8n1yMZ'
        'v5/Mgyp4Y6EJc+2Tv9trRwGD2tQq+3iO3h/TTH/VnYDpuonAZPi6ng4+RkSbpMjJnM2Q1vw2UODWH2+hgH7qvZgR42czIbk6O50I'
        'LjkMhY14oGhu28n3sgE8cri8vk+dlZjG5Mz5cl77ZjsfTGj2BW/eL+J6/miN+rSQ6RhiyxwAmcSfsD/mVlDOl3B0zRHhssOW+XKM'
        'OQCdke0l3PLiC0CyD3SX8h1DMm12bBdMybDEe6GAtZuYAP6nAfda5Ngnx7VoMXFlz80vj3CVjvRFk++40AP80n5dDrmmwjMhY4Fo'
        'Mrp3H1BQFjskfFhU441keS/Ci8i07K8hL9GxML860PEzsb0w4A9B3sQfuD12tWzfFn6LmiJ87WZEhBdDEXSNa35Ib8/2+Fu2nef5'
        'ehRPymaez1xF6L5IWbS9x53Dnk8dH5tO/gz5N4qzKz9T8j1KKDNysgunbKKpvFAcf6Lr6KljOR1j8Nh8T8jkb4lFpWsaZJNjMd6W'
        'LHDg+g+BpbuEoJcBtvse5Q/6zzYvv5Vh5PrcOva9lZ+MzZjSPrHaW7SI2gNbx+wtLRRWLIUeROhGSjstpQkBBRXChwgPLia5XK+P'
        '0RoHRLu0Ww1vBUPTJ+QsEFo+pVhYEjLM72ShncH1SLKRzzSEz0wbJtUdJDodx8tzQB2mn5Tgy6pVTns7B6RoswA7S3cP2S9xsLWC'
        'eNzLc7lRnc136aSjnxPbYFKD3GOIlp7RXIbqSlLaINKnyJ9ryOFObKdvIcVWZUhLybTOvDdS6k03PJDS7GYTzTp2g/6qP+CZMgO7'
        'LwW13c3/AGVE7Y197WON+8QUm5DWjjGx/wD67UK5KKJcf5LiDR2RtdJPqWJFG0W512K/yT+I3x+EMfJqORPuNARRBoP3snpapOZs'
        'xjo3vEcZ72RMaf8AIKs8ixYsSATaXLqWVlbAN74w1kR45/6KDu39KSuH8G8695j49pbH42j4LMjPHAfkvpkfHZ7s+9Af2XOMrU5J'
        'dYh1DV9QblbJQ8sjiHIuyAXWQP2pUMLGAuGdrePhvPLtzXF5/mig5OPorSP/ADeDJvmy51f2IUG236zpVxC8QDyybLydazc2GMfl'
        'ZJHSb2zGWNgvgyzf1O+wP7I3j/kM0GKI5AJ9O9Vsjo5wQ3IcOAaHIb31yf7r2Nuklji03GdqIaTtjfG6SGz8sIorYdN8P1HNAn16'
        'DCjjFh3p7onuvpo28AD4FX8qy3+iDrfRbxPG8b1TyRuXj4udDkGzKG1Ix77uhQpjQOK5/ddSGu6bgtdjZMUpjaKc/wDUL+L5VLia'
        'doujaa2HHhETnC6k+oCv9EtkZkId6PoNyGfrG32+6opEdYWHkEmnZ8kUM8Oa/Ejbvmax4DXA8NDubPPsmYBoEb49Nh/NyNhiJEbJ'
        'XbIzyadzxfHC1eJ4xix+JM2LKyX36V257Qbo/CutM14HT8h+NBHPmzO2ZDnNLWs+B9z/AKrolEWyxyMTHzml78KPEY+LcGFw3OBP'
        'HHzf9kvF49pUGJO7NwoHOhjDo/pBJfxVn5Fj2pLZ+pjHyN2oxtkn2iaRxOzYG8Mbz8n+Sjz5gm0+IY8gdBK4y3ZO6/gn2/6WmfJ5'
        'guDGnuLXC/ZXuPMCKctZw5SHdq0gnUkwFuGNebtDnAaeAoY77AN8IzvqRAxcNeUKdjvcpw0AhSAO6SgK18I55JUPQ45BVl6W7hEb'
        'iWFHkrDJlUyMeyJ6IAFVacfjiM9KD2tr7qP7iZRMgxpaz5QMmXaDxwjSybBRKrcuW+AUit6UAOHqyFQfigjtFjjaGbwaKgyVzH24'
        '2F0K1+Q5ohGC5GZBZ6R4IeAmWRfZF3pEFiwUeAmTHX3HuCiQsoIn2KX8CNf0a1qn4e4GvzyZeP6sGTVOMbuP7FawzxWTSs9+Nm+l'
        'kPZzta2iR7WuoYeVJiSiWEgOHsej+6LrEGLrkUckbmw5TeHsAou/Y+6RWv79KcdZ4yp0HGOlwBmPiMOQ9oIgY4UB8klC1nWMXGy4'
        '4nRerldll3X7JvMhZhRtDcaN8xFb5H2WD+CtffiY8Dpch0skkpH1HbfJ+CqrPpfuGycnEzWtGY5zZDy4N52/91jFbi0TBBIyNxob'
        'hyQh6VgCMBzmbHP5LXdD7lW8LKf2CGmm8cfunVkmIt0luRLJmSY4i2sMcYYacR837IeJjuwg3Cxw2Iho9SQjnd/y/JVzIzcygdxH'
        'N3wEzpWlTyxS5TIXOx4T/wASWvpBP+p+yz5G34KVkekYT8Fz89kr5RL6kb91m2+7+fq5qh19kQxh8TGuo7Who4rgCk5mfW6mghje'
        'Gg9qELCSBSM/6ZsrzA9hsAosD3g0Vbflwa4CicQB10nQoOKZzWjlWMEhLflBGMA0IkQ2/SVmzE3OLlmMKDzTlKN3Kh39FaDsajho'
        'UGEUpByjyPUDCGQwEfKVfBxacc+u0GZ9ArkwafCpzGcfKpssODrFq7msl1jhV+VFfsjFOX6VTFcXIG3Y9MysiLLaUjJG5r6AUJC9'
        'lV0umK0dFtA0VyjNASkMn3TAf9KrnpFhGHled0hbj2veoPdF/ApEJXEFYgmLH7gaIPBQsl/ul2S2Vx8n0dSi31FukavCItRxLd0J'
        'ATf7qlGhw4Di7By5XQk/pcd3P7FHEn3TWDg6hqTyzAw58lw7ETSaRi6bwdLCukx53R2HCVx4Fdn+FLNx9R07SHajlQuDA4Mjja4b'
        '5XE1TQuv+L/hjHmYr8jyfLxsfJxabKzCm/4gFAj1q4Dq7r97VH+IU/jmiapiaTprIM7EzGb34kszn1Tdwkaf1MIoiwR2u79nzWNP'
        '7dPF9Nb0DAzHviwtW0HUsPNyWb4YZaG9h6e09Oo0HN7G61uflGq4eZoGmadopjiwooA6SKNm0GT3J+TYK495F+Imdky4GPpuoZ8j'
        'NOym5OG/Im9R7NzQBG15+pwvjk9ArcfGYX4GSdMyJTtzYzlY+7ipTzJGPgG9wH7p5lJtIXkcuV5jITs465WMSOym9Ric23NCBgHk'
        'lZERtjKCkWg8lTB46USOU5sM7RSBK3a8FMNNBLZD+UtGMSEEWssKECXBZY4gri5HgMG2lStBY5TJUOwDzncWgSmwpyOFJTIk2NKy'
        'QUCnka0G0mZQ7tQmkMhKEOPlFzoyCSAHlKZLdwoJgOB7Kw4AhUicGMxNrhFohSoNI4RA2wupi4CBKi5Ee2kJ13tHJPQStaZC07h0'
        'Ut+kp/LwsqHHfkTY0sULBbpJGFrWj7k8LHjcOn61rOLpeHPLqOXkSBghw221vPb5SNrQBye1CuN08Q6eDfi2hal5DqLMPTsd0nI9'
        'WQnayJvy5x4C6VrflPj34d+OzYmlabAMgsDZZ3yySeu7lpEbqFu3fSQNvdi1t2fHpv4Y+CwsYI5HAPExkdUUzywk7/b2oWK/a186'
        '5fl+v65nRyYmNjaHp0b3Fn5SMMlePlrtp2X39NWuzi4p4lr+iNui+8l8u8vycXCzgx+izjDdDO18gblzsdXPpk7trR0XUbqj2tKb'
        'i4sWZi5uHu/MtkazcS5xcz+oG/arVh6EZlkmDbkkNveeXOP3J5KxmZUem6bkZs3O1pbGCe3npB33Y8/8+mq6Xh4ud542BvpiKCTe'
        'GsbtAa3oAfC33zfU34emYksPqOy/zcYxvTNP3X7fxa0z8OsIME+qyW6SZxa1x+Pdb6IsTMdBJkxMkfA/fGXf0u+QjLxiMvMhwmjj'
        'ma4ubNGJAT3yOb+92lIo9hJC9C4MjLGuO3cXAE9X3SPAAbJTgMscpnpRLdp46WWgl1IhIl1JeblyYnaQgAXdpWZoiwLLhRtZto91'
        '40R2uTknUDDAfSkJLSzyQaUS8j3XI1gQkshCTypC4UiFxNqHp27lOvTCojKg6N3sni0N5pQ3MJ2ilaJGXpWSbmOURN3dhP5MQPIC'
        'Qljp3SqpDhabLU2W1vQJHseAVMAArL2+4ToQWklyhM50UOAI+abLG+Q/bneB/kq2bP8AIZd2Nj5uRCdhZI6GJmJCRuv+j63HjiiO'
        'D2rcBCkbRR7NI2Gman4tnZ+r4s+VrGTl4jXF+TFPK9243Y2gnr9yvof/AGeNKxcDTPIs+TS42ZmnznFja4bW02JkhaOODbgDx7Ba'
        'z+EnjQ1/yW5mu/LYrfUeWgE3dN7/AL/wrjyrH1fwjD1nSm6j+bxdTy35BlMtzbpGm7Bo2C2rFgghGV53r8G/xGk+eeZxeeZcz4dN'
        'nwsE7fUhldfqSDk3XBokjjg8fstf9K644CbigZHE1kbWsY3gNAoBZ9PkrlfL3rRsFo4XOeGtBJPAC1Pz2b89q2NoOJIXhjgJK6s9'
        'rbdezIdG0ObNleGTPaW44J5v5Wq+C6a4Mk1bKt0sx+gu7r5XRC8A3+DYsLGZiY8ePEA1kbQAAnY3EDgoNojDwqOQFhjSlwFqzxn8'
        'KhheWuVvhvBATSAecOLWYKtYPICyxtchNQUZyKSZ/UjZD6NIMYBNrlu8KJA5RSi20yWAqIZfSn20GCsrCRY7SUj3A82rWSMtF8qv'
        'youbClc/kGHoPqoplrD8JbGBulZwtG3mkqYMK7LBa08KuJohwKuM4NLSFVFu0OHHa6uNaMkHa4SRc9pOVn1FTZJTdt8LxIIJVuuD'
        'JFi7kLzeQotPHKy0+6mmSMHgrBFqZB7Pv0jYOOcnOgx2izJI1n9yiFHYfwy0bMwPB25GLPNp+VludIclscbmsAH072uFlv7EHn2X'
        'E9Q1/I8n1KbV8zIdPM1zsUuIofQ4g0Oa5v8AgBfR3mObJoXg+RjNgfCyPC2QztcNodVVXYPv8cL5P8VgzdNZLhZUTp4J8iaaHKA+'
        'pri422QfBqw754TfqPI6o0fdZsDvpjL9pdQuh7rOnuxcvDlymSPDY2F7mObRFfJPCzEHPcGNFl3S1Tz/AF5r2M8d0V3qOkdWQ5v9'
        'R+P2C5eCU/qHvwpMzLn828ibG8GPHx+AAeNoW8NhZDCyFgAawUAq/wAX0jH0jADWsb67xcjvv8KykNq036IBI+F5p9l5xvheA46V'
        '0xdCRO5pWuE+qVMBTuFYYrzwmRi+ieC1Eb1ar4ZCByno3BzUKYyBTtB5Qm8BTmeASEFzuOCuDkr0ogu7kLD3U7jpLucQs77ACEvU'
        'MMOkBbRSkwBNHpEF+6k6in+oRi0f0uTsbm7ewlnj7dKG6j2ub4w5obIjafdV+QxoceUSWeuNyrsrJo/qXZw/DYDloO4WGOsFJzZQ'
        '3H7qUMhpdLCi/byPupCr4Q2PBCw/LxcZhdml0MQ7lbyB+4UUiIKTAkimdkafmzYkjiC5jadG892WGxf3Cl4w/wAmy/MtM06V2DDI'
        '53qMmxYzcm0ixtPAdyPeis6FnwaxkT4+DI2aWN5Aa3tw9iPlbV4BHL/vhhRBz4JS8tDttlpruirynvoDon4tP1GDw0YmVM/La6Rh'
        'fJIwCRvuDTaFXwfjhcXe228Bds/HiSSDx7TcYyOlL5S1z3VZodmlwjyvU8fQdGOZNK0ZD+IYj277pOWFbGh4it8z8iHjeH6MIY7N'
        'yGHae9g/6qh8E0qVsDtUzY/+PMSWbhyB8pXxfSMnW9QOu6tuczdcMbvf7/st79MbaAoKN+LqjffQQ5XiFPYbUwxJPgui+3lS28Ix'
        'jUXCgrJhAEUU3iCygEApjGG3pVTMPxik5Bw1JRutMMcQEKCgeY6jYQGPJKNkAvQ2s2hcVx6Ppgus0psYaUeAb6Ug/ngpEmmHSZHF'
        'FeBBQ5XENsIcMp3USrKfADLgCPgpScd9Iz37UpPKL47UXHoUxPKdx2qbNkNkAq5ySHNJHaqZYi5/S6eGcM2JNa9xCcg3tFEI8GN1'
        'wmRjCl0EneD0VUiOiZMwskaHtPYIS8RNpyD9lJmF3aZi+oyWOP0JWfoki+lzf5C23wXN1eTyTScOaeKeNk30zOYBK0V7u9wqJv7J'
        '7SZDBmwTGf8AL1I1okPsTwE/HT0x0H8fptC0Dx2DV5ck+j+a2yMEpeDKWmvc10V8wwxZnnXk51DJDhp8JptdUOgF2z8U/AoszwaR'
        'sU7nb81k2WXxOYZCTXqMBJ+nmuCtD0nQ5/Ho/S015mxP6oX9j7gqt/QfCyigZDG2OJoaxooAewUttBEiImAewGj2CKIRfT+yhSQW'
        'Ltb8he22eEYsI9issZ9lLBcBiPjlBmYB0U717JbIbzxaaRkhQtpHh4UHDkI8bOuFdDYHiTTBwl4288AplrTXuszI8QKQJD8JgtNd'
        'FQMd+ylU6MKSEkKLXBp7TMsXBoKvnDx0Es8YGFll9rWGdWErG1zjz2noY+OQbRqTIg6S20lH2TasjAD/AELwxQR+gf2SzOhwqHjh'
        'BMdmwFeHFFfpURjN9m/5K0zgGVsLa7CMWjanHY7D22il5oXgHbyExGkf/9k='
    ),
    'white_throated_sparrow_05.jpg': (
        '...'
        'AGxJMCY74CoysFn/2Q=='
    ),
    'white_throated_sparrow_06.jpg': (
        '...'
    ),
    'white_throated_sparrow_07.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABAUDBgcCCAEA/8QAPRAAAQMDAwIFAQYEBQQC'
        'AwAAAQIDBAAFEQYSITFBBxMiUWFxFDJCgZGhCCNSwRVigrHRJDPh8BZDU3Lx/8QAGgEAAwEBAQEAAAAAAAAAAAAAAgMEAQUABv/E'
        'ACoRAAICAQQCAQMEAwEAAAAAAAABAgMRBBIhMRNBIgVRYRQycYEjkbHR/9oADAMBAAIRAxEAPwDIXFfy8Ch0oK3MAEn2FdqzgACr'
        'DpGKlchKyBn3PaufOzasgL5PCOrDpyW+gPvIUhHYEcmmVxWbc3txwPmrkuTGixAkkdKol9mNSZKkpwoVGrHYy1wVUe+Su3G5OPPE'
        'jIx7UOZK3BhRP61NNZ2nceK5ZZSpOSKLAlZkDJZSpXIqGRBAG4HFFuHyl8civxc80bUijgsmSikKGo6luYAJ5p/arcBhSxUtviJT'
        '6lCmYISABxT0khaRM0ENAACpFvBKDg80It5Cec5I7VCHC4v4o0EFBxSlUQhWcJFCAhIoiKk5z1rGzwRt4oSZ0+KOIO3mgpmSMYry'
        'eDWxDNI345oP7M2tfNN3IpWrvXJhKbIO3I96124QmViQAmKlPCTRTSkspz1NfnyGh6hilUubhRCQTUsrNwnyZGrs4bcA0skyHVrB'
        'CuKDQ+pRyo/kK4L2VcGhTPZLfp5xOU561olgWSlOKyewurU+lKT3rV9MoIbSVGo71gjuiXK2jIFP4qRikVuUBinkVXArmyyc23gP'
        'SABXDhFfQrioH18UmRMzyeSCrA96sVocRGYCgrCu9VLziFA+1Su3BwN7UmvqpR3H1kHjksl0uzjxLaV/Gc0rjueU4VE5J96VNSVb'
        'Tk8+9cuyV9jQKvAxzb7GslQdVuUc0O4+EcDihWn8I5OTXwZfcAA4r2OcDE+MhbKS6eRwaYR4baBu6VDDSG080St0beOBTYxwefJ0'
        'taWxgUK7MOdqailOKOcVzEiPLOfLUon2FGLbwSNuFauTRrOBUZgSUeryHB/pNdNpWFhK0Ee+aGTaQLkkHsNKc5AplEiO5A2mo7ao'
        'BAAFNmnUoTycVLK15Eyt+xAqE8BynIoN6Gcmmr85tKOVChS8l05SaDysHMsdixCGwdqsZFTFlBQSO9FNwvNd3EDFSy4m1klI7Vu/'
        'Il5zllRvbSUoPaqy40VKP1qx3kqUSnkEUn2HqeK2PIUGmwB9opHegiFBfNO3Skj7uSKAMdTjg47+1NwhzaRZtERgt1K1c1q9oQEo'
        'TjgVmmkP+nKQrkVpVudSWgcgGpbo5QuyCcSyw1AAU6iuAY5qrw5A6E03jSOBzXKnHBx748j5LoI60NKc460OmRx1oeU+Ck81O0Sb'
        'TymVgYya4U5k1EsEVz+IV9afVIJBr6eR7Vy0MkAZJNGsQ1qwVCtCSyQMMrcUPam0VoNpzivjaEtD2r85IGMA1iiGkTp3OOBKQcmt'
        '28J/AaZdUt3TWXmwYCkhTUZCwHXQRkEn8I/es78CLANS+I9rhLI8ptzz1j3CPVj88V6P/iH1tI0pppFvgOFqfNyhKgc7Ed1fB5p1'
        'aTy36NbwhXMi+CmgHi2YVudmI5w6kyXB8HdkA0C545aNgLU3bNMbgDxtaQgH9K83LDsl5TjrilrUSVKJySaKYaCU5xS5arHERLmj'
        '0A9/EPbwkbtLpKc91J4/avkjxh0Ffz5d50u2W9pxvYQv1e/T6158WhSycJyPpX4BDaCSrB9hSXq7M8MT5n9j0GiH4HXxpHlPO2qQ'
        '5wS0sowT8EEYoO9+Bz8lkydK6jiXBBGUtPHYsj4UMg/tWAuzw3yOcU105q+9WuQHbdc5McjslZx+lZ5oy/fFGboS7WP4HWpNFaks'
        'DxRebXKjgHAcKcoP0UOKBt8VxSgOiR1JrTNH+M1/ba+z3tlm6xvxh1IyR89jVkj2nQOuS4bS9/g1yXyGUowjPyk8Y+mK94q5/sf+'
        'xkas8p5MqaQllI9qCuspKUYQetWrxE0TqPSLCn5MQyoX4ZcbK2/9XdP51lsucpQOVEk0i2mUHyjJxTRxcVNqJzik76UbTtziuZ0l'
        'W/OaAdlHBBNFGDQpQwdp2leD70Y2yjg0qQvKs01hguAYNa8nn2PLOnaoEdKtcSQUoHNVa2goIGaesq2pHNY1kavkh/Dl+oc808iS'
        'xtHNUhLpbXkE4plGnekc1DdTyc6+t5LgZgx1oaRMGOtIxN4+9UL8vI61L4uSTYYc8nbUCQSrA60etrzDkVI1ECOetfQppHcUkTWu'
        'N+JQ5pvhKEe1CRSEAV1JcycJ5NbvQ6M0CTXinOKDQtTixijhFdezkU90LpaTfNTwbUwwtwyHkpVtHROfUfoBmgnZjoB25eEegf4Q'
        'tHIiWuTrSVkrWFMR0qTgIA+8rJ6+3HzWc+Oeq1am8QJriF7osZRjsAdMJ4J/M1vniremPDnwsMW14b3JEWKgJxs9PJyO/evHTkpS'
        'ny4TlSjk0y1uupR9sG6WPih5HaTtBNcvrDYwKjhygW8Gh56iQSmosCc8HbkrCSCcD4pbKk5GAefaoX3VkYPFCFQ39a1cmJnaG5D6'
        '8BJxmn1ltLyhvcGE/wC9cWNhtakrcPFW1p1lDYAwAK3CYUY7mBpb+zowelcvXJxpSTFWUKTyFJOCD9a+3KY0R5YOfmk76eNyFD6Z'
        'piSQbljhGm6M8Zb1Zw3FvBM6GobTvG4hPQ5B4NWe++HuifEuAu7aIlxrXdSNzkbOGXD/APr1Qfpx8V5/fVuGCAFY4NS2u9XHT84S'
        '4E9xh1GDlvPX2+RVMJNcdodH5LDBtZ6bvGmLs7a73BdiSWzjCxwoe6T0I+RVZSjLmVDIr03pbxE0l4lWpGmfEGI19oxtYln0kH3C'
        'uqTVK1l4Ku2DU7LTd3ZmWaSkuMONqHnkf0FPY8/e6fnxWTgtrlE1VtvajNNPWaVeZPkQ2wEpGXHVnCGx7k1oY8PIltjiRLua9nl/'
        'd2YJV/V39J7Dqauka22zS9pbL7KWk5xGhtpy46vtx+JXfngdTXwW58br1q11LSkAqagpWQloe6ieqvc9foOKRCtz+T6KP0tcOHyz'
        'P5FoHkuy2GlxYbKcBb2St1Xvjt/ah4Sioc96K1dqBV7lhmL/AC4LR9DaRgE+9RW9khAzWOOHwTOK3fE6kHCKDEsoOCaOnpw0T8VU'
        'blKLLpyeKGde5CLq8lpRP9PWuXJwwcGqim5+n7371+NxKuNx/Wp/ARfp22cR4YxjNFCEMDmnDdvAT6k4NfPLDZKT271Q2UbxSYBA'
        '74rlqGPNG48U1dfbSnHerj4WeGd+11PQ800uHaUK/nTHEnbjuEf1K+n51iTk8I9HLYn0dpa6amni22SAqS/t3HkAIHuSeAK9E+HG'
        'iLH4UwXdQ6lukY3JTe0nfhDKT1SnPJJ96411rrRfhBZRZ7BDim4qQE+U1jccD7ziuv615Z11rPUOsbouXd5i3EgkoaScNoHsBVaj'
        'Grl8srwofybn4t+Iujtdaaft7t3VDI3OsbWt5Kk52pPtn39q87RGS45170HtIwPemdsAQQVUiybl2T3zcuR5bIBKSSBXc+JtBGKM'
        'tshAbA713NebcGM81JObJm2kV12DvBCU0vftZR6lZSewq9W+ClaNxHHvS/UjSGGio46Vtcm3gbSm+yptSjGJBVjHzU/+MuKO1Cs/'
        'NILi+XHiE5x3r5HWU4BqtVlMY8lgExRPJJJqdtwrFJ2Xs9aYx3ARRuA3xJ8kzwyDS6TtJwvcR04pmRlNWDQmi3dTTi7ILjNsYWBI'
        'dSPUono2j3Wf2HJo4jVD0gTwy0LL1DKFwk+Y1aWXAkrSdqn19fLQf9z0SPyreWbeLagy5TZefUA1FjtnngeltGefqe3U809stnjW'
        '+Kw2iKhpDKNjEdA9LSf6R7nuT3PNRX+52nTsGVfbs+hK0I2b8Z2g9G2x3Ucfn3wBSpNTePRXBeKOfYqFoEBiTqS8AyJ7DBcWWUFb'
        'cJoc7UDk/U9Sa8+6/wBeTdT3FSEFUe3pV6GSeV/5lfPx0H15OgaF8e5MfWrybpHaZsspXlhojJaT0yo/iJ7/ALccVz/EB4Vw2oh1'
        '3odsuWl9Pmy4rQyI+T99OPwHuO30p+3fHEe16IrbW+ujNLQ8FEAmrXB5QDVDsCj5iQeavluP8ocVNgyPJ8uKctGqDqEYWfitAnZL'
        'ZAFUq9w3nnSNhxTEj0kVferoM5o2DFfeUMA496bW6yercsU+Yhtx28hIFbwFCrJYpDbZG48GkN1UEqOMUwdlJKTlVMfDXR0rXuqx'
        'BStTEGOPMlvAZ2pz0Hye1RRk5vCOVXzwiw/w++FyNZ3J29X5l0WSKcJQDt+0Of05/pHfHwK0zxz8UI3h/Z06a02IqZqm/LbS0ABF'
        'QP8AKOM+1T+K+v7X4Z6JY07p1ptqf5XlR2xyGk45Wfn+9eQrjMl3Cc9OmvLffdUVLWtWSTXQbjRHC7LG/Gtq7I7vMlXKY5NmvuPv'
        'uKKlrWrJJq1eGOgZerHXZUiSi22mMnfJlujhKc449yeg9zVdskNufdo8Z3cGlKy5t67Rya07xR1E5Atdr8JdG7XbrdC0/cVoSUgB'
        'SQUM57JSOSfb86HT1xlmyx4S7Mqg5M/X+VoqxrFm0zbkT3s7RJkoC3nFHAACe2Tz0/8ALO0+GetL5mcIDUZawQkSEhPA4z045z9c'
        'GtF8FfDyx6LtaZF/lQ5V6eeLjktbfKem1Cc+xGeOelbGuVaolsVIelNxo6errx2j461RptVpdSt1ElJfhplkoyjw0eUleEmt0yVB'
        'MVhClNlW1Lm3ICsbR7HHP0r6jwxvTahvSQ6SohCz6cA44OAT+lbrqfxR0bY47yGbi3NmYIQlr1EqPTnoB781TPDfUk7VzNxk3ItS'
        'n0qCWldEoQc9Eis1WnhKOUja663++JmQhzo6/sqo4Lqc8IUD0qqawh3l1ji1TylXAUGFYORx2rdtUIsjK1NzoAWylsq8wErBHfHz'
        '7iqg3AjzQLhZtNTJsfzNzZcdTHWyoHA2nOVDHZVctJQeSp/ToPmLweeC0tt1SXEKQQcEKGMV0GwD0r0+dGNL+0LctbDTUxoeely4'
        'GQ6g9cpCkBCcdxzVM1L4aWdbOIz7sF5JKlO+UXEqH9JCBgH5wKp8qi8MT+kkl8XkxZKSCDTGCT3pn/gK7k0pOnoc1amUlch6ektI'
        'QB1wAnn8iTQyYiY81lh1UgoUQHHG4jpCAep6civRvjI9LT2Q7RaND6amamuiIrCVJZRhT7uMhCf7k9hXoTTlpi2mIw1HYS0zHThp'
        'schPuSe6z1J/8AV/w1kaOtVrYgWa6omS3ypSkOfy1qV0B2qweBViv14iWe2vy5nlsojJ+4g7io9gO3Whdqa+I6uvYsyONU6jt1gt'
        'r11uskNNJTtAT99f+VI968r+JmubprK6ec/liEySIsVKspbHufdR7n+1M/EfUdw1TdlS5R2Mp4ZYSfS2P+fmqW8ycHAxWwW1fkjv'
        'u3vC6AFOKWR5meOM963T+HXxQNlfb0zqB9L9qk5aaLh3BvdwUq/ymsSTDceXtQkkmndn07L3kr4StJH09j+VHHcnlE6ynlGn+MXh'
        '2dL6jXebCwo6cmOZZKckMLPJQfYe3xQFnQpbSc1pHg/qpi62MaL1aftTBQWwsjlacYH+oHBBpBq7S8rSd+dtr2XGD64z2OHWz0P1'
        'HQ/NNsgp/Nf2MisC1MRsoyrmlVxjtBR9Ipv5mE8mq9qCUEggHmlpjkkAvONNn04pfLmOr9KeBURcUvNckpHWvFMEPNPwJd6vUO2w'
        'WFSHZDqUJbH4sn/avVGuJdj8M9EGYwiNDkBny22WxgPuAcDA6/Wqz/D1oyPpXSy9bX8JZkPslbO8f9hnH3vqr/b61jvjDq6TrXUT'
        'spS1fY2SURWieEp9/qetZViivfLtnGh/hhn2zPr/AHa4X67v3K4vF195WSew+B7CljiSDjtTcxwE89qEDHmSkNEpTuUE5UcAZ7mp'
        'VJzeWByOvDi3quVwnQmw6h9+Etth1CCry3CUlJx84I/Otv0rpyz2a/3XVN5QV3ibIDYcdaKfKbSkISlPt05x71P4cRdPaYtCWrZJ'
        'gMOLAMi6TFJ8xau/lt9Up9s4NWF+Vp25OAxrldbpJb4LjKgtsE+6dpFe1f0uvW6aVFkmlLGcPHXr3/Z1dK3Th+zOvESXqS7t+dp6'
        'MVOx1hbf2pW1p0A8pwCDg8c5GcfNN9Gafu0rSjMTUdxcEh1wvPMRXSWmEqwQkckEn88Zq7q0dqBiE7LtQiXBpQSsRVq8teR12qxj'
        'kYGCBS+3PzYYQ5doMi3vqJIZk7dwyefunnmun9P+nUaOmNdEdsV6/wDTbb3J/krty8KbS8wlxEtUeT5oIUobhtyOPrWh6btUe0wF'
        'paAO/BIHUgDA/wDelK33HLhIYZacLgz1UnASB3V/49qby3nGWkQ4QDsjy9qSAeOQM88AdapuSXODK5OXGSr3V0x3XC1EU+4pwnyH'
        'SEoAPfFQLnobdUTGLDkgjhAO0kDA9xTOBpa6Py3XJjjW0kgL3FXOaOa0a4ptAffWtac8hRSDn47fr/xXEdVrbcUdjy0pYkzPL3e3'
        '23HUssK85r7yVDCvqM8H9aBtcye5cQ01dnrVNWguR3kHLLo/pUk9/wBat2sdEynnT9gebQheA7vyQhA67cclRoBjTqtI6beiXGQ3'
        'cWZag6x5w3uRkg/dJA455BpU6bs5kM89G3EEZnr63z7VPVcYs2/LfJC1OsR0FJUevpzjGc9qj01NVdNNPXC7i4yX0SQyFyreGkgb'
        'ckZGQpVSalmaduc37ErU0q2yCrb/AC5ykFPzhQFWZenIka1wmWNQTbxCZaBU++rKX1cneBnGMEAZ54pFz+HPH/f+GQsiplb8hiQ1'
        '5bMfYB0KQEH9ua7dbvKYEm3yXW5cN5vahL753tK/CpPHOPamD/2JpCll4NMoxvUlJ4z0wf7DNB6wYmxLfFXHZVCZl5LJXw+8jutQ'
        '/CnsB1pGkrtcsxfQOs1VWzDRS7hZ32Y7jj0mIkpHpbLmVrOegSP78UtjWp18+pBAq0wba0kbnBuUepPWjyw2hHA4rtQy+zgOOecC'
        'O02Zple5SQTTzy0JTwBQD0kMqOTigpV+jtAhSufanNG8IctuKiSG5TCy262oKSodjWwNXhXiJpFu2mGhc+IgrbdCvWlY6jH9JFec'
        'lX1Uh8JRwKvHh1qWZp67t3GMs8cLT7igVm1noNZ5OZ63Y6nGnUFDiCUqSrqCKpt1kqcfIJre/FqyMXbSUXWdqilAUkJmdMqz0Wcd'
        '88H6ivPlwSA+TnvRShhL7MNYyRoc5xX1ShjNQ9+DUTiyD1oSuEkj0l/FZ4gCIGtG2l4ISlIVM2HGB+FH/vxXnlE8qT1oO/3CVd7r'
        'IuMt5Tr76ytaic5JoEKUgd6XqJeSWfRwLZOTyPjJBTnNCOuAqJpeXiB14r4Hyrg0mEMA8sZKc3tlBPBFep/BV6wvacju211CH1IH'
        'mIScEKGMkpH58mvJfmZT1q4eEMyUjUTrUeYuMVoG4JP30g8g/wC9V0vbNFVTaPX0zU8WyyUBxTa+wQFjcfnH/NILremdUTG5LtqU'
        'jyUlLQdGCM9eR9BSyz6ftbjTUh1Xnq4wpZOffj6e9WSLbmW3lNpK9uAEDOB812YpLlDXyQQoqWcoRt3HqOgA/wCKncgsrK90lzc4'
        'kE7eCO2OKPaYZbCFFsAdDzmgLmypsIXHhiUptzJG7YpI7YNLsW5DK3t6CrOlbaPJW1JAaG7epXCuT880cp5ZHIUUjk8D+1BNsKeU'
        'kvICFY9O1VGNF5tBCFBWOACKT49obluIA+l0bmwgYPBxyKrepIU58rfDeWx6irGSfyq0xou5OXUo3rUTgDAB+alkJeQCgoTjphIp'
        'VlUZ9hQm4vg8/wB/07Bl3FN3laSj3O4sqIYT5akBIxgFW04WQecHil+jtAatj2p+GqU3CjFZW22+vf5Q7gben61ugDkYLMrG0nHP'
        'HFB3h9MeKsspKkK4UAOlc6yqOMPko8r9GX6Y0Q5GmmVcJrNw9X8ouja22oewBx+dVbVk5y53995xe5LR8pv2CRxwOwq+amucaIkS'
        'EOOuJZ9KW0qwFLPPNZo+sFxbisAqJJoq4rHRNP5M+BQT9aDuFxbYbO5Qz7UHcZSwTsqvTluOKJWommqOAHLjBDebqtxStmRVbfec'
        'WsqUs5ptJbGOKUSU4VRNCwy1LIdSau9sfwhIzzVCt6ihYq32U70gk0po9E9BeBeoo8+JK0jdiHYshtQQhXcEYI/vWK+KWm5eltXz'
        'LQ/lSG1bmXMcONnlKv0pnYZz1quMefHJDjKwoc9fitZ8a7TH1v4aRtW2xsqmW5vLuBypk/eH+k8/rVFfzg4Pv0a3jk8zbznBr4sj'
        'FRuLwspIxUSl570jAe86bxivyyAmoS8k8g5rlbuRjNLSOdhn1WFcV8S2c8VyFZV70VHTk0e0YoZPiEqAINFWidItF1j3GN/3GV7s'
        'HoR3BqaOyD1r5Jjp7USiOjHCNy0rrP7VFbmN7QCgqd3Z9H0rUrNqGHOS2NwUspxgqySOxP615AtF0kWt1Q9So6xhaArH5itJ0Zfr'
        'tEdSuLJS5E27khPKgenJPsKphqHFcjEelFzYiI6guQhKU8pPal8e5LXEcfJRKTvPCeCfjFZ3bboiU4l0vuhW30jI610dTvwLttLR'
        'aYWnCnVe+cdulPjfFrJvZokC5RXEJR5ClIJ9WVEbaeBcVLCVsvJ244Gc/vVNsdyZCkhbzTrDuTz1z7Z7VZW2oTa0rbTuURg45/UU'
        'allnsrBDPuiGAVpLanhjCcHmh2LtKPrWQhKjyFDOKJVPtoQWnEp9atiUnk5ql3a7qeeuLVscbU5BSlTjThwQk55+nBqe/CWcjq4y'
        'l0i0XWT58VSGZTO3qVK6fSqbeJtxhRkKjJYdZKsLSTzg9wazy8+IiIN0NsuhZRIWcJRHUXVZ+UgZFWu3WW/3OBltBSwtG5PmKwr8'
        '6mw7ekLnJxeGUvX13afuKWmHEhP3lIT2PzVbCi5yTxX7U8VEG+OsiQh5QOVFJ3YPfJ7moG3QE9q1pxeGeXRHNCQg1X5nKjTmc6Np'
        '5pDLWCTzWbhcpAr+NnWkkw4UaZSHTyKUSlEqPFbuMTydxV+oDNXGwE7RVHiZ80VdtPfdGaCQSRZmz6Mmtc/h31Iy4/M0pcNrjMhC'
        'i2hZ4UCMKT+9ZBu9H5VBabu/Y9RQ7qwohcd0L4PUdx+lDCbhJNGSeALxf0u9o7Xlxsykq8lC98ZR/G0rlJ/Tj8qqAWc16e/ibsjG'
        'qPD+1a6t6Q45FSlD6h+JlfIJ+iv968xBI3U29YlldMS208C+OskYzRJJwAaCZ4VTBABSKV0xb4Z9Z5UBTOI3nFAto9XSmMFJSevF'
        'a5JdhKeA5tGBmuXBmp0OISnBxUTymycJIoPNFhq1AT6O1ONMXhFuw26j07shQ7UscT2FRbTuxyaLKfsJSNJN5CHEF+O602obkOoO'
        'UH8x0/OmrV3K2Sk7JDZwDxk/FZ3ZZk6MgNFRUwf/AK1dvp7U5eVFeaPl72lHuOP9v7UKTi8oankt8q5SFNpcYlqbWD/2+n71AnxQ'
        'WhTbaJEkSGSUqVuICh7VWPtbzUdKEOJcI7uc/v1//tCwrSdQXANBCFAHLhaGdifemqx9ewl3wbj4dzmL9fmrrcnlB2NBMp1sIOwJ'
        'WCEbuyldTRDmoNFWnUf2CdMtca6Sic71j7QokAAfAOOh45qkNXJ7RulrkxYpDche5poPPpyRHIXnjPXJ49s1k17v+lf/AJlI1Lc7'
        'e7KuMyKpBj7kLZbdxsDigcLQU4B24OfcCnJPHPsq3YSwemo9ksMq5s3NdpjJlqCkoWlAT5xAGNwHfpz2+aoeuPGIOMSbRbrOph0Z'
        'acLxwAQcEFIqraN8aGbMxKtrkK4XqM+tP2eWeFNHHO4ng1T9S3ZV+1HMu6mEsGSvdsT2wAOfk4yfmvK2WMZEWxi3lHCXVrWXFnKl'
        'HJ+tSh/A5ocEBFcLPppTAP0t/KSM0okLJJFESFGhF+qlNiJdgjoPbvQT6OeRTRSRQUtFHE2KBY6QHRVusGeKqbA/mirbYvw16SGI'
        'seD5f5UmuwIzinqElTY4oGbCLuetJbS7Am0kbf8Aw5XSNq3w8uuiLmorCW1NAH/8awcEfQ15u1HZplmv820yUFL0R9TS/qDitH8G'
        'rivSutos0khh0+U8P8p7/rVi/iBsCVayN9YRli4tpcJxjCwACD89Kb5ozp/MREppx3fY82tqo+Ocge1J0rIXRzDwSKySeD0oDZKg'
        'nvRDT4SM5pSp0kda+B4gcGkuDYt1seGSCnk1GV5VlKsUsYcKj70wYaUcE5FKenYGxonStRwMZplbohWd6xUcNtIHTmnkJO5ICRz8'
        'ViTgxkE0zkMJSOKkiMSJclMSI0XHl8ISBmiZDS2VKaebU24ngpUMEflWk+D2nXIpRMmoLcq6tKMBJSMltB9R57H3HtVUZ54LtOvN'
        'NRMgu9hurDkaxASPOWAFKCSCsdyPirjpC0SbBbptpYDiXXvU/wCVkkJTjHPXuTWp6ng3ESY79vjsvha1NbkDKingkEk8ng9KpV8M'
        '+BqRm6WW8xY00EJyvnacYIx3yOx+aqrpw97K2oRk1Hoo+sZCUMrjRELCywr7uQQAOvzxWUwUQJMpT78G4SXlHZ5rb6UhSj7+knnk'
        'dfmrn4ktXCTJkSlMr8sqwX22FISonsM/2r7pCJcXoDL90cylA2xmy2ElKemTxk/n71uouVUdzE6q+NUcsXw7aiHHW2yz5ZdXvWSr'
        'ceOic+wzUrbak9as7sNBFAyIiU1z69VuZzK9TvYuT92vjnAqdSQnih3zgHFU5Ld3AvlUEVermp5SzmglqoHgU5E6lAihpAyk1+8y'
        'viyCmtTM3JA7Kf5v51cdOMKXtJ6VWYEcrdBI71f9ORwlCScZpVtqie8qG7DHoAxX1cbP4aYNN+mpQ2MVy7tTgjvvFbMUpWCByOa1'
        '66tJ1N4T+aoJVLikOjjnKRtV+3NZyhn4q7+HM5TanratX8t0HCSMjkYNDo9UvLtl0+Cem3LcH7PHa07Tmvwcx3rp1YKaCW5g19E4'
        'nZ2jNp0EDJoplsungGksdzKutW3T8bz9uBmlye1ZAnwjmBCVvBIp+xFO3pTCNbkoQCR+1duLbaTzjNc222bZzrJyyCIZKDzTCFMX'
        'CfbkMOFt1tQUhQ6pI6GlsmagDjFd6cutmRf43+PJK7fk+bhRHbjpzjPtQR3t8mQc20i62huZraWhaipychxKJThGErR2Xn+rAOfc'
        '9K0jW0ZLSmtTwCT/AIMppmAA4fLS3twcp4yVcj/0Un8OoOnrdc5i4kptn7Qnc0tcghUcKT6UkdCD1GeRmgfEObM07bW1R3UTmkrW'
        '3IZcO5IRnjBx1x0+a7VWnS+UuX7OxSnFfk+TNWKuWmX5Vjnus3QPl+Ol1W3DoVlYHbAHOaznV/iUu52lu3zbZaTPaU4tU1qWkh3d'
        'yPSnOCCOxH71RbteP8TupNvX9heWCEjd5Yzk5wc45Hfil9rt76ZK2J3lBhJBUUlKie42qGf2o52eOOch2SUU2zVfDfV5mWKdbr5Y'
        'G1x3Eq+yzE53tKIPTJ5579ualQKq0G4Mx2EMs4Q2jhKQegprHuiVJ+9XGvnZdw+kcTUSnc+uBu4QlPNLZjicHmonbiFDANCLdU6c'
        'DOKGqva8syitQeWRPKBVQkhXBo9UdSjwKgfiKIxirPJHBe7Y4EMkZNDKRTZ+E4CT1oRxkpBzQ7skjs5ACk+1dstFa+nFSLRzRURo'
        'Dk15ywjd/ARb2QFjjFXmxIHlpyKqcIJCxVws6hsFc6+wDcO0nAFSIwetCFzbxXaHgBk1yLptvBHa22HJIApjp+YI12YcCserB/Oq'
        'xInAHAVULU4hxKkr5BzQ1QluTQFcZbsmAOr60K5knNdkkmu2Wi4sJAr7o+myd21lbryUgd60rS8ZEdpO4cmqxZYAbAURyasrEhLL'
        'YycYoJJGNZH0ySlto4Iqp3m5hCyAqubreQElKVZNViQ8p5zco55pTqiJdcQ2RPcVn1Zq0eGdlZuNwau1xVvjxpTYTHIBDxzkg5/C'
        'OM/Wu/CPQknVl486THdFpjpK3nfuhZA4QD3yeuO2aul+vFttl1fiKYSwlQUlpbSRtaSSR6ewwMUSpi/QyFMe2K7vMtuq/EEqjThp'
        '8cuNyHOq1IPpB5AII/ICgrtqrUUp9Nnk277QpBWlaWvQ28gD74V0wAAc4ql6/TPbVHiSI7exlBU1IR1caPIJPce3t0pFIv8AeLq1'
        'Hiz5jrzEdstoH3SUddpI6gfNUN45Y3hE9+uzd3UylVuhRPJ4LjCcFePcjr/7zULMopASg7UgYAHagZBSThA2oHQV+ZJyM9KRJb+x'
        'Uvl2OUSV7eFGi4lxcAwVc0paPpqZltajwDWbYoHaiwxJS3CAVVYLZggZqpQGHwoYSasdtW6ggEVNdUn0T20qXRYkNIKQa+LZSRwK'
        '4iKUoDPSjgkY4Fc1xlF8kE65Q7EsuMopOBSeXFXyMVbXWd3GKgVCSRyM0yNyiArUilGM4FZKakQMHFWh63JweKVTIRbO7tTvKpjY'
        'z3Asfd5gxVqtLwbbBWeaqrbu1eB2ppEcUoDKuKnsqbGqD9lnMpKh1qB6YlIPqpO/PSwj3NLnJzr2SBgVMtNl8geLkbSZiM5K6ijz'
        'G1OfewO9Vi4y3BwM0I3OWkfeNWV6dIbCEYlTYZU4oAA81YrXbAkBSk81JabelAClCn6EtoR0Fdp8HSfALsDTdK7lPKUlKTzRdykp'
        'AKUHn4pY3AkzXkoabWtSjgJSMkn6UlzJ5W8izc484AMqUo4AHetI8K/DCdqe+w27n/0kJayFpKsOKAGeB2Hb364qzeGPhVHXCcue'
        'oYjzj5wqJGzgJxzuWB1z2Hwc1tFpgw2NPFhpIZfbcQ6Fp+9lOcE4x3NHHMmi+mhyrdjFjTDem3F2+MhBisteQWANvo7qHbjHf61k'
        'fixdoNkuU9F0tDm5xI+xutJPllQ6bknsRyccg1qWqLqES1svQ/MbeZ2hRePTGCUK64+D04rBdaS7hGjP7bkzM6JU1IWVPFHUDGeA'
        'Bgd/3xVLaXQCyimTr+qZcESLNGVa31tqS6ylzLJyPwpVn6/WhLf9kXaXUkLEkO5QrbkLTjnJrqHF2ulxshl9R3JCSVFtJ68nvjpR'
        'v2dKUBKBgAYFIlMTOzkRPIKTmvzR4xTKUwMZ70Ghn11ikAphVvZU8sDHFXK02cKSkkftSXTsdJWnIrRLUwhLYOKj1Nrj0Taq5xXA'
        'LGtKAkenmiEQkoHSmgAHGAKhlLCUnB5qBXTZDXqJbgVlAbOO1EpcTnjrSWXNLasE0O1ccK+9TvBKayVWQlYsljCio810OKAhSw5j'
        'JpgMKHFTTrceznzrcfRyoA8UBcW0ls5o8pVQN0VtaNDXncZVncUy7yBGWSOOa5t1yW4cZ/egtRrBWRml9sWQ4Oa7Macx5O5XDMeS'
        '8NteekFRFTGOhtskDtQtpdKm0g80zfI8k49qBUrITrRUr0QlRpG47z1pvfT6lGq+pXNOVaSFOo//2Q=='
    ),
    'white_throated_sparrow_08.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgIDAQEAAAAAAAAAAAAABAUDBgECBwAI/8QAPhAAAgEDAwIEBQIDBgUE'
        'AwAAAQIDAAQRBRIhBjETQVFhByJxgZEUMhWxwQgjM0JSoSRTctHxFmKS8IKywv/EABoBAAMBAQEBAAAAAAAAAAAAAAIDBAEFAAb/'
        'xAAmEQACAwACAgICAgMBAAAAAAAAAQIDERIhBDETIkFRBSMUQmGB/9oADAMBAAIRAxEAPwD5hhyj5xTKG4wozULQgVC6kDioepEm'
        'qQXNd4XvigWuN75qGfdUIyDT4QSRRBYg7xc1qZBmhd3Fas5o8DGMEozTCGVSOTVejkIomK4YedKlXoLiO2ZT2oaVN1QW85b60XG+'
        '7ypecTMw1jgxHnFQyRsO3amSKDFWqoGbFap4GpYhHLDluRUsFqSeBTOWBWkwBTXTtPDAfLWO4CMtYmt4WjbkU5sJNpBJwKMk03zA'
        'oWa2MeQKCf3Qc4ahmb+OBAUYHPcVBc6xGynBxScwNkkk5oS7jK55pK8WBM6UT3uoeIT6Unu5C2TWzhs1GYyTzVMK1E2MEgcBj7VP'
        'Bu863EWKkiSmchnIyRkdqgmU4o4JUcyCg5g8xPLkHtUJNHXMfORQciYNOjLRiZoa13Vlq1xRhFikzioCM1JI3vUYNRRRHGLRDKnF'
        'DulHOuRQ7rhqogyiAMUrVlorZk1gx0zRgHjmt48ipCnNbpHXtPPolsxluaawrxwMUFapg0xTt2qeyQqUyZSPDIFRIGDHiiYEG3kV'
        'tsJbtUzsEytI7NC84BFXHSrUCMcc0h0q2DTAmrpp0aqAMUtvR3jreyGazHh5ApNeWy5OVq23CDw8+VIr5UwSadyxFdnSK1exFB8q'
        '5pTPGSx3ZzVjkXcSc0q1BQCcd6yNusjcuxJMmDkVFtwe1FTcZzQ0jCnrs1IzhcVkHHaoTJxWpkNa4mOLCw4xzWsjAjihi5A8qjMx'
        'BrOAHAzPzQrLk1K8m7isouaNdDY9AbxnPaseFTDws1qYsUfMLkTn3rAqN5CTWEbBrOJnElOaiY5Nb7s1lE3NW+gksPRR5qRoeKMg'
        'g47VM1uAvagdnZ7kJXTBrZQKIu4tr8ChuQa9umS7QXCvnRaEYFAROc0TGxJxSZJk8oNjS1+bimMdrkZpPbSMrjFN4Lk4xUdif4J5'
        'RYw06NUcCrLYkDFVW3k+YHIFOrO6O4AVtaZd4yaHN5JiLikV0wKmmchaRMmk2qDYjEcVRJJosnW2hfO6qDild182a9cXQVsFjQzy'
        '7hxS41NPSR14B3YAU0tkYg80xuQWzS+4TBqut4auiAvWpfmsMK8Eph7TzOTxUbZNTLHzWxT2otSNSBwCDREPvWBHzW+0g0Mnp6RO'
        'uMVo5FeBOK1IzSRJGUYmveE3pTOO2zjiiEtgDjbRStSD54KI4mJ7UZbwHPNHrbDPYVKsQHlSZX6Lncb2sIKip5YVC1rEwSsyTBqn'
        'cnoj5GLLqLLHil80GGpzKQaGdATT4TY2NguVCpqQHaRU7RjNY8PHfmmboxNMmtm7E8GjUkxS+PI4qUZoXBMaqtGUU+D3pzpEzPIK'
        'rkHB5NWPpxQ0orJRUUUU14y2W8BeAHFKNctsRnI8qtllEBbrx5Uq6hhHgsceVSqx6db40onLtRTE5HpUaftorVVH6tgKE3bRg1cu'
        '0cqzGyOY4BoC4ORmiLiUc80DLLnIooxJnFkTd68vNaM3Nbxmm4ZhNGKnSPIrSGp1OKVJsFyNDEBWhUUQqSP+1SftXmhbtjBodM3Q'
        'criiLSwubk/3URI9fKibC2V5lDDzq3WMCpGoVcD2oW8AxsrNuAccUQi5fmhYzt7VIkuO/NIl2A/sGFABmhpXCnvWGnbbgGgZ5Cc8'
        '0MYNiuLZJLc4PeoDdc4zQcrZNRFzVEakOjWhiLjNbCTzoGNjipQ2BmvOCPOC/ASzjvUbSc1A0hq59AdE6lrdr/GXD29ksnhxzGIP'
        'lvNgDwcfzreorWOqrbfQF010tqOsjxRLa2MGcCW7lEYY+gB5P17e9WSP4WdTXKk6dNp18EOGMc+0A/8A5AZ+2a7Z0z8NtAvOn/DZ'
        'Z3v5Fy15LLlgc84XJA71Xeovh3qvTMt7qcFxbPaQpuiEO5ZVyeSCP/poJ2NHTrqg1mnJNe6A6w0GAXWpaJcrbYyJo13pj1yO33rX'
        'peRRIOfOunaP8R9dtLiyi1EJeWMG1JIWQEMvOeW/zd6s170H0j1ZD/HdDS40OeUsWXaTESD3K5+UH1BxQuzVjCVTg9RTrKYeAvPl'
        'SvqKcGFh7VteR3GlXMtlc43xOULL2JFINcvNyNzUnHsqtsyBT9UcG7c5pZcTbal1CYGd8etL5fm710q/RwubciKWQuc1CealK4rU'
        'ISe1UIYmR7ealjWpFhY9xREEBB4FY2ea09bwuxFPNM01GIaQbvrQ9hEC+SO1WCyKgAUp9sXw00a3QLgKB9qV3cAVjVgmUbcikmqy'
        'IoIB59qwaoLAS2cJIG9KslhdRsg+YD71SmuGVu1e/VSH/MR9KFx0U1gQZOfashsHvQXi81JE+fOh44I44gtm4+tCy81I7cVCzViQ'
        'vXoPIuT2rXZUrNzxWDR6M5msamtyDjisIKlJG3FeCihl0avTh1yM9Uz3MVgO/gxbyx98HOK+h+j9b6PnNppuha9Jc6ckbRywyrzE'
        'v+VwpHke/FfNUFuJByM010oTWVwk9tK8bqcgqcVkopvWWVtxXR3DprSZ+jbfVNMg6gnut85e2jjcgoD827jkZ+X6YpnaXPUNpcWF'
        '5pmoX9zHMzLObmV5sYXJbwyMBQccjvSDRJ7e76asNRvImS/uWkR7lEUb9vCkY7beFOe+6r/p1gvT1pDMsj3ctzEr3Dy5BXgvjaeB'
        'nt+a2cE0PhY29BoNavdT0ee8vtH0rZGhxPJGqh9pw3AyQOR+aWaJrdhpssqJdvb286EFll4yOynHHtmjry6udQ6fklELQTtH4ixD'
        's6gcHPblfLH8qqbaXqFhJDPD4a298u/cEBUEDkFecd+TUjrzCyEtT0LvZ7LqKK9W8e8Ecf8AeRoiKHJA5YEqNwHfGM4zzXKusLK5'
        '06YxSI4jdd0TnBDqR3BBIP5rp3T09zb6sNK1C2iMMsu6ISTCUswGQEYjH8qP6k6Kstf0O6SyvU8aFwttHJGoEG7naSORznHlxg+t'
        'b9dFzhJx9HzVcRO0rHNafppSAFXcT5Cmt3YXVvqM1ncQPFPDIY5EYYKsDgim2l2aKu5ly3qatXRy1ArsGkyv80gwPSpXsVj4C4q2'
        'NCuO1LNQiABIr3IJiFoQvet4lAra5bDYNeswGk9hW8jFJB1lHkkds+dGjxY+U5rS2HbNETH+74pTmbySBbq4uWj2+Jt9hS6ZCe/J'
        'oxz5moXZT580LmA7BVcRgH3oNywPBNNLlR3zS+ZeaOM9FfJrIlf1qaOTBqIW04GTE/4rwRwcFSPtTGg5IK8TPnWrEY71CAwHY/is'
        'tux2peE7R7PNZDYqFmIrG6tw9gQjDNbFuaF3kedSW+52ArcGReDuxXcgwKZwRH0qDR4flAYU6giBO1YzIQCSq+g5NTTs7wpjan0d'
        'i+F8sEnw7tozBG5t55cMVyVYndknPA4XinjXpvoIFlmmETwySyoXzkkDYo5J7ZXj1rnPwQnuLvU7rp+5nMXiL+ot12nEpXGUx5Z4'
        'OfQV02Kzi/XQoy26Rc7EMwZip7cDkYPGD6Z7UfKTS7LK+OJg0t5eWt+NJtraOe6W1jkSFu00TDJYehQggii4dNtY9V0WBRLJaTSP'
        'JHOvDRMcqyMPQH1o+DRXHUNrqgmcXNs23ZkEbc8qD68mjOl0jj6v6gtwihvEWSOI5AOcElQfXzAoZdjo+uih/oIdQs73R7qXN9pV'
        'ySs2NsigNjPvzj3xWumDUrTV7szzbTFC36gYOJrcn93/AFKefz71auqrb9J1/c63HH49lLZ/8UqgFlH7ST58EDvzS5ryCw1iz1OA'
        'm6sL+0kjeF1yrADJXPkSMilS+ssHQ+0WUT4v6FEktprCF5NyCCV2XlwMeHJkcNuUjmufAxxjAIr6K6g6bhbpq201JZLjTbqAtbSt'
        'y0QPz+GfoOR9CK+ZOrGfSL+4sbjKzwuUI7Hjsfv3qmEtRzvIjxeoLmu40BLMB96S6hqIlOyEcetJZr2WXO5uPStoZVomiJybC1jM'
        'hy3OakihETZU8GtoJE2jkVs7rng0PYHZPHNs4Jx716e8QR7d4zS+aUgYDGhiMkkHNDwPSbwLkuh/qzUDXLHtUUi8VA5OSK1RFxTZ'
        'O0ue5oaV6idmFQSSHzpkaxnDD6Du+krULgRrz7Urfo23ZseGPxV2uNRtmGfEWoIr+3L4DrV/1YvWVy26GtXT5oQftQmo9CWxBCxg'
        'D6V0iwmhaP8ActaXTw4OSK84QYHHTkL9CxBsbM59qGuuhURcqtdSlaJpeMd6lhs5b+5S2tIGmlbsqj+foKHjBLTVGTeI403RJzwD'
        'T/pr4W6nqbr+js5HBP7zwo+5r6G6b+HtnD4c+pAXE2M+Hn5FP9avcFnHDGIreBUCjAwuMVzfI8uC+sFp2PH/AIycu7Hn/Dk3QHwU'
        '0yz2XOskXsoP+GeIl+3+b78e1dTi6a0K3iVLfTrVQBj5YlGKYwWcm0fM3uBTG3tIlX58nPvXPTc32dSNdVKyKKP1d0sJdOS+0uIJ'
        'qNg4ubUgYDMvJQ48mGR965hrupHpTr6PXYIlm0u8hjeQSxgqyZ/acjhlD8Hy2/Wvo0PZwgKWHAxtHOK5L8V4dCXS77RtY067SC5k'
        'LWd1DGDsLDPbI4DZBHow+z4J7mi7JcluGtpFperyLFpWsC3O1ZImuVIIYHlWI4DAjuOKtl5YWd86EyJb6jHh1ljAIVx6MO49veuJ'
        '9FWutaTbYu1a6/RhW8SMZE0Y4J4zzsA4PmnvXUtPukiu4gzgwllIY9jHIMq1Pa3onikgvrLRLiMXOs6dEo1BottxCBxKmNrEA98j'
        'mqzJpDtpSnSomiL2Uhe0YfNvwc4z2cd8eYzXUtFka5Wa3u1VpLRVzu/zKex+mK5n8QLXU9A6yt7+aYtpt+UiJT9u3smPQjOCff3r'
        'J1t9/o2Fn4M2vUt9ZdL6Xf3lrFdaXNaAXCBNrgA7HZT5MME/+a4r/ab6Qb9bZ9SaWjSRD/gbz5QGDrkxyHHHzqT+BX0TqNlZ3eg2'
        'un3Tr+nmibwpQMAcDPHvknH1ql3kVvq/S15pVyjSfpYDbFzkmSS3xh//AIkDmiT4tfoXZFTi/wBnx5JZ3UY+aFhUYWXyVs/Su46j'
        '01bOduwfil46Ttg2fDH4roOk46s05HHNOgwUb8VutxIzfsbP0rr8vSloUBEQ/FDr0hblv8JfxXvhPfIjl0Ec08gUI34p7p2jGQDc'
        'Ca6fpvSdsF/wlz9KaRdNQRjCxgfapPJrsS+oqyxv0ctbQU2cLzQVxoAHOw12D/0+inO2sS6BHs/YO1c+KvQuM2jik+hZXO0ikeo6'
        'W8ZPBrvFx0/GRjwxSi+6ThnzlP8Aaqqnan2OjaUVupL3sST96xF1HeKwbLfml4jyKysakfSi+QNYXPTerLlI1yT2qW56ruH4DGqx'
        'p0ck80dtBG0krkKiKMlifIV9O/CL4Q6PpNrb6v1FFHfamwEixPzFB5gAf5j7n7UMr3Eqo8eVrxFQ+G/Q+s65Gmo63PLY2r8xwIuZ'
        'XHqfJR/v9K7X0707Z6bbiKxtTGmfmYcs31PnViEcG0KqgKOwAwKy92lrHhFIx2xUs7ZT9vo7FPjwpX1XZ6Czcd0Io2O3VRufAx60'
        'mk128J2xWxf0LHFaImoaid13KVj/AOWvAoE4oc1J+2MLnVLdG8G2Vp3/APb2H3qHN7Pw7+Ep8l7/AJoi2tI4UCxpz9KYQWbyDB+3'
        'FY4ykY5wrQFb2yoAP3BuDW2o9PWmrWX6a9XcgOQfNT607tLAR91z60Z4ManKqN1V1+M81kVvl9/U5Bd2S6HrsdjLZSMjD+4uYhhJ'
        'B5qw7bh78GpJNPsk1K30y4tRBbTxlbV4CVwScmIhs4wcMv3HaujdRQ25tEJ2GRG49feq/qcEN/pwtJN6HcGjkjHzRup4P5/nWy/r'
        'eaejL5Fyw2s54xapGQFkhiEPiYwTg9m9qTdSpPf6Hd6YljDfhAZBbyHDxkf5oz5jt8te0a9kvUnjuVVdTsX8K8ReFk4ysg9mHI+4'
        '8qxeyJ4kcqSvDLEd0U6Hle3J9V9RRqfXZrgvwc7+Fup3msfxTpfU7jN9aTNeWbsMeJC5O4D/AKXPbyDe1T6E66fqUumXrRs11LcT'
        'ghgcITHGfyQ1C6/qVonXVrI1m2m6xMZILh42HhCQ8hlHcbiFPpyaJ0jTbbqTX11syyQSxxNDJAmP7xtwJdc9u3I9Tmssj19TYPv7'
        'HN9U1K3t7yWCRwHidkYe4OD/ACqGLUrdzgOv5rmfxY1K7sfiZ1DaJlEW+dlXPYN8w/8A2pDBr94pzvP5rofK8040qsbO6teQeEPn'
        'X81iG+ti2N6/muLr1HdsMF2qW3164D53H80LvYpwO/afdQYB3L+aL/VQ7s5HNcS0/qS6wBuandpr9xJgMxzU9nmRS7FNnVhNE3bF'
        'ZnZPC49Ko9jqkrKOSTRkurOF2kmo158NA0cSzRDvUQkiZuMVUdU1SRclXNARa/Indqqr8yDQSOfK5C4NRtIAc1Z7jSkTPy0h1ay2'
        'DK1LXbGb6Hxkm+i6fAlILjrhWmAbwoiy58jkZP4z+a+r9N1RTFknjHAr5S+AVpJ/E76+YdlEKn3PJrvlnevKI7aAn5jtyKT5PU8R'
        '9L4EEqUX2PU2mOy3VpW88dhRkFvNIQZgT9O1RdNwRRwx7Rn5e1Om7fLx7UKjq0dKWPEAtGsQ4jyfSibSKVyCVIGeBUsar3b91Sfq'
        'jC37VI+tElnbFyk30kHWdoScnim0KKi+VV+PUJnOI4WC+p4ogG4uBiRyE/0DgVXVbCHpayC2qbf2eDGW9QEpCPFb27D71qolY7pW'
        '4P8AlHatLeJRwqkYotI89/LtTU52vsnfGPSAmtEmBEkasDyBjzrUaZCGOMg4+U+ntTMrgcd69tGKZ/jIz5pfhlI6h0N7LVYtZtV2'
        '7l/T3AHZkJyp+qtn7Map3UWorbzvGRg43xtxgk8EH2NdjvIY7i1kgkGVdSprifWGm3pAa9eB4IUKmOFD4h5PJbPY8Z448qRb/T0v'
        'RXQ/kXftCO4TT9Y6ms3nNm9/aIrrC0+ySXbyqlSPmYcZwckYoHXLySwuDe2UT2kts5d4FDZY89gAcZJ5z5VnU00u50STT3srYwKA'
        '0cRXA47EY7HvyOa5jq+r65ofhywanJq2lpMY5LO9HiTRR7cgxy/uOMHgny86TGzkG017KZ/aCg0+TruPUdNDBr+yjnuUL7tkoZkI'
        'z64Rc++a54I3HkavPUN9b63PHLDp36VI92CX3MwYg4P05/NKprFSuAvanq9JYcyyS5PBBGjedMtOtWkI4rZ7Ta444ptpSqg570Nt'
        'uLoTPcCrOwIxxTCCwldgEJGPOibDa45ptA0SADIrmSnKTJcbZLpgeJQp7VPesNuRyajeRFTIIoK8u1jjJJpTr16FKGCjU5n3nmkV'
        'xJJuPJphfT7pM+tD+GHHIqqHSFvott1ErxE4GarepW4IYU/juQYsGgpoRKefOleOnB9lFVbi8Ld8PLdNO6fCoAGkJYn1NdF6IkM/'
        '9+Tnb51RrYLa6Hbg8OQcCrp0yv8AD9JhDD5pDnNbOWvT7GuChBRR0fRdSRGClu3FPBfCTkMK5npl+RO2TnBwKtNnO7BcHv5UcZAT'
        'itLKLoYAB5oiIqxy2PrSU4CcEk4/FRrftEdhyfvRP9sBJP0Wy2Ma4GOaYQZYjAFV/TZ2kUMRinVrcBQOQKfW1+SK+D/A4jUAZxW9'
        'L/4hCi8yKB6k1F/EJZyVt4mA/wCY4wPsO9dBXwiujn/DPfQfcXKQjkFj5AVzvqn4ydFaBqE+n3usK13AdskFtE0rK3+kkDaD7Z4q'
        '23WnLdwFbqaQq/DBHKnH1HP4xS3Reg+jtMcmx6d0uF85LrbqWJ9ckZpU7LJ+h0Y1RXfZyXqr+0JNDaM+idNXpQj5bi9BjT64Gc/k'
        'U80fV313pXS9buNniXloksgUfLuPcD7g11PUdMt5bZoJIY5YWG0xuoKkemK5nfabbaDpkenWsYit4p5EiVRgIpkLYH0DVLPluSK4'
        'OLjsCjdU2scM0ksbEJLG20eQNcn1S4Nve3NvNFuD+DcxHHcEOCPvk11nqFv1UagPxtOB6Z2/965N1BNHci3uTt3JbiGQjyCscjHu'
        'GFKjD7aFJ7ESRaYNuY1yp7H2rz6ZtH7e9WLQE3WhikA3wsVP0/8AOR9qlvIlxxS5OSkQWVqPZSbrTlUEkUKqCIcCrPewFiQBSi7s'
        '8HnNOhJS6YvjGXRHZXO3hmxU02ogHCtk0EbKRv2nAqKSJYuCPmFOjVH2bHx0g2XWCifOSP60tn1ZpZMk/agrzOdxpc0oDHmj+CLA'
        'spQ+ScSHcTUqS+lV4XZUYzREV78nely8dkVlBe7WIvjPFNLawDTwp5vIFoLTvmIzVh0jD6nFjG2FdxPue1Qz3kdGipzvihrJb+Jq'
        'llasvyw/M3oAB2q26lMqJFGn7e44pLotqJYru8k4JLFST5UbEGmgUk8gcAUt+j6VrsJ08usnieX9asVnqTRTwocjd6+VLdJtmexV'
        'mGMnPI7Vtc28r7Dn5k7ECsbcWZFKXsv1tPE9uoBGTWY7XfIr7d3vilXTsZkjQbicd6uFlAm0AjnyqyL5IkkuLwjg3RqNq5OKyltf'
        'XcxWa5Edv6RDDk/X0+1HCADyFFWqgAYFbw1gu3iujexsYIEwidvMnJ/JosEKu1RUauM8V4nninRSj6IZNyesJkEcag57/wC1RwS7'
        '2LKMIOB74oed0QF5DgAZOewqk9a/FbpbpiAo90Lu5A+W3tiHb7nsv3pvNLsD4+i9ahciNOWArmHWF6lwkhidWCXbRk5zg/J/3riP'
        'W/xb6r6qvwtrcNpFipISG3f5j7s3cn6YFWroyWSL4eQGR2kla+ldnZskktHySaTKXKQ+GRjiANQ1FbdrJHL7ZnWI58mIbH+6iuf6'
        'mqpqF/ZnIVpH+24Ef/0PxT3rScQdNC7DNutrmOTH/TIRVe6ymWHXzJHkrMgYe4Ix/QVi6C9k3TtyfH+fjx4wSfVv/Ib80bO3zE5q'
        'r2t2YrptrZ2SHb9D8w/++9PiWbDA8HkfSgtj+SW6La6MyKp5PeoJ7dWUjbn3okROxHHFbuNqYbg1HGTUiGDakV+5iaM9sUnv2Ac0'
        '71acA7RzVa1C4UE8ZNdSp6jox9AN2GcHnFKZlIJxzR8krSNgjAqGaI96emC46LmzWU3Y71LLERzWiq2e5o9J5RSOk2d1tAParv0r'
        'ZSz2zSAfPLJtP/SMVTukdK/iWupDL88caGVwT6ftH5xXZuitLCxLuX9o3H61B59K8d8N1nU/jv7NszCbUrdbHR/AQcuoAoq20510'
        '5GCkfL51Ldwm9uDIeY0OAPTBqyzWyfoVTaRlR2rnJamdOTxoA0i38XTlbbgqTTWx0vxSm4YGOaz03aO1uY25Xee1WvTrVFjCqOPf'
        'zpqhuMRKfHUBadpiW65VcY7U5sIn37SMVPFDngYx6URGpjfIU/enQhjJbLtWEk0AWMcYOKzbIFXLD6VI+JI+fvUTsFQk4UCq8Sek'
        'abawwR83pQmsapY6TZPd3txHDEgyzOcUu1nVb6xsprqOz8aONS3+IFyB9a+UepOttS6r1ebUL64kMTuTFb7yUiXyAHb70tyxag5r'
        'gtZ0f4lfEi76gim0zSi9vpzfK79nmH9F9q4vq8ZTI8qcpdgp3pJrEwc96lTlKWsl1zlrINNAaVRXWdJmW36HtRwB47tj1/vIh/Su'
        'QWEvhyA1f3vtvSNku7HG7/5Tgf0p6X2L/wDXBH11dCXpPUYw3+rj6TtVb1m68bS9BvWYsXtowxz3K4B/kax1FqG/pzUELE7g5+3j'
        'yVXnuS3SejKSdyyyIPpvz/Wm8ev/AEBPsLuLtY70qMD5eOe5Q1a+n7uO6s1O7JjJQ/bt/tiuc6zciLUmbGAk7Z9+xqwdGyzRSTbw'
        'fBJ258gw7flSPxWWx+mi5P2dBjdVHrQuoTAqdo5oZbtQncZoK9ucKTu5rl4+RzJtKQu1TsSTgVXp3TcR3o3U7suSA2aSSlixPNdO'
        'pPDoVWrDdymMg4rUupHJFCTSN2rwyRzVKXR6y1L0bykNWqR14DmiIYye1BKWHOstZ2L4QwRS6rfvj+8EMYAPc8nPf3FdnsrcWVgB'
        'GB4k3ArgvwZ1SKPrGGEklbuBot+3AJX5l/ka+gLRgWJb5gv7QaV/LQf+Q/8Ap9D/ABcl8CRrbW5t7co/JOSpNOWIbT1ccFVA+h7U'
        'vvZo5Yj5Mp4/rTWGMTS28SABRgn6VDCPWFVkvsmN9DtDFaocctyad28eADzUdpGvhgeWMc0fboO3n2NURjpHZPCeBBj3ogBc5IqF'
        'MAedYbcTgd6fH6rCKXbNpWUDGOPaohAZTlu3pRMVsP3Oa2nlit4izMFAGabx3uQKnjyJzT+0F1DD0t8MNUufFCz3Ef6W2HmzuMf7'
        'DJ+1fFWm35iwM8Vc/wC1B8RZOsPiBLY2Vw7aRpTGCBQfleUf4kn54HsPeuXwTE+dF8f1Atlrz9F4h1NWjxnmoJ59/JNV62mIxzR4'
        'kLCp3WkxtSTD4nGRzVlvZGbQ7QKTtxbpkdhmQsf5GqcHKjOaL07Wzbq0VwZZYlGYkDcK3P4HJNeS0fJrCu6leN/CLhck+JAG/Mz/'
        'APepvCb+C6Ep7fqySPQHB/oak1U2F3IQqukfhLCAMZKq2cn3rY3KOsalRtjOUHpxj+Rpr9Ezlx7Ndb0mC71KadZtsTukgGOT/qFF'
        'm7MUaRR4VEULxxuwMAn3oSS5zwKFMuWJzQ8dXZPZcx1HqTkY5oa9vnYYBpZ45U4FQTTlj3oFUt0j3Wbz3DeRqEzMwraJQx9aK/TK'
        'VyBTliNXP8AGzd3rYLjgmppI/DOcULLKNxo12eyRKpAai7ZwBxSp5cdjWYbrHnQyrbPKHes//9k='
    ),
    'white_throated_sparrow_09.jpg': (
        '...'
    ),
    'white_throated_sparrow_10.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAGwAAAwEBAQEBAAAAAAAAAAAABAUGBwMCAQD/xAA5EAABAwMDAwMDAwMEAgAH'
        'AAABAgMEAAURBhIhEzFBIlFhFHGBBxUyI0JSkaGxwTNiFhckNHLR8f/EABoBAAMBAQEBAAAAAAAAAAAAAAIDBAEABQb/xAAoEQAC'
        'AgICAgMAAwACAwAAAAABAgARAyESMQRBEyJRMmFxFEIjUoH/2gAMAwEAAhEDEQA/AMALcjBCiaIt0R9xzBcKR96KfktF4kCuS5jr'
        'ZBZbpBLS2ljVTTkNAUHT+a9syi+k/wBTkUjeenyxtKTiibTFkIUQ5nPvXcT2Z3JehPFwekh3alwgZ8GjrREmuDcFkiui7S88vqIS'
        'SMU6sDrcJBQ+kAitBsQTYMBiRnjc20OpOM1UanSY9nCGAEkp70u/cYz1ybDZTwaI1NcGlNts5HPFS5NNKsZsSNskd1cxQceySeap'
        'HLU+kdRKioY4FJUQunNDjDpBJ7Zq0h/VNRApad/HtQPn/JqoPcz69wnlvEKaP3oeLHVET1ElQPfimWsr8WXumuOpPPcDiudku8B5'
        'rDyk/Y01ctCzFtjs6nqz6gdalBLx4BxT++XNSo6XWjnseKjr/Ihl3MYJz8U4sCzLiBLgyMUGXOoFiFixNdGUFtlKnx9jgyMeam9V'
        'abS6S40gA/FPo6hCaykYxXH9/jKylzB+9HjyDILEzJjKmjM8XaH96Wg0SonA4rYv0u063bIAekjkjJzQNiYiTHg+pKQByKc3m6oj'
        'xxHjqxgeKzK4Aqbhx2bhepJUdIPSWAaz+8TpDhLfUJH3o9xx1/KlLJz81zbtnUVvWeKk5H1K6k4ILr6sgnFFsWdtPLnJp679NFTg'
        'YzSWfckJOAa3rQma7MIEOO2nsKHdlsRycYpbIuYKD6v96RzZZcUcKJp+PAW2YjJnC6Eb3S8JxhOKUOSlv/8Aj70G4kuI5r9bVlt8'
        'BQzg1R8IXYk5zM3cLTb5sjkBVDSrPLRyd1XNjlR9oCk4ptIjxX28hIruVTglyEjkuSSlPPNNA0+0jJaBpBYnnmpW5aMjPmrFM9Km'
        'wFtYB81pyEQRiDRbHnLSv1sYFN48qOpIX/H4NEW5EGSvaUhJPvTCVY2Q1vZUCKYMp9wDg3qCPXyJFi7QU5rlbXY1wSrcpOVfNT17'
        'tyS9tBwc+KXfSz4qgqM4qlAn9jiB7EsHrS3EeL7S8nvyaAuLTktYUpRBT25pQm6z2E4lKVj5Ne7beUuydiiAKTlDNsR2MoujGkOO'
        '62pLhdJ21VWq/EBMdSN/jtUvJcLY6jZJSaNslyiIO5YBVULt9bqU8RKHUtlh3K3KdU2kKI9qzEWDpSVoGQAeK0R+7l1ohHCKWNFl'
        '50k4zSkzMBuM+BZNtWZCcEjJqisbKGkhAFGfQh3hFeFRXoqwQk4oHDEWDD4gRnJjJchK48VIJs4MwuOqwgK7VZsvb4JB74pDOZfd'
        'bWGgcntR+Pl+LRi8+MOAYdHdZYjBthQSaHWla1FS3M1Psxbm24c7jzTmBGmLH9RJpmTJvcFB+TzIktxzkq7UruGqUoSW2lc1QydP'
        'PTGiOefalKv09cJLmVZo8L4z/IwMoyf9RJty6vPrySea5KS5IXtBJJp85pCWyeEEgUOqC/CX60dvivRxZMN0DIciZj2Ikn299pAO'
        'TzS7oOI5Vmq19RkABTROO2BQ5t63fSI6sU/kD1FcK7kyqQE+iu8IoDgUoU7c00tQ39JQrmzaHGlYUhRx8V1EzOp0ZlNoAIOKK/d3'
        'cbGQSTQ6oCQQNmKd2C3sdUFQCjSMlKLj8VsajBdjhtuDalORS6+sIZUlDfc+1Ng4pSitasChpXQUgvqUDt7VRmRQIONjFjLZaAUl'
        'WF0zj3d4ILTv+tImZBdlqWrO0dqKyXVFQxipTo6jwLgl1WFS+qVECvv7klLYSEg8eBXNbYelbFGushMSGnbwTQzjE91fVIPKAkUj'
        'cJad3JOCKoXUpfc3BOBQUqCHXglI/wBKehqT5BfUIhXofSdNxWTivdsktqeJzzmgX7K4yAoK/Fc47DjTmQDms+JPU75X9ymM55Kg'
        'kD00Ym6RmUAqI3UqtjqnE7VprlcoBWN4B47GpMnjIx3K0zsBYlfZ75FK+VCqJiTCmDAWk1krLKuiduUqFcI1zuMN8hDqiPmlP4Wq'
        'UxieZv7Ca1LjFsHpkEV0tjKMELAzWcxdaPtOBD+SPNOYurUuepBzSD4+Wuo758Te5bKYjBfqSK9rXGaHATis5m6udW4UpCsih3b5'
        'Pdbzgge9CPEyHZnHOg0Jr9qkx3fSnbmmymgpPA4rE7NqR+Gre4SRVA9+oW9gNtZK+2BWN4eUTl8lJojjTCEkrSk1LXeHHlPHCRip'
        'tWqZa0Zd3AH5oZnUh6ignn81R4Xhqrcsp3E+R5BIpJYWm025eUFKdw8USYMNtwpDY4rObNqeSjUfTcCg2s8VpYUl5kOp/uTX0OIi'
        'qE8lxuzA23YDkv6XCQqiXLHGKslAwazS+3Jy36vaXuO0q5Fava5KZtvbcSrJIo8bA2DMdfYiebpphadyBQ0OzJiO7hVMHekCFdqQ'
        'XS7Ijydq8JB81mRMfuahcdTPL/cJXQ2R0qJPtXy2Lk/tx+oQckeaHbubYIKwCKNN8jlsJ2JqQpfcb8lHUCWtaOzagPtRUR8qTjBF'
        'eXr7CKQ30wVfFER3I7bZefGwEZANYQqioauWMVzHui8V5xSWTPW/LA3E819vkxUuWpLGdme4oyyW1teFOAA+9JUcZrtyNCdGlKSg'
        'Z80WxwpJACTRoiRw6lJPFHOwm0JC2Rke1bc4CBLjuvBJOcURHtjCfUeVV6deygISCT8VyckrjJ5Bz7Yrrnf7OqGENPHCQBX4utKU'
        'WyBg19guOS0HLeM1zXEW1I3d/NcQJoP5OztuSG9w4BoJ6ztuAlBBNM1OrUjpnIx7V7hxXVryCAPNZzAE74yTIe4Wh5DxyOPHFd7H'
        'DU3IIX2NXs+FHUzg43fNJmIrDL+VYxWfJqpxxUYvYgNLuQAQSnzxVCu1xShKNoT80VDSwUb20An7VxW048//AIj70JVj6hqyjVzy'
        'rT0Dp+pQ59zSqXYGmXN0Ygn4ppcmlMN563P3oS2yXg5lYJHzRIrCCxUwCVbpS2sEFNDx7JJCS4ME/eruM6xIjkKCQcUEFBvchBTi'
        'mfIIvgZCvRXmZSXVNkKSfFaRYboHLaEnuBU/NhSFq3dPcD7c1zYfchekpP2rVylehOKD9ivWUHrz/qMKyDkEVR6J1EiKwmNIXjHH'
        'NJ5t0Qs4UhWftXJuMiXjHpV3oseRgbgZAoHc05c9mVtU1yDUl+pEdxUELj8L9xXu1y0QIux1XIHvXK43H6tsZTlOeKc+TksBBRom'
        'QF60/cYizgqKfFAwLLdpTpShK8DzitPfWqclvcjv3p9A/bokcJCU7yOeKjy5eJoSlMPIWZk+nNNTl3YLlAhptWTkUx1fEVMuCGWl'
        '9NlCeccZq5uKQiO462nak+azS4LlPzloQpWM+KxDZ5NOcUOKwVmKhh8toG6mjR6KQAnbmubMV5hAUtPJplEgOycKWnArWIMxVqcY'
        '/qXleTVLaSytkpVz96CZght0N7Bj3NNItvQwrqrVhPt4oCIwCfWWoYdypIrrMEAgbGwo/avbb1vccKEKSVD2rwpAVu6aMkUIMIqI'
        'zs1tZks7kpSkD2FJ74iJGlbNwyKo9OKXHhOqfwE4qOuaES7w7LAKmmz+DTjjtRFh6MEelMJdwePvX03FtGEoWB+aXXNsynwptIST'
        'xQyrcW1BalnND8U45T6jxS3XkFSVFRpTKbnOElLe0DyaLhPuMo4TkCi03RsJIUgdu5okUK24LsWGoFYrqtDn060kq7U7kFxSdyfT'
        'mpKXc2WZhW0jJz4pnFubkhIOwgUWTW4OIAmoySW85fWPzzQ8qZGQrDeMe9By0uvcIITnzXyJakqUFOL3fc0gvKBjrqfnrupJKGgV'
        'E/4iuKJU9asj0807atsdIyE7q+uJjMYztT8dzQhtwis5QbnKY2h5srHvTByZGdT1X0gfGKU3iciPHBQjk9uKTKkyJDJK17R7VSty'
        'R1WH3abBLwLQTnPig1z1pcBa4oNuAXlFxKicV0jsj17s5TR9AyNwWdR6hTT7khwF5Zx96oGXo30u3AFTcFrqOerIHzTFa2gNiVdq'
        'AXK0VVj1a3mm8JbI/FD/AFzDLa3HnRvA7ZrRZumlOIJZIPHis01nZW4UgJnK6e84B7VNkwvcsTKtRFN1dJebXGSjDYOM+4pfCuTR'
        'eClAZ+a9y9Or6anYj4WnGcGgrdYprzpLiS2gHk08KtScs1ykhSUTJSRwQKeypbTMcNoGVUqstvYZbJS4Nw8iiN0ZLpDi9xrKEIEz'
        'yqa9xtR2819W3dboyW46iB75r87Jaey003nHxTizyXIkElKAn71g4+4R5HqI4NlkWtZfkvqKvbPFNGHpCXUqbBUlRrkSu4Syhxwn'
        'ngVT2GzqPKxwjnmgs/JVam8R8d3uebsFiG22Mp3jkChHYKU2ZzpJGcU6LJkSCVY6aOM0p1HfIUBhUdhO5XxVXe4nrUkP22ahov4x'
        'jnFLsynnDkfxpmvUHVHSyBnxXuElDrhAHJpTZAehCXEfZit2b0gGlDmurccON7lEDI8V9n2pZe6iu2c06t0drpJ3HgD2pSZWa6jG'
        'xqtXJtNua6mQgrNFfSSWwOm0rH+lP3p8GEeWgTQsi5l9OWW8Z7UzgT3ADgdQBqJIUUhSkpz+aoIFlSGw666cfNLoCVIc6sxeB3Ar'
        '3KnvS3OkyVIaHn3rDj9CEMvsw+ShhoFsPZPsDStyJHS8l150hP3oSWktuDLpKj816VGU81hZJ/NMXCFFmJbMW1O10XbZJShJGE/N'
        'LpaYKGglvGfivSrcWhwBk+KJjwUpx1UbVHtxREju5gB6qc2Yq0Rt7QOCKHsUCXKuK20sqwT3IqsitmMwkOoGw9qPgvBpYUw0AT5x'
        'WHJ+Qhgvcn7xp2dFY3tJ5PvU6bZNUo7l7ffFarIRIksnqKGMVKXOK2wpSlPY/NcjEzGQDuaTAmubOqleUHtzSvWVoiakglt3HUHY'
        '1mWlNbvCSmFKXhGcAmtQtz7EhAcbdBB54NaGsUZ1b1MrvVpl2VITlfTSe9fbNMXMkBhY2oI7itL1FFiSoikulKuKyJ5xcS9lhgch'
        'WE4pTLXUYG/ZXO6db2bo76gTzgGhk2d9nKizuPvQ7z97YAWplWD2Ka6ovt0aQkFvcT4IrDymjjOzLUls7hE4zzxTRSW5EbaPQrH8'
        'a8wr68WAJMYN7uBkd6XTLiI80uFGQfApZx2dxgcqNT1brTKE4OpWMA+avrW4UQ1JcICjxkVHsz2lthaHCgn3o+Tdo8GIhTkhOPPN'
        'MTu4tjCb1KLA6DZIQo+o0mk2OPLQXArORnvS3UOqguMXIzaVgeak4usrg3I3EHZnkCtycj/GYhUfylE3p+ImXl0YwacfQxorZWys'
        'E47UrY1VbZzALhCF+a5Jni4reatpKloT+KWCWFEbhkKpsGVFnk2yTGUxKCQuipVmhORz9I8En2zWVwlyfqnFTJRZUg8jtVVZA7cG'
        'dzFx4SfeuXGwNwi6kVUD1LZ7ghY2ILifimumoKWYwXOSEqHYGnzXVajBAUl9Y70rkl1+Vh1tSQPGKY2RiKixjUGfJbLDzxWP4jtS'
        'yW08oFMYYPvVE1BWtA6aMoxyRXCRCjstnY6ep7GhVjCZF9yXZs8198KWvzT36BUaKVE5KRXzMpAIR+DSmZfJEZS2Hxk05HJG+4h8'
        'YBnSOh91zrKzweAKKVFnXCW220yUnIwcULar0wt1tvaAontWl6ekRPplP7E70jihfIca3UJEDnuQf6luPWi1xGErHXOM4oXSt4nS'
        'IX9WOrCf7sd6I1ZbrheNQpckMqUyT6MciqiM9ZbNAajPqaQ6eNpIqXGeR5MdSlxQ4rETV8nSpwjMRl7BwokYoPUlsclZbLuxZ8Cq'
        'uRcoUdshllvKuyk0M1HElwSnE5IpGTzVTJxAjE8Ysn27im9fpXEfSZMJ5Ta+4ANM9PWV20wQw6+pRHfJp4xf2nMpQtPHzSHWN/jx'
        'mW2lOJStZ5Oa9JMQCmzckLG7AqfLjb5T5OyTge1KmtHgvJmLeJXnNPdK3233KV9LwVJTyc96oymEnc4tSUtpp2PEn7Fs7GJYqmGW'
        'w3MCVhI7kVC69vsBD4at+0LQeSKYfqBe2GErTBcC94wAk9qy9tsOOlcgOkqOexped7BVYeIBSC0qrRdJVymtpeUS03zwOKrkR4dx'
        'hLKBh1HuKntISokZsNOQ1oT5WoYp7Ju8JgkxQPkYpKAKmzuG5LtqSt4YuGS1FSoAHHFKZFtvKh/9UXVJPYVeuXCGUh4uISrGcfNB'
        'K1CHAQthJCexrRk/IJx/siDBmoT017gP8TQzyFRxtU1/tWiw5tsubm1xaG3B2AppHsNuf5UttY+a75D7mfEPRmRsLjoTlTSs/atD'
        '/S1hhbbryEbVqODmmdw0vbikllKM/ApdZ2JtmmEpR/RJ8USuIJxtP2t9G/ULXMjEhSv5JFRcBUqzLWzlbZJrW27y1JT00pO48HIp'
        'LqK1xZCkANhTyzgYoj/UEAgyRh32dHlh1t5Xzk8U/h6tD0lLbzAKzwceaI/+Xz720tvgZ8Giz+njlvj/AFSXt76eyc1xwPV1CGUd'
        'XKu1BoxBJbJGRymkuu22F2srjKDDvfdmm9gjzkW/L6AheMAUu1dp167Q0tB/pk/yIoFwZCbIjGyLVAzOLddLs0g9VYcbT/dSS83Q'
        'OzVPLVnxitBtejHbaSlckvMqHINTuptDoMoLhu7G1nlJ7CmkcexEfY9RfYGm1zWJLqlIYPdVanaYrUiEtMWSVFQ45qfttiZg6VXF'
        'kKQVgelZqft0m42h7eiThsHI5qLykbMhVTKsDLjNsJfNpvNtZU04kucHasjtWf3fTU6dOVJdkOrWVZyT2qrtn6jsyEfSzEjd2yRV'
        'Faf264DKHUoKvevJVM2C5b9Mkn9HaZlFO6XJUpKewJqrDBbwyAAkcUTIgrgoHRfStHfg0rl3ABeUnkd6dixWeTGczcRQkkuxXy3s'
        'uSm1B0A+oZqdVbpt9kq3Bx1zwkeKvvoNQSyG5K1MNK7jPiqjTMG1WKMVtp675PqOMmvWxlj3qefkodbmVaetk/T85TzkN85TjgV0'
        'v18vE0GJGjutBXc4NbVIfEra4iGnB8EV7atNsDn1EiO2V+wFNYgdGLW6qYLp/Rd7ffROmBQjk9lCqK+z7DZmRE+iQ5JxxhNbFKYe'
        'dtq22IqQCDsGKwnUMZ613h16fbnVuFXCinIoTSiwJq77MBnaiMgISzAPHBATXqC8ZSwhyCtAPfij4U1QVvEJCQeeRXu5XOWmMVsN'
        'IRgc8Uqmb1G2F7MJXpyG80lxB2nHYmhpVjjJaI+oSOOQKFt0mfJjKK3T6hxjxXXTMPrFwuKccUFHJV2ruB7Mzmo0BcFgRbTHk+lW'
        '9Y75p7GU2lJLSc58Ckl3tTrElbsdJKlH+IolhM2Nb+qpOxfuqsCX7h8uPqO0yXGxt2YJpbNcvGT6UlBPAr1aLqXpKPqmx0hwVVVW'
        'r9suVxUlrKg0M4piYbOot82txfpGJLlLDcmIEJ/zx3p1Is6Y80P59Ce2acsOsMqOMICR2rkt1EiRuc/8Q7D3r0F8VQKkxzMYEXll'
        'JWgFCR/dXeO844gK6m8V1lPR1I6SUAp9hXNyVFitb3VIZbT3J4pxBB1AXrc/OOjqKbcXsCU7iT2xS4XOA6vYiY2sfCqhv1I1ql2S'
        '1DsrvVA/8ikjOfipD6aVNQZFvS/HlJ5W3zhX2qTJ5JRv6jlQN1NM1FquDa3hHTh1xXbFSt01DOnDDbKceMHmp63Wm9y3At6G6o5/'
        'kuqWNp+XwpWGTj3qfJmLauGmP3UX3GTdH4KYuxwDHJFLIq5qHkNPR1vhPZOOTVUuHcdoZaktEj80XYESI17ZclBBSnudvek2K3DI'
        'N6k8IcGcz1ZMYxHUq/jjBrSNN2mx3KI0wxK2PJSAcKwa4a4iR44bvbLIU0tOFpQM4PvUTHfRFuCLjHlLYVnJSK1lQj6zlLq25psj'
        'RVxZypq6uKbHYKPipTUVqej/ANNi4es8LxXiV+oMyYExHJjcZkdyThSv/wBUn1Be44sb8iO8HCrjcD5pKofceWE1q4QZJeU865wB'
        '2oGYu6vRkNW+ChpRV6nFHx71IPa0mT1GVMUIsXHoGeTTSz61hCC447KSFtjvnvQ/LkE7hjMdXKfPj29xCHgp9Ccce9ZncNWamgPL'
        'C5AK85AJ7VcRdRWW9211bRIkJPf3NSi9LNXacZMlRSCc/wAqYc47MAYD6h+lf1auaNrN0QhQHGQaqZWs7PdGMPxEu5Ht5qbgaEsJ'
        'UN6ySO+FUdI0/py3NDc66j2wTWjyvyZ/xaG480zDs94Q59QwhpYPpHxTNzQ9leaUgqwFfNQRm2OMR9Pc3m1A470YvV8SOyEiS67t'
        '7nzTVzj/ANYBw/3PGq9Iossd2RCmDYlPCKS6c1FE6SYTzSY7h7rI71QQLlEuq1pW6SSMhK6DuUW1h8BUVpS/BIxmgcBz1UNLToxM'
        '9eWI9xcaaUJbh/jjsKX3Z6VdZTUQu9EZG5INVVst1tMkLdt4bz/cgZo86asq5P1WHc5zu7CiDKswo7T7ZdGx247QW6pxKkg04i2O'
        'DZOo+ysh1f8AbR0OdCQwltKshAwCDmlmob5AthTIUy7IcWcJSkZ5qtcuIbAiTifomEpiLdPWeCglQ7CuTqFdVLTeD+aQxtaP3a/M'
        '2BpgsF3+avKBVg1p6LEUh1Ml1ak99x71vzsw+sDiFO9xS84WDtISCKkdRNN3OR05S5Cm/wDBAIBrSnLXE6ZecGw5yVKNcAiEkZQl'
        'tzHkYqdi7aZo4cRsCZ7a7VboikJhWgKV5UsU/ftkwNdVlmOj4A5pxNVDyE7EpPkiuDaoSnQ2qUGsnuVUs40A2YQdvQiKTEfZjBxx'
        '8Ao7gV5hMRLmhUdClBwjnJqscsCXob7hWHF4ygZ71L2KFIj38qRHWpKV84HGfak/IhsL6h02iYy0xolMM/VSl7vVwlVPrhZYLmNr'
        'QSCO+KZLTPcShX9NOTynGfxQ10uarY29InMttRmUblOKOB9vvSWDse49SiiSL9mkRmZDIk7YxBUrqH0gVj97n7pjkS1/1TuILoHH'
        '4qn/AFG1hcdQluFDH0sBfKkDha//AMvj4rlovSE64oDrbBTGBwt0j+R9hVKDgLYybI3yNxQRXpjSqX1/VTkLkqByQT6Qf+6bXiwS'
        'LjtQt2PBho9z/wBVU6jgu2iChuPlRIwltJ/3NIrVZF3i4J+ukOLQjkst9vzRgHJuDxXH9R3JObap10kb5c9tptHG1J4SK83TTaos'
        'BBtz65alnkJHaqhyP9GsOTkRo4dVgb1dzTSwX+1tIdYMmGpSTkhI5/HvQ/KCOpvwm9mSZfk2DTraWoy1yFkZAT/rT7Supm5cNwyY'
        'LjS0Yykg81T2282WW8lLzTYbUe6h2qpj2O2Oth6P01JI4KU96W2/UYor/tJFlTbjiZMZwpaI5T5r9dCy83tO5agM9vFNbu7HtqlB'
        'yA6Gx2WlHBqOd1uyq6FlMIttJ4Soj1H8VoycRRWccfI6adZNgXKjhbTCdiucngiuRs0Vlj+vJQhQ7iqaDfWZcUOxkFwDugJ5H4ry'
        'm72KaVtSIWHU9ypBFYuR+yJzY06iyJIsNrSl5SC65jBUOxo43hiVt2xWAhPZSzzRi2LH0UlxLAbVyMivjka0tMJfY6DiT/FIHmgN'
        'n2Ya69CPtK3G2vgxVIbL3jCeCKc3CJEcj9NbQCVeBUNBvqbfIyLUkp/zQORXebr2OQplTfRVj0rXT8YQCmETkLk2pg+sd2lYRl2q'
        '1CSp1ewFSyUo47kVEWXVer3pxSuFGlc/xUzjH2NaHCek3G2uRpuHGXsKStRwfxX4Q48RCkMhA2+w5NVjCzbBoRPIDR2ZDWSHPt2r'
        '273OYDTbiiVq/wAc1qqbgxKbS426lae4we9RF61BaWHhbH3FOPrTkt4zgfNIILtyQtX7VJbYSVf00rTn0+1Lf6mhuFxE0HWt6jR9'
        'NvJklY6g2BKP5H7VmMfV0y3RRGtNtkOD/N4k1eWqDcbjBC7uphzHYhPb5qJ1rdLbadhiZX1M9H3d8bh7Jz+TXBSdtBLBejGNrf1B'
        'c4pUsBt5zsAOBQbkFyAtS7rdk9cHhlpWT+T2FKrTC1DNt5vF0mrjwW9xKFL2kgDOP+KHDcV5lS9ykodAO4KKsjjgfNZ9Rqplk+5Y'
        'ufqK9AgfRISiQ8kYHSUVFI8ZV2zQemf1IvMRqUmPGgKVkrW5JXkA44GBUDPS2wr6fMl0DlSUs4APyR3rn10RvWh8JK+wcTnb8fal'
        'nGp9Qw1e5b3L9T9ZKDTka5W4ylKJcZjRwtDTYHKlH71O6u1hfNRpKp9weU0kANtYCQSO6sD3NKrgqdJltNxnSFrSG1lgBO/yRkeK'
        '9Qo7blxceODGZUSOSQrHYc981wAHUH/Y30fHtrctpN7mdJo/1Hcn1K/9Qa2dv9QdLt2n6OOhEdoI6bYTxgVjNwgO3Szxy0hDqWgd'
        'y2SCVZOefapxm3XMulpoKKQecjtQlWbYhoQuiJrdxvum9gZMiTKczncHCD/qKUp1dcYCnTaXEFnd2djIKh8bk4/4qATEuEeQkPtF'
        'CSQN3irJ+GtiNHbcSpIQd2QeFE9icVqox/qNtWFATuqM/qaNHgJdjqeeWS0FcHI8VLPWN5l5SVsrQpCilacdiO9NLfO+kurMmOW3'
        'iydzawrBRn4oq2XnrXeQmSFltxZVvWngqriqt0YCMVNGLbHIdjSEtLG9kK5SruK0nSNzdkyi3FecZaT2Ge/xU5KgREPGS2jCFDPp'
        'FF2qPJiPJmQlBa++w9jQBCNAxho7mqGE5JhqL7u5OP4kVi1zsIjz5TL6VdRDqghWO48VqMW7yI8Nxx9h1oEfxX4qKm3pd2nFxcZS'
        'HUKKDhPpUB2OfeiVaP2mFrGoqswk2+ciXFWptaRhQzwofNalDvGnP2aPImsNl5SdqgEAlSqzWV/TX1VJIBHZI70rfugbWDKAjNqO'
        'UhawCB70fEVB5WdytvVx3SXlJgBqAcFpYTyPuK+qhxRa3RDmBThIdQd38T7UFaJrD6dxmNSIykFIwoEJ484rhJRp60xjcZtwW3Fd'
        '9GwAnKvj3ouAGhAJJ3PH0t9uryG4cpKSR4TkE/NOHYFtsUBhd/SZ0pxYShKW87leyRXrQpQnr3JcwxbcTtjIyE7k/wCRJ5JNO5dw'
        'jPXJLjR3ttoIQVDJOe5+KsxY1Cg1sxDFuRE6JnW/pqbSppS2wNzeMKR7VnGrZ1yvMpyPbpK2YzSuwyndjuc/9VUyr5GXcTAeCEur'
        'STwgZwPOaSqlMBpxLIQ4rJAJrz/P8vK9Y10fcr8bCqjkZ3TZoj2oLddX3EoU0yOruGQ5hPGa7S7nanpSUsJSAVY3bdqRU+1dJpn4'
        'fWlcRJHCk+pGPHyKobncdMft6di3HXFdmkt81nNCeQmEP1ANX6gcQI1ntyyI8lYbdUDk7BgrP2wag4MCTrbVy4qZWYkfCErSnAS2'
        'DhISK66tmrZltPsMrZaMd5tslPkjPj7UgsFxuNltpftra1rlEb19sYHA/wB6pVlKj+5HksORLbXssTrgxp+2upXAteEujO0OLHcf'
        'bxXVyHCRCR9I4242BgBKzwruQPsai7e+4CtxQWtSyVOKxnk9+aprQno2taG2luLXlSEo7jNHw1qDzs7i+TIhI3olTXIru0dNTSco'
        'SPKVDv7Upbt8uRKZVFQzMYcUB1UOBXHkEdx+aef/AA/BlsuKfedblFzbhSgU/ArvHsUaNAcahrQ5LdGFvKOAgZ4SB4+5+anKkR4c'
        'dQM2i4tzMJiAJ2FLbCXE7iTwPTnPvXd+AbdZH2piXGlrQQrgjbu4BJ/NPbBpZaGkrklHXB3h/f2542mvWpH3/wBmK1MiW0XekhDi'
        'CtLqs5PbngD/AIrKYVqECjdHcD05oa62u5x3bfcQ9DfTlRHG3j/SqOSiNDuAgyOg286PSpR27zS61T5kWI2iKp5hG0FEcoyEK25K'
        'Se+OO9T36hXBL8yKqW6pT6E7wM/+KubKlTceLJ/8lk8zaZDwtUh9nrqGQgqBrhqZj6NhorXuSRtyKyyG0zKmJlIdWwsqylwrKd3N'
        'aBqC+W/9tjNvumY+kcpJ2DIGMkCs5m6EagrZg+ldM3e8vPyrYy0kxRu/qkhKiQTt+5/7q/0bY7LOiiXNUxJJB/oBso2q7EKGe4PG'
        'Kd6Ziw7XZmIcB7ekp3qWOeoT/dmkU579g1I9Me2twpS09cKGOmsjAc+x7K/BqEcgtCOaibhV1VHW45HLLTTKAEIQgABNTj8d6M5v'
        'ZfK2WV7lJA/7oy6TlmY6WkkgnIO3KT9vilQuLSlKYWr1Dxu4Nep44HxLciyE8jGn7862gltwqSSCRnNcLdekzmltrWllXVUhxoIw'
        'UjP8s+2DSR11BfWhJ2DuU+1MYTwWltlhsl1whCEgfyJ4p7gN3FAleo3VACLXLmpy4xGaU644oYxjskfJOBWV3GEZzqVLeJeAy5nn'
        'jPitN/Uq+NWawMaOguIcluJDs9aedo7hP3Jx+APes9iJTHYW8tSVLIwnPck142XLeW16nq4sf/i+0UMIlwv/ALJ55tCuSEq7/emT'
        '12/dVsR78hqU2wNidg2FI/Hc1+dcbbioSTtP8R4PyaCUx9VIQxDbUtRWASlPAOcAmhXI5NA9w2xoo/yaJEkxpkVpuAouRmUpbYTj'
        'BBPcY9x2pg5GlRgkuYSsH8Gp/wDTmCxbrw6iS8eqEEqC+EpI8/gZpjqnV0V9pcWCCGAcqfUcFXyPivWQBcXNzv1PNYk5OKjUSXW7'
        'D97XtilCkf0eqfOeTx7VzbkJ67Uk425II8ZFG6JYjahffeLqxHQemF7Qd6sePgUHqq0yrHJQxIQtLDysNO44V9vn4qTPjc/eo3Gw'
        'H1BnFKlzZi1AhuOScnx+KOgw22VqdbQXGV8lZ5UmiGbbGMRDbMkhISCobTnNLJ70Ric3H6zqtiN2As5z9hU2JlY0I1wRsw68x4zl'
        'tdfKOoI56iB3Tke4qeShmLZEPIirKVgf0yArAB4OKcYTIgvNjLLC0FK1K4wDTWxRppQBa1NkMjG4oCknjt/p5qvGxC0BJ8iW1kyG'
        'a1DMjpMW3BnovfzzHA2fAo+EsEJW44s7hyltWCa+6wFmZkpcCkRn1LKXAnhO/PPHjmvItFyQ2hyMpmWHTtSpCsk/irVzaoyQ4jeo'
        'Y2pqU7sDoS6kkAFr1Eecn3pta4iFvp+naVjOSpRIQAP+TQdm0dqB8B6UWmEjJHVc9WfsKdxIM2OxumupBJ2oaCtwUffHtXMV9zQr'
        'eo0TG/cFpKkvhhH8lfxLgH9qR4HzXO7dB2WhbrxjtNp2tNIHoSPNKp8qc9CmN/WssNZDaXEnb7bgKVdKE0lDka9NvLSrCklWU49u'
        'TUvlZKGhKPFx73Kduy4WJkeWhxRHCVAgEYpff9OO3fTj70uFFU8x6k9UhLqh5ShQx4rq3fZyo/SYagvD+IW24CrPsR4oWc7OS3IZ'
        'VHckuJaU+6rbwykDP8v7s8/aoLTlYuXHkVqZ7edNSITSXrc8JkbJ9KTlaAPcDuPkUJAlpfUliUwVNoTjqjuj757iqdqbbwtyRHfc'
        'StSBlsg5P57fmuD0iBOLjKoDKHVpIIKv7scY7eaJcoOmEAqR1NhuT0Kw2ZZtyG2ZTje6O20lSm0487TgDx7d6hDef1AuoePTtEqN'
        'y24y43sLgI5SD9qNvc26OSXC250yMNvtpa9KxnGADz+a6fVqjkwWZDLCdhDbpb5SsnyngH3qvGEv7xDFiPrAbDqadDtr2nHEstXK'
        'NlCHHVBzYCeAB/eU5xT226ZW5FRIvE9U5e3BC46EEH3ygA5+5rPxaYq1Jb1M7iRFeWoPsu8yEKUTuUMcd/BzSy5zV2eP9XZtTPOR'
        '1ObQ0HFbx+Ox+9blfM38G0JiJjX+QmpzrDZUtqkuSnmEJTlSlKGB9yamlaps+npDj1lfN0uGza04pva1HPlX/sah2F6m1SyoNrmz'
        'mmQVYWr05HsO2aKY0xcgwQ4Q1JIyGCM7T/7q4APxmkXmIpmuNVUBsCc1POSZr06ZJUVOkqdecVys96d2ez3fUiQq1Q8sNnHVcOxv'
        'J+T3P2zXTQ2jFypDkvUCVOrjLGIpyEjJ7kee1bNb2X0xG0NMtJYxgJHOz5x2NIdAp3uV42LDWhMP1vpS+6bgs3K5pEiOtexwxsks'
        'nxn4Pv71WaXtKLdphq5lCgqSgLwoYKQe1aDf4kibYZVunsh5LrYB2+k8nxWZ3f8AUeEy6q3ftk5xplXSXubA2EcHAzz2+Kq8bid/'
        'km8gEH/Yl1NdHkXRbW1SUKT01OAe4OBn/ShmUdVgJcAwRtUCODQki1tXxxy4266ber6lMupWNhHkgJIA/NOrDEj3CQzATc2XprmT'
        '02ORwOeTim5By/2TpY/yFaSujMCYiDGaDSEDgDt8mtJclQrzbxHnM/UMqAIz3B8EexrNp+mpVmmIfUhxzKiUqSRgDzk8Y/NeHtSJ'
        'gpLarnCYA7bn96h+Gwf9zVWPyVriwiTgYGxKe8abtSGA05cJo3j0lK9pJ9uKCg6assWE86w6px1hO91xTh4T5J/Galk6mQ671FTV'
        'vN9lrQrClJ+M9jX6XqWKzHfi25+SW3lBSlukBZPbBPb/AEpX0B0NQzyIG9yluETT8dh155hTqFjAStw+on25pl+nK0LhriMxmmo8'
        'd3A2jBwrkjPms0mXRc5aXn396kjA8JSPYCjGrxcbC2iSxIVDL38UrAO8DztP/NF3sag1Qo7MoNVWXTkm93JTtmU4+paiXQ4oAnHc'
        'eM0bpv6O1QwhloJJAHPgDwKiXdWrfUouuJBVyceT713iXmTMeDcVt151R4J4ArgwrcJVFzQ5N8K1NpZGwk4Ks8gUJEjyrnJPRdWy'
        '00561dyRjJCfk/8AdZ+L+uPIKHwQpJwtJ4NUti1rChQ3EBaW3lHO5z1A+2APIoWcmbQ9SK1jcbzqO6uWyJb5TMeKsoRGbbyUYODv'
        'PbPvXi3aHuim+o7MEfAyUIUXFD77eP8Aen921QZAWzGXtZWdy1bACT8Y/wCe9eGdSpj28sb1lZPp2p7/ABSxZEwqt/sVi3PxUKSi'
        'ZMcUOMnCR/zXZE+7FsRm7o+ztBHHBVnwT5rsbbepu2R9BKSwU56j+G0/fKsZoyzQ1Mkl1cQrCwRtSpzjtj0pIA85zSSRetxgMC/Z'
        'rgxHQ9vLzTh2ggcgnvXNi3rU6ogOKW0QQrGSD47VWGYt4Nwmf6O04SQFBKzjkYIHjNWGlLXbbLbnrq8UvTRgodeALbH/APCeTUhJ'
        'LdVKdBdbn//Z'
    ),
    'white_throated_sparrow_11.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgIDAQEAAAAAAAAAAAAABQYEBwIDCAEA/8QAQBAAAQMDAwEHAgMHAgQG'
        'AwAAAQIDBAAFEQYSITEHEyJBUWFxFIEykbEVI0JSYqHBFtEIM+HwFyRjcoKyU4OS/8QAGgEAAgMBAQAAAAAAAAAAAAAAAgMBBAUA'
        'Bv/EAC8RAAICAgIBAwMBBwUAAAAAAAECAAMRIQQSMRMiQQVRYTIUFUJxgaHRI1KRscH/2gAMAwEAAhEDEQA/AMUIcfdBIITRu3xm'
        'kEEgE1DdcbSsJQKlxHORzXiUIPiILEw2NobAAGKwU5xWLawpuo7zmDjNIvGRiSDNi3OeKmxHv3eM0HKzmpUBzJwaxrT0OowGE1LJ'
        '86juulNbE5NRpXGavcSxmGYJWYKeyqs0uYHWoK14OTXinuOtW78lZwXEnFwE5rJhfjqA27kVvaUPWqYWGNQ3He8OM1tU/joaFNPb'
        'Ris0OEmlMm8mWAwEnKeO7rUuK4TjNDGzuXU+OMqGKsUDG5DtmGWPwjFbFqwOtRWF7cZrGRITjrV23k+3UTiQb0A+0tlXKFpIUPUG'
        'uVe3bSjWnLuy9A3phyhlGVFRQsdRk/auoJ7+G1rOdqQSSBXN2t7VqzXMW43xbqBCtzixGQpBSqQgHkgeXH51P0my31jYxwv/AHGd'
        'cDMsX/hzlWtvRaBGcUuYFYllZyrPl9sVbzU0FHBrmH/hzuvc3C4W9RwVtBafsef1q9W7ihphTjrqW20jKlqOAkepNa1qObSBFlcw'
        '9OmtoQpS1BIAySegqrtZ9pJi3NNm08yqfPd8Ke758R6YpO7Qtb3DVF2Gl9IqcfCztceQSAr1OfIe9N3Zzo+NpeMZDyhJuTqf3r55'
        '2/0p9BRNWvGHd9t8CSQBJnZ92asNTf8AUOqii4XdxXeJaV4m2Cf/ALH3qz1tt7fwigUGZ5bqIuzE9315rPblNactEMMncH3gJAIF'
        'AMYcNFprneZJNQW2tznSsvl2facqRT+uBXkHJolAkE45rRbbHarqyXLNqJhb23chiUju1LPkkKBKc/OKjtJlQZiosxhxl5s4Uhac'
        'EVs8nhWcYbE4LGyItRbrS8o7jX1tfStvB9K2ONb1ZBrEusIMaFmpvxVLiJwqtSG9oqRG/FVBl7GT1kvdtT1qFKfA4JrfIOE0FmrO'
        'TzW5wKQRidiZSHhjrQ92Wd2AaxU4SDQ9e4rJq9yKVUYnZhNqYR1NTmZgOOaWFuqTW+HIO7BNVRxhiKLbjcw9uA5qewQaXYkgYGaJ'
        'sS0gDmq9nHInCz7w21gVOYdQlPWl8TBjOa0u3UI88V1VJMYHBjO7MSkdaiOywfOltVz3n8VZtyytPWuPGZzCzCUySpQxu49PKg8x'
        'wJaUgpGwggpxxg1JDm8VGkI3HnpVyvjBTkwHszOcYMtWi+05zflMdEhSFgcZbVyP1FNetNVuaqmN6ZsTryg4AnCCPEvPO4/ygedB'
        'u32Ey1qhl5kALdjjf7kE/wCKB6X0xqx62/ta0xXUQnVBtx1vHeFIOSQOuOPKvQKyhA50YxG8S8+zzSMLSltUyjD01zBffI5J/lHs'
        'KaCvAwaEdnSZrlhbbuDDzbyTgF1JBUPXmmZ2IMZKaw+TySbCDuSUwcQeCtJ3oBrBc50EhWcCp6UlAIxxUeSw2tOQMH0qn3AMUVmh'
        'Eku0UtzOcGhLbBSvjyo5aFYxmqXIr7DMnE5DgXG/6ZuhiTDJiuIPLTuR+Xr+lXTortKs9/is2vUiVqdawhqSCO/b9gf40+x+2Ksa'
        '5aRtuoIZjXK2MS0YwC4nlPwrqPtVQ697BrtbQ5c9JvmSByYTqsOj/wBi+h+Dg+5r3NfKWxerjUgMD5loLgLYiiXDfRMgk8PteXso'
        'dUn5rUzMKVYNUNorXmoNOXD9nze+bcZUEKadSUqyDylQPX4NXdZLvZtVtlcVxq33HI3NE4ZWT/8AQ/2+KyOb9F75fj7/AB/iH4hh'
        'p9LnnUlkYWDnigT7cuBLMeWytpwcgHzHqD0I9xU2PNwBk15t6GRsEbheYRlkYzQOavk81PkykqR1oavDhNa/CBQSDIqQVViWRg1L'
        'Q14qzU2BR8izJzBMCyWeTxUNJ7twc0ZloKvKhb7OFZNFScxLCS2nsAYNTYzi1HrQqMOPaiUUk9BxRW1kmJYQiXCEcmhM10lRweKm'
        'urwjrUFTZVkmrFFENBNLbq9+M0VYdIAANCkoKSTU2MeQTVt6wi+IxtCF2ljbWDqyTWlCxt45ryW8mIyt2SQ0lCCs7zjgDPnWY/bO'
        'FErnJ8Sgu1yYq669ditc92pLCfngfqa6A07BattliW5tAKGGkt4+Byfzrm3S7xv3anFWpJUH5xeOfQEq/wAV0o28pKsmn849VVPx'
        'Hu/UgQ8zgAKFZuPpPFBEzVg149LynOTmsZaSzQ0tzqFHHE+1D35KEqwTQxya6nICsih8iYtS+atpwnJyRHeRGRpxKuRU6M4E8pNK'
        'Ea5d2obumaNwJbbwylWKv/u7KxZ1LPW9FjnYNvHkK1OPh5JAbGPLNLzrpYkHvXMkHrRG2T0PEJGMV5O/6ny739uh9oSgL5i7rns9'
        'sGrY+LrBH1AH7qWz4Xmz7K8x7HIrn3XmhtYdnkkTWJDtzti1ZEppBy3joHE/w/PQ118lA25BFQ5idxKFISUqGCCMgir3D+pcvjHx'
        'kfaMPXGQZzh2ddrPeoZtN9jpmR1YSnf/AAn+k9Un+1WzMsTM+L9VYnFOqxlUZfDqfXj+L7flSX2raAtOnHBrKx28NICz9dHaQCnK'
        'hhKkjqnxHkD1FUPC1vebdfXpLb60rCsKClKKuPc8ivWKvH+p0ixxg/3H+YrOfEv2RJcbWppYUFpO0pIwQfipFvRKdOUtqIoDoLtT'
        'sWoVts6jjpMkAJ78YDo+/wDGPnn3q47TDtkiL9TbXm5TA6rR1T8jqKzvqFb8Gvsq9l+4/wDZIJ8GJgadQPG2Qa+UkEe9ONxgNlBI'
        'A5pdnwVhJUgdKy6bxylyBJZcQFL4yKEv5yRRKWh0ZOCRXtutL00qcV4Gm0lbizwEpAyST5AAda0a6ygyYk5MFsBQPAJojHS8EZDS'
        'seZxUzRGnbx2gSAdOlVosLS9j9ydQFPSFAgqSyny9Nx4Bz8VcFv7KLNB75/v58iSptaEl2UshO7P8OQOM4HnWtXwXYdm1OFf3lKh'
        'XeKwa3Ka8IGKsdzs3t9pP7S1BeVRrVAjZlSFOcvLySonIxwOmMemKS9PXfSWrL+7atOM3bCGy4h6ZsSlaR5gDmrC0BNwxUc6gGQN'
        'qsVHckhsAZp3naUKSQc59QeKX5+kZKlEoWr8qTcysMQG+xkSJMUja42spWDlJHUGqq7arBeiy7qBq8SpMZToVIjrP4CeAoY6jyOa'
        'tUWS4RiAUFaRUDVsIv6XuUd5sjdHWcEeYGazg7VWhhFB2U4HiUx2ExVOa6TIWMdyw4oZ9SMf5roEKzxVE9ibika2LAIyphaSPjn/'
        'ABV8ssKPWlfUctaD+JFw9084AqG8tZXgHipUnwjFamGS4QcGgorkVgzStvKM+dDJKCHMcmmUQyrGRxXn7MSV5IrRDYlvOBF4sJU0'
        'Bj/pUuzx3++AGSjNFzbsKA28UVtkJCMbU071cCKLyXPRIfmubgUJz5ipUFAj8jrTVeoDce2rStGVkcLx0NKYXsUQrGRXnKaaQO1Z'
        'zmBeGVtwrGnOhYycipD0wKxjGaC99tGc1rclZ86sZAiATDK3FlOeCPzpR1b2caU1k+t25WVP1y+DLjnu3fuRwfuDRRE9SB14qdbr'
        '4iG+lwgKGeRTKbQG84jFO5z1r7/h11PZ0ruGlnXLmwnKvp1gIfSPbyV/Y0m6P7RNV6Jun0shcllxhW1xp5JQ6j2IP6Gu3ZerIr0A'
        'FLfiNI2udE6Y7QYGy6wUiUB+7lNAJeb/APl5j2ORWqnKZHwD2U/8y3saMC6D7UtOasjttTX24UsDBWBhCj/UPL5HHtTjcYYWyC0p'
        'C21DwqQQpKvgjrXLnaD2S6u0Koz7aVXK3NK3iTHTh1of1o6/cZHxRPsw7ZrhYyiHPcS8ys4UHhuQfkeR9xih/ZeNfl6CA39s/kfE'
        'LOZfadOOPFSh5+WKr/tgF0kWq16K082VSby86p9bajuDTBUkoUB5KWCR693Vr6K11pi/NIKJCYTyvJatzZ+FDp9/zquot0laQukl'
        'y6wnZUtuEGkOsp3Ng988vg/wkhYJ588dRiqTrzaVcivLj9P2JPz/AEjKUVmHY4EsS+9o1k7MtJRmlNkMRkIjttoGVHAAGT+p9TVa'
        'az7dL3qq0SLXpGOtpwgKkSEyAtTSc8AbOMk8cn86rPX13uupLglV9ZeRb8q2tNgqQQT1WUnI6DHpip2g+4sMBxuDBlR4DoD7jwaW'
        'UOEfhBWRyRngDpnPFaf0vj8iunHIs7Mdn/Ak3shb2jAhO62DWGqH3pepdWOuo3JwhbpLacDnCc7Rjp0q3tEQbRCsNvXY5TaUR2lN'
        'ulkcDOc+InxHIz60jtXBiZazFTkpWMHanKEn3I68/nij9lfhWXTYQxKJKRnY4PCVZ65HlnyHFahUARSncPzL/IYu30zaHDFwBg5J'
        '+SOg596KQ7lbJfgYksKdAyprvE7/ALDqR8VXz95dlzV7LuzDUoZQUSShSfYJPCvgdK8hXPWUiJ9UXrVe7cwopEWRhh95I4BSo8BQ'
        '+2aybFDsZdfjrYg7eZZjbMZ44IH3qHqPT0aZaZTQbTuWytIOPVJoWq6PxLXHuxZlOxXMlyIsoVIbAHVCs+Mj+VRJx0NFrRqGDcGQ'
        'uHKS+0QMgpKVJz6pPI/T0NV2UqPdM9+M1fnxOPux90R+0+3MFQPeOLaP/wDJrpSQgNpwK5YjS29MdrapSwoNQLssqAHOwOHP9q6U'
        's+obJql+Z+wZa5aIrqWlud2UtrUoZAQT+I+1Ty6S2GA+IDqSczYWStWanQYxAyanW62uPtpdSkqSRkcUVjQMLSFJ4B5qiSUO5yrB'
        'yI5ODt4HTitzcYdcU+2e1R3Y43BGfioN8syGAVtACrvoOy91OYTAxOW0lJIrJlSUGtstBSTniozKCV1Va7GoHUy11MszGC2tIIUK'
        'rDVkVFsuhYQvIPI9qeYNzEaAtchQSUAjmqj1De/rrrIkqV4dxCc+gry303sBmW+dgLJjkvI25qXZYT91khlvIHmqk1d2aQcqcH51'
        'aHZTLjSICnmykrFeg4yCx9+JmBSZru+lH4zSe6cJUfI0CuenLk1HL2/BTzirDky3H5eOcChGprghMZSCoJ4p1qVoSy/0ja13Fi2h'
        'P0yS+cbeSCaIt3WIXU9w6AocYoTbmlzo7pSrjnpURqxvl/ehzINZvIeyj2E4zuWz7o7d8iXGPPi6A1V2u+w21atS9NtYTarsrJC0'
        'J/cuq/qSOnyP7072SJMjykpWtQQeoNWNBLLMRO0DOPSl8SnkXXerU3UjyfvJOEG5xRpi0XXs01ZOTrIuQHITYVDj53NzHVEhLiT0'
        'KEgFR99oI8qPP9oMKSEGPCk3mWckOPIw0FeZCDnJ91Z+BXTmsdO2PV1sNsv8BuWwSS2TwtpX8yFDlJ+K51132OX/ALPXVah0+Xrr'
        'ZmV99vSnL8TBzlaf4k/1J+4Fexo5DOvU+YGRGHsyg3mWk3iWqMouuJSWlNhKEtn8ZSAOSMADOByas7VGsbA1aHLZIgLmSHUbEwW2'
        'xk7upz0A46+1UWO06OmC1/plPeT5DfgZCDtac8/kZJI9aYuzrTs0pN8vkl+VNlEbt6jgAnoR/wBir1RwMCCck7hHSmjpU58LYbVH'
        'jt8lAc/z0+wpzmaTiobYkXBEqRswoCOdyUkf09Tj06UdfkQIzTMFofSlbWMJxz5Z46Ut3h29wYMhcW5xng0DtYUgLJCRnGPMn/FG'
        '5ysJNHMG3zS67hKck6chxnrhEwA466ErwpIOQMDBx5UMvUy02k/s+beJS5IZ753u1rbUg85HHQ8HCfOilguDLTTVztEVlx10FbqN'
        'yu8ST1x/saXe0KKq6NN3y3SUN3YNkOKaQFFSOuxaDwR7/pWdZTkZl4cgqfbBhudtbkoU3crk2lfjDRcU44vCc70BOVpUMeZANMuk'
        'jqKW66krkxrW+tC/rLkERfAsEhwAkqO4jBwgZ8yec1DZ9TPSbi5EkhlErBT3aQW1Oc87ecYwOg5+aKO3SQ8hS4zri3WRgtrWd6PY'
        'KHIHtj70heoOCNxnq+oPMrntejx09o95LDjawuUtYLQO3n0J6g9egq/OyzsVi2e0Wi+uXe7NXBbCZD0dp4JZDiuRkYzwMA1zteHH'
        'LvrJKnXE73HkoKlnA69VH09/SusJHaXD03arcxLQm7D6dJfkQF70oxxnGPGOOqc/HnR2BiuAcTPKsT7I/Wz6ZDZbUgJX0Vx/epDz'
        'EcJKhikjTmuLDq5pLlgnNyFk4U0lQ7xPynOR9xTlEHeMeL0rGu5Rq1asj0TMGbj9M7tCuPmt0q4F5GFHNBZ0Jz6jKTxnipCUqab8'
        'fIrMp+qkWFVOo4IQNyDdWwUk4/KhsbAXzRqU33jRUg59qHtwFBzJz6gU/u9rZMF1HxFfUt6uMtK07u6SrqAar68S5LW5Keae5Mi2'
        'zYwe/aEVorGQlSxkUvybZbZZKBeoQUfVY/3rWXgOo6qmotiW8yr59zkofO9Rxn1pw7OtdyLI/gEqaV1HpWVy7O5VwcUIVztzhH/q'
        '4zQp/s41RagHXW4ymyoAKQ8DWgnGYJjpAMtl/tNaWyVMN+NQoHI1Qq4b++WQSOOaWHtJX63x2y8lpbq+e7QrKgPWoc223KOhDjkd'
        '5vHUlJxVc8UaBG5K5BxiPFqv7kNBQg5FHLffXJLraWUFSs8gUp6F0+/fLghpSilkDKj5n2q7rJpS1WdobGwVYzk8mncvg0XrlhJ7'
        'kTRCKnIgKk+MDNb4F2S0ssSAU46ZqY4lttzwAULvYhqZJU4hCzwCTjmsvjcY0DFcW7lmhlh5l5RWlY9etThdY7bWxTiUn5qq5Uq4'
        'Q1lLL6Ck9MrxShq+VqdhpExxa0x3VYStCspJ9MirlfJZT1CbhemfmMmq9D6Sjaj/AGrZ4UeC5MWFyENAbCsH8SU/w588cGi1iQWB'
        'tUncn1PGcVX2nJlyfYlOSVhx1qOtbAVnPAyRn4H9hR93UkhVmZQwyd4RjkYB8uD71q0WZTJk9euoYvcpc1SHGi2lbRxgdR/0pPvs'
        't4SVrYmqSsJJda3EHGOfvio/7WfLKlNbw6VkBOOpGTQG4zlvPpmuJTvUNrwUrk/FO75nQsq9IjKZmwWyFYws54A8jWDt9+seTKYU'
        '23LGQMJ6jzBz1FAUSjCdUuOA4wsctq5x7CpqHY7sQvR45EgDPd8FCx/KUnzoHwRiSrEHMhan05Hu8VTjrcdh8+M92MDPqkj/AL96'
        'X2Yl8SoWi6RJEp1pJMO5wz++YBOAXvVGfNXvVkWRa5UZK5NtLDZPBUnn4x/1oB2g6Ncu0SXdbJcpKZzUdRVF3HY6lOSMDqCMnHUe'
        'XFJagEbh9/tKbdUn/WD6ivaWnVkrxxxnninrQmo1CUqyy/HHdQpxo4Kg2oDJPAJAIHUA4/Oq4tDMibc22E+N+QrbweST/micVmRb'
        'bylyYp1hUdYUk7iyST6qH/LPorofWlEah1tiNGpbNDeuTd3tUly2zkq3tyYixhwjzSoEAn3Bz61afYPrHtAv1xftU5uLNaiJ/eyX'
        'gpt1PpvAOMnB5CevXrmqafubpccdUVrP4nXW2gHfl5n8Kx/6iefPJNFdMapl2CW3coMltonwpdCi5Gc/oV5o/wDarp6ik2p6lZUj'
        'P843WZ14Xmys7FlaPIkYz9q1z1FxkhAwfKkns81/a9SITDl7Lfd0gboyleF0fzNk9R7dR70/tJSRzXlv3Xiw5GIBYjzAkL6kvkOD'
        'ijTbCSATXjwaaG44FQlSyskJ4TWpxeL6K4Y5kE9vE59032ZCHDI1C44++o+Has7cex86nSdBafQg7IqiR/WaQ9K9qM+2K7hx5YZ2'
        'bS3gFIx5lB4p3g9qWlpW1FyT9MVdH4+Qn7oJOD8EVdv4V5btW+fx4lYg/EDTdKwI7gWwX2lJORtdVQa/NaiW7mFd30t4A7tSyRx5'
        '1ZoZgXtoOWW4xpoUPCjPdrP2VwT7Amlq72+VFf7qRGeZX1w4gpyPXmioHIr02Z2WiY/ctaKklxExaXC2lBXv8hTRbNW9oybeG2Vx'
        'JKW+u9AJNaCw4QQEHk4BxRayuvQG3NzXGMg586s+rapk92zuStMdrGsbdlx/SDT6Eq8TjScGnCD/AMQlkLiE3mzTYav4jtzVUwL3'
        'Pg3d7LmIziiSnPQ0cl3Sw3OOGZXd5P8AOgfrRC2xj7l1JOz4loy+1TR94YSq231EZZHKXUYNAJl1dul5YiR5aJJYWl5S0qwjHpVV'
        '3TTmnlK3sOIQT5pNDY9tvEO4tphXhCYh4WvOF49Pem9U+RCCgfEujUTV4kp7yPbnHmyTy3hX6UHRPnx7Q9BlocSwrJU24g8HyPNJ'
        'kRvUkJ4Js+sH28nhC+RU97WfaNaFhiW3b7sg9QpHJofSqByDgmEQcRhgsS/2M7IbUrvCNyUggFSfNI+RkfehLDr02SWmVrw14h4s'
        'YT1xj1HnUq19qYZfZa1DodbSicJdYGB/tQHXOotLy703M09IlQpbqsPMuICUBYGN2fU9PyoUqNKn5EWVPmGrq64WmS25t+mXvyk8'
        '9ME58zg0szXGi8sqfV3ajkjgj59qwbu8pKiHj3g/mBxj7UKuSnln93hIUepGR8UB5H2i4RiXBpqQWkBS0bcgKdwT64P6Zo1bZsNt'
        'fdPbVx3/AMQdUFZ9vY+4IpByqO4kfh2DKc8/aopmPYUppIUQsnZ5+tMW/MmXNp65RLflpqQkMFWA2rgp9vejRudlbcDrsyGvYrhb'
        'zuSnPUYyCT9jVIi7rdYSgvJVH2g7VJ5Tn368UYskpMSYl59a3jjwKUrcMU5bwfM7cRLeURdbtht0tobn+BaFFJA7zgg/FWnq+wtS'
        'ws3WTKbWnwtOOlB49ycEj7YqptUBLeqpim/An6grT7ZOasZWpP8AUSUtQtPRlMNtBEt2Ykqw7xktlCfPnr0zQDGDDBiNJjO2yaGF'
        'qCm2l/uylwp2+7a+qD7HKTW9lUJUzAkSIch0AFwRhlwejjOdjg90HP8ATTDfbbaURFuJQvZ1Kd2Nn2Iz+lK0bCpSIgYC46gVbFjc'
        'kj19vtSPUwYwNDNruK7etBZUyjunBsW2omPu/pKsLYV/SoY+K6P0D2gR7lb22LiCxMZAbdUo5SpQA5z7jmucG0suLTtccCgnYC4Q'
        'VBP8ocwSpP8AStKx8UyaOvCLQ8pp5lpbCklO1QKm8Yx5KJR9iR7jpQsEfcYGBGDOjZtzS6AttxLqCcBSFbkn71G+q2p8SsZ8hXNj'
        'urb3Yrssx5Kvp1K3pQhRx+R6H2GKsbTfadZrultu493GkdCQe7z9jx+VZ99VjbUzsDwphKDoCFdLW3CcTa7vE27mSvAdbHpnqPzq'
        'stZdmtjjzXosORKgyGzy05+9b+x4OPuaXbFr2dCUAt1aHACnPnRDSeow/qCTPvLj8iOpBT+L8HPXFaL1qo71E5/nK5yTkxdftWo7'
        'KVhDj7zA5QqMskJPrjgj8qPaT7TL9FWqDc56nY6cBKXk7h15ODxnGfKrGJgPREyYZadZUMpWnn8/Sly+Wy1XAYmQ2XFdAvbhQ+FD'
        'mop5jfxDEkZIk5vXlsvao8afFWh9KVhEmOdpwPPA8JGMEhYIrbdrNcxFW+0FrZA3pfZSdm0/zoPKOvUEp+Kr6Zo6TEc+os0srKeQ'
        '24dqvgKHB++K22zWWo7JHEBcmREej+JtLqiMjzHnnPtxVkXJdoHcEkHULuWh505dfSc+grJNjQ2gkvqJpjZvWnr7GYeU4iHI7oGQ'
        '+ynwb+M72xgAe6cefBrTcIEuO0l5KUyIyzhuQydzaj6Z8j7HB9qz7670+dfiLPYQHHtAKsd+v70ct9hbcAJeUcV5Di7B3klW0fyj'
        'rX0ibIWr6eCOfarXHDBctGKTJDsSFDcG2QO+6c0btUSM82lx9alHywM0BtGmnnZokzXzk84zVi2eBDabSAndj1rrxg9owNAjmmXp'
        'ja1RlbdySBzkD7Ul3HsXu084VJATngpGSBV6QG20EbQlPxRZL4Snrn3qkea/icTKDtvZPd7fG7tVxceHklaM4+9CL/pu72lpRlxl'
        'FgH/AJqRlP39K6OW+hzIJAP96Fz0k5SQkg8cjg0oMzHJMSQZzLIjsvtlCxweM1CXbg0AptfOeD8VeV87PLPcULkR3DAkKySGxls/'
        'KfL7Yqtbvpi5259TfdiQ2Dwto5H5dRT+roM/ELqYmfQuIecTkdys5A9D/it7SXW3ClJISOnpRCVGdaWULSpsjyWMY/OoiXHG3dqk'
        'Ak8DA5ohZmTiLOpubnuVkKKRnNPGlbReHrHEXCggtuJKi6rG05J5PXFJutIr7N2S3LZW04UAlChgj0q4uzO8WuxaFi3LeqZdHgtt'
        'qOcBLO1RAOPM9Dk9PmrII67EYiFjiDo+iZLLAmahfShbn/IiI4UR/Mr0/tX0PSi5DyzAtqpDpHi7seFI9M9KbLVYbtfAL1Md+sLx'
        'JUhh5KlpAPmOeKcLdb4jcYNx0kKT1Sv8QP3pF/qVp26wLbMaUaiazpqL+w2mZ9vgNTErOShA3bMdF8YznzqAiywIzw2wWEkH8SUC'
        'nyVFHQp/tUEwhu5TwawruY5MV6pJzBC7BYLlbXVXS2MPd3gIdbPduJz/AFDr9waU7l2ZWuconT9+aS75RrgA2fgODwn74qx3Lcsx'
        'VoaO1K+ooGqzuIkHKTg+dHTzzoRwcEbgPVeg7LfApwMiJKPR1oYyfcedVFqPTN70y+pMlsuRVHCXkcpV8+ldDyn40ZorfeQgD3pP'
        '1VqOAbe+WI6ZqEAlxKlDbt88in8F+QrYGxEVh8xA7OdRx7ZdmvqHSIigUvNHlKs+eKsiQnStxeabi3sQJTwKmkvpJZX7bhyk/NUR'
        'c3WZ9yck2q1vxGP/AMaSVhJ+a9alvBQD6lZQCEBXG0+tbxAOmEcJe83Tl5tbaHpcQqjLPgkNKDjSvhQ/zUS5WiPPihmXFQ6hQJSH'
        'E/3H/Sq70zrW82WS0Yd+fYQRlxC1ZbUfTb0NM8jtERedYwbhqFTbzMTaNjCNja0+YKQcc+dUn+nBn7I2Io1AnOYBu2kJUNwvWR5T'
        'oSDlhZ8XwFdD8H86laK1LNtUpTE12Q0ELwthSD4BjgFJ4Kc4znjBp6j6ptmo9ZMRrPAgssJQtTrMZgt+EdDyeT9qk690xbdTx2W0'
        'qVFlR87Hm0J3Hj8KsjJA9M008luO4Rtj7w/06i/e7xYbhNQy07Ktb7ijtBRvjr9uCVoPxuHxRO3xnIC0pfbCNwyhQIUlY9QocEfF'
        'VferDqPTcpAcbclsoOUOtoKgAPUdQaK2LXUoOqaEhtthwJK2lICkOLHmQcjJ9cZHrVsdLPcJJlqNygnBSelGrXN3YG6kmRqS1usx'
        'WosGO0XtrjjzTq1qRxgpIKikJ3Hr16UYtshJwpt1C0HgKQoKSce4onq7LiSpj/GljaDmpSp/hwDStEfWR1NEorbshAWkjFYHIXo0'
        'MjMJJf3KyTUxCklI73GPIedBW0Lbe8Shn0qW4RgHfzVQXntgSSFTXzBGppjjUxqNHUUNuqwfcVg/bpDO1YbG1XRQ5rVMX391Cdv4'
        'BuyelF5k8xYKJEpOEkYaQPxOH2Hp71ea8ugWNVWt9qeYr67sFqkaRmKu74YUpvDTgTle4cgJHn6feqbtmnL7a9OSNVtW6TJixVbU'
        'JdZUpK8g+MYIOE9SrGBxV0CMm53AXC+p74J/5UY/gQPLI8/j86PP3RctvuEoSljG0pI4I9MelLHJFI90K0VUrgbP3nI98vCrzcDN'
        'faWxJUBvSCCjIHUenxVu9m8G2XTs3MC5NhcQvrL0lrCX4q+Nqwr056HwnkGlXXGkYlpv7jU9DrUWRJJivx28DYedvPhyORjI6A9D'
        'UHQOpP8ASupnm7g8+m3lC2n2Sj/mcEJKk/r7Gtqi9WAdZXADHZ0Y2u6X1npWe0bVf48ptw/+VcbdKQ8OuMHgK9U5o1bO1nU9ldTE'
        '1jYlOIScBxTeFY9lUvWfVI05qK4aZuaGpdiLgcZacUSGwpIUgpV1GAR05GKtG2MMTmXVF2FdrUWQookvBDyR6oONrnzwfannv5TY'
        '+0H5xNTGvbDqSKI1mm/s+Y4sYXLO8J59OKsyDZYC4SVOvtrKk8OpPhWfaqI1VoCwuwo9/sy34DUokR3C2UtLUPTy/SsdK6s1Jo5a'
        'v2hc2ZdrSMOISrvAk+Qx71Xt4tV/61wZLVdfIl1uQe6UUY6VodhAg+EUrWDtMs94lByG9HYdU2U/TSPwFR889RVgMtocYSr6mI6v'
        'aCoNOg8/rWBf9KtrHZRkfiJYdfEpteiTNCv2hrCytD0+oWrP5JqGvs2kfTLRaZkW4jkrRElJKyPdKsEj86odVyvLZAVNeSQABlZG'
        'MdKIW/UF+hOsvmUtaWuoWpWFegPPIr0HpU4wMj+sYCfIMfZ+nJEAHv4MxlOcZcaUkZ+SKiIs0GUNj8dDgPnjn86FW/tV1ZDYXGF0'
        'lFhZBWgrJSrHSj0PtmcdKWrlp+0zwBgqcioSSB/UkA5+9KbiE/peC2SJgnsugXOK69bpzzExAylpzCkH29RSzqLs+1fp2Am6XSzu'
        'tw1KCQ8CCkkjIHr0FWlpftI0ZJmI+osrlvWroqPKVtHsQrdUbWcjRl+ubiJN/DLORujfRtgtqHAysHJwPXHNHRx7wfcwxOUH5lLs'
        'T5EWYqaypbb5GMpOMU/aU7TpUINt3lgT2VHJczh5A6cK8x7KrDXVksDlsiL05IEuQhXdqQkJ3lAHCjzz5Z4+9ITkOdDfw7EfbXj+'
        'Js8iiZM5DiT/ADE6IjzbXrNtDNjusFT6sfuJbqY7qT64UcKHwc+1V5qCwWuU/KacaZRLiu90+WSEuIV7jooH15HvVavSnS8pxs7Q'
        'fIeVE9OT24WoGpUt15TLjexxQOSAcDPPp6VC1qi+2FnUmyLbeLQVOsFUuJghWzOQnzBHp8UR0xq6RaZKRHIcjEpU6y4cocPToOQf'
        '6k4Pz0qxNO6ckXeCu4wLlaFw2wVOOLmpQUJHmpJ5FIOn7InWs2QzFft0CdF3kyVuJZQ+orO0HcQknb6c9DzTlJwPzBG5b2j79Evs'
        'JMiKhTTieFtKUFc9fCocKGMeh9qixtXS4PaXI05IjLZhyWgYzilZSp4JClDpwSOMeo96XbHo7Wlk0/IjBNkiLeJ3rfu8ZKRnrjCz'
        'uyBn4FMNqas1qabf1Try2LlIQMFkmc4OASjC0lJGQnk5/D5YpVvBS3OdZE7sY5NrUcuvOtNpzgqcWEj8zUhbsVCcrmRT/wDvRj9a'
        'pXtX1+3ekxoen1udwwlSVSnWkBa9wwcAJG3PxmkBm+3BktqdkPPBs4SHHT5+1Z6fSQnzmNQVfx5nQtx1BFtzbs2F3M10ud0lW3LT'
        'asZxzjcr26Dz6is9LyGp0VF2ul4YlznxlRU4EhkfygHz9ao+ZrGU9pmHGjrDD0SU4pfhSoOBYBCskHnjH/xFCxqu4IbCSG1jOdxB'
        'G77Dj8sVaHE1j7S6OTSi9Vzj+86VkvwVrS2zIYcUeRtWCP8AascYGBVEWztWvtshiOiLbHWkq4Q4yo++OoNSf/GbUja1ERLIoqI8'
        'P0pITgDp4vP5NZl/0ayxuwYTOuIc6jl27Mpd0IHFDJZmNqH3Ck/5pG7F0pu2rP2fcgmZF+jeCWX0hST4enIzjrUTXnaPc9U2pcGZ'
        'Bhw45cbW0mOgDO1JB3HzJPPl8UvaQu8yxXqJdoBSl6O4FArAKc+hB4I5NXaOE1VBq7b+8AAhcSyu03s7nsOLvlp3TWdoMhgDxNbU'
        'gZQPNOB8j3pIhX66wWkfQTH2kNZIQgnAB6+fSrhs3bDbkx0qvFgbkFThQl+DI7oJA/mGFJzn0A4oHqVjsp1LOVc4twvFikOEqebb'
        'itvNrPUnG5OD+WfSn8VOQntsIP5kDt8xi0PcxqiyNWS6Tipp1IUw26jaNw64Prz981I1F2cExg0ltC2EHIbCdo/tS5pO22BuQEs6'
        'xgFhleWvq4rrJPofCFDOPXH9qfHH9XS7hEhwb9Y/2UofvZEWc0p0JGcbS4OSRjOR9jinXU2OwZDGGxyMZ1Knl6EjNziliU7b5CEK'
        'cSF+JBKRnGfKlyFqi/2+aFsS3l+LzJOTXSOqNK26629ch24w47hTtwHUOlYxzyn8Jzx0qpdW9nsuAlMi1bJrbag7lg7jxz06jp6Y'
        'pf8ArofcsgMvXcZZTTLuQ8w04PPegH9aG6h0zpVq0/UzYCxIcTlITH2tpxjO8g8DB60xR5C4ExEpttpa0ZwHEBQ/I1GmXKdMaWzJ'
        'eU82rJWlR4UM9D6/FYtdgUbO4FDopy24gM6E03c46XUw344V5NvEf7jHnQ+d2RwiCqFeJDPol1sLH5jFWS2W0J8KUpz6V8tea79r'
        'uU+0xTOc5EqK26Ph2S/sR9T6kXbmloWqG9FhKfCnBjgg4CfU9aHa10/J03dRH+sZvDUgBbcuOlYStRzlODg7hjOPcVck1tt1shaE'
        'lWCEqIBKSRjI9KVmrUtFoat91aUH2nN4CiCps5459cfrV+vnMVGRGo4K5Mp9Znx31L7t1pR81JKTU+BqK5W+a1PZlqL7KwtIUcjj'
        'yI8x5Y8xVuurQ6TsCig+Sxz71p/YsGWoLft0Z0A5ypoGnfthHkTvUMrN9yNf7shENsIbSwXZT7bJOMcqO0cnHAwPWgyijepOScEg'
        '8YNW9fNLSlN7dOOwbaiQ2GpKQztOM53ApGSTwMH0pV1joONZtLuXFiTJkTm3El0/wqSTycdfvmmi5GAPgwzALVxd7lhmGpTSz4F4'
        'VtSpPvUht16IoqLiHCpSlneokH5z1oHb2pL6CWIryiOPCOp9B6n2ojG0/qabJCW7bIKW3MKDp7sA/Jx/amq28TszOZcnWwpjvnEh'
        'X4kE9OPMdKgGUCk5c2nHOVfp701DRE5UyPZ3ozaPqB3jMtCs9yAPEhRxzjjH296NyOx2Wbb9RDuf1SgtJVGbbIWScgkE8HHHPHBP'
        'FMPtEkZMrJMopVypXvXhU4ptSyc+g86vlrsJ0+IbJkXq6If2J77YWykqxzjKeBmg977LtKW2VHban3N8k+NDjqfH9wkYHxSWvUHE'
        'L02xmU6hWCS4SoYHn1xWMhwKO4FCR6DNdDQdDaPaSFfsGItQAxv3K/U1vkad08lwbbBbBjpiKj/aq7c4KfEUWxOay6StQAHPtXyi'
        'tRKiME+gxXUUbTGm32Vtu6ftiwv8Q+lQM/kKlxtCaP2DOmLX4RgZYBoD9SQeQZCtmcqKWspS2pZKR0BPSt6HAGCgbsk/lXTdx7Nt'
        'DS8pXp9hgkY3R1qaI/I4oC52J6XL6louF2baP4Ww4g7fuU5oR9SpPnIjOplLW+4SmWX2lvBLTwG5kJGwnjnHQHgc8Vuid3t3tubc'
        'nB9v+/Wrck9jFgSr91drmB5BQbVj+wqE52QIaCjC1AtHHR6MDk/INF+8KSfMDsBqV6ytba1d4ok4yhQVgZB4PvUpq7ym1EZIzzj/'
        'AGpok9lmoG421i7W6QB0SoKQfzINLd401qOyJL0yES0DgusqDiR+XI+4FNXlIxwDJBzDttu13lMOMol7AsgAKGQrHTnqOOOK3W/U'
        'WoobpkpckMlHjITztGSM+opLauDqUhW5XJqYxeXsEFZweOacLSJMvGYME0MeOKLy05zQqQk15ppTkcuEHGa2IcJrSUjPPWs2xiuX'
        'c7M13CUYraXUOx23wsKZ78ZSpQOcdOehOD1xQq4S35dykvyFsuKU4SXWVEoWT1Izgj4xTNHatUgxo9ygxpiC+HFIW4QshI/CBkD8'
        'uaVdSXKHNnCLbmGIwhJ7p1toHwrySckjkjIH2rRqrwm5oIoPH8fM3REI3grOfajWEhodAMUs21ZCsqJNFVOLWPEcD0qyeqjURkLM'
        'lSloWW0cjyJrNhzKtyzuJ65qOoA+VZIJHvVOwExbsWkruAHVuhRKVAYbwAlJ8yOOprUrIVzUhlYUjFSIcFcp3IT4R50zjOQ0Fckz'
        'ZZo7ktYSAdo607Qm0RI4TgDAoba2ExEYAx71pvd0SxGUAobscCtgv2XUv1oBszRqXUAjAttqBWfIUkuPOyrohxxRJzXksrkPKcWc'
        'kmtsQbJKPOqDAK24L29jgR1tkfckbhRJVvaICsZodAfCUpz0oy28gt5KhWdyGIGoAAM0NNtt9AK3GQlCOMUNmTEJcPiqKZY8jms1'
        'WZmxAbAk16Zhec1iLmnGCaBzpCs5BqCiWS9hSuM1pUVBhsQ1fIjeHkuIyFDNQpEkIPiPAoe1JATjPFQLq+padqFcedPfhARbDclv'
        '3RS1lKDkVlHdysKeXtFA47iG8+IE1vYSuUrepWxA9+tLFYrnB8TOXpPSlwkLcctjYWo5UptSkZP2OKFzezmwLUVMiZHB6BL2QPzB'
        'ppjhthsbVZNeOuvuqwE4FNaz/aYeiJveOa1oh9/Gfd3gFsZwfOp6m215zUd2IraQ2ogEciq3oMDsZlPEBKThVeg+tS3ojifLNQJe'
        '5sbcc0nqV8zguTIV1eHdONBwp3oKCpJwQCMHBoHao5jW9uEXi7g5Kj1J5/3qTcVLU5tHOa+Ya7pOepNXKlIXZlrt0XElRo6UdBW7'
        'OK9inKDnmvHE4NHncRmYk18lXNfEZ4qdarY7KcB2kI9anrmSoJOBN1oiuSnAADt8zTtBiNsRwkJwcVhZrehlAATgCvb/ADkQ2ShB'
        'ysjgUa1Y2ZerqCDJgy+zxFRtbV4z0pYdU6+SpxROa2PLVIcLizkk1taRxWpUB1i3s7eJAEck9K92d3IQPiiaWwa0vM/+aQcedZ3L'
        '02ZWJ3CbS9oHNbDJcCMBXFaNuOhr7BrLLdvMV6hkWU4tSsk1i3IKcjNbHxxUB8lNdWBCzmSVq7wetQnmVJcBT1zW+IVKTxW4oUfH'
        'tJxWjUmMER6CR1OlCAlXBqPOdP05IPlxWi6OkO85xWkOlxAFXXsCruS2p7CZJ8aycelT0l5RCWwQkVnCjlaQVcJHlU4hKAEisW2/'
        'JlYvmeRSUkbjmiLDoUM4ocCAqpcVQzihrYNoxiWT/9k='
    ),
    'dark_eyed_junco_00.jpg': (
        '...'
    ),
    'dark_eyed_junco_01.jpg': (
        '...'
        'f//Z'
    ),
    'dark_eyed_junco_02.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAQUBAQEAAAAAAAAAAAAAAAECAwUGBwQI/8QAQxAAAgEDAgQDBQYDBgUD'
        'BQAAAQIDAAQRBRIGITFBE1FhBxQicYEjMpGhsdFCwfAIFRYkUuFEYoKSlCVUomSDhJPx/8QAGgEAAgMBAQAAAAAAAAAAAAAAAAEC'
        'AwQFBv/EACURAAICAgICAwACAwAAAAAAAAABAhEDIRIxBEETMlEicRRhgf/aAAwDAQACEQMRAD8A+OO1JTqQ0IQ00UtNP4UwDmfl'
        'RRRTAKKKO9AB3ooNFABQKKO9AAaKK9ek6ddaneLbWqAsebMxwqDzJ7Ck2ltgeTPenKrN91Wb5DNb7SuHtL0i1N1eNaaneOoCQybv'
        'DQn5dTSQSXAb3GzAQOwLlF2rjsAPKqvmT6Jcf0wHOgV27hnSbJnPvFq8qkbkIVRvx15Ht61Dxjwtw7JHJdTWBtHJwrW8i5Y45HAG'
        'Pxqn/LXKqJ/E6OL0Vda/w/caZmVGaaBThmKbWQ+RH8xVLWqMlJWitpoKWk70tSEFLSUtIQUUlFAwoxRRTAfSGiiooBDTTzpxpp60'
        '0AfOiiimAdqKKKACgUUYoAKWkFFADo0aSQIgJJ7VvOFhY2sEglRniRAjmNQDIx8z8/0rFaYxW+i2sV54zjOK66vDVlDp1pLBqK+L'
        'JCJJ0fGAR3GOvKsvkZFGkyzHFvo8WqWfvPhXZt4baAuVVAdzdMZOPl+ZqNrBg6bGBYHpmvfPpN9BcOHntSsSjI8QBXPkPX/evYmn'
        '3R0J9UnEZtozgEj7vz9M8qzrKq7JuDs9WgWuq3U/g3CH3eMj4s7V9M+nyqzvdEi1a4WLT5Hu57ch3IOIx6A/lXk0i/vdbijtbfwL'
        'K3t4TvnmPhxouRzLHr2A/StnwINGs9Fn1CHWILud7gRMkAyRjnkDqwPngVnyScdrsnFWUOo6Bbyu+mXkK/bEFgCMgDrjIOa4v7Te'
        'ErnhvV/EW0kWwn5xSg7kJycrkAAH0r6S1+4MkwmmgWAg4jLSDn+Hc1z72m6YdQ4Snha6+BD40Q8MyBGB6cuYz07/ACp+NmlGavol'
        'OCaPn/tS0FSrFWBBHIg9qO9dkxhS0mKWkAUlLSUAFJjnS0nSmBJSUtJUQENNpxptNAFFFAoAKWkoNMAooo70AFAoopAeizimZxLG'
        'rYQ8yK7f7NLa91yDTmv7MXMVscpJtJfrjZ6rz6dK4vZfFaiGPfvkk+nkK+1fYhpmm6Tw1Z28KpNcxQKJHGPhOOYz2rm+fLSSNfiJ'
        'cnZzz2g8JaxNq0DjQ2jt/DYI8SblToeg6Dl+tc64hvb06rPpdoktvZswHhFiy7gPiIz3r7OuWj9xmmubiCBNvbB5epNfOPHtloEz'
        'lbVHe8aZvBmLHbJ8QyAADnkcdqyYpKLVmjLC1a0c84qg1iHSrKPQFljt2jeOcxfF4jEbTu645Z/E9M10v2OaK3DPCEl9ryRJcXHx'
        'wrIuZQMYyM9M5qn4alsrewaDU4mtchtjFXwSp+HIHljrz54r16nPDeSvD7zfOhwGdYwdoB6DsfnVknKS4eihUtlm9zfahdxK0Stb'
        'I2VCfaYHqcYz8qNcMlndRw+7K6uwCKD0z3x2FQ6LfDTNUWOzkSWKVfukAuPn2q9sNI1jXNYu/c1EbqgJSYAYXHY88dqok6mTX12c'
        'N9u+jWNlqNte2Vm8LyqfeCq/Zlux3dz1865iK797VrMWXCmqxSRW0l2rBrhZTk4PLcmO4PMH59a4DXX8WblDfoyZVsKWk+lFaSsK'
        'KKKACkopaAHUUUVFAIabTjSGpIBBRRRQAHrRzoooAMnFHfpRRTAKVQWYAcyelJ8hVlZ2pUgH7x5k+XpUJS4jWySyb3KWGWIo8sLh'
        'wHXKs3Xp3FbTRvafxdaRQ6fBrUltCH+IpGvUnmTyzWSm0+9hmYPB4bsoZdw5kfKvRb6ZIJI5XVvuksfUVkyRjP7bLYSlH6nUNQ4x'
        '1HVVntbu5e7bx8hvEbZ4fkB5ZI9eVezhfUU09WYrFdW4k+OF3K7c5+JT2Nc90Qk3kYYtjz74rVTxs1rIYGADAHHZsfz686oeOKXE'
        'seSUnbZZrDcX12ot3kyGZlhEpG0n58uf8q9ttp+owXRtJbWYTuoAJYMqjHzxip+D3gto4JLtGZSQ0iMAeX8PyNXl3Pw/ZXc17phn'
        'kllUhkkPNc/p/XKqpzcdIaV7PBpWiXF9dyRrd+6GJeWwMdpB8ieefQ4rp+j8X21lpUenKpnvokKe/ZADeYx1Hlz8q5D/AHndTyiI'
        'Wq2sYwGeMFWbyJPXyoj07UbiQ3Cq0Vi4zMuNzdeoP55H1qtxb3Id/hL7cdZn07SXlTSEuYruJ7eaZufhZ6edfNx6+lfXUGk2F/pL'
        'WNzI95p9ynhyJJgkeWPL6V8wcc6BPw3xJdaXMrhUbdEzKRvQ9DzrZ4OSO4eyGeLdSKOiiiuiZgooooAKSlpKYD6KKKiAhpppxpDT'
        'ASiiigAoooPlQAUUVNaQtPMqKM0N0rAmsIMtvchR1yew866F7P8ASIZrhJZGKOZN6FkyFQH7x/r1qsstIs7ZIIpWaS5lIdwoBwOo'
        '6duVdA0yzFhoUeo6fAJbqchGGc7FBzjqTXN8nNa17NGKFsseKdB0/UWihs5ftYQS0zdyeh+p6CsvDo8oaTxV3PJiNcjbnnjPP61Z'
        'LfzWEwnSX/NB9sgH3VJ7D1APbpVlqNo6T+BCzSQxRq7MW5hmz1z35Gs+NuKpsskk9mXXQ0tyZLYu6YJU4/hxgn8cipdOmdZ3t3iK'
        'ADJ3jkDzyPlmvcLW5iuBJb3gdAFQxZ5D9udMu9JuJbgXOSzFAZCmCVx1B+mKvTvtlb/0EeoPa3QaVUmO3JhOdp+ZHP0yOnkaudL1'
        'S11K/WF7aW0kIy7ZzuOO5A5cu/Q9arksrd7IL4ab4yXid2wV/wCU+ee1e3QbeKC4e+mnjd3XdkHaQR0GP671CVNDjZp+H7dUZLba'
        's67fDZPFVXXn94KwGfkM16uIZ7eytzb7XjkXlkSKCw6YIBqkMyXd5C4kdveASMD7rA8mH+lh/I9qz3EU2p3epmNpmeGMHfuYEKR1'
        'x5c+3Oq4wcnslaRc6VcIt2LaS6ZRtDRjOc+QP+1Z/wBtsB1LQIZr+Jba7s3Kwzuc+OCMlCfPlkH5ijSkmnv7cRMGMjhRGOZAHc1u'
        'faNaS6nwjrGjX0cag2YuLeUfeEiDcM+hwR9fSiUlDLFkopyiz5UNFK3WkrtGIKKM0UAJR3paSgB4ooopAIaaadTaaAKKXNJTAO9F'
        'HWigAHM4rRxXWm6bCtvY273UsmBNPIMEeYRf5nrVFbJl95GVX9e1WOnwz3dyILYKGwS8jHCoo6kk1ny09E4/pruEbRtTvS6QvBGx'
        'ARFIzjGM8+/rXW+H9Cj0bQJkskaS5ZcoZBnB+XT/AHrEez2yAAMSMygDLMf0Fdm4WtPEljdxlB2rieTlbnSOhhxpRtnFeIInGpIr'
        'uQ0TByoIwenP55zVvZXMsl3PLcJldoml3ZJOxWGAPUt+dbT2icJ20Vw1/CjxHrvUdKwZtntrNES53XPieLJt6sTyAJ8hz/GtEJKc'
        'VRROPFntkmjuMxRW6rGI0kQodrbGAPPz7j6VP4T29kXtw8bMoEjE4PXPL8vwqrmkuJ/ARQRJEnhuAuMjJ6eYxirDx2FoqSE5yQMg'
        'jvViRCyCfwJWaV7h5Gfm4xjP9ZNSx/Hb27CN0jwVG5cDKnBP8/rTLWwv2t2b3JQVfkzDqMc/wxVnFpV1Jau1yrs5ICRjOFHf+VDa'
        'QLY2O8jDZXaT3YDpWfu7iQzylY96nIC98npXuWzmiLxyCRSzAKNvPJ54NSyaPfW0UdzLE4RiACR0PnTTSFRNwhbzQ6zBHCqFRJh3'
        'DA7um4A10viWyF7ZPIiEZVo3VuYIqu9mfDcjX8JnhzEMu5xyz25+fStvrEUNsTGuBgeXWuf5OROVm/xoumfD/GmmppXEt7YxjCRy'
        'HaD2B5iqaumf2itLWw48NxGmxLu3SUY7kZU/oK5nXfwT544yOZljxm0FFFFXFYUlFFAD6KSlpAIelNNPPSmUwD6UUdqKYBRR9KAC'
        'TjIFAHoSbEcUKLyGcnzJq84ZiSW5KkHZ35DDY881VWljIx8UlDGD1BJBNafhm1AnjAC8zzOMVk8iSS0W402zrvBcCRWSNtxy8uVd'
        'Q4PiCnczYB+tc94WjUWaHl0HPsP6FbTRLtvHMSfcI2g9q85kf8mzrx+tF/qNs2r3qQxxkwk4dmHWo9e9n+nLYma3s18RFJPL73qf'
        'M1q9CghEaSBclenzq8R0kX4hmtGHoy5O9HzrNwjeSSbxBIQCSDt9Kv8AQfZtc32oRR3aSiJObnHmOld5srCzZVBiX8Kt7a0t41cR'
        'xgFsCtS5P2Z5NHNLT2b6VYxyRNulOCR55xXki4St4JJWs0LtH1VhnBBzj9K61Pa+FCZSOg5mq8JawK8gxvkPMCq8kNkoyOeW/DWn'
        '6iEvJbJVlLLvyvMEdKvrXhHTnje3urSN7cptww/A1tba3t1hULEoUjypLtFCgDlUljpC5bMy2lWlnYpb2cKoExzHU1keLLVVcY6m'
        'uiXMQV85+E1keIljl8bIHwLyrJmgbMEz4y/tGarbajxisEBb/JRiE59QG/UmuWVpPaRO1xxvrMr9TeSD6BiB+QFZs16Lx48cUUc3'
        'M7m2FFFIKuKgo70UUAOFLSUtIANNpxpppoBKKKKYBRRQKQFxosryRSK5yFxzrUaIAsikAk9sedZnQVAids/xYPOttwxIsc4DRLIh'
        '6ggVgz9ujRDpG50TWV90hVdqYOH/AEro3DckRjjnz9mRu59q5PZ2UJlZkDKp6AnNbnhY3cVsxlVmhTmvqRXLnFGraVnZdGuoxECp'
        '5sMgVb2Tkscnoedc/wCGNQcAGVebAAfPv+tbnTl3IGViCedOCoqlIv7Z2Vg2auLKUcixzVJaNuYKeoAr3XEixKSrAEdKvjorey1v'
        'JVmtXgY43qRnyyOtUdtYzRSC4lOZN53AU271mCOzR5JAm5tmT2Jqz02VXjG98h+57GhvlIOkexCdmPSvNO3MZ7VNI+x/OvJcyKWJ'
        '8qsZFEVyQYWye3Kud8W3PuzySsfs3G0+lbfUrlYrKaQnGAa4J7cOKRpnBl5cmYJNIphtxnmzn9uZrJlXNqK9mvBpOT6Pkrii5981'
        '+/uiQfFuJHyPViaqzUkzbnJqOvQRVKjnyduw7UUUVIiJRS0lADqBSU4UgA00040ymgCigUUwCiiikBb8PtuEseccwc1rdLuJIWVl'
        'IYr6dazeh2LLavKy7ZGxtz1x1q707MQAJznvWDPTkzRD6nRdClS5iEgcAjrGTz+nmK6hw80UljHbrje6hsEcwK5HwzLaSbEkjUc+'
        'bGusaLPBGFm3BDsxny5HH61zckKZcp2qLLS5wpjRl2jcwyfQ/wBGt3DdJFDDKjcshetYeFElsQ6kM4JYGpn1pYYYbebKZUHJ86Iq'
        'iL2dG069UzHB6DHyr1Xt0nwzeIAvQqa53DqU1vcpOkoKOQOvX0q/gZdVhJimGxgQ4zgr2z+IpuWhUXaWlvfxGN/ijb4iueh6/Sre'
        'wW4gnVd6GALj1z//ACs3bTPp8iJMznljJ5gY9a91xq0a4ZGzleQHb1qvHO5bHJaL6S9Ubmzk5wPnXmkmyjsTknrWdFzu2neeRzgd'
        '6g1jVmhgZN4XPIYPOtDkRSG8XayEtvd0dV3c3YnkF718Ye3DjH/EnEr29rITp1mxjhx0c/xP9e3pXVf7RHG/91aUNHtJVN7eqd5D'
        'fFFGep+Z6D618yyOWJNafDwW/kl/wMuSo8ENJ50lFFdMyhSd6WigApKKKAHUopopRSAU0004009aaAQ0UUoyTyyT2FACVe6FpG4r'
        'c3Qwo5qpHX1NP0fSBHi4vAN3UIe3z/arK4mLjavJf1rPky+olkIXtkxuYw5x9OXWnW7gty7Hp5VV3MywRF269h503RTNvedjyc8h'
        'WeUbVl3RtNOm8F0YH4eta7TeKkjj+03DB2lT/p8/nWFtpVaNfi68sVb6eLGaCaC6PhPjdHMPPyI7/wBfKs8kn2So3Y4oSC0Vobnf'
        'tO1gD2PQ/wBedXJ1KDW9NhROc8fNefMjyriMjP4joj9D1B5fOrTTNeltLNQVb3iNwQ4fky9MEVF4/wAD2df0WS9TxbHUUbwZMY3s'
        'Mg+f6/UGtVpk9xoN1HLkz2kqAOOuP9v0Ncph4jFzYW/iTlLmDlvB5nuPnWu0TiS51S1WBWiEwYHHY/tWTKpEonU7nU9OvI1lLx7S'
        'PhBO1gf5157SWIxNK7hIem49wKpLb3Y7VulQunxYJ+6fT514uJ9ZtTE9pDIp8NGLHPwLy5A/z+dGKHtie9Fle8VRTOEsYzs6Icdf'
        'U/OsZ7TeOrThfRZJrqZZtRlTENuWyQf5AdzXL+LvadFpJeDSLhby85gyAfZx/vXH9a1a/wBXvnvNQuXnnfqzH8vlXRw+M5u5dFUp'
        'qPQ7X9XvdZ1OfUL+dpriZtzsx/L5VXUlA6V00klSKLFoFFFMQZopKWgBDRRRyoAUU4U0U4UhgaaetONTWVpJdPhfhUHm1Fpdh2RQ'
        'RSTSBI1LMa0Wl6bFaKJpcPL2Pl8v3qa1tYbOIKqgt3/3p8jFufWsuTK5aXRbHHXYsshf0HYV57iZII98h+Q86juryOAYB3v/AKRX'
        'ltoHvZPGuGymeSioJe2TbGwQzahceI/KIf1ir2KIIoVRgAUsEYVQqqAB0ApLq5itoWdzyA/H0pN8mCRrOAOHrfWr2WS/ZlsbVfFm'
        'YNjaACc5+hrC6lxIq6hOtoryWwdhGX5FlzyyBW+sLqXQvYZqGsSMFutdl93iXuEPI/8AxU/jXG2OSTTwYuTk5Ess+MUkaKz4idn8'
        'OWNIwT96rEalCx3KyYHfeKxf1oq+Xjxb0UrKzf2WtW6v4azqxHxYBq6t+NDo11G1vII3kwoCcz8+dcmHLpTi7lgxYkrjBJ6Y6VW/'
        'Ei+x/Mz694Iiu7/QP8Q6rJJschY49xMjt/KsF/ay1KDTtZ0zRNFlSCJ9PjnvkhON0rE8j9AOXrXL9H9oev2Mdrbe9ye6wNu8NTyJ'
        'zknFa/jbSrfjzRY+IdBUyarFGPeIFyTOoH8I818vKsWLC8OZPJ0adZMb4dnH2JJ50hpzKykqwIIOCD2puDXXMICiiimIKKKKAFpK'
        'KKACiiigBwpRUwtLk/8ADy/9hp4sro/8NN/2Go8kS4s83zq7tL6yt1VVfAA5cqrWsbzGfdZ8efhmmGxvP/az/wD6zUJRU+2NXH0W'
        '76pa45SE/wDTXhutTLnEeQvkKgXTdQc4Sxum+UTH+VIdN1AcjY3Q/wDst+1RWOC9kuUiJJvjy/3T1x2q40m5t4wUaVPMZOKrRpuo'
        'scLYXRPl4LftTjpGqgbjpl4B5+A37U5xjL2JOS9Ghur+GCLIkQkjkAartGtLnijiK00tHEaSyAM5HKNe7H5CtRwB7NH16ya81S9k'
        'sUaFpYIkQF2UfxnPRevqfStG/An+A411y1u7q+uX+xhVYR8JOCzcie2cVleXHBuKf8i+OKc6daHf2h4I7Dh3hrTrBGSwt1ePBXb8'
        'ShQPyzz+dcVr6R4ujh4x4RsrOUXPhSEEyxJvZHGcHH4iufaV7OrNItTi1aHUvEtU3rJGNvLP8IwQ1Q8byoQx1Lssz4JSncTl4xii'
        'r/XeFNV06/8ABt7W5vLeRBJBPFCxDo3ToOR7EdiDXi/uDXAu46NqG3z92fH6VvWSLVpmNwkvRW0oGatY+HNdbpouo/8Aiv8AtU8f'
        'Det8s6RqH/jP+1DyR/Q4S/CmSMk9K1Xs+4gueHtVjfxXW2dgXC/wnsw/nXmj4b1kddJvx87dh/KugcLeye8u9Ks9U1G4a1a4YNFa'
        'qmXKdQWJ+7nt6VRny4+L5PRZihk5ritllxx7OrTiTSZOIuH45I9UEfi3NuIz4dwepKns/cjofnXEntypIIwR1FfStnd6hb8c2ugW'
        'jyW9rGmVlckbycZ5nAOBgemTWd4k9mml3GuXU0T3ttC10VKxqGRcjOVJHTPfOB0rJg8viqm/6NXkePbuPZwkxHyphjPlXR9T9nV+'
        'mnLf6RcLqaF9rRRoRIPkOe7HfFUTcIcSZK/3BqeR1/yr8vyrdHyINaZjeGafRkypFJitQ/CHEnP/ANA1MY/+lf8AaozwdxMemgam'
        'f/xX/ap/NH9I/HL8M3SVozwZxR24f1T/AMV/2pp4K4sIyOHNV/8AFf8Aan8sP0Pjl+GeoNaP/A3GOcf4Z1bPl7q/7Un+B+MN23/D'
        'WrZ8vdmz+lHyw/Q+Of4fYkMdrtMgiVct8O0r9aQZaElIspvCyO0WBjPng4+nWol8MW4iCh1xnLIhP44yKYEeQAxRxCXafsxGhB9c'
        '4ryVnoD1RWgefcwRVz68x0B6Zpm62YYMEKFCQx94Az+Jz+VRPIEtk8S3WMjGZEUg5+eMUiReJPuG047I6ZB8+a86TbHRPbPA0gET'
        'xyMOyzAHHljGTUrtEwMkdqqjdzBkJP41AJcTfaOsisOWUAPI+YFRSnKiNY5CVOS25gzfP4cUAWEcYyFMI8Y81KkHORn8u9ea9Rrm'
        '0uLZbkJceCyjeT8IK4B5HoDjrioBFuHxeL05jbkjyx8NTw2izybBdI0gUlVkiP5Hb1+dOxUJpUFhp1nFZ20UrJEAisUBJAHXIJGO'
        'Xap7ieNYZLgrNG688eGSD/y4H0piafA83htGVkK890YXB9etRSWiqzAeGpVtpVsfFjuPgpNvsEkjmvsn4khmkmhlIheW4m2xgFjz'
        'Yt5jpnGK6YuxmSWYsI3wNywtnPpg1zfgHhuyi1TiK18B0u4L/MbFgNqc2XC47+hGa6Q1s4tw5j3swzhDGDjucbetXZ+PN0Qx3Wx9'
        'tHZ2J/y/wwSSF0Uplcnqdo75yak3QCIbnGFOM7GGTnzxUcdunjF/GYgKMbmKkcu5A/lUj2xU+Hgq/I7PGbJHbyql7JkcmoW38MqI'
        'wAXnNjf65xUgvIrmMHc7gclBnJGfIcv1pk3hxsWlDQouD8UzEEj/AKuVJGE2Bveo/NlLhuR7YLEGgCo4teea0sNMt0kPv06pMrDm'
        'Ilyzg4HQgY+tXAS1i3ye6ytLO2/eHYjyIAxj6Yp629hI3iLDaTyD4m+zTIz5YeoZEgR9z2cahM42fCV/+dSb1QvdlNrJjueKtCt1'
        'VzLCs07eJnKoNvIch1YD8K1E0yQQqiwyKY1A5Ixz6EEA+vKufcOhta4y128jjnVbeVLOEKxIAX7xOSevPOD3Fbn3YBFeeNwwYAEt'
        'kg9h1qU040hJ27MtxhpcdpbQarp0bRTR3RmnhjbCHdgMQCeXXcfkfOtRb3ObJZDFNGxjDKg3Z+XU155oLW9aSKRd0b/eDruBPkaS'
        'W1hjxuht3UHB+ENnPQ9ai5WiSjs9MMqll8aSdVPxBzE2Af8AtOKkVlDKWkk5HLhgfyGBn5V4fc4grAxISF3sNiHAr0w2sBZUW0hY'
        'MuR9jHuxnr186iB6TdwxjBkYxhjmM5G8H9KgLwvL9jEWXoqs+SPPnjnQY7LwnaIIwVSdrKgIA79fPyrzB4OSsQhA5bdg/HzoA9by'
        'mN1QtIoyAdj5IHn06cq8y3Ecju7sgIHwlovvfPnTxeQGIAxxYzhSVUdOXPaa87LArHCkd+Yjxz8+WRQwREx3II/BmXGQw2ABufXq'
        'OdTHOSS5Y42/aKFwP++pXCToZXijicqNuyQAAgdCNtEIuhHuQXBjJGcOjAn6dKegtgg8ALsn8JG/0AH0/wBRqYKkauryQP4RznHQ'
        'dPPnXme4uGUuplKNzIDIR+ZqLNzzdYpwh+8dvwg/QmgRNJLD4ePeItm3au4rkD8M0/Fu5OLqOR1XCKoboPp+dRQqplDNJ4h7HJ7d'
        'eh5V6riIeEzRxvjqSszcwemcijsZGTApHxHABBVsqQfTnz+VRPGhdSm/4hldykdumT1pPEDMyXFoMyDlhxkeWM/KiIYkjBhk2jI2'
        'FgDk+VAWTo+2JvFkjYctkjrt249RkfXFRsiJ8MssWzG7J2yKxx3GOvIdakbccbVeBkyuxlBIA7HkQfwrzs+yTG2FT1wQ2T+VAkVK'
        'WSR6/d6ojwslzBGjpGmMyISN2NnLkRVyXiDjc4Y8lBaPHLqf4ajFwigKyBzGck7tufy/Smi8VpGYQI3PkVnGc/h+VD27Y+idfcTl'
        'TF4wUHa3hEHPkcL1qWPw5ADHEp7lfDABH1AqAS/5cReAoAGG+EFs559flXqkmluQJFEWABlfuHHpzoFZ5jtjmaTBXc2WRCowfT0+'
        'tTIzvuLtcFtuQEYKfxGRimSh3USxiVM8ubA8s9OvnQi38QRVNwVJ5jIyB1yMNSGTIsaIX8ORmwcq6KcdOY8/wqr4q1H+6eH77UIE'
        'dp0hKxIYwQHY7VPMDuRVnP8AbpseHxpRlt0q5yPrzx071Deadb3sAgvbWEW+5XC4CltrBuZB7HHLvU1ppsi3ooPZ/pF1o3DiRLlp'
        'DI7O2zOTkZbz5kfpWlS4eK6wqqJNu3wyhAX1AIznn1FQ/wCRguUHgwPuG8ZIGw9eoP60lzIZ5AJo4wmOe1QQMfMZzSbbdsFrSJpG'
        'uzbgGKYOCS7rnJHkRg8qabi7ErG38QNg/AE6+ePhNQW0b7Mxszg5yqghiuPLFS2848JFiiPjp8BLOy59NuAPzpDFaF7h1LPOJFG5'
        'vFKrg+g29MUq27LH4v2csRIGA6EHPIc8UNMYYgz2g6nLqSVBzzHcd6WN5GUQoyxhupWQ4Iz8j5+VH9hsI5BZy7EwFY5G5s8/LO3r'
        '9aPE8dBHt2kZwq5U9fRetK1lcssuDFKoyuVG0n1GVGO3XFeYeFbu6GGMsOTcwrKQe4zRVB2ehJmikjYCQjK4beSuc9M46/OiSNnl'
        'IEkmxWOdxIYMfQHpUUbRXEojWRIifhXJ5DHTrQSQ23cokT7mQGGfxBpDP//Z'
    ),
    'dark_eyed_junco_03.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHQAAAAcBAQEAAAAAAAAAAAAAAAECAwQFBgcICf/EAD4QAAEDAwIEAwYEBAYA'
        'BwAAAAECAwQABRESIQYxQVETImEHFDJxgZEVI0KhCDNSsSRicqLB0RY0Q2OSsvD/xAAZAQEBAQEBAQAAAAAAAAAAAAAAAQIDBAX/'
        'xAAjEQEBAAICAgICAwEAAAAAAAAAAQIRAyESMUFRBCITFDJh/9oADAMBAAIRAxEAPwDDR4KgfGWV6s5Kj3rX8PxmZKVNKbGCN89a'
        'r5GtX+DXpLmMtED+YP8AsVL4YgXP35KiFBOeRr5GHV0ekLj2wNNRm3IzeHE9qgWHhKdMZ8XcBQ7V1W5WYvRkLcSFDG9WvDcRhmAp'
        'GgAp5V6LwTK9rPbmlv4aVBfCiMkbCnbnwzImL8NSVaCOVdPt9qEqWSUeUGtA1ZWBzQKcP48ktbzy305nwzwS23CCXWQdu1S1cKto'
        'dU6EAaBjaunpjMR2ScADFUNwUXvEDI2IxivRcZIxI5hMgR25KkoAIVsU+tQbpaER2230AoOd8jnU6/xZUS7pfUvSgKyRTlwntv2/'
        'R5V52BFfPzxlt27eO501nAqWDGTrCdhzq9u0iK02EJ05NcxtVymxGlIAJSeWKCp1wkvBbilaR07V1nPjjjrTnljY1zltjSZIdVjB'
        'qTbocRmQsEp2O1Z9q7qbjhGNwNqq37vPU6otbZO1Z/s8eNZuUdEvMqM1AASRkCuXXW5OKecbBISo42qwVJmvN/nOH5Vm7ytxEgJa'
        'Rnuaxy/keSTOCU0tLalJIx1NUkpDr0gKUshKeQrSRkq92w4g4NUt/aeaGWEnauOWds6LC4T7zKS202pRPLAq34ebmuycrjqxntUr'
        '2e+7OIC5Lf5nYiugsCInSGWhqPYV7eLjnjtdqRy4rgR/OjTgb1QHiBcmckNpUU55CtxP4feueQkEJV6UVs4HRCcDmCCPSs58XLll'
        '1embatOGIrsyOlTiCMjrWkFiQG8Y361DhT2rWgIUnIHpVxAv0KUQlKvN2r3YYyTVUm3WpMdWUjAqzLKtIFONuJUMp5U4DXWQRm42'
        '+VGngykczTlCg8/SuC1SYnhtqLT6cLZc/pUORqz4PmolOOW2cwI92ieV5r+sdFp7g1tG32FsBRQcjbas5xHZfxKXHulrcEW7Qzlp'
        'w8nE9UK9DXiy49f5WdXbSORkqt5ykDArNRbg2JxYb3GcHHerRXETErhx5wI8GY2fCfjqPmbX2+XY1n7PZJjaWnXBhayVqJ7mtZWy'
        'ai9bba1vNssKWrAJpEq/NMqwVbUiJb3HUABeQBvg01LsTQXqcUPrV3lZ06axnslV0dl5SgKOeVLtaVJWQsZJFWlktjIRgEZFLkxE'
        'syUkEAZrWON+WcrPhheOLUp9lxwIVkjoKxHDVnmLKUPJJTqI3ruN0jNu+7teUhwkH7VTxrexHUtIQNl9q48nB5XZM9M/G4fIb1KQ'
        'M9sVXXmIzDBKRpV1rqCGWjG1JAzjauZ8eocTKSByzvisc2H8eCZZs40torKlHAzS35cVCRpCc1BuMZ9cUllKiewqmiWm+PSkD3Zz'
        'Rnma+RfOXqOcx3WlbeUtpSinAHKjtNvTKlgrGrO6tqtrZZpriEMlkgddq29h4S8NvxFjCjXq4OHkyvcdrMJ6c6v8ENoIZQduwqC1'
        'aX3o6PGYVjmSRXbG+FopOXEA/OnJdgi+BpS2PlXv/q9Odu3FGCzbtRwEnpV3wtf4bs1DXM53JHOtPJ4EamS9a04TnkOVXdu4KtsL'
        'SpLKARvnFcODh/Imfd6S/wDF9aAw7HSpKANu1Tiwg7aR9qRCZQwyEp6U/kV9WQQpNqiyE4W2n7VFj8PxWHdbadJq3zQJ9auomiW2'
        'koTgUvFJ1Cj1Cqo6FAHNCgyVvhNeI4jvvipJtrTOXDgAdSaqLlxFbbbMAS6HHFHAQDuagXh28XG3uylrVFYCfKOprj0Mhxu4uXxt'
        'GNiiGQ+yk+9BBx4iOgPr2rSwTLvcBqVDc0Mq2IIwUkbFJHQirH2V8NotdvcnP5ckyFlalr3JocTA8JXZfEEdkrtMpQFxaSP5SuQe'
        'A/8At96xMPmkulzbC5Dh+GQFqA3OKzPEVynqk+ElCkoP6q2cIxpbKH2FpcacSFJUncEHrUK921LjSlJQOXPFbyx3Ol3WbtdxuUdB'
        'yhSiBUe63i5qOpLKiM5rQ2Xwysx1oGsYGe9W0m3MKA8g39KkxtntLWBe4hnIehLeaUkaiAT3IqdEnzZLqlJTsRmrjiq3x2oUBfhj'
        'aW2k7dCcVOhMRYjbqltpGlJwaTG77EC2T3Vue7qxkU9LtDUxR8VIyeuKgcGOJn3eU/pHhBZSn6VrpDI/TWpjLOz2zqeGYqU/CCPl'
        'VjBtNvZbwtCcj0qSsuMtnOSelQFokSlkjUkH1qeEx9RfhbMRIScKQkUV1u0a2RFOLUlISOtNQ47iEAEnNMXOxN3IaH/Mg8wa1q66'
        'RkEe0hMm4eBGYecTnGoJOPvW3gznJUZKyggkdaat/DFthgeGwgEelXDMdppISlIAqYY5T/VSb+TDCl4+GpOCrnSgAOQpQrppSNNH'
        'ppVAZqhODQxnrSqPFA3ppWnFHjrR0BAYo+lDbFGKDiXsx4bCLg45dXVSpaTjUvfBrb8TLMyZHtTOMZGsCsBw/wATIi3MvuKwSnzf'
        'Otpwg8bjcH7ovfOyK4cetaiRq4zYYZQ0jZKRgUchlqSw4y+hLjS0lK0kZBB50YOTRSHA2wpRPSuqufcGyl8H8Tq4Um6/wqWsqtMh'
        'Z2SeZZJ7jp6V0d9GptQIztVBxPw7H4k4XXb3VFp8/mx3k/Ey4N0qH1qF7N+JX7pbH7Vd8NXu2K8CWg/qI5LHoRvWJ10ifa44RdnV'
        '42zV78R5VCiBCXlrzuakuyENAFVanSq7iport7OBnTJbV/uFVnFS1pV7u1tkHP1q4v7yVWR54A4SAr7GosaMZs73h1HlwCKURuCY'
        'KYbKkgYwN/U1o1AnlTbEZLWdI50+lJxVk1AgshYwqltx20jYUsA4pWDV0AEJHIUoDtRCjqg80M0Qo9NAKOixRgUAoUeKJSkp269q'
        'AZNK3pC3m0DzKAPam1vqVs2k/OoHt+tDektJUBlR3pdUFgmhuKOhQednLFqvbKZI8NtQ36bV0eyAQkoixUkoHXvWP4salolJacBb'
        '1DCFDoa1XAVyT7mY8tIEhoYUT17V5cP1ukaWMqUpzJBCaO4qWW0N9VGrCKlRZSpY3O/yqL/NluOfpaTt867qFv8AGUhZztnArGcf'
        'WaTaL3G43tiSXGMNXFtA/nME7nHdPP5ZrfwkhEZOcAnenlttvNKacCVoWCFA7gg0uO4VBhNIeYbkMua23EhSSORBqSqIheNYzVPw'
        'sk2mU9w84fy2suwyerZO6foa0fLerBXXiOj8HkpA28M7VJgBCoTC0gYU2k/tR3BOuBIT3bV/ameH1BdkiK/9oCqJmB2o8elHQzvV'
        'AxtQFMzpAixVvEZx8I7k8hS0ay2kubKI3A71AvIzSVLAoFNAJFUNJdUpWANqeGrrQwMbbU2p9tHNYz2FQPBJolqQ0nUtQAqGq4ZV'
        'obSSe+KaehKlqBedVp7Zqb+gmVeEa/CjJLiuW29HHEt0ZPlzUqNbozA8qRUoITyFJL8iM1EQnzOK1GpKShIwBigEijCRiqBmiBOa'
        'UAKPFUFzoUZosUHMeNptsnBDTC0KW3hSiDy7CqZ2cxHuERxvy+JgOdsDcGtfaOFrXHjIKUh1WBqX3NVPF9jZVOjJZToS00tasd+l'
        'eXKZa2NkxcWHbYH0uJyU96hGU57iURU6lr3UquWNXCSm7Kt7brngrIUAOgPQfvXQXLwIBgQVRykSVaUqI9K1jySjQx4DhbR4rxyE'
        'jlRuRpEf8xpwkDpTkiezFWhDziUKX8IJ50zKuBJDMYeK4rtyFdehR8dzRDtbHETOferavWpCdyts7LH23+lXNqkvXqOxcG16IjqA'
        'tAHUGm7bYtXjKuB8bxgQUHcAdqp/Zu8q03C48HSlHVBc8WIVfqYVuMfLcfSp3tGyeTmOtH+Uj9qruEF6uH44H6dSfsTVuQCk57VS'
        '8FkfhLqBvokup/3GtKud6MCjGOtNyn2o0ZyQ4rCG0lR+lUVcl4zOImrejdqKjx3/APUdkD+5q2Iqq4biqZjvTXgfeJrnjOZ5gfpT'
        '9BirQqxvjNIBjbtUSRNbbVobHiL7Cm5nvklXhNDw2+qutPxIzcZICU5V1UaCM5HnS0bueCk9Bzp2Fa246fMtSz1JNTNZzS8kjlTQ'
        'aDTKTyGaAW1r0jnS9AO+KSGkhWrG9A6AKMDekjOKG9UK0+tDGKTlVEFL7UC8YoYpBK+1J8RY/TUDuTQFMLfcHJommHJy2ykLaIye'
        '1NjknD96vlrWqJN1OspOEO4PmFTXuKS9OfamNFD3g6G8/rBPOthaIcSRFShxpKlNHSc1mONYTL8mXHjMoExCUKaOPhxmvLljljju'
        'UCwswEuIfdaw4FaWQofEomtLxNCZVMsxdUn8p0Eq9cVi7Fc3Z7Z94iFtUYaE4G6VHmat25y7jOhwpQcPhK1Z7irjf1RW+0hSr5xt'
        'aLbb31hLJK3ig9K0yYsuzOJfi5eRgawedLs9oQ3fXpQaCVKT5D1xWhMYZCVrGVdKcfHrK5X3Q1arxHnJwlWlwc0nmKzftDactdzt'
        'nF0VO8JzwpeP1MrIBz8jg/epl7sUjWJduc8F5G+3I0X4h77Y5Nvu7HncbU2U89eRiu1+hqRIYMZMkuJDSk6tRO2KwXDd9mvO3SBZ'
        'IZfS3MWTIUcIGd/rSvZw1KutsVb7y6rVbXCx4Ocakj4SfpitHwtHYiXG8xWW0tth4KAA7pqXeWtA+DbtIu0V/wB5QEuMultWORIp'
        'd6cTMu0ayo3z+fI9EA7D6mmuC2mokC4SFEJQZLiyT0ANQvZwpd0/EuJnckT3ymPnoyjZOPnufrTC3xko1ugAYApJGKWSRRAE10UX'
        'TagMUsD0oad6oTgZoZSKWE+lJcU0ggLUkE8hnc0BahQSoKOwpfhp59KIJxyoDCc0eg0aUqPKn2mFKoI+k9qGmmZF84civFiVxFZ2'
        'HgcaHJzSVZ+RVmp7KWZDIejPNPtHktpYWk/UbUEXGKGKeW0R0ppQxtigTRLbbVjUAaVtRZyOVBmLXCeYmSJJc/Kd8wHauf3O8yLl'
        'xNcGba0CgKDLrxOBgc0itVcrhcbsyuyWAhLqUaZMrmlkHoO6v7VD4T4Sas0WY0tRdcDgwonck15c5bqQt2rzKkxZYuSWGvCSkNyE'
        'DqOhHyrYxxDjsfiD/gJTpzqGOVPosMYxw0ttJGMEHrVG203a7h+COt+8l0+JESTkY6pPyrUlxRHuHENzmXBKLNH0eQhJcG6/kO1W'
        'Ui13WNa13F+et2e20VYGyR1xiplqhqj3Z5yQtCn1IGkAbJHYVayda47iCMhSSKsxurbRGsEtdysseYT5nEZPzomLPiWZLzniLJ27'
        'Cq32fqeRanIriFJ8F5aU5HMZrUoHetYXyxlGUdYVaeOmJCNmLk0WnO3iJ3SftmrGCQxxRckKO7jKFgfLIo+NWFKsxlMj82ItLyD8'
        'jv8AtmoV0kIF/hS0KA94jYHryq3oZnjK6PReA1W2EsiZdpiorOOYClYJ+2a6DYILdqskS3Mp0tx2UoAHoK55ara9c/aYxFeUHItk'
        'bLp7eK4dvqB/euo4AO9Z4922kDUT0pQO1EAmm5UmNEZU9JeQ02kZKlHFdVPjfpUK73a32lnxp8lDSegPMnsB1rOXTiWbNZUmxNht'
        'nOn3t1OxPZCeajT/AA3wylqQLpdnXJk0jy+Mc6PpyB+VTe/QafvHE16SW+H7cmEwdve5oI+oRz++Kf4a4Udt9wVc7reJd0nKGNTh'
        'w2j/AEpGwrTgjGwwPSiKkgbmp4/YM0pKcmmi4gcjTEq4lnCWIzsh08gkbfetC1jNEn0rxx/Ff7eJ9xvT/B3BN0fi2qGpTU2VHWUL'
        'lOg4UkKBz4Y5dMnPTFdW/ik9oNz4J9niW25IZu95UqPGabOPBb0/mOnuQCAPVXpXhQJCzqXkg9TzNNiG6tx1ZJJUonmdya0ns942'
        '4w4F4gYuXDl0mRHG1grYDivBeTndK0ciD8qgNNtIIUOY5Y2NJX5ntwoqJKlKKiSSee9RXurgbi7iH2hS4kFi4ENraDk19o6UNjG+'
        'kADmdhmuwsRW4UVuM0D4bSQlOTk7d6+cvsa4nuXCHtDtd6iTXV4eS060pw6XGicFJJ6b9tq+kZV4rDbwGAtAUPqKmE17b5M/L1NI'
        'pG+1EBvSjzos71tzYH2YJFttH4OHm5S45UXH0c1qzzV61fKiyfeVLCwELcCyPlVVwRb/AMJsjUcNaXVeZ1R5qUeZNaNBKiM1x45f'
        'GbEeQ+tryJVqeVskVDudqKonvLSsz2j4ja/UdPkat0sp1aykau9JZd/xbkZwAKCQtB/qTy/Y/wB63YK5mU1LZh3FsaSVaXAeaTyI'
        'P1q3wCMdKoJrP4fclNjaJOOx6Nu9/rV40lSW0hZ1KA3pA4y022CltITk52peSKQFdKBJPM1QicEuw3mVbhaCk/asixFekQIcic2W'
        'k25CiVE8wOVbIJBG4qPcLeifEVFdJDS9lAbZHapZsUXs8t5jWx+5P/8Ambi8qQ4euD8I+gArSkkjagzGRHaS22AEpGAOwpxKKsmo'
        'IcsTfBV7sElzHl1cqx17t0vxWl3d1dxluH8iE18JPcjsO5rWXi6iG81Aio94uL/8trPIf1K7AVKtkERsvvqD0twfmPEbn0HYVLNi'
        'qtFllNqbmT1tqkpThDaB+WyOyR39auAh0HnmpZwaICrJoRgl2lpSetPj5UCBVCENk8wMVOjNoSCtelKUjKlE4AA5kmmWk5POuP8A'
        '8VPtCHDvC44RtylJuV6aUl50HHgxs4Vj1V8PyzUysxm1xx8rp5n/AIjuKX/aH7Up9wak+JaYSjEtqAfKGkndY/1qyr6jtXN/wZ5a'
        'ceI2FdgDWgUltKwhBzvRq0jltXkvNlfT1zhxZ/8AApoP5UllQ5ebaiTY7ruCylRG/lcAJA+daRxKXEeZXI5Bx1o0vFJTkgLHXNJy'
        '5peHBk34c6Ktt6Oy6y+0sKGRnSRuCOh3FfRv2GcVjjf2TWW9rfQ9NDIYnYTpKH0bKBHQ8j9a8GSpoUnSCB0PXNdi/hL4yTw5xauz'
        'vulFuu+G3RqOlLgVhDmOhydJ9D6V2x5ftyvH9PXTicdaRj13qVIQU7EZI61G5c67uKkaSTjA2qU2k7E0oDA2FLAxzrIIEY5E1S8Y'
        'STChMzkr8PQstqUOeFpKR/u01eJTWf8AaQyHOB7qQcFtnxQexQQr/is5+qLWbERNiJYeGUnB9QRT7bZQkJyTilxVhyIw7/W2k/cC'
        'nfLjerA1oPOloQMb70oEA4HKhvjaqD0gb0AcDIou1HjNAee9UXGPE8awRUIbbMm4SDoixkbqcV/wB1NXugK2qqlQbNCuIub7SFzX'
        'AG2zjUtX+VIpdiHYIrdmgu3a7PeJcZOFSHSMnJ5NoHYcgKsoAmSn/fZpUw2P5Mbqn/Mvur05CnWIniPplywC6P5aOaW/l6+tS1Jz'
        '0zUkAzvmjyedFg9BtRjYVoGNqUME0gGnW8k8qoeaKUJUtXIDJrw77fOLE8Y+0afc4ikKhRv8HGVy1pRzI75UVH7V6s9tnFbfCvBL'
        'qWndM6cFNMAc0gDK1fQf3rw1dkB02hsqJDqlbk81HOM/WvNzZd+LvxY67MvrEfQ255FFIJ23NQ5crQ4EIQtSlHCcVJeX74tCZDaG'
        '3kq8Nxe/JP8AbYUqA+htZmrisobQo+C85nO+AAE9Tt+9cvDU3XXz3dQJRej4aebOopBKe1J91WpJU8st+g5/WnXntGZElZQpR1AE'
        '+dXqe3yqodu8lbp903APxKTlP0H/ADTHG30ZZSezzkQoISTkBJKsnGKlcNzlxLw17sXQpChqwc6Dg8z35feq9y5SHUeCIzba1nA0'
        'KJJNT7HaZVv8Jx0AhxWTg5xz/fvWrNTtjG7vT6KcKXFV44PtF1WQVyYTTiz3VpGr981LVzqj9lTYb9lvDiUk49xQfuSavFggk16s'
        'fUefL3UTHbegR0zvUK23JqU4uM42qPLbGXGF8wO47p9RUw5PL96m0KAqr4xSlfCF4SesJ3p/kNT46HEAl1wrUT9BUTiNt2Vw/cIj'
        'CdTr8ZxtA7kpIFL6D9jWldkgr5gxmz/tFS81V8NMyItggRZScPsx0Nub53CQDViAeZpPQX9qMHApOAdzQTvyqhYxtk0YAztSNSED'
        'KiAB3rDcWcV3qbIVZODYK3pChh2c4khpr/TnmazlnMZ2bXHFPFke1Sm7Vb2jcby+cNRWzsn/ADLP6Uip9htj0ce/XR5Mu5ODzuYw'
        'lsf0IHQD7mqX2fcJ/wDh2O9IlZk3OUrXIkLVkk9hnpWs86uw+VTGZXvJDoV3xQ6ZzTYSojdVKSnPM4ropY5c6Az0AxSQOxpQ5EE0'
        'B5T1pxnmOdNApHLenWiSr5UHnX+KOS85xX4b7uhhmCEMg8kFYO5+ZIz6CvN9wQ8u2MBxCkuRXATjp5iDv9q9efxI8Ii5oiXHWG0S'
        'CIzzqtg2P6iegxn7V5GvanI91ebbdfUw/wCdAKcFaFclYzsFDf5GvL4/tdvT5frNClS1OsmYuOhxSl+GlOjZZI8xwPTH3pqW94IQ'
        '9KWlUhKdKEAeRgdgOp/5pLsoR0IZZwhSNyoqKUpPU5qGmbGdkpZiNv3Wa4ryoYQSnPoev0q2bSXRh6POuK8rQUMp385Cc+pJoSvd'
        '4baEtutvKIwEtHVnPLfrUziyxcR2iNBlXuGmC1OAMYL3UT/SUnfUPlShYJENTTi0eJrTr1qGM9OtXpm3s5ZIZEZUgoPjrICB1T1P'
        '/FXcNKn5UaI3lQOwAGfmajWxiQpoqKsrwRk7BOe1dd9i3s+dut5YddbJCVpLiiPhQOdcMrcrp3xkxm3qPhSEbbwhZ7eU4UxCaQR6'
        '6RmpSyPpTz6j0AwNgKjlRPSvfJqaeO3dU95tqZobeadVHmMHUw+nmk9Qe6T1FOwJTr0cB9jw30+VaR8Oe4PapOjPM9KUkAchWdIS'
        'Ac75pQHLpSvLj4aAI5YqggntRgd6IqoZJ6UB4FFgAc6G5OwowkjIxQF4SVDcZpbaEJ2SkJA6AUQ2PWjz0yaBdAgetJAz1NGBgcxQ'
        'GCNxjNDn0o8nuDR5HI0BA7kUecHkaHI9xQHbeqFbK20ipMdsk0w0klQFeY/4k/4kZVluE3gn2fZbuDCixOuyhnwV9UMg81Dqs8un'
        'ehrbbfxY+1jhnhvhWVwc2hq632UE5ZCsoiYIILndR/o7Hfbn4fnXW4yJr0px8pfdXqcWg4Kj/wBftTDTkmfNVNnPOOKUoqK3llSl'
        'q3O5O5UTuaFlaXMlYIOCcE4zgVyt+a6TrprfZ1w7I4vvwjz3NMJpBfmSVN6gy0n4lY6qPIDqSK6JwfxTwxw/xYiz2DhOVHcfCg1L'
        'mr8QpxukaUgbHmTn9q7D7JOC7BwxwDPRdJLLFyUY7kpJ8ykFxGW2dIySoZxgD4iqrh72fWN2M+9MgMuOrQUt9Mgis+Pk1b4uM8U8'
        'AzeMb01eeK79+a2QWmW1gBtGc6UJHLPfnRcYWMyG4/urQZaSkISkHVsO3c9PvXR2eEUa1haUspSN3FqwABU7hW3cJSpcv3W5e8iP'
        '4fnQnqQQoYO6SFJI33rGWFrWNk7Yf2bezqTdLhrlslLLQyU42yOnr/3XpDg2yxeG7T4DaR47pKnl43z0H0qlt1ytltjBiGkIH7n6'
        '1LTe2lqxqO423rrx8cxc887k0ins5Gf3pOvPPNUzNwQvrv0qUiQVcx9q6uaZtigVYob70MjG+M+tQAE5oE7YNDn0IoAHIxQDG+QK'
        'Bz3+1HpPQ0eMHfFAM+v0oAmhkUY3+fegMZPXaj3HIUnSOYJ+9DcdMigV5s9KPSc/Oi1qxsMUEkq9aA9uWAfrRhXdP7UQVsNgPTnQ'
        'JVt6mgXqGB5aBJKvh+VIwrfc/OgfUmgcQog5GB9azN39nnAF5uL1yuvCFnlzHiC485GGpRGdyR1351ojjONVEcb7kfSgw/G3sk4A'
        '4m4Wj8PuWRm3RorxejrgIS042sjBOcHVkYznPIVh7T7AOC+HXmplvl3FyeyvW29KUhxIOcjLekA/Wu1ukY+M1WzACk+bnUuMrUys'
        'cJ4n4Onx+PI16TcpiXnT4y5w0afeAokeQ7A4zjbnV5ahKtbCkuSpk55ZJdfkPFSnCTkEjkMchgDatvdoKZQwR8KtScDke9U7fDyk'
        'ICEuLKRy1b1icfj6bvJcvbA8Ufiz7zc623B6I8yoqUlWVNuIxugpzj1zz2rnnDEHjGRcJd6gzRbVT3FpcSqMdkkglSUnbJOcE969'
        'Es8MheNYBz6VYxeGoyf/AEsdMYqXjtpM9RzHhK3TrbBbipVIex8bjyypTiuqjnqfStpbGJZwVg1q27I0lHk8vqBnFON2uW2nUwpl'
        'wjo4gpP3FdJjpi3aDBjuJI1FRPrVtHaWE8+dCJ44WUSIK2cbBSSFpP23+4qybYxvk/LFWVKc0p65pLpUGz4SUrWOQJwD9aGk9cE0'
        'oA0QfmxvgfKizvt+1A7dck9qG/PFAe59PpQ/c0BnO+1HjIHOgGT1O9KxncmiATyx+/Kosq5W6MdL8+M2f6VODJ+nOm1kt9JYA70e'
        'RnkfvVbEvNslPpZjPqeUrkUNLKf/AJYx+9WO5AG9SWX0WWeyte2MbCjCscxik4zzzQUFadsE+tVCunQUYSnJyo0n6Gj+eaAYQOpN'
        'HhIH/wC3oiduQ+tFgnkQKBWW+hptxaE9iaAQrVlSsg9MURZznegiPvIP6efpUB1ZOdtugq2VFzuf7Ugwkk7igpw2SdWmpDccKHw1'
        'YphhPSnUtAH4aoiNRxjlT6Ggk0+UkqTg4A9OdKGM7poEBA6jelad+WaV5TR4HQnHagTgkY0iiCTnHL60o5AJTue1EMnBKcE/tQf/'
        '2Q=='
    ),
    'dark_eyed_junco_04.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABQYDBAcCAQgA/8QAOxAAAgEDAwIEBQIEBQMF'
        'AQAAAQIDAAQRBRIhBjETIkFRBxQyYXFCgSORobEVFlLB0UNy4SQlM2Lwgv/EABoBAAIDAQEAAAAAAAAAAAAAAAMEAQIFAAb/xAAn'
        'EQACAgEEAgEFAQEBAAAAAAABAgADEQQSITETQSIFFDJRYSNxgf/aAAwDAQACEQMRAD8AZLabadhonHCrpnPegcjKAWDY9ah/x2O1'
        'GHfA96ayB3F9pPUNS2imQHNSL5fJkYoF/j0MwGyQE1Il7IfOynb967/k7aRDFzZrKvGDSd1BbvBLjkAmm2xvFYcsKEdV7X8496xv'
        'rBY1bRNj6SAtm4xat4yeSe1cXj7MjP4q+kDNFlDz9hVOWxvJH/8AjLEdjisPS6L45sGJs6jXANhTB0IuhNuBG0mmbR7mcOgbsO9U'
        'rS0uEGGt2x74o9o9q2SShz+K9RoaVrXgzzmuvNrciHIdWdYwg5PvXNxcPOMHI/Fc2McTN5gODzmuhJbRXGCwwPStMZImYTiXNNtX'
        'wHIP71bv7iO2gJZwMCh2qdSWVlaM28AgdqWoNVfXVdUPl5qzFVGJygmTaj1DEJSFcGurTW2lHDcUp6rpYsbkvMxKsf5UH1bXU06M'
        'mEsOKVa0DsxlayepoV9rZi4jbLY75oJeXF5fHzyEL3OazVespTMWcHHoK/T9bzuQihlFBe5P3DLS/wCpqWm2dr9Uh3N61xdtAk21'
        'cAUqdO63M9oJJCfNzg16t9Nd3TbT3ND3AjIlgpBxD+rSkWxEWCccYqpoNndzxBmTCnuSO9EenrF52xIu7b3zTNHalV8GJMY7GrUr'
        'k5g7T6gaTpeyvIP4nDke9Dl6FNtOJYHb8D1pySyaOPBYlhVyz3ImZMnFFspR/wAhBrYy9GBNP0a6jjHmIxXV2Fg8sj4amEXO8bFX'
        'FLHUlo7TiXceDnFKtoKf1DLqX/cliie6Xw0AYGgPUXQl3ejfHhH7q2O1GLSV7UJJjcB3xTNYa9DIqodpP3pb7DacqYwNUCMGZze6'
        'rtjK7h+KUdSuJbucqJGwewzWnDoiO4hDuCTj3oDN01Bp+rooj3t6DvzSg1Nrdxo11L0JJ0D0vK5W7vSVQfSpNHOrdsMfg2wG4HHl'
        '9Knkkn0yx8WTjC9vQVn111jG+oyLvBAblj61pUOQn9iFiAtGDTrqeFwkjZ/NFbySK5iG4g1nl91IJZd8IJ+/YV2vVDRW5Z17D3oG'
        'pR7Oo1p2rr/KaVpht4wFIX8Uw6ZZR3bgJGMe9YdpvWQkuVGeM4719A9BTRzaTFMMEsoNZ702bgHPEa8lbfgJeg6btyuXXNRXWn2N'
        'op4UHFXtX1QWcDNn0rNupeo5XV2DnFMMyUgLX3FxWXOWnHVepQ2bkQSY54xSnNqtxNmQyYz7UG1TUZLmcu75we2aD3+pm3Q+bt96'
        '06rWRPkYk9QZuBGPf85MRNKSPbNNXR8UcEoVMYrH7LqDzk7yPNitI+H9+JxuDZAb3oa3s9oHqGbTqlZPuOvUmiDULIso5xjism1j'
        'Q50meGaNm2nufWt502dJYtpwQRil7rDSgfPEi5H2omr0q2DeTjEHp9Q1fAGZhM2iqpOIf6VUTT4jdLGEGc81sS6LBJbksoyaVbvS'
        'Y7TUC+0DPcmstK1Z9oaaT2sqbiJR2RQ2QRVAParfT9mzXClF+o4rue1jkddo4pt6I0sNOJWXIHatTbxtEyy3OY1aJoy29kHYeYjJ'
        'qa4AtELkZbHpROa4hjiEeRwKC6zNvIxyKYQbYBju5kNhJLczFnyBniiMqjw+/wCaGWLqzbUOKJi3keMnPFE3CDIncIijiLHvQHVp'
        'lnfaOKI3JlWIxoCTQ4WrFSX71M4SOGe1Ft4UgX2qp4IWbdA2BmgmuSG3uMIx9jU2lX+0ZZuMfzqhIMuBNYlmgtouWAApE1DUYk1p'
        'pPCzg98ZoxdZfcDJkUJnFqhyQC1IroAB3GG1RM81POtwNCSyREdvU0j6n8PrYuzxOVY896eLaWIthDtzV6K0Ex4bOabShVEC1pJz'
        'MXvNEutLcs6GaNe+B2pL6u6giGYY1CyLxjGK+n5dDhmB8WNCD7ikrrD4TaXrG6RIVSQ/qTg1xq/UgWT500XUJpNRjVeVJGa+u/g1'
        'ffMaBCu7OFxyaxc/Ca40iQvEfFUf6hzWl/B1ZrEPazgoVbgelZurDYxiaOiIDZzG7r5pltiynAxWeXttcXVttiG4mnT4i6pGsRgD'
        'cnil2yvYorLcCCcUqiBn7jVrEDMUbnpu/VDK7IM+maXdU6fnugYwzBvsacdY16aQmNFULn0r3QyZZQWTJ9eKdrUOeeojY5TruJfT'
        '/wAOru5nDTzOEJ7AVrPTPS0OkWwSMHgfvRbS7eRYwwi2j3Ir9qGrwWXDuufbNNpVXV8oBrHt4luB3gYYFW7iXx4fN3paj1y3mbIO'
        'B+acOnrJL2ESscr6UlqrmtOxI3p6RUN79wE8QAOeKV+pIIy27GT61oHUtj4EZaH070i36i5kwTxWdUrVWfKNOwsTiUNPsDOy7V71'
        'oujW8Vhp4XA3Ee1COnbBBgqBgcZo1qIEUeM8elbunO8bpkX/AAOIJ1KYxTGQtxQm41hZZ/DUBjRO5s5ryJlTPPrVOy6akhJeTlu+'
        'TR8EwG6RW07w3at6H2pss5ZLhAFU4NC7DTl+ZAfDY4pptIo4E+nAAqwWdnMFSQFZNuBSZ1hrR0acEHKscEZ7U46/fw20byhwCKxn'
        'ra7lvZWZj5M8VV+BLKMxms7a219BcRSAOfY5BqjrMEmlJtkUgY7+lKvS+ry6VJuEhxnOCa0Sw1Sw6kszbzbS5FKs5WGCgwl4zTIc'
        'NnNUp4yiF+TU2l28q26q5BbFXmtmClmANP4i2YBUSyMWXirdreXFqQTu/Y17LNHHIRxQzUr3zeSoxOjSnUKCPYx834qQ6wxA2+tJ'
        '1nHJcsCQaYLSz2oNzVVrMS6pu6lye98VAHUHNULWSO0ujKg25HNdXkWFwGxQTUROFOw5NBZlcYhUBQ5gX4gaq8l0HBOAKB9O6vcX'
        '96bOJC7Z9PaptZhku2MLBtzcLgetaL8Gugflh83cx5djuJxWYlJ3GPW6gbQBOtC6Aa8CyzqcNg/iniz6W0zR7XxXiQEDue9ONrBH'
        'axElQoUVkvxK67tk1NtMhlHH1kHsfam0GOBEm55MH9bdWLZborSEsBxnHFYzrXVs95fkNIVOcFR2FH+q9ejmt3RV3E1mt7EXmdwD'
        'uJyK66svxC0Ps5j1pd5JI6GO52kketfSfRGF0WLJySo/tXxppl7e217AoclPEAIx96+v+hi8ugW7r3KjNKrWKzGS5sMva1iQPHWY'
        '6s5tdS8Ij6j5a0fVAyS5Y4pY1ixtbi7SdsZQ5z7US2kWqMdwa2GskmEtCA+VQIMtV7U4raGFWvry3gB7eLIFz/OsU66+JNzYX8mn'
        'dOsFaPyNOMEk/wD19KyXX+oNUuZ5G1O8nurgP2kctt/NaNahEAxELCXbJn2Hpeo6KqbYdTsZWI4CzqxP9a8vrqK4YxRsAT6iviIX'
        '9y43+IQfYcYph6Y+IHUXT1xHLFeyXcAPnt52LLj7HuKv5PWJTxz7C0e1jtSWuJgc8ipLzUIMskbA/iso6U6uHVenC8t7l4ypCyxE'
        '+ZDj+33p00l4ljAd9x9TUhsycYgnrJXMTTu5CjnGazLWLtZpNmeBT18UNVRLdbeLBZu9ZiqST3BBJGf6Uvc4Uw1SMwg/UJnRgqEg'
        '/ajnSN7NZ3sU24jJwRVVtEfxPEeXt6VGsq22oRx5zgjtSyurnuGep1HImy/PG2PIIP3qK41aSRcA96NS21pJ9Srk1TudBV8tGMAj'
        '0rSV8xIoR1Fq4nEhJyM1xawiZvMc11rmh6jbK0sAyo75ofoVwyHbI3m/NSzccTlU55jVZxJGmQAKtGQqPqAzQj5h9nHP3ruJnkXl'
        'sUu2IUEyS+uWDAb+1QRSLJIFPc1Wv8wsCzkg1+0iymlv0nVjsJGRS7HEOoJjd0t0rBd3i3UkY49xWt6RaW9jacKFCil3pvZBaqcA'
        'cVS636vtdKgEDSqrvwOavBcZgH4ofEqy0aK5tIJA86oQAPevmW9vry8vpL2dmZpG3NWx9X2/Tk1q15dSRO7+Ylj61lGs6toMJaK3'
        'ZCO3DVbxkHkzt2YR0aG31FQsoGc4NMUPQlvcpuiAyexFKvRV9Yve7fFAyffg1uXTVkpgSSP6aBbaKu4VK98ybVehrqBldLfxAvqo'
        '5FbZ0BffLaBFDIpVlUZB/FE0s4imJEB/ahuutaWEK7HVWPZR3NAa8XcCHrXxtmWdVvEmRmLYwO9Yd8R+uJS0mnaPMoXJWSXP1fYU'
        '6dVXtzqWnPZQSfKo+RIwOWI9vtWJdVaVcaYWPh70PZ/em9KmOWgNQ+4/GBry9haM4XFy3BYjsPtQWVdxJbJBPNSO8ZfOCPavHTyn'
        'MmVNNk5ioGJBJDGFIiBXIyMnNexWzGGRiQ74GwdsnPP9KkhiTlTtPtXWDGynGQME54oZEvGfoXWf8t38kxUiLIU4PlznjJ9q1XQu'
        'vYNS1JbMQ29u0mVjKS5DsBn1rD7rz6c4glkCbcOCo59q1D4W9E23yem9Q3IJQ24ZIixIV9xBb8EY4rv5OHMN9SQyTB7ubkAUmQ3q'
        'vcnBxinzrRv4Biizg1l7wyRyN3HJrFZy9jAzcWpa61IlvVNYljk2xOc/arfTyCdxLcJuYnv7UCjtDLKWLZNGLad7eAkEArTlaBF3'
        'GI22l32ia7a6wBKpuZoxz701WWr2c0YEcqN+DWbWXw21OaJTqWpSljydpxTv0t0jb6UB/EaTHcsc1o7G/UzfKIbnubeW3ZZFJUj2'
        'rONcksLO/Z4goOeeK0yQ20AIkVQuKyXrq6s5NUkjiUfkVcKBwZ24mfm6mtYu+01FF1VBLJtgiZj9uaCRWG9cyJ5WPBIo1pGlRWSG'
        'dY1I74pc5LYEKOpZkmvNSwDGYhjjPeiuizXNpcIs0hMakdqDXOpXEin5VQuPYUDu7zWrh/CiVy2e4FQyqe5wYibtD1XYW1mA0ygg'
        'c81l3xK1zTtTYyfMLujOVwfWgs/TvUF5pmxHcOwzkn1rnp/4SazeODf377T3Cipzj1KzJ+qdXu7mVomupWiB8q7uKAJbyyncgLV9'
        'K3PwHs5VG+aQn1OasaT8FtI02ZZHLvtPIduKqR7lok/A74bXmoSpqmo7ki7pGf7mvohLSDS7RY1wAq8k0Nt7zT+nLRYnmihXGACe'
        'T9gKCdSdQm8K+EWS2HLNt5Y+1Y91VmoswOo+hWlP7Cer9SrHEYbR1ViMeI3AX8Uk6jqkMZLy3hYnO525/lQ2+uZJS0hDKPcjA/rQ'
        '0SW0iFpmQgHj1xWhXQtIwIsbC5yYRnvt4Dxs3gkYDe9C9SEd0hjVGkRvqxyP5VV+bikvBEZt6t2HYAUx6VPY2ysswWONhglOWx+a'
        'KqbjyZUvgdTONT6VS6lzbWsqHspC8GlbU9Av7FmDxsyqTnjBX9q1+fU4Be74T/DjbyAntjtQrUL83FxNLgAk7c0XdiDxmZCwCrgn'
        'LZ5AHp+a88OfeHRyc/pbkGtWh6XsNavIp1tAoPEu0lQff96odYdFDSHSW3JFo3HJ5B9quGB7lcHuI0k4t4hHOpxL5WweBW3/AAu1'
        'V7n4bwQOoL2ztFEyjAZQePz3rF9WijX5d48SRhyr5/TkfetY+DENw3ScxI3Ri5bwuPTAz/WuPHclcmPF7oa3WifMyDz4DCsi6iQ2'
        '11Mmwrt963S6M/8AhIQrtBwOaVdb6SGqWEgZdrlThvUVhjC2knqbrFmqCr3MV0jUgblkYjvjn0q5e3O8GOIbmNeXnRWp6ZqLqULI'
        'WzuH+9GtO062slD3RBbHOTWmE8wwvUyQ3iOWHM+g47xZiqyMFIqvf6zBbAxRNvc+goPqCSSkSwsdp74qxoUFglyBO4aU8+b1pxb9'
        'w4iZq2nmQXdjrOskKZWt4jz5e+KO9P8AQGixYnuUE03qznJoh40Hz8USsFVV9PWiS3ESDCOKRutO6M118QZ1V0tpdxpDRwRKjKvB'
        'AxisdRr9dQfTIkMgUkEj0rcLq8jELCV8AjBpOlhsorh7iBU3se5FFoy45lbCFPEW7PRbiBf4iDzd+KOaLpEKyb3hH71LcXMrR881'
        'JY3cgjwVo4qx7gzZn1D0UFuqgKFBojb3EMC7iV4pWmuX4IJBqrf6klvF5maSU/TGnc/8VbaPcpuMbbrX4UYgEHFLWua/NPuWKQWk'
        'Q7zvjn/tz/egTnVboFnxbRnn7gfvVSSO1ik3yMJ39TIxY1BTPAEkPjucy3UULmaBmmdu8zx72J99zf7CgutaozHEqtK/opOB/wAU'
        'ZbxZ/LHbyEZ9to/5qvNoF1dFhcJEF/TgHy/vUirAneQZgO12SQGW6y5J4XOF/wDNePE8gIKrGncKPSrer2EunWrXEl1GNvGQuT/+'
        '/FDLK7eaMB2W09227y38zxQzUSeZcWDHEr3NpGq529jwTxk1Bd3V0FzHEUUYAZuCT60Zt30gSuBcM7ryZJgcH8Ghmqa7osW4FZpm'
        'HbAABrvHO35gWxgvHZ0iHiSMS2SfQ+tWHhkUBZjGFLbQFfu3tVK86ltIsm10xgT2LSf8VBoUV/rWpJJPCILVWBxGuCx9snmq4k5m'
        'u9LWfyOlBTFmVuSM5x9qu6skNzYra3aDOQSCvYe9R6O8tmqW3h8d9236j+TVyKDxdRM9267V5bPoPauccyymYvfdIT6h1HPoSSKq'
        'hvHEhBAZSeK234f9Ow6HotppUQJKjdJl9w3epH2NVNN0iKTqafVomDBrYIox28xOP6Uw9Oanp41j5aW4UTA9s0jrWYjgx7QhQ3Ih'
        'rX7HZpJVVAYCs/6h6mbRbSMTJ5T5SRTv8Stdg0zSmKSAsRgAetZr1Jpsmq9NR3k6kHbuxilVpDgA9xuy41scS9Y31jq9uspCEmlL'
        'rnSIXGYWwT7VFpUNzaIGhDBPX2pr6UtINavv/XAMkXofeuRLqHwOpSx6r0ye4a6WuRNFtY5U+9UeodPddXt7qGVo9jZwD3FNeldK'
        'SWdufNg+tAuoJDHOsG3JB702jNSDv69RRlW8jbKcd1dnU4z4hYYxReHUpFufCYtuHoaWjNMt8pQHIbAzV7VbLqhGF4mlGXjjae4r'
        'qWDZLSNQmwjEP6pewTQCPxgH9RmhrvaKoV5CCeM5rjQYX1AbtR0+S1lHcyECqvUFjZJEzHVbaKQHyr4oJ/kKcW9QuAIqaiTnMvQq'
        'sbeGsniK3bntVu1ilZ9q4/J7VmfUHVdn04kZh+Y1GY/phGFH7mlu8+LvVUr4sNAtoR2BlYv/ADAwKIrk9wbJ+pseo3kTDZat4zhi'
        'HfB2r+PehwuIrIGVlYyHu5HP9ax7/OnxE1SQg3sNqCeEhtgo/bip4dM66vkMkl/dOD6+GTRAecgShAxzNRuuoNHUlrqeVsD1PFVY'
        '+senQSI0ld+wxC2P50j2vQHV08bSPc3mB9RXjP4qNehdYN0I7iLUZQ3rI7cGib3HqD2pHq6670u3i5kAYfbaB/PFANR+JVlnCSqz'
        'DjAfOf2AalbqLoA6fbrdXJMYJyc84HtQnQ5rC3vo4EtiVyQ0pXJ/YVBawyQqR6HUL6tbH/0kzRtwA0RwD/8A0cf0riDTXyJHiAVs'
        '7RJOz4P4UCjWkzRPYeAkCRxNgqN3I+5+9ECnhRptUTN7k8V2OJYQK2gs1r5rh45DyBAmBj8mhEmg79x+ZYY4JIAJp58eaSH+OFSV'
        'fpwe32quIYVBlZI2z9RIzQ2xLrmK+ldMWLygFfEk7bmGadNEsI4T4Yt41jTs2MZqsjQqVKxhT2qc3xiJiKEuw8vtVePUtC8k4lUL'
        'uUBDgD1qpK/i3AjSVhGD5yO5qGzspyR4o5x3z6mur9o9OTcXBIPag2HAzCKMnEvXWofJaZJJGNhK4UVm1jb30+tHUfGZJi2Rg9qb'
        'Zr9L9BHHk471xZ2Est2rRRHA78cVjWbiSZqooAEoalBfajqFql/ctIpYALmtWv8AToYunkiZQFEWOfxS7pnTVzeavazOAscbbiKc'
        '+oYneP5VDhduKtWTnM6zHRmfadpsEsBSMj8UW6c0JIrhmjOMnJxXOlaDdWs7v4g2mrkeoTaaCuzcq8sRTg13kTBHMV+z2P3xPNc+'
        'JFhHFJAfDSY8AIwY/wBKSrvqQXQ8SK2lIXkvLhAf50T1P4b2T3JubR/lzn/uOPsT2q1Y9H6darvaATMO7SsX/p2qc0NyTmQEvQYA'
        'ixb6vfNOJLe0hnY8r/DLBf3o6b74i6pb+DGboKM4ULsGPbt/vVy4vnsFK2yxxqo/SgH9qQF+LNzba/JBqFyny6NtG0cimUarqLWJ'
        'b20boug+sNQfxLueNAw/6lyB/uatD4U3AIbUOodPt07sdzN/wKnttTOrWAubTUZZIpBkbJMf2pV6n6ebV4HgeaQuTwWYmmCyKM4g'
        'ArMcZjC3Rvw1sJAurdcWXidtqSRgk/zJoxY6H8JrZA6GbUMcjG45/kBXzjqHQ93YawsEkhCscq4HY086Yb/S7SNJm37B9WaA+qYL'
        'mtRGK9IhbbYxmyf5s6O0Q7dL6Pk3jszQqv8AU5Ndab8SrfUJzbXejLY5+ljhlI9s471mr6lHfWW2Uh8jAPrQWCOaymEiznwwc4Jo'
        'Nf1K9jgCGs+mVIM7pv0mu26KHWEOrDunIqnLcjUVaSCNkCHkkYNZlaatGyp8tcFXA5Abj+VFbTqeW2cLcwCT0ypxTX3Zz8oodHx8'
        'OYY6m0i01ixZbgZYKyrjvzWXaz0rFZlflBtC8se5+9arI0V9B4lrIFLDnaQaC6hpyBQskrMT6geUH70wmpBi7UkRP0WG3lZI5JhF'
        '6YyAf2H/ADVy0sZ5XmMk3hwJxguCc0yaXpmnQss0qWzuTw2wf3qW806zijlkMUcRfnyjFcxXGZy5ziKFwDFIGVg4zkkHvUizXHjC'
        'ZcGEjGDySakksbZEknbLNngE8H7Vzaq23xHAVQPpIpfIh8SaCB5EZpGw5PAHpRC3ijVIi/DA/WaExSP8ygXcdw9+34ogr8qsgL49'
        'DVSZaWtRvZIEJgZdo/U36R6n80AlnfUpRaxbnZjkmo+o71PFSB2ILthYx60c6esl0+ETOn8Z+efSs/UXnP8AJoaegf8AsLaFpNvZ'
        'wL4wBf1FMFrFDtzGoQUHhdnHYkmitqHO3cu0etJpvubAEecLSucxp6Yt3AaRgCOwr9rkYEhcnaD61JZarZ2Ngc5YgUsa71C16pij'
        'gZATwTTa6dweBE3vQ9z3U70JAVgJPGM0viVl067muyAu08mi1gounSFlOPvSx8YdStLLTF0i2BFxJ9WPahDTNTl3Mu2pW3CII7XV'
        '07NsHYn0qjqF1JFH4aDJPep9Lh+ct0uIpeT3B9DQ+6uZI9We1e3zj9Z7USvTsPU6zUA+4H1GJhG00xAjC5IrLOpuj7HVGlubRRHI'
        '/OfetW1pDdQNHnKk4P2oD/hYRTtk247ZNOIMRGxsmBPh9aDpy0SGa7ce4J4FPEFxDdfxUkQ49QaSdSvbeDfDdyRYHqDQBOp9PsWb'
        'ZdYGewND2urcGdlMciaPrFl80RJhXYds1UvNMa4smgdcZGMgdqS5fiTBHEqxuGwPWrmkfE60UkXce5D7UatwvBEo4ZjkGQW2kazp'
        'tyYcGaEk7T7UZuNFluLPaxZSRyBUFx8SdCZ9yxvkemK5tviPpM0vhsnh5OMmqiukNkQhut27TLuhdLLawmRJnMhORk5xX68gvfnY'
        '7cRszsQoPvTVoAi1jY9nIrR4yWHpTNpdhZQ3oO3xrhe3HapurQiRTawPEK9NaLa6ToUYlRd7Lls1Wv8AS4LhXaIbV/vRgo74a4bA'
        'A4Wo2gMmH3L4ec4HeqVLaMEDiDtesnuKV7oUbQFUcwMvKtjODSneWHVokfxcXkKuMmJOdtNnxA6i6gs4vlNA6UmvrjsGZlC/35rO'
        '9H6s+KkWom6uuirkLC2HEK7M/wAzg0wzZPIgVXHuHpLZ3Rc7kwPpI5/8VJLbQpF5zvyORjimSy1rSOsLfwJ//bdR5Vd8exlfHKsp'
        '7H7dj6Ula9pHVGi6gDqUEtzCDmOWBf4bj7D0/FQ52ruXkQoQk4n4QhrjfvEYAwAKsEhITtI3HjPvX5JBqgjNpbyB8c8YorpvTl9c'
        'yBJYyqGurIIzIIwcQPY2VvcagsskQd15H5pngspp8F0xjtRG06Z/wwCZssM88UTkgXw8oD2ob1o55jFZdeQYNtbeFQFx5h3qS/XY'
        'qNG/5FexwTQDcfU17cW0so3EbRjk0VFA6lHJPc7WaE24RuXPpUMdim8vMyqo5oHqmsW1pcfKWx8SX9T+i0n9XdbtaSw28LSSkHzB'
        'RQn1KocDkyRUWGT1G/qLquHQ7hRHCzffFZd1Rrn+Pa+blkb6cAYp26f6i0fVolgv1VWcYAcYzRc9EaNKzXViIyx5AzkUn4nuP+jR'
        'gWrWPisbdOtI9PAWG6jnQ9wODQTqmaFLsPhgxHGK6v7e9cv4MwbHABGCP3odFqUSyNp+rokcv/T8Xjd9wfSmqr1aLWI6dwpY6fFN'
        'YqzKdknekz4gdAa1cQNc9Oaq6OBnwHPBp90kmGBAGDwMO3+mrJdkOR78UWzaBmdV8hgz426k03qixv3g1SKeOYf6iefxRno/RRrm'
        'nz2N5CUuFGYpPWvp/qDQdH123xqNuhb9L45FJMHRttoeoNPFMjwnt7gUu7qR8e4Va2Ruep88SdO6slzJD4ZYoxUkV42larA21raT'
        'n2FfRVp09ZTTvKFyXJoxH0Zp0qfxfKR2pxawRkxNnIPE+Y4NH1W4YBbaQH78UQtumtYkukhlspVU/qAzX0hZdHaXbTiSS8RlA+kk'
        'VJc6VodvlxeKJT2VSDVSKweTLDyEcCZl0xrNz0HpUwit57ySTsnqPxW0/DkX8+kxatrMAt7u6G9IM8xKe2771R0jpa3u7mPVb23L'
        'vGMQgjGf/saaJrmOzgYIA0uMA44FXVF/KBdzjbLks7FhlQdxwGxUckpjj2qBuH9DQY3epzt4HEKMfLJnkr/tV6C2FvDlSZZOAdxz'
        'RM5gduBOZGjEuHQFu455/aqchHis6AjHGM8Cp548b5H8zAYBA9/QVU1iNxavFAAJgoJ+59qqQZZYC6ne2Fo8cvhpcO2IwCCQferH'
        'S2qX/wAo2n60ovLTGI2c+bbjsT/v3rzSdB2hZ7xVe6IPJGdufSrcsdjAALq4KsD5IYxmRz+Pb71yIwOYcOD8VnkIgtbjbYNvjIO1'
        'HAymPv60Tj1dI7Ml7uG3K43SKuR/Wlz/AC/e3tybuS7l06PdvSKJvP8Auf8Aai2kaVZ72ZhJcOpAMkzZzirjHsSGcQnpPUr6g1xA'
        'loXCfRJIhVX/AHrjT7yW7klF1YTWYSXwyz/Tn0/b79qsC2jdzjCYIJA7GriweLGySYKNwVP6qo1QPU4agjucXNl4cLM5B2/fgUh9'
        'Y9SCSX/DNNkAI4kkH6ae9d06XUdMntI7prXKcOD3P3rBeuNN13RtVghNg6aezZa5Q7g/2z6fvQrEf8RCJap5hkadcupkjUmM/r9z'
        'Sxq+j3MM7zgBwftyKMf5xRNtpAUWNAAc0Qj1LT7qMFpVB78mqipF6nF2PcUIIGRQbuEsnuO4q3ax69b/AMTQtZkKH/pu2cU6SR6a'
        'LVZpduwjGQKAX3TjoXvNMudgbkYPFS2nDiQtpUzU0819MpA9D/MUudc28d3EZ5rdZGtBujwfTHP/AO+1HrkG1vlk3bg6AEfiqWrw'
        'GRDJCB5h5hWLv8VmJueMWpBvSmvTX0cMdpbrJEBskP8ApoBrWvdSv1adE0uOIKXAEh5wPXihkOl30dzOIbuW0ctgFHIDJ98VW029'
        '1Dp3W49Qmk8Yb+ZSM5A7qc1pC2uwbZmNRbUN2Js1toixaMEvJZLi7IyWzjmqraXbFVh25fuSTk0x217Y3emwX8EoeK4QMo7ke4/a'
        'hcjQLLLNEXLf6afRExwIgzufyMXblNOj1JId8ls0bAEspCn9+1HLuzSSLcMsrdmU965a4MkOx4FYvwQwyDUdvLJYTxQQQtJaS53j'
        'ORCft9qMFTHyg9zeop630rd6hdn/AA+4licHldxIP3ox0v0da6DML7Wb0TzjGxG+lSf7mmOE3MLkwQqgbuW+o+3HtUZ08XBDXf8A'
        'HlzlmIwP2pVtNSH3BYz91aV2k8S1FcvczM0cTJCDhfvUkkckk4XaAc8ZqCSeOwhCoixRKCTg8Cu9NuVktS0bl84O6Q980URc/wAn'
        '68S0QLJdTBCPp5xk/apotiQDG4kg/UeTVO8tI7m7ikuSGjTlEI9fvVpImeQt+jjGT6VcCUP9kSHdywwhPb7114CtKXLFmY5walZV'
        '7Kce/wBq8UBGYgZcepPcVbGJHc/ONqiKJRvYHcxHYe1UbbSYLe4a5D7pmIJdzkge1XhKwbKYxjnPtVKa8fd4UUe8E4P/ADUE+5fn'
        'qQ6pcq062sJZscsQO9TWCNHGpWNQD9WO1exxWqKrMMk8licfgVchiHpnFVk5wJ7EhMhCqxH1DHYVay6MgKkhu5B+muYW2tx/4r84'
        'kMqkNge1STKTq4iaY+GrEIxw2fb7VBcvaWluLQwNPvJGzbv49zn0rtQGn2rMQQM4zwa9nitRIyhm3nu2f6VEtxMn134V6HrOrvJp'
        'tzcafLK27bEu6NfckHsPwayr4h9NdSdEas0V9umsd+ILpFIjkHp+D9q+sbO2itF2wx4kI5xyf3NC+q9P0rUtNlstUgt5rdx5o5uz'
        'H7e1QagRLC0gz5UsOsNRuGjsIIXmaTyKg5JNbp8P+gtdXSEuNYuNqONwgz9I+5oXpHRPS+i38usaIHVlwPCmJdY8dyp9vzTVq3VL'
        'y6QVlmENqi5Kg4Zz/wDvSlH8inAjKbGGTF/pjWX1sSXEj+YSnbgYwCOP7UdckRFEcgt3yaVenNNm0m7uInGEIQLxjtmjk8hB3BuS'
        'MkZ7Vga6weU4notEhFQzKWqR4VGAwSCpH4oNPAk1tMshyrKQwxRe8m8WFQx7Nmg91dRJnBHm/pQ67T6hnrB4Mk+H3VVrpOmzWGo3'
        'T+GkpMfHOPxRnUviF03b27+FcyvL6KEOTWe640EOn3eoOgDRxZGPbI/5q/8ADrpGbX/C1nXLdrTSvqjRxh7j8D0X+9a9Wpvf4p1M'
        'W7S0V5ZzzHLpPUta1qMapLbLa6WWwhkY75f+0U6WsE8scDZPhpnj3Oe5x61VNzE9t4dnbosVt/8AErABBj/aqltrS6lOVSR1iQlQ'
        'ImwCQcHkelaKAjgnMzHIY5AxGdlMTRg5fIxuz3qSGJQoVg4T2zxQmDUmVI/mCqMWxz3qx848jLGkZcH9Q9vvRhAkGftRsbKZDHIA'
        'FHJXcQGx/q+1VWnt3YpblQwOAM4PHFWVSaXymIDdlduM1KunkurlVQJxtUDzfmrYzOyBK6zOspUKzsMAZwQfvxV/fI0KlHVW7NkZ'
        'zU8NvBCvCbQeO3JrgrGoLt5R7HipAxIJEqNN4cXiEcHsPVqhhleVgGTCtyRnn71KkvzahUTYoPfg5xXskaRKNjsJWODtGa48zup7'
        'PEZCNwIA7n3qlcEwB5okOFjIAIyaMlTtHiOFOOQKhucmIiM5wfbvUGSDAMFkb2QNd7/dQfWjaSRxMsbtjI44r20jVE8Rxtb15zj7'
        'VKqmTOxB7KT6VGJxnaMcjAzntgdq/b2kdk2sSDycVEF2DhuRz37mp45cKA+1WPrmukYnUarHnaoDnksa9CLt34XPua4j3szEsNh5'
        'x60I1rVkjhaKCVchsd+O1WHEiWNZ1iOyUxRsjP6lTn9qTdWvrm7dWkfMfJwB2ruVxdYluCN36QnrUF2/0xALlSc7D2/NULQiricO'
        '8i6eqRx7IyxD8cEUgfPTax1FLaSAxx2Uhi2Lzub1P2p7MiSDYiHYP1GhVvb29t1PdXMUAJkKufL6kDJ+9J6xyibo3pUDPgxj6oup'
        'Ft4wLfwnJOHJzuoHaSq2WkmDuRQmyOrJp0Njqd5HdmF28LbJvZB/pJ9f3q1biGF95OT7V5jUpsJE9LpiCgIn7VD4YQISATyTQ9Ua'
        'aZbeKPxZX4UDnNFl0691y8WC0UrAgy8uDjPsPvTz030uukzlgyRYGS7YZ2yO2fT9qb0OisvAY8CLazXV08DkwFoXSNvbKs2spHdz'
        'P2tQuVH596Yr+wNxJu3FBEB/CU4Ufai0EKW9x4ccTzSsM+JjtVo2YR2EjKFHLDHP716Sqla1wonnLb2sbc0XpNBubixMMki+FKmG'
        'jx3HsPWp7HRo9JjJjjYKUAKqAAMff0o6ZI4kHiLg/SPNyQPWq0sk87I4VI7YfVuzubnsPejAKIHcYOjjjmbxbiAJEH8it3+xzROM'
        'RiRVfy7j5UQcfua9hKSSP4cJKYPLcAEewqBpyY2MaBVB5Oc5rup3cvEQIQFjG4HgL3r8A7oTIGhB7A9zVRVMZF08pSPH0kck/wDF'
        'RNdZuvGuLuOCALgIT5j96ndKlZZu7kRjZFlnUYHPGarNDJNGnzUhkPptGP51WudQs7fe8UTzs36jXFlrBcFZMIm4YyPT81UtLBTL'
        '8UJjiCW4VCc4FTRxrEA7SB3zQ7UtRYW7PbtCCvY5B5PpQGfVbt2TZNxxvI7nioLSduY4SSqMq7gZGQx9KqwxzNK88dwXUdlJwP50'
        'tvfz70ilmBUrlcjuO+PzVG81Yx3Swjxj4jCPZ4nf749qrultnEdpZgkSGeeEN3IU9/xXTXdumQ0wA9ee1Js+5/DdwSU4X1G30qm8'
        'tywLtKuI/qBz/OpJIkBczQNyqhdWBzjAz61Rn1BEw7qY8EjnApWSeYsAkxcjAODgZ9DXmqM6QD5k5LevJFXXBlSCIUu9YuARHBI0'
        'YI59Tn/agEz3B8SMbHOTkt6VLFJFhZZZTvZsADv9+ar6kQUV7dmDBu7etVMsBJYkYfX4m7AGVPFSRwAAPyq4zgcGqUNzeorSLsGB'
        '2bkVPZzT3CASuc7cthcAiozLATpVQbtu0Ix7etDtX322pJLDja8QyTzyKJ74EMQWI7mPrQzqy5ENjHcPBtChxxz2GRSeuGaTGtIf'
        '9RP/2Q=='
    ),
    'dark_eyed_junco_05.jpg': (
        '...'
    ),
    'dark_eyed_junco_06.jpg': (
        '...'
        'DPHGOGmvsf/Z'
    ),
    'dark_eyed_junco_07.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAABQYDBAcCAQAI/8QAPBAAAQMDAwIEBAQFAwQC'
        'AwAAAQIDBAAFEQYSITFBEyJRYRQycYEHFSORQlKhsdEzwfAWJDRyJWJDU4L/xAAaAQACAwEBAAAAAAAAAAAAAAACAwEEBQAG/8QA'
        'JxEAAgICAgICAgMAAwAAAAAAAQIAAxEhBBIxQRMiBWEUMlFSgcH/2gAMAwEAAhEDEQA/AOtQahemqMSAg5UcFVLsmKlo4Ud7h+ZR'
        'p6Ta4MZsoZAK8cqPU0v3GAhLqlk8VhX1NWpK+BKzRVfaUHgQOB1q66tWG9/TGBUxbBd28HJqG9J2MgA8pIrOs5JtT42nMxK7jTou'
        '5u5MNas7eUEnt6VZ1I54xXjKQoYzSfaphhSW5IGdowRR203BN2MhkjGDlI9K1eFys1Cv2JyPjUREKnQ5zjYcUUhROc006bdYlx1y'
        'XU/qt8A+9UbtaZjNwRKQ3vaV5SKPNRo1mgB2QjAV5iB61dLEjxGqN5jlp1cdUdDjzfGOc1BrS82ZmHtbYaDmOoFZ7c9avsJ8GJjZ'
        '6UHgOTbzcUOSdyWd2Tn0omsUJ0UZnYNhwJptmZZesyZTrCdyuRkUt3Fa40svI7nG3sRRqXeYse2ojtEAJTtwKXw1cJ8o+HGdXnph'
        'JxSlr6AYE0VSvtj1CSHY05jacZIwQe1DpOn0oSp2PhChzgd6rXqxX63RTO+FcbQOcg1Npe//ABu2PJO14cc96sghxhhK91ahsKcx'
        'ZukZTzq0rTtcT3xQ9Ml2GQUKIUk1oV7trDyS4k7XfX1pFuMQtyyh1JxnqOlZ/J4hZgVlWxIXsV4mS3Q2R+9Frm0sskudag0pHipU'
        'hXf1o7fIwVHJb5GKC7juRswQIixp7ltuAeQrAzzTSu7szmAknIUOlJ93YIKuKH2+W4w+EKWdv1oqLDWpWT+o0SkFrO3gdq70zfRF'
        'nBhauc8GqjstDsbG7zEUHQws3FpSOu/qKQENrHMEDM2OXPbXADqj2zSq9fowWpJWODVx+O+uykIySEVmbjLqJS/GCgd3eq9/Hak5'
        'IgYImtWHUlubCUleKNy7xBfj5bdQcisbhPtpAxRKMX3V/plQH1quvOYNhhqX6vyVleFHiNb9xAdUlKuO1UnZpJJzQl4uMqBXmuvE'
        'DieDzV3AbYg8jltccmN13uDLZJSs+xFAHrmXUqTuGKNSrawWzueCvSlm4RENO7WlZOegqxzaLS3ZTqVGUkzphQ8UKIxzXkplcle3'
        'nBqJht0HKkqx24oxbCyjBc/rVGngG27LeJHXsYGciLjpLah9D615pxxxi/eU4BTyKOXAsvpKAAcdK4sluaVNU4TghBHNaKcQU2dg'
        'dQlq+wxL7cpEqU22ojYjlVK2vr2VKEJHzIOcg0XS4zEgSHSfMCQD6mk56CZbq3nCSpRzzTLeUFYZj2brqCo09tUgJcbGc9cU+WiM'
        'Vww45htBHHY0Gs+no6HRKmjYynkZHWjL8lLn6bACGwMJNWK7AydoHydRkQ7p5NpbnNpko8UlQ69KaL1q1NrkqiwoUfwwkYWKzlp4'
        'NJ3NcudM1Qu1wXHZUpZys+tZvJ5lth+Hj+f9iQ7McCNF+1nIfaKJawtGPkHSs1ud1QxOVKitFHOQOnNU5t7U8+SsYHSqUyUh5vaK'
        't8bifF9mJLHzHovXfuGbfreUt7ZMBKT0UKOtzY85IWlSVVnTTQU5jBo1DhO43NqWk+1WmsCDcJm3kx1txCHQkeXnqmm+3p8aMULO'
        '7jvWd2duep5IOVJHUnrWiWl/wY36ycAClmxbPEHR8Rc1RaSkKcQk4NZ5c2H23jhJFajqC/RGklKsAH1pTdMSc7lGOaQVCnJkdYpJ'
        'lSmeuSBRS0XJHxTZV1B70Xa06/cJiIcNoOPuZ2pyBnjPehUa0hi8GPKQW3Gl4Wk8EGi0BkQlTJwJr1qlMyLWE4GSmkvVENtS1KbT'
        'g066fjxDCQG1gHGMZq4/YG5CexzVG/lFhhhL54DEamP2mA6uUkKHlKq/QFg0TZrlppLjI2SUJ5I65pOe021H3O7Qnbyat6e1ui3z'
        'kQY54Udqs1RFi9gWGpn20NSftPrroKTIiOSIrwUpsnKT3pIfhuxXSy4gpUnggitDveqjZLqlwO/oyeqe2TQi8Nfmp+KZbwVc4Fai'
        'UI9XanREQy/5MfY1NdXVhHiY3U/6PtsqekPyVbjWXMI8NSVpHIPStn/DZcr8vD8pstNgcA9TWiqjOWjiIxxtPtLASEDB9q9u2jVp'
        'hF8YT6CjtjkGQvxF7W20HOCetLettbFdyTDinay2cLx3NRdfXSnZjI7YinOtM2IvO1Sk19BZllKlKCkgjFNtru8O4BDSwkr/AL1f'
        'lx4vxDDKChO45VQdUdeynzCVx6me3eAQy3FSlWB5iaFu+Fbkb3SCodEmtYvFhIjLdQhKlbSUmsOnwbjMu7iJSVgpWQEgHpVK6le3'
        'Zh4gO2Tky7EvJm3BDc5RDGfkT0oo6mO44fh0KCB0zVGFY5jYU4GQhKOgV1NEIjSlr/7g+Ekdj3rmL3jCwMsdCfREBx5KBwlPU1V1'
        'DC8VKiBkdsVcm4QQIZwf71LGS8tnDqQfWrnE4i8dcezGIvUTMrjBWHCNpFRMxD0Ip5u9uSpSlBBB+lBkRkpVhXWnOcDMMmBoMZXx'
        'QJHQ1oNktwcbRlIOaXmYiS+AlPNaDp2EoR2z14qm7uT41CqrNhxJI1rQ2nOBn0FUby+/EbIQknimtDPOMVBOtokpxtFQ7gD6zQT8'
        'a3uZizYpeo5vneDQ9TTnp38KHsgi5hQ9sV9Ls5jqC2tyT7GvYt4uVqVubdUUg9M13FYnTiU76GqbBnupdA6ltAEm35kFs7klHBpX'
        '1va0SYTl9jynmLoy2hU2GplWAem4K/atj05rxl5CUvLSrsQqrWunbWuyJu8eGw8FKSxLbI4W0o4P7HFaBqXr9YlT6n5vsuqLnEWA'
        'pZUketapo/VTc1tKVrwruDQbVP4YSYbaLjbEF6E+N6QOSgHt71RsWlp7DyXkEpAPKayOQvXPYSxVy3qP6jl+Id1MWyfoL5d44rHm'
        '5Tjc5DoUQQrNahrG3PP2FPlKlI61lTjS0u7Np3bqp0qGzB5VxuYH1NfjaXOrrEysukLAG0571Dsn6VktwZ58RPQK9RTh+FrXw+nm'
        'XFEpCUd6RPxTua7ld/BZXuKDwodq3UVaKQZUY4lPQ2irciOm43dwE9cHoKJ6kvkZhkRrYAAngEVR1Be7c5a0xoa1lfQgUswWZEh4'
        'ICSVKOBWbzuaQvSryYNtvoQ1b7xclOqK5ikJ75OBXsiZYmRvfU5MkKOTt6UM1AtmOEW1AHiJ8zih61QQ0FJHrWO1xoHRtn9xPbrq'
        'F3r00zlyDHLBx1z0oS/fLg5I8UyHAodDmuHsqATjpVZbKj7UoXO2mMEsfcYG9cXpLjRclqUhAxt7Gmm03u2XFvx0NoEgfOkjmsrd'
        'bUDmnH8PLC87I+NfVsbHygnrWtwLLe+Acg+cxlZOY4vqRMQRs6e1ArlZC6oFO4HPGDTi1Eba/wBNxP0r6Ux4ZyQOeabyFamzsvuG'
        'cgxAmsGCzuWnzAUEe1E2ydpJSR6in+8wGpaeU844rM9X2sRHCodz0p9HLbSmNV5bbv8AGkZSpxPPvUjL8ZTm7CTSGGVh3IBzVpJd'
        'SNqitNaHfMZmO6X2BISPKCT2pus9wCEJGOBWTR0OZS6lSiUnPWtAs76HrSHc4WBiqr8sKcYml+P4ycnsO2CNx2TKEhseGoA+tcsS'
        'HWnNq+R6g0nWu7qbdKHSQAeCKKrubasFCxmiZK28GJPLtD/28Rhmo8dkqb70Bnw1ZwR9aMWyaytoJWrGfSpXktuAlPNQAUPWX7Ol'
        '9fye4jy7OsqLsRxTTnseDXce6XuPbn4U1lbsdxJQrb1+tMDrW1zgYzVBxb8Z1TmwqR3FMe01jMxGXc0zRd7aZ0Pa3Z2CwrLOVY8q'
        'h0B9K6u79kQoPJU22pXTB60h6Mmwr8q5aaXvbfmICoairCQ+nnke471TYs0pE1TU1TilsqKfmyARxS7eUrV7GYJEdJKWH2SlICkG'
        'gTelLMZXxD6Ug8kD3onHDjUPclO7A4FfRm3JrgVsKcdj0FY9NTZLqdCSmAex8T6bePy+yrhRWskDCcVnbKHVSHJEpA3Ek81pFzts'
        'RqOVyZSUn0BpalQYLzai24tfpitMm4qPlIAi+rMfEQprZZfKEpwnqDU0OYqKPEQMrPAqwy4w6hKpCc7aEX93wpaX46ClocCszg0d'
        'yLQfEQozuHUWUXxC5EdeyQBkhXehJhT2VKQpheU9SBxTNpJ1t1gSWlYUB5xRtLja1JSAn9RXI9q0OTwKrgD4MMVBpnCGlF0lQIV6'
        'GrQZG3zDmnTU1tthUHWMpfx8gHWllKVDKS1gg/xCsu78Var+dSPhLNF2e42x5lJ/pViJrCQwkNMoKUgYAqxNhCQop+Y5qFjT6y4k'
        'lokE+lX+LQ1IwPMJa+uoatOsZTigHWifetAiThcILa8EHGDmlXT1kZZOVtg/UU2RUIaSEoSABRXCwaeSVI8yJbZzzSLryIpx9OBw'
        'OeK0CRnaSMUvXaOiUClScqHTAoKkycyR5mdWSCybiEvJwM96cL3pmG9aw42E7scEVRdgtsyE8cg00R1oELw1KG0CtEdkXJhnW5nl'
        'utD4eLW0kZ600Q7K8xb1FKiDkkimC0Qml/qBINFX2QlgpSMZrPdCxLQ6eS1LdhM6RHLTygtPPrXD2S+lCeM0fucRR3bUc0GRDlKk'
        'BZSAAaQlbs2YAPuMVpgvGLvBPA7Vba8ZkkrVn2r62zUxowQpWD3zXr8lLoJ4Oa1q7M6mmfj+Mb3PkyCpznr71Wv97tkWN4TriVOH'
        'smvEbFKOeM0GuWl4ktSnEvL8Q91Ux2VfImexkdp1Dbok1EmNHJdQsLSfcH1rV9QzIifgLs00kxLkwHkq9F/xJ/esXVpGe1kxngvH'
        'SnjS8a7338Nbtpl9K27hbD8ZBWE/Mn+NOahWRxjEEHOoxRLva5Syww8gLHbNBNSagRbwqOh4JX14rP8AS0K8fG+MWXSskJ780c1p'
        'pOZFgHUMaUZkNL3hvgjzMK7Z9j61nd/iBCDDH/ySB9ZcgXdD6FvziVpAyAo1EdXRMFLCUIx2NK8ed4wDQUMYPFQy7PFcWVDegnnh'
        'VLoVmHazcBsgYBmt2/RdilqWwptxPoSrrQ3VmhYsWH4LLu4L4CVdRTFqdx/S8fM5t9p48NqxlCz7K6Gs7u2qX7hMQlUpQcVxx0rZ'
        'K10jqoxB8akmn9Oy7St0uvBTRHAB5orAslynXNCUAsJxncR2rmI58MhsyHSvB3KyaaourLYltKytCXEDoaiqsE5MLwI16V0vbbey'
        'HpjaZD5HK180L1dbtJS1raX4Ud7HBHFKN+/EGVMeEa2AhJ8pX2FTQ7B8fF+KllbzvXcVGrDMGHUCcuQYoyLW3b7mpCHA61u4VTAw'
        '7AZjjft3Yqpe4KYyQhKsYOeTQeQUhkqUvtwM1W7CuEF9mH03GGXAlJAr03HJ2sJK/cUiMyT8Zt34pis6ytw5dQy2OVLWcBIqhyeQ'
        'GHUDJ9SCR4jO1IQywHZoByMhGfr1/aoG72tJ2tkeErqG8gH2OBn7VBJjRrs4iPY5LM5YSS4hTnhrcVjGEZ4+2aV72q62aWkP2i4r'
        'fWspMcpUjA7E9zk/atKqpaxgDccB1HiaEzKizGsTYzMlCiAfEZ+X9sq/tQLV9nNpCZcVanYDhA3Hqgnsfb0NdQIAKkhYiM70pOUq'
        'KlJUQCUj1wePtTzp51txKrXcI7EiO4NgJAAUnuCM5/aitq7riQwDDBippp6KqKApQBAri83BpgHCwR1pZ/EiG/o3UTsSKpxEN0eL'
        'G3HJCD2z7HIpVavcieShRJPSsu61xWV67ErOCNRrlXtlZOOTXzF0ZKcFs80FhW2W55wyrFWHYsxtaUIjuLVngJGSftWIbeWx14ix'
        '2Mao0dl9oKKcgjNDRht1TYV5QatWBU+SyW2Ij7pAwQhskirydEape3TPyxbDOfmeWlvr7E5r0tZ61AgZMtLoSi3HKk7grFfDKVbV'
        'dabP+gb8wlpLku3edG4/rfKe46c18dHNNf8AnX+CwvIAABV/ig79x99SNmLrCsEAUx6VmG3XqJNXy2lW11OOqFcGpWdM2RmTsf1U'
        'wkAEkpayOPvV+JZYMt1DcG+Q3AVYSVnaeO9FV8Qb+wzJAMB6mQLVe5aGQlLAUpbZ9UHkY+1LcK8oWuXAeexGmoLa0HoT2rR9f26V'
        'NhRpuxt1+KgsyC10UlJ4UPXisnk2pSJu9pBdTuynb/v9Kpc+thYtqeoROsRViWx+Dcnm1J4Sopz681fWSVHij70ZTqdzjex0HasZ'
        '5BqsYG0FW3pT1KhBEk5m0wNSw48D4TUq2X2XDs8FxrdvPrt5H7Uqar0FpWVdo0iyxHLZLU9t8MKKmlZBO4j+HsOP2rD4Mi8TEB60'
        'XBKHQCA226Uhz/8AknhWaYtI67ulouCWri3JTJaUEuIcIyvnnOe/of61fLh9R2pc1Tab/Cvhg3GI/DIGW9w8jg9Uq6EfSq8OLb2p'
        'QTL3yF90g4FbmzqK26ksLceU01Jhrx5XE8pJHO09UqHse1Iupfw4lNJeuemluXCMjlbB/wBZv/ZX259qnp1H1gkf5JdNKsDhQ05A'
        'bQn6dKc2ZmmkwliNMbQpA6BXSsTN4mwULbajlLg8qt4wQe9AfjpTjy1blJKjzih+froCdgxk/EW+ATVNx1hSvUdMUAizXJDB3ZyR'
        'UUuMHU5WrcrFSW9tKcAdKq8jBHacdCcRYq1SfEOcdq0Cx6Quki1idI8GNFUjLZccTucPYJT3J98UCs7TCZjBlJUWQsFQSOVD0+9a'
        'GuaPzRqO8oeKoEqaCsBlIHlQMdOMZxVPhUrynNj+B4ECsdjmKrdusq5qnH2X7dJSj5kkobJAP2OeKNtw4FygIdTcJSlNoShDbh8V'
        'O487QVcj6dKuTn0zXJcF9tlK1NBCWuNqUcqJH/2/fGapLt9vmpU+puTFS8tCVhpRSpBx5VYGQc9+M1uYEeJHFEMFcZaw9NSNuxOF'
        'K49McfemG0tNQWzJmqKVBIABXlWPp9+nFLf/AE5+XFEpMtRJXgLUkpUcepJ54647ZohGk29tbSpDEV1ha0/rAZShWcDOex6VGcDU'
        'MLkwvrnTkfXVgCG2/BmRgVxiBws48w47HA9wayjS2j5iLyqG3BkPSEL2rbDZJSff0rc7VLjB1EUMLSU5W24jKdqx/Dn0IqeTdZbN'
        'hdkW18NPOLKkqcbykK6bFkdOeAe30pYGTkwrKwRqUrVoRTMVLlzfjQkBAUpO4FYHp6ZqL820pYg6/YG/zCckbN6XElQB4P0GOtZn'
        'fLjqaVOWm8vSE4V5meUhJHbFRquJbYIACTjsMUg8hMEDUplwviNj+uNStOluI1Eis8+RJJP7gUu3PVmpJwlNSJpSh5QyEjgAdsVX'
        'tkhT6dyuSaimNYKj3rD5vJcaVtQPkaVHrrdVRUtyri64pKlEOJJQcHtwfarsBUJ1nLwLxxjLiyo/1oFcydpxQuPOdZcLeTiq3E5B'
        'yQ+4Bds+Y2iNGL2xLDe3/wBakk2eOy2ZjSlsrUNo8Jwpx78d6G2qWZL7TIVgrOMmiWqh+W+C0hwuZGTzW3Q1aUlyMgRiHAzC2mF6'
        'rgSXPAuDUyIQnYJKjuPqCOlF3r3Y3HVt37TD8JWcfEReR9eOMUsae1CApLLo2Z4znimdbzctshIyQOMUddtdqE0mELMyNix2S6Ou'
        'uWG7NPOED9J9QSo8du1ULtYLrb0kSILmMcFI3D9xXz1hTKdS+ltbDiTnxG1bFffHWp4Uq/WyUGYNzecA/heGQPvUVo3/ABzJGG3M'
        'mdsrctb06E6uE/u4UR5XMdyO31qm+pDhWxdYzqinhL7agtST2OfT2p8Ta0XCMUwZcdW7ondzSrd7NcrYFKeiOFAPzpGRTXVxjAhl'
        'v8nWjtWy7DI8F9xciGtYJWOcEdFbex9u4radPaveEUqizElqUoKSRyFK7c+tfn+15lSHEJbQpIGVIUOv37GrrNxesctLlrW+wgnc'
        'uNIB2n/P1HNWFuXPXO5OQZ+i9Q2jTupEOfmDXhzyNoksYCt2O46KH1/esr1hoK8adBlFpMyEflkseZI/9h1T9+PepNK6wRPlJRv8'
        'Bw/wbs5x6ev960a0aoRFSlDzvkUMFJVkH/AppVXkzBTGkPHCB5fWjNvtrYYUVKwpI/c1rsrT2mrq4t9LDcNbhzuaOEKPrgdPtSJr'
        'OK7pcpX8G4plw+R3GUK+iun2pT1hVyZDEYgy3SWkOBKkKDiCCk470WmKkol/FxtqWgS96qUs8bT60AgattipCUy2EoBPJA6VoMOP'
        'bLxa1v2mWh11ACtieScdvrQUKoP08QFYA6gX4yT+a/GuRg+URcx88bz/ABk+h/xTDCU7GdL4hpXDmAKcAOVtqA65H/OKWo5ddWzG'
        'aSpCkBaty+qTnp+1FIEqY02HoxQkMLBkxSflTjBx7VYMepjc5JhTYKxIbYejDAUl0hKgfUHt9aRplzt8C8mMI8V9GzbtWTteQeeO'
        'gOCOo96PxWgm6SXISGZLDre4sOLwUnHIBqK5QrLORsvFtUxHcTlK2EgYPclSRQrqGxBnKdXRfhW2d3hKQnYgob+UDsR7enWvoeok'
        'MSXw24lLz3nWzu8rw9Unp/znHWurdo7TUtO1m4OBlGMgr/VOOnmP1o1B0RpdghSw88pJKkKcdH6ZI5I9+fp7VOQTO2BBT1ysNzjO'
        'G5BbYzsClpIeQrsMHkj659jSrcLP8O6laXGpUNw/pSGvlV7H+VXsaq/i1GcsswzUXUXBlCABEcUSpCO4Dg6euDkc0s6c1k40lCUb'
        '0NvJwY0gABwZ7Hor7YPomlW8dbRuJcBo4sW1SEbo+T7VO5a5S2/ECOtdWG6MXBtUiClYaQoJcQ4QC2r0J7/8zTGxKbDZSocnmqbf'
        'jFcbiDXM1vERbSylxO0+9L0tnYoqxk08aydbWCUDBTS3bGhIkAEZJ9azB+NNduMwDXILRHdDCpK0rSB8pr52Q9IP6zhXjuac2IoX'
        'b/BWEgAdqFIsCnJBUjIHpV/kcJjWK6zDZCFwIGbTtAKTzT5pKQhtkF0gnHU0BNlDTyRjn3oxaIIU+htBPJGar8Sn+K57QFXEZn5r'
        'a2D4Qz24riNGUpreeFKq7KgIREDTQAAFDEynmVbM8Jq7fzlrGFjGbGhPzozMeiuh1h1bageNqiK0nQ+vWvATb7yA4FHAWsZB+tZn'
        '4ZJ5rrwwkjkj0qarTX4k6mxXnTdvkzfzC1JbZURlxKeih6ig8yCgNltaApPQgjNLEDUMxthpgOL3J4znqKb7UuVcWgQgbMcqNRfx'
        '05B7powSMxUuFkcQoSLcva6gghBPXHoexqxD1EtSG0TmnmFL+QrSQFfQ0Q1Baru6rwobDjg9UigDumNRFQTJgSlt/wAq+RTOMLkH'
        'WzcYufcerTqRbSk+GtCm/DISD69KcdParix4jkOclt6O5wpl9IWhefY1k8XSWqIzHjQI7hGM+A4ePsf81QbvwjzFxbmy9ElIOFId'
        'Twn/ABV4MR5hzSNS6M0NdQmbbpD9sdcVhTLI3t5z2B6f29q4atn5DDRbrA0tAe5ekqWdyyO6jxgDPCR/WleJc0pLbjp/SIzlC+T+'
        '1HY2sG2YzzNwR4sM+RlKUbl564B7etCEQHsBicF3qWw7AgMqjNyCtTgKXfFV/qZ9+3WvWmnIrvxyZPjJUkNuIPVSO+cd6AKlaTnJ'
        'WhUq5RXzylawFJHtg4yPvVyzOIgLbakyw+0RuaPhkEA/U9K7MZ8Txwh2xpKFTkzHFtJxhbZw42e6SO6avKTcYrf6F1Y8JQyUONeG'
        'VZ9ccfegse4oZWpKwVIWgpVt7iiBcCoSUoCFjGEoX5sj29amBmdSXW21ockMHeR0Z2q+4I/xXjMpRfCYoWrf/wDjc6D3/wCc1ats'
        'ltTYZ/TjKA9MD7VDNcLEgpLO0uDAWkZB9zj/ADUaELJMiv2lLffQ21NU8whSgt1O8AKA7cmvLn+GOmJ8ZCRIissoSQrKOSfZX8P7'
        'VFFYeL7jqlh5JIVgg8eoGelXX7xaLcfECUOuHhTaVbh967OYPiXIbOh9OWwWdkSjJd5ddjO8nA43bgUn7iqD7MeW+Grfc7lgjAD8'
        'FLqR90lNVxqyO64ERLW2j1JFWlamnspCmI7SSO+KIGDmcXLREmZHHjNMlRTgPtFaTntuQof1BNCG9G/l7W3xkuSldh0FWrhrK+Pg'
        'pLwQOmAKDqnz1vF8vKKz3zS3KA5MgkQzbdLzkEuSXm0J+uaNCzw2Y/iGc1nHbFLUC7SySiQtS0EYqyxBLytzSipKv4SaHsvkSNGX'
        'FwrevLy5G8p4GKvWx2AErQ2gJX0Boem3lvjHHpUzEYpkJwKp22hWOoHbHiMimNzKTuHSgb9lkvbloUnBV600xY+6Mk//AFolaLUF'
        'xxuHfNUagjkgiCMZ3Py6vSy5bqlR/wBEK6IVVG7aQu1vKS42lQX0waeb/qKwRsPQnw652Smlh2/ybtcW1vKKW08AVr3rVWP3GQAL'
        'LOaSFuNlIBps0s5co8ctRnUhIP8AHXshtyQjIUT6YquxHktK8Ra/DQKqU2/fQneIxuT9QQm/FbkRzzyODQa5621IvKC62MeiRRa2'
        'RmpLIHiEkjk5qK9WFIQHAOnWrp+TyDqTkwdadT6lm/piWpI6ZAFV9QWl66r/APlFl5YHCiOR9DT1pDTbKYiHSjJPOcVLq6AyywFA'
        'YUDxS7PkCE5gsT6mLv2O6WrcYLnxbB6tnhQ/57VPZJ5f2syssLQSQhwdzjP9qcXYx3Zz16VXeszMxQLre1wfK4kcj/NIo5ZY4acl'
        'h9yKCywpxbrkZClbFEBXIJAyKngsPR1IlSfEcSrCsheMmoExH7e4lt/CmlHAWOh+voaKsBbiCw0rcFeUn0+lX1YNLAYjxCFtuTE6'
        'QGUqPjdEhSetGlMPEgHKdp6HtQWNY0JWpSxlQ+VYOMH60xWCZOlzE26VFVKWkf6qeCE9yT6D3osSCcnMr+HJ8LwkNkrPyFNSfDSG'
        'Y6pchDq0oPm2dOKYXUIDDke1rbkOgkLUVALPsken96XZk27sNuQni602oYKFowcelLc9VzIJxAOrtQSFtMt2jxI5wQ6SAMg9qCWx'
        'p8t7lqJJ5JPU0fUwyteHEg1YZiR0p2pqn8luMkRRJi+2+uPL5PFMsOSl9nk0IukHd5kA5FRQVutYHIqxS5I3OEJSWMryPWp4sfcj'
        'BBJrqKrxB5hRa2tIVx1rnI8zjjMpRYJLvTij0CGlBBCtpq5Et25O5Irh9hxDuACKBGAMHOJcbYD/AJSRuA6+tcqhqbfTlP3rqAFJ'
        'Vk5zRlrCwAtOaTd1YzjgyzbmitlCAM5pqgR0sMgEcAUJtSUoSnAxiiMucGmSM9qqU9UJJgjU/DLMTYguLwMdBXLLzrb4KkkJzxRJ'
        'mOtxI3JOal8JtJ2rSD7VqmneY7EYbDOYUyPEx0qpqi5MoTtbV1HaqzQQy1kDakj7UGuCEurJ3k/Wqj8Uq3bMHriXtO6jXEkobcBU'
        '2VevStehFm4QkbgMKFfn5xlxCtyR09K0TR+qFjwGCy6t04TsQnOfpT6rlQdWM7xNQhLVASWyD4Y6GlrWtyQ8tDSSMZ9aa7W7KU40'
        '4YiVKOfI4Mk+uB3pinwtP3aD4cqzx15GFt7di0n1Qocg0+xfkXAnGY62tpyOSFA7RmuXFpKUbSBmi+vNAzNPxV3azSlzbUsjclf+'
        'qxn1xwR70oRnVqUhKyfKOlZ9lBRhiQFjI0w2+yW1pStKhgg17BsE8PqMYtrZOMbl4Un296jt6yG93bpTnYUD4Er9qZ8rVjMYTgSK'
        'ysocl/DOwE4bADmVFSj9B0ovODKGlxLa38O24nzOpO47uw+lBrpOVHS4pGUrdwgqBxxn/Gav2GQX0AKGfZPar9bd1zGBtReUxKtk'
        'tTYcCXUjnnIIPeici7ypkJlmQ+UraOUn19j61Bqe1hmUZLLjiQ4cgk5GfSgSpDzKv1Qoj+ZJ4oSWWCYzv6aYuaQ9CltoeUMlKhhK'
        'j7Y6UFnWafAUn4llxoHgEjg/Q9KIWmZ4iQpDmAO9NdvuKHFJZku721fNuTkUKOraOoMSURStggpJIGaoogjxsFOK1d+z2R1tKi24'
        'z4g4W1wM/TpQx7RwefSmBOadJPAcG3+vSp6Y8QSIlphhKOKu25opIxR65aWu1v8A/IiL2/zI8yf3FQQ4pSeQeKTYpgkQ/YW0uMDI'
        '5FSyrbveJArmyfpr2+tGXEk4UKDRgwKm3ltWccVbaaCQKv4yMYr1DIJ54FIuG9TsSeEghsGq1wyQoc0TjoAbG0YFePd0qbQtPfI5'
        'quKuxxO9T8ewpD3iDa2HAfUUZVEiLhmTIT4Sx0ApHi3uUVhMZGT64q+7MmuYVMeyOyR0rc7AQ9xrgsRJDBbGCaB36wvpc3xfuK+g'
        'PKba8RBJIqaPfVuPltYJx61BUMIUDKtc1CPMgnHXFH9LtiOytCNzcpwYLn8qfQe5plskGVPG74VIbIyPEWlJI9geanvemXrHeI8m'
        'YENsPDckIWCSodvYe9Z/KqIXNcXYCRqEdPTFQX2kyC746E+VJOf2pquF2fW7EW26fCcOFAf1Ffad/L7j4ImW6IQEnyuEnKfUK6iv'
        'L/Y0W6CibCLq7c4vocLMdfbkfwn1NOCslePUUAQITTcH0Q1qRhQH+ohQBCx6EHg1lutIDUW7fFRCgR5OVpSkcJPdPtTPfLimNZ5D'
        'ijjxG8DzdSfaku/TVf8AT2Fgl5lbZBx1BGDQverWBTGK+8TmEp0EBShg9q0WzqH5VtBBOKzzTkSfcGg8hpCGR1ccWEpFNtrnJhoD'
        'aGkvKBIK1HI+woeQagBkxpnV3SsuJISVJBznHGa6s8x2O8cHYFHn0oki7SlR1tukBop8wKPLihKHY6lKCTkA8EU7jWK4+skHMdYb'
        'bN5gPRXkJTuHC/5T2NI1xhPQpLsd5BC0HBx0PvTRpW4vNEpbGf5selGdUWv4uEma02FONp3HHdPerZGZMzCO1+sQCpGe44ok2qUx'
        'tWpwrRkebPIq6wwy8vO1JNWjEb+UDI9DVV1OcwTmGrNcWUxghUjzfw8HGasx7qy4vBcSDu57Yodbo7a4brJA4BAP9qjiwGknk5zU'
        'gvnE7cc49z8UpLT68AYwpWavqEWSkGREZUMcqA5+uetKEeP4QHhOqz70SivvpxuIP0NEbP8AZGYY/LoJWFRlra74UciryIJAADjb'
        'gHPBoUxI3K8xopGcBHzUvKGRozqTDDfmQcioUIycURadUkHKdw712ER1jcW9pPcUDoG2DOwJE0kBIFQyOCf61bU2EK4PGKpyD1Pr'
        'VfHWCZ+JmWY0VzYkJGe1XRGZkYBI+lV/xNs6bRPbuFvcU5bpROw5z4S+6D/t7UvQLitSgAo5HvWkIfkTSbXZ21McEDNAtQwl2p8S'
        'EALSDkg1BFvkmKgKCzj0JqlqC/OzmSnb1HOKBr6x9SZGYVbkmYVyo8l5ZGCrJwpHt9KP6n1I6/o2KEMNrcbwlSSfOkDvnrg1nNnm'
        'fDuJ8ZBKc98jPtTVIQl2IHUtbUrRyARkA1jXUNxSXrGVbz+ot1K7Eu/hnrJmOVR586Y9NkrKPDUrKEJHTB7U73DW93hlqHbUMobx'
        'lxKlg7weNpB4PFfne9MrjylgLCsHgoVn+1GtFXqUpTsVR3uhpQQ4s/KPvThZZ8f0gbYam6SUxZ9uamPtJYCBuQxnIUseh9BS8l60'
        'kPiay/cXVup3tBe1A5zjjnPtQzTGp5dohKbcbTIJOxAVztKhwR6dq8i2iTEiMzZcpmOXj4mzflzr1wKp3NYMNWN+5I9GN8bwEXVU'
        'eLBehMKbwlpagQAfUVeg2l5hOVrQpJPKknIFLFnnQluLaeQvy8+LuJUSelPFmukBtpIQlTycYUSnrVxakuANhjM5nkjTl0l7FsuN'
        'toxtALmN+eev+1VnLFNhqUiQypCx2oq/LXIdAaUvwhghPTmiy7i85bkxZCQsoPlUeo9quUuueiDQjAYpw31w3xkEJPCsVoenLkHU'
        '4UrAI8oPXHvSTcWkKUc4FTWWYph0NhYwRjKjVxWky/qiyGBOMyIn/tXjnA/gUe309KGq8QJCutOtsWxcrcuO75gpOBtHI96X5kNT'
        'Lq47gAUjj6+9Ler7ZBkETvTuVlYPOeKhSVhwj0OKs2NktOkZ56jFdyI625C/IQncSOKRcWAGILZxJopJHJxV9nnFUY6eQByaJMxn'
        'vDDu3y1Xr7E5gDMtxkArAPSibaEJIKcpz1oZHUUmr6VZSk03MmXjuCcg5qWKSVVwx5m6sRBhVCc5kiSyT+nk0IfVk4oncFcYoS4T'
        'u9c0tzucZ+WtXIhFqXbi6XG1J3oyeNw5H+9ZG+PBkktHFOFwMr4x5p1t0PJykoI5CvSgydNXaO8w9cobrUVRypw9CPT6mrdbMQSR'
        'OQ6hOwwQ/Ebm3N4xoJOEnGVuf+o/3pzs7tsaAbtVmjpP/wC10eIv6kngfalRbrC7gC+rAQgJZZx5UJHtTFb5zMKOpTBHIoPlTJJg'
        '53DtzfeatzouTrUiOpBCmSyFcH+31pIuL7DUFcaLuS11QFckD3qpfr3Mwsl1Y8Tjg8BPpQ223aPJIYkOpbcTwkqwAoemfWuWw2Ds'
        'sPyNQDM3odcASFJV2zUdkZnOy1fl2S6kHcgkDKavTWFlbigFJTnOPb1ozobT0sJemOsOqL6drTSEkqI/mwKlsKP3JC5heJ4cV1gl'
        'CnXktglkHgL9T9KKx3bjdJYdVFdSP4iRhKQBwKtwdN3Z1QUxDDL+MeHKwjf9MnINeKdMN39WDJgyEfOAvKFY9M0r+N2/vqDYAdCM'
        'dj0+qU8lyW8EISkJyEdcdKPOsx4TqGI2VnGSpX+KVI98kuNpS04W0JHY4/eilmntKdLi3PEcIxk9qlaKicKP+5wELw3n1vlSlnA7'
        'DoKsS7k42nAzivYjaHPkOM19NtzhyEpKquYwMCMEGqnuOE5PWpo/jOKBAUa6YtLviAr4FG4sRDaRnP3rgp9zoa0ZcDCWQvypVxx6'
        '0fv8BFxhplxkBMhpPmG75x7fSldDSSlKmzjHWmmxSkoa2uuDKP5v80zOsGdF6HlMhBwCelN8VEaXFDLyAVJGM0FvMZtuQiRGIU04'
        'cnA+VXpUrclTDyVD5SkZpQOGwZBnk+G3DV5eQTwa7hyXE4Gcp9KvS43x1tMhKTuB8uO9DITC1kgkIAPOaRcCrDrAIhBYSoBSB9qn'
        'aSvZkpIH0q5EhlLQLewjv61LKK246QOMnmoZCASZGJ1BSdhyQKuMpIOaHxVEEURZztBpKtmSJWnqJXiqgwnk9aszCMkjr6VVSNwy'
        'o89qBgczp//Z'
    ),
    'dark_eyed_junco_08.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHQAAAgEFAQEAAAAAAAAAAAAAAwQCAAEFBgcICf/EADwQAAIBAwMCBAUCBAYA'
        'BgMAAAECAwAEEQUGIRIxBxMiQTJRYXGBFJEVI0KhCBYzUrHBFyQ0U9HhYmTw/8QAGgEAAwEBAQEAAAAAAAAAAAAAAQIDAAQFBv/E'
        'ACsRAAIBAwMDAwQCAwAAAAAAAAABAgMRMRIhURNBUgQiQjJhobGBkRQjgv/aAAwDAQACEQMRAD8A8wvHx2oZGKyEsYxwKTlWuaEr'
        'nRKNioWpsH00pCtOovoFLNbhgXj7VeUZWqQYqbqemp9yiWwqvcVOQemqC80RxxTPIosBzU3HHar9mq8gyKLyAAftWQsvhFIsKes/'
        'hFCeBDJoMpQ3FGiUmLtVugnmuXDJvIv01EqO1MMlCcYpkEAy85oZHOKM4oZFOgoiBVEVILVyKIQecVNW4qDiqH5rIxMnmpUINUuq'
        'mFLseKgWq7dvahmijFMaBIe4qbmhMaZGYy3wUjOPpT4+GlJ15NGOS8t0Chp2P4KUhXmn4k9FCoaANe9Gx6KgB6qMoytJIZCuOak4'
        '9NS6eTUyPRWbNYTGS1EI9PNUF9VEI4pmKkLMKds1yo4pWQc07Z/CMVpvYRmWjAEP4qYQFe3NRj5hH2ogPGK5GifcXkXFLyDinZRS'
        'snNPEwsRUWAovTxnFDYZpwohV+9VirgVhgcooYok/wAVDHeisAZcirDiiDtUWooBEnioNUj2qDU6AQcUMiptUCaYA2OFpWb4qMzc'
        'Us5yaZIq2SgHNZCMeikbYc1kYx6alUyUp4AH46PGOKGy+ujRjikk9h0gDL6qky+iiKuWqUi4Xihc1hED1URh6asB66I3w1QmKSU3'
        'Zn0UrN3piz+GmkthGZiD/RFFU8UK25gFT4ArleSZZzkUs/0oztgd6CDk5pkYrp4obLRhUWAxRGQv0VILRMVRFa4RSRcuTUQhpjHN'
        'UErNmBBeKhIKZ6aFMvtWjLcDQs1QapsKgTVkICc0M0VqHTIxNjmoiMk0eCLqxTYtwAKZySLKNxKBcGn4z6frQjH0vRUHFQm7lYqx'
        'Eg9dGQEDgUNfio4HpoSGQJODzU5Pg5qKiiOP5ZzSvJhDjqqTHirkc1FhwaqiQrLnNNWfwjNLutNWwwtO8E2Za1IEAzVnbmr2wxBU'
        'XwK5Hkmwcjd+aGp4xVSc1DqNUSCgnXjNUXzQHb8VeNuKNghu9XzxUQc1VIxisfSrgVIdqr3pGwlEYFBcUVjmoEZrJ2MJTrg0Bqcu'
        'V4zilCK6Iu6EaBtUKIwxQz3qiFMnaxgKDimz2qESgKKIRUpM64qws6+o1WMA0Rx6qsw9JodxgKfEaZXlDQIxTKr/ACzRkKgcYqc3'
        '+mavCtWuBhaXuESHxc1TCrqPVUnHNU7iCj9zR7f4RUJV5okK8CneCbMnC2IPahu2TUVYiKh9VcttybKY0InmpOaGapEBFzV4jzir'
        'NUYzhxT9gjSmrluaip4qvepMYKDxVveqWqapsYtnJNSAz3qkGTip9NAAGcDp4FIsO9PSmk371aArBOKCwGeKM/ahe9WQpmo/hFTN'
        'Ahb0ijnlaSSsdaewI96jJwtTocx4oJbhb2IwjJFODhKBbpk0xIMLijLIFgtGDQ7vgUWLtzQrs+1a25uwoi5Oakw5qUX2q8nvTiCs'
        'nejQjIFAkPNGgPFGWBGNNxH9aB70Vj/LNBJwK57bk2WY1AkVcmoE0yMWY1Atg1TnnvQmaqpAHUcEVNDk0lFJmmY2471OURkxkdqu'
        '1DV6rqzUGhxiMDFXaoRkdIqzGgAHN9DST98U5IaUlGDVqYrAvQ6I1CPBqyFHoXINNeZwBSbKVPbtVF+fxTuNyydhsyUF2yaGX4qU'
        'C9TUqjYZu49ajgVOc84qUKEKKhJzJU8sfCJQ0G874pmIUvdfHissgeAMORV3Hfmrx1dh3p0KxKQd8VK3HFSlFM2NuD0vKxjjP9WM'
        '/wBqrGDqPSiU5KKuxuy025u8BQEU9mYgZ/esnBodqV6Tl5PYlsgfgUpDPPJ0orDC+nHuayFo0PV0yO0bDsyNXq0PRUYLdXf3PPnV'
        'k3sJaja6ZYIouCDIxwIxHyf71htQEUOXNrIkWMhlbJH3Ht+1ZLW5WlmWZZOqSI5A4yfrWu+Vfz6jPdXlzJcTsoTEinqKgYGRjAAA'
        'qfqKdNuyiVpt2u2NPaPLYtfWredbrjrYDlMnHI+/vSDc1tvhpoF9fa+kNvPbpa4/80sjY82M90C4OSe30Nbl4i+BW6NFibWduW0u'
        'uaLIvmr5CE3Nup/pkj78fMZ+wrhlQkt1gvq5ORRDmmoxxigRqVZlYEMDggjBBphO1c0wpk1yDVzUR8VTqLQ6YWNsrVE0KM8/Spk0'
        'LGuRY0CUZWjGhS/CaMQMWahNiiNQz3q6EM1dQ45ArGyqQ9bFKgKdqxU8OXP3oxmdMoiSAmnrRMAGqtoOcU4kfT2pZzDCBMnpSlwc'
        'yUSc4Wgwj1UseRpDkYwlJzZ6806eI84pU8tRigSIIvpqpBxRsYWhSHvToRi4R3fCIzt7ADNPXUl280MUVogmZVUrCS3V8uPmazm3'
        'm0XTdtXWqaik11cSSCOOKFunpA5wTj3x2+VIDdypaJFFothDeAk/q4xh1H09s+2TXoekUUm2zjrt3tYRuWdfSGeB1Hr6T6vz9axS'
        '6hcQXJKSSyL2JPqNSurv9TOVVenPLHvk0jcQyISwKMfmgP5rpqVL/SShHkveXUzuWYD7mqt9Ru5mWJ5JTEp4QMeB780GPzXkCzKx'
        'U9jis7pNksFykjRFlHGCMjk1yTlJJtF4pXsdF8JtS27p1qbnceiRFJ5cRXMxIyMZ9m9P3FdY1Xduh6npENvt7xTj2beW59JEqzQT'
        'D2D85AH5ri++dp6q+n22o6c8d3ZQWyKLeHh4AEHVx/UM5ORzz2rnDcV59L1Cq2lZbf2dVSclHRI2rxN1LUdS3S8uranoer3ir0ya'
        'jpSqEuvkzlVUFvbJAPzzWujtSynmjq2Rinnu7nMia1PNQWp1Fjot71MN86iRUTShJmgTHipM2BxQZXNZIwJvtUDVM1WTlu1XQhtn'
        'X6aTcZYmi9XpoQ5NSasdpOFQBmiMeKsnC1CVgBiha4QNwcnFTt0yaCxy9PWye9O9kLllTnpSlk5NHvD7UuhPV+KaOBZZCScKBUtL'
        'sptT1e0023/1rudII+M+pmAH/NBkb1AGszsBLt966TJZRmSaG7jlwBk4VgSfwOaPYVm5f4hdM0/aMOj7Z0ryjaWaN1OP9S5l7M7D'
        '69/t9q4w83rJUdJPtXevHTSf4jFFq5fy4o29fSoBdvY5PNcCuwFkdm6gxPzrvUtlbByNb7k1LsDhWKnv9TTdrbJIGVwzEHCnOOax'
        '9tJ6gAuSeM07Ld+hQQF6fSB9fnTRavditPsZmG1trLq8siXrwz9Q7GspbagkboXVEhlwCqqMEfStKe+lhLqfcck1LQriSfUIIJEa'
        'ZC4AXPzParz9TGMWkhIUm5ZO2aTqJtZAyysOrAaNgARg4pDfmz9N1PTX1nSYZIbzrKSiMAxO/f1D+k4+X7VhtQFxZbjkj7t0IzKT'
        'wPSMj685roO2gujWFpa3ht7/APUSNczyQMWniB4Ckdmx345FfLVouk+tF5PXVpLpyWDz1KskMrRSqyOpwykcg1NG5ruvjjsfTY9j'
        'ncySW8epROkoEYx+otpCoDY7HBYdu2eQM1wWM12enrqvT1I4qkOnKw4jUQGlVajBuKZoCYTNQdqiWqLNS2CWY0JzUzUHpkjAmz7V'
        'UXx4qjVRHD04DOeZlRzVI3NJCTgUxG/bmjKJ0KQ2G4oE8lXLgUtI2W70sYhcicJJcVlIDhaRs057U83oXvQmGACdsvVIh70HqzIa'
        'bQ+mmwK9xOb46NprTJfwtBO8EgbKyIcFfqKFcMOqhCQqGxwSMU1hD0FbCy3pss2lpMZZYG8ifr5y4APf657/AErmGreGAiV5bq7e'
        'J3ZvLiK5PHtmp+Cu54tD3G9lesBZagBGzN2ST+lj9OSPzXYtahtp3WKVAXByHJ+EfIV1ens1Z9jlq3TPMsGnQwxzyFWbymwo9wfr'
        '+1T1SwlltGkitDkcsQP716Bi2ftq5uXuRDjqwWQHgmoa7ocTQsltbRIrLjHbmrOy2FTbPL4imnwojYkHv/1W/eG+3PLuV1K4BUKM'
        'hSPeuhaPsmzsoxHqFrDM8jF1de39qyqaYvnLDHaMiJxwO1cVdvTsdfp9Oq7Ne3psfUN0R22oaRKsN3bKUYnKh1HI5Hy/7pzww21u'
        'OK6FlrkatbyD+TKH6iG9uRyPvXQdIQ20IiTIPGR1YC8VvWy9OjEii6BUFh09J7fmvDrVanTcWtj03Cm5ar7mH3foltuLw413Q0to'
        '76aCJVtfV61cAAuTj09jnGe/avIGqbU3Bpk0iXGnSFU5LxkOpHzBHevoTqcdrpUMV5CI42aTBxz1DBz/ANVqm9doaXqG4bDXvKWK'
        '1FqFuIkUqGYfDyO3H/Arr9BKEadpux51aDlK6PBQyp6WBB9wRzRA1etd0bA2bqqObS1hd8k9DP18++Dw3/Nc23B4T6FG58uS+08n'
        '3U+an7HBx+a79F8Mhg4mWq2c1vV34Z6iWcaVqdhqBU4EfX5bn8Nx/etY1rb2taM/TqemXVqM8M6ek/ZhxSuLDcxhNRaqNQJpLGIt'
        'UVOHqieaGTzToA7KSpBoscuMc1O/s7qIEvazoB3zGR37Ut+nuwWU204ZRlgYzkD6/Kq6blL2GjLn3qy5ZhUILO9mm8mO0uXkxnoW'
        'Ji2PnjFbVtnYO8tdt7q40rbWpXMVmMzssBHR9OcZPHYUrVgppmJtBgA1O5kwpFbrtLwo3/uO3u5tN27dBbQDr/UDySxP9Kh8dR+1'
        'M7i8EvEvS9JtNRm27NMLpwgt7Y+bNGT261XtSKEpPA7nFK1zm0bZfNMl8JXQNI8CvFG80wahFtiZEMhTyppUjl44J6WIOPrSEHhT'
        '4h3WuHQ4tp6kL0KXIkj6ECg46us+nGfrTOEs2FU48miO2WxUHyK6ref4fPFS106O9OgRzM5PVbw3KNKg+ZGcc/QmtY17w137ounL'
        'qGqbU1S3tWUv5hhJCgf7sfD+cU6pyeyQuqPJpgJHPaulbF3w36NNK1ic9ScW9w/cj/Yx/wCDXOhb3DNhYJST7BDWQs9A1y7s2ubX'
        'RtQngVuhpI7Zyob5E4xmjCM7+1Cys1udkTVfIYMj4Unmp3G4kChWcEH3B4FcrSbUNMtxZu9zczRkCVVQslvnspYDk9qy38G3tHLB'
        'B/ljV+u6lEUSm2bJYkAA5HGc+9dFpv4sm4OOTc5tdjjkQebx35phNxWr3SW/UBKx+ftWlansPxQg1STTf8l6xJPCnW/lQFlA+YYc'
        'H8GkI9oeKX6ouuy9dR4sO7NZvwCe54pJ0ajX0saDV8nY5Lx7OS0OWcTdnHYAfP8A/vat22zuzRGk9WqWy+Qhe5kMi9MSrjljnAzX'
        'PvCrw13ZvCWabc9lq1tY2qYWJo/KE7Z5jBJBH1rP+KngHq1xoMdts6wht11K7ie7t1m6FtelSOo8eocnIB74715cvTVKj02sdbqR'
        'j3MT4p+PWh2sjpozxazdIOiCGPP6eEf7nf8ArP0X9xXCN2+Jm+d2IYdY3BctbE/+mgIii+3SuM/nNdF3d/hd3zo2kNqGlXVnrpWQ'
        'Kbe2VklC+7erAwPvWC1D/D/4mWcFrPb6LHqaTxlybOdW8sjujdWPV8gM59q66foo00rK5yyqylk0HTtb1u0INtq17HjsBO2P71uO'
        'h+KG6rEeXc3K30X+2Uc/v/8AVVaeDviZLYSXqbL1byo3MbAxgOSDjhSeojPuBim9weDniLoU9pBd7Yu55LpAyC0Hn9J/2t0Z6W57'
        'Gqf47fxF1W7me03fegaswfU0hsZx2aSI4+4ZRWS/zLt5XEC7mtDEw5Us7J9iGXFaXaeEniRdQSTRbO1YJFMsL9cXQQxIHY4JHIyQ'
        'MD3pjXPBbxK0nVY9Ml2reXU0kfmK9oPOjxnBy68AgnkGh0KnY2pD+4NI8M71GuLrcFjZzOhcPYBm5z2KAYz9sVyfXrWzs70x6fqK'
        'ahb4ysyxsh7nghuxrpG6fBLxI2/Ym9vNvST2/WsfXaSrOct29K849icd61LVNhbusNVtdM1DQL+ynunRYzPCyp6iACWxgDnk+1Be'
        'nqvtc2pGok0NzXUdweBHiRp+sjTrHRv46CgY3GmN5sKk8dLMcdLD5H257U/P/hp8XUvzaDQbaTEPm+at7H5Z/wDwyT8X0p40J5sa'
        '6PVF34v7Hlyvm6X0kg9M0RPVjse3tRLPxV2RMHJuNEMkg/mkADqP1yOew/avGN1HuLCM0F3GuOhW8s4J/Iq0Q163iVXhcxkEgumQ'
        '347126ksw/ZDQ/I9sQ+Jm1v1JuIJtJd8FepFCufpn5U4fErTTEDHcWSZOWy45/vXhqX9erK6oEZxn0rgfY1NJtQjEaeXkEYZuvj/'
        'AJo9WCzD9m6cu0j3DB4h6fIQj3FpIoOQvWO/tzmnxv21UDLWvI4Hm4zXhWC71J8ymJWVh6VWQkkj65ot1JqLKkjSmH0n0mcgfTua'
        'HVpd4G6c/I9xLv62Yg+ZZIB3BmzRo97I5JU27gjC4b+9eGLeW6jXqa7VEI4IuCf3qbXd2hDw6ndsW5UpMSP7Uyq0fAHTn5HuGXc1'
        'zcspiuooV61JVMZIHcZPzqd1uqSQqgSDy84dD6g4+RzXjOB9YlszPBqlzEqr/XdkA4+nekota1FrSKR9dvPNOcqJ2yKPWo+AOnPy'
        'Pay6vp7zCb+F6cGAIB8pcjP1o8O4reKBreKCxiiPHQoAXn6DivFJn3ESjjcV4GYZEf6g5I+2ar+Ja4qRdeu3CZbJxcNkD5Gi69N/'
        'H8mVKXke1k1LTwJWhstNjeYjzSsS5kI7dXz/ADTsGtTykKVhYjkGvEdnrmtQiXO751XPAMxJP71mdP3FuJOmW33ddFSOWY8Ckdam'
        '/iMqcvI9l/xiVI+UUkdyxxQ/4pL1iXzT9UyOmvINzuzdojUJuq4ui3YJ3H70zbbo3MIy8u7pVU/1FAePlS66fiNplyetH1nN0iic'
        'LxnpC8GnEv5GBcPGR/tIxXko7t1+0KMu8ImLLkeZHjFNQb+3UHES7rsWbPYgVnKnwbRLk9R3+sGyjQyyKc59QFY6beVlEnLRjBx2'
        '9688WXidulnEUmqaHMhOCJgQayF7u/cMgBNjtqQEZ/1sEj7UVKlwDTPk7Nf+IFpbzxh7uzhyfhZx6hV5t+wquP1NiueQ4cc/3rzt'
        'q2sXF/MstxtrTGyQC0NzgCkrrXrJetW2/C3lHlUusj708alHxFcJ8nokeJluzFTcaePV04Mn/wB0ebxJs7YhJ5bfJ915/wC68xDV'
        'rB5laLRYU6hnP6j/AJqrq6jkUl7CNlPsLkUznR8UDRPk9I/+LWihyrahbqR/TisZdeNGjxTMHuEdV7qsOa882t3p/S3TpDiQHBX9'
        'QP3pqG/0xHfzdMbrZefWMik6lPhB6cuWd1Xxy0bHUkgCt/8Arn/5rFT/AOIfSIDIrLcTDPBWDBH964/JfaAsSLDp0+RySGXj8UO7'
        'vttOEdtNuBxj09PNbXT4X9B0P7mnz7s11I0SDUJHwACJgpyacsd7a4kSiS4gdjjA/TIwH0Jp6TVNnXDKF0GW2SJulyjdbuc/MntR'
        'LG72EjmV7LUYwnwjuByf3qVrYl+x/wCAQ3xfheiXStOlVFzk2wAP2x9ag+t2kkMTzbZseojqKhyDg+x44py8uNn3qoI2u0MYAVTF'
        'jgfM/erQ2m2ZgfJ12GMnHUsidJDZ+fbFNql5AsuAx1LQWtmkm2xCPMUNmG4x0/ioifZoPnXGi3XlhCAqv1LnPfvnNOtpdnaLDBaa'
        'nZzrN1ENJgDOaYfa11OFS21HTVJB6mEqn24GO/8AanWt9/0D2mLR/D2eRpXtL22V1wyxpkfgHNC/T+HWR0ajfRsMtgoRzzgfb6Ua'
        '72hqcV2EvLwXPlr1xSQdHQPvSkW09SlX9VbiB2HpjUcM/wAzig+px+je3kINP8P4xEqa/cJh8urLjOT2H1q91pG231DzNN17pthx'
        'lk9Sjn96Be7e1Z75Q+mtkOA7qmRk4xSt5pOqRLLI9qYrfqxkx4wffHGaD12+n8BVuTMjTtHF5E38dhZljwrY5+fP5o38D0GeLzJt'
        'aBKAqAhAC5+/1rCWWkCO2BW2lky3V6UPUv34qn0G/iAuJYetXAOGY8j2wKVavEO3JtKba2xcxeVcaupIUepAufyaYs9gbdWBwu5T'
        'GX7AlSDWrPCk8yRKgRo8cdRHVgUVNNu3DlYY2wTyzGjd+Jrfc2e62bBIUWLcVm0OQoPmKGH70V9k3EUkf6a+srqInkdYOT8zWoeW'
        'YHMc1nnp7sc9APHY0NovOaVI7S6ToJGVbANJ/wAjfybtdbE1GRUWSGKdsghklXpA+1Sh2RqMLKz6dbeaxPUY2GSB29+K0+zvZ7K0'
        '9UV4x+FVBJxn6im2n1S3kERgvyzLk9TNlf70drfSb+TON4T67dWn6qGOKJ2kJQNJ6gPqKTufDHc5kLyqkkinp4P9xWIhu9z+uKO9'
        '1GJSfSodu33puF9Ymla2l1e+a6wPLAlbDH5cGldn8Qj1v4ZbhiLeah6+jqAWTAP1rGx7A3JE/mC1eU9RynXgGlp5t1wXPlyy6n1q'
        'cSKZ3x+KJNqeuI56LjUArLjiRsZrWS+JgOpbG1yOU3Q0JliP9Ky5IP7UT/Im4prZbmHT5u2AplHH2qy3WuWtuzzXmrlWOSQ7HArH'
        '3Oua0ImitLvVSQcktI3Aplp8RXfkZj2Dujyuk6fI8uf/AHMHFSg2LuVi0f8AD5+ocdXXSljuLVoV/nalqAYj+qVuKtPujXoVLxax'
        'ecn0+s1rw8Te7kztvsXXoleGXTZg3T8ec1jbvZGr4CrbXDY4xmr2m7d0mFVXWb3LDk5qdruncMbM0mv3QA4PWAT/AMVtVPg1pGsl'
        'PLtnsprfqk5brjmwyA8/LOTV0wqwLJaztyC4VQ6ke3vxTsOu6F+paZtEhYf1BkOOAMYArK6funbli7zHRYnZ0CHp6sZ/J703s5/A'
        'vu4MLNDYQ2UizQTKQPMjk8rp9+zfKk7SITwSrbw+YS56WYAkD7Vtq63tK7Je402WGNVIeIuW5znv+P70hFfbKDNJbC9ikZR0MsbF'
        'Qw4LEZ5rOMOfwZOXAvJaad5KJLJNJI2D0iPAJxyfl2+VLMLa1vBBC0zR+V1I0cnSy4PvWwW9pojpDLDf3S4TpIkiJQ5PLDIyDTWo'
        'aPtseZNb63Es0jetpYCAM4HA+1HpxeGjanwaXd32otcGO2v3lVgcoJCSv3q2nanqsErus7yycAjpBx7VsNroLXkskQ1bTnTLBehW'
        'XgD05xzzmp6NsicIpe906BgcgGU9TfIE/nNL03z+Q6kI6fretuyLLeLFGULANxx7URtw6xBCywTiSbOWz1NgH/rtTNztfUIDm3ii'
        'lySnWJR0hR8vc/Ok7zbGtpcrNHGrKEU4jGc/Sj05oGqJmbDdu40gRHgiUMML1MeW70xJvPXJCsDRWkjA8A5JX27fKsOdC1jzFAtJ'
        'CkeO7HOR27cUzebe1eKQST2zN5salyIyVHyAIplCp9zXiZSPdeqJNGgsbBnCkl+nH45q43tq17duDpmnOqsR1dJHSRWGuYb7TbuO'
        'RInhkjPpymAcj607oFsn87zL10eU5dTF1YJ57imUan3B7TNJuO7kmI/QWDKrZYEEr9sj61GfeUsQaU7bti7N8Kt3PzAxWKuSunwP'
        'ELroMp8xh5JGT7Ck7vWrSO1WS6R5GkGSoyWGDj8UklNDRszaovESFrVom2+F7EAdOARU7jxOlmvPKGhpJcKg6eV+GtRhvLdbUvbT'
        'XLx5LAPFnB+R4odldJIha2hXzI/XMz9x9uKRylyNZG3TeIFysrsuiOrIP5ilAeKjF4iXFpMt2+37QSA4VmTJUVr1pctJdDyX/UrK'
        'OW6QMfTmsfreoRxP0lB5oPKgDv8ALikbkHY3WTxjuTcOF0OIluSzRcY/+Kxw8T5kAuE0W3eNXPUgXnJrXbm7bU7eK6j07pkYiNem'
        'PHb5ijX9xMt0jpZRwsF6XEcROCPfFBOQdjab7xK1OfTUuodIgjRm6CjReoisTBvm+tLiSa50dJVk5CC2A4+WaQtNW1No2SW3E8Q9'
        'miwqj5jHvSF5eaxeosVrFJFOjYVej2pk53FsjZTvzQXt41utsoXZsn0AUHUNx7fubJC+h+Q0T5yq8H6Vr2n2mpXMyHUrYyLkq3TG'
        'QVpy80m7kidIYS0SnqAORnFV/wBotojsOv7ZV1EmnvEmfXhDRLfX9oMZcWrNg/EQRx+1YS10fWr2MzWti2W46SuQaj/lncU86Wkd'
        'h5CjknHeg41ODLSf/9k='
    ),
    'dark_eyed_junco_09.jpg': (
        '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0d'
        'Hx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4e'
        'Hh4eHh4eHh7/wAARCADgAOADASIAAhEBAxEB/8QAHAAAAgMBAQEBAAAAAAAAAAAAAwQCBQYHAQAI/8QAOxAAAgEDAwMDAgMHAwME'
        'AwAAAQIDAAQRBRIhBjFBEyJRYXEygZEUI0KhscHRBxVSM+HwJENygmKD8f/EABoBAAIDAQEAAAAAAAAAAAAAAAECAAMEBQb/xAAs'
        'EQACAgICAgEEAAUFAAAAAAAAAQIRAyESMQRBEwUiMlEUQmFx0YGRobHx/9oADAMBAAIRAxEAPwBe4jL0u9qx5xVhGVAwakxQArxX'
        'immjIo3srRZbzzQtT0RdU0x7Jh71y0DfDfH51bxYyMUwpVe360sMk8c1JDxVGO6SuGlD6begpd2/BB7staJbZd3aqnrDT3Ei65pw'
        'C3VuQZAP4h8/2qx0PVItTsUuIwASMOvlW8irPLxWvlh0/wDsLirsa/Z8DIFRkt1KZAppG9p+BUDIBwawoiiKRxmFty9xTkOoMqbc'
        '+KDKQw70lI4U+KvjkaVAaoZnlDMWJzSEjkk4NSMoIIzUFZSRRSvYjdgjC8jZwaMlsyCnLX0zgUScqq8VKYFH2IEke0mvCMncPFCm'
        'kG8gURXymPNFKwxGIZF80zDIob5qrZyKktxtHJquVhTfZZzOpOOKkkasMVUG53MB4qxtZxs5pXFoZSCiEBjxX0igD70KW6UHPxSs'
        '14M98ZrSlobkkTnTIya8ihBxk4qIlVo857V6lxgUOO7Fuz29wI8f0qguELy/TNWl5cg5IxSAG5s/WruIHsCEKMMU7G5281CVCeR2'
        'o9rEzJk5oKAKGri5KcZxQlumc4JNQaJpXAxTlrYEsMjmpKaF4t9B7dyQO9FlfAxRVtigGPihyQk9h2pNMba0LmYg4IBHYg9iKzdw'
        'jdOayLyLc2mXR9wH8B/yP6Vo5IyFbjmhPbx3dlLZXCj05BwSPwnwa0YJxjcJdMaL9MtrZkkiDqwZWGQQeDQpUyck1melb+awvX0C'
        '/OHjJ9Bj5Hx/itS3Iyax5vHeKdMcWbODSFxkMSatJsD86QmQk9u9SKEmIO+MjsaX9Vt3BNOSQkk4oL2zAZq6MVRUGtZ2xwaK0rSd'
        'zSIYocUQSdsUrTRA4jLuFHmm1gVeO5pWGZUbJOacWdNm49z2pYL9jRoTkUb257cUvKMUSaQbiRkc0s7knAplGyOR6Dz9aZSfanfF'
        'ARcDkUC4ZgeO1HgSxxptzE5paaTGfpQYmOcVC4kOTgcnirFEFjUcp45OKhJOwYgE4oNuWYfFEMRPbzRIDLb3xmmFGM7aEsBR8mmF'
        '5TAoWGJ4jksFNWtomRx2HzVKAwcGrezLemp7/FMMmWEduqv2FPWqIDzSu/LfSjRsQeKxpNssTSG3CnmhNFkZr4EnzRVU5odOiVYl'
        'LblhjFB/ZW+OKt129jiihY8dqWUmTiYvq3QpLy0F9agreW3uUr3YD+4p3pXU01fTgXAW5i9syfB+fsa0zKqLn9RXPuooLjpzXF1u'
        'wTNrIds8Y7YPcf3FbsM/4nH8cvyXX+A+jaC0EjcdqlNpuRwK+0O/gu7dLmNw0bjcpq7jmidNowTXNySlB0CkzKz2BU9qDLaewkit'
        'LNGjE0jNGuCKfHnYjSMlc2x3+anBbEp9atrmEEnAqCptXGK1KaYtbKmW3w/0qYX2gU1cYJ7Um7MrceaHLdA0gpi3eM0q8G1yT2+K'
        'tLYe0ZFBePfKQRTXQXGxRE7/AFoUtuzHjsKuobIFc4/lTC6cPT3EdvmineycdGfhtuDmgy249Q4+K0H7Icnap4pS4t8HG3nyKX5G'
        'mCqK6CDAzimoos+O9NRQEr2qUUeDgiq3m2OkK3FtiPOOaTRCshH9avmjUxnNIPCDIccUPl/RHEVjhLPjHBq1sol9MAjtQ4osECml'
        '9naisw0IkYW3eaYQjGarVn9PiiC6BOM/emjoS/2WsWXNMk7Uqttpsjg0dpsnANLKN7HiwnqYOQc16Jn+eKHEVPJNeyMFHGKokh60'
        'FaYsQD2pW+givIXt5lDRyDDCvFkPPmvllIb702NuLtC2ZDRppum9ZbR7qTNrM2YHPYE/2NbKO5kjfjJpDqLRY9b0xkGBcR+6Jh/S'
        'kOjdU/bYJNOvDtv7X2sD3dR5/wA10c2KOePyrv2Ro1cNwSO/egTN7jUI9wByK9fGMmuf8SRKsgse4c9qFNHjPxRwRt74pe4ck4Xm'
        'lSZONCbRgk8UuYcvnHmrMRMV3EUF1O/gcVclWxXEJaw/uxxzigMhFxnHGasbRNyivrm2xlhUlLWhhiy9MRDcBmmCY2GGPHx81UJK'
        '0YxzR45HY5zxTRehux+QRkbEAzjv8VV3aorfX5os9y0abQKRJd2LtnNV5GkhJIbSNfT44zSdwCjYowlZVoMp381lVsLZAyHbjNLb'
        'wHzkd6hdS7c9qUhlBfJbzxTJCWXUPvIAFFkT2EjxStjLg89vmmZ54lQnNFMujtFTeBgCQKWjkYt3q8uLUsh4qpktisnbzW+MSiUW'
        'OWxcAYOc00XI80WwtwIRlc/NTu7chdwGM0kkPGOrArcKOM4rySYnGO1Lek2786bgt5HjJRSQPOOKr4OTpDJt6PEbFekjdmlLt5YN'
        'wKbcd8/4qok1qeObaywbfALe41fDwcz2kRxZrbRipyTjisl1pYy2d6mv6YpSaNsygdj/AP3zU06us1bZcRtH7tu5Tu/lV5ZXVlqc'
        'GYZo542HuXPj6itMY5fHdyWg/wBAuh6nBq2nRXVuQNww6+VbyKYuFwcCsPI03R3UHqqGbTbo+4eF+v3FbCC7S4CyIwZWGQR5FU+V'
        'jUPuj0+gexpY/bXkcQ3c0UMNnFBLY5+KyJ2MNGL2c9qUaBm7DzRP2hm4qw06JXiOfxVIx5SoD2IxIYT7qYd49nJ70TUwscPJ5Bqi'
        'luGD4zTShxdMVuhmVMtxXiybBtqMch280KQkmq26HiSkYMeamiqELNilQxBx5qUshVeaLhYWfXMgAxSbz8EZodzPuGAfuaXGTgnt'
        'S8EtFa2yN8XKkg0hAWVxu7Zq4kjDw4pV4MMPn6VdHHaBKPsaichaXuJnYHnNMLGfTGBzQHi3SYpOCQ6tI14jBjwKRuIFznFEFzjt'
        'XjPuBJFXWR7PbWXYAp7UdiZRjxSJbDfFX/TMcs13GbVYHeM5YSkbf0p8UfklQI7dC1t09fXEfq+g0URGfUcEA/b5/Ksxr3Vcelyy'
        'abaJL+5zuZ1K72+1dn1CTUp4U9bWLKOQDG3YSB9BXKutNIuP90Wa8udNvA52qY3COPyPf+ddbx8EcburNrwJRu9mL1HqyFrIT3pV'
        'nYEAclh9CPn/ABWF1HWnuJSYwdhPtLDBA/WtfJ0yLq6eGa3eQAnPJBT9PvQpOiFSVprhRDBGRhFbcZB9/Fb04oytMyCTyzcpGQB3'
        'IPemYru8sJlube4kjcc5U+a0V1p8KD0raMIiDtms/cRsrsmSV802n2LRsNM6psuorE6XrCpFcMMJJ2Vj8/Q0Tpa8l0rUzod82ELf'
        '+ncn+X2Nc9ubaSGQn2gYyMGn49XlurRLa8Ys8X/Rm/iX6E+RWTL4UZRcV0/+P7AO2xnjBNClOD34qn6F1tdZ0v0p2H7ZAAJB/wAh'
        '4arS93Kprz8sEsTcZehn0eCYA96ctr70+Q3FZ55Tzihi5ccE1VCLUrE5F/e3jTHGcilEjLNk0lbzmRxzmrq3QbQcZp5psKjbADIr'
        'wIzCjSLljjzUlTC/eq1Hex0KhDu4GahcRkqafWMeO9fS2xPirk1VEooRbln+lGaDCgAdqekRYzzQ1YHgVXNipJCDqytjxXgGTzTd'
        'wqjJOMmlgwbtTKTSA9Bl8ZFCZMSZ+tHhYYweTQ5CBncftQ48iXoZhk3nFNM2AB4roN5oejzBWa0jViP4BtP8qqpukreRC0U8kWR7'
        'c8+cV1JfTskeiJGIllG8UW1unSQMrFSOxBq3vOjtRj90MsEwJOOSpP61WTaNqVtkyWkuB3IXIrNPx8mN9CqLCXN3NcwtHJNKQfO8'
        '5qv0rQllvp3G+Usu7Ej524+M/wDnFMIrAe4EVc9KJv1VIzkblYA/lT+LlnHIk3plqk/bELWMNGfVGXzyB8/X8sVDXLVZtPJjBRWO'
        'CBkZxWpv7QCeWWGENklQAMnGB/as3rKpb7reTdI78sgTcMYrtposox1nYBopROVLngRx8sQfP2HzVbrFjDEiqka+omAoUZzz4+TW'
        '3sNDmhtJGitoo2m5wDgqPFFtenra2RpbgrNMpymOQKEskY7JHG5aObW/TN7ql6YlgeNCAMkfPetCOg9Mgs83EczTjICeoOwrVs5i'
        'GFIDkcY8VU3886MSGYjHZv7VS88pdFiwpdlRa6JpFjKHWO4hkxwwlP8AUU5LfXEURUuLlFbBOeR+dDmuknyXibIGAPFVMl5HHdD0'
        '0Zl/iAJGPrSuCmvu2LKKLuCeC5BCONy90PcflQZ0wxArO69cujRzwM8WOd27kH6Gp6P1PBLJ6F+yqc4EvYH7jxWPN4UorlDZmlGj'
        'UadE2RV5EzJEATzSWnKhRZEYMp7EHNPvjbmubu7ZE6PYF3yc041uMcd6StyFlDZ4q0WVCPrQSsKkJQgrIc+KaLIYz84pW6O1i1JG'
        '8IO0nApKaYXKj69GWOKWjXDimZJPUXIoSAbs4xSSexbPZoS6bqS9Pa2DVsSDF3qvnbLgCncgS0Al3IOKBvLyAE8U1LyMYpeNMSHj'
        'tS83QqOn2ev252pPCYQ+cMpyB9803b6lZTb3iu1CJuUmUY5HfisajQhYgsaOi+1T2JzkZJ7fNIy3kM87xhJiV4BU5AA79/Jr1yys'
        'vo6TFIzhRbSxuTg5zn8yP0r0rJ62GJI54/Dn8659BdOHMsTCBif4HwcA4A+1Mt1BqMO9lnEshxt77Mfn3o/ImCjWmzt5iweBZAw9'
        'u7Br6DSLW1lju44xFIFJPfA+aopup5ItMW8lgiKIgaUBvfn6AZq2sNbgu5YYZI5oZJQGKSkDHkD5oOGOW62SiwSwkuUeRC67+CcY'
        'wKjcdNpBC10Nr3Cj93u7CtJpcICTPg7dwIHxxn+9A1CY7WVskck4rBkm+bR0seOPC2ck6h1ya0LRyAxupxgjvWbuOp/YwYHOOwNa'
        'X/UC2hmaRsujd8+K5TeACRtshP5+Kvg7WzNJU9GpTXY2TcuDxwCardR1b3hssvyQcgis7As53AE4+RRERyhV5FP50eCFtlql2ZcG'
        'MsR+lLX+7PqKzbh7iQMZqNq1vCmwysrnuCOMVYR+lIF9OSF+OxYgmj0TbEiUezLXbRKuOMnz8Vn7yy/bVFxp6Bufco4rV3qoiFJN'
        'NWVe+VkH96o7m5sV2jT3GnTK2SSTk/2qyEmVyiLaHrmqaLMtsztFFn8EoJUV0PR9fgvlVJ/3E/HtLZVvqD5rnd3JfzbY21G3nz5K'
        'qcfrVX62oWUh2SeqnYjORVWbxcedb0ytxO6I68c03byHOa5F011tNausF+rywdgT+JP8iupaLd29/bCe1mWSM+Qex+D8Vxcvi5PH'
        'l93X7E2NXgymQcmqZ4neYAGrq5XA70CKIF88VnkrYrVhLe2PogAdh5oM8YRgMVYYZIe/ikZsk5J5pcySQaoUuZCoIBpWI7n7805J'
        'FvPbOajFalWyVIrMmxXbZFovbnmgMoQ8/NWbBRHgik2gMjggU7dLYzVbJOkjRW8cjhZmHuZeFU4PGPz/AJV5aSRS23pO0TxiTCen'
        '33Z+vzUZIhb28bxyADOCo8tnkD6VH1If2yJkVYvSXJAXhv8Al9K9WXh2mKz/ALQwAyezHAx8/wDagtclJ42kLEgcKDwTzgHz5+lf'
        'SBrnAjX902EHGfIJPPjmpTLHDd4845fdg5+QD9KJBO6a2w7u2GQ/wk4QnwT81c6XqUlvfWqZLhpFI3gk4B+aq2WFyZIwX3A74RwB'
        'zjn6/wCK+nAhtVkEm0QBSgQZzzkfnnHNG67Ifoe3jYadG+FBlUMcfQY/tVHrMDoWbnbjwasehNUg1jpWB2ZPWgT050H8Lf4Pf86T'
        '1l5Fl27htIOKxzjUmdHHJOCOa9VpckSejLKgPyQwFcf6iR4bl5JLtGcd/acn6V2vqkSSRSKm1HXvnPIrk3VUcQhywQyHIA5P54rR'
        'hM2VUUMuo74FVCo29yo5NVlzd5PuyMfxDvS0wm3kRQbv/iD/AHpWeSVPbNAyn61pUChseW9kD5Egbxz3FFN+Y13Sxsq+GU1RG5lH'
        'CqBz96hIZHHuZ8Z/Km+NC8mXo1sJONskjKO28ZFOHVtPugRc28RPjb7TWTMbDsx/SpmGXYWWQMP50XBEtmojGlu4EakMTwCuc/Sq'
        '6+sofe8FwIz32K2BVNidfcB28g1ETXAP8f61FFr2AM5uYyfcSMYyOateluor/Rb717WYjP4435WQfB/zVMZXfl1YH5AxUlLg5zu+'
        '60zipKpCtHeNI6hs9c09bi2Yq+B6kRPuQ/4+tWdmxZhzXANP1C806dZrWZ4nA5x5+9bfQP8AUIqRFqkH/wCyMY/lXB8n6bNScse1'
        '+vYKpnVJXAUe7IFV0twpcgHzVauqwXdmLi1mEkbjKkUqJ2J5PGa5U4t6YrezQ2siluasUVWXgDms1DcbNvNWtrdl1zuquMHYYv0T'
        'ulw2MUaGJRAOwJocYa5lAHPzU7v93CTj6CtEYXtgdmd1Bri11G3kIAGfU2sQ21fA+Ac17tZJfRkBkiGZCDhSRt7Ejt389+au9Q6I'
        'hS4v/wDb7y8tlt541AVyRtbxgk5NK33TWvWcl3nUoJ4YJlgKXEOC4J4yRj+Vela/RdQvaXDiBZo3corhhuxkEDwKhaRwy3DXE6bv'
        'T2mPdwSucZOfJ7cUG/tdUtp7iG90n9yjrCz2r5wT8BgCa8hv7SylgtprSe3njZvUadWVIzxgE+e3A+tCmif3DyM0F1MJWEe8ZiGO'
        '44r6RpCUUNmNGUyIYxtye2P171H1obhS7zRNk7slgML2O368f1pu3xKJJIWYJsHqW5JbI5A5GT2oMho+g9duOndZS6jgllspgyXC'
        'g5QRj+Lt3B5/lXTtQjgvoFubCSOWCRdyMGyCD8VwPUxdQw/s8by4IZxGSQCox+IfGMfFXvSfU1zo4MVrMt3FCAzxu+FU8Agf9qnF'
        'SVMaM3F6NZ1Lp0ssBOxlccfBFcn6n0t443kKZZchlPiuwNrtlr+nbrK7jW5iJDwOwytc418XOZFmXcGP3GKHxuDLfkUkcsnWOMmQ'
        'Iyn6V5Fd25BE0TOfBY1dazat6rskICsMcL/5is7c2khb8Jq5SvsraPbpdNuAN8ccbHuwXH9KRl0y1MBmt7tWGeQwwRRWsm85qP7J'
        'jycU6pdAeyskt9jY3ZP0NeLA54ViPzq1FtGBk8Y+aiYw4/dLu+vxRsUqpbdwfxk/nXsNg038Zz8Zqzj02WTliR+VGkht7FNzyFn+'
        'AaDn6QCtTRZCcmT9KKuhTNjZPg/XIqcmqy7tqIoH/I811DpaytpNBs52hT1ZIgzNtySaz+T5UsEVJqyaMb0Z0xJcXspvrWa6iRQQ'
        'Efg8+a3MWh9GXFo1tc6OtvOv8Yd0f88nBo53Wkm+JihHleKMNVabAuUjuB2965P61kj9ThJ/cqFU60xa26Yt7UbdHjkEI5ZGnOGP'
        'nGe1OS6bZGRUE09k+ORdp7Pydc/zAqduLV23w+vbP42vuX9DTrftf4SYb2PH4Rw36GncvFzP7uw0pCN309qsFst3+yNLbNyJoSJE'
        '/Vc4qFhkcEEU7pt2NMvUnsZ7vSJg2dnIjbx27Vfz67BeMx17RrK8B7XOnsIZQPkj8LH7ikl9NhLeKX+j/wA/+AUaYhorIu9mxnsK'
        '+11kW2YrjOMCpxJol5Lt0PWkaXk/sl6BBMPoMna35H8qrtUWdHNvOjxyDurLgisOfDkwKpqidrRsL/qPRjLqEYvYR60MT7lJyzDu'
        'OPI+tNy3umdRXlrp9lq1vbmfmWdj+BgMj25HcisH0l0Z1D1Us17dr/tVjF7i/ogvJnwo+ODyad1i36b6eTAkuryVf4TdAbiPk9v0'
        'Br0XG/xHTa7NrqbaTpschv7iGd55Q0kqxtHHlex4BI+fOarbLV9J1a9uE0oKRglmI27nIwOCP51mbTrXpeeB4TpV3NfSHZMkT7be'
        'NAPaM/iY588UnbjWbGaX/bemp1tLhcOQzKQf4SST3pIxn/OM3F/iXWo6FZ2/TmoataC0u9UkV1U29uzqh7OpZsYP2GK5DaJ1TFuk'
        'sX1NBECzeiGwg89uwrsunP1Dp9p/ts+t20Gn5luzbjJdRjLKcf07VkLnqm/khK9L9OTmAu0ZuGiLKx88Dj9SaEZN3xYZRSrkjF6d'
        'e6/rOppZvdtNPMfTxcPgEfVjwAK1c/SPUOnaPp8l7FAi3cxjjjt2EkkoOOfgADkfer/o+z0yH9qk1Tp2e8vpY8IspZIQ2RufHAGM'
        '8DOOK10OmPJoUdnLcpHFGGC+ioMo3cFN3f6fSkfOTtdEqC0c9vdMi0KBdRm1GF7mViIVjO+YN23+3j9eK0tpo97eWo/aJIpX7GVO'
        'VP3+velrnprX45XTS47GKzJ9qGVizHA5Zsc/07169t19aRKkN1bFsBAFmI4+cYx8/wA6ZdbGSiYrrC8g0m9a2uYXXHZgOGHyKyd1'
        'rVgTlAfyFaLqPorrrVJ3lu4beXHAYTgAjx3+9ZO46E12C6aG4jWJhE0ocyKVOBnGQcAkZxTxUfYkq9ApNRR2/dwMR80JpJHPtgAH'
        '3qgDTpKQJG4OO9NQ3d0oBEnH2p+KQharBLIfe2BTKJBbpln/AFqikvbsnJfAHxxQJZpHI3OTnvzS8WSy21DVgqlbfk9t2KqJZJHO'
        '+QlifJNRyBnBJ+9eNjgGilQGLyPuII7BsV2vpOTGh2UfgQr/AEriypmYe3CKc/euw9Mtt0ezbA/6K4/Sud9TV40BsvbuMMucVVGE'
        'rJ3q2SUSJj4oDpzuIzXBTYsqbG7FR6Qx380dnG7B7ilbMMOBRbgFVJxzStbG5Uh6KZ/S2bt6f8WG4foardUtrWaNgkTQOezQsV/l'
        '2qNvO3Y5o3D80cebLCX2snK0Yu/0LVjIf3jXMPgrgsv/ANT3/KqjUx1LbIrR3tzLHB+BQzbo/wD6tyB9uK6WwIXdjGKWYQ3OUnjV'
        'x43DtXWx/UclVNWS17L3qDqoW1rJDqOtzIp7Wtu+d3/yxx/OlumrNda0aTWLNWkXeUaNYlQ5BzhnyTgj4x3rzrOyfVtbsdLtGitr'
        'h3/fhRzg88DyAATWk6dMGm6fPpVqkjWUBZ2Yw4PqYAJ3cA5xjAFd2UG4qMXQU929laLjpLTUOp6ba2lrLE26SKcbirjx55q2vP8A'
        'cusuhpL+3sUt1ZXaPMoCEoPcV+RjNJdLaJHHaX+rGE3DTzMsMJXBVR/EQfP1+Ks7jWL7/cdOt7SOBtNhhKzwSylIyxOcrtHjzng0'
        'JQbVIaE+Lsp/9NdHtLg3guI7gy+g0byTyFgqEd1z4JH8qo+ierHg1SPpqy0xtQJmJOyTBAzkkDtxya1nU+tai00FhplpazG6bYuQ'
        'Vyp8jceR9SBxQ+mejhoPU79QpcWpnaBojFEmEy3BIOeMVXDHxm5WWZMnKCjXRswIPTAeCaJgQ+2Vf5A5x+hqtARJJpIo403k75O+'
        'ec81K6uJcxCTMkhwCPP1J+lLk+qwYybFEf8A01PHPyf7VcylB5JAAqqDjjBHGR+felriSOFJHMijnLPnkZ8Z+lJ6tq9lp8LNLIhw'
        'pZRu4H0+9cs616zklZlEgjDxriPGMAnJHH5UqjZLNd1X1Zb21u6wygKFBySCWXzjnvz5rjPU/UlzqRZPUcpjZnsWXJxnH0wKrNT1'
        'Ce7fLueRjg96QcnHNNSQNsAAPW+55ohwu1QRjzXgIUHjknvQ3bIA+nxSPYx9IQxPx4qCgZwfHxXxJwPocnFeMcN7u4ogPWwR2qI5'
        'P+a8dxggd/vX0ZIwTj45oEDNwoGP1rqvTQP+w2PH/siuUsdx48V2DptAdAsB2PoKf5VzPqbrGhJFhBleO1HVc0F+Fx2xTFh+8IB5'
        'rz9p9AQa0iYNkDii3YLKRim8KqgDjApdZFeU5IIFXRI36Eorcq3Pmn7S3Gee1QlYLkt9/tU7a4G0Z4oUuRIVZK8hxGcVSzIUkBHa'
        'r2WX1FCgUjdwriklfIeVei+srm3t7tLmCKNdXjuTJOyIZHVM47+OOMZAq51zWrOV3voHvJrdjhQkQYLjuCM8EUL/AFR6h0nQbOfT'
        'otMRRcvloV9hk+rFeT89+9c26OtYNe1d7dbA2kCqXdrRmBHPAJYkH+vxXtVfbC6XRdX3VkE15HY2kOpGWQkJH6AG8dyDzgjFaT/T'
        '/UbabrGx1YCWaK0BE9uItsbSEEAkn4z2+lOaT03pmlksLCCW4lIKXSDBjUjsOfa3POOatUSKztwkYXkjAwBz2zwM57CpbfYKLzrL'
        'Ux1DGEZP2S1RshYiFbj5bv8ApxWOfQtPVPZPfOh4z6xx8fPPirkbpvaypjs3Oc5+aR1O9jgjklkeKIAcDkAL2AFTroL32Vj9OWSq'
        '7G5ujgjJadhj7+KzfUraPYxyQwXV2zR/iYzk7Qe9KdR9XpCFhilVYEJJwfxk9zyf/DXLNZ1m6vp2eSZiMnHOByc5+p5plfsUY1vV'
        'JDMwhv550J43ngfaqC4mkcsXbcxPOa+lJ53dvFBZsrgA4FK5WFKiLEc4zmgu3luT8VJzlcYJxQjzjNKEkze3FDZ89zxXrsVccZx5'
        'oMjZYkUCEgVB55HmhbjuzyK+OD5r7HGCaBD7OaIq8cZobAg985807ZWs11LGkKM7McbVHNK2krZAUcUk0qhOSTjAHcmu06ZC0Fjb'
        'wY5jjVT+QxVJ0n0tDYCO6uwHuccL3Cf9619rF7u3avP/AFHyllahHpCtWBeBmXtRrKBohuxVn6ShBxQZiq854rnJKKtgaF552/CP'
        'ilFcq2c1YWlsbl2bsB5qV1prR+6pUq5FfFsSZmlxuOAKlESGwe1TdQgpCSVhLgGq022P+KLaOVVB/rUXdXOSaSWYbCPNRhnAbBNX'
        't0wXo6Pr/Qul6jp9pe67c3F9qcsCb5VYxe7HKhMZGPkn8ql0to2n6HpR0yx9RELtI8pwXd/g/YcCrEyMoCEJkPuOBwM9gD3/AF5o'
        'SoVBMj4Uc4B5AzzXsIY1Hdv/AHL5Tb0RuJSIRHCjOx9uQ2O/B58HzQ1KoQSzxqU92B4z/KoXklpEwLO5/ExXGSFB8nx4rF9UdXRp'
        'E6wzxxYYlcNvyMd/HParKbYtmh1nW7PTP3SN6ki+5QXwM4/pXLOqur3nnZRL6hUEFsD+VZvW+obm6kkCS4VuGJ/iIGP6Z/Ws+8pd'
        'iS33Oad0hOw17ey3MrNLIz5P8R80mSMsrY45FeEgdh9aGWIO/PHf86rbsdI8lfa3fIIqCbiOSK8lOSPnuRQ2cDOc8UCH0h2n6fHz'
        'QwwJNRL8MaE7nAA7Ec1GQ9dzncOag5K4we/cVE9sZqQPsGcDmlIeJ+HJxUh8DFeIGdtqqST9Oa2PSnSL3rLcXqtFF3Vf4m+v0FV5'
        'MkcauRLKXQtFvNYuBDbodndnb8Kj5NdT6d6fstIgxEgecj3ykcn6D4FO2enJZwJb20KxRr4Aqxt0IYL5+tcTyfJlmddIVuwCoQO1'
        'GhLL7mGKOiYDbvHmhH/8sfSuXONMVaCmdiuKXmkJXmpZ3DFDl7YPbzSzVjcbLjQJY9vpt5IzVrdIrQZPmqDSFDSA5xitFIUMGSRw'
        'Oa1L8KGhozV4h34+KrJ4Tv3Y5rQyAPISACPFI3seEJArM9MElasqjtVR80MHLAipzZJ7c19BGzHOKrcipKz/2Q=='
    ),
    'dark_eyed_junco_10.jpg': (
        '...'
        'DDQSGgqpEDq0qtYSIdiAd7YqX//Z'
    ),
    'dark_eyed_junco_11.jpg': (
        '...'
        'xkWryGFABMF223LyEcsUvbuMJV24n//Z'
    ),
}

**Module 3/4:** `src/bioclip2_biodiversity_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Classification metrics and trivial baselines for BioCLIP 2 species-classification adaptation.

Pure Python (no scikit-learn): accuracy, macro-F1, per-class precision/recall/F1/support, and AUROC
(binary: positive class = the last entry of `classes`; multiclass: macro one-vs-rest), computed by
the Mann-Whitney rank statistic with average ranks for ties. The zero-shot baseline lives on the
pipeline (`zero_shot_evaluate`) because it needs the model; the two baselines here need nothing.
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import decode_image` removed — names are kernel globals defined by the carried modules


def _prf(hits: int, n_pred: int, n_true: int) -> dict[str, float]:
    precision = hits / n_pred if n_pred else 0.0
    recall = hits / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)}


def auroc(y_true: Sequence[int], scores: Sequence[float]) -> float | None:
    """Area under the ROC curve for binary 0/1 labels; None when only one class is present."""
    n_pos = sum(1 for y in y_true if y == 1)
    n_neg = len(y_true) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    order = sorted(range(len(scores)), key=lambda i: scores[i])
    ranks = [0.0] * len(scores)
    i = 0
    while i < len(order):
        j = i
        while j + 1 < len(order) and scores[order[j + 1]] == scores[order[i]]:
            j += 1
        avg = (i + j + 2) / 2.0  # 1-based average rank of the tie block
        for k in range(i, j + 1):
            ranks[order[k]] = avg
        i = j + 1
    rank_sum = sum(r for r, y in zip(ranks, y_true, strict=True) if y == 1)
    return round((rank_sum - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg), 4)


def classification_metrics(
    y_true: Sequence[str],
    y_pred: Sequence[str],
    scores: Sequence[Sequence[float]] | None,
    classes: Sequence[str],
) -> dict[str, Any]:
    """Discrete and ranking metrics over one evaluation split (labels are class names).

    `scores[i][k]` is the score of class `classes[k]` for record i (softmax outputs from the
    pipeline; any monotone score works for AUROC). Class order is preserved exactly as given.
    """
    if len(y_true) != len(y_pred):
        raise ValueError(f"{len(y_true)} labels vs {len(y_pred)} predictions")
    class_list = list(classes)
    unknown = sorted((set(y_true) | set(y_pred)) - set(class_list))
    if unknown:
        raise ValueError(f"labels outside the class list {class_list}: {unknown}")
    n = len(y_true)
    correct = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == p)
    per_class: dict[str, dict[str, Any]] = {}
    f1s: list[float] = []
    for c in class_list:
        hits = sum(1 for t, p in zip(y_true, y_pred, strict=True) if t == c and p == c)
        n_pred = sum(1 for p in y_pred if p == c)
        n_true = sum(1 for t in y_true if t == c)
        prf = _prf(hits, n_pred, n_true)
        per_class[c] = {**prf, "support": n_true, "predicted": n_pred}
        if n_true:
            f1s.append(prf["f1"])
    result: dict[str, Any] = {
        "n": n,
        "accuracy": round(correct / n, 4) if n else 0.0,
        "macro_f1": round(sum(f1s) / len(f1s), 4) if f1s else 0.0,
        "per_class": per_class,
        "classes": class_list,
        "decision_rule": "argmax over class scores",
        "auroc": None,
        "auroc_definition": None,
    }
    if scores is not None and n:
        if len(scores) != n or any(len(row) != len(class_list) for row in scores):
            raise ValueError("scores must be one row per record with one column per class")
        if len(class_list) == 2:
            pos = class_list[-1]
            result["auroc"] = auroc([1 if t == pos else 0 for t in y_true], [row[-1] for row in scores])
            result["auroc_definition"] = f"binary AUROC with positive class {pos!r} (last class in the list)"
        else:
            values = []
            for k, c in enumerate(class_list):
                a = auroc([1 if t == c else 0 for t in y_true], [row[k] for row in scores])
                if a is not None:
                    values.append(a)
            result["auroc"] = round(sum(values) / len(values), 4) if values else None
            result["auroc_definition"] = "macro-averaged one-vs-rest AUROC over classes present in the split"
    return result


def majority_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Predict the most frequent training class for every evaluation record (EVAL11)."""
    counts: dict[str, int] = {}
    for r in train_records:
        counts[r["label"]] = counts.get(r["label"], 0) + 1
    majority = max(sorted(counts), key=counts.__getitem__)
    metrics = classification_metrics(
        [r["label"] for r in eval_records], [majority] * len(eval_records), None, classes
    )
    return {"baseline": "majority-class", "predicted_label": majority, **metrics}


def color_features(image_bytes: bytes) -> list[float]:
    """Six numbers per image: mean and standard deviation of each RGB channel on a 32x32 thumbnail."""
    im = decode_image(image_bytes).resize((32, 32))
    raw = im.tobytes()
    pixels = [raw[k : k + 3] for k in range(0, len(raw), 3)]
    n = float(len(pixels))
    means = [sum(p[c] for p in pixels) / n for c in range(3)]
    stds = [(sum((p[c] - means[c]) ** 2 for p in pixels) / n) ** 0.5 for c in range(3)]
    return [round(v / 255.0, 6) for v in means + stds]


def color_baseline(
    train_records: Sequence[Mapping[str, Any]],
    eval_records: Sequence[Mapping[str, Any]],
    classes: Sequence[str],
) -> dict[str, Any]:
    """Nearest class centroid in a six-number colour space, fitted on the training split only (SPL8).

    Colour is the confounder worth ruling out in a species task: if the classes have different
    plumage colours or typical backgrounds, a model can score well without seeing any structure.
    The tutorial's four species were picked to be similarly coloured so this baseline has little to
    use; on a user's own data it says how much of the task is colour.
    """
    class_list = list(classes)
    sums: dict[str, list[float]] = {c: [0.0] * 6 for c in class_list}
    counts: dict[str, int] = {c: 0 for c in class_list}
    for r in train_records:
        feats = color_features(r["image_bytes"])
        sums[r["label"]] = [a + b for a, b in zip(sums[r["label"]], feats, strict=True)]
        counts[r["label"]] += 1
    centroids = {c: [v / counts[c] for v in sums[c]] for c in class_list if counts[c]}
    predicted: list[str] = []
    for r in eval_records:
        feats = color_features(r["image_bytes"])
        best = min(
            sorted(centroids),
            key=lambda c: sum((a - b) ** 2 for a, b in zip(feats, centroids[c], strict=True)),
        )
        predicted.append(best)
    metrics = classification_metrics([r["label"] for r in eval_records], predicted, None, class_list)
    return {
        "baseline": "colour nearest-centroid (RGB mean+std on a 32x32 thumbnail)",
        "centroids": {c: [round(v, 4) for v in centroids[c]] for c in centroids},
        **metrics,
    }

**Module 4/4:** `src/bioclip2_biodiversity_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""The embedded sample dataset and the labelled-image-dataset contract for species classification.

The tutorial dataset is 48 real photographs — 12 each of four North American sparrow species
(Passerellidae) — taken from research-grade iNaturalist observations whose photos carry the CC0 1.0
licence, centre-cropped and resized to 224x224, and embedded in `sample_data.py` so the standalone
notebook needs no dataset download. The four species were chosen **a priori** to make colour a
weak cue by construction: all four are streaked or grey-brown "little brown birds", so a colour
baseline has little to work with and any separation the model achieves comes from finer structure
(head pattern, breast streaking, bill colour). This is sanity evidence for the adaptation contract
on real field data, not a benchmark: 48 images, one seeded split, no dispersion estimate.

Two honest caveats, stated here and in the tutorial. The images are unfiltered beyond licence and
quality grade — some subjects are small in the frame, one is held in a hand, and backgrounds vary —
which is what field data looks like. And a random split of observations from one region can share
photographers, locations and seasons across splits; the observers are recorded so a reader can see
how many distinct sources there are, but the pipeline cannot detect that kind of leakage for you.
"""

from __future__ import annotations

import base64
import csv
import hashlib
import json
import random
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import ACCEPTED_FORMATS, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, decode_image, image_digest` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .sample_data import (` removed — names are kernel globals defined by the carried modules

DATASET_REPRESENTATION = "io.github.kurtvalcorza.dataset.vision.image-labels.v1"
SAMPLE_CLASSES: tuple[str, ...] = (
    "chipping_sparrow",
    "dark_eyed_junco",
    "song_sparrow",
    "white_throated_sparrow",
)
# Scientific names are what BioCLIP was trained to match; the dataset keys are what the metrics report.
SAMPLE_CLASS_PROMPTS: dict[str, str] = {
    "chipping_sparrow": "Spizella passerina",
    "dark_eyed_junco": "Junco hyemalis",
    "song_sparrow": "Melospiza melodia",
    "white_throated_sparrow": "Zonotrichia albicollis",
}
SAMPLE_COMMON_NAMES: dict[str, str] = {
    "chipping_sparrow": "Chipping Sparrow",
    "dark_eyed_junco": "Dark-eyed Junco",
    "song_sparrow": "Song Sparrow",
    "white_throated_sparrow": "White-throated Sparrow",
}
SAMPLE_SIZE = len(SAMPLE_RECORDS)  # 48: 12 per species
MIN_RECORDS = 8
MAX_RECORDS = 5_000
MAX_CLASSES = 50
MIN_RECORDS_PER_CLASS = 3
MAX_ID_CHARS = 64
MAX_LABEL_CHARS = 64
REQUIRED_COLUMNS = ("id", "image", "label")  # `image` is a file path in CSV/JSON; bytes in memory
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".webp")


def generate_sample_dataset() -> list[dict[str, Any]]:
    """The 48 embedded sample images as `{id, image_bytes, label, ...provenance}` records.

    Deterministic by construction (the bytes are literals); `dataset_digest` over the result is the
    tutorial's dataset identity.
    """
    records: list[dict[str, Any]] = []
    for rec in SAMPLE_RECORDS:
        file = str(rec["file"])
        records.append(
            {
                "id": file.rsplit(".", 1)[0],
                "image_bytes": base64.b64decode(SAMPLE_IMAGES_B64[file]),
                "label": str(rec["label"]),
                "file": file,
                "scientific_name": rec["scientific_name"],
                "common_name": rec["common_name"],
                "source": rec["inat_observation_url"],
                "license": rec["license_code"],
                "observer": rec["observer"],
            }
        )
    return records


def sample_provenance() -> dict[str, Any]:
    """What the sample is and where it came from, for the dataset manifest and the card."""
    observers = sorted({str(r["observer"]) for r in SAMPLE_RECORDS})
    return {
        "source": SAMPLE_SOURCE,
        "image_license": SAMPLE_IMAGE_LICENSE,
        "image_side": SAMPLE_IMAGE_SIDE,
        "n_images": len(SAMPLE_RECORDS),
        "n_observers": len(observers),
        "classes": {
            c: {"scientific_name": SAMPLE_CLASS_PROMPTS[c], "common_name": SAMPLE_COMMON_NAMES[c]}
            for c in SAMPLE_CLASSES
        },
        "location_data": "not collected",
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over the canonical (id, image sha256, label) rows; recorded in provenance (OUT9)."""
    canon = json.dumps(
        [[r["id"], image_digest(r["image_bytes"]), r["label"]] for r in records], separators=(",", ":")
    )
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    classes: Sequence[str] | None = None,
    min_records: int = MIN_RECORDS,
    min_per_class: int = MIN_RECORDS_PER_CLASS,
) -> dict[str, Any]:
    """Check a labelled image dataset against the contract; return its manifest.

    Every error names the record and the violated rule (VAL4/DAT19). Images are decoded — that is
    the check — and their sizes and digests recorded; nothing verifies that an image shows the
    labelled organism, and the manifest says so.
    """
    if isinstance(records, str | bytes | Mapping) or not isinstance(records, Sequence):
        raise TypeError("records must be a list of {'id', 'image_bytes', 'label'} mappings")
    if len(records) < min_records:
        raise ValueError(f"dataset has {len(records)} records; at least {min_records} are required")
    if len(records) > MAX_RECORDS:
        raise ValueError(f"dataset has {len(records)} records; ceiling is {MAX_RECORDS}")
    seen_ids: set[str] = set()
    seen_digests: dict[str, str] = {}
    counts_by_class: dict[str, int] = {}
    widths: list[int] = []
    heights: list[int] = []
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in ("id", "image_bytes", "label") if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required field(s) {missing}; "
                "required: ['id', 'image_bytes', 'label']"
            )
        rid = str(rec["id"]).strip()
        if not rid or len(rid) > MAX_ID_CHARS:
            raise ValueError(f"record[{i}] id must be 1..{MAX_ID_CHARS} characters")
        if rid in seen_ids:
            raise ValueError(f"record[{i}] duplicates id {rid!r}")
        seen_ids.add(rid)
        data = rec["image_bytes"]
        if not isinstance(data, bytes | bytearray) or not data:
            raise ValueError(f"record[{i}] ({rid}) image_bytes must be non-empty encoded image bytes")
        try:
            im = decode_image(bytes(data))
        except (ValueError, TypeError) as exc:
            raise ValueError(f"record[{i}] ({rid}) {exc}") from exc
        digest = image_digest(bytes(data))
        if digest in seen_digests:
            raise ValueError(f"record[{i}] ({rid}) duplicates the image of {seen_digests[digest]!r}")
        seen_digests[digest] = rid
        widths.append(im.size[0])
        heights.append(im.size[1])
        label = rec["label"]
        if not isinstance(label, str) or not label.strip() or len(label) > MAX_LABEL_CHARS:
            raise ValueError(
                f"record[{i}] ({rid}) label must be a non-empty string of at most {MAX_LABEL_CHARS} chars"
            )
        counts_by_class[label] = counts_by_class.get(label, 0) + 1
    if classes is None:
        class_list = sorted(counts_by_class)
    else:
        class_list = [str(c) for c in classes]
        unknown = sorted(set(counts_by_class) - set(class_list))
        if unknown:
            raise ValueError(f"labels {unknown} are not in the class list {class_list}")
    if len(class_list) < 2:
        raise ValueError(f"classification needs at least 2 classes, found {class_list}")
    if len(class_list) > MAX_CLASSES:
        raise ValueError(f"{len(class_list)} classes exceeds the ceiling of {MAX_CLASSES}")
    thin = [c for c in class_list if counts_by_class.get(c, 0) < min_per_class]
    if thin:
        raise ValueError(f"classes {thin} have fewer than {min_per_class} records each (class coverage rule)")
    return {
        "verdict": "accepted",
        "representation": DATASET_REPRESENTATION,
        "validation": (
            "decode, format and pixel-size checks only; "
            "nothing verifies that an image shows the labelled organism"
        ),
        "n_records": len(records),
        "classes": class_list,
        "class_counts": {c: counts_by_class.get(c, 0) for c in class_list},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "ceilings": {
            "min_image_side": MIN_IMAGE_SIDE,
            "max_image_side": MAX_IMAGE_SIDE,
            "accepted_formats": list(ACCEPTED_FORMATS),
            "max_records": MAX_RECORDS,
            "max_classes": MAX_CLASSES,
            "min_records": min_records,
            "min_records_per_class": min_per_class,
        },
        "digest": dataset_digest(records),
        "findings": [],
    }


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 42,
) -> dict[str, list[dict[str, Any]]]:
    """Stratified random train/validation/test split (assumes independent images, SPL3).

    Field-image datasets are rarely independent: several photos of one individual, one
    photographer's style, one site's background. A split by observation, site or photographer is
    the right tool there; the tutorial sample keeps one photo per observation and per observer
    within a species, which is why a random split is defensible here, not because it is generally.
    """
    if not (0.0 < val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("val_fraction and test_fraction must be in (0, 1) and sum to less than 1")
    manifest = validate_dataset(records)
    rng = random.Random(seed)
    by_class: dict[str, list[dict[str, Any]]] = {c: [] for c in manifest["classes"]}
    for rec in records:
        by_class[rec["label"]].append(dict(rec))
    out: dict[str, list[dict[str, Any]]] = {"train": [], "validation": [], "test": []}
    for cls in manifest["classes"]:
        rows = by_class[cls]
        rng.shuffle(rows)
        n_val = max(1, round(len(rows) * val_fraction))
        n_test = max(1, round(len(rows) * test_fraction))
        if len(rows) - n_val - n_test < 1:
            raise ValueError(f"class {cls!r} has {len(rows)} records; too few to leave one per split")
        out["validation"].extend(rows[:n_val])
        out["test"].extend(rows[n_val : n_val + n_test])
        out["train"].extend(rows[n_val + n_test :])
    for part in out.values():
        rng.shuffle(part)
    return out


def load_byod_dataset(source: str | Path) -> list[dict[str, Any]]:
    """Read a user-supplied dataset: a CSV with `id,image,label` (image paths relative to the CSV),
    a JSON array / JSONL of `{id, image, label}` objects, or a directory of `<label>/<image files>`.

    Image files are read as bytes and nothing about them is rewritten (VAL7). The records are then
    validated with `validate_dataset`, whose errors name the offending row.
    """
    path = Path(source)
    records: list[dict[str, Any]] = []
    if path.is_dir():
        for label_dir in sorted(p for p in path.iterdir() if p.is_dir()):
            for img in sorted(p for p in label_dir.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES):
                records.append(
                    {"id": f"{label_dir.name}/{img.name}", "image": str(img), "label": label_dir.name}
                )
        if not records:
            raise ValueError(f"no <label>/<image> files found under {path}")
        base = path
    elif path.is_file():
        text = path.read_text(encoding="utf-8-sig")
        if not text.strip():
            raise ValueError(f"BYOD dataset file is empty: {path}")
        suffix = path.suffix.lower()
        if suffix == ".csv":
            reader = csv.DictReader(text.splitlines())
            header = [h.strip() for h in (reader.fieldnames or [])]
            missing = [c for c in REQUIRED_COLUMNS if c not in header]
            if missing:
                raise ValueError(f"CSV header {header} is missing required column(s) {missing}")
            for row in reader:
                records.append({c: (row.get(c) or "").strip() for c in REQUIRED_COLUMNS})
        elif suffix == ".jsonl":
            for line_no, line in enumerate(text.splitlines(), start=1):
                if not line.strip():
                    continue
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"line {line_no} is not valid JSON: {exc}") from exc
        elif suffix == ".json":
            try:
                data = json.loads(text)
            except json.JSONDecodeError as exc:
                raise ValueError(f"file is not valid JSON: {exc}") from exc
            if not isinstance(data, list):
                raise TypeError("JSON dataset must be a top-level array of objects")
            records = data
        else:
            raise ValueError(f"unsupported BYOD file type {suffix!r}; use .csv, .json, .jsonl or a directory")
        base = path.parent
    else:
        raise FileNotFoundError(f"BYOD dataset not found: {path}")
    out: list[dict[str, Any]] = []
    for i, rec in enumerate(records):
        if not isinstance(rec, Mapping):
            raise TypeError(f"record[{i}] must be a mapping, got {type(rec).__name__}")
        missing = [c for c in REQUIRED_COLUMNS if c not in rec]
        if missing:
            raise ValueError(
                f"record[{i}] is missing required column(s) {missing}; required: {list(REQUIRED_COLUMNS)}"
            )
        img_path = Path(str(rec["image"]))
        if not img_path.is_absolute():
            img_path = base / img_path
        if not img_path.is_file():
            raise FileNotFoundError(f"record[{i}] ({rec['id']}) image file not found: {img_path}")
        out.append(
            {
                "id": str(rec["id"]),
                "image_bytes": img_path.read_bytes(),
                "label": str(rec["label"]),
                "file": img_path.name,
            }
        )
    validate_dataset(out)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the images to `<path's directory>/images/` and a CSV (`id,image,label`) beside them,
    so users have a BYOD template that `load_byod_dataset` reads back."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    img_dir = out.parent / "images"
    img_dir.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(REQUIRED_COLUMNS)
        for r in records:
            name = r.get("file") or f"{r['id']}.jpg"
            (img_dir / name).write_bytes(r["image_bytes"])
            writer.writerow([r["id"], f"images/{name}", r["label"]])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `2957b322090f…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BioClip2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bioclip-2",
  "modelId": "imageomics/bioclip-2",
  "revision": "2957b322090f9cb17ae72c71981c7218a28d81e0",
  "files": [
    {
      "path": "README.md",
      "bytes": 20723,
      "sha256": "807e4daa2b3486bc542a7c796c84a117be02818e671c934a66cd1898c3c098c1"
    },
    {
      "path": "open_clip_config.json",
      "bytes": 534,
      "sha256": "1bf947e96e943fe50efd5c3e26c37f843a2fa3c358967719a68c8a6d17ce68c8"
    },
    {
      "path": "open_clip_model.safetensors",
      "bytes": 1710517724,
      "sha256": "b7b2bf6fbc95799e42630e394cf95803892ab447c1a8ab629dbc82fbeaf7dfef"
    }
  ],
  "totalBytes": 1710538981
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BioClip2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample photographs, validation and split

The default dataset is carried inside this notebook: 48 JPEG photographs, 224x224, decoded from base64 in the `sample_data` module above, each traceable to its iNaturalist observation (the observation URL, photo id and observer login are recorded; the CC0 licence code was checked per photo when the set was built; no location was collected). `validate_dataset` decodes every image — that is the check — and reports sizes, class counts, the ceilings and a dataset digest before any model runs. `split_dataset` shuffles within each class and cuts 20 % validation / 25 % test; the sample keeps one photo per observer within a species, which is what makes a random split defensible here.

Look for: 48 images, four classes, 12 each, splits 28/8/12, the contact sheet, and a written `outputs/bioclip2_biodiversity_sample_dataset.csv` with an `images/` folder beside it — the BYOD shape. The photographs are unfiltered beyond licence and quality grade: some birds are small in the frame, one is held in a hand. That is what field data looks like.

In [ ]:
import io
import json
import os
import zipfile
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}
BYOD_CLASS_PROMPTS = {}  # e.g. {'song_sparrow': 'Melospiza melodia'}: one scientific name per BYOD label

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_root = Path('work') / 'byod'
    byod_root.mkdir(parents=True, exist_ok=True)
    if file_name.lower().endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(payload)) as zf:
            for member in zf.infolist():
                target = (byod_root / member.filename).resolve()
                if not str(target).startswith(str(byod_root.resolve())):
                    raise ValueError('zip member escapes the upload directory: ' + member.filename)
                if not member.is_dir():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(zf.read(member))
        tables = sorted(p for p in byod_root.rglob('*') if p.suffix.lower() in ('.csv', '.json', '.jsonl'))
        byod_source = tables[0] if tables else byod_root
    else:
        byod_source = byod_root / file_name
        byod_source.write_bytes(payload)
    records = load_byod_dataset(byod_source)
    data_source = 'BYOD (' + file_name + ')'
    class_prompts = dict(BYOD_CLASS_PROMPTS)
else:
    records = generate_sample_dataset()
    data_source = 'embedded CC0 iNaturalist sample (' + str(SAMPLE_SIZE) + ' photographs, four sparrow species)'
    class_prompts = dict(SAMPLE_CLASS_PROMPTS)

dataset_manifest = validate_dataset(records)
CLASSES = dataset_manifest['classes']
missing_prompts = [c for c in CLASSES if c not in class_prompts]
if missing_prompts:
    raise ValueError('supply a scientific name for every class in BYOD_CLASS_PROMPTS: ' + str(missing_prompts))
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(records, 'outputs/bioclip2_biodiversity_sample_dataset.csv')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'classes': CLASSES, 'class_counts': dataset_manifest['class_counts']})
print({'class_prompts': class_prompts, 'validation': dataset_manifest['validation']})
print({'image_width': dataset_manifest['image_width'], 'image_height': dataset_manifest['image_height'], 'ceilings': dataset_manifest['ceilings'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})
if not USE_BYOD:
    print({'provenance': sample_provenance()})

from PIL import Image

thumb, per_row = 96, 12
shown = records[:48]
sheet = Image.new('RGB', (per_row * thumb, ((len(shown) + per_row - 1) // per_row) * thumb), 'white')
for i, r in enumerate(shown):
    sheet.paste(decode_image(r['image_bytes']).resize((thumb, thumb)), ((i % per_row) * thumb, (i // per_row) * thumb))
sheet.save('outputs/bioclip2_biodiversity_contact_sheet.png')
try:
    from IPython.display import display
    display(sheet)
except ImportError:
    print('contact sheet written to outputs/bioclip2_biodiversity_contact_sheet.png')

## 5. What the input validation accepts and refuses

`validate_inputs` decodes each image with Pillow, records its size and SHA-256, and raises exactly what `embed_images` or `classify` would raise. The ceilings are the checkpoint's and the runtime's, not biology's: a shorter side of at least `MIN_IMAGE_SIDE` px, a longer side of at most `MAX_IMAGE_SIDE` px, JPEG/PNG/WEBP, at most `MAX_IMAGES_PER_CALL` images per call and `MAX_LABELS` labels per zero-shot call. The cell shows three refusals — undecodable bytes, an unsupported format, a too-small image — and one acceptance. What it cannot show is a *semantic* refusal, because there is none: the schema says so in words.

In [ ]:
print({'max_images_per_call': MAX_IMAGES_PER_CALL, 'image_side_pixels': [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE], 'accepted_formats': list(ACCEPTED_FORMATS), 'max_labels': MAX_LABELS, 'image_size': IMAGE_SIZE})
print({'validation': INPUT_SCHEMA['validation']})

def encoded(image, fmt):
    buf = io.BytesIO()
    image.save(buf, fmt)
    return buf.getvalue()

probes = {
    'garbage bytes': b'not an image',
    'BMP file': encoded(Image.new('RGB', (64, 64), (200, 30, 30)), 'BMP'),
    '16x16 image': encoded(Image.new('RGB', (16, 16)), 'JPEG'),
}
for name, payload in probes.items():
    try:
        validate_inputs([payload])
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

input_manifest = validate_inputs([r['image_bytes'] for r in test_records[:4]], names=[r['id'] for r in test_records[:4]])
print({'verdict': input_manifest['verdict'], 'n_images': input_manifest['n_images'], 'requires_remote_code': input_manifest['requires_remote_code'], 'first': input_manifest['inputs'][0]})

## 6. Image embeddings (representation, not prediction)

`pipe.embed_images` runs the verified image tower and returns one L2-normalised 768-dimensional vector per image — a point in the joint image/text space. Embeddings are representations: they carry no label and no metric of their own; a downstream task is what gives them meaning (EVAL9). The cell embeds the eight validation images, writes them with their ids to `outputs/bioclip2_biodiversity_embeddings.csv` (OUT4), and prints the mean cosine similarity within and between species as an inspection, not an evaluation.

It also checks reproducibility the honest way: calling the same batch twice returns identical vectors, and embedding one of those images on its own differs by at most ~1e-7 — float accumulation order changes with the batch, which is noise, not stochasticity. A reader who needs bit-identical embeddings across batch sizes must fix the batch composition.

In [ ]:
import csv
import math

embed_records = val_records[:8]
embedding_result = pipe.embed_images([r['image_bytes'] for r in embed_records], names=[r['id'] for r in embed_records])
vectors = embedding_result['embeddings']
print({'n_images': embedding_result['n_images'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling'], 'norm_of_first': round(math.sqrt(sum(x * x for x in vectors[0])), 6)})

repeat = pipe.embed_images([r['image_bytes'] for r in embed_records], names=[r['id'] for r in embed_records])['embeddings']
single = pipe.embed_images([embed_records[0]['image_bytes']])['embeddings'][0]
cross_batch = max(abs(a - b) for a, b in zip(single, vectors[0]))
print({'same_batch_twice_identical': repeat == vectors, 'single_vs_batch_max_abs_diff': cross_batch, 'note': 'batch composition changes float accumulation order; this is noise at the 1e-7 level, not stochastic inference'})
assert repeat == vectors
assert cross_batch < 1e-5

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))

within, between = [], []
for i in range(len(embed_records)):
    for j in range(i + 1, len(embed_records)):
        (within if embed_records[i]['label'] == embed_records[j]['label'] else between).append(cosine(vectors[i], vectors[j]))
print({'mean_cosine_within_species': round(sum(within) / len(within), 4) if within else None, 'mean_cosine_between_species': round(sum(between) / len(between), 4) if between else None, 'note': 'inspection only; embeddings are unlabelled representations'})

with open('outputs/bioclip2_biodiversity_embeddings.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'label', 'sha256'] + [f'dim_{k}' for k in range(embedding_result['dimension'])])
    for r, meta, vec in zip(embed_records, embedding_result['images'], vectors):
        writer.writerow([r['id'], r['label'], meta['sha256']] + [f'{x:.6f}' for x in vec])
print('wrote outputs/bioclip2_biodiversity_embeddings.csv')

## 7. Zero-shot species classification — the model's own prior

`pipe.zero_shot` embeds one prompt per label (`a photo of <scientific name>.` by default, BioCLIP's convention), embeds the images, and takes a softmax over the scaled cosine similarities. No training is involved: this is what the pretrained model already knows about these species, and it is the baseline the fine-tuning in Section 9 must be read against. `zero_shot_evaluate` scores the whole test split this way.

Three probes make the score semantics concrete (UNC1–UNC4). The scores are a softmax over **the label set you supplied and nothing else**: a blank grey image and a noise image still receive a confident label, because closed-set zero-shot always answers. An open-set probe — the same sparrow photo against `a rock`, `a car`, its species and `a domestic cat` — shows the model choosing the species. And common names are tried alongside scientific names, because which vocabulary the text tower knows best is an empirical question on your taxa.

In [ ]:
baseline_zero_shot = pipe.zero_shot_evaluate(test_records, class_prompts)
print({k: baseline_zero_shot[k] for k in ('baseline', 'template', 'n', 'accuracy', 'macro_f1', 'auroc')})
for cls_name, row in baseline_zero_shot['per_class'].items():
    print({'class': cls_name, 'prompt': DEFAULT_PROMPT_TEMPLATE.format(class_prompts[cls_name]), **row})

if not USE_BYOD:
    common = pipe.zero_shot_evaluate(test_records, SAMPLE_COMMON_NAMES)
    print({'with_common_names': {k: common[k] for k in ('accuracy', 'macro_f1')}, 'note': 'same images, common names in the prompt instead of scientific names'})

example = test_records[0]
single = pipe.zero_shot([example['image_bytes']], class_prompts, names=[example['id']])
print({'id': example['id'], 'true_label': example['label'], 'predicted': single['predictions'][0]['label'], 'scores': {k: round(v, 4) for k, v in single['predictions'][0]['scores'].items()}, 'logit_scale': round(single['logit_scale'], 2)})
print({'decision_rule': single['decision_rule']})

open_set = pipe.zero_shot([example['image_bytes']], ['a rock', 'a car', class_prompts[example['label']], 'a domestic cat'])
print({'open_set_probe': {k: round(v, 4) for k, v in open_set['predictions'][0]['scores'].items()}})

blank = encoded(Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128)), 'JPEG')
import random
rnd = random.Random(0)
noise = encoded(Image.frombytes('RGB', (IMAGE_SIZE, IMAGE_SIZE), bytes(rnd.getrandbits(8) for _ in range(IMAGE_SIZE * IMAGE_SIZE * 3))), 'JPEG')
for name, payload in (('blank grey image', blank), ('uniform noise image', noise)):
    p = pipe.zero_shot([payload], class_prompts)['predictions'][0]
    print({'probe': name, 'label': p['label'], 'score': round(p['score'], 4), 'note': 'closed-set zero-shot always answers; a confident label is not evidence of a bird'})

## 8. Trivial baselines on the test split

Two predictors that know nothing about birds set the floor (EVAL10/EVAL11). `majority_baseline` predicts the most frequent training class — 0.25 on a balanced four-class split. `color_baseline` reduces each image to six numbers (mean and standard deviation of R, G and B on a 32x32 thumbnail), fits one centroid per class on the **training split only** (SPL8) and assigns each test image to the nearest centroid.

Colour is the confounder worth ruling out in a species task, and the four species were chosen to make it weak. On your own data, read this baseline first: if colour already separates your classes — a red bird versus a yellow one, or two species photographed against different backgrounds — a model can score well without having seen any structure.

In [ ]:
baseline_majority = majority_baseline(train_records, test_records, CLASSES)
print({k: baseline_majority[k] for k in ('baseline', 'predicted_label', 'accuracy', 'macro_f1')})
baseline_color = color_baseline(train_records, test_records, CLASSES)
print({k: baseline_color[k] for k in ('baseline', 'accuracy', 'macro_f1')})
print({'centroids_rgb_mean': {c: v[:3] for c, v in baseline_color['centroids'].items()}, 'note': 'six-number colour space; fitted on the training split only'})

## 9. Bounded fine-tuning, starting from the zero-shot classifier

`pipe.adapt` copies the image tower, puts a linear head over its L2-normalised projection, and — because `class_prompts` supplies a scientific name per class — **initialises that head from the zero-shot text classifier** (the text embeddings scaled by the checkpoint's logit scale). Epoch 0 in the history is therefore the zero-shot classifier evaluated on the validation split, and every later epoch is comparable to it. AdamW then runs with the hyperparameters below (FT4/FT6): tutorial values, not production settings. Validation metrics are **monitoring only**; the final epoch's weights are kept (EVAL14). Training loss going down is optimisation evidence, not task-quality evidence (FT7) — Section 10 is where quality is measured.

`TRAINABLE_BLOCKS = 0` trains the head alone on features the frozen tower computes once, which is what 28 training images support. Setting it to 1 unfreezes the last of the 24 tower blocks: on the sample that took a saturated zero-shot classifier from 12/12 to 11/12 on the test split — 12.6 million parameters pulled around by 28 images — which is the recorded reason the default is 0.

In [ ]:
import time

EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 1e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_BLOCKS = 0  # @param {type:"integer"}

started = time.perf_counter()
adapt_result = pipe.adapt(
    train_records,
    val_records,
    classes=CLASSES,
    class_prompts=class_prompts,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    trainable_blocks=TRAINABLE_BLOCKS,
    seed=SEED,
)
adapt_seconds = round(time.perf_counter() - started, 2)
print({'method': adapt_result['method'], 'head_initialisation': adapt_result['head_initialisation'], 'trainable_parameters': adapt_result['trainable_parameters'], 'total_parameters': adapt_result['total_parameters'], 'precision': adapt_result['precision'], 'device': pipe.device, 'seconds': adapt_seconds})
for step in adapt_result['history']:
    print(step)

## 10. Held-out evaluation

`pipe.evaluate` classifies every image of a split with the adapted head and reports `accuracy`, `macro_f1` (the unweighted mean of per-class F1, which exposes a model that ignores a class), per-class precision/recall/F1 with support, and `auroc` (macro one-vs-rest, ranking quality independent of the argmax). The **test split** was never used for training or monitoring, so its numbers are the independent evidence (SPL6/SPL7). These are tutorial metrics on a 12-image split (EVAL6): one holdout, no dispersion estimate. The report — with the zero-shot, majority and colour baselines and the deltas against them — is written to `outputs/bioclip2_biodiversity_evaluation_report.json`.

Read the delta against **zero-shot** first. On the sample it is zero, because the pretrained model already separates these four species perfectly and head-only adaptation preserves that; the value of adaptation shows on a dataset where zero-shot is *not* saturated — a local morph, a poorly named taxon, a camera-trap angle — which is what BYOD is for.

In [ ]:
val_metrics = pipe.evaluate(val_records)
test_metrics = pipe.evaluate(test_records)
print({'split': 'validation', **{k: val_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
print({'split': 'test', **{k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
for cls_name, row in test_metrics['per_class'].items():
    print({'class': cls_name, **row})

evaluation_report = {
    'task': 'species classification (bounded fine-tuning of a BioCLIP 2 head initialised from zero-shot)',
    'evidence': 'tutorial sample-sanity metrics on one stratified holdout of 48 field photographs; not a benchmark',
    'estimation': 'single train/validation/test split, seed ' + str(SEED) + ', no dispersion estimate',
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'classes': CLASSES,
    'class_prompts': class_prompts,
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'zero_shot': baseline_zero_shot, 'majority': baseline_majority, 'color': baseline_color},
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'delta_vs_zero_shot': {k: round(test_metrics[k] - baseline_zero_shot[k], 4) for k in ('accuracy', 'macro_f1')},
    'delta_vs_majority': {k: round(test_metrics[k] - baseline_majority[k], 4) for k in ('accuracy', 'macro_f1')},
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'trainable_parameter_names'},
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/bioclip2_biodiversity_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
print({'delta_vs_zero_shot': evaluation_report['delta_vs_zero_shot'], 'delta_vs_majority': evaluation_report['delta_vs_majority'], 'report': 'outputs/bioclip2_biodiversity_evaluation_report.json'})

## 11. Inference on held-out images, artifact export and fresh reload

`pipe.classify` returns, per image, the argmax `label`, its `score`, the full `scores` dictionary in class order and the image digest. The scores are softmax outputs of a head trained on a few dozen images — **not calibrated probabilities** (UNC2); the only decision rule is argmax (UNC3). The six inputs here are held-out test images the head never saw in training.

`pipe.save_artifact` writes the trained tensors — with the default head-only setting, the 4x768 weight and the 4-entry bias, about 12 KB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the class order, the tensor names, the file size and SHA-256, and the adaptation configuration including how the head was initialised (OUT8). `BioClip2Pipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digests **before** deserialising, rebuilds the classifier and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical labels and scores within `1e-5` (VER4).

In [ ]:
new_records = test_records[:6]
new_source = 'six held-out test-split images (never used for training or monitoring)'
inference_result = pipe.classify([r['image_bytes'] for r in new_records], names=[r['id'] for r in new_records])
predictions = inference_result['predictions']
print({'new_source': new_source, 'decision_rule': inference_result['decision_rule']})
n_match = 0
for p, r in zip(predictions, new_records):
    n_match += p['label'] == r['label']
    print({'id': p['id'], 'predicted': p['label'], 'score': round(p['score'], 4), 'true_label': r['label']})
print({'matches': n_match, 'of': len(new_records), 'note': 'sanity check on held-out images, not an evaluation'})

with open('outputs/bioclip2_biodiversity_predictions.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'sha256', 'predicted_label', 'score'] + [f'score_{c}' for c in CLASSES])
    for p in predictions:
        writer.writerow([p['id'], p['sha256'], p['label'], f"{p['score']:.6f}"] + [f"{p['scores'][c]:.6f}" for c in CLASSES])

artifact_dir = Path('outputs/bioclip2_biodiversity_adapter')
pipe.save_artifact(artifact_dir, metadata={'data_source': data_source, 'dataset_digest': dataset_manifest['digest'], 'test_metrics': {k: test_metrics[k] for k in ('n', 'accuracy', 'macro_f1', 'auroc')}})
with open(artifact_dir / ARTIFACT_MANIFEST_NAME, encoding='utf-8') as f:
    artifact_manifest = json.load(f)
print({'format': artifact_manifest['format'], 'base_model': artifact_manifest['base_model'], 'requires_remote_code': artifact_manifest['requires_remote_code'], 'n_tensors': len(artifact_manifest['tensors']), 'tensors': artifact_manifest['tensors'], 'files': artifact_manifest['files']})

reloaded_pipe = BioClip2Pipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.classify([r['image_bytes'] for r in new_records], names=[r['id'] for r in new_records])
max_score_diff = 0.0
for before, after in zip(predictions, reloaded_result['predictions']):
    assert before['id'] == after['id'] and before['label'] == after['label'], f'reload parity failure on {before["id"]}'
    max_score_diff = max(max_score_diff, abs(before['score'] - after['score']))
assert max_score_diff < 1e-5, f'reload score drift {max_score_diff}'
print({'reload_parity': 'PASS', 'labels_equal': True, 'max_abs_score_diff': max_score_diff})

## 12. Result export and provenance

The last output, `outputs/bioclip2_biodiversity_result.json`, gathers what a reader needs to interpret the files above: the notebook source revision, the model id, immutable revision and licence, the dataset source, digest and (for the sample) its iNaturalist provenance summary, the class prompts, the adaptation configuration, the zero-shot and trivial baselines and held-out metrics, the held-out predictions, the artifact manifest, the reload-parity result, and the runtime versions and device (OUT6/OUT7). No credential is involved anywhere in this notebook, so none can leak into it (OUT10).

In [ ]:
import platform

result_payload = {
    'task': 'species classification adaptation (BioCLIP 2)',
    'pipeline_class': 'BioClip2Pipeline',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'remote_code_executed': False,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'notebook_source': NOTEBOOK_SOURCE,
    'data_source': data_source,
    'sample_provenance': None if USE_BYOD else sample_provenance(),
    'dataset_manifest': dataset_manifest,
    'class_prompts': class_prompts,
    'embedding_summary': {'n_images': embedding_result['n_images'], 'dimension': embedding_result['dimension'], 'pooling': embedding_result['pooling'], 'single_vs_batch_max_abs_diff': cross_batch},
    'evaluation_report': evaluation_report,
    'inference': {'new_source': new_source, 'decision_rule': inference_result['decision_rule'], 'predictions': predictions},
    'artifact_format': ARTIFACT_FORMAT,
    'artifact_format_version': ARTIFACT_FORMAT_VERSION,
    'artifact_manifest': artifact_manifest,
    'reload_parity': {'labels_equal': True, 'max_abs_score_diff': max_score_diff},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'open_clip': open_clip.__version__,
        'safetensors': safetensors.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/bioclip2_biodiversity_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The pretrained model separates four similarly coloured sparrow species from their scientific names alone, and a head fine-tuned from that starting point keeps the separation on an independent split. That is the claim: BioCLIP 2's joint space already encodes fine-grained taxonomic structure that colour does not explain, and the adaptation contract preserves it rather than destroying it. The test split has 12 photographs, the metrics come from one seeded holdout with no dispersion estimate, and the four species are common, well-photographed North American birds — the easy end of biodiversity. So a perfect score says the contract works on this sample, not that BioCLIP 2 identifies rare, cryptic or poorly represented taxa, and not that it reproduces any published benchmark.

Three things to carry to real data. **Zero-shot first:** run `zero_shot_evaluate` on your labelled set before you train anything — if it is already saturated, adaptation has nothing to add, and if it is poor, check the prompt vocabulary (scientific versus common names, full taxonomic strings) before blaming the model. **Splits:** photographs of one individual, one photographer or one site leak across a random split; split by observation, site or photographer. **Unfreezing the tower:** with a few dozen images, even one unfrozen block degraded a perfect zero-shot classifier here; unfreeze only with hundreds of images per class and a validation split you trust.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model, build it from its pinned config, validate the demonstrated dataset contract, classify zero-shot, execute bounded fine-tuning, evaluate against trivial baselines on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or that any photograph shows what its label says.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_BLOCKS = 1` to watch the last tower block over-fit 28 images; change the prompt template in `pipe.zero_shot` (for example `'a photo of the bird {}.'`) and compare; or bring your own labelled set through BYOD with scientific names in `BYOD_CLASS_PROMPTS` and read the zero-shot and colour baselines first.

## References

- Repository README: https://github.com/kurtvalcorza/bioclip2-biodiversity-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bioclip2-biodiversity-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/imageomics/bioclip-2
- Upstream code: https://github.com/Imageomics/bioclip-2
- Gu, J., Stevens, S., Campolongo, E. G., et al. (2025). BioCLIP 2: Emergent Properties from Scaling Hierarchical Contrastive Learning. NeurIPS 2025. arXiv:2505.23883. https://arxiv.org/abs/2505.23883
- Sample photographs: iNaturalist research-grade observations with CC0 1.0 photo licences; per-image observation URLs are recorded in the carried `sample_data` module.